# Water consumption of U.S. hyperscale data centers — reproducibility notebook

This notebook supports the manuscript **The hidden water geography of U.S. hyperscale data centers in the AI era**.

It is configured for a GitHub repository layout. Input files should be placed under `data/raw/`; generated figures and tables are written to `outputs/`.

Key outputs:
- facility-level and pathway-resolved water estimates;
- hydrologic-basin Scope 1 maps and tables;
- balancing-authority Scope 2 maps and tables;
- hotspot typologies and manuscript-ready figures.

Facility-level raw data are not included in this repository if restricted by data-use agreements.


In [ ]:
# ==============
# 0) Imports
# ==============
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

print("Versions:",
      "pandas", pd.__version__,
      "| geopandas", gpd.__version__,
      "| matplotlib", mpl.__version__)


In [ ]:
# ==============
# 1) CONFIG (keep your paths + column names)
# ==============
CONFIG = {
    # Facilities:
    "FACILITIES_GEOFILE": None,   # e.g. "data/gdf_h.parquet" (GeoParquet / GeoPackage)
    "FACILITIES_TABLE": "data/raw/facilities.csv",

    # BA polygons (must include region_B_1):
    "BA_POLYGONS": "data/raw/balancing_authorities_polygons/balancing_authorities_EPA.shp",

    # BA mix table (your file: BA_egrid.xlsx):
    "BA_MIX_TABLE": "data/raw/BA_egrid.xlsx",

    # HydroBASINS polygons (must include PFAF_ID) + Aqueduct (optional):
    "BASINS_POLYGONS": "data/raw/hydrobasins/hybas_lake_na_lev06_v1c.shp",
    "AQUEDUCT_TABLE": "data/raw/aqueduct/Aqueduct40_baseline_monthly_y2023m07d05.csv",

    # Lower‑48 states boundaries:
    "STATES48": "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_state_20m.zip",

    # Output folder
    "OUTDIR": "outputs",
}

COLS = {
    "lat": "latitude",
    "lon": "longitude",
    "mw": "current_mw",
    "company": "company_name",
    "subregion": "subregion_id",

    # Water (m³/yr) — if present we will use them
    "s1_m3": "s1_m3_mid",
    "s2_m3": "s2_m3_mid",
    "tot_m3": "total_m3_mid",

    # Facility stress / basin id
    "bws": "bws_01_score",
    "pfaf": "pfaf_id",

    # Grid attribute (we will create this from BA_egrid.xlsx if missing)
    "grid_L_per_kwh": "grid_L_per_kwh",
}

SCEN = {
    "HIGH_STRESS_THRESHOLD": 3.0,
    "FACILITY_MWH_COLUMN": "mwh_fac_A_mid",   # if present, used for Scope-2 water if needed
    "FALLBACK_UTILIZATION": 0.66,             # only used if no MWh column is available
}

# Lambert Conformal Conic (Lower‑48)
LCC = "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 +datum=WGS84 +units=m +no_defs"

# Lower‑48 clipping box in lon/lat
L48_EXTENT = (-125, -66.5, 24, 49.5)

os.makedirs(CONFIG["OUTDIR"], exist_ok=True)
print("OUTDIR:", os.path.abspath(CONFIG["OUTDIR"]))


In [ ]:
# ==============
# 2) Helpers (fixes previous notebook errors)
# ==============
def safe_num(x):
    """
    Convert scalars/Series/arrays to numeric safely.
    Returns:
      - pandas Series if input is Series/Index/array-like
      - float (or np.nan) if input is scalar
    """
    # pandas Series / Index
    if isinstance(x, (pd.Series, pd.Index)):
        out = pd.to_numeric(x, errors="coerce")
        return out.replace([np.inf, -np.inf], np.nan)

    # array-like (list, np.ndarray)
    if isinstance(x, (list, tuple, np.ndarray)):
        out = pd.to_numeric(pd.Series(x), errors="coerce")
        return out.replace([np.inf, -np.inf], np.nan)

    # scalar
    out = pd.to_numeric(x, errors="coerce")
    if np.isinf(out):
        return np.nan
    return out


def require_cols(df, cols, name="dataframe"):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"{name} missing required columns: {missing}")

def _read_tabular(path):
    """Read csv / parquet / xlsx (first sheet by default)."""
    if path is None:
        raise ValueError("Path is None")
    path = str(path)

    if path.lower().endswith((".parquet", ".pq")):
        return pd.read_parquet(path)
    if path.lower().endswith(".csv"):
        return pd.read_csv(path)
    if path.lower().endswith((".xlsx", ".xls")):
        # default: first sheet
        return pd.read_excel(path)
    raise ValueError(f"Unsupported table format: {path}")

def to_wgs84(gdf):
    if gdf.crs is None:
        return gdf.set_crs("EPSG:4326")
    return gdf.to_crs("EPSG:4326")

def bbox_clip_points(gdf_pts):
    # assumes EPSG:4326
    return gdf_pts[
        (gdf_pts.geometry.x >= L48_EXTENT[0]) & (gdf_pts.geometry.x <= L48_EXTENT[1]) &
        (gdf_pts.geometry.y >= L48_EXTENT[2]) & (gdf_pts.geometry.y <= L48_EXTENT[3])
    ].copy()

def bbox_clip_poly(gdf_poly):
    # assumes EPSG:4326
    return gdf_poly.cx[L48_EXTENT[0]:L48_EXTENT[1], L48_EXTENT[2]:L48_EXTENT[3]].copy()

def savefig(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(CONFIG["OUTDIR"], f"{name}.pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(CONFIG["OUTDIR"], f"{name}.png"), bbox_inches="tight", dpi=350)
    plt.close(fig)

# Matplotlib defaults (Nature-ish; editable PDF text)
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 350,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


In [ ]:
# ==============
# 3) Load inputs (facilities, BA polygons, BA mix, basins, states)
# ==============

# ---- Facilities ----
if CONFIG["FACILITIES_GEOFILE"]:
    gdf_h = gpd.read_file(CONFIG["FACILITIES_GEOFILE"])
else:
    df = _read_tabular(CONFIG["FACILITIES_TABLE"])
    require_cols(df, [COLS["lat"], COLS["lon"]], "facilities table")
    df[COLS["lat"]] = safe_num(df[COLS["lat"]])
    df[COLS["lon"]] = safe_num(df[COLS["lon"]])
    gdf_h = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df[COLS["lon"]], df[COLS["lat"]]),
        crs="EPSG:4326"
    )
gdf_h = gdf_h[gdf_h["company_type"].str.contains(r"hyperscale|hyperscaler|hyper\s*scale", case=False, na=False)].copy()


# ---- BA polygons ----
ba_polys = gpd.read_file(CONFIG["BA_POLYGONS"])
require_cols(ba_polys, ["region_B_1", "geometry"], "BA polygons")

# ---- BA mix ----
mix_raw = _read_tabular(CONFIG["BA_MIX_TABLE"])

# ---- Basins + Aqueduct (optional) ----
gdf_basins = None
if CONFIG["BASINS_POLYGONS"]:
    gdf_basins = gpd.read_file(CONFIG["BASINS_POLYGONS"])
    require_cols(gdf_basins, ["PFAF_ID", "geometry"], "Basins polygons")

aq = None
if CONFIG["AQUEDUCT_TABLE"]:
    aq = _read_tabular(CONFIG["AQUEDUCT_TABLE"])

# ---- States ----
states48 = gpd.read_file(CONFIG["STATES48"])

print("Loaded:",
      f"facilities={len(gdf_h):,}",
      f"BA_polys={len(ba_polys):,}",
      f"basins={None if gdf_basins is None else len(gdf_basins):,}",
      f"states={len(states48):,}")


In [ ]:
# ==============
# 4) Standardize BA mix from BA_egrid.xlsx (fixes KeyError: missing subregion_id + grid_L_per_kwh)
# ==============

# Your BA_egrid.xlsx uses these names:
# - Balancing Authority Code
# - BA coal generation percent (resource mix), etc.
# We convert to: subregion_id, share_* (0-1), share_fossil, grid_L_per_kwh (derived)

def standardize_ba_mix_from_egrid(mix_raw):
    df = mix_raw.copy()

    # Identify BA code column
    ba_code_col = None
    for cand in ["Balancing Authority Code", "BA Code", "BA code", "Balancing Authority Code "]:
        if cand in df.columns:
            ba_code_col = cand
            break
    if ba_code_col is None:
        raise KeyError("Could not find 'Balancing Authority Code' in BA_egrid.xlsx")

    df["subregion_id"] = df[ba_code_col].astype(str).str.strip().str.upper()

    # Map eGRID percent columns -> our share columns (0-1)
    percent_map = {
        "share_coal": "BA coal generation percent (resource mix)",
        "share_oil": "BA oil generation percent (resource mix)",   # optional (we fold into 'other fossil')
        "share_gas": "BA gas generation percent (resource mix)",
        "share_nuclear": "BA nuclear generation percent (resource mix)",
        "share_hydro": "BA hydro generation percent (resource mix)",
        "share_biomass": "BA biomass generation percent (resource mix)",
        "share_wind": "BA wind generation percent (resource mix)",
        "share_solar": "BA solar generation percent (resource mix)",
        "share_geothermal": "BA geothermal generation percent (resource mix)",
        "share_other_fossil": "BA other fossil generation percent (resource mix)",
        "share_other": "BA other unknown/ purchased fuel generation percent (resource mix)",
    }

    for out_col, src_col in percent_map.items():
        if src_col in df.columns:
            df[out_col] = safe_num(df[src_col]) / 100.0
        else:
            df[out_col] = np.nan

    # Combine into the standard set you used previously
    df["share_other"] = df[["share_other", "share_biomass", "share_geothermal"]].fillna(0).sum(axis=1)
    df["share_coal"] = df["share_coal"].fillna(0)
    df["share_gas"] = df["share_gas"].fillna(0)
    df["share_nuclear"] = df["share_nuclear"].fillna(0)
    df["share_hydro"] = df["share_hydro"].fillna(0)
    df["share_wind"] = df["share_wind"].fillna(0)
    df["share_solar"] = df["share_solar"].fillna(0)

    # treat oil + other_fossil as "other fossil" and include in fossil share
    df["share_other_fossil"] = df[["share_oil", "share_other_fossil"]].fillna(0).sum(axis=1)

    df["sum_shares"] = df[["share_coal","share_gas","share_nuclear","share_hydro","share_wind","share_solar","share_other","share_other_fossil"]].sum(axis=1)

    # Normalize shares if they don't sum to ~1 (minor rounding differences are common)
    with np.errstate(invalid="ignore", divide="ignore"):
        for c in ["share_coal","share_gas","share_nuclear","share_hydro","share_wind","share_solar","share_other","share_other_fossil"]:
            df[c] = np.where(df["sum_shares"]>0, df[c]/df["sum_shares"], np.nan)

    df["share_fossil"] = df["share_coal"].fillna(0) + df["share_gas"].fillna(0) + df["share_other_fossil"].fillna(0)

    # Derive grid water intensity (L/kWh) using literature-style technology factors.
    # These are *defaults* and should be moved to Methods/SI; change as needed.
    # Units: L/kWh (operational consumption)
    WATER_FACTORS_L_PER_KWH = {
        "share_coal": 1.9,
        "share_gas": 0.7,
        "share_nuclear": 2.5,
        "share_hydro": 18.0,   # very uncertain; hydro can dominate; treat carefully
        "share_wind": 0.02,
        "share_solar": 0.02,
        "share_other": 0.5,
        "share_other_fossil": 1.2,
    }
    df["grid_L_per_kwh"] = 0.0
    for share_col, factor in WATER_FACTORS_L_PER_KWH.items():
        df["grid_L_per_kwh"] += df[share_col].fillna(0) * factor

    keep = [
        "subregion_id",
        "share_coal","share_gas","share_nuclear","share_hydro","share_wind","share_solar","share_other","share_other_fossil",
        "share_fossil","grid_L_per_kwh"
    ]
    return df[keep].drop_duplicates(subset=["subregion_id"]).reset_index(drop=True)

mix = standardize_ba_mix_from_egrid(mix_raw)

print("BA mix standardized:", mix.shape)
print(mix.head(3))


In [ ]:
# ==============
# 5) Harmonize CRS + clip to Lower-48
# ==============

gdf_h = bbox_clip_points(to_wgs84(gdf_h))
ba_polys = bbox_clip_poly(to_wgs84(ba_polys))
states48 = bbox_clip_poly(to_wgs84(states48))

# numeric MW (subregion_id may not exist yet!)
require_cols(gdf_h, [COLS["mw"]], "facilities")
gdf_h[COLS["mw"]] = safe_num(gdf_h[COLS["mw"]])
gdf_h = gdf_h.dropna(subset=[COLS["mw"]]).copy()

# ---- Standardize BA polygon key ----
ba_polys = ba_polys.copy()
ba_polys["subregion_id"] = ba_polys["region_B_1"].astype(str).str.strip().str.upper()

# ---- Ensure mix has subregion_id ----
mix = mix.copy()
if "subregion_id" not in mix.columns:
    raise KeyError(
        "mix table is missing 'subregion_id'. "
        "For BA_egrid.xlsx this should have been created from 'Balancing Authority Code'."
    )
mix["subregion_id"] = mix["subregion_id"].astype(str).str.strip().str.upper()

# ---- Assign facility subregion_id (if missing) via spatial join ----
if COLS["subregion"] not in gdf_h.columns:
    # spatial join needs matching CRS
    pts = gdf_h.to_crs("EPSG:4326")
    polys = ba_polys[["subregion_id", "geometry"]].to_crs("EPSG:4326")

    # Use "within" (points within BA polygons). If you have edge cases, switch to "intersects".
    joined = gpd.sjoin(
        pts, polys, how="left", predicate="within"
    ).drop(columns=["index_right"], errors="ignore")

    # write back
    gdf_h = joined
    gdf_h[COLS["subregion"]] = gdf_h["subregion_id"]

# clean subregion id
gdf_h[COLS["subregion"]] = gdf_h[COLS["subregion"]].astype(str).str.strip().str.upper()

# optional: warn if some points did not join to a BA
n_missing = gdf_h[COLS["subregion"]].isna().sum()
if n_missing > 0:
    print(f"⚠️ {n_missing} facilities did not match a BA polygon (subregion_id missing).")

# join mix to BA polygons
ba_polys_m = ba_polys.merge(mix, on="subregion_id", how="left")

# project to LCC for nicer maps
gdf_p = gdf_h.to_crs(LCC)
ba_p = ba_polys_m.to_crs(LCC)
states_p = states48.to_crs(LCC)

# basins + aqueduct
if gdf_basins is not None:
    gdf_basins = to_wgs84(gdf_basins)

    # join aqueduct to basins if available
    if aq is not None:
        pfaf_col = "PFAF_ID" if "PFAF_ID" in aq.columns else ("pfaf_id" if "pfaf_id" in aq.columns else None)
        if pfaf_col is None:
            print("Aqueduct file loaded but missing PFAF_ID/pfaf_id; skipping join.")
        else:
            aq2 = aq.copy()
            aq2[pfaf_col] = aq2[pfaf_col].astype(str).str.strip()
            gdf_basins["PFAF_ID"] = gdf_basins["PFAF_ID"].astype(str).str.strip()
            keep_cols = [pfaf_col] + [c for c in ["bws_01_score","bws_01_cat"] if c in aq2.columns]
            gdf_basins = gdf_basins.merge(
                aq2[keep_cols].drop_duplicates(subset=[pfaf_col]),
                left_on="PFAF_ID", right_on=pfaf_col, how="left"
            ).drop(columns=[pfaf_col], errors="ignore")

    gdf_basins = bbox_clip_poly(gdf_basins)
    basins_p = gdf_basins.to_crs(LCC)
else:
    basins_p = None

print("After clip:", len(gdf_h), "facilities (Lower-48)")


In [ ]:
# ==============
# 6) Attach basin stress to facilities (if basins available)
# ==============

# If facilities already have bws_01_score/pfaf_id, we keep them.
# Otherwise we spatial-join facilities -> basins and transfer bws_01_score and PFAF_ID.

if basins_p is not None:
    if COLS["pfaf"] not in gdf_h.columns or COLS["bws"] not in gdf_h.columns:
        # join in WGS84 for stability
        fac_w = gdf_h.copy()
        bas_w = gdf_basins.copy()
        # Use representative geometry: ensure basins valid
        bas_w["geometry"] = bas_w["geometry"].buffer(0)
        joined = gpd.sjoin(
            fac_w,
            bas_w[[ "PFAF_ID", "geometry"] + ([ "bws_01_score" ] if "bws_01_score" in bas_w.columns else [])],
            how="left",
            predicate="within"
        )
        if COLS["pfaf"] not in fac_w.columns:
            gdf_h[COLS["pfaf"]] = joined["PFAF_ID"]
        if COLS["bws"] not in fac_w.columns and "bws_01_score" in joined.columns:
            gdf_h[COLS["bws"]] = safe_num(joined["bws_01_score"])
        # reproject updated
        gdf_p = gdf_h.to_crs(LCC)

# High-stress flag
if COLS["bws"] in gdf_h.columns:
    gdf_h[COLS["bws"]] = safe_num(gdf_h[COLS["bws"]])
    gdf_h["is_high_stress"] = gdf_h[COLS["bws"]] >= SCEN["HIGH_STRESS_THRESHOLD"]
else:
    gdf_h["is_high_stress"] = False

print("Stress available at facilities:", COLS["bws"] in gdf_h.columns)


In [ ]:
# ==============
# 7) BA summaries from facilities + merge back to BA polygons
# ==============

ba_sites = (
    gdf_h.groupby(COLS["subregion"], as_index=False)
         .agg(
             n_sites=(COLS["mw"], "count"),
             hyperscale_mw=(COLS["mw"], "sum"),
             mean_bws=(COLS["bws"], "mean") if COLS["bws"] in gdf_h.columns else (COLS["mw"], "count"),
             share_high_stress=("is_high_stress", "mean"),
         )
         .rename(columns={COLS["subregion"]: "subregion_id"})
)

ba_p = ba_p.merge(ba_sites, on="subregion_id", how="left")
ba_p["hyperscale_mw"] = safe_num(ba_p["hyperscale_mw"])
ba_p["mean_bws"] = safe_num(ba_p.get("mean_bws", np.nan))
ba_p["grid_L_per_kwh"] = safe_num(ba_p.get("grid_L_per_kwh", np.nan))
ba_p["share_fossil"] = safe_num(ba_p.get("share_fossil", np.nan))

print("BA coverage:",
      f"polygons={len(ba_p):,}",
      f"with MW={ba_p['hyperscale_mw'].notna().sum():,}",
      f"with grid_L_per_kwh={ba_p['grid_L_per_kwh'].notna().sum():,}")


In [ ]:
# ==============
# 8) Plot utilities (consistent legends + smaller dots + decluttering)
# ==============

def draw_base(ax):
    # BA outlines faint
    ba_p.boundary.plot(ax=ax, linewidth=0.25, color="white", alpha=0.9, zorder=1)
    # states
    states_p.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.6, alpha=0.9, zorder=2)
    ax.set_axis_off()

def point_sizes_from_mw(mw_series, smin=6, smax=60):
    """Compact sizes for dense maps; clipped."""
    mw = safe_num(mw_series).fillna(0).clip(lower=0)
    if mw.max() <= 0:
        return np.full(len(mw), smin)
    x = np.sqrt(mw)
    x = (x - x.min()) / (x.max() - x.min() + 1e-12)
    return smin + (smax - smin) * x

PT_SIZE = point_sizes_from_mw(gdf_p[COLS["mw"]], smin=6, smax=55)

def add_size_legend(ax, sizes=(10, 50, 150), loc="lower right", title="Facility MW"):
    # sizes in MW, translated to marker sizes using same scaling function
    svals = point_sizes_from_mw(pd.Series(sizes), smin=6, smax=55)
    handles = [
        Line2D([0],[0], marker='o', color='none', markerfacecolor='black', alpha=0.35,
               markersize=np.sqrt(s), label=f"{mw} MW")
        for mw, s in zip(sizes, svals)
    ]
    leg = ax.legend(handles=handles, title=title, loc=loc, frameon=True, framealpha=0.9, fontsize=8, title_fontsize=9)
    leg.get_frame().set_linewidth(0.5)

def plot_choropleth_with_points(
    poly_gdf, value_col, cmap, title, cbar_label,
    vmin=None, vmax=None, missing_color="#d9d9d9",
    annotate_top=False, top_n=6, fig_size=(12.5, 7.5)
):
    fig, ax = plt.subplots(figsize=fig_size)
    draw_base(ax)

    vals = safe_num(poly_gdf[value_col])
    if vmin is None:
        vmin = np.nanpercentile(vals, 5) if np.isfinite(vals).any() else 0
    if vmax is None:
        vmax = np.nanpercentile(vals, 95) if np.isfinite(vals).any() else 1

    poly_gdf[vals.isna()].plot(ax=ax, color=missing_color, linewidth=0.0, zorder=0)
    poly_gdf[vals.notna()].plot(ax=ax, column=value_col, cmap=cmap, linewidth=0.0, vmin=vmin, vmax=vmax, zorder=0)

    # points (rasterize to reduce heavy PDFs + overlap artifacts)
    pc = gdf_p.plot(ax=ax, markersize=PT_SIZE, color="black", alpha=0.30, linewidth=0, zorder=3)
    try:
        pc.collections[0].set_rasterized(True)
    except Exception:
        pass

    ax.set_title(title)

    # colorbar
    sm = mpl.cm.ScalarMappable(cmap=plt.get_cmap(cmap), norm=mpl.colors.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, fraction=0.030, pad=0.015)
    cb.set_label(cbar_label)

    add_size_legend(ax, sizes=(25, 100, 300), loc="lower right")

    if annotate_top and "hyperscale_mw" in poly_gdf.columns:
        tmp = poly_gdf.dropna(subset=["hyperscale_mw"]).sort_values("hyperscale_mw", ascending=False).head(top_n)
        for _, r in tmp.iterrows():
            p = r.geometry.representative_point()
            ax.text(p.x, p.y, str(r["subregion_id"]), fontsize=8,
                    ha="center", va="center",
                    bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=1.0),
                    zorder=5)

    return fig



In [ ]:
#PT_SIZE = 15
# ==============
# 10) Basin maps (shown explicitly) — Aqueduct baseline stress + facility overlay
# ==============

if basins_p is None:
    print("No basins provided; skip basin maps.")
else:
    # (A) basin stress
    if "bws_01_score" in basins_p.columns and basins_p["bws_01_score"].notna().any():
        fig, ax = plt.subplots(figsize=(13.5, 8.0))
        # outlines + states for context
        states_p.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.6, zorder=2)
        basins_p.plot(ax=ax, column="bws_01_score", cmap="YlOrRd", linewidth=0.05, edgecolor="white", vmin=0, vmax=5, zorder=0)

        pc = gdf_p.plot(ax=ax, markersize=PT_SIZE, color="black", alpha=0.30, linewidth=0, zorder=3)
        try:
            pc.collections[0].set_rasterized(True)
        except Exception:
            pass

        ax.set_axis_off()
        ax.set_title("HydroBASINS level-06: baseline water stress (Aqueduct) + hyperscale sites")
        sm = mpl.cm.ScalarMappable(cmap=plt.get_cmap("YlOrRd"), norm=mpl.colors.Normalize(vmin=0, vmax=5))
        sm.set_array([])
        cb = fig.colorbar(sm, ax=ax, fraction=0.030, pad=0.015)
        cb.set_label("Aqueduct baseline stress (0–5)")
        add_size_legend(ax, sizes=(25, 100, 300), loc="lower right")
        savefig(fig, "basin_map_aqueduct_stress_clean")

    # (B) high-stress-only overlay
    if COLS["bws"] in gdf_h.columns:
        g_hi = gdf_p[gdf_h["is_high_stress"].values].copy()
        fig, ax = plt.subplots(figsize=(13.5, 8.0))
        states_p.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.6, zorder=2)
        basins_p.boundary.plot(ax=ax, linewidth=0.08, color="lightgrey", alpha=0.8, zorder=0)

        # all sites faint
        pc1 = gdf_p.plot(ax=ax, markersize=PT_SIZE, color="black", alpha=0.12, linewidth=0, zorder=1)
        # high stress highlighted
        if len(g_hi):
            PT_HI = point_sizes_from_mw(g_hi[COLS["mw"]], smin=10, smax=80)
            pc2 = g_hi.plot(ax=ax, markersize=PT_HI, color="black", alpha=0.55, linewidth=0, zorder=3)
            try:
                pc2.collections[0].set_rasterized(True)
            except Exception:
                pass

        ax.set_axis_off()
        ax.set_title(f"High-stress sites only (Aqueduct score ≥ {SCEN['HIGH_STRESS_THRESHOLD']})")
        add_size_legend(ax, sizes=(25, 100, 300), loc="lower right")
        savefig(fig, "basin_map_high_stress_sites_only_clean")

    print("✅ Saved basin maps to:", CONFIG["OUTDIR"])


In [ ]:
# ==============
# 11) Paper-friendly tables
# ==============

# BA summary table
ba_table = ba_p[[
    "subregion_id","hyperscale_mw","n_sites","grid_L_per_kwh","share_fossil","mean_bws","share_high_stress"
]].copy()

ba_table.to_csv(os.path.join(CONFIG["OUTDIR"], "table_BA_summary_for_paper_clean.csv"), index=False)

# Company summaries if company_name exists
if COLS["company"] in gdf_h.columns:
    dfc = gdf_h.copy()
    dfc[COLS["company"]] = dfc[COLS["company"]].astype(str)
    comp = (
        dfc.groupby(COLS["company"], as_index=False)
           .agg(
               n_sites=(COLS["mw"], "count"),
               total_mw=(COLS["mw"], "sum"),
               share_high_stress=("is_high_stress", "mean"),
               mw_high_stress=(COLS["mw"], lambda s: s[dfc.loc[s.index, "is_high_stress"]].sum())
           )
           .sort_values("total_mw", ascending=False)
    )
    comp.to_csv(os.path.join(CONFIG["OUTDIR"], "table_company_exposure_all_clean.csv"), index=False)

print("✅ Wrote tables to:", CONFIG["OUTDIR"])


In [ ]:
# ==============
# NEW) Basin-level aggregation (scarcity × hyperscale intensity)
# ==============

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl

def basin_area_km2(gdf):
    # expects projected CRS in meters
    return gdf.geometry.area / 1e6

def ensure_projected(gdf, name="gdf"):
    if gdf.crs is None:
        raise ValueError(f"{name} has no CRS.")
    # if geographic (degrees), warn
    if gdf.crs.is_geographic:
        raise ValueError(f"{name} is geographic CRS; project it before area/density calcs.")

# Requires basins_p (projected) and gdf_p (projected)
if basins_p is None:
    raise ValueError("basins_p is None. Set CONFIG['BASINS_POLYGONS'] and re-run.")

ensure_projected(basins_p, "basins_p")
ensure_projected(gdf_p, "gdf_p")

# join facilities -> basins
# basins must have PFAF_ID
if "PFAF_ID" not in basins_p.columns:
    raise KeyError("basins_p must contain 'PFAF_ID'.")

pts = gdf_p.copy()
bas = basins_p[["PFAF_ID", "geometry"] + [c for c in ["bws_01_score", "bws_01_cat"] if c in basins_p.columns]].copy()

# If facility already has pfaf_id we could skip this, but this ensures correctness
joined = gpd.sjoin(
    pts,
    bas.rename(columns={"PFAF_ID":"_PFAF_ID"}),
    how="left",
    predicate="within"
).drop(columns=["index_right"], errors="ignore")

# write basin id to facility
joined[COLS["pfaf"]] = joined["_PFAF_ID"]
joined = joined.drop(columns=["_PFAF_ID"], errors="ignore")

# compute basin stats
mw_col = COLS["mw"]
tot_m3_col = COLS["tot_m3"] if COLS["tot_m3"] in joined.columns else None

# Use facility stress if present, else later we can rely on basin stress
bws_fac_col = COLS["bws"] if COLS["bws"] in joined.columns else None

agg = {
    "n_sites": (mw_col, "count"),
    "hyperscale_mw": (mw_col, "sum"),
}
if tot_m3_col:
    agg["total_m3_mid_sum"] = (tot_m3_col, "sum")
if bws_fac_col:
    agg["mean_bws_site"] = (bws_fac_col, "mean")

basin_tab = (
    joined.dropna(subset=[COLS["pfaf"]])
          .groupby(COLS["pfaf"], as_index=False)
          .agg(**{k: v for k, v in agg.items()})
)

# attach to basin polygons
basins_m = basins_p.copy()
basins_m["PFAF_ID"] = basins_m["PFAF_ID"].astype(str).str.strip()
basin_tab[COLS["pfaf"]] = basin_tab[COLS["pfaf"]].astype(str).str.strip()

basins_m = basins_m.merge(basin_tab, left_on="PFAF_ID", right_on=COLS["pfaf"], how="left")

# basin area and densities
basins_m["area_km2"] = basin_area_km2(basins_m)
basins_m["mw_per_1000km2"] = basins_m["hyperscale_mw"] / (basins_m["area_km2"] / 1000.0)

# define basin scarcity score (prefer Aqueduct basin score if present)
if "bws_01_score" not in basins_m.columns:
    # fallback: if you only have facility scores, use mean site stress
    if "mean_bws_site" in basins_m.columns:
        basins_m["bws_01_score"] = basins_m["mean_bws_site"]
    else:
        basins_m["bws_01_score"] = np.nan

print("✅ Basin aggregation done.",
      "Basins with sites:", basins_m["hyperscale_mw"].notna().sum(),
      "Total sites:", len(joined))


In [ ]:
# ==============
# NEW) Basin-level aggregation (scarcity × hyperscale intensity)
# ==============

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl

def basin_area_km2(gdf):
    # expects projected CRS in meters
    return gdf.geometry.area / 1e6

def ensure_projected(gdf, name="gdf"):
    if gdf.crs is None:
        raise ValueError(f"{name} has no CRS.")
    # if geographic (degrees), warn
    if gdf.crs.is_geographic:
        raise ValueError(f"{name} is geographic CRS; project it before area/density calcs.")

# Requires basins_p (projected) and gdf_p (projected)
if basins_p is None:
    raise ValueError("basins_p is None. Set CONFIG['BASINS_POLYGONS'] and re-run.")

ensure_projected(basins_p, "basins_p")
ensure_projected(gdf_p, "gdf_p")

# join facilities -> basins
# basins must have PFAF_ID
if "PFAF_ID" not in basins_p.columns:
    raise KeyError("basins_p must contain 'PFAF_ID'.")

pts = gdf_p.copy()
bas = basins_p[["PFAF_ID", "geometry"] + [c for c in ["bws_01_score", "bws_01_cat"] if c in basins_p.columns]].copy()

# If facility already has pfaf_id we could skip this, but this ensures correctness
joined = gpd.sjoin(
    pts,
    bas.rename(columns={"PFAF_ID":"_PFAF_ID"}),
    how="left",
    predicate="within"
).drop(columns=["index_right"], errors="ignore")

# write basin id to facility
joined[COLS["pfaf"]] = joined["_PFAF_ID"]
joined = joined.drop(columns=["_PFAF_ID"], errors="ignore")

# compute basin stats
mw_col = COLS["mw"]
tot_m3_col = COLS["tot_m3"] if COLS["tot_m3"] in joined.columns else None

# Use facility stress if present, else later we can rely on basin stress
bws_fac_col = COLS["bws"] if COLS["bws"] in joined.columns else None

agg = {
    "n_sites": (mw_col, "count"),
    "hyperscale_mw": (mw_col, "sum"),
}
if tot_m3_col:
    agg["total_m3_mid_sum"] = (tot_m3_col, "sum")
if bws_fac_col:
    agg["mean_bws_site"] = (bws_fac_col, "mean")

basin_tab = (
    joined.dropna(subset=[COLS["pfaf"]])
          .groupby(COLS["pfaf"], as_index=False)
          .agg(**{k: v for k, v in agg.items()})
)

# attach to basin polygons
basins_m = basins_p.copy()
basins_m["PFAF_ID"] = basins_m["PFAF_ID"].astype(str).str.strip()
basin_tab[COLS["pfaf"]] = basin_tab[COLS["pfaf"]].astype(str).str.strip()

basins_m = basins_m.merge(basin_tab, left_on="PFAF_ID", right_on=COLS["pfaf"], how="left")

# basin area and densities
basins_m["area_km2"] = basin_area_km2(basins_m)
basins_m["mw_per_1000km2"] = basins_m["hyperscale_mw"] / (basins_m["area_km2"] / 1000.0)

# define basin scarcity score (prefer Aqueduct basin score if present)
if "bws_01_score" not in basins_m.columns:
    # fallback: if you only have facility scores, use mean site stress
    if "mean_bws_site" in basins_m.columns:
        basins_m["bws_01_score"] = basins_m["mean_bws_site"]
    else:
        basins_m["bws_01_score"] = np.nan

print("✅ Basin aggregation done.",
      "Basins with sites:", basins_m["hyperscale_mw"].notna().sum(),
      "Total sites:", len(joined))


In [ ]:
# ==============
# 9) BA maps (bigger, cleaner) — fossil share, grid water intensity, hyperscale MW, mean stress
# ==============

# (1) fossil share
fig = plot_choropleth_with_points(
    ba_p, "share_fossil", cmap="YlOrBr",
    title="Balancing authorities: fossil generation share (overlay: hyperscale sites, size ∝ MW)",
    cbar_label="share (0–1)", vmin=0, vmax=1,
    annotate_top=True, top_n=6, fig_size=(13.5, 8.0)
)
savefig(fig, "map1_fossil_share_by_BA_clean")

# (2) grid water intensity
fig = plot_choropleth_with_points(
    ba_p, "grid_L_per_kwh", cmap="viridis",
    title="Balancing authorities: grid water intensity (L/kWh) (overlay: hyperscale sites)",
    cbar_label="L/kWh", annotate_top=True, top_n=6, fig_size=(13.5, 8.0)
)
savefig(fig, "map2_grid_water_intensity_by_BA_clean")

# (3) mean site stress by BA (if available)
if "mean_bws" in ba_p.columns and ba_p["mean_bws"].notna().any():
    fig = plot_choropleth_with_points(
        ba_p, "mean_bws", cmap="YlOrRd",
        title="Balancing authorities: mean site water-stress score (overlay: hyperscale sites)",
        cbar_label="stress score", vmin=0, vmax=5, annotate_top=False, fig_size=(13.5, 8.0)
    )
    savefig(fig, "map3_mean_stress_by_BA_clean")

# (4) hyperscale MW concentration by BA
fig = plot_choropleth_with_points(
    ba_p, "hyperscale_mw", cmap="plasma", 
    title="Balancing authorities: hyperscale capacity concentration (MW) (overlay: hyperscale sites)",
    cbar_label="MW", annotate_top=True, top_n=8, fig_size=(13.5, 8.0)
)
savefig(fig, "map4_hyperscale_capacity_by_BA_clean")

print("✅ Saved BA maps to:", CONFIG["OUTDIR"])


In [ ]:
# ==============
# NEW FIGURE 2) BA grid water intensity + basin scarcity outlines
# ==============

# Choose a scarcity threshold for outlines
thr = SCEN["HIGH_STRESS_THRESHOLD"]

fig, ax = plt.subplots(figsize=(13.5, 8.5))
ax.set_axis_off()

# BA water intensity background
col = COLS["grid_L_per_kwh"]
vals = pd.to_numeric(ba_p[col], errors="coerce")
ba_p[vals.isna()].plot(ax=ax, color="#d9d9d9", linewidth=0.0)
ba_p[vals.notna()].plot(ax=ax, column=col, cmap="viridis", linewidth=0.0)

# Basin scarcity contours
bws = pd.to_numeric(basins_m["bws_01_score"], errors="coerce")
basins_hi = basins_m[bws >= thr]
basins_hi.boundary.plot(ax=ax, color="black", linewidth=0.35, alpha=0.9)

# State borders
states_p.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=0.55, alpha=0.8)

# points
gdf_p.plot(ax=ax, markersize=7, color="black", alpha=0.20)

# colorbar
norm = mpl.colors.Normalize(vmin=np.nanpercentile(vals, 5), vmax=np.nanpercentile(vals, 95))
sm = mpl.cm.ScalarMappable(cmap=plt.get_cmap("viridis"), norm=norm); sm.set_array([])
cb = fig.colorbar(sm, ax=ax, fraction=0.028, pad=0.01)
cb.set_label("Grid water intensity (L/kWh)")

ax.set_title(f"Grid water intensity by BA with high-scarcity basin outlines (BWS ≥ {thr})", fontsize=12)
savefig(fig, "Figure_BA_grid_water_intensity_with_high_scarcity_basin_outlines")


In [ ]:
# ==============
# NEW FIGURE 3) Facility-weighted: basin scarcity vs BA grid water intensity
# ==============

# Build a facility table with both basin stress and BA grid water intensity
fac = joined.copy()

# attach BA grid intensity by subregion
# ba_p is polygon; easier: use mix table or ba_p attributes via subregion join
if "subregion_id" not in fac.columns:
    fac["subregion_id"] = fac[COLS["subregion"]]

mix_cols_needed = ["subregion_id", COLS["grid_L_per_kwh"]]
mix2 = mix.copy()
for c in mix_cols_needed:
    if c not in mix2.columns:
        raise KeyError(f"mix missing required col {c}")
mix2["subregion_id"] = mix2["subregion_id"].astype(str).str.strip().str.upper()

fac["subregion_id"] = fac["subregion_id"].astype(str).str.strip().str.upper()
fac = fac.merge(mix2[mix_cols_needed].drop_duplicates("subregion_id"), on="subregion_id", how="left")

# basin stress from basins_m (Aqueduct)
bas_stress = basins_m[["PFAF_ID", "bws_01_score"]].copy()
bas_stress["PFAF_ID"] = bas_stress["PFAF_ID"].astype(str).str.strip()
fac[COLS["pfaf"]] = fac[COLS["pfaf"]].astype(str).str.strip()
fac = fac.merge(bas_stress, left_on=COLS["pfaf"], right_on="PFAF_ID", how="left").drop(columns=["PFAF_ID"], errors="ignore")

# x/y with weights
x = pd.to_numeric(fac["bws_01_score"], errors="coerce")
y = pd.to_numeric(fac[COLS["grid_L_per_kwh"]], errors="coerce")
w = pd.to_numeric(fac[COLS["mw"]], errors="coerce").fillna(0)

mask = x.notna() & y.notna() & (w > 0)
x = x[mask]; y = y[mask]; w = w[mask]

fig, ax = plt.subplots(figsize=(8.5, 6.5))
hb = ax.hexbin(x, y, C=w, reduce_C_function=np.sum, gridsize=28, mincnt=1)
ax.set_xlabel("Basin baseline water stress (Aqueduct BWS)")
ax.set_ylabel("BA grid water intensity (L/kWh)")
ax.set_title("Where hyperscale load sits: scarcity vs grid-water intensity\n(hex color = total MW in bin)")
cb = fig.colorbar(hb, ax=ax, fraction=0.035, pad=0.02)
cb.set_label("Total hyperscale MW")
savefig(fig, "Figure_hexbin_scarcity_vs_grid_water_intensity_weighted_by_MW")


In [ ]:
# ==============
# NEW FIGURE 3) Facility-weighted: basin scarcity vs BA grid water intensity
# ==============

# Build a facility table with both basin stress and BA grid water intensity
fac = joined.copy()

# attach BA grid intensity by subregion
# ba_p is polygon; easier: use mix table or ba_p attributes via subregion join
if "subregion_id" not in fac.columns:
    fac["subregion_id"] = fac[COLS["subregion"]]

mix_cols_needed = ["subregion_id", COLS["grid_L_per_kwh"]]
mix2 = mix.copy()
for c in mix_cols_needed:
    if c not in mix2.columns:
        raise KeyError(f"mix missing required col {c}")
mix2["subregion_id"] = mix2["subregion_id"].astype(str).str.strip().str.upper()

fac["subregion_id"] = fac["subregion_id"].astype(str).str.strip().str.upper()
fac = fac.merge(mix2[mix_cols_needed].drop_duplicates("subregion_id"), on="subregion_id", how="left")

# basin stress from basins_m (Aqueduct)
bas_stress = basins_m[["PFAF_ID", "bws_01_score"]].copy()
bas_stress["PFAF_ID"] = bas_stress["PFAF_ID"].astype(str).str.strip()
fac[COLS["pfaf"]] = fac[COLS["pfaf"]].astype(str).str.strip()
fac = fac.merge(bas_stress, left_on=COLS["pfaf"], right_on="PFAF_ID", how="left").drop(columns=["PFAF_ID"], errors="ignore")

# x/y with weights
x = pd.to_numeric(fac["bws_01_score"], errors="coerce")
y = pd.to_numeric(fac[COLS["grid_L_per_kwh"]], errors="coerce")
w = pd.to_numeric(fac[COLS["mw"]], errors="coerce").fillna(0)

mask = x.notna() & y.notna() & (w > 0)
x = x[mask]; y = y[mask]; w = w[mask]

fig, ax = plt.subplots(figsize=(8.5, 6.5))
hb = ax.hexbin(x, y, C=w, reduce_C_function=np.sum, gridsize=28, mincnt=1)
ax.set_xlabel("Basin baseline water stress (Aqueduct BWS)")
ax.set_ylabel("BA grid water intensity (L/kWh)")
ax.set_title("Where hyperscale load sits: scarcity vs grid-water intensity\n(hex color = total MW in bin)")
cb = fig.colorbar(hb, ax=ax, fraction=0.035, pad=0.02)
cb.set_label("Total hyperscale MW")
savefig(fig, "Figure_hexbin_scarcity_vs_grid_water_intensity_weighted_by_MW")


In [ ]:
# ==============
# NEW FIGURE 4) Fuel mix × scarcity exposure (facility-weighted)
# ==============

# Ensure these fuel share columns exist in mix (or rename if your mix uses the eGRID percent columns)
fuel_cols = ["share_coal","share_gas","share_nuclear","share_hydro","share_wind","share_solar","share_other"]
missing_fuel = [c for c in fuel_cols if c not in mix.columns]
if missing_fuel:
    print("⚠️ Missing fuel share columns:", missing_fuel)
    print("If your BA_egrid.xlsx only has '... generation percent (resource mix)' columns, add a mapping step earlier.")

# Compute fossil share if available
if "share_fossil" not in mix.columns and ("share_coal" in mix.columns and "share_gas" in mix.columns):
    mix["share_fossil"] = pd.to_numeric(mix["share_coal"], errors="coerce").fillna(0) + pd.to_numeric(mix["share_gas"], errors="coerce").fillna(0)

# Facility table already has: subregion_id, MW, basin bws_01_score
fac2 = fac.copy()
fac2["mw"] = pd.to_numeric(fac2[COLS["mw"]], errors="coerce")
fac2["bws_01_score"] = pd.to_numeric(fac2["bws_01_score"], errors="coerce")

# merge BA fuel/fossil shares onto each facility via subregion_id
keep = ["subregion_id"] + [c for c in (fuel_cols + ["share_fossil"]) if c in mix.columns]
mix_small = mix[keep].drop_duplicates("subregion_id").copy()
fac2 = fac2.merge(mix_small, on="subregion_id", how="left")

# define scarcity class bins (0–5 score)
bins = [-0.01, 1, 2, 3, 4, 5.01]
labels = ["Low (≤1)", "Low–Med (1–2)", "Med (2–3)", "High (3–4)", "Ext High (4–5)"]
fac2["scarcity_class"] = pd.cut(fac2["bws_01_score"], bins=bins, labels=labels)

# plot: distribution of fossil share by scarcity class (weighted by MW)
dfp = fac2.dropna(subset=["scarcity_class", "share_fossil", "mw"]).copy()

# weighted mean fossil share per scarcity class + weighted SE (optional)
grp = dfp.groupby("scarcity_class").apply(
    lambda d: pd.Series({
        "mw_sum": d["mw"].sum(),
        "fossil_mean_w": np.average(d["share_fossil"], weights=d["mw"]),
    })
).reset_index()

fig, ax = plt.subplots(figsize=(9, 5.8))
ax.bar(grp["scarcity_class"].astype(str), grp["fossil_mean_w"])
ax.set_ylim(0, 1)
ax.set_ylabel("Facility-weighted BA fossil generation share")
ax.set_xlabel("Basin scarcity class (Aqueduct BWS)")
ax.set_title("Are stressed basins served by more fossil-heavy grids?\n(bar = MW-weighted mean fossil share of serving BA)")
for i, (mw_sum, v) in enumerate(zip(grp["mw_sum"], grp["fossil_mean_w"])):
    ax.text(i, v + 0.02, f"{mw_sum:.0f} MW", ha="center", va="bottom", fontsize=9)
savefig(fig, "Figure_fossil_share_by_scarcity_class_weighted")


In [ ]:
# =========================
# FIGURE v3 (Nature-ready): BIG 2x3 BA panels, overlap-controlled points
# Requires: gdf_p (points), ba_p (BA polygons w/ attrs), states_p (state lines)
# Columns expected on ba_p: mean_bws, hyperscale_mw, share_fossil, grid_L_per_kwh, share_high_stress
# Column expected on gdf_p: current_mw, bws_01_score (optional)
# =========================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# ---------- style ----------
plt.rcParams.update({
    "figure.dpi": 180,
    "savefig.dpi": 450,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})
def safe_num(x):
    """
    Convert scalars/Series/arrays to numeric safely.
    Returns:
      - pandas Series if input is Series/Index/array-like
      - float (or np.nan) if input is scalar
    """
    # pandas Series / Index
    if isinstance(x, (pd.Series, pd.Index)):
        out = pd.to_numeric(x, errors="coerce")
        return out.replace([np.inf, -np.inf], np.nan)

    # array-like (list, np.ndarray)
    if isinstance(x, (list, tuple, np.ndarray)):
        out = pd.to_numeric(pd.Series(x), errors="coerce")
        return out.replace([np.inf, -np.inf], np.nan)

    # scalar
    out = pd.to_numeric(x, errors="coerce")
    if np.isinf(out):
        return np.nan
    return out


def draw_base(ax):
    # BA boundary thin white, states thin red (keep your convention)
    ba_p.boundary.plot(ax=ax, linewidth=0.25, color="white", alpha=0.9, zorder=2)
    states_p.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=0.55, alpha=0.85, zorder=5)
    ax.set_axis_off()

# ---------- point sizing + overlap control ----------
mw = safe_num(gdf_p["current_mw"]).clip(lower=0)
# More conservative scaling than before -> fewer giant circles
s = 8 + 140 * (np.sqrt(mw) - np.sqrt(mw).min()) / (np.sqrt(mw).max() - np.sqrt(mw).min() + 1e-12)

def plot_sites(ax, df, sizes, top_only=False, top_q=0.90):
    """Two-layer points: outline + fill. Optionally plot only top quantile to reduce clutter."""
    d = df.copy()
    ss = sizes.copy()

    if top_only and len(d):
        thr = np.nanquantile(mw, top_q)
        keep = safe_num(d["current_mw"]) >= thr
        d = d.loc[keep]
        ss = ss.loc[keep]

    # outline (helps separation on busy maps)
    d.plot(ax=ax, markersize=ss*1.15, color="white", alpha=0.60, zorder=8)
    # fill
    d.plot(ax=ax, markersize=ss, color="black", alpha=0.28, zorder=9)

def add_stress_hist(ax, gdf, title="Site water stress"):
    iax = inset_axes(ax, width="28%", height="26%", loc="upper right", borderpad=1.1)
    vals = safe_num(gdf.get("bws_01_score", np.nan)).dropna()
    if len(vals) == 0:
        iax.text(0.5, 0.5, "No stress data", ha="center", va="center")
        iax.set_xticks([]); iax.set_yticks([])
        return
    iax.hist(vals, bins=np.linspace(0, 5, 11), edgecolor="white", linewidth=0.5)
    iax.axvline(3.0, linestyle="--", linewidth=1.0)
    iax.set_title(title, fontsize=8)
    iax.set_xlabel("bws score", fontsize=7)
    iax.set_ylabel("n", fontsize=7)
    iax.tick_params(axis="both", labelsize=7)

# ---------- bivariate palette ----------
BIVAR_3x3 = np.array([
    ["#e8e8e8", "#b5c0da", "#6c83b5"],
    ["#b8d6be", "#90b2b3", "#567994"],
    ["#73ae80", "#5a9178", "#2a5a5b"],
])

def quantile_bins(series, n=3):
    s = safe_num(series).dropna()
    if len(s) == 0:
        return None
    qs = np.quantile(s, np.linspace(0, 1, n+1))
    for i in range(1, len(qs)):
        if qs[i] <= qs[i-1]:
            qs[i] = qs[i-1] + 1e-12
    return qs

def assign_bin(val, edges):
    import pandas as pd
    if pd.isna(val):
        return np.nan
    for i in range(len(edges)-1):
        lo, hi = edges[i], edges[i+1]
        if i == len(edges)-2:
            if val >= lo and val <= hi:
                return i
        else:
            if val >= lo and val < hi:
                return i
    return len(edges)-2

def make_bivar(df, xcol, ycol, n=3):
    xedges = quantile_bins(df[xcol], n=n)
    yedges = quantile_bins(df[ycol], n=n)
    out = df.copy()
    if xedges is None or yedges is None:
        out["_bivar_color"] = "#d9d9d9"
        return out

    xb = out[xcol].apply(lambda v: assign_bin(v, xedges))
    yb = out[ycol].apply(lambda v: assign_bin(v, yedges))

    colors = []
    for i, j in zip(xb, yb):
        if np.isnan(i) or np.isnan(j):
            colors.append("#d9d9d9")
        else:
            colors.append(BIVAR_3x3[int(i), int(j)])
    out["_bivar_color"] = colors
    return out

def bivar_legend(ax, xlab, ylab, loc=(0.02, 0.06), size=0.18):
    x0, y0 = loc
    leg = inset_axes(ax, width=f"{int(size*100)}%", height=f"{int(size*100)}%", loc="lower left",
                     bbox_to_anchor=(x0, y0, 1, 1), bbox_transform=ax.transAxes, borderpad=0)
    leg.set_xticks([]); leg.set_yticks([])
    for i in range(3):
        for j in range(3):
            leg.add_patch(Rectangle((j, i), 1, 1, facecolor=BIVAR_3x3[i, j], edgecolor="white", lw=0.6))
    leg.set_xlim(0, 3); leg.set_ylim(0, 3)
    leg.text(1.5, -0.55, ylab, ha="center", va="top", fontsize=8)
    leg.text(-0.55, 1.5, xlab, ha="right", va="center", rotation=90, fontsize=8)

# ---------- build figure ----------
fig = plt.figure(figsize=(19, 10.5), constrained_layout=True)
gs = GridSpec(2, 3, figure=fig, wspace=0.02, hspace=0.05)

axes = [fig.add_subplot(gs[i, j]) for i in range(2) for j in range(3)]

# (a) bivariate: mean_bws x hyperscale_mw
ax = axes[0]
draw_base(ax)
b1 = make_bivar(ba_p, "mean_bws", "hyperscale_mw")
b1.plot(ax=ax, color=b1["_bivar_color"], linewidth=0, zorder=1)
plot_sites(ax, gdf_p, s, top_only=False)
ax.set_title("(a) Water stress × hyperscale capacity (BA-level)")
bivar_legend(ax, "Higher stress", "Higher capacity")
add_stress_hist(ax, gdf_p, "Site water stress")

# (b) bivariate: mean_bws x share_fossil
ax = axes[1]
draw_base(ax)
b2 = make_bivar(ba_p, "mean_bws", "share_fossil")
b2.plot(ax=ax, color=b2["_bivar_color"], linewidth=0, zorder=1)
plot_sites(ax, gdf_p, s, top_only=False)
ax.set_title("(b) Water stress × fossil share (BA-level)")
bivar_legend(ax, "Higher stress", "Higher fossil share")
add_stress_hist(ax, gdf_p, "Site water stress")

# (c) grid water intensity
ax = axes[2]
draw_base(ax)
col = "grid_L_per_kwh"
vals = safe_num(ba_p[col])
ba_p[vals.isna()].plot(ax=ax, color="#d9d9d9", linewidth=0, zorder=1)
m = ba_p[vals.notna()]
vmin, vmax = np.nanpercentile(vals, 5), np.nanpercentile(vals, 95)
m.plot(ax=ax, column=col, cmap="viridis", linewidth=0, vmin=vmin, vmax=vmax, zorder=1)
plot_sites(ax, gdf_p, s, top_only=False)
ax.set_title("(c) Grid water intensity (L/kWh) + sites")
sm = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=vmin, vmax=vmax), cmap="viridis"); sm.set_array([])
cb = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.01)
cb.set_label("L/kWh")

# (d) mean stress (BA)
ax = axes[3]
draw_base(ax)
col = "mean_bws"
vals = safe_num(ba_p[col])
ba_p[vals.isna()].plot(ax=ax, color="#d9d9d9", linewidth=0, zorder=1)
ba_p[vals.notna()].plot(ax=ax, column=col, cmap="YlOrRd", linewidth=0, vmin=0, vmax=5, zorder=1)
ax.set_title("(d) Mean site water stress by BA")
sm = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=0, vmax=5), cmap="YlOrRd"); sm.set_array([])
cb = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.01)
cb.set_label("stress (0–5)")

# (e) hyperscale MW by BA
ax = axes[4]
draw_base(ax)
col = "hyperscale_mw"
vals = safe_num(ba_p[col])
ba_p[vals.isna()].plot(ax=ax, color="#d9d9d9", linewidth=0, zorder=1)
m = ba_p[vals.notna()]
vmin, vmax = np.nanpercentile(vals, 5), np.nanpercentile(vals, 95)
m.plot(ax=ax, column=col, cmap="plasma", linewidth=0, vmin=vmin, vmax=vmax, zorder=1)
ax.set_title("(e) Hyperscale capacity (sum MW) by BA")
sm = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=vmin, vmax=vmax), cmap="plasma"); sm.set_array([])
cb = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.01)
cb.set_label("MW")

# (f) share high-stress sites by BA
ax = axes[5]
draw_base(ax)
col = "share_high_stress"
vals = safe_num(ba_p[col])
ba_p[vals.isna()].plot(ax=ax, color="#d9d9d9", linewidth=0, zorder=1)
ba_p[vals.notna()].plot(ax=ax, column=col, cmap="Reds", linewidth=0, vmin=0, vmax=1, zorder=1)
ax.set_title("(f) Share of sites in high-stress (≥3) by BA")
sm = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=0, vmax=1), cmap="Reds"); sm.set_array([])
cb = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.01)
cb.set_label("share (0–1)")

# Save
outpath_pdf = os.path.join(CONFIG["OUTDIR"], "Figure_BA_a_to_f_v3_big.pdf")
outpath_png = os.path.join(CONFIG["OUTDIR"], "Figure_BA_a_to_f_v3_big.png")
fig.savefig(outpath_pdf, bbox_inches="tight")
fig.savefig(outpath_png, bbox_inches="tight")
plt.close(fig)

print("✅ Exported:", outpath_pdf, "and", outpath_png)


In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# ============================================================
# Plot styling (Nature-ish defaults; Matplotlib only)
# ============================================================
plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 450,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

OUTDIR = CONFIG.get("OUTDIR", "outputs")
os.makedirs(OUTDIR, exist_ok=True)

# ============================================================
# Utilities
# ============================================================
def savefig(fig, name, tight=True):
    if tight:
        fig.tight_layout()
    fig.savefig(os.path.join(OUTDIR, f"{name}.pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(OUTDIR, f"{name}.png"), bbox_inches="tight")
    plt.close(fig)

def safe_num(x):
    """
    Convert scalars/Series/arrays to numeric safely.
    - If Series/Index -> returns Series
    - If array-like -> returns Series
    - If scalar -> returns float (or np.nan)
    """
    if isinstance(x, (pd.Series, pd.Index)):
        out = pd.to_numeric(x, errors="coerce")
        return out.replace([np.inf, -np.inf], np.nan)

    if isinstance(x, (list, tuple, np.ndarray)):
        out = pd.to_numeric(pd.Series(x), errors="coerce")
        return out.replace([np.inf, -np.inf], np.nan)

    out = pd.to_numeric(x, errors="coerce")
    if np.isinf(out):
        return np.nan
    return out

def require_cols(df, cols, name="df"):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"{name} missing required columns: {missing}")

def scaled_point_sizes(mw_series, smin=10, smax=240):
    mw = safe_num(mw_series)
    if not isinstance(mw, pd.Series):
        mw = pd.Series(mw)
    mw = mw.fillna(0).clip(lower=0)

    if len(mw) == 0:
        return np.array([])
    if mw.max() <= 0:
        return np.full(len(mw), smin)

    x = np.sqrt(mw)
    x = (x - x.min()) / (x.max() - x.min() + 1e-12)
    return smin + (smax - smin) * x

def normalize_subregion(s):
    return s.astype(str).str.strip().str.upper()

# ============================================================
# Base map layers
# ============================================================
def draw_base(ax, states_layer=None, boundary_layer=None):
    if boundary_layer is not None:
        boundary_layer.boundary.plot(ax=ax, linewidth=0.2, color="white", alpha=0.7)
    if states_layer is not None:
        states_layer.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=0.6, alpha=0.8)
    ax.set_axis_off()

# ============================================================
# Bivariate palette + legend
# ============================================================
BIVAR_3x3 = np.array([
    ["#e8e8e8", "#b5c0da", "#6c83b5"],
    ["#b8d6be", "#90b2b3", "#567994"],
    ["#73ae80", "#5a9178", "#2a5a5b"],
])

def quantile_bins(series, n=3):
    s = safe_num(series)
    if not isinstance(s, pd.Series):
        s = pd.Series(s)
    s = s.dropna()
    if len(s) == 0:
        return None
    qs = np.quantile(s, np.linspace(0, 1, n+1))
    for i in range(1, len(qs)):
        if qs[i] <= qs[i-1]:
            qs[i] = qs[i-1] + 1e-12
    return qs

def assign_bin(val, edges):
    if pd.isna(val):
        return np.nan
    for i in range(len(edges) - 1):
        lo, hi = edges[i], edges[i+1]
        if i == len(edges) - 2:
            if (val >= lo) and (val <= hi):
                return i
        else:
            if (val >= lo) and (val < hi):
                return i
    return len(edges) - 2

def make_bivariate_colors(gdf, xcol, ycol, n=3, missing="#d9d9d9"):
    out = gdf.copy()
    x_edges = quantile_bins(out[xcol], n=n)
    y_edges = quantile_bins(out[ycol], n=n)

    if x_edges is None or y_edges is None:
        out["_bivar_color"] = missing
        return out, x_edges, y_edges

    xb = out[xcol].apply(lambda v: assign_bin(v, x_edges))
    yb = out[ycol].apply(lambda v: assign_bin(v, y_edges))

    colors = []
    for i, j in zip(xb, yb):
        if pd.isna(i) or pd.isna(j):
            colors.append(missing)
        else:
            colors.append(BIVAR_3x3[int(i), int(j)])
    out["_bivar_color"] = colors
    return out, x_edges, y_edges

def draw_bivariate_legend(ax, xlab, ylab, loc=(0.02, 0.06), size=0.18):
    x0, y0 = loc
    leg = inset_axes(
        ax,
        width=f"{int(size*100)}%", height=f"{int(size*100)}%",
        loc="lower left",
        bbox_to_anchor=(x0, y0, 1, 1),
        bbox_transform=ax.transAxes,
        borderpad=0,
    )
    leg.set_xticks([]); leg.set_yticks([])
    leg.set_frame_on(True)
    for i in range(3):
        for j in range(3):
            leg.add_patch(Rectangle((j, i), 1, 1,
                                   facecolor=BIVAR_3x3[i, j],
                                   edgecolor="white", lw=0.7))
    leg.set_xlim(0, 3); leg.set_ylim(0, 3)
    leg.text(1.5, -0.55, ylab, ha="center", va="top", fontsize=8)
    leg.text(-0.55, 1.5, xlab, ha="right", va="center", rotation=90, fontsize=8)
    leg.annotate("", xy=(3.05, 0.15), xytext=(0.05, 0.15),
                 arrowprops=dict(arrowstyle="-|>", lw=0.9))
    leg.annotate("", xy=(0.15, 3.05), xytext=(0.15, 0.05),
                 arrowprops=dict(arrowstyle="-|>", lw=0.9))

def add_stress_hist_inset(ax, gdf_sites, bws_col, title="Site stress"):
    iax = inset_axes(ax, width="27%", height="27%", loc="upper right", borderpad=1.0)
    if bws_col not in gdf_sites.columns:
        iax.text(0.5, 0.5, "No stress column", ha="center", va="center")
        iax.set_xticks([]); iax.set_yticks([])
        return

    vals = safe_num(gdf_sites[bws_col]).dropna()
    if len(vals) == 0:
        iax.text(0.5, 0.5, "No stress data", ha="center", va="center")
        iax.set_xticks([]); iax.set_yticks([])
        return

    iax.hist(vals, bins=np.linspace(0, 5, 11), edgecolor="white", linewidth=0.6)
    iax.axvline(SCEN["HIGH_STRESS_THRESHOLD"], linestyle="--", linewidth=1.0)
    iax.set_title(title, fontsize=8)
    iax.set_xlabel("score", fontsize=7)
    iax.set_ylabel("n", fontsize=7)
    iax.tick_params(axis="both", labelsize=7)

# ============================================================
# 0) Ensure grid_L_per_kwh exists (derive from BA_egrid.xlsx if needed)
# ============================================================
def ensure_grid_water_intensity(mix_df):
    """
    Ensures mix_df has:
      - subregion_id
      - grid_L_per_kwh (computed if missing)
      - share_* columns (0..1) where derivable from BA_egrid percent columns

    NOTE: factors are placeholders; keep these transparent and test sensitivity (ED7).
    """
    df = mix_df.copy()

    # key
    if "subregion_id" not in df.columns:
        if "Balancing Authority Code" in df.columns:
            df["subregion_id"] = normalize_subregion(df["Balancing Authority Code"])
        else:
            raise KeyError("mix table missing 'subregion_id' and missing 'Balancing Authority Code'.")

    # if already there, just standardize key and return
    if COLS["grid_L_per_kwh"] in df.columns:
        df["subregion_id"] = normalize_subregion(df["subregion_id"])
        return df

    # map percent columns -> share columns
    colmap = {
        "BA coal generation percent (resource mix)": "share_coal",
        "BA gas generation percent (resource mix)": "share_gas",
        "BA oil generation percent (resource mix)": "share_oil",
        "BA nuclear generation percent (resource mix)": "share_nuclear",
        "BA hydro generation percent (resource mix)": "share_hydro",
        "BA wind generation percent (resource mix)": "share_wind",
        "BA solar generation percent (resource mix)": "share_solar",
        "BA biomass generation percent (resource mix)": "share_biomass",
        "BA geothermal generation percent (resource mix)": "share_geothermal",
        "BA other fossil generation percent (resource mix)": "share_other_fossil",
        "BA other unknown/ purchased fuel generation percent (resource mix)": "share_other_unknown",
    }

    # Placeholder factors (L/kWh)
    FACTORS = {
        "share_coal": 1.9,
        "share_gas": 0.7,
        "share_oil": 1.5,
        "share_nuclear": 2.5,
        "share_hydro": 15.0,      # dominates; sensitivity required
        "share_wind": 0.05,
        "share_solar": 0.1,
        "share_biomass": 1.0,
        "share_geothermal": 2.0,
        "share_other_fossil": 1.0,
        "share_other_unknown": 1.0,
    }

    # build share columns
    for src, dst in colmap.items():
        if src in df.columns:
            df[dst] = safe_num(df[src]) / 100.0

    share_cols = [c for c in FACTORS.keys() if c in df.columns]
    if len(share_cols) == 0:
        raise KeyError(
            "Could not find BA generation percent columns to compute grid_L_per_kwh. "
            "Verify BA_egrid.xlsx columns."
        )

    df[share_cols] = df[share_cols].fillna(0)
    ssum = df[share_cols].sum(axis=1)
    df.loc[ssum > 0, share_cols] = df.loc[ssum > 0, share_cols].div(ssum[ssum > 0], axis=0)

    df[COLS["grid_L_per_kwh"]] = 0.0
    for c in share_cols:
        df[COLS["grid_L_per_kwh"]] += df[c] * FACTORS[c]

    df["share_fossil"] = df.get("share_coal", 0) + df.get("share_gas", 0) + df.get("share_oil", 0)
    df["subregion_id"] = normalize_subregion(df["subregion_id"])
    return df

# ============================================================
# 1) Build BA summary from sites and merge into BA polygons
# ============================================================
def ensure_subregion_id_on_ba(ba_gdf):
    out = ba_gdf.copy()
    if "subregion_id" not in out.columns:
        if "region_B_1" in out.columns:
            out["subregion_id"] = normalize_subregion(out["region_B_1"])
        else:
            raise KeyError("BA polygons must include 'region_B_1' or already have 'subregion_id'.")
    else:
        out["subregion_id"] = normalize_subregion(out["subregion_id"])
    return out

def build_ba_summary(gdf_sites, ba_polys_joined):
    """
    Returns BA polygons with BA-level summaries:
      - n_sites
      - hyperscale_mw
      - mean_bws (if available)
      - share_high_stress (if available)
    """
    ba_out = ensure_subregion_id_on_ba(ba_polys_joined)

    subr = COLS["subregion"]
    require_cols(gdf_sites, [subr, COLS["mw"]], "gdf_sites")

    sites = gdf_sites.copy()
    sites["subregion_id"] = normalize_subregion(sites[subr])
    sites[COLS["mw"]] = safe_num(sites[COLS["mw"]])

    bws_col = COLS.get("bws", "bws_01_score")
    has_bws = bws_col in sites.columns
    if has_bws:
        sites[bws_col] = safe_num(sites[bws_col])

    agg = {
        "n_sites": (COLS["mw"], "count"),
        "hyperscale_mw": (COLS["mw"], "sum"),
    }
    if has_bws:
        agg["mean_bws"] = (bws_col, "mean")
        agg["share_high_stress"] = (bws_col, lambda s: (safe_num(s) >= SCEN["HIGH_STRESS_THRESHOLD"]).mean())

    ba_stats = sites.groupby("subregion_id", as_index=False).agg(**agg)

    out = ba_out.merge(ba_stats, on="subregion_id", how="left")
    if "mean_bws" not in out.columns:
        out["mean_bws"] = np.nan
    if "share_high_stress" not in out.columns:
        out["share_high_stress"] = np.nan

    return out

def ensure_ba_summary(ba_p, gdf_p):
    """
    Hard guarantee that BA table contains:
      subregion_id, hyperscale_mw, mean_bws, share_high_stress
    """
    ba_p = ensure_subregion_id_on_ba(ba_p)
    missing = [c for c in ["hyperscale_mw", "mean_bws", "share_high_stress"] if c not in ba_p.columns]
    if not missing:
        return ba_p
    return build_ba_summary(gdf_p, ba_p)

# ============================================================
# FIGURE 1: Basin stress + site distribution
# ============================================================
def fig1_basins_context(basins_p, gdf_p, states_p=None):
    if basins_p is None:
        raise ValueError("basins_p is None. Load HydroBASINS + Aqueduct join first.")
    require_cols(gdf_p, [COLS["mw"]], "gdf_p")

    stress_candidates = ["bws_01_score", COLS.get("bws", "bws_01_score")]
    stress_col = next((c for c in stress_candidates if c in basins_p.columns), None)
    if stress_col is None:
        raise KeyError("Basins layer missing stress column (expected bws_01_score).")

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    ax0, ax1 = axes

    # (a) basin map
    draw_base(ax0, states_layer=states_p, boundary_layer=None)
    vals = safe_num(basins_p[stress_col])
    basins_p[vals.isna()].plot(ax=ax0, color="#d9d9d9", linewidth=0.0)
    basins_p[vals.notna()].plot(ax=ax0, column=stress_col, cmap="YlOrRd", linewidth=0.0, vmin=0, vmax=5)

    s = scaled_point_sizes(gdf_p[COLS["mw"]], 10, 260)
    gdf_p.plot(ax=ax0, markersize=s, color="black", alpha=0.25)
    ax0.set_title("(a) Basin water stress (Aqueduct) + hyperscale sites (size ∝ MW)")

    sm = mpl.cm.ScalarMappable(cmap=plt.get_cmap("YlOrRd"),
                               norm=mpl.colors.Normalize(vmin=0, vmax=5))
    sm.set_array([])
    cb = fig.colorbar(sm, ax=ax0, fraction=0.035, pad=0.01)
    cb.set_label("baseline water stress (0–5)")

    # (b) site stress distribution
    bws_col = COLS.get("bws", "bws_01_score")
    if bws_col in gdf_p.columns:
        site_bws = safe_num(gdf_p[bws_col]).dropna()
        ax1.hist(site_bws, bins=np.linspace(0, 5, 11),
                 edgecolor="white", linewidth=0.6, alpha=0.9)
        ax1.axvline(SCEN["HIGH_STRESS_THRESHOLD"], linestyle="--", linewidth=1.2)
        ax1.set_xlabel("baseline water stress (0–5)")
        ax1.set_ylabel("number of sites")
        ax1.set_title("(b) Distribution of site water stress")
    else:
        ax1.text(0.5, 0.5, "No site stress column found.", ha="center", va="center")
        ax1.set_axis_off()

    savefig(fig, "Fig1_basins_context")

# ============================================================
# FIGURE 2: Grid water intensity heterogeneity (+ optional CO2 nexus)
# ============================================================
def fig2_grid_water(ba_p, gdf_p, states_p=None):
    require_cols(ba_p, [COLS["grid_L_per_kwh"]], "ba_p")
    ba_p = ensure_ba_summary(ba_p, gdf_p)

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    ax0, ax1, ax2 = axes

    # (a) map
    draw_base(ax0, states_layer=states_p, boundary_layer=ba_p)
    vals = safe_num(ba_p[COLS["grid_L_per_kwh"]])
    ba_p[vals.isna()].plot(ax=ax0, color="#d9d9d9", linewidth=0.0)
    ba_p[vals.notna()].plot(ax=ax0, column=COLS["grid_L_per_kwh"], cmap="viridis", linewidth=0.0)

    s = scaled_point_sizes(gdf_p[COLS["mw"]], 10, 260)
    gdf_p.plot(ax=ax0, markersize=s, color="black", alpha=0.22)
    ax0.set_title("(a) Grid water intensity by BA (L/kWh) + hyperscale sites")

    norm = mpl.colors.Normalize(vmin=np.nanpercentile(vals, 5), vmax=np.nanpercentile(vals, 95))
    sm = mpl.cm.ScalarMappable(cmap=plt.get_cmap("viridis"), norm=norm)
    sm.set_array([])
    cb = fig.colorbar(sm, ax=ax0, fraction=0.035, pad=0.01)
    cb.set_label("L/kWh")

    # (b) water vs CO2 scatter (optional)
    co2_candidates = [
        "co2_kg_per_mwh",
        "co2_intensity_kg_per_mwh",
        "co2_lb_per_mwh",
        "BA annual CO2 total output emission rate (lb/MWh)",
    ]
    co2_col = next((c for c in co2_candidates if c in ba_p.columns), None)

    x = safe_num(ba_p[COLS["grid_L_per_kwh"]])
    mw = safe_num(ba_p["hyperscale_mw"])
    if not isinstance(mw, pd.Series):
        mw = pd.Series(mw, index=ba_p.index).fillna(0)

    if co2_col is not None:
        y = safe_num(ba_p[co2_col])
        bubble = 20 + 180 * (np.sqrt(mw.fillna(0)) / (np.sqrt(mw.fillna(0)).max() + 1e-9))
        ax1.scatter(x, y, s=bubble, alpha=0.7)
        ax1.set_xlabel("Grid water intensity (L/kWh)")
        ax1.set_ylabel(co2_col)
        ax1.set_title("(b) Water vs CO₂ intensity (bubble size ∝ hyperscale MW)")
        ax1.grid(True, linewidth=0.3, alpha=0.35)
    else:
        ax1.text(0.5, 0.5, "CO₂ intensity column not found in BA table.\n(panel optional)",
                 ha="center", va="center")
        ax1.set_axis_off()

    # (c) MW-weighted distribution
    xvals = x.dropna()
    w = mw.loc[xvals.index].fillna(0)
    if len(xvals) > 0:
        bins = np.linspace(np.nanpercentile(xvals, 1), np.nanpercentile(xvals, 99), 16)
        ax2.hist(xvals, bins=bins, weights=w, edgecolor="white", linewidth=0.6)
        ax2.set_title("(c) Hyperscale-MW-weighted distribution of grid water intensity")
        ax2.set_xlabel("Grid water intensity (L/kWh)")
        ax2.set_ylabel("MW-weighted count")
        ax2.grid(True, axis="y", linewidth=0.3, alpha=0.35)
    else:
        ax2.text(0.5, 0.5, "No grid intensity values.", ha="center", va="center")
        ax2.set_axis_off()

    savefig(fig, "Fig2_grid_water_heterogeneity")
    return ba_p

# ============================================================
# FIGURE 3: Nexus bivariate maps (stress × grid water, stress × capacity)
# ============================================================
def fig3_nexus_bivariate(ba_p, gdf_p, states_p=None):
    require_cols(ba_p, [COLS["grid_L_per_kwh"]], "ba_p")
    ba_p = ensure_ba_summary(ba_p, gdf_p)

    require_cols(ba_p, ["mean_bws", "hyperscale_mw"], "ba_p (after summary)")

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    axA, axB = axes

    # (a) bivariate: mean_bws × grid_L_per_kwh
    b1, _, _ = make_bivariate_colors(ba_p, "mean_bws", COLS["grid_L_per_kwh"], n=3)
    draw_base(axA, states_layer=states_p, boundary_layer=ba_p)
    b1.plot(ax=axA, color=b1["_bivar_color"], linewidth=0.0)
    gdf_p.plot(ax=axA, markersize=scaled_point_sizes(gdf_p[COLS["mw"]], 10, 260),
               color="black", alpha=0.18)
    axA.set_title("(a) Nexus: mean site stress × grid water intensity (by BA)")
    draw_bivariate_legend(axA, xlab="Higher stress", ylab="Higher grid water", loc=(0.02, 0.05), size=0.20)
    add_stress_hist_inset(axA, gdf_p, COLS.get("bws", "bws_01_score"), title="Site stress")

    # (b) bivariate: mean_bws × hyperscale_mw
    b2, _, _ = make_bivariate_colors(ba_p, "mean_bws", "hyperscale_mw", n=3)
    draw_base(axB, states_layer=states_p, boundary_layer=ba_p)
    b2.plot(ax=axB, color=b2["_bivar_color"], linewidth=0.0)
    gdf_p.plot(ax=axB, markersize=scaled_point_sizes(gdf_p[COLS["mw"]], 10, 260),
               color="black", alpha=0.18)
    axB.set_title("(b) Nexus: mean site stress × hyperscale capacity (by BA)")
    draw_bivariate_legend(axB, xlab="Higher stress", ylab="Higher capacity", loc=(0.02, 0.05), size=0.20)
    add_stress_hist_inset(axB, gdf_p, COLS.get("bws", "bws_01_score"), title="Site stress")

    savefig(fig, "Fig3_nexus_bivariate_maps")
    return ba_p

# ============================================================
# FIGURE 4: Operator exposure (portfolio risk)
# ============================================================
def fig4_operator_exposure(gdf_sites, ba_table=None):
    require_cols(gdf_sites, [COLS["company"], COLS["mw"]], "gdf_sites")
    bws_col = COLS.get("bws", "bws_01_score")
    if bws_col not in gdf_sites.columns:
        raise KeyError(f"gdf_sites missing required stress column '{bws_col}' for Fig4.")

    df = gdf_sites.copy()
    df[COLS["mw"]] = safe_num(df[COLS["mw"]])
    df[bws_col] = safe_num(df[bws_col])
    df = df.dropna(subset=[COLS["mw"]]).copy()
    df["is_high_stress"] = df[bws_col] >= SCEN["HIGH_STRESS_THRESHOLD"]

    comp = (df.groupby(COLS["company"], as_index=False)
              .agg(
                  n_sites=(COLS["mw"], "count"),
                  total_mw=(COLS["mw"], "sum"),
                  share_sites_high_stress=("is_high_stress", "mean"),
                  mw_high_stress=(COLS["mw"], lambda s: s[df.loc[s.index, "is_high_stress"]].sum()),
              ))

    comp = comp.sort_values("total_mw", ascending=False).head(12).copy()
    comp["share_mw_high_stress"] = comp["mw_high_stress"] / (comp["total_mw"] + 1e-12)

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    ax0, ax1, ax2 = axes

    comp_a = comp.sort_values("share_mw_high_stress", ascending=True)
    ax0.barh(comp_a[COLS["company"]], comp_a["share_mw_high_stress"])
    ax0.set_xlim(0, 1)
    ax0.set_xlabel("Share of MW in high-stress basins")
    ax0.set_title("(a) Operator exposure (share)")
    ax0.grid(True, axis="x", linewidth=0.3, alpha=0.4)

    comp_b = comp.sort_values("mw_high_stress", ascending=True)
    ax1.barh(comp_b[COLS["company"]], comp_b["mw_high_stress"])
    ax1.set_xlabel("MW in high-stress basins")
    ax1.set_title("(b) Operator exposure (absolute)")
    ax1.grid(True, axis="x", linewidth=0.3, alpha=0.4)

    # Optional quadrant: x=share MW high stress, y=MW-weighted grid water intensity
    if ba_table is not None:
        ba_table = ba_table.copy()
        ba_table = ensure_subregion_id_on_ba(ba_table)
        if COLS["grid_L_per_kwh"] in ba_table.columns:
            tmp = df.copy()
            tmp["subregion_id"] = normalize_subregion(tmp[COLS["subregion"]]) if COLS["subregion"] in tmp.columns else None
            if "subregion_id" in tmp.columns and tmp["subregion_id"].notna().any():
                bt = ba_table[["subregion_id", COLS["grid_L_per_kwh"]]].copy()
                tmp = tmp.merge(bt, on="subregion_id", how="left")

                def mw_weighted_avg(sub):
                    w = safe_num(sub[COLS["mw"]]).values
                    x = safe_num(sub[COLS["grid_L_per_kwh"]]).values
                    m = np.isfinite(x) & np.isfinite(w) & (w >= 0)
                    if m.sum() == 0:
                        return np.nan
                    return np.average(x[m], weights=w[m])

                comp2 = (tmp.groupby(COLS["company"])
                           .apply(mw_weighted_avg)
                           .reset_index(name="mw_weighted_grid_L_per_kwh"))
                comp = comp.merge(comp2, on=COLS["company"], how="left")

                bubble = 40 + 260 * (np.sqrt(comp["total_mw"]) / (np.sqrt(comp["total_mw"]).max() + 1e-9))
                ax2.scatter(comp["share_mw_high_stress"], comp["mw_weighted_grid_L_per_kwh"], s=bubble, alpha=0.8)
                for _, r in comp.iterrows():
                    ax2.text(r["share_mw_high_stress"] + 0.01,
                             r["mw_weighted_grid_L_per_kwh"],
                             str(r[COLS["company"]])[:12], fontsize=8)

                ax2.axvline(0.5, linestyle="--", linewidth=1.0)
                ax2.set_xlabel("Share MW in high-stress basins")
                ax2.set_ylabel("MW-weighted grid water intensity (L/kWh)")
                ax2.set_title("(c) Portfolio risk quadrant (size ∝ total MW)")
                ax2.grid(True, linewidth=0.3, alpha=0.4)
            else:
                ax2.text(0.5, 0.5, "Quadrant needs facility subregion mapping.", ha="center", va="center")
                ax2.set_axis_off()
        else:
            ax2.text(0.5, 0.5, "Quadrant needs BA grid_L_per_kwh.", ha="center", va="center")
            ax2.set_axis_off()
    else:
        ax2.text(0.5, 0.5, "Quadrant plot optional (needs ba_table).", ha="center", va="center")
        ax2.set_axis_off()

    savefig(fig, "Fig4_operator_exposure")
    return comp

# ============================================================
# ED6: Rank shift proxy (volumetric -> scarcity-weighted)
# ============================================================
def ed6_rank_shift(ba_p):
    require_cols(ba_p, ["subregion_id", "hyperscale_mw", "mean_bws"], "ba_p")

    df = ba_p[["subregion_id", "hyperscale_mw", "mean_bws"]].copy()
    df["vol_metric"] = safe_num(df["hyperscale_mw"])
    df["scar_metric"] = safe_num(df["hyperscale_mw"]) * safe_num(df["mean_bws"])
    df = df.dropna(subset=["vol_metric", "scar_metric"]).copy()

    df["rank_vol"] = df["vol_metric"].rank(ascending=False, method="dense")
    df["rank_scar"] = df["scar_metric"].rank(ascending=False, method="dense")
    df["rank_shift"] = df["rank_scar"] - df["rank_vol"]

    top = df.sort_values("vol_metric", ascending=False).head(20).copy()
    top = top.sort_values("rank_shift", ascending=True)

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(top["subregion_id"], top["rank_shift"])
    ax.axvline(0, linewidth=1.0)
    ax.set_title("ED6: Rank shift after scarcity weighting (proxy)")
    ax.set_xlabel("Rank shift = (scarcity rank) − (volumetric rank)\n(negative = more important under scarcity)")
    ax.grid(True, axis="x", linewidth=0.3, alpha=0.4)
    savefig(fig, "ED6_rank_shift_scarcity_proxy")

# ============================================================
# ED7: Hydro sensitivity (with/without hydro)
# ============================================================
def ed7_hydro_sensitivity(mix_df):
    if "share_hydro" not in mix_df.columns:
        print("ED7 skipped: mix_df lacks share_hydro (need it computed from BA_egrid percent columns).")
        return

    df = mix_df.copy()
    require_cols(df, ["subregion_id", COLS["grid_L_per_kwh"], "share_hydro"], "mix_df")

    share_cols = [c for c in df.columns if c.startswith("share_")]
    nonhydro = [c for c in share_cols if c != "share_hydro"]
    ssum = df[nonhydro].sum(axis=1)
    df_non = df.copy()
    df_non[nonhydro] = df_non[nonhydro].div(ssum.replace(0, np.nan), axis=0).fillna(0)

    FACTORS_NO_HYDRO = {
        "share_coal": 1.9, "share_gas": 0.7, "share_oil": 1.5, "share_nuclear": 2.5,
        "share_hydro": 0.0,
        "share_wind": 0.05, "share_solar": 0.1, "share_biomass": 1.0, "share_geothermal": 2.0,
        "share_other_fossil": 1.0, "share_other_unknown": 1.0,
    }
    df_non["grid_L_per_kwh_no_hydro"] = 0.0
    for c, f in FACTORS_NO_HYDRO.items():
        if c in df_non.columns:
            df_non["grid_L_per_kwh_no_hydro"] += df_non[c] * f

    tmp = df_non[["subregion_id", COLS["grid_L_per_kwh"], "grid_L_per_kwh_no_hydro"]].dropna()
    tmp["rank_with"] = tmp[COLS["grid_L_per_kwh"]].rank(ascending=False, method="dense")
    tmp["rank_no"] = tmp["grid_L_per_kwh_no_hydro"].rank(ascending=False, method="dense")
    tmp["rank_shift"] = tmp["rank_no"] - tmp["rank_with"]
    top = tmp.sort_values("rank_shift", ascending=True).head(25)

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(top["subregion_id"], top["rank_shift"])
    ax.axvline(0, linewidth=1.0)
    ax.set_title("ED7: BA rank shift when hydro is excluded from grid water intensity")
    ax.set_xlabel("Rank shift = (no-hydro rank) − (with-hydro rank)\n(negative = more important without hydro)")
    ax.grid(True, axis="x", linewidth=0.3, alpha=0.4)
    savefig(fig, "ED7_hydro_sensitivity_rankshift")

# ============================================================
# Run suite
# ============================================================
def run_figure_suite(gdf_p, ba_polys, mix, states_p=None, basins_p=None, LCC=None):
    """
    Minimal assumptions:
      - gdf_p: facilities already projected to LCC
      - ba_polys: BA polygons (WGS84 or LCC ok)
      - mix: BA_egrid table loaded
      - states_p: optional LCC state borders
      - basins_p: optional LCC basins+Aqueduct join
      - LCC: CRS string if reprojection needed
    Returns:
      (ba_p2, comp, mix2)
    """

    # 1) Ensure grid water intensity exists in mix
    mix2 = ensure_grid_water_intensity(mix)

    # 2) Standardize BA polygons key, merge mix into BA polygons
    ba_polys2 = ensure_subregion_id_on_ba(ba_polys)
    ba_polys_m = ba_polys2.merge(mix2, on="subregion_id", how="left")

    # 3) Project BA polygons if needed
    ba_p = ba_polys_m
    if LCC is not None and (ba_p.crs is None or str(ba_p.crs) != str(LCC)):
        try:
            ba_p = ba_p.to_crs(LCC)
        except Exception:
            # If already in LCC-like string, ignore
            pass

    # 4) Ensure BA summary columns exist
    ba_p2 = build_ba_summary(gdf_p, ba_p)
    ba_p2 = gpd.GeoDataFrame(ba_p2, geometry="geometry", crs=ba_p.crs)

    # MAIN FIGS
    if basins_p is not None:
        fig1_basins_context(basins_p, gdf_p, states_p=states_p)
    ba_p2 = fig2_grid_water(ba_p2, gdf_p, states_p=states_p)
    ba_p2 = fig3_nexus_bivariate(ba_p2, gdf_p, states_p=states_p)
    comp = fig4_operator_exposure(gdf_p, ba_table=ba_p2)

    # ED
    ed6_rank_shift(ba_p2)
    ed7_hydro_sensitivity(mix2)

    print("✅ Figure suite written to:", OUTDIR)
    return ba_p2, comp, mix2

# Example call (use your in-notebook objects):
# ba_p2, comp, mix2 = run_figure_suite(gdf_p, ba_polys, mix, states_p=states_p, basins_p=basins_p, LCC=LCC)


In [ ]:
ba_p2, comp, mix2 = run_figure_suite(gdf_p, ba_polys, mix, states_p=states_p, basins_p=basins_p, LCC=LCC)


In [ ]:
def basin_hotspots(gdf_sites, basins_p, ba_table,
                   stress_col="bws_01_score",
                   fossil_col="share_fossil",
                   mw_col=COLS["mw"],
                   subr_col=COLS["subregion"],
                   pfaf_col=COLS["pfaf"],
                   q=0.750,
                   stress_hi=3.0):
    """
    Returns basin summary table with hotspot flags + category.
    Requires:
      - gdf_sites: projected GeoDataFrame (sites) with mw, pfaf_id, subregion_id
      - basins_p: projected basins with PFAF_ID and stress_col
      - ba_table: BA table with subregion_id and fossil_col
    """

    sites = gdf_sites.copy()
    sites[mw_col] = safe_num(sites[mw_col])
    sites = sites.dropna(subset=[mw_col]).copy()

    # join BA fossil share onto sites
    bt = ba_table[["subregion_id", fossil_col]].copy()
    bt["subregion_id"] = bt["subregion_id"].astype(str).str.strip().str.upper()
    sites["subregion_id"] = sites[subr_col].astype(str).str.strip().str.upper()
    sites = sites.merge(bt, on="subregion_id", how="left")

    # normalize pfaf
    if pfaf_col not in sites.columns:
        raise KeyError(f"Sites missing {pfaf_col}. Add basin join first.")
    sites[pfaf_col] = sites[pfaf_col].astype(str).str.strip()

    # basin stress table
    b = basins_p.copy()
    if "PFAF_ID" not in b.columns:
        raise KeyError("basins_p missing PFAF_ID.")
    b["PFAF_ID"] = b["PFAF_ID"].astype(str).str.strip()
    if stress_col not in b.columns:
        raise KeyError(f"basins_p missing {stress_col}.")

    basin_stress = b[["PFAF_ID", stress_col]].drop_duplicates("PFAF_ID")

    # aggregate sites -> basins
    def mw_weighted_mean(x, w):
        x = safe_num(x)
        w = safe_num(w).fillna(0)
        m = np.isfinite(x) & np.isfinite(w) & (w > 0)
        if m.sum() == 0:
            return np.nan
        return np.average(x[m], weights=w[m])

    basin = (sites.groupby(pfaf_col, as_index=False)
                  .agg(
                      n_sites=(mw_col, "count"),
                      mw_total=(mw_col, "sum"),
                      fossil_mw_weighted=(fossil_col, lambda s: mw_weighted_mean(s, sites.loc[s.index, mw_col])),
                  ))

    basin = basin.rename(columns={pfaf_col: "PFAF_ID"})
    basin = basin.merge(basin_stress, on="PFAF_ID", how="left")

    # thresholds
    mw_thr = basin["mw_total"].quantile(q)
    fos_thr = basin["fossil_mw_weighted"].quantile(q)

    basin["hi_mw"] = basin["mw_total"] >= mw_thr
    basin["hi_fossil"] = basin["fossil_mw_weighted"] >= fos_thr
    basin["hi_scarcity"] = safe_num(basin[stress_col]) >= stress_hi

    # category code 0..7 (bitmask)
    basin["hot_code"] = (basin["hi_mw"].astype(int) * 4 +
                         basin["hi_fossil"].astype(int) * 2 +
                         basin["hi_scarcity"].astype(int) * 1)

    labels = {
        0: "None",
        1: "Scarcity only",
        2: "Fossil only",
        3: "Fossil + scarcity",
        4: "MW only",
        5: "MW + scarcity",
        6: "MW + fossil",
        7: "MW + fossil + scarcity",
    }
    basin["hotspot_type"] = basin["hot_code"].map(labels)

    meta = {"mw_thr": mw_thr, "fos_thr": fos_thr, "stress_thr": stress_hi, "q": q}
    return basin, meta


def plot_basin_hotspots(basins_p, basin_table, states_p=None, title=None):
    # merge categories back to geometry
    b = basins_p.copy()
    b["PFAF_ID"] = b["PFAF_ID"].astype(str).str.strip()
    t = basin_table[["PFAF_ID", "hot_code", "hotspot_type"]].copy()
    out = b.merge(t, on="PFAF_ID", how="left")

    # discrete colormap (8 categories)
    cmap = mpl.colors.ListedColormap([
        "#e0e0e0",  # 0 none
        "#fddbc7",  # 1 scarcity
        "#d1e5f0",  # 2 fossil
        "#92c5de",  # 3 fossil+scarcity
        "#c7e9c0",  # 4 MW
        "#41ab5d",  # 5 MW+scarcity
        "#74a9cf",  # 6 MW+fossil
        "#045a8d",  # 7 triple
    ])
    norm = mpl.colors.BoundaryNorm(np.arange(-0.5, 8.5, 1), cmap.N)

    fig, ax = plt.subplots(figsize=(10.5, 6.5))
    draw_base(ax, states_layer=states_p, boundary_layer=None)
    out.plot(ax=ax, column="hot_code", cmap=cmap, norm=norm, linewidth=0.0)

    ax.set_title(title or "Basin hotspot typology (MW × fossil mix × scarcity)")

    # legend
    handles = []
    label_map = {
        0: "None",
        1: "Scarcity only",
        2: "Fossil only",
        3: "Fossil + scarcity",
        4: "MW only",
        5: "MW + scarcity",
        6: "MW + fossil",
        7: "MW + fossil + scarcity",
    }
    for k in range(8):
        handles.append(mpl.patches.Patch(facecolor=cmap(k), edgecolor="none", label=label_map[k]))
    ax.legend(handles=handles, loc="lower left", frameon=True, fontsize=8, title="Hotspot type")

    savefig(fig, "Fig_hotspot_typology_basins")


def plot_hotspot_overlap_counts(basin_table, title="How hotspots overlap"):
    counts = basin_table["hotspot_type"].value_counts().reindex([
        "MW + fossil + scarcity",
        "MW + fossil",
        "MW + scarcity",
        "Fossil + scarcity",
        "MW only",
        "Fossil only",
        "Scarcity only",
        "None"
    ]).fillna(0)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(counts.index, counts.values)
    ax.set_title(title)
    ax.set_xlabel("Number of basins")
    ax.grid(True, axis="x", linewidth=0.3, alpha=0.4)
    savefig(fig, "ED_hotspot_overlap_counts")


In [ ]:
basin_tbl, meta = basin_hotspots(gdf_p, basins_p, ba_p2, q=0.80, stress_hi=3.0)
print(meta)

plot_basin_hotspots(basins_p, basin_tbl, states_p=states_p,
                    title="Hotspots where hyperscale capacity intersects fossil-heavy grids and water-scarce basins")

plot_hotspot_overlap_counts(basin_tbl)


In [ ]:
def basin_hotspots_fixed_thresholds(
    gdf_sites,
    basins_p,
    ba_table,
    mw_col=COLS["mw"],
    subr_col=COLS["subregion"],
    pfaf_col=COLS["pfaf"],
    stress_col="bws_01_score",
    coal_share_col="share_coal",
    gas_share_col="share_gas",
    fossil_threshold=0.50,
    mw_quantile=0.75,
    scarcity_threshold=3.0,
):
    """
    Basin hotspot classification using fixed, interpretable thresholds:
      - High MW: basin mw_total >= 75th percentile of basin totals
      - High fossil: MW-weighted (coal+gas) share >= 0.50
      - High scarcity: basin stress >= 3

    Requires:
      - gdf_sites has pfaf_id, subregion_id, current_mw
      - basins_p has PFAF_ID and bws_01_score (after Aqueduct join)
      - ba_table has subregion_id and share_coal/share_gas (or equivalent)
    """

    sites = gdf_sites.copy()
    require_cols(sites, [mw_col, subr_col, pfaf_col], "gdf_sites")
    sites[mw_col] = safe_num(sites[mw_col])
    sites = sites.dropna(subset=[mw_col]).copy()

    # Normalize keys
    sites["subregion_id"] = sites[subr_col].astype(str).str.strip().str.upper()
    sites["PFAF_ID"] = sites[pfaf_col].astype(str).str.strip()

    # Basin stress table
    b = basins_p.copy()
    require_cols(b, ["PFAF_ID", stress_col, "geometry"], "basins_p")
    b["PFAF_ID"] = b["PFAF_ID"].astype(str).str.strip()
    basin_stress = b[["PFAF_ID", stress_col]].drop_duplicates("PFAF_ID")

    # Attach BA coal/gas shares to each site
    bt = ba_table.copy()
    require_cols(bt, ["subregion_id"], "ba_table")
    if coal_share_col not in bt.columns or gas_share_col not in bt.columns:
        raise KeyError(
            f"ba_table missing {coal_share_col}/{gas_share_col}. "
            "Make sure ensure_grid_water_intensity() creates share_coal/share_gas, "
            "or rename columns accordingly."
        )

    bt2 = bt[["subregion_id", coal_share_col, gas_share_col]].copy()
    bt2["subregion_id"] = bt2["subregion_id"].astype(str).str.strip().str.upper()
    bt2[coal_share_col] = safe_num(bt2[coal_share_col])
    bt2[gas_share_col] = safe_num(bt2[gas_share_col])
    bt2["share_fossil_cg"] = bt2[coal_share_col].fillna(0) + bt2[gas_share_col].fillna(0)

    sites = sites.merge(bt2[["subregion_id", "share_fossil_cg"]], on="subregion_id", how="left")

    # MW-weighted mean helper
    def mw_weighted_mean(x, w):
        x = safe_num(x)
        w = safe_num(w).fillna(0)
        m = np.isfinite(x) & np.isfinite(w) & (w > 0)
        if m.sum() == 0:
            return np.nan
        return np.average(x[m], weights=w[m])

    # Aggregate to basin
    basin = (sites.groupby("PFAF_ID", as_index=False)
                  .agg(
                      n_sites=(mw_col, "count"),
                      mw_total=(mw_col, "sum"),
                      fossil_cg_mw_weighted=("share_fossil_cg",
                                            lambda s: mw_weighted_mean(s, sites.loc[s.index, mw_col])),
                  ))

    basin = basin.merge(basin_stress, on="PFAF_ID", how="left")

    # Thresholds
    mw_thr = basin["mw_total"].quantile(mw_quantile)

    basin["hi_mw"] = basin["mw_total"] >= mw_thr
    basin["hi_fossil"] = basin["fossil_cg_mw_weighted"] >= fossil_threshold
    basin["hi_scarcity"] = safe_num(basin[stress_col]) >= scarcity_threshold

    # Bitmask code: MW=4, fossil=2, scarcity=1
    basin["hot_code"] = (
        basin["hi_mw"].astype(int) * 4 +
        basin["hi_fossil"].astype(int) * 2 +
        basin["hi_scarcity"].astype(int) * 1
    )

    labels = {
        0: "None",
        1: "Scarcity only",
        2: "Fossil (coal+gas) only",
        3: "Fossil + scarcity",
        4: "MW only",
        5: "MW + scarcity",
        6: "MW + fossil",
        7: "MW + fossil + scarcity",
    }
    basin["hotspot_type"] = basin["hot_code"].map(labels)

    meta = {
        "mw_threshold_quantile": mw_quantile,
        "mw_threshold_value": float(mw_thr) if np.isfinite(mw_thr) else np.nan,
        "fossil_cg_threshold": fossil_threshold,
        "scarcity_threshold": scarcity_threshold,
        "notes": "Fossil defined as coal+gas (MW-weighted by sites within basin).",
    }
    return basin, meta


In [ ]:
def plot_basin_hotspots_typology(basins_p, basin_table, states_p=None, name="Fig_hotspots_basins_typology"):
    b = basins_p.copy()
    b["PFAF_ID"] = b["PFAF_ID"].astype(str).str.strip()
    t = basin_table[["PFAF_ID", "hot_code"]].copy()
    out = b.merge(t, on="PFAF_ID", how="left")
    out["hot_code"] = out["hot_code"].fillna(0).astype(int)

    cmap = mpl.colors.ListedColormap([
        "#e0e0e0",  # 0 none
        "#fddbc7",  # 1 scarcity only
        "#d1e5f0",  # 2 fossil only
        "#92c5de",  # 3 fossil+scarcity
        "#c7e9c0",  # 4 MW only
        "#41ab5d",  # 5 MW+scarcity
        "#74a9cf",  # 6 MW+fossil
        "#045a8d",  # 7 triple
    ])
    norm = mpl.colors.BoundaryNorm(np.arange(-0.5, 8.5, 1), cmap.N)

    fig, ax = plt.subplots(figsize=(11, 6.8))
    draw_base(ax, states_layer=states_p, boundary_layer=None)
    out.plot(ax=ax, column="hot_code", cmap=cmap, norm=norm, linewidth=0.0)

    ax.set_title("Basin hotspots: MW (≥P75) × fossil (coal+gas ≥50%) × scarcity (BWS ≥3)")

    label_map = {
        0: "None",
        1: "Scarcity only",
        2: "Fossil only",
        3: "Fossil + scarcity",
        4: "MW only",
        5: "MW + scarcity",
        6: "MW + fossil",
        7: "MW + fossil + scarcity",
    }
    handles = [mpl.patches.Patch(facecolor=cmap(i), edgecolor="none", label=label_map[i]) for i in range(8)]
    ax.legend(handles=handles, loc="lower left", frameon=True, fontsize=8, title="Hotspot type")

    savefig(fig, name)


def plot_hotspot_overlap_counts(basin_table, name="ED_hotspot_overlap_counts"):
    order = [
        "MW + fossil + scarcity",
        "MW + fossil",
        "MW + scarcity",
        "Fossil + scarcity",
        "MW only",
        "Fossil (coal+gas) only",
        "Scarcity only",
        "None",
    ]
    counts = basin_table["hotspot_type"].value_counts()
    counts = counts.reindex(order).fillna(0)

    fig, ax = plt.subplots(figsize=(9.5, 5.2))
    ax.barh(counts.index, counts.values)
    ax.set_xlabel("Number of basins")
    ax.set_title("How hotspot drivers overlap (basin typology)")
    ax.grid(True, axis="x", linewidth=0.3, alpha=0.4)
    savefig(fig, name)


In [ ]:
def top_hotspot_table(basin_tbl, top_n=20):
    df = basin_tbl.copy()
    # A simple “severity” ranking inside each class:
    # log(MW) + fossil + stress (all increasing)
    df["severity"] = (
        np.log1p(safe_num(df["mw_total"]).fillna(0)) +
        safe_num(df["fossil_cg_mw_weighted"]).fillna(0) +
        safe_num(df["bws_01_score"]).fillna(0)
    )

    out = df.sort_values(["hot_code", "severity"], ascending=[False, False]).copy()
    cols = ["PFAF_ID", "hotspot_type", "n_sites", "mw_total", "fossil_cg_mw_weighted", "bws_01_score", "severity"]
    return out[cols].head(top_n)


In [ ]:
basin_tbl, meta = basin_hotspots_fixed_thresholds(
    gdf_sites=gdf_p,
    basins_p=basins_p,
    ba_table=ba_p2,           # or mix/ba table if that’s where shares live
    mw_quantile=0.75,
    fossil_threshold=0.50,
    scarcity_threshold=3.0,
)
print(meta)

plot_basin_hotspots_typology(basins_p, basin_tbl, states_p=states_p)
plot_hotspot_overlap_counts(basin_tbl)

top20 = top_hotspot_table(basin_tbl, top_n=20)
top20.to_csv(os.path.join(OUTDIR, "table_top_hotspot_basins.csv"), index=False)
top20.head(10)


In [ ]:
def plot_basin_hotspots_selective(
    basins_p,
    basin_table,
    states_p=None,
    highlight_codes=(7, 6, 5, 3),   # default: triple + key duals
    show_sites=False,
    gdf_sites=None,
    name="Fig_hotspots_basins_selective",
):
    """
    Cleaner hotspot map:
      - all basins light gray
      - only selected hotspot codes colored (default: 7,6,5,3)
      - states boundaries on top
    """
    b = basins_p.copy()
    b["PFAF_ID"] = b["PFAF_ID"].astype(str).str.strip()
    t = basin_table[["PFAF_ID", "hot_code"]].copy()
    t["PFAF_ID"] = t["PFAF_ID"].astype(str).str.strip()
    out = b.merge(t, on="PFAF_ID", how="left")
    out["hot_code"] = out["hot_code"].fillna(0).astype(int)

    # background
    fig, ax = plt.subplots(figsize=(11.2, 6.8))
    draw_base(ax, states_layer=None, boundary_layer=None)
    out.plot(ax=ax, color="#efefef", linewidth=0.0)

    # highlight palette (only for selected)
    code_color = {
        7: "#045a8d",  # MW+fossil+scarcity
        6: "#2b8cbe",  # MW+fossil
        5: "#41ab5d",  # MW+scarcity
        3: "#74a9cf",  # fossil+scarcity
        4: "#c7e9c0",  # MW only (optional)
        2: "#d1e5f0",  # fossil only (optional)
        1: "#fddbc7",  # scarcity only (optional)
    }

    # plot only highlighted codes
    for code in highlight_codes:
        sel = out[out["hot_code"] == code]
        if len(sel) == 0:
            continue
        sel.plot(ax=ax, color=code_color.get(code, "#444444"), linewidth=0.0)

    # optional: outline highlighted basins (thin)
    out[out["hot_code"].isin(highlight_codes)].boundary.plot(
        ax=ax, linewidth=0.25, color="white", alpha=0.7
    )

    # states boundaries on top (clean)
    if states_p is not None:
        states_p.boundary.plot(ax=ax, linewidth=0.6, color="black", alpha=0.9)

    # optional sites overlay
    if show_sites and (gdf_sites is not None):
        require_cols(gdf_sites, [COLS["mw"]], "gdf_sites")
        s = scaled_point_sizes(gdf_sites[COLS["mw"]], 6, 160)
        gdf_sites.plot(ax=ax, markersize=s, color="black", alpha=0.22)

    # legend with only what’s shown
    label_map = {
        7: "MW + fossil + scarcity",
        6: "MW + fossil",
        5: "MW + scarcity",
        3: "Fossil + scarcity",
        4: "MW only",
        2: "Fossil only",
        1: "Scarcity only",
    }
    handles = [
        mpl.patches.Patch(facecolor=code_color[c], edgecolor="none", label=label_map.get(c, str(c)))
        for c in highlight_codes if c in code_color
    ]
    ax.legend(handles=handles, loc="lower left", frameon=True, fontsize=8, title="Hotspot class")

    ax.set_title("Hotspot basins (selective highlight)")
    ax.set_axis_off()
    savefig(fig, name)


In [ ]:
plot_basin_hotspots_selective(
    basins_p, basin_tbl, states_p=states_p,
    highlight_codes=(7,6,5,3),   # clean “story”
    show_sites=True, gdf_sites=gdf_p
)


In [ ]:
def plot_only_hotspot_basins(
    basins_p,
    basin_table,
    states_p=None,
    highlight_codes=(7, 6, 5, 3),
    name="Fig_hotspots_basins_only",
):
    b = basins_p.copy()
    b["PFAF_ID"] = b["PFAF_ID"].astype(str).str.strip()
    t = basin_table[["PFAF_ID", "hot_code"]].copy()
    t["PFAF_ID"] = t["PFAF_ID"].astype(str).str.strip()
    out = b.merge(t, on="PFAF_ID", how="left")
    out["hot_code"] = out["hot_code"].fillna(0).astype(int)

    code_color = {7:"#045a8d", 6:"#2b8cbe", 5:"#41ab5d", 3:"#74a9cf"}

    fig, ax = plt.subplots(figsize=(11.2, 6.8))
    draw_base(ax, states_layer=None, boundary_layer=None)

    # faint land background (optional, minimal)
    if states_p is not None:
        states_p.plot(ax=ax, facecolor="#f7f7f7", edgecolor="none", alpha=1.0)

    # plot ONLY hotspot basins
    sel = out[out["hot_code"].isin(highlight_codes)].copy()
    if len(sel) == 0:
        ax.text(0.5, 0.5, "No hotspot basins in selection", ha="center", va="center")
    else:
        for code in highlight_codes:
            s = sel[sel["hot_code"] == code]
            if len(s) == 0:
                continue
            s.plot(ax=ax, color=code_color.get(code, "#444444"), linewidth=0.0)
        sel.boundary.plot(ax=ax, linewidth=0.25, color="white", alpha=0.85)

    # states boundaries last
    if states_p is not None:
        states_p.boundary.plot(ax=ax, linewidth=0.7, color="black", alpha=0.95)

    handles = [mpl.patches.Patch(facecolor=code_color[c], edgecolor="none", label=str(c)) for c in highlight_codes]
    label_map = {7:"MW+fossil+scarcity", 6:"MW+fossil", 5:"MW+scarcity", 3:"fossil+scarcity"}
    handles = [mpl.patches.Patch(facecolor=code_color[c], edgecolor="none", label=label_map[c]) for c in highlight_codes]
    ax.legend(handles=handles, loc="lower left", frameon=True, fontsize=8, title="Hotspot basins")

    ax.set_title("Hotspot basins (minimal)")
    ax.set_axis_off()
    savefig(fig, name)


In [ ]:
plot_basin_hotspots_selective(
    basins_p, basin_tbl, states_p=states_p,
    highlight_codes=(7,5,6),
    show_sites=True, gdf_sites=gdf_p,
    name="Fig_hotspots_selective_maintext"
)
plot_hotspot_overlap_counts(basin_tbl, name="ED_hotspot_overlap_counts")


In [ ]:
def basin_hotspots(
    gdf_sites, basins_p, ba_table,
    stress_col="bws_01_score",
    coal_col="share_coal",
    gas_col="share_gas",
    mw_col=COLS["mw"],
    subr_col=COLS["subregion"],
    pfaf_col=COLS["pfaf"],
    mw_q=0.75,
    fossil_thr=0.50,
    stress_hi=3.0
):
    """
    Basin hotspot typology with policy-facing cutoffs:
      - High MW: basin MW >= top mw_q percentile (computed among basins with mw_total>0)
      - High fossil: (coal + gas) >= fossil_thr  (BA-level, MW-weighted to basin)
      - High scarcity: stress >= stress_hi (basin-level Aqueduct)

    Requires:
      - gdf_sites: GeoDataFrame with mw_col, pfaf_col, subr_col (already hyperscale-filtered)
      - basins_p: GeoDataFrame with PFAF_ID and stress_col
      - ba_table: table with subregion_id plus coal_col & gas_col (or precomputed)
    """

    sites = gdf_sites.copy()
    sites[mw_col] = safe_num(sites[mw_col])
    sites = sites.dropna(subset=[mw_col]).copy()

    # ---- build BA fossil share (coal+gas) ----
    bt_cols = ["subregion_id"]
    missing = [c for c in [coal_col, gas_col] if c not in ba_table.columns]
    if missing:
        raise KeyError(
            f"ba_table missing {missing}. "
            f"Need '{coal_col}' and '{gas_col}' to define fossil = coal+gas."
        )

    bt = ba_table[bt_cols + [coal_col, gas_col]].copy()
    bt["subregion_id"] = bt["subregion_id"].astype(str).str.strip().str.upper()
    bt["fossil_cg"] = safe_num(bt[coal_col]).fillna(0) + safe_num(bt[gas_col]).fillna(0)

    # join fossil share onto sites
    sites["subregion_id"] = sites[subr_col].astype(str).str.strip().str.upper()
    sites = sites.merge(bt[["subregion_id", "fossil_cg"]], on="subregion_id", how="left")

    # normalize pfaf
    if pfaf_col not in sites.columns:
        raise KeyError(f"Sites missing {pfaf_col}. Add basin join first.")
    sites[pfaf_col] = sites[pfaf_col].astype(str).str.strip()

    # basin stress table
    b = basins_p.copy()
    if "PFAF_ID" not in b.columns:
        raise KeyError("basins_p missing PFAF_ID.")
    b["PFAF_ID"] = b["PFAF_ID"].astype(str).str.strip()
    if stress_col not in b.columns:
        raise KeyError(f"basins_p missing {stress_col}.")
    basin_stress = b[["PFAF_ID", stress_col]].drop_duplicates("PFAF_ID")

    # aggregate sites -> basins
    def mw_weighted_mean(x, w):
        x = safe_num(x)
        w = safe_num(w).fillna(0)
        m = np.isfinite(x) & np.isfinite(w) & (w > 0)
        if m.sum() == 0:
            return np.nan
        return np.average(x[m], weights=w[m])

    basin = (sites.groupby(pfaf_col, as_index=False)
                  .agg(
                      n_sites=(mw_col, "count"),
                      mw_total=(mw_col, "sum"),
                      fossil_cg_mw_weighted=("fossil_cg", lambda s: mw_weighted_mean(s, sites.loc[s.index, mw_col])),
                  ))
    basin = basin.rename(columns={pfaf_col: "PFAF_ID"})
    basin = basin.merge(basin_stress, on="PFAF_ID", how="left")

    # thresholds
    mw_base = basin.loc[basin["mw_total"] > 0, "mw_total"]
    mw_thr = mw_base.quantile(mw_q) if len(mw_base) else np.nan

    basin["hi_mw"] = basin["mw_total"] >= mw_thr
    basin["hi_fossil"] = safe_num(basin["fossil_cg_mw_weighted"]) >= fossil_thr
    basin["hi_scarcity"] = safe_num(basin[stress_col]) >= stress_hi

    # category code 0..7 (bitmask): MW=4, fossil=2, scarcity=1
    basin["hot_code"] = (
        basin["hi_mw"].astype(int) * 4 +
        basin["hi_fossil"].astype(int) * 2 +
        basin["hi_scarcity"].astype(int) * 1
    )

    labels = {
        0: "None",
        1: "Scarcity only",
        2: "Fossil only (coal+gas)",
        3: "Fossil + scarcity",
        4: "MW only",
        5: "MW + scarcity",
        6: "MW + fossil",
        7: "MW + fossil + scarcity",
    }
    basin["hotspot_type"] = basin["hot_code"].map(labels)

    meta = {
        "mw_thr": float(mw_thr) if pd.notna(mw_thr) else np.nan,
        "mw_q": mw_q,
        "fossil_thr": fossil_thr,
        "stress_thr": stress_hi,
        "fossil_def": f"{coal_col}+{gas_col}",
    }
    return basin, meta


In [ ]:
def plot_basin_hotspots(
    basins_p, basin_table, states_p=None, title=None,
    fname="Fig_hotspot_typology_basins",
    outline_hotspots=True
):
    # merge categories back to geometry
    b = basins_p.copy()
    b["PFAF_ID"] = b["PFAF_ID"].astype(str).str.strip()
    t = basin_table[["PFAF_ID", "hot_code", "hotspot_type"]].copy()
    t["PFAF_ID"] = t["PFAF_ID"].astype(str).str.strip()

    out = b.merge(t, on="PFAF_ID", how="left")
    out["hot_code"] = out["hot_code"].fillna(0).astype(int)
    out["hotspot_type"] = out["hotspot_type"].fillna("None")

    # palette (muted, print-friendly)
    palette = {
        "None": "#f7f7f7",
        "Scarcity only": "#fddbc7",
        "Fossil only (coal+gas)": "#d1e5f0",
        "Fossil + scarcity": "#92c5de",
        "MW only": "#c7e9c0",
        "MW + scarcity": "#41ab5d",
        "MW + fossil": "#74a9cf",
        "MW + fossil + scarcity": "#045a8d",
    }
    order = [
        "None",
        "Scarcity only",
        "Fossil only (coal+gas)",
        "Fossil + scarcity",
        "MW only",
        "MW + scarcity",
        "MW + fossil",
        "MW + fossil + scarcity",
    ]

    fig, ax = plt.subplots(figsize=(10.8, 6.6))
    ax.set_axis_off()

    # background (very light, minimal ink)
    if states_p is not None:
        states_p.boundary.plot(ax=ax, linewidth=0.45, color="#9e9e9e", alpha=0.9)

    # plot NONE basins as almost-white *or* skip them entirely
    none = out[out["hotspot_type"] == "None"]
    if len(none) > 0:
        none.plot(ax=ax, color=palette["None"], linewidth=0.0)

    # plot hotspot basins by category (only those != None)
    for lab in order[1:]:
        sel = out[out["hotspot_type"] == lab]
        if len(sel) == 0:
            continue
        sel.plot(ax=ax, color=palette[lab], linewidth=0.0)

    # crisp outlines only for hotspots (optional)
    if outline_hotspots:
        out[out["hot_code"] > 0].boundary.plot(ax=ax, linewidth=0.25, color="white", alpha=0.8)

    ax.set_title(title or "Hotspot typology: high hyperscale capacity × fossil-heavy grids × water-scarce basins",
                 pad=10)

    # legend (compact, journal-like)
    handles = [mpl.patches.Patch(facecolor=palette[k], edgecolor="none", label=k) for k in order]
    leg = ax.legend(
        handles=handles,
        title="Hotspot type",
        loc="lower left",
        frameon=True,
        fontsize=8.5,
        title_fontsize=9.5,
        borderpad=0.8,
        labelspacing=0.5,
        handlelength=1.6,
        handletextpad=0.6,
    )
    leg.get_frame().set_edgecolor("#cfcfcf")
    leg.get_frame().set_linewidth(0.8)
    leg.get_frame().set_alpha(1.0)

    savefig(fig, fname)


In [ ]:
def plot_hotspot_overlap_counts(basin_table, title="How hotspot criteria overlap", fname="ED_hotspot_overlap_counts"):
    order = [
        "MW + fossil + scarcity",
        "MW + fossil",
        "MW + scarcity",
        "Fossil + scarcity",
        "MW only",
        "Fossil only (coal+gas)",
        "Scarcity only",
        "None",
    ]

    counts = basin_table["hotspot_type"].value_counts().reindex(order).fillna(0)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(counts.index, counts.values)
    ax.set_title(title)
    ax.set_xlabel("Number of basins")
    ax.grid(True, axis="x", linewidth=0.3, alpha=0.35)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)

    savefig(fig, fname)


In [ ]:
basin_tbl, meta = basin_hotspots(
    gdf_sites=gdf_p,
    basins_p=basins_p,
    ba_table=ba_p2,
    stress_col="bws_01_score",
    coal_col="share_coal",
    gas_col="share_gas",
    mw_q=0.75,
    fossil_thr=0.50,
    stress_hi=3.0
)

print("Hotspot thresholds:", meta)

plot_basin_hotspots(
    basins_p=basins_p,
    basin_table=basin_tbl,
    states_p=states_p,
    title="Hotspots where hyperscale capacity intersects fossil-heavy grids (coal+gas ≥ 50%) and water-scarce basins (stress ≥ 3)",
    fname="Fig_hotspot_typology_basins_newcutoffs"
)

plot_hotspot_overlap_counts(basin_tbl, title="Overlap of hotspot criteria (MW≥75p, coal+gas≥50%, stress≥3)")


In [ ]:
def ensure_facility_water_columns(gdf_sites, ba_table):
    """
    Ensure gdf_sites has:
      - s1_m3_mid (optional; if missing, stays NaN)
      - s2_m3_mid (computed if missing and mwh_fac_A_mid exists)
      - total_m3_mid (sum if possible)

    Scope 2 compute:
      s2_m3 = (mwh_fac * 1000 kWh/MWh) * (grid_L_per_kwh L/kWh) / 1000 L/m3
            = mwh_fac * grid_L_per_kwh
    (Nice simplification!)
    """
    g = gdf_sites.copy()

    # normalize keys
    g["subregion_id"] = g[COLS["subregion"]].astype(str).str.strip().str.upper()
    bt = ba_table[["subregion_id", COLS["grid_L_per_kwh"]]].copy()
    bt["subregion_id"] = bt["subregion_id"].astype(str).str.strip().str.upper()
    bt[COLS["grid_L_per_kwh"]] = safe_num(bt[COLS["grid_L_per_kwh"]])

    g = g.merge(bt, on="subregion_id", how="left")

    # Scope 1
    if COLS["s1_m3"] not in g.columns:
        g[COLS["s1_m3"]] = np.nan
    else:
        g[COLS["s1_m3"]] = safe_num(g[COLS["s1_m3"]])

    # Scope 2
    if COLS["s2_m3"] in g.columns:
        g[COLS["s2_m3"]] = safe_num(g[COLS["s2_m3"]])
    else:
        # compute if MWh present
        mwh_col = SCEN.get("FACILITY_MWH_COLUMN", None)
        if mwh_col is not None and mwh_col in g.columns:
            g[mwh_col] = safe_num(g[mwh_col])
            # because mwh * L/kWh = m^3 (see derivation)
            g[COLS["s2_m3"]] = g[mwh_col] * g[COLS["grid_L_per_kwh"]]
        else:
            g[COLS["s2_m3"]] = np.nan

    # Total
    if COLS["tot_m3"] in g.columns:
        g[COLS["tot_m3"]] = safe_num(g[COLS["tot_m3"]])
    else:
        g[COLS["tot_m3"]] = safe_num(g[COLS["s1_m3"]]) + safe_num(g[COLS["s2_m3"]])

    return g


In [ ]:
def build_ba_water_summary(gdf_sites, ba_polys):
    """
    Returns BA polygons with:
      hyperscale_mw, n_sites, mean_bws, share_high_stress,
      s1_m3_ba, s2_m3_ba, tot_m3_ba,
      plus any BA grid attributes already present (grid_L_per_kwh, share_coal/gas,...)
    """
    # ensure subregion_id on polygons
    out = ba_polys.copy()
    if "subregion_id" not in out.columns:
        if "region_B_1" in out.columns:
            out["subregion_id"] = out["region_B_1"].astype(str).str.strip().str.upper()
        else:
            raise KeyError("BA polygons need subregion_id or region_B_1.")

    s = gdf_sites.copy()
    s["subregion_id"] = s[COLS["subregion"]].astype(str).str.strip().str.upper()
    s[COLS["mw"]] = safe_num(s[COLS["mw"]])
    if COLS["bws"] in s.columns:
        s[COLS["bws"]] = safe_num(s[COLS["bws"]])
    else:
        s[COLS["bws"]] = np.nan

    for c in [COLS["s1_m3"], COLS["s2_m3"], COLS["tot_m3"]]:
        if c in s.columns:
            s[c] = safe_num(s[c])
        else:
            s[c] = np.nan

    ba_stats = (s.groupby("subregion_id", as_index=False)
                  .agg(
                      n_sites=(COLS["mw"], "count"),
                      hyperscale_mw=(COLS["mw"], "sum"),
                      mean_bws=(COLS["bws"], "mean"),
                      share_high_stress=(COLS["bws"], lambda x: (safe_num(x) >= SCEN["HIGH_STRESS_THRESHOLD"]).mean()),
                      s1_m3_ba=(COLS["s1_m3"], "sum"),
                      s2_m3_ba=(COLS["s2_m3"], "sum"),
                      tot_m3_ba=(COLS["tot_m3"], "sum"),
                  ))

    out["subregion_id"] = out["subregion_id"].astype(str).str.strip().str.upper()
    out = out.merge(ba_stats, on="subregion_id", how="left")

    # stable columns
    for c in ["n_sites","hyperscale_mw","mean_bws","share_high_stress","s1_m3_ba","s2_m3_ba","tot_m3_ba"]:
        if c not in out.columns:
            out[c] = np.nan

    return out


In [ ]:
def fig_scope2_vs_scarcity(ba_p, gdf_p, states_p=None, fname="Fig_scope2_vs_scarcity_BA"):
    require_cols(ba_p, ["s2_m3_ba", "mean_bws", "hyperscale_mw"], "ba_p")
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    ax0, ax1, ax2 = axes

    # (a) Map: Scope 2 water by BA
    draw_base(ax0, states_layer=states_p, boundary_layer=ba_p)
    vals = safe_num(ba_p["s2_m3_ba"])
    ba_p[vals.isna()].plot(ax=ax0, color="#f0f0f0", linewidth=0.0)
    ba_p[vals.notna()].plot(ax=ax0, column="s2_m3_ba", cmap="viridis", linewidth=0.0)
    gdf_p.plot(ax=ax0, markersize=scaled_point_sizes(gdf_p[COLS["mw"]], 8, 220), color="black", alpha=0.20)
    ax0.set_title("(a) Scope 2 water consumption by BA (m³/yr) + hyperscalers")

    sm = mpl.cm.ScalarMappable(cmap=plt.get_cmap("viridis"),
                               norm=mpl.colors.Normalize(vmin=np.nanpercentile(vals, 5),
                                                         vmax=np.nanpercentile(vals, 95)))
    sm.set_array([])
    cb = fig.colorbar(sm, ax=ax0, fraction=0.035, pad=0.01)
    cb.set_label("Scope 2 water (m³/yr)")

    # (b) Scatter: scope2 vs mean site scarcity (bubble ∝ MW)
    x = safe_num(ba_p["mean_bws"])
    y = safe_num(ba_p["s2_m3_ba"])
    mw = safe_num(ba_p["hyperscale_mw"]).fillna(0)
    sizes = 30 + 280 * (np.sqrt(mw) / (np.sqrt(mw).max() + 1e-9))
    ax1.scatter(x, y, s=sizes, alpha=0.75)
    ax1.axvline(SCEN["HIGH_STRESS_THRESHOLD"], linestyle="--", linewidth=1.0)
    ax1.set_xlabel("Mean site water stress in BA (bws_01_score)")
    ax1.set_ylabel("Scope 2 water (m³/yr)")
    ax1.set_title("(b) Scope 2 water vs site scarcity (bubble ∝ hyperscale MW)")
    ax1.grid(True, linewidth=0.3, alpha=0.35)

    # (c) Bivariate: scope2 × scarcity
    tmp = ba_p.copy()
    tmp["s2_log"] = np.log10(safe_num(tmp["s2_m3_ba"]) + 1)  # stabilize extremes
    tmp2, _, _ = make_bivariate_colors(tmp, xcol="mean_bws", ycol="s2_log", n=3, missing="#eeeeee")
    draw_base(ax2, states_layer=states_p, boundary_layer=tmp2)
    tmp2.plot(ax=ax2, color=tmp2["_bivar_color"], linewidth=0.0)
    gdf_p.plot(ax=ax2, markersize=scaled_point_sizes(gdf_p[COLS["mw"]], 6, 180), color="black", alpha=0.18)
    ax2.set_title("(c) Bivariate: higher scarcity × higher Scope 2 water")

    draw_bivariate_legend(ax2, xlab="Higher stress", ylab="Higher Scope 2 water", loc=(0.03, 0.06), size=0.20)
    savefig(fig, fname)


In [ ]:
def fig_scopeX_vs_scarcity(ba_p, gdf_p, states_p=None, water_col="s2_m3_ba",
                           label="Scope 2", fname="Fig_scopeX_vs_scarcity_BA"):
    require_cols(ba_p, [water_col, "mean_bws", "hyperscale_mw"], "ba_p")
    # identical to above, just replace y-col and titles accordingly
    # (I’m keeping it short—copy fig_scope2_vs_scarcity and replace "s2_m3_ba" with water_col and label text)


In [ ]:
# 1) ensure facility water columns exist
gdf_p = ensure_facility_water_columns(gdf_p, ba_p2)

# 2) rebuild BA summary with water totals
ba_w = build_ba_water_summary(gdf_p, ba_p2)

# 3) plot scope2 vs scarcity suite
fig_scope2_vs_scarcity(ba_w, gdf_p, states_p=states_p)


In [ ]:
# --- 1) Compute facility water columns (Scope 1/2/total) ---
gdf_p = ensure_facility_water_columns(gdf_p, ba_p)

# sanity check
print("Facility water columns present?",
      COLS["s1_m3"] in gdf_p.columns,
      COLS["s2_m3"] in gdf_p.columns,
      COLS["tot_m3"] in gdf_p.columns)
print("Facility Scope2 (m3/yr) non-missing:", gdf_p[COLS["s2_m3"]].notna().sum())

# --- 2) Aggregate to BA ---
ba_p2 = build_ba_water_summary(gdf_p, ba_p)

# sanity check
print("BA water columns present?", all(c in ba_p2.columns for c in ["s1_m3_ba","s2_m3_ba","tot_m3_ba"]))
print("BA Scope2 (m3/yr) non-missing:", ba_p2["s2_m3_ba"].notna().sum())


In [ ]:
def plot_ba_scope2_vs_scarcity(ba_p2):
    require_cols(ba_p2, ["s2_m3_ba", "mean_bws"], "ba_p2")
    df = ba_p2[["s2_m3_ba", "mean_bws", "hyperscale_mw"]].copy()
    df["s2_m3_ba"] = safe_num(df["s2_m3_ba"])
    df["mean_bws"] = safe_num(df["mean_bws"])
    df["hyperscale_mw"] = safe_num(df.get("hyperscale_mw", np.nan)).fillna(0)

    m = df[["s2_m3_ba","mean_bws"]].notna().all(axis=1)
    df = df[m].copy()

    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    s = 30 + 260*(np.sqrt(df["hyperscale_mw"]) / (np.sqrt(df["hyperscale_mw"]).max() + 1e-9))
    ax.scatter(df["mean_bws"], df["s2_m3_ba"], s=s, alpha=0.75)
    ax.axvline(SCEN["HIGH_STRESS_THRESHOLD"], ls="--", lw=1.0)

    ax.set_xlabel("Mean site water stress by BA (Aqueduct bws_01_score)")
    ax.set_ylabel("Scope 2 water (m³/yr) aggregated by BA")
    ax.set_title("Scope 2 water vs water stress (bubble size ∝ hyperscale MW)")
    ax.grid(True, lw=0.3, alpha=0.4)
    savefig(fig, "ED_scope2_vs_scarcity_scatter")


In [ ]:
def map_bivariate_scarcity_x_scope2(ba_p2, gdf_p, states_p=None):
    require_cols(ba_p2, ["mean_bws", "s2_m3_ba"], "ba_p2")

    tmp = ba_p2.copy()
    tmp["mean_bws"] = safe_num(tmp["mean_bws"])
    tmp["s2_m3_ba"] = safe_num(tmp["s2_m3_ba"])

    tmp2, _, _ = make_bivariate_colors(tmp, xcol="mean_bws", ycol="s2_m3_ba", n=3)

    fig, ax = plt.subplots(figsize=(10.5, 6.5))
    draw_base(ax, states_layer=states_p, boundary_layer=None)
    tmp2.plot(ax=ax, color=tmp2["_bivar_color"], linewidth=0.0)

    # sites overlay
    s = scaled_point_sizes(gdf_p[COLS["mw"]], 8, 240)
    gdf_p.plot(ax=ax, markersize=s, color="black", alpha=0.20)

    ax.set_title("Bivariate: water stress × Scope 2 water (by BA)")
    draw_bivariate_legend(ax, xlab="Higher stress", ylab="Higher Scope 2 water", loc=(0.02, 0.06), size=0.18)
    savefig(fig, "Fig_bivar_stress_x_scope2_by_BA")


In [ ]:
def build_ba_scope2_summary(gdf_sites, ba_polys_joined, subr_col=COLS["subregion"]):
    """
    BA-level summary for Scope 2 water:
      - s2_m3_ba: sum of site Scope2 water in BA
      - hyperscale_mw: sum MW in BA
      - mean_bws: mean site water stress in BA (site-side scarcity proxy)
      - share_high_stress: share of sites with bws >= threshold
    """
    # ensure BA key on polygons
    ba = ba_polys_joined.copy()
    if "subregion_id" not in ba.columns:
        if "region_B_1" in ba.columns:
            ba["subregion_id"] = ba["region_B_1"].astype(str).str.strip().str.upper()
        else:
            raise KeyError("BA polygons must have 'subregion_id' or 'region_B_1'.")

    # ensure site has required cols
    require_cols(gdf_sites, [subr_col, COLS["mw"], COLS["s2_m3"], COLS["bws"]], "gdf_sites (for BA summary)")
    sites = gdf_sites.copy()
    sites["subregion_id"] = sites[subr_col].astype(str).str.strip().str.upper()
    sites[COLS["mw"]] = safe_num(sites[COLS["mw"]])
    sites[COLS["s2_m3"]] = safe_num(sites[COLS["s2_m3"]])
    sites[COLS["bws"]] = safe_num(sites[COLS["bws"]])

    # aggregate
    out = (sites.groupby("subregion_id", as_index=False)
                .agg(
                    hyperscale_mw=(COLS["mw"], "sum"),
                    s2_m3_ba=(COLS["s2_m3"], "sum"),
                    mean_bws=(COLS["bws"], "mean"),
                    share_high_stress=(COLS["bws"], lambda s: (safe_num(s) >= SCEN["HIGH_STRESS_THRESHOLD"]).mean())
                ))

    # merge to polygons
    ba["subregion_id"] = ba["subregion_id"].astype(str).str.strip().str.upper()
    ba2 = ba.merge(out, on="subregion_id", how="left")
    return gpd.GeoDataFrame(ba2, geometry="geometry", crs=ba.crs)


In [ ]:
def build_basin_scope1_summary(gdf_sites, basins_p, ba_table,
                              stress_col="bws_01_score",
                              pfaf_col=COLS["pfaf"],
                              subr_col=COLS["subregion"],
                              mw_col=COLS["mw"],
                              s1_col=COLS["s1_m3"],
                              coal_col="share_coal",
                              gas_col="share_gas"):
    """
    Basin-level summary for Scope 1 water:
      - s1_m3_basin: sum of site Scope1 water in basin
      - mw_total: sum MW in basin
      - coal_gas_mw_weighted: MW-weighted average (coal+gas share) from BA mix
      - scarcity: basin stress score from basins_p (Aqueduct)
    """
    # sites required
    require_cols(gdf_sites, [pfaf_col, subr_col, mw_col, s1_col], "gdf_sites (basin summary)")
    sites = gdf_sites.copy()
    sites[pfaf_col] = sites[pfaf_col].astype(str).str.strip()
    sites["subregion_id"] = sites[subr_col].astype(str).str.strip().str.upper()
    sites[mw_col] = safe_num(sites[mw_col])
    sites[s1_col] = safe_num(sites[s1_col])

    # basins required
    b = basins_p.copy()
    require_cols(b, ["PFAF_ID", "geometry", stress_col], "basins_p")
    b["PFAF_ID"] = b["PFAF_ID"].astype(str).str.strip()
    basin_stress = b[["PFAF_ID", stress_col]].drop_duplicates("PFAF_ID")

    # attach coal+gas share via BA
    bt = ba_table.copy()
    if "subregion_id" not in bt.columns:
        if "region_B_1" in bt.columns:
            bt["subregion_id"] = bt["region_B_1"].astype(str).str.strip().str.upper()
        else:
            raise KeyError("ba_table needs subregion_id or region_B_1")

    # need coal+gas shares (computed earlier from BA_egrid)
    if coal_col not in bt.columns or gas_col not in bt.columns:
        raise KeyError(f"ba_table must include '{coal_col}' and '{gas_col}' (from ensure_grid_water_intensity).")

    bt["coal_gas_share"] = safe_num(bt[coal_col]) + safe_num(bt[gas_col])
    bt = bt[["subregion_id", "coal_gas_share"]].drop_duplicates("subregion_id")

    sites = sites.merge(bt, on="subregion_id", how="left")
    sites["coal_gas_share"] = safe_num(sites["coal_gas_share"])

    def mw_weighted_mean(x, w):
        x = safe_num(x)
        w = safe_num(w).fillna(0)
        m = np.isfinite(x) & np.isfinite(w) & (w > 0)
        if m.sum() == 0:
            return np.nan
        return np.average(x[m], weights=w[m])

    basin = (sites.groupby(pfaf_col, as_index=False)
                  .agg(
                      n_sites=(mw_col, "count"),
                      mw_total=(mw_col, "sum"),
                      s1_m3_basin=(s1_col, "sum"),
                      coal_gas_mw_weighted=("coal_gas_share", lambda s: mw_weighted_mean(s, sites.loc[s.index, mw_col])),
                  ))

    basin = basin.rename(columns={pfaf_col: "PFAF_ID"})
    basin = basin.merge(basin_stress, on="PFAF_ID", how="left")

    return basin


In [ ]:
def map_ba_scope2(ba_p2, gdf_p, states_p=None, title=None):
    require_cols(ba_p2, ["s2_m3_ba", "geometry"], "ba_p2")
    fig, ax = plt.subplots(figsize=(10.5, 6.5))
    draw_base(ax, states_layer=states_p, boundary_layer=ba_p2)

    vals = safe_num(ba_p2["s2_m3_ba"])
    ba_p2[vals.isna()].plot(ax=ax, color="#e6e6e6", linewidth=0.0)
    ba_p2[vals.notna()].plot(ax=ax, column="s2_m3_ba", cmap="viridis", linewidth=0.0)

    s = scaled_point_sizes(gdf_p[COLS["mw"]], 8, 240)
    gdf_p.plot(ax=ax, markersize=s, color="black", alpha=0.20)

    ax.set_title(title or "Scope 2 water consumption by BA (m³/yr) + hyperscale sites")
    sm = mpl.cm.ScalarMappable(cmap=plt.get_cmap("viridis"),
                               norm=mpl.colors.Normalize(vmin=np.nanpercentile(vals, 5),
                                                        vmax=np.nanpercentile(vals, 95)))
    sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.01)
    cb.set_label("Scope 2 water (m³/yr)")
    savefig(fig, "Fig_BA_scope2_m3")


In [ ]:
def map_basin_scope1(basins_p, basin_table, states_p=None,
                     stress_col="bws_01_score", s1_col="s1_m3_basin",
                     min_sites=1, top_q=0.85, title=None):
    b = basins_p.copy()
    b["PFAF_ID"] = b["PFAF_ID"].astype(str).str.strip()

    t = basin_table.copy()
    t["PFAF_ID"] = t["PFAF_ID"].astype(str).str.strip()

    out = b.merge(t[["PFAF_ID", "n_sites", s1_col, stress_col]], on="PFAF_ID", how="left")
    out[s1_col] = safe_num(out[s1_col])
    out["n_sites"] = safe_num(out["n_sites"]).fillna(0)

    # show only basins that actually have sites and are high enough
    thr = out.loc[out["n_sites"] >= min_sites, s1_col].quantile(top_q)
    show = (out["n_sites"] >= min_sites) & (out[s1_col] >= thr)

    fig, ax = plt.subplots(figsize=(10.5, 6.5))
    draw_base(ax, states_layer=states_p, boundary_layer=None)

    # light background basins
    out.plot(ax=ax, color="#f2f2f2", linewidth=0.0)

    # highlight selected
    out.loc[show].plot(ax=ax, column=s1_col, cmap="Blues", linewidth=0.0)

    ax.set_title(title or f"Scope 1 water hotspots by basin (top {int((1-top_q)*100)}% shown)")
    sm = mpl.cm.ScalarMappable(cmap=plt.get_cmap("Blues"),
                               norm=mpl.colors.Normalize(vmin=np.nanpercentile(out.loc[show, s1_col], 5),
                                                        vmax=np.nanpercentile(out.loc[show, s1_col], 95)))
    sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.01)
    cb.set_label("Scope 1 water (m³/yr) in basin")
    savefig(fig, "Fig_Basin_scope1_hotspots")


Scope 1 at basin level ✅

Scope 1 is on-site cooling water → it draws from (or discharges into) the local hydrologic system. Basins are the right unit for water scarcity and permitting/utility planning. That’s exactly the point of scarcity weighting.

Scope 2 at BA level ✅

Scope 2 is electricity-related water consumption → it is governed by the generation mix and power-system region. BA is the correct accounting unit (and matches how eGRID is structured).

One sentence you can use in Methods/Discussion

“We report Scope 1 water against hydrologic basins (local scarcity context) and Scope 2 water against balancing authorities (power-system attribution), because the relevant physical and governance boundaries differ across pathways.”

In [ ]:
# ============================================================
# Compute missing Scope-2 (site-level) water: s2_m3_mid
# ============================================================

def attach_ba_mix_to_sites(gdf_sites, mix_df):
    """
    Adds BA mix columns to sites via subregion_id key.
    Requires sites[COLS["subregion"]] and mix['subregion_id'].
    """
    sites = gdf_sites.copy()
    sites[COLS["subregion"]] = sites[COLS["subregion"]].astype(str).str.strip().str.upper()

    m = mix_df.copy()
    # Ensure subregion_id exists in mix
    if "subregion_id" not in m.columns:
        if "Balancing Authority Code" in m.columns:
            m["subregion_id"] = m["Balancing Authority Code"].astype(str).str.strip().str.upper()
        else:
            raise KeyError("mix_df missing 'subregion_id' and 'Balancing Authority Code'.")

    m["subregion_id"] = m["subregion_id"].astype(str).str.strip().str.upper()

    # keep only needed cols (but include shares if present)
    keep = ["subregion_id"]
    for c in [COLS["grid_L_per_kwh"], "share_coal", "share_gas", "share_fossil"]:
        if c in m.columns:
            keep.append(c)

    sites = sites.merge(m[keep].drop_duplicates("subregion_id"), left_on=COLS["subregion"], right_on="subregion_id", how="left")
    return sites


def compute_site_scope2_m3(gdf_sites, mix_df):
    """
    Computes COLS["s2_m3"] (default 's2_m3_mid') on sites.

    Uses:
      - Facility electricity column if present: SCEN["FACILITY_MWH_COLUMN"] (MWh/yr)
      - else fallback: current_mw * 8760 * FALLBACK_UTILIZATION
    Then:
      s2_L = (MWh * 1000 kWh/MWh) * grid_L_per_kwh
      s2_m3 = s2_L / 1000
    """
    sites = gdf_sites.copy()

    # Attach BA water intensity (grid_L_per_kwh) to sites
    mix2 = ensure_grid_water_intensity(mix_df.copy())
    sites = attach_ba_mix_to_sites(sites, mix2)

    # Need grid_L_per_kwh
    if COLS["grid_L_per_kwh"] not in sites.columns:
        raise KeyError(f"After join, sites still missing '{COLS['grid_L_per_kwh']}'.")

    # Get MWh
    mwh_col = SCEN.get("FACILITY_MWH_COLUMN", None)
    if mwh_col and (mwh_col in sites.columns):
        sites["_mwh_used"] = safe_num(sites[mwh_col])
    else:
        # fallback from MW
        require_cols(sites, [COLS["mw"]], "sites (for fallback MWh)")
        util = float(SCEN.get("FALLBACK_UTILIZATION", 0.70))
        sites["_mwh_used"] = safe_num(sites[COLS["mw"]]) * 8760.0 * util

    # Compute S2 water
    kwh = sites["_mwh_used"] * 1000.0
    sites[COLS["grid_L_per_kwh"]] = safe_num(sites[COLS["grid_L_per_kwh"]])

    s2_L = kwh * sites[COLS["grid_L_per_kwh"]]
    sites[COLS["s2_m3"]] = safe_num(s2_L) / 1000.0  # L -> m3

    # Cleanup
    sites = sites.drop(columns=["_mwh_used"], errors="ignore")
    return sites


# ---- RUN THIS ON YOUR gdf_h ----
gdf_h = compute_site_scope2_m3(gdf_h, mix)

print("✅ Computed:", COLS["s2_m3"], "present?", (COLS["s2_m3"] in gdf_h.columns))
print("Non-null count:", gdf_h[COLS["s2_m3"]].notna().sum())
print("Example rows:", gdf_h[[COLS["company"], COLS["subregion"], COLS["mw"], COLS["s2_m3"]]].head(3))


In [ ]:
# ============================================================
# Compute missing Scope-1 (site-level) water: s1_m3_mid
# ============================================================

def compute_site_scope1_m3(
    gdf_sites,
    wue_mid_L_per_kwh_it=1.5,   # baseline WUE (ISO: L/kWh_IT)
    pue_mid=1.25,               # used ONLY to convert facility -> IT electricity when IT MWh not available
    it_mwh_col_candidates=("mwh_it_A_mid", "mwh_it_mid", "mwh_it"),   # preferred: IT MWh/yr columns
    fac_mwh_col=None,           # fallback: facility MWh/yr column; else derived from current_mw * 8760 * u
):
    """
    Computes COLS["s1_m3"] (default 's1_m3_mid') on sites.

    Preferred:
      IT electricity in MWh/yr (any of it_mwh_col_candidates)

    Fallback:
      Facility electricity in MWh/yr (SCEN["FACILITY_MWH_COLUMN"] if present)

    Final fallback:
      Facility MWh/yr = current_mw * 8760 * u  (u = SCEN["FALLBACK_UTILIZATION"])

    IMPORTANT (per assumptions in this project):
      - current_mw is *facility* (nameplate) power, not IT power.
      - PUE enters ONLY when converting facility electricity to IT electricity
        to use ISO WUE (which is defined per kWh_IT).

    Then:
      IT_kWh = (IT_MWh * 1000)
      s1_L   = IT_kWh * WUE_mid
      s1_m3  = s1_L / 1000
    """
    sites = gdf_sites.copy()

    # If already exists, do nothing
    if COLS["s1_m3"] in sites.columns and sites[COLS["s1_m3"]].notna().any():
        return sites

    # 1) Try IT MWh columns first
    it_col = None
    for c in it_mwh_col_candidates:
        if c in sites.columns:
            it_col = c
            break

    if it_col is not None:
        it_mwh = safe_num(sites[it_col])
    else:
        # 2) Fallback to facility MWh
        if fac_mwh_col is None:
            fac_mwh_col = SCEN.get("FACILITY_MWH_COLUMN", None)

        if fac_mwh_col and (fac_mwh_col in sites.columns):
            fac_mwh = safe_num(sites[fac_mwh_col])
        else:
            # 3) Final fallback from nameplate MW + utilization
            require_cols(sites, [COLS["mw"]], "sites (for fallback facility MWh)")
            util = float(SCEN.get("FALLBACK_UTILIZATION", 0.66))
            fac_mwh = safe_num(sites[COLS["mw"]]) * 8760.0 * util

        # Convert facility -> IT electricity for ISO WUE
        pue_mid = float(pue_mid)
        it_mwh = fac_mwh / pue_mid

    # Compute Scope-1 water from WUE (ISO: L/kWh_IT)
    it_kwh = it_mwh * 1000.0
    s1_L = it_kwh * float(wue_mid_L_per_kwh_it)
    sites[COLS["s1_m3"]] = safe_num(s1_L) / 1000.0  # L -> m3

    return sites


# ---- RUN THIS ON YOUR gdf_h ----
WUE_MID = 1.5  # L/kWh_IT (edit if you prefer 1.2, 0.8, etc.)
PUE_MID = 1.25 # used only for facility->IT conversion for ISO WUE

gdf_h = compute_site_scope1_m3(gdf_h, wue_mid_L_per_kwh_it=WUE_MID, pue_mid=PUE_MID)

print("✅ Computed:", COLS["s1_m3"], "present?", (COLS["s1_m3"] in gdf_h.columns))
print("Non-null count:", gdf_h[COLS["s1_m3"]].notna().sum())
print("Example rows:", gdf_h[[COLS["company"], COLS["mw"], COLS["s1_m3"]]].head(3))


In [ ]:
# ============================================================
# RUN: Build tables + write figures
# ============================================================

def run_scope_figures(
    gdf_h, ba_polys, mix, states48,
    gdf_basins=None,  # can be None
    do_filter_hyperscale=True
):
    # -------------------------
    # 0) Hyperscale filter (your rule)
    # -------------------------
    sites = gdf_h.copy()
    if do_filter_hyperscale:
        if "company_type" not in sites.columns:
            raise KeyError("Expected 'company_type' column for hyperscale filter.")
        sites = sites[sites["company_type"].astype(str).str.lower().str.contains("hyper", na=False)].copy()

    # -------------------------
    # 1) Ensure required numeric columns
    # -------------------------
    require_cols(sites, [COLS["mw"], COLS["subregion"], COLS["bws"]], "sites")
    sites[COLS["mw"]] = safe_num(sites[COLS["mw"]])
    sites[COLS["bws"]] = safe_num(sites[COLS["bws"]])

    # If Scope-1/2 water exists, keep them numeric
    if COLS["s1_m3"] in sites.columns:
        sites[COLS["s1_m3"]] = safe_num(sites[COLS["s1_m3"]])
    if COLS["s2_m3"] in sites.columns:
        sites[COLS["s2_m3"]] = safe_num(sites[COLS["s2_m3"]])

    # -------------------------
    # 2) Clean BA keys everywhere
    # -------------------------
    sites[COLS["subregion"]] = sites[COLS["subregion"]].astype(str).str.strip().str.upper()

    ba = ba_polys.copy()
    require_cols(ba, ["region_B_1", "geometry"], "ba_polys")
    ba["subregion_id"] = ba["region_B_1"].astype(str).str.strip().str.upper()

    # -------------------------
    # 3) Ensure mix has subregion_id + shares + grid_L_per_kwh (+share_coal/share_gas)
    # -------------------------
    mix2 = ensure_grid_water_intensity(mix.copy())
    mix2["subregion_id"] = mix2["subregion_id"].astype(str).str.strip().str.upper()

    # Join mix to BA polygons
    ba_m = ba.merge(mix2, on="subregion_id", how="left")

    # -------------------------
    # 4) CRS: project everything to LCC for maps
    # -------------------------
    sites_p = sites.to_crs(LCC)
    ba_p = ba_m.to_crs(LCC)
    states_p = states48.to_crs(LCC)

    basins_p = None
    if gdf_basins is not None:
        basins_p = gdf_basins.to_crs(LCC)

    # -------------------------
    # 5) Build BA Scope2 summary (s2 at BA level)
    # -------------------------
    # Needs site s2_m3_mid present
    if COLS["s2_m3"] not in sites_p.columns:
        raise KeyError(
            f"Sites missing '{COLS['s2_m3']}'. "
            "Compute Scope-2 water at site level first (or add the column)."
        )

    ba_p2 = build_ba_scope2_summary(sites_p, ba_p)

    # -------------------------
    # 6) Build Basin Scope1 summary (s1 at basin level)
    # -------------------------
    basin_table = None
    if basins_p is not None:
        # Need pfaf_id on sites and s1_m3_mid
        if COLS["pfaf"] not in sites_p.columns:
            raise KeyError(f"Sites missing '{COLS['pfaf']}'. Join sites to basins first.")
        if COLS["s1_m3"] not in sites_p.columns:
            raise KeyError(
                f"Sites missing '{COLS['s1_m3']}'. "
                "Compute Scope-1 water at site level first (or add the column)."
            )

        # Basin stress column in basins layer should be bws_01_score (Aqueduct)
        stress_col = "bws_01_score" if "bws_01_score" in basins_p.columns else COLS["bws"]
        basin_table = build_basin_scope1_summary(
            gdf_sites=sites_p,
            basins_p=basins_p,
            ba_table=ba_p2,
            stress_col=stress_col
        )

    # -------------------------
    # 7) FIGURES (write to OUTDIR)
    # -------------------------
    # BA Scope-2 map
    map_ba_scope2(ba_p2, sites_p, states_p=states_p,
                  title="Scope 2 water consumption by BA (m³/yr) + hyperscale sites")

    # Basin Scope-1 hotspot map (clean: only show strongest basins)
    if basins_p is not None and basin_table is not None:
        stress_col = "bws_01_score" if "bws_01_score" in basins_p.columns else COLS["bws"]
        map_basin_scope1(
            basins_p=basins_p,
            basin_table=basin_table,
            states_p=states_p,
            stress_col=stress_col,
            s1_col="s1_m3_basin",
            min_sites=1,
            top_q=0.85,
            title="Scope 1 water hotspots by basin (top-tail shown)"
        )

    print("✅ Wrote figures to:", OUTDIR)
    return sites_p, ba_p2, basin_table





In [ ]:
sites_p, ba_p2, basin_table = run_scope_figures(
    gdf_h=gdf_h,
    ba_polys=ba_polys,
    mix=mix,
    states48=states48,
    gdf_basins=gdf_basins,
    do_filter_hyperscale=True
)


In [ ]:
def map_basin_scope1(
    basins_p,
    basin_table,
    states_p=None,
    stress_col="bws_01_score",
    s1_col="s1_m3_basin",
    min_sites=1,
    top_q=0.85,
    title=None,
    fname="Fig_Basin_scope1_hotspots_clean"
):
    """
    Cleaner, more interpretable Scope 1 basin hotspot map.
    - Only top-tail basins shown in color
    - Clear state boundaries
    - Log color scale (optional)
    """

    b = basins_p.copy()
    b["PFAF_ID"] = b["PFAF_ID"].astype(str).str.strip()

    t = basin_table.copy()
    t["PFAF_ID"] = t["PFAF_ID"].astype(str).str.strip()

    out = b.merge(t[["PFAF_ID", "n_sites", s1_col]], on="PFAF_ID", how="left")
    out[s1_col] = safe_num(out[s1_col])
    out["n_sites"] = safe_num(out["n_sites"]).fillna(0)

    # Define top-tail
    eligible = out["n_sites"] >= min_sites
    thr = out.loc[eligible, s1_col].quantile(top_q)

    show = eligible & (out[s1_col] >= thr)

    # -------- Plot --------
    fig, ax = plt.subplots(figsize=(11.5, 7.0))
    ax.set_axis_off()

    # Background basins (very light gray)
    out.plot(ax=ax, color="#f5f5f5", linewidth=0.0)

    # Highlighted basins (log-scaled)
    if show.sum() > 0:
        highlighted = out.loc[show].copy()
        highlighted["log_s1"] = np.log10(highlighted[s1_col] + 1)

        vmin = highlighted["log_s1"].min()
        vmax = highlighted["log_s1"].max()

        highlighted.plot(
            ax=ax,
            column="log_s1",
            cmap="Blues",
            linewidth=0.0
        )

        # Crisp white outlines for highlighted basins
        highlighted.boundary.plot(ax=ax, linewidth=0.4, color="white", alpha=0.9)

        # Colorbar (log-scale but labeled in m3)
        sm = mpl.cm.ScalarMappable(
            cmap=plt.get_cmap("Blues"),
            norm=mpl.colors.Normalize(vmin=vmin, vmax=vmax)
        )
        sm.set_array([])
        cb = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.02)
        cb.set_label("log₁₀(Scope 1 water m³/yr)")

    # Clear state boundaries
    if states_p is not None:
        states_p.boundary.plot(
            ax=ax,
            linewidth=0.8,
            color="#333333",
            alpha=0.95
        )

    ax.set_title(
        title or f"Scope 1 water hotspots by basin (top {int((1-top_q)*100)}% of basins with sites)",
        pad=12
    )

    savefig(fig, fname)


In [ ]:
# ---- CALL IT ----
sites_p, ba_p2, basin_table = run_scope_figures(
    gdf_h=gdf_h,
    ba_polys=ba_polys,
    mix=mix,
    states48=states48,
    gdf_basins=gdf_basins,   # set to None if you didn’t load basins
    do_filter_hyperscale=True
)

In [ ]:
import matplotlib.ticker as mticker

def map_basin_scope1_hotspots(
    basins_p,
    basin_table,
    states_p=None,
    s1_col="s1_m3_basin",          # basin_table should already have this
    min_sites=1,
    top_q=0.85,                    # top-tail cut (e.g., top 15%)
    title="Scope 1 water hotspots by basin",
    subtitle=None,
    fname="Fig_Basin_scope1_hotspots_clean"
):
    """
    Hotspot definition (Scope 1):
      - Basin must contain >= min_sites hyperscaler facilities
      - Basin Scope-1 water >= the (top_q) quantile among eligible basins
    """

    b = basins_p.copy()
    b["PFAF_ID"] = b["PFAF_ID"].astype(str).str.strip()

    t = basin_table.copy()
    t["PFAF_ID"] = t["PFAF_ID"].astype(str).str.strip()
    t[s1_col] = safe_num(t[s1_col])
    t["n_sites"] = safe_num(t.get("n_sites", np.nan)).fillna(0)

    out = b.merge(t[["PFAF_ID", "n_sites", s1_col]], on="PFAF_ID", how="left")

    eligible = (out["n_sites"] >= min_sites) & out[s1_col].notna()
    if eligible.sum() == 0:
        raise ValueError("No eligible basins found (check n_sites and s1_col).")

    thr = out.loc[eligible, s1_col].quantile(top_q)
    show = eligible & (out[s1_col] >= thr)

    # ---- Plot ----
    fig, ax = plt.subplots(figsize=(11.5, 7.0))
    ax.set_axis_off()

    # Background: very light basins
    out.plot(ax=ax, color="#f3f3f3", linewidth=0.0)

    # Highlight only hotspots
    hot = out.loc[show].copy()
    if len(hot) > 0:
        # log10 scale for color
        hot["_log"] = np.log10(hot[s1_col] + 1)

        vmin, vmax = hot["_log"].min(), hot["_log"].max()
        hot.plot(ax=ax, column="_log", cmap="Blues", linewidth=0.0)
        hot.boundary.plot(ax=ax, linewidth=0.35, color="white", alpha=0.95)

        sm = mpl.cm.ScalarMappable(
            cmap=plt.get_cmap("Blues"),
            norm=mpl.colors.Normalize(vmin=vmin, vmax=vmax)
        )
        sm.set_array([])
        cb = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.02)
        cb.set_label("Scope 1 water (log₁₀ m³/yr)")

        # Make ticks human-interpretable: show approx m³/yr
        ticks = np.linspace(vmin, vmax, 4)
        cb.set_ticks(ticks)
        cb.ax.set_yticklabels([f"{10**t:,.0f}" for t in ticks])
        cb.set_label("Scope 1 water (m³/yr, log-scaled colors)")

    # State boundaries (visible, not red)
    if states_p is not None:
        states_p.boundary.plot(ax=ax, linewidth=0.9, color="#2f2f2f", alpha=0.9)

    # Title + subtitle
    ax.set_title(title, pad=12)
    if subtitle is None:
        subtitle = f"Hotspot basins: ≥{min_sites} sites AND Scope 1 ≥ {int(top_q*100)}th percentile of eligible basins"
    ax.text(0.5, 1.01, subtitle, transform=ax.transAxes, ha="center", va="bottom", fontsize=10)

    # Explicit “definition box” with the actual numeric threshold
    # Put it bottom-left like a Nature figure callout
    box_txt = (
        "Hotspot definition (Scope 1)\n"
        f"• Eligible basin: ≥ {min_sites} hyperscaler site(s)\n"
        f"• Hotspot basin: Scope 1 ≥ {int(top_q*100)}th percentile of eligible basins\n"
        f"• Cutoff used here: {thr:,.0f} m³/yr"
    )
    ax.text(
        0.02, 0.03, box_txt,
        transform=ax.transAxes,
        ha="left", va="bottom",
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.45", facecolor="white", edgecolor="#444444", linewidth=0.8, alpha=0.95)
    )

    savefig(fig, fname)
    return {"s1_threshold_m3yr": float(thr), "top_q": top_q, "min_sites": min_sites, "n_hot": int(show.sum())}


In [ ]:
def map_ba_scope2_hotspots(
    ba_p2,
    states_p=None,
    s2_col="s2_m3_ba",
    min_mw=None,          # optional: require BA has some hyperscale MW
    top_q=0.85,
    title="Scope 2 water hotspots by balancing authority",
    subtitle=None,
    fname="Fig_BA_scope2_hotspots_clean"
):
    """
    Hotspot definition (Scope 2):
      - Eligible BA: hyperscale_mw > 0 (and optionally >= min_mw)
      - Hotspot BA: Scope-2 water >= top_q quantile among eligible BAs
    """

    df = ba_p2.copy()
    df[s2_col] = safe_num(df.get(s2_col, np.nan))
    df["hyperscale_mw"] = safe_num(df.get("hyperscale_mw", np.nan)).fillna(0)

    eligible = (df["hyperscale_mw"] > 0) & df[s2_col].notna()
    if min_mw is not None:
        eligible = eligible & (df["hyperscale_mw"] >= min_mw)

    if eligible.sum() == 0:
        raise ValueError("No eligible BAs found (check hyperscale_mw and s2_col).")

    thr = df.loc[eligible, s2_col].quantile(top_q)
    show = eligible & (df[s2_col] >= thr)

    fig, ax = plt.subplots(figsize=(11.5, 7.0))
    ax.set_axis_off()

    # background BA polygons light gray
    df.plot(ax=ax, color="#f3f3f3", linewidth=0.0)

    hot = df.loc[show].copy()
    if len(hot) > 0:
        hot["_log"] = np.log10(hot[s2_col] + 1)

        vmin, vmax = hot["_log"].min(), hot["_log"].max()
        hot.plot(ax=ax, column="_log", cmap="Purples", linewidth=0.0)
        hot.boundary.plot(ax=ax, linewidth=0.35, color="white", alpha=0.95)

        sm = mpl.cm.ScalarMappable(
            cmap=plt.get_cmap("Purples"),
            norm=mpl.colors.Normalize(vmin=vmin, vmax=vmax)
        )
        sm.set_array([])
        cb = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.02)

        ticks = np.linspace(vmin, vmax, 4)
        cb.set_ticks(ticks)
        cb.ax.set_yticklabels([f"{10**t:,.0f}" for t in ticks])
        cb.set_label("Scope 2 water (m³/yr, log-scaled colors)")

    if states_p is not None:
        states_p.boundary.plot(ax=ax, linewidth=0.9, color="#2f2f2f", alpha=0.9)

    ax.set_title(title, pad=12)
    if subtitle is None:
        elig_txt = "hyperscale MW > 0" if min_mw is None else f"hyperscale MW ≥ {min_mw}"
        subtitle = f"Hotspot BAs: ({elig_txt}) AND Scope 2 ≥ {int(top_q*100)}th percentile of eligible BAs"
    ax.text(0.5, 1.01, subtitle, transform=ax.transAxes, ha="center", va="bottom", fontsize=10)

    box_txt = (
        "Hotspot definition (Scope 2)\n"
        f"• Eligible BA: hyperscale MW > 0" + ("" if min_mw is None else f" (and ≥ {min_mw} MW)") + "\n"
        f"• Hotspot BA: Scope 2 ≥ {int(top_q*100)}th percentile of eligible BAs\n"
        f"• Cutoff used here: {thr:,.0f} m³/yr"
    )
    ax.text(
        0.02, 0.03, box_txt,
        transform=ax.transAxes,
        ha="left", va="bottom",
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.45", facecolor="white", edgecolor="#444444", linewidth=0.8, alpha=0.95)
    )

    savefig(fig, fname)
    return {"s2_threshold_m3yr": float(thr), "top_q": top_q, "min_mw": min_mw, "n_hot": int(show.sum())}


In [ ]:
def run_scope_hotspot_maps(
    basins_p,
    basin_table,
    ba_p2,
    states_p,
    s1_col="s1_m3_basin",
    s2_col="s2_m3_ba",
    top_q=0.85,
    min_sites=1,
    min_mw=None
):
    meta1 = map_basin_scope1_hotspots(
        basins_p=basins_p,
        basin_table=basin_table,
        states_p=states_p,
        s1_col=s1_col,
        min_sites=min_sites,
        top_q=top_q,
        fname="Fig_Basin_scope1_hotspots_clean"
    )

    meta2 = map_ba_scope2_hotspots(
        ba_p2=ba_p2,
        states_p=states_p,
        s2_col=s2_col,
        min_mw=min_mw,
        top_q=top_q,
        fname="Fig_BA_scope2_hotspots_clean"
    )

    print("✅ Wrote:")
    print(" - Fig_Basin_scope1_hotspots_clean.(pdf/png)")
    print(" - Fig_BA_scope2_hotspots_clean.(pdf/png)")
    print("Thresholds:", meta1, meta2)
    return meta1, meta2


In [ ]:
def fig_scope_hotspots_2panel(
    basins_p, basin_table,
    ba_p2,
    states_p=None,
    s1_col="s1_m3_basin",
    s2_col="s2_m3_ba",
    top_q=0.85,
    min_sites=1,
    min_mw=None,
    fname="Fig_ScopeHotspots_2panel"
):
    # Compute thresholds (so the callouts match what’s plotted)
    tmp1 = basin_table.copy()
    tmp1[s1_col] = safe_num(tmp1[s1_col])
    tmp1["n_sites"] = safe_num(tmp1.get("n_sites", np.nan)).fillna(0)
    eligible1 = (tmp1["n_sites"] >= min_sites) & tmp1[s1_col].notna()
    thr1 = tmp1.loc[eligible1, s1_col].quantile(top_q)

    tmp2 = ba_p2.copy()
    tmp2[s2_col] = safe_num(tmp2.get(s2_col, np.nan))
    tmp2["hyperscale_mw"] = safe_num(tmp2.get("hyperscale_mw", np.nan)).fillna(0)
    eligible2 = (tmp2["hyperscale_mw"] > 0) & tmp2[s2_col].notna()
    if min_mw is not None:
        eligible2 = eligible2 & (tmp2["hyperscale_mw"] >= min_mw)
    thr2 = tmp2.loc[eligible2, s2_col].quantile(top_q)

    # Merge basin geometry
    b = basins_p.copy()
    b["PFAF_ID"] = b["PFAF_ID"].astype(str).str.strip()
    t = basin_table.copy()
    t["PFAF_ID"] = t["PFAF_ID"].astype(str).str.strip()
    t[s1_col] = safe_num(t[s1_col])
    t["n_sites"] = safe_num(t.get("n_sites", np.nan)).fillna(0)
    basin_g = b.merge(t[["PFAF_ID", "n_sites", s1_col]], on="PFAF_ID", how="left")

    # Hot flags
    basin_hot = (basin_g["n_sites"] >= min_sites) & basin_g[s1_col].notna() & (basin_g[s1_col] >= thr1)
    ba_hot = eligible2 & (tmp2[s2_col] >= thr2)

    # --- Plot ---
    fig, axes = plt.subplots(1, 2, figsize=(15.5, 6.2))
    axA, axB = axes
    for ax in axes:
        ax.set_axis_off()

    # Panel (a): basin scope 1
    basin_g.plot(ax=axA, color="#f3f3f3", linewidth=0.0)
    hotA = basin_g.loc[basin_hot].copy()
    if len(hotA) > 0:
        hotA["_log"] = np.log10(hotA[s1_col] + 1)
        hotA.plot(ax=axA, column="_log", cmap="Blues", linewidth=0.0)
        hotA.boundary.plot(ax=axA, linewidth=0.35, color="white", alpha=0.95)
    if states_p is not None:
        states_p.boundary.plot(ax=axA, linewidth=0.9, color="#2f2f2f", alpha=0.9)

    axA.set_title("(a) Scope 1 hotspots by basin", pad=10)
    axA.text(
        0.02, 0.03,
        f"Hotspot: ≥{min_sites} sites AND Scope 1 ≥ {int(top_q*100)}th pct\nCutoff: {thr1:,.0f} m³/yr",
        transform=axA.transAxes,
        ha="left", va="bottom", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="#444", linewidth=0.8, alpha=0.95)
    )

    # Panel (b): BA scope 2
    tmp2.plot(ax=axB, color="#f3f3f3", linewidth=0.0)
    hotB = tmp2.loc[ba_hot].copy()
    if len(hotB) > 0:
        hotB["_log"] = np.log10(hotB[s2_col] + 1)
        hotB.plot(ax=axB, column="_log", cmap="Purples", linewidth=0.0)
        hotB.boundary.plot(ax=axB, linewidth=0.35, color="white", alpha=0.95)
    if states_p is not None:
        states_p.boundary.plot(ax=axB, linewidth=0.9, color="#2f2f2f", alpha=0.9)

    elig_txt = "hyperscale MW > 0" if min_mw is None else f"hyperscale MW ≥ {min_mw}"
    axB.set_title("(b) Scope 2 hotspots by balancing authority", pad=10)
    axB.text(
        0.02, 0.03,
        f"Hotspot: ({elig_txt}) AND Scope 2 ≥ {int(top_q*100)}th pct\nCutoff: {thr2:,.0f} m³/yr",
        transform=axB.transAxes,
        ha="left", va="bottom", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="#444", linewidth=0.8, alpha=0.95)
    )

    savefig(fig, fname)
    print("✅ Wrote:", f"{fname}.pdf/.png")
    return {"scope1_thr": float(thr1), "scope2_thr": float(thr2)}


In [ ]:
# Example: top 15% hotspots, >=1 site per basin
meta1, meta2 = run_scope_hotspot_maps(
    basins_p=basins_p,
    basin_table=basin_table,
    ba_p2=ba_p2,
    states_p=states_p,
    s1_col="s1_m3_basin",
    s2_col="s2_m3_ba",
    top_q=0.85,
    min_sites=1,
    min_mw=None
)

# 2-panel Nature-ready composite
meta_combo = fig_scope_hotspots_2panel(
    basins_p=basins_p,
    basin_table=basin_table,
    ba_p2=ba_p2,
    states_p=states_p,
    s1_col="s1_m3_basin",
    s2_col="s2_m3_ba",
    top_q=0.85,
    min_sites=1,
    min_mw=None,
    fname="Fig_ScopeHotspots_2panel"
)
meta1, meta2


## new hotpots

Scope-1 hotspot (basin): a basin is flagged “S1-only” if the basin has on-site water consumption (sum of facility S1 in basin) ≥ 50th percentile among basins, and basin (nor touching neighbors) is not highly stressed. A basin is flagged “S1 + Scarcity” if basin S1 ≥ 50th percentile and basin OR any touching neighbor has Aqueduct baseline stress ≥ 3. Basins with stress ≥ 3 but not high S1 are “Scarcity only”.

Scope-2 hotspot (BA): a balancing authority is flagged for Scope-2 if the BA total Scope-2 water (site MWh × BA grid_L_per_kwh) is in the top quantile (default top 25%). We also mark BAs with unusually high grid water-intensity (grid_L_per_kwh in top 25%); the combined typology separates traffic driven by large load, water-intense grid, or both.

In [ ]:
def ensure_grid_water_intensity(mix_df, verbose=True):
    """
    Ensures mix_df has:
      - 'subregion_id' (BA code)
      - 'grid_L_per_kwh'
      - 'share_fossil' (coal+gas) for other plots

    Uses BA_egrid.xlsx generation percent columns.
    """
    df = mix_df.copy()

    # --- find/standardize BA code -> subregion_id ---
    if "subregion_id" not in df.columns:
        # common variants
        cand = [c for c in df.columns if c.strip().lower() in [
            "balancing authority code",
            "balancing authority code ",
            "balancing_authority_code",
            "ba code",
            "ba_code"
        ]]
        if len(cand) == 0:
            raise KeyError(
                "Cannot build subregion_id: mix table lacks 'subregion_id' and no BA code column found.\n"
                "Expected something like 'Balancing Authority Code'."
            )
        code_col = cand[0]
        df["subregion_id"] = df[code_col].astype(str).str.strip().str.upper()
        if verbose:
            print(f"[ensure_grid_water_intensity] subregion_id built from: {code_col}")

    # If already computed, just return (but still ensure share_fossil if possible)
    if COLS["grid_L_per_kwh"] in df.columns:
        if verbose:
            print("[ensure_grid_water_intensity] grid_L_per_kwh already present.")
        return df

    # --- map eGRID columns -> internal share cols ---
    colmap = {
        "BA coal generation percent (resource mix)": "share_coal",
        "BA gas generation percent (resource mix)": "share_gas",
        "BA oil generation percent (resource mix)": "share_oil",
        "BA nuclear generation percent (resource mix)": "share_nuclear",
        "BA hydro generation percent (resource mix)": "share_hydro",
        "BA wind generation percent (resource mix)": "share_wind",
        "BA solar generation percent (resource mix)": "share_solar",
        "BA biomass generation percent (resource mix)": "share_biomass",
        "BA geothermal generation percent (resource mix)": "share_geothermal",
        "BA other fossil generation percent (resource mix)": "share_other_fossil",
        "BA other unknown/ purchased fuel generation percent (resource mix)": "share_other_unknown",
    }

    # technology factors (L/kWh) – placeholders; you can swap your final factor set
    FACTORS = {
        "share_coal": 1.9,
        "share_gas": 0.7,
        "share_oil": 1.5,
        "share_nuclear": 2.5,
        "share_hydro": 15.0,
        "share_wind": 0.05,
        "share_solar": 0.10,
        "share_biomass": 1.0,
        "share_geothermal": 2.0,
        "share_other_fossil": 1.0,
        "share_other_unknown": 1.0,
    }

    found = 0
    for src, dst in colmap.items():
        if src in df.columns:
            df[dst] = safe_num(df[src]) / 100.0
            found += 1

    share_cols = [c for c in FACTORS.keys() if c in df.columns]
    if verbose:
        print(f"[ensure_grid_water_intensity] share columns found: {found} mapped, using {len(share_cols)}")

    if len(share_cols) == 0:
        raise KeyError(
            "Could not compute grid_L_per_kwh: none of the expected generation percent columns were found.\n"
            "Check your BA_egrid.xlsx sheet/columns OR update colmap to match your file."
        )

    # fill missing shares with 0 and renormalize
    df[share_cols] = df[share_cols].fillna(0)
    ssum = df[share_cols].sum(axis=1)
    df.loc[ssum > 0, share_cols] = df.loc[ssum > 0, share_cols].div(ssum[ssum > 0], axis=0)

    # compute grid_L_per_kwh
    df[COLS["grid_L_per_kwh"]] = 0.0
    for c in share_cols:
        df[COLS["grid_L_per_kwh"]] += df[c] * FACTORS[c]

    # fossil share you asked for elsewhere: coal + gas
    df["share_fossil"] = df.get("share_coal", 0) + df.get("share_gas", 0)

    if verbose:
        print("[ensure_grid_water_intensity] computed grid_L_per_kwh OK.")
        print(df[[ "subregion_id", COLS["grid_L_per_kwh"], "share_fossil" ]].head(3))

    return df


In [ ]:
mix = ensure_grid_water_intensity(mix, verbose=True)
print("mix columns include grid_L_per_kwh?", COLS["grid_L_per_kwh"] in mix.columns)


In [ ]:
print("COLS['grid_L_per_kwh'] =", COLS["grid_L_per_kwh"])
print("mix has columns:", [c for c in mix.columns if "grid" in c.lower() or "kwh" in c.lower()])
print("mix head:", mix.head(2))


In [ ]:
# ============================================================
# WATER + SCARCITY FIGURE PIPELINE (BEGIN → END)
# Runs on your machine with YOUR paths + YOUR column names.
# ============================================================

# ==============
# 0) Imports
# ==============
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1.inset_locator import inset_axes


# ==============
# 1) CONFIG (KEEP YOUR PATHS / COL NAMES)
# ==============
CONFIG = {
    # Facilities:
    "FACILITIES_GEOFILE": None,
    "FACILITIES_TABLE": "data/raw/facilities.csv",

    # BA polygons (must include region_B_1):
    "BA_POLYGONS": "data/raw/balancing_authorities_polygons/balancing_authorities_EPA.shp",

    # BA mix table (BA_egrid.xlsx):
    "BA_MIX_TABLE": "data/raw/BA_egrid.xlsx",

    # HydroBASINS polygons + Aqueduct:
    "BASINS_POLYGONS": "data/raw/hydrobasins/hybas_lake_na_lev06_v1c.shp",
    "AQUEDUCT_TABLE": "data/raw/aqueduct/Aqueduct40_baseline_monthly_y2023m07d05.csv",

    # Lower-48 states boundaries:
    "STATES48": "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_state_20m.zip",

    # Output folder
    "OUTDIR": "outputs",
}

COLS = {
    "lat": "latitude",
    "lon": "longitude",
    "mw": "current_mw",
    "company": "company_name",
    "subregion": "subregion_id",      # BA code attached to each site

    # Water (m³/yr) — if present we will use them; if not, we compute
    "s1_m3": "s1_m3_mid",
    "s2_m3": "s2_m3_mid",
    "tot_m3": "total_m3_mid",

    # Facility stress / basin id (we will compute/join if missing)
    "bws": "bws_01_score",
    "pfaf": "pfaf_id",

    # Grid attribute we will compute from BA_egrid.xlsx
    "grid_L_per_kwh": "grid_L_per_kwh",
}

SCEN = {
    "HIGH_STRESS_THRESHOLD": 3.0,
    "FACILITY_MWH_COLUMN": "mwh_fac_A_mid",   # used if present
    "FALLBACK_UTILIZATION": 0.70,             # used if no MWh available and you want a fallback
    # Scope-1 WUE mid (L/kWh_IT). You can later make this climate-stratified.
    "WUE_L_PER_KWH_IT_MID": 1.5,
    # PUE mid, if you need IT energy from facility MWh
    "PUE_MID": 1.25,
}

# Lambert Conformal Conic (Lower-48)
LCC = "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 +datum=WGS84 +units=m +no_defs"
# Lower-48 clipping box in lon/lat
L48_EXTENT = (-125, -66.5, 24, 49.5)

OUTDIR = CONFIG.get("OUTDIR", "outputs")
os.makedirs(OUTDIR, exist_ok=True)


# ==============
# 2) Plot styling
# ==============
plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 450,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

def savefig(fig, name, tight=True):
    if tight:
        fig.tight_layout()
    fig.savefig(os.path.join(OUTDIR, f"{name}.pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(OUTDIR, f"{name}.png"), bbox_inches="tight")
    plt.close(fig)


# ==============
# 3) Core helpers (robust)
# ==============
def safe_num(x):
    """Convert scalars/Series/arrays to numeric safely."""
    if isinstance(x, (pd.Series, pd.Index)):
        out = pd.to_numeric(x, errors="coerce")
        return out.replace([np.inf, -np.inf], np.nan)
    if isinstance(x, (list, tuple, np.ndarray)):
        out = pd.to_numeric(pd.Series(x), errors="coerce")
        return out.replace([np.inf, -np.inf], np.nan)
    out = pd.to_numeric(x, errors="coerce")
    if np.isinf(out):
        return np.nan
    return out

def require_cols(df, cols, name="df"):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"{name} missing required columns: {missing}")

def _read_tabular(path):
    if path.endswith(".parquet"):
        return pd.read_parquet(path)
    if path.endswith(".xlsx") or path.endswith(".xls"):
        # default: first sheet
        return pd.read_excel(path)
    return pd.read_csv(path)

def to_wgs84(gdf):
    if gdf is None:
        return None
    if gdf.crs is None:
        return gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs("EPSG:4326")

def bbox_clip_points(gdf, extent=L48_EXTENT):
    xmin, xmax, ymin, ymax = extent
    g = to_wgs84(gdf)
    g = g[(g.geometry.x >= xmin) & (g.geometry.x <= xmax) &
          (g.geometry.y >= ymin) & (g.geometry.y <= ymax)].copy()
    return g

from shapely.geometry import box as shp_box

def bbox_clip_poly(gdf, extent=L48_EXTENT):
    xmin, xmax, ymin, ymax = extent
    g = to_wgs84(gdf)
    bbox_geom = shp_box(xmin, ymin, xmax, ymax)
    try:
        return gpd.clip(g, bbox_geom)
    except Exception:
        # fallback: manual intersects filter
        return g[g.intersects(bbox_geom)].copy()


def scaled_point_sizes(mw_series, smin=8, smax=220):
    mw = safe_num(mw_series).fillna(0).clip(lower=0)
    if mw.max() <= 0:
        return np.full(len(mw), smin)
    x = np.sqrt(mw)
    x = (x - x.min()) / (x.max() - x.min() + 1e-12)
    return smin + (smax - smin) * x

def draw_base(ax, states_layer=None, boundary_layer=None,
              state_edge="#2b2b2b", state_lw=0.35,
              ba_edge="#ffffff", ba_lw=0.20):
    # BA boundaries (thin white) give the “clean” cartographic feel
    if boundary_layer is not None:
        boundary_layer.boundary.plot(ax=ax, linewidth=ba_lw, color=ba_edge, alpha=0.7)
    # State boundaries (thin dark)
    if states_layer is not None:
        states_layer.boundary.plot(ax=ax, linewidth=state_lw, color=state_edge, alpha=0.9)
    ax.set_axis_off()


# ==============
# 4) Ensure grid_L_per_kwh exists (compute from BA_egrid.xlsx)
# ==============
def ensure_grid_water_intensity(mix_df, verbose=True):
    """
    Ensures mix_df has:
      - subregion_id
      - grid_L_per_kwh
      - share_fossil = coal + gas (share)
    """
    df = mix_df.copy()

    # Build subregion_id from BA code if needed
    if "subregion_id" not in df.columns:
        cand = [c for c in df.columns if c.strip().lower() in [
            "balancing authority code",
            "balancing authority code ",
            "balancing_authority_code",
            "ba code",
            "ba_code"
        ]]
        if len(cand) == 0:
            raise KeyError(
                "Cannot build subregion_id: mix table lacks 'subregion_id' and no BA code column found.\n"
                "Expected something like 'Balancing Authority Code'."
            )
        code_col = cand[0]
        df["subregion_id"] = df[code_col].astype(str).str.strip().str.upper()
        if verbose:
            print(f"[ensure_grid_water_intensity] subregion_id built from: {code_col}")

    # Already present?
    if COLS["grid_L_per_kwh"] in df.columns:
        if verbose:
            print("[ensure_grid_water_intensity] grid_L_per_kwh already present.")
        # also compute share_fossil if possible
        return df

    colmap = {
        "BA coal generation percent (resource mix)": "share_coal",
        "BA gas generation percent (resource mix)": "share_gas",
        "BA oil generation percent (resource mix)": "share_oil",
        "BA nuclear generation percent (resource mix)": "share_nuclear",
        "BA hydro generation percent (resource mix)": "share_hydro",
        "BA wind generation percent (resource mix)": "share_wind",
        "BA solar generation percent (resource mix)": "share_solar",
        "BA biomass generation percent (resource mix)": "share_biomass",
        "BA geothermal generation percent (resource mix)": "share_geothermal",
        "BA other fossil generation percent (resource mix)": "share_other_fossil",
        "BA other unknown/ purchased fuel generation percent (resource mix)": "share_other_unknown",
    }

    # Placeholder factors (swap later with your final Macknick et al. set)
    FACTORS = {
        "share_coal": 1.9,
        "share_gas": 0.7,
        "share_oil": 1.5,
        "share_nuclear": 2.5,
        "share_hydro": 15.0,
        "share_wind": 0.05,
        "share_solar": 0.10,
        "share_biomass": 1.0,
        "share_geothermal": 2.0,
        "share_other_fossil": 1.0,
        "share_other_unknown": 1.0,
    }

    # Build share columns
    for src, dst in colmap.items():
        if src in df.columns:
            df[dst] = safe_num(df[src]) / 100.0

    share_cols = [c for c in FACTORS.keys() if c in df.columns]
    if len(share_cols) == 0:
        raise KeyError(
            "Could not compute grid_L_per_kwh: none of the expected generation percent columns were found.\n"
            "Check BA_egrid.xlsx column names OR update colmap."
        )

    # Fill missing and renormalize
    df[share_cols] = df[share_cols].fillna(0)
    ssum = df[share_cols].sum(axis=1)
    df.loc[ssum > 0, share_cols] = df.loc[ssum > 0, share_cols].div(ssum[ssum > 0], axis=0)

    df[COLS["grid_L_per_kwh"]] = 0.0
    for c in share_cols:
        df[COLS["grid_L_per_kwh"]] += df[c] * FACTORS[c]

    df["share_fossil"] = df.get("share_coal", 0) + df.get("share_gas", 0)

    if verbose:
        print("[ensure_grid_water_intensity] computed grid_L_per_kwh OK.")
        print(df[["subregion_id", COLS["grid_L_per_kwh"], "share_fossil"]].head(3))

    return df


# ==============
# 5) Load inputs (facilities, BA polys, mix, basins+Aqueduct, states)
# ==============
# ---- Facilities ----
if CONFIG["FACILITIES_GEOFILE"]:
    gdf_h = gpd.read_file(CONFIG["FACILITIES_GEOFILE"])
else:
    df = _read_tabular(CONFIG["FACILITIES_TABLE"])
    require_cols(df, [COLS["lat"], COLS["lon"]], "facilities table")
    df[COLS["lat"]] = safe_num(df[COLS["lat"]])
    df[COLS["lon"]] = safe_num(df[COLS["lon"]])
    gdf_h = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df[COLS["lon"]], df[COLS["lat"]]),
        crs="EPSG:4326"
    )

# ---- Filter hyperscale only (your rule) ----
if "company_type" in gdf_h.columns:
    gdf_h = gdf_h[gdf_h["company_type"].astype(str).str.lower().str.contains("hyper", na=False)].copy()
else:
    print("[WARN] 'company_type' not found; skipping hyperscale filter.")

# ---- BA polygons ----
ba_polys = gpd.read_file(CONFIG["BA_POLYGONS"])
require_cols(ba_polys, ["region_B_1", "geometry"], "BA polygons")
ba_polys["subregion_id"] = ba_polys["region_B_1"].astype(str).str.strip().str.upper()

# ---- BA mix ----
mix = _read_tabular(CONFIG["BA_MIX_TABLE"])
mix = ensure_grid_water_intensity(mix, verbose=True)
mix["subregion_id"] = mix["subregion_id"].astype(str).str.strip().str.upper()

# ---- Basins + Aqueduct ----
gdf_basins = gpd.read_file(CONFIG["BASINS_POLYGONS"])
require_cols(gdf_basins, ["PFAF_ID", "geometry"], "basins polygons")
gdf_basins["PFAF_ID"] = gdf_basins["PFAF_ID"].astype(str).str.strip()

aq = _read_tabular(CONFIG["AQUEDUCT_TABLE"])
pfaf_col = "PFAF_ID" if "PFAF_ID" in aq.columns else ("pfaf_id" if "pfaf_id" in aq.columns else None)
if pfaf_col is None:
    raise KeyError("Aqueduct table must have PFAF_ID or pfaf_id.")
aq[pfaf_col] = aq[pfaf_col].astype(str).str.strip()

keep = [pfaf_col] + [c for c in ["bws_01_score", "bws_01_cat"] if c in aq.columns]
gdf_basins = gdf_basins.merge(
    aq[keep].drop_duplicates(subset=[pfaf_col]),
    left_on="PFAF_ID", right_on=pfaf_col, how="left"
).drop(columns=[pfaf_col], errors="ignore")

# ---- States ----
states48 = gpd.read_file(CONFIG["STATES48"])
# Drop AK/HI/territories if present (keeps the map clean)
if "STUSPS" in states48.columns:
    states48 = states48[~states48["STUSPS"].isin(["AK", "HI", "PR", "VI", "GU", "MP", "AS"])].copy()

print("Loaded:",
      f"facilities(hyperscale)={len(gdf_h)}",
      f"BA_polys={len(ba_polys)}",
      f"basins={len(gdf_basins)}",
      f"states={len(states48)}")


# ==============
# 6) Harmonize CRS + clip to Lower-48
# ==============
# ============================================================
# Assign each facility to a BA (subregion_id) via spatial join
# ============================================================

# Ensure both are WGS84
gdf_h = to_wgs84(gdf_h)
ba_polys = to_wgs84(ba_polys)

# BA key from polygons
if "subregion_id" not in ba_polys.columns:
    if "region_B_1" in ba_polys.columns:
        ba_polys = ba_polys.copy()
        ba_polys["subregion_id"] = ba_polys["region_B_1"].astype(str).str.strip().str.upper()
    else:
        raise KeyError("BA polygons must contain 'region_B_1' (EPA BA code).")

# Do the spatial join (facility point -> BA polygon)
# predicate="within" is standard; "intersects" is a bit more forgiving for boundary cases
sites_join = gpd.sjoin(
    gdf_h,
    ba_polys[["subregion_id", "geometry"]],
    how="left",
    predicate="within"
)

# Write back to gdf_h with the exact column name you expect downstream
gdf_h = sites_join.drop(columns=["index_right"], errors="ignore")
gdf_h[COLS["subregion"]] = gdf_h["subregion_id"]

# Clean/standardize
gdf_h[COLS["subregion"]] = gdf_h[COLS["subregion"]].astype(str).str.strip().str.upper()
gdf_h.loc[gdf_h[COLS["subregion"]].isin(["", "NAN", "NONE"]), COLS["subregion"]] = np.nan

print("BA assignment coverage:",
      gdf_h[COLS["subregion"]].notna().mean().round(3),
      f"({gdf_h[COLS['subregion']].notna().sum()} / {len(gdf_h)})")

# Optional: drop sites that don't match any BA polygon (outside polygons or edge cases)
# gdf_h = gdf_h.dropna(subset=[COLS["subregion"]]).copy()

states48 = bbox_clip_poly(to_wgs84(states48))
gdf_basins = bbox_clip_poly(to_wgs84(gdf_basins))

# numeric MW
require_cols(gdf_h, [COLS["mw"]], "facilities")
gdf_h[COLS["mw"]] = safe_num(gdf_h[COLS["mw"]])
gdf_h = gdf_h.dropna(subset=[COLS["mw"]]).copy()

# clean subregion id on facilities
# If your Harvard file doesn't already have subregion_id, you must have added it earlier.
if COLS["subregion"] not in gdf_h.columns:
    raise KeyError(
        f"Facilities missing '{COLS['subregion']}'. "
        "You need the BA code per site (subregion_id). "
        "If you compute it by spatial join, do it before proceeding."
    )
gdf_h[COLS["subregion"]] = gdf_h[COLS["subregion"]].astype(str).str.strip().str.upper()

# merge mix onto BA polygons
ba_polys_m = ba_polys.merge(mix, on="subregion_id", how="left")

# Project to LCC
gdf_p = gdf_h.to_crs(LCC)
ba_p = ba_polys_m.to_crs(LCC)
states_p = states48.to_crs(LCC)
basins_p = gdf_basins.to_crs(LCC)

print("After clip:", len(gdf_h), "hyperscale facilities (Lower-48)")


# ==============
# 7) Join sites → basins (to get pfaf_id + stress at site level)
# ==============
# Spatial join: site point in basin polygon
sites = gdf_p.copy()
bas = basins_p[["PFAF_ID", "bws_01_score", "geometry"]].copy()
bas = bas.rename(columns={"PFAF_ID": COLS["pfaf"], "bws_01_score": COLS["bws"]})

sites = gpd.sjoin(
    sites, bas, how="left", predicate="within"
).drop(columns=["index_right"], errors="ignore")

# Ensure pfaf_id exists
if COLS["pfaf"] not in sites.columns:
    raise KeyError("Basin join failed: sites missing pfaf_id.")
sites[COLS["pfaf"]] = sites[COLS["pfaf"]].astype(str).str.strip()
sites[COLS["bws"]] = safe_num(sites[COLS["bws"]])

gdf_p = sites  # keep as the canonical projected sites



# ============================================================
# If facility annual MWh is missing, compute it from MW capacity
# ============================================================

def ensure_facility_mwh(sites, mw_col=COLS["mw"], mwh_col=SCEN["FACILITY_MWH_COLUMN"], util=SCEN["FALLBACK_UTILIZATION"]):
    sites = sites.copy()
    if mwh_col in sites.columns:
        # already there
        return sites

    require_cols(sites, [mw_col], "sites (for fallback MWh)")
    mw = safe_num(sites[mw_col])

    # Annual facility electricity (MWh/yr) ≈ MW * 8760 * utilization
    sites[mwh_col] = mw * 8760.0 * float(util)

    print(f"[ensure_facility_mwh] created '{mwh_col}' = {mw_col} * 8760 * {util}")
    return sites

# ---- call it ----
gdf_p = ensure_facility_mwh(gdf_p)

# ==============
# 8) Compute Scope-2 water at SITE level if missing (s2_m3_mid)
# Scope-2 uses facility electricity * grid_L_per_kwh
# ==============
def compute_scope2_site_m3(sites_p, mix_df):
    s = sites_p.copy()
    # attach grid intensity via subregion_id
    mt = mix_df[["subregion_id", COLS["grid_L_per_kwh"], "share_fossil"]].copy()
    mt["subregion_id"] = mt["subregion_id"].astype(str).str.strip().str.upper()
    s["subregion_id"] = s[COLS["subregion"]].astype(str).str.strip().str.upper()
    s = s.merge(mt, on="subregion_id", how="left")

    # choose electricity MWh column if present
    if SCEN["FACILITY_MWH_COLUMN"] in s.columns:
        mwh = safe_num(s[SCEN["FACILITY_MWH_COLUMN"]])
    else:
        # fallback: if no MWh, you must supply your own; we refuse silently guessing capacity→MWh
        raise KeyError(
            f"Sites missing '{SCEN['FACILITY_MWH_COLUMN']}' needed for Scope-2. "
            "Add facility annual electricity (MWh/yr) to the Harvard table or compute it earlier."
        )

    # compute: L = (MWh*1000 kWh/MWh) * (L/kWh) then /1000 = m3
    s2_m3 = (mwh * 1000.0) * safe_num(s[COLS["grid_L_per_kwh"]]) / 1000.0
    s[COLS["s2_m3"]] = s2_m3
    return s

if COLS["s2_m3"] not in gdf_p.columns:
    gdf_p = compute_scope2_site_m3(gdf_p, mix)


# ==============
# 9) Compute Scope-1 water at SITE level if missing (s1_m3_mid)
# Scope-1 uses IT energy * WUE (L/kWh_IT)
# ==============
def compute_scope1_site_m3(sites_p):
    s = sites_p.copy()

    # need facility MWh to estimate IT kWh
    if SCEN["FACILITY_MWH_COLUMN"] not in s.columns:
        raise KeyError(
            f"Sites missing '{SCEN['FACILITY_MWH_COLUMN']}' needed for Scope-1. "
            "Add annual facility electricity (MWh/yr) or compute it earlier."
        )

    mwh_fac = safe_num(s[SCEN["FACILITY_MWH_COLUMN"]])
    # IT kWh = facility kWh / PUE
    it_kwh = (mwh_fac * 1000.0) / float(SCEN["PUE_MID"])
    # liters = it_kwh * WUE
    liters = it_kwh * float(SCEN["WUE_L_PER_KWH_IT_MID"])
    s[COLS["s1_m3"]] = liters / 1000.0
    return s

if COLS["s1_m3"] not in gdf_p.columns:
    gdf_p = compute_scope1_site_m3(gdf_p)

# total if you want it
if COLS["tot_m3"] not in gdf_p.columns:
    gdf_p[COLS["tot_m3"]] = safe_num(gdf_p[COLS["s1_m3"]]) + safe_num(gdf_p[COLS["s2_m3"]])

print("Computed columns present:",
      COLS["s1_m3"] in gdf_p.columns,
      COLS["s2_m3"] in gdf_p.columns,
      COLS["tot_m3"] in gdf_p.columns)


# ==============
# 10) BA summary table (Scope-2 at BA level + attrs)
# ==============
def build_ba_summary_from_sites(sites_p, ba_polys_joined):
    out = ba_polys_joined.copy()
    out["subregion_id"] = out["subregion_id"].astype(str).str.strip().str.upper()

    s = sites_p.copy()
    s["subregion_id"] = s[COLS["subregion"]].astype(str).str.strip().str.upper()
    s[COLS["mw"]] = safe_num(s[COLS["mw"]])
    s[COLS["s2_m3"]] = safe_num(s[COLS["s2_m3"]])
    s[COLS["bws"]] = safe_num(s[COLS["bws"]])

    ba = (s.groupby("subregion_id", as_index=False)
            .agg(
                hyperscale_mw=(COLS["mw"], "sum"),
                n_sites=(COLS["mw"], "count"),
                s2_m3_total=(COLS["s2_m3"], "sum"),
                mean_site_bws=(COLS["bws"], "mean"),
                share_sites_high_stress=(COLS["bws"], lambda x: (safe_num(x) >= SCEN["HIGH_STRESS_THRESHOLD"]).mean())
            ))

    out = out.merge(ba, on="subregion_id", how="left")
    return gpd.GeoDataFrame(out, geometry="geometry", crs=ba_polys_joined.crs)

ba_p2 = build_ba_summary_from_sites(gdf_p, ba_p)


# ==============
# 11) Scope-1 BASIN hotspots (your updated definition)
#   - basin is “hot” if:
#       (A) basin stress >= 3 OR any neighbor basin stress >= 3
#       AND
#       (B) basin Scope-1 consumption >= 50th percentile among basins
#   - No fossil here (per your rationale)
# ==============
def basin_neighbors_table(basins_p):
    # spatial index neighbor finder (touching)
    b = basins_p[["PFAF_ID", "bws_01_score", "geometry"]].copy()
    b["PFAF_ID"] = b["PFAF_ID"].astype(str).str.strip()

    sindex = b.sindex
    neigh = {pid: set() for pid in b["PFAF_ID"].values}

    for i, geom in enumerate(b.geometry):
        pid = b.iloc[i]["PFAF_ID"]
        hits = list(sindex.intersection(geom.bounds))
        for j in hits:
            if j == i:
                continue
            if geom.touches(b.geometry.iloc[j]):
                neigh[pid].add(b.iloc[j]["PFAF_ID"])

    return neigh

def build_basin_scope1_table(sites_p, basins_p):
    s = sites_p.copy()
    s[COLS["pfaf"]] = s[COLS["pfaf"]].astype(str).str.strip()
    s[COLS["s1_m3"]] = safe_num(s[COLS["s1_m3"]])
    s[COLS["mw"]] = safe_num(s[COLS["mw"]])

    # basin totals
    basin = (s.groupby(COLS["pfaf"], as_index=False)
              .agg(
                  n_sites=(COLS["mw"], "count"),
                  mw_total=(COLS["mw"], "sum"),
                  s1_m3_total=(COLS["s1_m3"], "sum"),
              )
            ).rename(columns={COLS["pfaf"]: "PFAF_ID"})

    # attach basin stress
    b = basins_p[["PFAF_ID", "bws_01_score"]].copy()
    b["PFAF_ID"] = b["PFAF_ID"].astype(str).str.strip()
    basin = basin.merge(b, on="PFAF_ID", how="left")
    basin["bws_01_score"] = safe_num(basin["bws_01_score"])

    return basin

def define_scope1_hotspots(basin_table, basins_p, stress_hi=3.0, q=0.50):
    tbl = basin_table.copy()
    thr = tbl["s1_m3_total"].quantile(q)

    # neighbor stress
    neigh = basin_neighbors_table(basins_p)
    stress_map = dict(zip(
        basins_p["PFAF_ID"].astype(str).str.strip(),
        safe_num(basins_p["bws_01_score"])
    ))

    def neigh_high(pid):
        pid = str(pid).strip()
        ns = neigh.get(pid, set())
        vals = [stress_map.get(n, np.nan) for n in ns]
        vals = [v for v in vals if np.isfinite(v)]
        return (len(vals) > 0) and (np.nanmax(vals) >= stress_hi)

    tbl["hi_s1"] = safe_num(tbl["s1_m3_total"]) >= thr
    tbl["hi_stress_self"] = safe_num(tbl["bws_01_score"]) >= stress_hi
    tbl["hi_stress_neighbor"] = tbl["PFAF_ID"].apply(neigh_high)
    tbl["hi_stress_any"] = tbl["hi_stress_self"] | tbl["hi_stress_neighbor"]

    # hotspot type
    # 0 none
    # 1 stressed only (self or neighbor)
    # 2 high scope1 only
    # 3 high scope1 + stressed (self/neighbor)
    tbl["hot_code"] = (tbl["hi_s1"].astype(int) * 2 + tbl["hi_stress_any"].astype(int) * 1)

    labels = {
        0: "Neither (below median S1; stress<3)",
        1: "Stressed basin (self/neighbor ≥ 3) only",
        2: "High Scope-1 (≥ median) only",
        3: "Scope-1 hotspot: High Scope-1 (≥ median) + stressed (self/neighbor ≥ 3)"
    }
    tbl["hotspot_type"] = tbl["hot_code"].map(labels)

    meta = {
        "s1_quantile": q,
        "s1_threshold_m3": float(thr),
        "stress_threshold": float(stress_hi),
        "definition": "High Scope-1 if basin S1 ≥ quantile; stressed if basin stress≥thr OR any touching neighbor stress≥thr."
    }
    return tbl, meta

basin_s1 = build_basin_scope1_table(gdf_p, basins_p)
basin_s1, meta_s1 = define_scope1_hotspots(basin_s1, basins_p, stress_hi=SCEN["HIGH_STRESS_THRESHOLD"], q=0.50)

print("[Scope-1 hotspot meta]", meta_s1)


# ==============
# 12) Plot Scope-1 basin hotspot map (clean, with borders + clear legend)
# ==============
def plot_scope1_basin_hotspots(basins_p, basin_table, states_p=None, title=None, name="Fig_Basin_scope1_hotspots"):
    b = basins_p.copy()
    b["PFAF_ID"] = b["PFAF_ID"].astype(str).str.strip()
    t = basin_table[["PFAF_ID", "hot_code", "hotspot_type", "s1_m3_total"]].copy()
    out = b.merge(t, on="PFAF_ID", how="left")

    cmap = mpl.colors.ListedColormap([
        "#e6e6e6",  # 0 none
        "#fddbc7",  # 1 stressed only
        "#c7e9c0",  # 2 high scope-1 only
        "#238b45",  # 3 hotspot (high scope1 + stressed)
    ])
    norm = mpl.colors.BoundaryNorm(np.arange(-0.5, 4.5, 1), cmap.N)

    fig, ax = plt.subplots(figsize=(11.2, 6.8))
    draw_base(ax, states_layer=states_p, boundary_layer=None,
              state_edge="#2b2b2b", state_lw=0.35)

    # Draw basins, but only color where we have a code; keep others light gray
    out.plot(ax=ax, color="#f2f2f2", linewidth=0.0)
    out.plot(ax=ax, column="hot_code", cmap=cmap, norm=norm, linewidth=0.0)

    ax.set_title(title or "Scope-1 water-consumption hotspots (basin level)")

    # Legend with explicit definitions + threshold callouts
    s1_thr = meta_s1["s1_threshold_m3"]
    stress_thr = meta_s1["stress_threshold"]

    handles = [
        mpl.patches.Patch(facecolor=cmap(0), edgecolor="none",
            label=f"Neither: Scope-1 < median AND stress<{stress_thr} (self & neighbors)"),
        mpl.patches.Patch(facecolor=cmap(1), edgecolor="none",
            label=f"Stressed only: stress≥{stress_thr} in basin OR touching neighbor"),
        mpl.patches.Patch(facecolor=cmap(2), edgecolor="none",
            label=f"High Scope-1 only: basin Scope-1 ≥ median ({s1_thr:,.0f} m³/yr)"),
        mpl.patches.Patch(facecolor=cmap(3), edgecolor="none",
            label=f"Hotspot: High Scope-1 (≥ median) + stressed (self/neighbor ≥{stress_thr})"),
    ]
    leg = ax.legend(handles=handles, loc="lower left", frameon=True,
                    fontsize=8, title="Hotspot definition", title_fontsize=9)
    leg.get_frame().set_linewidth(0.6)

    # Inset: distribution of basin Scope-1 totals
    iax = inset_axes(ax, width="28%", height="28%", loc="upper right", borderpad=1.0)
    vals = safe_num(basin_table["s1_m3_total"]).dropna()
    iax.hist(vals, bins=15, edgecolor="white", linewidth=0.6)
    iax.axvline(s1_thr, linestyle="--", linewidth=1.0)
    iax.set_title("Basin Scope-1 totals", fontsize=8)
    iax.set_xlabel("m³/yr", fontsize=7)
    iax.set_ylabel("n", fontsize=7)
    iax.tick_params(axis="both", labelsize=7)

    savefig(fig, name)

plot_scope1_basin_hotspots(basins_p, basin_s1, states_p=states_p)


# ==============
# 13) Scope-2 BA hotspots (electricity water) + scarcity context
#   Your goal: “water used to produce electricity serving hyperscalers”
#   BA hotspot if:
#     - BA S2 total is high (>= 75th percentile among BAs)
#     - AND BA mean_site_bws >= 3 (scarcity exposure where facilities sit)
#     - AND (optional) fossil share (coal+gas) > 0.5 (your earlier cutoff)
# ==============
def define_scope2_ba_hotspots(ba_p2, s2_quantile=0.75, stress_hi=3.0, fossil_cut=0.50,
                             require_fossil=True):
    df = ba_p2.copy()
    df["s2_m3_total"] = safe_num(df.get("s2_m3_total", np.nan))
    df["mean_site_bws"] = safe_num(df.get("mean_site_bws", np.nan))
    df["share_fossil"] = safe_num(df.get("share_fossil", np.nan))

    s2_thr = df["s2_m3_total"].quantile(s2_quantile)

    df["hi_s2"] = df["s2_m3_total"] >= s2_thr
    df["hi_stress"] = df["mean_site_bws"] >= stress_hi
    df["hi_fossil"] = df["share_fossil"] > fossil_cut

    if require_fossil:
        # 0 none
        # 1 stress only
        # 2 s2 only
        # 3 s2+stress
        # 4 fossil only
        # 5 fossil+stress
        # 6 fossil+s2
        # 7 triple
        df["hot_code"] = (df["hi_s2"].astype(int) * 4 +
                          df["hi_fossil"].astype(int) * 2 +
                          df["hi_stress"].astype(int) * 1)
        labels = {
            0: "None",
            1: f"Stress only (mean site stress ≥{stress_hi})",
            2: f"Fossil only (coal+gas>{fossil_cut:.2f})",
            3: f"Fossil + stress",
            4: f"High Scope-2 only (≥P{int(s2_quantile*100)})",
            5: f"High Scope-2 + stress",
            6: f"High Scope-2 + fossil",
            7: f"Scope-2 hotspot: High S2 + fossil + stress",
        }
    else:
        # simpler 4-class, no fossil
        df["hot_code"] = (df["hi_s2"].astype(int) * 2 + df["hi_stress"].astype(int) * 1)
        labels = {
            0: "Neither",
            1: f"Stress only (mean site stress ≥{stress_hi})",
            2: f"High Scope-2 only (≥P{int(s2_quantile*100)})",
            3: f"Scope-2 hotspot: High Scope-2 + stress",
        }

    df["hotspot_type"] = df["hot_code"].map(labels)
    meta = {"s2_thr_m3": float(s2_thr), "s2_quantile": s2_quantile,
            "stress_thr": float(stress_hi), "fossil_cut": float(fossil_cut),
            "require_fossil": bool(require_fossil)}
    return df, meta

ba_hot, meta_s2 = define_scope2_ba_hotspots(
    ba_p2, s2_quantile=0.75, stress_hi=SCEN["HIGH_STRESS_THRESHOLD"],
    fossil_cut=0.50, require_fossil=True
)
print("[Scope-2 hotspot meta]", meta_s2)


# ==============
# 14) Plot Scope-2 BA hotspot map (clean)
# ==============
def plot_scope2_ba_hotspots(ba_hot, states_p=None, title=None, name="Fig_BA_scope2_hotspots"):
    df = ba_hot.copy()

    if meta_s2["require_fossil"]:
        cmap = mpl.colors.ListedColormap([
            "#e6e6e6",  # 0 none
            "#fddbc7",  # 1 stress only
            "#d1e5f0",  # 2 fossil only
            "#92c5de",  # 3 fossil+stress
            "#c7e9c0",  # 4 high s2 only
            "#41ab5d",  # 5 high s2 + stress
            "#74a9cf",  # 6 high s2 + fossil
            "#045a8d",  # 7 triple
        ])
        labels_order = [0,1,2,3,4,5,6,7]
    else:
        cmap = mpl.colors.ListedColormap(["#e6e6e6","#fddbc7","#c7e9c0","#238b45"])
        labels_order = [0,1,2,3]

    norm = mpl.colors.BoundaryNorm(np.arange(-0.5, max(labels_order)+1.5, 1), cmap.N)

    fig, ax = plt.subplots(figsize=(11.2, 6.8))
    draw_base(ax, states_layer=states_p, boundary_layer=df,
              state_edge="#2b2b2b", state_lw=0.35,
              ba_edge="#ffffff", ba_lw=0.20)

    df.plot(ax=ax, color="#f2f2f2", linewidth=0.0)
    df.plot(ax=ax, column="hot_code", cmap=cmap, norm=norm, linewidth=0.0)

    s2_thr = meta_s2["s2_thr_m3"]
    stress_thr = meta_s2["stress_thr"]
    fossil_cut = meta_s2["fossil_cut"]

    ax.set_title(title or "Scope-2 water-consumption hotspots (BA level)")

    handles = []
    for k in labels_order:
        lab = df["hotspot_type"].dropna()
        # use the mapping from the table if exists
        label = df.loc[df["hot_code"]==k, "hotspot_type"]
        label = label.iloc[0] if len(label) else f"code {k}"
        # add thresholds to key labels
        if "High Scope-2" in label:
            label = label + f"\n(BA S2 ≥ P{int(meta_s2['s2_quantile']*100)} = {s2_thr:,.0f} m³/yr)"
        if "Stress" in label and "≥" in label:
            label = label
        if "Fossil" in label:
            label = label

        handles.append(mpl.patches.Patch(facecolor=cmap(k), edgecolor="none", label=label))

    leg = ax.legend(handles=handles, loc="lower left", frameon=True,
                    fontsize=8, title="Hotspot definition", title_fontsize=9)
    leg.get_frame().set_linewidth(0.6)

    # Inset: BA Scope-2 distribution
    iax = inset_axes(ax, width="28%", height="28%", loc="upper right", borderpad=1.0)
    vals = safe_num(df["s2_m3_total"]).dropna()
    iax.hist(vals, bins=15, edgecolor="white", linewidth=0.6)
    iax.axvline(s2_thr, linestyle="--", linewidth=1.0)
    iax.set_title("BA Scope-2 totals", fontsize=8)
    iax.set_xlabel("m³/yr", fontsize=7)
    iax.set_ylabel("n", fontsize=7)
    iax.tick_params(axis="both", labelsize=7)

    savefig(fig, name)

plot_scope2_ba_hotspots(ba_hot, states_p=states_p)


# ==============
# 15) Optional: “old-style” bivariate nexus map (BA water intensity vs scarcity)
# ==============
def fig_ba_waterintensity_vs_scarcity(ba_p2, states_p=None, name="ED_BA_waterIntensity_vs_scarcity"):
    require_cols(ba_p2, [COLS["grid_L_per_kwh"], "mean_site_bws"], "ba_p2")
    df = ba_p2.copy()

    def quantile_bins(series, n=3):
        s = safe_num(series).dropna()
        if len(s) == 0:
            return None
        qs = np.quantile(s, np.linspace(0, 1, n+1))
        for i in range(1, len(qs)):
            if qs[i] <= qs[i-1]:
                qs[i] = qs[i-1] + 1e-12
        return qs

    def assign_bin(val, edges):
        if pd.isna(val):
            return np.nan
        for i in range(len(edges)-1):
            lo, hi = edges[i], edges[i+1]
            if i == len(edges)-2:
                if (val >= lo) and (val <= hi):
                    return i
            else:
                if (val >= lo) and (val < hi):
                    return i
        return len(edges)-2

    BIVAR_3x3 = np.array([
        ["#e8e8e8", "#b5c0da", "#6c83b5"],
        ["#b8d6be", "#90b2b3", "#567994"],
        ["#73ae80", "#5a9178", "#2a5a5b"],
    ])

    xcol = COLS["grid_L_per_kwh"]
    ycol = "mean_site_bws"
    x_edges = quantile_bins(df[xcol], 3)
    y_edges = quantile_bins(df[ycol], 3)

    colors = []
    xb = df[xcol].apply(lambda v: assign_bin(v, x_edges))
    yb = df[ycol].apply(lambda v: assign_bin(v, y_edges))
    for i, j in zip(xb, yb):
        if pd.isna(i) or pd.isna(j):
            colors.append("#d9d9d9")
        else:
            colors.append(BIVAR_3x3[int(i), int(j)])
    df["_bivar_color"] = colors

    fig, ax = plt.subplots(figsize=(11.2, 6.8))
    draw_base(ax, states_layer=states_p, boundary_layer=df,
              state_edge="#2b2b2b", state_lw=0.35,
              ba_edge="#ffffff", ba_lw=0.20)

    df.plot(ax=ax, color=df["_bivar_color"], linewidth=0.0)
    ax.set_title("Bivariate: grid water intensity × site scarcity exposure (BA level)")

    # Legend inset
    leg = inset_axes(ax, width="18%", height="18%", loc="lower left", borderpad=1.0)
    leg.set_xticks([]); leg.set_yticks([]); leg.set_frame_on(True)
    for i in range(3):
        for j in range(3):
            leg.add_patch(Rectangle((j, i), 1, 1, facecolor=BIVAR_3x3[i, j], edgecolor="white", lw=0.7))
    leg.set_xlim(0, 3); leg.set_ylim(0, 3)
    leg.text(1.5, -0.55, "Mean site scarcity ↑", ha="center", va="top", fontsize=8)
    leg.text(-0.55, 1.5, "Grid L/kWh ↑", ha="right", va="center", rotation=90, fontsize=8)

    savefig(fig, name)

fig_ba_waterintensity_vs_scarcity(ba_p2, states_p=states_p)


# ==============
# 16) DONE
# ==============
print("✅ All figures written to:", OUTDIR)
print("Created at least:")
print(" - Fig_Basin_scope1_hotspots.(pdf/png)")
print(" - Fig_BA_scope2_hotspots.(pdf/png)")
print(" - ED_BA_waterIntensity_vs_scarcity.(pdf/png)")


## Now, multiple scenarios: efficienccy (low), baseline (mid), highload (high)

In [ ]:
# ============================================================
# SCENARIOS (edit / add as needed)
# ============================================================

SCENARIOS = {
    # Baseline (your current assumptions)
    "base": dict(
        UTIL=0.66,
        PUE=1.25,
        WUE_L_PER_KWH_IT=0.8,
        HYDRO_FACTOR_L_PER_KWH=8.0,
        S2_QUANTILE=0.75,
        S1_QUANTILE=0.50,
        STRESS_HI=3.0,
        FOSSIL_CUT=0.50,
        REQUIRE_FOSSIL=True,
    ),

    # More conservative (lower util, better efficiency)
    "efficiency": dict(
        UTIL=0.55,
        PUE=1.15,
        WUE_L_PER_KWH_IT=0.2,
        HYDRO_FACTOR_L_PER_KWH=8.0,
        S2_QUANTILE=0.75,
        S1_QUANTILE=0.50,
        STRESS_HI=3.0,
        FOSSIL_CUT=0.50,
        REQUIRE_FOSSIL=True,
    ),

    # Higher load + worse WUE (stress test)
    "high_load": dict(
        UTIL=0.85,
        PUE=1.30,
        WUE_L_PER_KWH_IT=1.5,
        HYDRO_FACTOR_L_PER_KWH=8.0,
        S2_QUANTILE=0.75,
        S1_QUANTILE=0.50,
        STRESS_HI=3.0,
        FOSSIL_CUT=0.50,
        REQUIRE_FOSSIL=True,
    ),

    # Hydro sensitivity: exclude hydro from grid water intensity (set factor low/0)
    "no_hydro": dict(
        UTIL=0.66,
        PUE=1.25,
        WUE_L_PER_KWH_IT=0.8,
        HYDRO_FACTOR_L_PER_KWH=0.0,
        S2_QUANTILE=0.75,
        S1_QUANTILE=0.50,
        STRESS_HI=3.0,
        FOSSIL_CUT=0.50,
        REQUIRE_FOSSIL=True,
    ),
}


In [ ]:
def ensure_grid_water_intensity(mix_df, hydro_factor=8.0, verbose=True):
    df = mix_df.copy()

    # Build subregion_id if needed
    if "subregion_id" not in df.columns:
        cand = [c for c in df.columns if c.strip().lower() in [
            "balancing authority code", "balancing_authority_code", "ba code", "ba_code"
        ]]
        if len(cand) == 0:
            raise KeyError("Cannot build subregion_id from BA mix file.")
        code_col = cand[0]
        df["subregion_id"] = df[code_col].astype(str).str.strip().str.upper()
        if verbose:
            print(f"[ensure_grid_water_intensity] subregion_id built from: {code_col}")

    colmap = {
        "BA coal generation percent (resource mix)": "share_coal",
        "BA gas generation percent (resource mix)": "share_gas",
        "BA oil generation percent (resource mix)": "share_oil",
        "BA nuclear generation percent (resource mix)": "share_nuclear",
        "BA hydro generation percent (resource mix)": "share_hydro",
        "BA wind generation percent (resource mix)": "share_wind",
        "BA solar generation percent (resource mix)": "share_solar",
        "BA biomass generation percent (resource mix)": "share_biomass",
        "BA geothermal generation percent (resource mix)": "share_geothermal",
        "BA other fossil generation percent (resource mix)": "share_other_fossil",
        "BA other unknown/ purchased fuel generation percent (resource mix)": "share_other_unknown",
    }

    FACTORS = {
        "share_coal": 1.9,
        "share_gas": 0.7,
        "share_oil": 1.5,
        "share_nuclear": 2.5,
        "share_hydro": float(hydro_factor),  # <-- scenario knob
        "share_wind": 0.05,
        "share_solar": 0.10,
        "share_biomass": 1.0,
        "share_geothermal": 2.0,
        "share_other_fossil": 1.0,
        "share_other_unknown": 1.0,
    }

    # Build share columns
    for src, dst in colmap.items():
        if src in df.columns:
            df[dst] = safe_num(df[src]) / 100.0

    share_cols = [c for c in FACTORS.keys() if c in df.columns]
    if len(share_cols) == 0:
        raise KeyError("Could not find generation percent columns to compute grid_L_per_kwh.")

    # Fill + renormalize
    df[share_cols] = df[share_cols].fillna(0)
    ssum = df[share_cols].sum(axis=1)
    df.loc[ssum > 0, share_cols] = df.loc[ssum > 0, share_cols].div(ssum[ssum > 0], axis=0)

    # Compute
    df[COLS["grid_L_per_kwh"]] = 0.0
    for c in share_cols:
        df[COLS["grid_L_per_kwh"]] += df[c] * FACTORS[c]

    # Coal+gas share for fossil cutoff (your definition)
    df["share_fossil"] = df.get("share_coal", 0) + df.get("share_gas", 0)

    if verbose:
        print(f"[ensure_grid_water_intensity] computed grid_L_per_kwh OK (hydro factor={hydro_factor}).")
        print(df[["subregion_id", COLS["grid_L_per_kwh"], "share_fossil"]].head(3))

    return df


In [ ]:
def ensure_facility_mwh(sites, util, mw_col=COLS["mw"], mwh_col=SCEN["FACILITY_MWH_COLUMN"]):
    sites = sites.copy()
    require_cols(sites, [mw_col], "sites (ensure_facility_mwh)")
    mw = safe_num(sites[mw_col])
    sites[mwh_col] = mw * 8760.0 * float(util)
    return sites

def compute_scope2_site_m3(sites_p, mix_df):
    s = sites_p.copy()

    # Normalize BA key
    s["subregion_id"] = s[COLS["subregion"]].astype(str).str.strip().str.upper()

    mt = mix_df.copy()
    if "subregion_id" not in mt.columns:
        raise KeyError("mix_df missing subregion_id (cannot join Scope-2 grid intensity).")
    mt["subregion_id"] = mt["subregion_id"].astype(str).str.strip().str.upper()

    # Ensure the mix side has the canonical name before merging
    if COLS["grid_L_per_kwh"] not in mt.columns:
        candidates = [c for c in mt.columns if c.lower() == "grid_l_per_kwh"]
        if not candidates:
            raise KeyError(
                f"mix_df missing '{COLS['grid_L_per_kwh']}'. "
                f"Columns include: {list(mt.columns)[:30]} ..."
            )
        mt = mt.rename(columns={candidates[0]: COLS["grid_L_per_kwh"]})

    keep = ["subregion_id", COLS["grid_L_per_kwh"]]
    if "share_fossil" in mt.columns:
        keep.append("share_fossil")
    mt = mt[keep].drop_duplicates("subregion_id")

    # Merge (ALLOW suffixes)
    s = s.merge(mt, on="subregion_id", how="left", suffixes=("", "_mix"))

    # ---- normalize merged grid column ----
    # If the merge created suffixes, pick the best available and write it to COLS["grid_L_per_kwh"]
    if COLS["grid_L_per_kwh"] not in s.columns:
        # common outcomes if sites already had the col
        candidates = [
            f"{COLS['grid_L_per_kwh']}_mix",
            f"{COLS['grid_L_per_kwh']}_x",
            f"{COLS['grid_L_per_kwh']}_y",
        ]
        found = next((c for c in candidates if c in s.columns), None)
        if found is None:
            raise KeyError(
                f"After merge, could not find '{COLS['grid_L_per_kwh']}' or any suffix version. "
                f"Columns include: {list(s.columns)[:40]} ..."
            )
        s[COLS["grid_L_per_kwh"]] = s[found]

    # require MWh
    if SCEN["FACILITY_MWH_COLUMN"] not in s.columns:
        raise KeyError(f"Sites missing {SCEN['FACILITY_MWH_COLUMN']} needed for Scope-2.")

    mwh = safe_num(s[SCEN["FACILITY_MWH_COLUMN"]])
    grid = safe_num(s[COLS["grid_L_per_kwh"]])

    s[COLS["s2_m3"]] = (mwh * 1000.0) * grid / 1000.0
    return s



def compute_scope1_site_m3(sites_p, pue, wue_L_per_kwh_it):
    s = sites_p.copy()
    require_cols(s, [SCEN["FACILITY_MWH_COLUMN"]], "sites (scope1)")
    mwh_fac = safe_num(s[SCEN["FACILITY_MWH_COLUMN"]])

    it_kwh = (mwh_fac * 1000.0) / float(pue)
    liters = it_kwh * float(wue_L_per_kwh_it)
    s[COLS["s1_m3"]] = liters / 1000.0
    return s

def rebuild_all_water_columns_for_scenario(gdf_p_base, mix_base, scen_name, scen):
    mix_s = ensure_grid_water_intensity(
        mix_base,
        hydro_factor=scen["HYDRO_FACTOR_L_PER_KWH"],
        verbose=False
    )

    sites = gdf_p_base.copy()
    sites = ensure_facility_mwh(sites, util=scen["UTIL"])
    sites = compute_scope2_site_m3(sites, mix_s)  # now robust
    sites = compute_scope1_site_m3(sites, pue=scen["PUE"], wue_L_per_kwh_it=scen["WUE_L_PER_KWH_IT"])
    sites[COLS["tot_m3"]] = safe_num(sites[COLS["s1_m3"]]) + safe_num(sites[COLS["s2_m3"]])
    return sites, mix_s



In [ ]:
def plot_scope1_basin_hotspots_sidepanel(basins_p, basin_table, meta, states_p=None,
                                        title=None, name="Fig_Basin_scope1_hotspots"):
    b = basins_p.copy()
    b["PFAF_ID"] = b["PFAF_ID"].astype(str).str.strip()
    t = basin_table[["PFAF_ID", "hot_code", "s1_m3_total"]].copy()
    out = b.merge(t, on="PFAF_ID", how="left")

    cmap = mpl.colors.ListedColormap(["#e6e6e6", "#fddbc7", "#c7e9c0", "#238b45"])
    norm = mpl.colors.BoundaryNorm(np.arange(-0.5, 4.5, 1), cmap.N)

    fig = plt.figure(figsize=(12.6, 6.8))
    gs = fig.add_gridspec(1, 2, width_ratios=[3.3, 1.0], wspace=0.05)
    ax = fig.add_subplot(gs[0, 0])
    axh = fig.add_subplot(gs[0, 1])

    # map
    draw_base(ax, states_layer=states_p, boundary_layer=None, state_edge="#2b2b2b", state_lw=0.35)
    out.plot(ax=ax, color="#f2f2f2", linewidth=0.0)
    out.plot(ax=ax, column="hot_code", cmap=cmap, norm=norm, linewidth=0.0)
    ax.set_title(title or "Scope-1 water-consumption hotspots (basin level)")

    # legend (explicit)
    s1_thr = meta["s1_threshold_m3"]
    stress_thr = meta["stress_threshold"]
    handles = [
        mpl.patches.Patch(facecolor=cmap(0), edgecolor="none",
            label=f"Neither:\nS1 < P{int(meta['s1_quantile']*100)} and stress<{stress_thr} (self+neighbors)"),
        mpl.patches.Patch(facecolor=cmap(1), edgecolor="none",
            label=f"Stressed only:\nstress≥{stress_thr} in basin OR touching neighbor"),
        mpl.patches.Patch(facecolor=cmap(2), edgecolor="none",
            label=f"High Scope-1 only:\nS1 ≥ P{int(meta['s1_quantile']*100)} ({s1_thr:,.0f} m³/yr)"),
        mpl.patches.Patch(facecolor=cmap(3), edgecolor="none",
            label=f"Hotspot:\nHigh S1 + stressed (self/neighbor ≥{stress_thr})"),
    ]
    leg = ax.legend(handles=handles, loc="lower left", frameon=True,
                    fontsize=8, title="Hotspot definition", title_fontsize=9)
    leg.get_frame().set_linewidth(0.6)

    # side histogram (what it means + threshold line)
    vals = safe_num(basin_table["s1_m3_total"]).dropna()
    axh.hist(vals, bins=15, edgecolor="white", linewidth=0.6)
    axh.axvline(s1_thr, linestyle="--", linewidth=1.2)
    axh.set_title("Basin Scope-1 totals", fontsize=9)
    axh.set_xlabel("m³/yr", fontsize=8)
    axh.set_ylabel("basins", fontsize=8)
    axh.tick_params(axis="both", labelsize=8)
    axh.grid(True, axis="y", linewidth=0.3, alpha=0.35)

    savefig(fig, name)


In [ ]:
def plot_scope2_ba_hotspots_sidepanel(ba_hot, meta, states_p=None, title=None, name="Fig_BA_scope2_hotspots"):
    df = ba_hot.copy()

    cmap = mpl.colors.ListedColormap([
        "#e6e6e6", "#fddbc7", "#d1e5f0", "#92c5de",
        "#c7e9c0", "#41ab5d", "#74a9cf", "#045a8d",
    ])
    norm = mpl.colors.BoundaryNorm(np.arange(-0.5, 8.5, 1), cmap.N)

    fig = plt.figure(figsize=(12.6, 6.8))
    gs = fig.add_gridspec(1, 2, width_ratios=[3.3, 1.0], wspace=0.05)
    ax = fig.add_subplot(gs[0, 0])
    axh = fig.add_subplot(gs[0, 1])

    draw_base(ax, states_layer=states_p, boundary_layer=df,
              state_edge="#2b2b2b", state_lw=0.35,
              ba_edge="#ffffff", ba_lw=0.20)

    df.plot(ax=ax, color="#f2f2f2", linewidth=0.0)
    df.plot(ax=ax, column="hot_code", cmap=cmap, norm=norm, linewidth=0.0)
    ax.set_title(title or "Scope-2 water-consumption hotspots (BA level)")

    s2_thr = meta["s2_thr_m3"]
    stress_thr = meta["stress_thr"]
    fossil_cut = meta["fossil_cut"]
    q = int(meta["s2_quantile"] * 100)

    # explicit legend text
    label_map = {
        0: "None",
        1: f"Stress only\n(mean site stress ≥{stress_thr})",
        2: f"Fossil only\n(coal+gas > {fossil_cut:.2f})",
        3: f"Fossil + stress",
        4: f"High Scope-2 only\n(BA S2 ≥ P{q} = {s2_thr:,.0f} m³/yr)",
        5: f"High Scope-2 + stress",
        6: f"High Scope-2 + fossil",
        7: f"Hotspot: High S2 + fossil + stress",
    }
    handles = [mpl.patches.Patch(facecolor=cmap(k), edgecolor="none", label=label_map[k]) for k in range(8)]
    leg = ax.legend(handles=handles, loc="lower left", frameon=True,
                    fontsize=8, title="Hotspot definition", title_fontsize=9)
    leg.get_frame().set_linewidth(0.6)

    # side histogram (BA S2 totals)
    vals = safe_num(df["s2_m3_total"]).dropna()
    axh.hist(vals, bins=15, edgecolor="white", linewidth=0.6)
    axh.axvline(s2_thr, linestyle="--", linewidth=1.2)
    axh.set_title("BA Scope-2 totals", fontsize=9)
    axh.set_xlabel("m³/yr", fontsize=8)
    axh.set_ylabel("BAs", fontsize=8)
    axh.tick_params(axis="both", labelsize=8)
    axh.grid(True, axis="y", linewidth=0.3, alpha=0.35)

    savefig(fig, name)


In [ ]:
def run_all_scenarios(gdf_p_base, basins_p, ba_p, states_p, mix_base):
    results = {}

    for scen_name, scen in SCENARIOS.items():
        print(f"\n=== Scenario: {scen_name} ===")

        # recompute water columns + scenario mix
        sites_s, mix_s = rebuild_all_water_columns_for_scenario(gdf_p_base, mix_base, scen_name, scen)

        # rebuild BA summary
        ba_s = build_ba_summary_from_sites(sites_s, ba_p)

        # rebuild basin table + hotspots
        basin_tbl = build_basin_scope1_table(sites_s, basins_p)
        basin_hot, meta_s1 = define_scope1_hotspots(basin_tbl, basins_p, stress_hi=scen["STRESS_HI"], q=scen["S1_QUANTILE"])

        # BA scope2 hotspots
        ba_hot, meta_s2 = define_scope2_ba_hotspots(
            ba_s,
            s2_quantile=scen["S2_QUANTILE"],
            stress_hi=scen["STRESS_HI"],
            fossil_cut=scen["FOSSIL_CUT"],
            require_fossil=scen["REQUIRE_FOSSIL"]
        )

        # ---- plots (side-panel versions) ----
        plot_scope1_basin_hotspots_sidepanel(
            basins_p, basin_hot, meta_s1,
            states_p=states_p,
            title=f"Scope-1 hotspots (basins) — scenario: {scen_name}",
            name=f"Fig_Basin_scope1_hotspots__{scen_name}"
        )

        plot_scope2_ba_hotspots_sidepanel(
            ba_hot, meta_s2,
            states_p=states_p,
            title=f"Scope-2 hotspots (BA) — scenario: {scen_name}",
            name=f"Fig_BA_scope2_hotspots__{scen_name}"
        )

        # optional bivariate map still works
        fig_ba_waterintensity_vs_scarcity(ba_s, states_p=states_p, name=f"ED_BA_waterIntensity_vs_scarcity__{scen_name}")

        results[scen_name] = dict(
            sites=sites_s,
            mix=mix_s,
            ba=ba_s,
            basin_hot=basin_hot,
            meta_s1=meta_s1,
            ba_hot=ba_hot,
            meta_s2=meta_s2
        )

    print("\n✅ Scenario figures written to:", OUTDIR)
    return results


#mix_raw = mix.copy()
# (optional) if mix was already merged with other stuff earlier, reload from file instead:
# mix_raw = _read_tabular(CONFIG["BA_MIX_TABLE"])
# before run_all_scenarios(...)
gdf_p_base_for_scen = gdf_p.drop(columns=[COLS["grid_L_per_kwh"]], errors="ignore").copy()
scenario_results = run_all_scenarios(gdf_p_base_for_scen, basins_p, ba_p, states_p, mix)

# ---- CALL IT ----
scenario_results = run_all_scenarios(gdf_p, basins_p, ba_p, states_p, mix)


In [ ]:

# ============================================================
# Quick summaries (liters) by scenario, BA, and state
# Assumption: current_mw is nameplate facility power; UTIL scales annual MWh.
# PUE is used ONLY for Scope 1 (ISO WUE per kWh_IT).
# ============================================================

def _pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def add_liters_columns(df, s1_m3_col=COLS["s1_m3"], s2_m3_col=COLS["s2_m3"], tot_m3_col=COLS["tot_m3"]):
    out = df.copy()
    # Ensure total exists
    if tot_m3_col not in out.columns:
        out[tot_m3_col] = safe_num(out.get(s1_m3_col, 0)) + safe_num(out.get(s2_m3_col, 0))
    out["s1_L"]  = safe_num(out[s1_m3_col])  * 1000.0
    out["s2_L"]  = safe_num(out[s2_m3_col])  * 1000.0
    out["tot_L"] = safe_num(out[tot_m3_col]) * 1000.0
    return out

def summarize(df_sites, label="scenario"):
    df = add_liters_columns(df_sites)

    mw_col = COLS["mw"] if COLS["mw"] in df.columns else _pick_col(df, ["current_mw","mw","MW"])
    ba_col = _pick_col(df, [COLS["subregion"], "subregion_id", "ba", "BA", "Balancing Authority", "Balancing_Authority"])
    state_col = _pick_col(df, ["STUSPS", "state", "State", "state_abbr", "STATE", "abbr"])

    total_L = df["tot_L"].sum()
    total_mw = safe_num(df[mw_col]).sum() if mw_col else np.nan
    L_per_MW = total_L / total_mw if (mw_col and total_mw>0) else np.nan

    print(f"\n===== {label} =====")
    print(f"Total water (L/yr): {total_L:,.0f}")
    print(f"Total water (m^3/yr): {total_L/1000:,.0f}")
    if mw_col:
        print(f"Total hyperscale capacity (MW): {total_mw:,.1f}")
        print(f"Liters per MW of nameplate capacity (L/MW/yr): {L_per_MW:,.0f}")

    if ba_col:
        ba = (df.groupby(ba_col, as_index=False)
                .agg(
                    n_sites=(mw_col, "count") if mw_col else ("tot_L","count"),
                    hyperscale_mw=(mw_col,"sum") if mw_col else ("tot_L","sum"),
                    tot_L=("tot_L","sum"),
                    s1_L=("s1_L","sum"),
                    s2_L=("s2_L","sum"),
                )
             )
        ba["L_per_MW"] = ba["tot_L"] / ba["hyperscale_mw"]
        ba = ba.sort_values("tot_L", ascending=False)
        print("\nTop 10 balancing authorities by total liters:")
        display(ba.head(10))

    if state_col:
        st = (df.groupby(state_col, as_index=False)
                .agg(
                    n_sites=(mw_col, "count") if mw_col else ("tot_L","count"),
                    hyperscale_mw=(mw_col,"sum") if mw_col else ("tot_L","sum"),
                    tot_L=("tot_L","sum"),
                    s1_L=("s1_L","sum"),
                    s2_L=("s2_L","sum"),
                )
             )
        st["L_per_MW"] = st["tot_L"] / st["hyperscale_mw"]
        st = st.sort_values("tot_L", ascending=False)
        print("\nTop 10 states by total liters:")
        display(st.head(10))

    return df

# scenario_results is created in the prior cell
for scen_name, df_sites in scenario_results.items():
    _ = summarize(df_sites, label=scen_name)


# create df for tables

In [ ]:
ba_p2.to_csv("table_BA_water_summary_updated.csv", index=False)


In [ ]:
ba_p2

In [ ]:
basin_s1

In [ ]:
gdf_p

In [ ]:
# inserting state too 
import geopandas as gpd
import pandas as pd

# Ensure GeoDataFrame + CRS
gdf = gdf_p.copy()
if not isinstance(gdf, gpd.GeoDataFrame):
    gdf = gpd.GeoDataFrame(gdf, geometry="geometry")
if gdf.crs is None:
    gdf = gdf.set_crs("EPSG:4326")
else:
    gdf = gdf.to_crs("EPSG:4326")

# Census cartographic boundary states (smallish, fast)
states_url = "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_state_20m.zip"
states = gpd.read_file(states_url).to_crs("EPSG:4326")

# Keep only the 50 states + DC (drop territories)
exclude = {"AS","GU","MP","PR","VI"}  # territories
states = states[~states["STUSPS"].isin(exclude)].copy()

# Spatial join: assign state to each facility point
gdf = gpd.sjoin(
    gdf,
    states[["STUSPS", "NAME", "geometry"]],
    how="left",
    predicate="within"
).drop(columns=["index_right"], errors="ignore")

# Add clean columns
gdf["state_abbrev"] = gdf["STUSPS"]
gdf["state_name"]   = gdf["NAME"]

# Replace gdf_p so downstream code sees the new columns
gdf_p = gdf

print("State coverage:",
      gdf_p["state_abbrev"].notna().mean().round(3),
      "| missing:", gdf_p["state_abbrev"].isna().sum())


In [ ]:
gdf_p.current_mw.sum()

In [ ]:
import pandas as pd
import numpy as np

# ----------------------------
# Settings
# ----------------------------
OUTDIR = "."  # change if you have a notebook OUTDIR already (e.g., "outputs")
UTIL_MID = 0.66  # document this in supplement
PUE_MID  = 1.25  # document this in supplement (used only for ISO WUE -> Scope 1)
WUE_MID  = 1.5   # L/kWh_IT (your mid scenario for Scope 1)

# ----------------------------
# Helper: safe numeric
# ----------------------------
def to_num(s):
    return pd.to_numeric(s, errors="coerce")

df = gdf_p.copy()

# Standardize BA key (subregion_id)
df["subregion_id"] = df["subregion_id"].astype(str).str.strip().str.upper()

# Ensure numerics
for c in ["current_mw","mwh_fac_A_mid","grid_L_per_kwh","share_fossil","s1_m3_mid","s2_m3_mid","total_m3_mid","bws_01_score"]:
    if c in df.columns:
        df[c] = to_num(df[c])

# ----------------------------
# 1) BA-level unified summary
# ----------------------------
ba = (
    df.groupby("subregion_id", dropna=False)
      .agg(
          n_sites=("id", "nunique"),
          mw_total=("current_mw", "sum"),
          mwh_fac_total=("mwh_fac_A_mid", "sum"),
          grid_L_per_kwh=("grid_L_per_kwh", "mean"),
          share_fossil=("share_fossil", "mean"),
          mean_bws=("bws_01_score", "mean"),
          s1_m3=("s1_m3_mid", "sum"),
          s2_m3=("s2_m3_mid", "sum"),
          total_m3=("total_m3_mid", "sum"),
      )
      .reset_index()
)

# Convert to liters
ba["s1_L"]    = ba["s1_m3"] * 1000.0
ba["s2_L"]    = ba["s2_m3"] * 1000.0
ba["total_L"] = ba["total_m3"] * 1000.0

# Liters per MW (guard divide-by-zero)
ba["s1_L_per_MW"]    = ba["s1_L"]    / ba["mw_total"].replace({0: np.nan})
ba["s2_L_per_MW"]    = ba["s2_L"]    / ba["mw_total"].replace({0: np.nan})
ba["total_L_per_MW"] = ba["total_L"] / ba["mw_total"].replace({0: np.nan})

# Optional: also compute a “facility electricity per MW” diagnostic
ba["mwh_per_MW"] = ba["mwh_fac_total"] / ba["mw_total"].replace({0: np.nan})

# Clean ordering + rounding for table-readiness (keep raw too if you prefer)
ba_out = ba.sort_values("total_L", ascending=False).copy()

# Helpful rank columns (for a “Top BAs” panel)
ba_out["rank_total_L"] = ba_out["total_L"].rank(method="min", ascending=False).astype(int)
ba_out["rank_s2_L"]    = ba_out["s2_L"].rank(method="min", ascending=False).astype(int)
ba_out["rank_s1_L"]    = ba_out["s1_L"].rank(method="min", ascending=False).astype(int)

# Export
out_ba_csv = f"{OUTDIR}/table_BA_scope1_scope2_total_unified_mid.csv"
ba_out.to_csv(out_ba_csv, index=False)
print("Wrote:", out_ba_csv, "| rows:", len(ba_out))

# ----------------------------
# 2) National one-row totals (for “example numeric outputs” panel)
# ----------------------------
nat = pd.DataFrame([{
    "scenario": "mid",
    "utilization_u": UTIL_MID,
    "PUE": PUE_MID,
    "WUE_L_per_kWh_IT": WUE_MID,
    "n_sites": df["id"].nunique(),
    "mw_total": df["current_mw"].sum(),
    "mwh_fac_total": df["mwh_fac_A_mid"].sum(),
    "s1_m3_total": df["s1_m3_mid"].sum(),
    "s2_m3_total": df["s2_m3_mid"].sum(),
    "total_m3_total": df["total_m3_mid"].sum(),
    "s1_L_total": df["s1_m3_mid"].sum() * 1000.0,
    "s2_L_total": df["s2_m3_mid"].sum() * 1000.0,
    "total_L_total": df["total_m3_mid"].sum() * 1000.0,
    "total_L_per_MW": (df["total_m3_mid"].sum() * 1000.0) / (df["current_mw"].sum() if df["current_mw"].sum() else np.nan),
}])

out_nat_csv = f"{OUTDIR}/table_national_water_totals_mid.csv"
nat.to_csv(out_nat_csv, index=False)
print("Wrote:", out_nat_csv)

# ----------------------------
# 3) OPTIONAL state summary (only if you have a state column)
# ----------------------------
state_col_candidates = ["state", "State", "state_name", "state_abbrev"]
state_col = next((c for c in state_col_candidates if c in df.columns), None)

if state_col:
    st = (
        df.groupby(state_col, dropna=False)
          .agg(
              n_sites=("id", "nunique"),
              mw_total=("current_mw", "sum"),
              s1_m3=("s1_m3_mid", "sum"),
              s2_m3=("s2_m3_mid", "sum"),
              total_m3=("total_m3_mid", "sum"),
          )
          .reset_index()
          .assign(
              s1_L=lambda d: d["s1_m3"] * 1000.0,
              s2_L=lambda d: d["s2_m3"] * 1000.0,
              total_L=lambda d: d["total_m3"] * 1000.0,
              total_L_per_MW=lambda d: d["total_L"] / d["mw_total"].replace({0: np.nan}),
          )
          .sort_values("total_L", ascending=False)
    )
    out_state_csv = f"{OUTDIR}/table_state_scope1_scope2_total_unified_mid.csv"
    st.to_csv(out_state_csv, index=False)
    print("Wrote:", out_state_csv, "| rows:", len(st))
else:
    print("No state column found; skipped state-level export.")


In [ ]:
states48

In [ ]:
sites[sites.stage=='{Operational}'].current_mw.sum()

In [ ]:
sites[sites.subregion_id =='PJM'].current_mw.sum()*0.66

In [ ]:
df

In [ ]:
df.groupby('state').sum()['current_mw']/1.1

# final figures

In [ ]:
import inspect
import os
import matplotlib.pyplot as plt

def _call_with_accepted_kwargs(fn, **kwargs):
    """
    Call fn(**kwargs) but only with the kwargs that fn actually accepts.
    Also supports functions that take **kwargs.
    """
    sig = inspect.signature(fn)
    params = sig.parameters
    accepts_var_kw = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in params.values())
    if accepts_var_kw:
        return fn(**kwargs)
    accepted = {k: v for k, v in kwargs.items() if k in params}
    return fn(**accepted)

def fig3_scope2_hotspot_typology(ba_gdf, sites_gdf, outdir, states_gdf=None, tag="base"):
    """
    Try to use your existing plot_scope2_ba_hotspots() (whatever its signature),
    else fallback to a simple Scope-2 by-BA choropleth.
    """
    os.makedirs(outdir, exist_ok=True)

    if "plot_scope2_ba_hotspots" in globals():
        fn = os.path.join(outdir, f"MAIN_Fig3_Scope2_hotspotTypology__{tag}.pdf")

        # Many plotting functions either:
        # - save internally, or
        # - return fig/ax, or
        # - take an outpath/filename parameter.
        # We pass multiple common names; the wrapper will keep only accepted ones.
        res = _call_with_accepted_kwargs(
            globals()["plot_scope2_ba_hotspots"],
            ba_gdf=ba_gdf,
            ba=ba_gdf,
            bas=ba_gdf,
            ba_p=ba_gdf,
            ba_p2=ba_gdf,
            sites_gdf=sites_gdf,
            sites=sites_gdf,
            sites_p=sites_gdf,
            states_gdf=states_gdf,
            states=states_gdf,
            states_p=states_gdf,
            outpath=fn,
            outfile=fn,
            savepath=fn,
            path=fn,
            fname=fn,
            title="Scope 2 BA hotspot typology (base)"
        )

        # If the function didn't save, but returned a fig, save it here.
        if isinstance(res, plt.Figure):
            res.savefig(fn, dpi=300, bbox_inches="tight")
            plt.close(res)
            return fn

        # If it returned (fig, ax)
        if isinstance(res, (tuple, list)) and len(res) >= 1 and isinstance(res[0], plt.Figure):
            res[0].savefig(fn, dpi=300, bbox_inches="tight")
            plt.close(res[0])
            return fn

        # If it saved internally, just return the intended filename
        return fn

    # ---- fallback: simple choropleth of Scope-2 totals ----
    def _pick_col(df, candidates):
        for c in candidates:
            if c in df.columns:
                return c
        raise KeyError(f"None of these columns found: {candidates}")

    s2_col = _pick_col(ba_gdf, ["scope2_m3_ba", "s2_m3_ba", "scope2_m3", "W2_m3", "W2"])
    mw_col = None
    for c in ["current_mw", "P_mw", "mw", "MW"]:
        if c in sites_gdf.columns:
            mw_col = c
            break

    def _autosize_points(mw_series, min_s=6, max_s=80):
        import numpy as np
        x = mw_series.astype(float).to_numpy()
        if np.nanmax(x) == np.nanmin(x):
            return np.full_like(x, (min_s+max_s)/2, dtype=float)
        z = np.sqrt(np.clip(x, 0, None))
        z = (z - z.min()) / (z.max() - z.min())
        return min_s + z * (max_s - min_s)

    fig, ax = plt.subplots(figsize=(12, 8))
    if states_gdf is not None:
        states_gdf.boundary.plot(ax=ax, linewidth=0.6)

    ba_gdf.plot(ax=ax, column=s2_col, legend=True)

    if mw_col is not None:
        sizes = _autosize_points(sites_gdf[mw_col])
        sites_gdf.plot(ax=ax, markersize=sizes, alpha=0.85)
    else:
        sites_gdf.plot(ax=ax, markersize=18, alpha=0.85)

    ax.set_axis_off()
    ax.set_title("Scope 2 water consumption by BA (m³/yr) + hyperscale sites")

    fn = os.path.join(outdir, f"MAIN_Fig3_Scope2_byBA_sites__{tag}.pdf")
    fig.tight_layout()
    fig.savefig(fn, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return fn


In [ ]:
# ============================================================
# MAIN PAPER FIGURES (4 candidates)
# Assumes you've already run your pipeline and have:
#   - sites_p : GeoDataFrame of facility points (from run_scope_figures or equivalent)
#   - ba_p2   : GeoDataFrame of BA polygons with grid + scope2 summaries
#   - basins_p: GeoDataFrame of HydroBASINS with Aqueduct stress attached
#   - states_p: (optional) GeoDataFrame of US states for basemap context
# And you already have OUTDIR set (as in your notebook).
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt

# ---------- helpers ----------
def _pick_col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"None of these columns found: {candidates}")
    return None

def _ensure_outdir(outdir):
    os.makedirs(outdir, exist_ok=True)
    return outdir

def _autosize_points(mw_series, min_s=6, max_s=80):
    x = mw_series.astype(float).to_numpy()
    if np.nanmax(x) == np.nanmin(x):
        return np.full_like(x, (min_s+max_s)/2, dtype=float)
    # scale by sqrt for less extreme bubbles
    z = np.sqrt(np.clip(x, 0, None))
    z = (z - z.min()) / (z.max() - z.min())
    return min_s + z * (max_s - min_s)

# ---------- FIG 1: opener map (BA grid water intensity + facility points) ----------
def fig1_ba_grid_intensity_with_sites(ba_gdf, sites_gdf, outdir, states_gdf=None, tag="base"):
    outdir = _ensure_outdir(outdir)

    grid_col = _pick_col(ba_gdf, ["grid_L_per_kwh", "grid_L_per_kwh_no_hydro", "grid_L_kwh", "grid_LperKWh"])
    mw_col   = _pick_col(sites_gdf, ["current_mw", "P_mw", "mw", "MW"], required=False)

    fig, ax = plt.subplots(figsize=(12, 8))

    if states_gdf is not None:
        states_gdf.boundary.plot(ax=ax, linewidth=0.6)

    # BA polygons colored by grid intensity
    ba_gdf.plot(ax=ax, column=grid_col, legend=True)

    # facility points sized by MW (if available)
    if mw_col is not None:
        sizes = _autosize_points(sites_gdf[mw_col])
        sites_gdf.plot(ax=ax, markersize=sizes, alpha=0.85)
    else:
        sites_gdf.plot(ax=ax, markersize=18, alpha=0.85)

    ax.set_axis_off()
    ax.set_title("BA grid water-consumption intensity (L/kWh) + hyperscale sites")

    fn = os.path.join(outdir, f"MAIN_Fig1_BA_gridIntensity_sites__{tag}.pdf")
    fig.tight_layout()
    fig.savefig(fn, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return fn

# ---------- FIG 2: basin stress map (Aqueduct) + facilities ----------
def fig2_basin_stress_with_sites(basins_gdf, sites_gdf, outdir, states_gdf=None, tag="base"):
    outdir = _ensure_outdir(outdir)

    stress_col = _pick_col(basins_gdf, ["bws_01_score", "bws_score", "baseline_stress", "stress", "bws"], required=True)
    mw_col     = _pick_col(sites_gdf, ["current_mw", "P_mw", "mw", "MW"], required=False)

    fig, ax = plt.subplots(figsize=(12, 8))

    if states_gdf is not None:
        states_gdf.boundary.plot(ax=ax, linewidth=0.6)

    # basins colored by stress score
    basins_gdf.plot(ax=ax, column=stress_col, legend=True)

    # facilities
    if mw_col is not None:
        sizes = _autosize_points(sites_gdf[mw_col])
        sites_gdf.plot(ax=ax, markersize=sizes, alpha=0.85)
    else:
        sites_gdf.plot(ax=ax, markersize=18, alpha=0.85)

    ax.set_axis_off()
    ax.set_title("Hydrologic scarcity (Aqueduct baseline stress) + hyperscale sites")

    fn = os.path.join(outdir, f"MAIN_Fig2_BasinStress_sites__{tag}.pdf")
    fig.tight_layout()
    fig.savefig(fn, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return fn

# ---------- FIG 3: Scope-2 hotspot typology (reuse your existing function if present) ----------
def fig3_scope2_hotspot_typology(ba_gdf, sites_gdf, outdir, states_gdf=None, tag="base"):
    """
    If your notebook already defines plot_scope2_ba_hotspots() or map_ba_scope2(), we call it.
    Otherwise, we fallback to a simple map of Scope-2 totals.
    """
    outdir = _ensure_outdir(outdir)

    # try to use your existing hotspot/typology plotter if available
    if "plot_scope2_ba_hotspots" in globals():
        fn = os.path.join(outdir, f"MAIN_Fig3_Scope2_hotspotTypology__{tag}.pdf")
        plot_scope2_ba_hotspots(ba_gdf=ba_gdf, sites_gdf=sites_gdf, states_gdf=states_gdf, outpath=fn)
        return fn

    if "map_ba_scope2" in globals():
        # your map_ba_scope2 writes to OUTDIR internally; still call it for consistency
        map_ba_scope2(ba_gdf, sites_gdf, states_p=states_gdf,
                      title="Scope 2 water consumption by BA + hyperscale sites")
        # we don't know the exact filename it uses; return None
        return None

    # fallback: simple Scope-2 BA totals map
    s2_col = _pick_col(ba_gdf, ["scope2_m3_ba", "s2_m3_ba", "scope2_m3", "W2_m3", "W2"], required=True)
    mw_col = _pick_col(sites_gdf, ["current_mw", "P_mw", "mw", "MW"], required=False)

    fig, ax = plt.subplots(figsize=(12, 8))
    if states_gdf is not None:
        states_gdf.boundary.plot(ax=ax, linewidth=0.6)

    ba_gdf.plot(ax=ax, column=s2_col, legend=True)

    if mw_col is not None:
        sizes = _autosize_points(sites_gdf[mw_col])
        sites_gdf.plot(ax=ax, markersize=sizes, alpha=0.85)
    else:
        sites_gdf.plot(ax=ax, markersize=18, alpha=0.85)

    ax.set_axis_off()
    ax.set_title("Scope 2 water consumption by BA (m³/yr) + hyperscale sites")

    fn = os.path.join(outdir, f"MAIN_Fig3_Scope2_byBA_sites__{tag}.pdf")
    fig.tight_layout()
    fig.savefig(fn, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return fn

# ---------- FIG 4: “who sits where” bivariate (grid intensity vs scarcity), sized by MW ----------
def fig4_bivariate_scatter(ba_gdf, outdir, tag="base", label_top=10):
    """
    Scatter: x = BA grid intensity (L/kWh), y = mean stress exposure, size = hyperscale MW.
    """
    outdir = _ensure_outdir(outdir)

    xcol = _pick_col(ba_gdf, ["grid_L_per_kwh", "grid_L_per_kwh_no_hydro", "grid_L_kwh"])
    ycol = _pick_col(ba_gdf, ["mean_stress", "stress_mean", "stress", "bws_mean", "mean_bws"], required=True)
    mwcol = _pick_col(ba_gdf, ["mw_ba", "MW", "hyperscale_mw", "mw_total", "sum_mw"], required=True)
    namecol = _pick_col(ba_gdf, ["ba_code", "BA", "code"], required=False)

    x = ba_gdf[xcol].astype(float)
    y = ba_gdf[ycol].astype(float)
    s = _autosize_points(ba_gdf[mwcol], min_s=15, max_s=250)

    fig, ax = plt.subplots(figsize=(9, 7))
    ax.scatter(x, y, s=s, alpha=0.75)

    ax.set_xlabel("BA grid water-consumption intensity (L/kWh)")
    ax.set_ylabel("Mean siting scarcity exposure (Aqueduct baseline stress)")
    ax.set_title("Grid water intensity vs scarcity exposure (bubble size = hyperscale MW)")

    # optional labels for top MW
    if label_top and namecol is not None:
        top = ba_gdf.sort_values(mwcol, ascending=False).head(label_top)
        for _, r in top.iterrows():
            ax.annotate(str(r[namecol]), (float(r[xcol]), float(r[ycol])), fontsize=8)

    fn = os.path.join(outdir, f"MAIN_Fig4_bivariate_gridVsStress__{tag}.pdf")
    fig.tight_layout()
    fig.savefig(fn, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return fn

# ============================================================
# RUN: build all 4 and save into OUTDIR
# ============================================================

# If you used run_scope_figures already:
# sites_p, ba_p2, basin_table = run_scope_figures(...)
# and basins_p, states_p exist from earlier cells.

TAG = "base"  # change to "efficiency" / "high_load" if you want scenario variants

fig1 = fig1_ba_grid_intensity_with_sites(ba_p2, sites_p, OUTDIR, states_gdf=globals().get("states_p", None), tag=TAG)
fig2 = fig2_basin_stress_with_sites(globals().get("basins_p", None), sites_p, OUTDIR,
                                    states_gdf=globals().get("states_p", None), tag=TAG)
fig3 = fig3_scope2_hotspot_typology(ba_p2, sites_p, OUTDIR, states_gdf=globals().get("states_p", None), tag=TAG)
fig4 = fig4_bivariate_scatter(ba_p2, OUTDIR, tag=TAG, label_top=10)

print("Wrote:")
print("  ", fig1)
print("  ", fig2)
print("  ", fig3)
print("  ", fig4)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import geopandas as gpd

def _sizes_sqrt(series, smin=8, smax=220):
    x = pd.to_numeric(series, errors="coerce").fillna(0).clip(lower=0)
    if x.max() <= 0:
        return np.full(len(x), smin)
    z = np.sqrt(x)
    z = (z - z.min()) / (z.max() - z.min() + 1e-12)
    return smin + (smax - smin) * z

def _legend_sizes(ax, breaks, smin, smax, title, anchor=(1.02, 0.25)):
    svals = _sizes_sqrt(pd.Series(breaks), smin=smin, smax=smax)
    handles = [
        Line2D([0],[0], marker='o', linestyle='None',
               markerfacecolor="none", markeredgecolor="black",
               markeredgewidth=0.8, alpha=0.9,
               markersize=np.sqrt(s), label=f"{b:,} MW")
        for b, s in zip(breaks, svals)
    ]
    leg = ax.legend(handles=handles, title=title, loc="center left",
                    bbox_to_anchor=anchor, frameon=True, framealpha=0.95,
                    fontsize=8, title_fontsize=9)
    leg.get_frame().set_linewidth(0.6)
    return leg

def _points_by_region(poly_gdf, region_id_col, sites_gdf, sites_region_col, mw_col):
    agg = (sites_gdf.assign(_mw=pd.to_numeric(sites_gdf[mw_col], errors="coerce").fillna(0).clip(lower=0))
                    .groupby(sites_region_col, as_index=False)["_mw"].sum()
                    .rename(columns={sites_region_col: region_id_col, "_mw": "mw_sum"}))
    out = poly_gdf[[region_id_col, "geometry"]].merge(agg, on=region_id_col, how="left")
    out["mw_sum"] = out["mw_sum"].fillna(0)
    out["geometry"] = out.geometry.representative_point()
    return gpd.GeoDataFrame(out, geometry="geometry", crs=poly_gdf.crs)

def plot_ba_map_clean(
    ba_polys, sites, value_col,
    region_id_col="subregion_id", sites_region_col="subregion_id", mw_col="current_mw",
    title="", cbar_label="",
    cmap="cividis",
    use_quantiles=True, q=(0.02, 0.98),
    dot_smin=10, dot_smax=240,
    dot_breaks=(250, 1000, 3000),
    states_gdf=None,
    outpath=None
):
    fig, ax = plt.subplots(figsize=(12.5, 7.5))
    ax.set_axis_off()

    # --- choropleth normalization ---
    vals = pd.to_numeric(ba_polys[value_col], errors="coerce")
    if use_quantiles and np.isfinite(vals).any():
        vmin = np.nanquantile(vals, q[0])
        vmax = np.nanquantile(vals, q[1])
        norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    else:
        norm = None
        vmin = vmax = None

    # --- fill polygons (no borders) ---
    ba_polys.plot(ax=ax, column=value_col, cmap=cmap, linewidth=0, zorder=1,
                 vmin=None if norm is None else norm.vmin,
                 vmax=None if norm is None else norm.vmax)

    # --- BA boundaries on top (thin, light) ---
    ba_polys.boundary.plot(ax=ax, linewidth=0.25, color="#f2f2f2", alpha=0.9, zorder=2)

    # --- states boundaries (optional) ---
    if states_gdf is not None:
        states_gdf.boundary.plot(ax=ax, linewidth=0.6, color="#333333", alpha=0.6, zorder=3)

    # --- aggregated dots ---
    dots = _points_by_region(ba_polys, region_id_col, sites, sites_region_col, mw_col)
    dots = dots[dots["mw_sum"] > 0].copy()
    dots["ms"] = _sizes_sqrt(dots["mw_sum"], smin=dot_smin, smax=dot_smax)

    coll = dots.plot(ax=ax, markersize=dots["ms"], facecolor="none",
                     edgecolor="black", linewidth=0.8, alpha=0.85, zorder=4)

    # rasterize dots for clean PDFs
    try:
        coll.collections[0].set_rasterized(True)
    except Exception:
        pass

    ax.set_title(title, fontsize=13)

    # --- colorbar ---
    sm = mpl.cm.ScalarMappable(cmap=plt.get_cmap(cmap),
                              norm=mpl.colors.Normalize(
                                  vmin=vmin if vmin is not None else np.nanmin(vals),
                                  vmax=vmax if vmax is not None else np.nanmax(vals)
                              ))
    sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, fraction=0.030, pad=0.015)
    cb.set_label(cbar_label)

    # --- size legend outside ---
    _legend_sizes(ax, dot_breaks, dot_smin, dot_smax, "Hyperscale capacity (sum MW)", anchor=(1.02, 0.22))
    fig.subplots_adjust(right=0.84)

    if outpath:
        fig.savefig(outpath, bbox_inches="tight")
    return fig


In [ ]:
def plot_facility_hexbin(sites_gdf, mw_col, title="", gridsize=130, outpath=None):
    fig, ax = plt.subplots(figsize=(12.5, 7.5))
    ax.set_axis_off()

    x = sites_gdf.geometry.x.values
    y = sites_gdf.geometry.y.values
    w = pd.to_numeric(sites_gdf[mw_col], errors="coerce").fillna(0).clip(lower=0).values

    hb = ax.hexbin(x, y, C=w, reduce_C_function=np.sum,
                   gridsize=gridsize, bins="log", cmap="magma",
                   linewidths=0, mincnt=1, alpha=0.95)

    ax.set_title(title, fontsize=13)
    cb = fig.colorbar(hb, ax=ax, fraction=0.030, pad=0.015)
    cb.set_label("Log10(sum MW in hex)")

    if outpath:
        fig.savefig(outpath, bbox_inches="tight")
    return fig


In [ ]:
# ============================================
# Paper-quality Fig1–Fig4 plotting utilities
# (choropleths + ONE dot per region + clean legends)
# ============================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# -----------------------
# 0) Helpers
# -----------------------
def safe_num(x):
    return pd.to_numeric(x, errors="coerce")

def ensure_crs_match(*gdfs):
    """Reproject all gdfs to the CRS of the first non-null."""
    base = None
    for g in gdfs:
        if g is not None and hasattr(g, "crs") and g.crs is not None:
            base = g.crs
            break
    if base is None:
        return gdfs
    out = []
    for g in gdfs:
        if g is None:
            out.append(None)
        else:
            out.append(g.to_crs(base) if g.crs != base else g)
    return out

def sizes_sqrt(series, smin=12, smax=260):
    x = safe_num(series).fillna(0).clip(lower=0)
    if len(x) == 0 or x.max() <= 0:
        return np.array([])
    z = np.sqrt(x)
    z = (z - z.min()) / (z.max() - z.min() + 1e-12)
    return smin + (smax - smin) * z

def legend_sizes(ax, breaks, smin, smax, title, anchor=(1.02, 0.25)):
    svals = sizes_sqrt(pd.Series(breaks), smin=smin, smax=smax)
    handles = [
        Line2D([0],[0], marker='o', linestyle='None',
               markerfacecolor="white", markeredgecolor="black",
               markeredgewidth=0.9, alpha=0.95,
               markersize=np.sqrt(s), label=f"{b:,} MW")
        for b, s in zip(breaks, svals)
    ]
    leg = ax.legend(handles=handles, title=title, loc="center left",
                    bbox_to_anchor=anchor, frameon=True, framealpha=0.95,
                    fontsize=8, title_fontsize=9)
    leg.get_frame().set_linewidth(0.6)
    return leg

def add_discrete_legend(ax, labels_to_style, title, anchor=(1.02, 0.70)):
    """
    labels_to_style: list of tuples (label, facecolor)
    """
    handles = []
    for lab, col in labels_to_style:
        handles.append(Line2D([0],[0], marker='s', linestyle='None',
                              markerfacecolor=col, markeredgecolor="#333333",
                              markeredgewidth=0.6, markersize=10, label=lab))
    leg = ax.legend(handles=handles, title=title, loc="center left",
                    bbox_to_anchor=anchor, frameon=True, framealpha=0.95,
                    fontsize=8, title_fontsize=9)
    leg.get_frame().set_linewidth(0.6)
    return leg

def rasterize_first_collection(gdf_plot_return):
    """Rasterize the first collection to keep PDF light."""
    try:
        gdf_plot_return.collections[0].set_rasterized(True)
    except Exception:
        pass

def representative_points(poly_gdf):
    # representative_point() is safer than centroid for multipolygons/coastal shapes
    rp = poly_gdf.geometry.representative_point()
    return gpd.GeoDataFrame(poly_gdf.drop(columns="geometry"),
                            geometry=rp, crs=poly_gdf.crs)

def aggregate_mw_to_polys(poly_gdf, sites_gdf, poly_id_col, sites_poly_id_col, mw_col):
    """
    Returns GeoDataFrame of representative points for poly_gdf with mw_sum added.
    Assumes sites already have the polygon id column (e.g., BA code, basin id).
    """
    agg = (sites_gdf.assign(_mw=safe_num(sites_gdf[mw_col]).fillna(0).clip(lower=0))
                 .groupby(sites_poly_id_col, as_index=False)["_mw"].sum()
                 .rename(columns={sites_poly_id_col: poly_id_col, "_mw": "mw_sum"}))
    out = poly_gdf[[poly_id_col, "geometry"]].merge(agg, on=poly_id_col, how="left")
    out["mw_sum"] = out["mw_sum"].fillna(0)
    out_pts = representative_points(out)
    return out_pts

def apply_quantile_norm(vals, qlow=0.02, qhigh=0.98):
    vals = safe_num(vals)
    if np.isfinite(vals).any():
        vmin = np.nanquantile(vals, qlow)
        vmax = np.nanquantile(vals, qhigh)
        if not np.isfinite(vmin): vmin = np.nanmin(vals)
        if not np.isfinite(vmax): vmax = np.nanmax(vals)
        if vmax <= vmin:
            vmax = vmin + 1e-12
        return mpl.colors.Normalize(vmin=vmin, vmax=vmax), vmin, vmax
    return mpl.colors.Normalize(vmin=0, vmax=1), 0, 1

def draw_outlines(ax, ba_gdf=None, states_gdf=None):
    # thin, unobtrusive outlines
    if ba_gdf is not None:
        ba_gdf.boundary.plot(ax=ax, linewidth=0.25, color="#f2f2f2", alpha=0.9, zorder=3)
    if states_gdf is not None:
        states_gdf.boundary.plot(ax=ax, linewidth=0.6, color="#333333", alpha=0.55, zorder=4)
    ax.set_axis_off()

def savefig(fig, outpath):
    if outpath:
        os.makedirs(os.path.dirname(outpath), exist_ok=True)
        fig.savefig(outpath, bbox_inches="tight")
    return outpath

# -----------------------
# 1) Fig1: BA choropleth + BA MW dots
# -----------------------
def fig1_ba_grid_intensity(
    ba_polys, sites, value_col, *,
    ba_id_col="subregion_id",
    sites_ba_col="subregion_id",
    mw_col="current_mw",
    states_gdf=None,
    cmap="cividis",
    qlow=0.02, qhigh=0.98,
    dot_smin=12, dot_smax=260,
    dot_breaks=(250, 1000, 3000),
    title="BA grid water-consumption intensity and hyperscale load",
    cbar_label="Grid water intensity (L/kWh)",
    outpath=None
):
    ba_polys, sites, states_gdf = ensure_crs_match(ba_polys, sites, states_gdf)

    fig, ax = plt.subplots(figsize=(12.5, 7.5))
    ax.set_axis_off()

    norm, vmin, vmax = apply_quantile_norm(ba_polys[value_col], qlow, qhigh)

    # Fill (no borders)
    ba_polys.plot(ax=ax, column=value_col, cmap=cmap, linewidth=0, norm=norm, zorder=1)

    draw_outlines(ax, ba_gdf=ba_polys, states_gdf=states_gdf)

    # Aggregated dots
    dots = aggregate_mw_to_polys(ba_polys, sites, ba_id_col, sites_ba_col, mw_col)
    dots = dots[dots["mw_sum"] > 0].copy()
    dots["ms"] = sizes_sqrt(dots["mw_sum"], smin=dot_smin, smax=dot_smax)

    p = dots.plot(ax=ax, markersize=dots["ms"],
                  facecolor="white", edgecolor="black",
                  linewidth=0.9, alpha=0.92, zorder=6)
    rasterize_first_collection(p)

    ax.set_title(title, fontsize=13)

    # Colorbar
    sm = mpl.cm.ScalarMappable(cmap=plt.get_cmap(cmap), norm=norm)
    sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, fraction=0.030, pad=0.015)
    cb.set_label(cbar_label)

    # Legends outside
    legend_sizes(ax, dot_breaks, dot_smin, dot_smax, "Hyperscale capacity (sum MW)", anchor=(1.02, 0.22))
    fig.subplots_adjust(right=0.84)

    return savefig(fig, outpath)

# -----------------------
# 2) Fig2: Basin stress choropleth + Basin MW dots
# -----------------------
def fig2_basin_stress(
    basins_polys, sites, stress_col, *,
    basin_id_col="HYBAS_ID",           # change if yours differs
    sites_basin_col="HYBAS_ID",        # change if yours differs
    mw_col="current_mw",
    states_gdf=None,
    cmap="cividis",
    qlow=0.02, qhigh=0.98,             # for continuous stress (if categorical, skip)
    dot_smin=12, dot_smax=260,
    dot_breaks=(250, 1000, 3000),
    title="Hydrologic stress (Aqueduct baseline) and hyperscale load",
    cbar_label="Aqueduct baseline water stress (0–5)",
    outpath=None
):
    basins_polys, sites, states_gdf = ensure_crs_match(basins_polys, sites, states_gdf)

    fig, ax = plt.subplots(figsize=(12.5, 7.5))
    ax.set_axis_off()

    norm, vmin, vmax = apply_quantile_norm(basins_polys[stress_col], qlow, qhigh)

    basins_polys.plot(ax=ax, column=stress_col, cmap=cmap, linewidth=0, norm=norm, zorder=1)

    # basin boundaries subtle; states optional
    basins_polys.boundary.plot(ax=ax, linewidth=0.15, color="#f2f2f2", alpha=0.75, zorder=2)
    if states_gdf is not None:
        states_gdf.boundary.plot(ax=ax, linewidth=0.6, color="#333333", alpha=0.55, zorder=4)

    # Aggregated dots
    dots = aggregate_mw_to_polys(basins_polys, sites, basin_id_col, sites_basin_col, mw_col)
    dots = dots[dots["mw_sum"] > 0].copy()
    dots["ms"] = sizes_sqrt(dots["mw_sum"], smin=dot_smin, smax=dot_smax)

    p = dots.plot(ax=ax, markersize=dots["ms"],
                  facecolor="white", edgecolor="black",
                  linewidth=0.9, alpha=0.92, zorder=6)
    rasterize_first_collection(p)

    ax.set_title(title, fontsize=13)

    sm = mpl.cm.ScalarMappable(cmap=plt.get_cmap(cmap), norm=norm)
    sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, fraction=0.030, pad=0.015)
    cb.set_label(cbar_label)

    legend_sizes(ax, dot_breaks, dot_smin, dot_smax, "Hyperscale capacity (sum MW)", anchor=(1.02, 0.22))
    fig.subplots_adjust(right=0.84)

    return savefig(fig, outpath)

# -----------------------
# 3) Fig3: BA hotspot typology map (categorical) + BA MW dots
# -----------------------
def classify_scope2_typology(
    ba_df,
    scope2_col, fossil_share_col, stress_col,
    *,
    scope2_quantile=0.75,
    fossil_thresh=0.50,
    stress_thresh=3.0,
    require_activity_col=None
):
    """
    Returns a copy with 'typology' column.
    typology:
      - "None"
      - "High S2 only"
      - "Fossil only"
      - "High stress only"
      - "S2+Fossil"
      - "S2+Stress"
      - "Fossil+Stress"
      - "Triple: S2+Fossil+Stress"
    You can simplify later, but this gives flexibility.
    """
    df = ba_df.copy()

    s2 = safe_num(df[scope2_col])
    fossil = safe_num(df[fossil_share_col])
    stress = safe_num(df[stress_col])

    if require_activity_col and require_activity_col in df.columns:
        active = df[require_activity_col].fillna(False).astype(bool)
    else:
        active = pd.Series(True, index=df.index)

    # threshold for "high S2" among active BAs with finite values
    qbase = s2[active & np.isfinite(s2)]
    if len(qbase) == 0:
        s2_thr = np.nan
    else:
        s2_thr = np.nanquantile(qbase, scope2_quantile)

    hi_s2 = active & (s2 >= s2_thr)
    hi_fossil = active & (fossil > fossil_thresh)
    hi_stress = active & (stress >= stress_thresh)

    typ = np.full(len(df), "None", dtype=object)

    # build combinations
    typ[hi_s2 & ~hi_fossil & ~hi_stress] = "High S2 only"
    typ[~hi_s2 & hi_fossil & ~hi_stress] = "Fossil only"
    typ[~hi_s2 & ~hi_fossil & hi_stress] = "High stress only"
    typ[hi_s2 & hi_fossil & ~hi_stress] = "S2+Fossil"
    typ[hi_s2 & ~hi_fossil & hi_stress] = "S2+Stress"
    typ[~hi_s2 & hi_fossil & hi_stress] = "Fossil+Stress"
    typ[hi_s2 & hi_fossil & hi_stress] = "Triple: S2+Fossil+Stress"

    df["typology"] = typ
    df["s2_threshold_used"] = s2_thr
    return df

def fig3_ba_typology(
    ba_polys, sites, *,
    ba_id_col="subregion_id",
    sites_ba_col="subregion_id",
    mw_col="current_mw",
    scope2_col="scope2_GL_yr_base",      # <-- set to your actual BA column name
    fossil_col="fossil_share",           # fraction 0..1 (or percent; adjust below)
    stress_col="mean_stress",
    fossil_is_percent=False,             # set True if fossil column is 0..100
    scope2_quantile=0.75,
    fossil_thresh=0.50,                  # 0.50 means 50%
    stress_thresh=3.0,
    states_gdf=None,
    dot_smin=12, dot_smax=260,
    dot_breaks=(250, 1000, 3000),
    title="Scope 2 hotspot typology (BA)",
    outpath=None
):
    ba_polys, sites, states_gdf = ensure_crs_match(ba_polys, sites, states_gdf)

    df = ba_polys.copy()
    if fossil_is_percent:
        df["_fossil_frac"] = safe_num(df[fossil_col]) / 100.0
        fossil_use = "_fossil_frac"
    else:
        fossil_use = fossil_col

    df2 = classify_scope2_typology(
        df, scope2_col=scope2_col, fossil_share_col=fossil_use, stress_col=stress_col,
        scope2_quantile=scope2_quantile, fossil_thresh=fossil_thresh, stress_thresh=stress_thresh
    )

    # Discrete palette (muted, print-friendly)
    palette = {
        "None": "#e6e6e6",
        "High S2 only": "#b3cde3",
        "Fossil only": "#fbb4ae",
        "High stress only": "#ccebc5",
        "S2+Fossil": "#decbe4",
        "S2+Stress": "#fed9a6",
        "Fossil+Stress": "#ffffcc",
        "Triple: S2+Fossil+Stress": "#fb8072",  # highlight
    }

    fig, ax = plt.subplots(figsize=(12.5, 7.5))
    ax.set_axis_off()

    # plot each class separately for discrete legend control
    for k, col in palette.items():
        sub = df2[df2["typology"] == k]
        if len(sub) > 0:
            sub.plot(ax=ax, color=col, linewidth=0, zorder=1)

    # boundaries
    draw_outlines(ax, ba_gdf=df2, states_gdf=states_gdf)

    # dots: BA aggregated MW
    dots = aggregate_mw_to_polys(df2, sites, ba_id_col, sites_ba_col, mw_col)
    dots = dots[dots["mw_sum"] > 0].copy()
    dots["ms"] = sizes_sqrt(dots["mw_sum"], smin=dot_smin, smax=dot_smax)
    p = dots.plot(ax=ax, markersize=dots["ms"],
                  facecolor="white", edgecolor="black",
                  linewidth=0.9, alpha=0.92, zorder=6)
    rasterize_first_collection(p)

    ax.set_title(title, fontsize=13)

    # legends outside
    order_for_legend = [
        ("Triple: S2+Fossil+Stress", palette["Triple: S2+Fossil+Stress"]),
        ("S2+Fossil", palette["S2+Fossil"]),
        ("S2+Stress", palette["S2+Stress"]),
        ("Fossil+Stress", palette["Fossil+Stress"]),
        ("High S2 only", palette["High S2 only"]),
        ("Fossil only", palette["Fossil only"]),
        ("High stress only", palette["High stress only"]),
        ("None", palette["None"]),
    ]
    add_discrete_legend(ax, order_for_legend, "Typology (BA)", anchor=(1.02, 0.72))
    legend_sizes(ax, dot_breaks, dot_smin, dot_smax, "Hyperscale capacity (sum MW)", anchor=(1.02, 0.26))
    fig.subplots_adjust(right=0.80)

    return savefig(fig, outpath)

# -----------------------
# 4) Fig4: Scatter (grid intensity vs stress) with typology colors
# -----------------------
def fig4_scatter_tradeoff(
    ba_df, *,
    x_col="grid_L_per_kWh",
    y_col="mean_stress",
    mw_col="hyperscale_mw",
    scope2_col="scope2_GL_yr_base",
    fossil_col="fossil_share",
    fossil_is_percent=False,
    scope2_quantile=0.75,
    fossil_thresh=0.50,
    stress_thresh=3.0,
    label_col="subregion_id",
    label_top=12,
    outpath=None,
    title="Grid water intensity vs siting scarcity (BA-level)"
):
    df = ba_df.copy()
    if fossil_is_percent:
        df["_fossil_frac"] = safe_num(df[fossil_col]) / 100.0
        fossil_use = "_fossil_frac"
    else:
        fossil_use = fossil_col

    df2 = classify_scope2_typology(
        df, scope2_col=scope2_col, fossil_share_col=fossil_use, stress_col=y_col,
        scope2_quantile=scope2_quantile, fossil_thresh=fossil_thresh, stress_thresh=stress_thresh
    )

    palette = {
        "None": "#bdbdbd",
        "High S2 only": "#80b1d3",
        "Fossil only": "#fb8072",
        "High stress only": "#8dd3c7",
        "S2+Fossil": "#bebada",
        "S2+Stress": "#fdb462",
        "Fossil+Stress": "#ffffb3",
        "Triple: S2+Fossil+Stress": "#e31a1c",
    }

    x = safe_num(df2[x_col])
    y = safe_num(df2[y_col])
    mw = safe_num(df2[mw_col]).fillna(0).clip(lower=0)

    fig, ax = plt.subplots(figsize=(10.5, 7.2))

    # plot by class for clean legend ordering
    order = [
        "Triple: S2+Fossil+Stress",
        "S2+Fossil", "S2+Stress", "Fossil+Stress",
        "High S2 only", "Fossil only", "High stress only",
        "None"
    ]
    for k in order:
        sub = df2[df2["typology"] == k]
        if len(sub) == 0:
            continue
        ms = sizes_sqrt(sub[mw_col], smin=20, smax=340)
        ax.scatter(
            safe_num(sub[x_col]), safe_num(sub[y_col]),
            s=ms, c=palette[k],
            edgecolors="black" if k == "Triple: S2+Fossil+Stress" else "#333333",
            linewidths=0.6, alpha=0.85, label=k, zorder=3
        )

    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Grid water intensity (L/kWh)")
    ax.set_ylabel("Mean baseline stress (0–5)")
    ax.grid(True, linewidth=0.4, alpha=0.35)

    # label top by MW (or you can label by Scope2)
    sub_lab = df2.copy()
    sub_lab["mw_for_label"] = safe_num(sub_lab[mw_col]).fillna(0)
    top = sub_lab.sort_values("mw_for_label", ascending=False).head(label_top)
    for _, r in top.iterrows():
        ax.text(safe_num(r[x_col]), safe_num(r[y_col]), str(r[label_col]),
                fontsize=8, ha="left", va="bottom")

    # legend outside
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5),
              frameon=True, framealpha=0.95, fontsize=8, title="Typology")
    fig.subplots_adjust(right=0.72)

    return savefig(fig, outpath)

# -----------------------
# 5) One-call wrapper: generate all 4 main figs
# -----------------------
def make_main_figures(
    OUTDIR,
    TAG,
    *,
    ba_polys,
    basins_polys,
    sites_gdf,
    states_gdf=None,
    # column mappings (edit as needed)
    COL_BA_ID="subregion_id",
    COL_SITES_BA="subregion_id",
    COL_BASIN_ID="HYBAS_ID",
    COL_SITES_BASIN="HYBAS_ID",
    COL_MW="current_mw",
    COL_BA_GRID="grid_L_per_kWh",
    COL_BA_STRESS="mean_stress",
    COL_BASIN_STRESS="baseline_stress",
    COL_BA_SCOPE2="scope2_GL_yr_base",
    COL_BA_FOSSIL="fossil_share",
    fossil_is_percent=False
):
    os.makedirs(OUTDIR, exist_ok=True)

    f1 = fig1_ba_grid_intensity(
        ba_polys, sites_gdf, value_col=COL_BA_GRID,
        ba_id_col=COL_BA_ID, sites_ba_col=COL_SITES_BA, mw_col=COL_MW,
        states_gdf=states_gdf,
        title="Figure 1. Grid water intensity by balancing authority and hyperscale capacity",
        cbar_label="Grid water intensity (L/kWh)",
        outpath=os.path.join(OUTDIR, f"MAIN_Fig1_BA_gridIntensity_plusMW__{TAG}.pdf"),
    )

    f2 = fig2_basin_stress(
        basins_polys, sites_gdf, stress_col=COL_BASIN_STRESS,
        basin_id_col=COL_BASIN_ID, sites_basin_col=COL_SITES_BASIN, mw_col=COL_MW,
        states_gdf=states_gdf,
        title="Figure 2. Basin water stress (Aqueduct) and hyperscale capacity",
        cbar_label="Aqueduct baseline water stress (0–5)",
        outpath=os.path.join(OUTDIR, f"MAIN_Fig2_Basin_stress_plusMW__{TAG}.pdf"),
    )

    f3 = fig3_ba_typology(
        ba_polys, sites_gdf,
        ba_id_col=COL_BA_ID, sites_ba_col=COL_SITES_BA, mw_col=COL_MW,
        scope2_col=COL_BA_SCOPE2, fossil_col=COL_BA_FOSSIL, stress_col=COL_BA_STRESS,
        fossil_is_percent=fossil_is_percent,
        title="Figure 3. Scope 2 hotspot typology by balancing authority",
        outpath=os.path.join(OUTDIR, f"MAIN_Fig3_Scope2_typology__{TAG}.pdf"),
    )

    f4 = fig4_scatter_tradeoff(
        ba_polys,
        x_col=COL_BA_GRID, y_col=COL_BA_STRESS, mw_col="hyperscale_mw",
        scope2_col=COL_BA_SCOPE2, fossil_col=COL_BA_FOSSIL,
        fossil_is_percent=fossil_is_percent,
        label_col=COL_BA_ID, label_top=12,
        title="Figure 4. Grid water intensity vs siting scarcity (BA-level)",
        outpath=os.path.join(OUTDIR, f"MAIN_Fig4_tradeoff_scatter__{TAG}.pdf"),
    )

    return {"Fig1": f1, "Fig2": f2, "Fig3": f3, "Fig4": f4}



In [ ]:
# 1) Confirm which object is being passed
print("ba_p type:", type(ba_p))
print("ba_p shape:", ba_p.shape)

# 2) Verify exact column membership (no assumptions)
col = "grid_L_per_kWh"
print("Has exact column?", col in ba_p.columns)

# 3) Show close matches (catches whitespace/case issues)
print("Columns containing 'grid' or 'kwh':",
      [c for c in ba_p.columns if ("grid" in str(c).lower()) or ("kwh" in str(c).lower())])

# 4) Print repr of suspect columns to reveal hidden spaces
suspects = [c for c in ba_p.columns if "grid" in str(c).lower()]
print("Suspects repr:", [repr(c) for c in suspects])

# 5) If it exists, confirm it’s not all-missing
if col in ba_p.columns:
    print("Non-null count:", ba_p[col].notna().sum())
    print("Min/Max:", ba_p[col].min(), ba_p[col].max())


In [ ]:
# 1) Confirm which object is being passed
print("ba_p type:", type(ba_p))
print("ba_p shape:", ba_p.shape)

# 2) Verify exact column membership (no assumptions)
col = "grid_L_per_kWh"
print("Has exact column?", col in ba_p.columns)

# 3) Show close matches (catches whitespace/case issues)
print("Columns containing 'grid' or 'kwh':",
      [c for c in ba_p.columns if ("grid" in str(c).lower()) or ("kwh" in str(c).lower())])

# 4) Print repr of suspect columns to reveal hidden spaces
suspects = [c for c in ba_p.columns if "grid" in str(c).lower()]
print("Suspects repr:", [repr(c) for c in suspects])

# 5) If it exists, confirm it’s not all-missing
if col in ba_p.columns:
    print("Non-null count:", ba_p[col].notna().sum())
    print("Min/Max:", ba_p[col].min(), ba_p[col].max())


In [ ]:
ba_p = ba_p.rename(columns={
    "grid_L_per_kwh": "grid_L_per_kWh",      # if your code expects this exact spelling
    "mean_bws": "baseline_stress",           # or "baseline_stress_mean" if you prefer
    "share_high_stress": "share_high_stress" # keep as-is; here for clarity
})


In [ ]:
print("basins id cols:", [c for c in basins_p.columns if "HYBAS" in c.upper() or "BASIN" in c.upper()])
print("sites id cols:",  [c for c in sites_p.columns  if "HYBAS" in c.upper() or "BASIN" in c.upper() or "PFAF" in c.upper()])


In [ ]:
basins_p["HYBAS_ID"] = basins_p["HYBAS_ID"].astype("string")
sites_p["pfaf_id"]  = sites_p["pfaf_id"].astype("string")


In [ ]:
ba_p.columns

In [ ]:
import numpy as np
import pandas as pd

# -----------------------
# 0) Column names (yours)
# -----------------------
COL_BA_ID = "subregion_id"
COL_MW    = "current_mw"
COL_S2_M3 = "s2_m3_mid"
COL_S1_M3 = "s1_m3_mid"

# -----------------------
# 1) Build BA totals from sites
# -----------------------
tmp = sites_p[[COL_BA_ID, COL_MW, COL_S2_M3, COL_S1_M3]].copy()

# Safety: coerce numerics
for c in [COL_MW, COL_S2_M3, COL_S1_M3]:
    tmp[c] = pd.to_numeric(tmp[c], errors="coerce")

# Aggregate
ba_agg = (tmp
          .groupby(COL_BA_ID, as_index=False)
          .agg(
              hyperscale_mw_sites=(COL_MW, "sum"),
              s2_m3_yr_base=(COL_S2_M3, "sum"),
              s1_m3_yr_base=(COL_S1_M3, "sum"),
              n_sites_sites=(COL_BA_ID, "size"),
          ))

# Totals + unit conversions
ba_agg["tot_m3_yr_base"] = ba_agg["s1_m3_yr_base"] + ba_agg["s2_m3_yr_base"]

# m3 -> GL (since 1 GL = 1e9 L = 1e6 m3)
ba_agg["s2_GL_yr_base"]  = ba_agg["s2_m3_yr_base"] / 1e6
ba_agg["s1_GL_yr_base"]  = ba_agg["s1_m3_yr_base"] / 1e6
ba_agg["tot_GL_yr_base"] = ba_agg["tot_m3_yr_base"] / 1e6

# -----------------------
# 2) Merge into ba_p
# -----------------------
# Make sure IDs match dtype
ba_p[COL_BA_ID]   = ba_p[COL_BA_ID].astype(str)
ba_agg[COL_BA_ID] = ba_agg[COL_BA_ID].astype(str)

ba_p = ba_p.merge(ba_agg, on=COL_BA_ID, how="left")

# Fill missing (BAs with zero sites)
for c in ["hyperscale_mw_sites", "s2_m3_yr_base", "s1_m3_yr_base", "tot_m3_yr_base",
          "s2_GL_yr_base", "s1_GL_yr_base", "tot_GL_yr_base", "n_sites_sites"]:
    if c in ba_p.columns:
        ba_p[c] = ba_p[c].fillna(0)

# -----------------------
# 3) Sanity checks (optional but useful)
# -----------------------
print("Check BA totals vs site totals:")
print("  sites sum s2 (m3/yr):", float(np.nansum(sites_p[COL_S2_M3])))
print("  ba_p sum s2 (m3/yr): ", float(np.nansum(ba_p["s2_m3_yr_base"])))
print("  sites sum MW:", float(np.nansum(sites_p[COL_MW])))
print("  ba_p sum MW: ", float(np.nansum(ba_p["hyperscale_mw_sites"])))


In [ ]:
# Example (adjust column names to match your sites df)
# sites_p must have: BA id, MW, and scope2 water (GL/yr) OR the inputs to compute it.

COL_SITES_BA = "subregion_id"  # OR whatever BA id column in sites_p is
COL_BA_ID    = "subregion_id"  # BA id column in ba_p

# if sites_p has per-site scope2 in GL/yr:
COL_SITE_SCOPE2 = "scope2_GL_yr"  # replace with your real column

scope2_by_ba = (sites_p[[COL_SITES_BA, COL_SITE_SCOPE2]]
                .dropna()
                .groupby(COL_SITES_BA, as_index=False)[COL_SITE_SCOPE2].sum()
                .rename(columns={COL_SITES_BA: COL_BA_ID, COL_SITE_SCOPE2: "scope2_GL_yr_base"}))

ba_p = ba_p.merge(scope2_by_ba, on=COL_BA_ID, how="left")
ba_p["scope2_GL_yr_base"] = ba_p["scope2_GL_yr_base"].fillna(0)


In [ ]:
import numpy as np

def ensure_scope2_gl(ba_p):
    # already there?
    if "scope2_GL_yr_base" in ba_p.columns:
        return ba_p

    # common alternatives people end up with
    candidates_m3 = [
        "s2_m3_yr_base",   # best / explicit
        "s2_m3_base",
        "scope2_m3_yr_base",
        "scope2_m3_base",
        "s2_m3_mid",       # if your "base" scenario is called mid in your pipeline
        "scope2_m3_mid",
    ]
    candidates_gl = [
        "s2_GL_yr_base",
        "scope2_GL_yr",
        "scope2_GL_yr_mid",
        "s2_GL_yr",
    ]

    # if you already have GL
    for c in candidates_gl:
        if c in ba_p.columns:
            ba_p["scope2_GL_yr_base"] = ba_p[c].astype(float)
            return ba_p

    # if you have m3/yr, convert to GL/yr (1 GL = 1e6 m3)
    for c in candidates_m3:
        if c in ba_p.columns:
            ba_p["scope2_GL_yr_base"] = ba_p[c].astype(float) / 1e6
            return ba_p

    # last resort: show what exists to pick from
    print("No Scope2 column found. Columns that look relevant:")
    print([c for c in ba_p.columns if ("s2" in c.lower()) or ("scope2" in c.lower())])
    raise KeyError("Could not construct scope2_GL_yr_base from ba_p columns.")

ba_p = ensure_scope2_gl(ba_p)


In [ ]:
print(ba_p.columns[100:])

In [ ]:
# pick BA stress column robustly
if "mean_bws" in ba_p.columns:
    COL_BA_STRESS = "mean_bws"
elif "baseline_stress" in ba_p.columns:
    COL_BA_STRESS = "baseline_stress"
elif "mean_stress" in ba_p.columns:
    COL_BA_STRESS = "mean_stress"
else:
    raise KeyError(f"No BA stress column found. Candidates tried: mean_bws, baseline_stress, mean_stress.\n"
                   f"Available cols: {list(ba_p.columns)}")

OUTDIR = "figures_main"
TAG = "base"

paths = make_main_figures(
    OUTDIR, TAG,
    ba_polys=ba_p,
    basins_polys=basins_p,
    sites_gdf=sites_p,
    states_gdf=states_p,
    COL_BA_ID="subregion_id",
    COL_SITES_BA="subregion_id",
    COL_BASIN_ID="HYBAS_ID",
    COL_SITES_BASIN="pfaf_id",
    COL_MW="current_mw",
    COL_BA_GRID="grid_L_per_kWh",
    COL_BA_STRESS=COL_BA_STRESS,
    COL_BASIN_STRESS="bws_01_score",
    COL_BA_SCOPE2="scope2_GL_yr_base",
    COL_BA_FOSSIL="share_fossil",
    fossil_is_percent=False
)

print("Using COL_BA_STRESS =", COL_BA_STRESS)
paths


In [ ]:
ba_p.columns

In [ ]:
# ============================================
# Paper-quality Fig1–Fig4 (improved)
# choropleths + ONE dot per region + clean legends
# ============================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import BoundaryNorm

# -----------------------
# 0) Helpers
# -----------------------
def safe_num(x):
    return pd.to_numeric(x, errors="coerce")

def ensure_crs_match(*gdfs):
    base = None
    for g in gdfs:
        if g is not None and hasattr(g, "crs") and g.crs is not None:
            base = g.crs
            break
    if base is None:
        return gdfs
    out = []
    for g in gdfs:
        if g is None:
            out.append(None)
        else:
            out.append(g.to_crs(base) if g.crs != base else g)
    return out

def sizes_sqrt(series, smin=18, smax=220):
    """Sqrt scaling for dot AREAS in pts^2 (moderate range for print)."""
    x = safe_num(series).fillna(0).clip(lower=0)
    if len(x) == 0 or x.max() <= 0:
        return np.array([])
    z = np.sqrt(x)
    z = (z - z.min()) / (z.max() - z.min() + 1e-12)
    return smin + (smax - smin) * z

def legend_sizes(ax, breaks, smin, smax, title, anchor=(1.02, 0.22)):
    svals = sizes_sqrt(pd.Series(breaks), smin=smin, smax=smax)
    handles = [
        Line2D([0],[0], marker='o', linestyle='None',
               markerfacecolor="white", markeredgecolor="black",
               markeredgewidth=0.9, alpha=0.95,
               markersize=np.sqrt(s), label=f"{b:,} MW")
        for b, s in zip(breaks, svals)
    ]
    leg = ax.legend(handles=handles, title=title, loc="center left",
                    bbox_to_anchor=anchor, frameon=True, framealpha=0.95,
                    fontsize=8, title_fontsize=9)
    leg.get_frame().set_linewidth(0.6)
    return leg

def add_discrete_legend(ax, labels_to_style, title, anchor=(1.02, 0.72)):
    handles = []
    for lab, col in labels_to_style:
        handles.append(Line2D([0],[0], marker='s', linestyle='None',
                              markerfacecolor=col, markeredgecolor="#333333",
                              markeredgewidth=0.6, markersize=10, label=lab))
    leg = ax.legend(handles=handles, title=title, loc="center left",
                    bbox_to_anchor=anchor, frameon=True, framealpha=0.95,
                    fontsize=8, title_fontsize=9)
    leg.get_frame().set_linewidth(0.6)
    return leg

def representative_points(poly_gdf):
    rp = poly_gdf.geometry.representative_point()
    return gpd.GeoDataFrame(poly_gdf.drop(columns="geometry"),
                            geometry=rp, crs=poly_gdf.crs)

def aggregate_mw_to_polys(poly_gdf, sites_gdf, poly_id_col, sites_poly_id_col, mw_col):
    """
    Returns representative points with mw_sum; coerces join keys to str to avoid dtype issues.
    """
    poly = poly_gdf[[poly_id_col, "geometry"]].copy()
    sites = sites_gdf[[sites_poly_id_col, mw_col]].copy()

    poly[poly_id_col] = poly[poly_id_col].astype(str)
    sites[sites_poly_id_col] = sites[sites_poly_id_col].astype(str)

    agg = (sites.assign(_mw=safe_num(sites[mw_col]).fillna(0).clip(lower=0))
                .groupby(sites_poly_id_col, as_index=False)["_mw"].sum()
                .rename(columns={sites_poly_id_col: poly_id_col, "_mw": "mw_sum"}))

    out = poly.merge(agg, on=poly_id_col, how="left")
    out["mw_sum"] = out["mw_sum"].fillna(0)
    return representative_points(out)

def apply_quantile_norm(vals, qlow=0.02, qhigh=0.98):
    vals = safe_num(vals)
    if np.isfinite(vals).any():
        vmin = np.nanquantile(vals, qlow)
        vmax = np.nanquantile(vals, qhigh)
        if not np.isfinite(vmin): vmin = np.nanmin(vals)
        if not np.isfinite(vmax): vmax = np.nanmax(vals)
        if vmax <= vmin:
            vmax = vmin + 1e-12
        return mpl.colors.Normalize(vmin=vmin, vmax=vmax), vmin, vmax
    return mpl.colors.Normalize(vmin=0, vmax=1), 0, 1

def draw_outlines(ax, ba_gdf=None, states_gdf=None):
    # very subtle BA boundaries; clearer state lines
    if ba_gdf is not None:
        ba_gdf.boundary.plot(ax=ax, linewidth=0.20, color="#f0f0f0", alpha=0.9, zorder=3)
    if states_gdf is not None:
        states_gdf.boundary.plot(ax=ax, linewidth=0.65, color="#2b2b2b", alpha=0.55, zorder=4)
    ax.set_axis_off()

def savefig(fig, outpath):
    if outpath:
        os.makedirs(os.path.dirname(outpath), exist_ok=True)
        fig.savefig(outpath, bbox_inches="tight", dpi=300)
    return outpath

# -----------------------
# 0b) Robust BA fields for Fig3/4: compute BA scope2 if missing
# -----------------------
def attach_ba_scope2_from_sites(ba_polys, sites, ba_id_col, sites_ba_col, site_scope2_col="s2_m3_mid"):
    ba = ba_polys.copy()
    ba[ba_id_col] = ba[ba_id_col].astype(str)
    s = sites.copy()
    s[sites_ba_col] = s[sites_ba_col].astype(str)

    if site_scope2_col not in s.columns:
        raise KeyError(f"sites missing '{site_scope2_col}'. Available: {list(s.columns)}")

    agg = (s.assign(_s2=safe_num(s[site_scope2_col]).fillna(0))
             .groupby(sites_ba_col, as_index=False)["_s2"].sum()
             .rename(columns={sites_ba_col: ba_id_col, "_s2": "scope2_m3_mid_from_sites"}))

    ba = ba.merge(agg, on=ba_id_col, how="left")
    ba["scope2_m3_mid_from_sites"] = ba["scope2_m3_mid_from_sites"].fillna(0)
    return ba

# -----------------------
# 1) Fig1: BA choropleth + BA MW dots (clean + missing handling)
# -----------------------
def fig1_ba_grid_intensity(
    ba_polys, sites, value_col, *,
    ba_id_col="subregion_id",
    sites_ba_col="subregion_id",
    mw_col="current_mw",
    states_gdf=None,
    cmap="cividis",
    qlow=0.02, qhigh=0.98,
    dot_smin=18, dot_smax=220,
    dot_breaks=(250, 1000, 3000),
    title="BA grid water-consumption intensity and hyperscale load",
    cbar_label="Grid water intensity (L/kWh)",
    outpath=None
):
    ba_polys, sites, states_gdf = ensure_crs_match(ba_polys, sites, states_gdf)

    fig, ax = plt.subplots(figsize=(12.5, 7.5))
    ax.set_axis_off()

    if value_col not in ba_polys.columns:
        raise KeyError(f"'{value_col}' not in ba_polys. Available: {list(ba_polys.columns)}")

    norm, vmin, vmax = apply_quantile_norm(ba_polys[value_col], qlow, qhigh)

    # Missing in light gray
    miss = ba_polys[safe_num(ba_polys[value_col]).isna()]
    if len(miss) > 0:
        miss.plot(ax=ax, color="#eeeeee", linewidth=0, zorder=0)

    ok = ba_polys[safe_num(ba_polys[value_col]).notna()]
    ok.plot(ax=ax, column=value_col, cmap=cmap, linewidth=0, norm=norm, zorder=1)

    draw_outlines(ax, ba_gdf=ba_polys, states_gdf=states_gdf)

    dots = aggregate_mw_to_polys(ba_polys, sites, ba_id_col, sites_ba_col, mw_col)
    dots = dots[dots["mw_sum"] > 0].copy()
    dots["ms"] = sizes_sqrt(dots["mw_sum"], smin=dot_smin, smax=dot_smax)

    ax.scatter(dots.geometry.x, dots.geometry.y,
               s=dots["ms"], facecolors="white", edgecolors="black",
               linewidths=0.9, alpha=0.92, zorder=6)

    ax.set_title(title, fontsize=13)

    sm = mpl.cm.ScalarMappable(cmap=plt.get_cmap(cmap), norm=norm)
    sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, fraction=0.030, pad=0.015)
    cb.set_label(cbar_label)

    leg1 = legend_sizes(ax, dot_breaks, dot_smin, dot_smax, "Hyperscale capacity (sum MW)", anchor=(1.02, 0.22))
    ax.add_artist(leg1)  # ensure it stays even if other legends added later
    fig.subplots_adjust(right=0.84)

    return savefig(fig, outpath)

# -----------------------
# 2) Fig2: Basin stress (DISCRETE 0–5) + Basin MW dots
# -----------------------
def fig2_basin_stress(
    basins_polys, sites, stress_col, *,
    basin_id_col="HYBAS_ID",
    sites_basin_col="HYBAS_ID",
    mw_col="current_mw",
    states_gdf=None,
    cmap="YlOrRd",
    dot_smin=18, dot_smax=220,
    dot_breaks=(250, 1000, 3000),
    title="Hydrologic stress (Aqueduct baseline) and hyperscale load",
    cbar_label="Aqueduct baseline water stress (0–5)",
    outpath=None
):
    basins_polys, sites, states_gdf = ensure_crs_match(basins_polys, sites, states_gdf)

    fig, ax = plt.subplots(figsize=(12.5, 7.5))
    ax.set_axis_off()

    if stress_col not in basins_polys.columns:
        raise KeyError(f"'{stress_col}' not in basins_polys. Available: {list(basins_polys.columns)}")

    # Discrete bins 0–1–2–3–4–5
    vals = safe_num(basins_polys[stress_col])
    bins = np.array([0, 1, 2, 3, 4, 5])
    cm = plt.get_cmap(cmap, 5)
    norm = BoundaryNorm(bins, cm.N, clip=True)

    miss = basins_polys[vals.isna()]
    if len(miss) > 0:
        miss.plot(ax=ax, color="#eeeeee", linewidth=0, zorder=0)

    basins_polys[vals.notna()].plot(ax=ax, column=stress_col, cmap=cm, norm=norm, linewidth=0, zorder=1)

    basins_polys.boundary.plot(ax=ax, linewidth=0.15, color="#f0f0f0", alpha=0.75, zorder=2)
    if states_gdf is not None:
        states_gdf.boundary.plot(ax=ax, linewidth=0.65, color="#2b2b2b", alpha=0.55, zorder=4)

    dots = aggregate_mw_to_polys(basins_polys, sites, basin_id_col, sites_basin_col, mw_col)
    dots = dots[dots["mw_sum"] > 0].copy()
    dots["ms"] = sizes_sqrt(dots["mw_sum"], smin=dot_smin, smax=dot_smax)

    ax.scatter(dots.geometry.x, dots.geometry.y,
               s=dots["ms"], facecolors="white", edgecolors="black",
               linewidths=0.9, alpha=0.92, zorder=6)

    ax.set_title(title, fontsize=13)

    sm = mpl.cm.ScalarMappable(cmap=cm, norm=norm)
    sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, fraction=0.030, pad=0.015, ticks=[0.5,1.5,2.5,3.5,4.5])
    cb.ax.set_yticklabels(["0–1","1–2","2–3","3–4","4–5"])
    cb.set_label(cbar_label)

    leg1 = legend_sizes(ax, dot_breaks, dot_smin, dot_smax, "Hyperscale capacity (sum MW)", anchor=(1.02, 0.22))
    ax.add_artist(leg1)
    fig.subplots_adjust(right=0.84)

    return savefig(fig, outpath)

# -----------------------
# 3) Fig3: BA hotspot typology map + BA MW dots
#    (robust: computes scope2 from sites if needed)
# -----------------------
def classify_scope2_typology(
    ba_df,
    scope2_col, fossil_share_col, stress_col,
    *,
    scope2_quantile=0.75,
    fossil_thresh=0.50,
    stress_thresh=3.0
):
    df = ba_df.copy()
    s2 = safe_num(df[scope2_col])
    fossil = safe_num(df[fossil_share_col])
    stress = safe_num(df[stress_col])

    qbase = s2[np.isfinite(s2)]
    s2_thr = np.nanquantile(qbase, scope2_quantile) if len(qbase) else np.nan

    hi_s2 = (s2 >= s2_thr)
    hi_fossil = (fossil > fossil_thresh)
    hi_stress = (stress >= stress_thresh)

    typ = np.full(len(df), "None", dtype=object)
    typ[hi_s2 & ~hi_fossil & ~hi_stress] = "High S2 only"
    typ[~hi_s2 & hi_fossil & ~hi_stress] = "Fossil only"
    typ[~hi_s2 & ~hi_fossil & hi_stress] = "High stress only"
    typ[hi_s2 & hi_fossil & ~hi_stress] = "S2+Fossil"
    typ[hi_s2 & ~hi_fossil & hi_stress] = "S2+Stress"
    typ[~hi_s2 & hi_fossil & hi_stress] = "Fossil+Stress"
    typ[hi_s2 & hi_fossil & hi_stress] = "Triple: S2+Fossil+Stress"

    df["typology"] = typ
    df["s2_threshold_used"] = s2_thr
    return df

def fig3_ba_typology(
    ba_polys, sites, *,
    ba_id_col="subregion_id",
    sites_ba_col="subregion_id",
    mw_col="current_mw",
    scope2_col="scope2_GL_yr_base",   # can be missing; we’ll repair
    fossil_col="share_fossil",
    stress_col="mean_bws",
    fossil_is_percent=False,
    scope2_quantile=0.75,
    fossil_thresh=0.50,
    stress_thresh=3.0,
    states_gdf=None,
    dot_smin=18, dot_smax=220,
    dot_breaks=(250, 1000, 3000),
    title="Scope 2 hotspot typology (BA)",
    outpath=None
):
    ba_polys, sites, states_gdf = ensure_crs_match(ba_polys, sites, states_gdf)
    df = ba_polys.copy()

    # If BA scope2 missing, compute from sites and use that
    if scope2_col not in df.columns:
        df = attach_ba_scope2_from_sites(df, sites, ba_id_col, sites_ba_col, site_scope2_col="s2_m3_mid")
        scope2_col = "scope2_m3_mid_from_sites"

    if fossil_is_percent:
        df["_fossil_frac"] = safe_num(df[fossil_col]) / 100.0
        fossil_use = "_fossil_frac"
    else:
        fossil_use = fossil_col

    df2 = classify_scope2_typology(
        df, scope2_col=scope2_col, fossil_share_col=fossil_use, stress_col=stress_col,
        scope2_quantile=scope2_quantile, fossil_thresh=fossil_thresh, stress_thresh=stress_thresh
    )

    palette = {
        "None": "#e6e6e6",
        "High S2 only": "#b3cde3",
        "Fossil only": "#fbb4ae",
        "High stress only": "#ccebc5",
        "S2+Fossil": "#decbe4",
        "S2+Stress": "#fed9a6",
        "Fossil+Stress": "#ffffcc",
        "Triple: S2+Fossil+Stress": "#fb8072",
    }

    fig, ax = plt.subplots(figsize=(12.5, 7.5))
    ax.set_axis_off()

    for k, col in palette.items():
        sub = df2[df2["typology"] == k]
        if len(sub) > 0:
            sub.plot(ax=ax, color=col, linewidth=0, zorder=1)

    draw_outlines(ax, ba_gdf=df2, states_gdf=states_gdf)

    dots = aggregate_mw_to_polys(df2, sites, ba_id_col, sites_ba_col, mw_col)
    dots = dots[dots["mw_sum"] > 0].copy()
    dots["ms"] = sizes_sqrt(dots["mw_sum"], smin=dot_smin, smax=dot_smax)

    ax.scatter(dots.geometry.x, dots.geometry.y,
               s=dots["ms"], facecolors="white", edgecolors="black",
               linewidths=0.9, alpha=0.92, zorder=6)

    ax.set_title(title, fontsize=13)

    order_for_legend = [
        ("Triple: S2+Fossil+Stress", palette["Triple: S2+Fossil+Stress"]),
        ("S2+Fossil", palette["S2+Fossil"]),
        ("S2+Stress", palette["S2+Stress"]),
        ("Fossil+Stress", palette["Fossil+Stress"]),
        ("High S2 only", palette["High S2 only"]),
        ("Fossil only", palette["Fossil only"]),
        ("High stress only", palette["High stress only"]),
        ("None", palette["None"]),
    ]
    legA = add_discrete_legend(ax, order_for_legend, "Typology (BA)", anchor=(1.02, 0.72))
    ax.add_artist(legA)
    legB = legend_sizes(ax, dot_breaks, dot_smin, dot_smax, "Hyperscale capacity (sum MW)", anchor=(1.02, 0.26))
    ax.add_artist(legB)
    fig.subplots_adjust(right=0.80)

    return savefig(fig, outpath)

# -----------------------
# 4) Fig4: Scatter (grid intensity vs stress) with typology colors
#    (robust: computes scope2 from sites if needed)
# -----------------------
def fig4_scatter_tradeoff(
    ba_df, sites=None, *,
    x_col="grid_L_per_kWh",
    y_col="mean_bws",
    mw_col="hyperscale_mw",
    scope2_col="scope2_GL_yr_base",
    fossil_col="share_fossil",
    fossil_is_percent=False,
    scope2_quantile=0.75,
    fossil_thresh=0.50,
    stress_thresh=3.0,
    label_col="subregion_id",
    label_top=12,
    outpath=None,
    title="Grid water intensity vs siting scarcity (BA-level)"
):
    df = ba_df.copy()

    # repair scope2 if missing
    if scope2_col not in df.columns:
        if sites is None:
            raise KeyError(f"'{scope2_col}' missing in ba_df AND sites not provided for repair.")
        df = attach_ba_scope2_from_sites(df, sites, ba_id_col=label_col, sites_ba_col=label_col, site_scope2_col="s2_m3_mid")
        scope2_col = "scope2_m3_mid_from_sites"

    if fossil_is_percent:
        df["_fossil_frac"] = safe_num(df[fossil_col]) / 100.0
        fossil_use = "_fossil_frac"
    else:
        fossil_use = fossil_col

    df2 = classify_scope2_typology(
        df, scope2_col=scope2_col, fossil_share_col=fossil_use, stress_col=y_col,
        scope2_quantile=scope2_quantile, fossil_thresh=fossil_thresh, stress_thresh=stress_thresh
    )

    palette = {
        "None": "#bdbdbd",
        "High S2 only": "#80b1d3",
        "Fossil only": "#fb8072",
        "High stress only": "#8dd3c7",
        "S2+Fossil": "#bebada",
        "S2+Stress": "#fdb462",
        "Fossil+Stress": "#ffffb3",
        "Triple: S2+Fossil+Stress": "#e31a1c",
    }

    fig, ax = plt.subplots(figsize=(10.5, 7.2))

    order = [
        "Triple: S2+Fossil+Stress",
        "S2+Fossil", "S2+Stress", "Fossil+Stress",
        "High S2 only", "Fossil only", "High stress only",
        "None"
    ]
    for k in order:
        sub = df2[df2["typology"] == k]
        if len(sub) == 0:
            continue
        ms = sizes_sqrt(sub[mw_col], smin=28, smax=280)
        ax.scatter(
            safe_num(sub[x_col]), safe_num(sub[y_col]),
            s=ms, c=palette[k],
            edgecolors="black" if k == "Triple: S2+Fossil+Stress" else "#333333",
            linewidths=0.6, alpha=0.85, label=k, zorder=3
        )

    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Grid water intensity (L/kWh)")
    ax.set_ylabel("Mean baseline stress (0–5)")
    ax.grid(True, linewidth=0.4, alpha=0.35)

    # label top by MW
    top = df2.assign(_mw=safe_num(df2[mw_col]).fillna(0)).sort_values("_mw", ascending=False).head(label_top)
    for _, r in top.iterrows():
        ax.text(safe_num(r[x_col]), safe_num(r[y_col]), str(r[label_col]),
                fontsize=8, ha="left", va="bottom")

    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5),
              frameon=True, framealpha=0.95, fontsize=8, title="Typology")
    fig.subplots_adjust(right=0.72)

    return savefig(fig, outpath)

# -----------------------
# 5) One-call wrapper: same signature as yours
# -----------------------
def make_main_figures(
    OUTDIR,
    TAG,
    *,
    ba_polys,
    basins_polys,
    sites_gdf,
    states_gdf=None,
    COL_BA_ID="subregion_id",
    COL_SITES_BA="subregion_id",
    COL_BASIN_ID="HYBAS_ID",
    COL_SITES_BASIN="HYBAS_ID",
    COL_MW="current_mw",
    COL_BA_GRID="grid_L_per_kWh",
    COL_BA_STRESS="mean_bws",
    COL_BASIN_STRESS="bws_01_score",
    COL_BA_SCOPE2="scope2_GL_yr_base",
    COL_BA_FOSSIL="share_fossil",
    fossil_is_percent=False
):
    os.makedirs(OUTDIR, exist_ok=True)

    f1 = fig1_ba_grid_intensity(
        ba_polys, sites_gdf, value_col=COL_BA_GRID,
        ba_id_col=COL_BA_ID, sites_ba_col=COL_SITES_BA, mw_col=COL_MW,
        states_gdf=states_gdf,
        title="Figure 1. Grid water intensity by balancing authority and hyperscale capacity",
        cbar_label="Grid water intensity (L/kWh)",
        outpath=os.path.join(OUTDIR, f"MAIN_Fig1_BA_gridIntensity_plusMW__{TAG}.pdf"),
    )

    f2 = fig2_basin_stress(
        basins_polys, sites_gdf, stress_col=COL_BASIN_STRESS,
        basin_id_col=COL_BASIN_ID, sites_basin_col=COL_SITES_BASIN, mw_col=COL_MW,
        states_gdf=states_gdf,
        title="Figure 2. Basin water stress (Aqueduct) and hyperscale capacity",
        cbar_label="Aqueduct baseline water stress (0–5)",
        outpath=os.path.join(OUTDIR, f"MAIN_Fig2_Basin_stress_plusMW__{TAG}.pdf"),
    )

    f3 = fig3_ba_typology(
        ba_polys, sites_gdf,
        ba_id_col=COL_BA_ID, sites_ba_col=COL_SITES_BA, mw_col=COL_MW,
        scope2_col=COL_BA_SCOPE2, fossil_col=COL_BA_FOSSIL, stress_col=COL_BA_STRESS,
        fossil_is_percent=fossil_is_percent,
        title="Figure 3. Scope 2 hotspot typology by balancing authority",
        outpath=os.path.join(OUTDIR, f"MAIN_Fig3_Scope2_typology__{TAG}.pdf"),
    )

    f4 = fig4_scatter_tradeoff(
        ba_polys, sites=sites_gdf,
        x_col=COL_BA_GRID, y_col=COL_BA_STRESS, mw_col="hyperscale_mw_sites",
        scope2_col=COL_BA_SCOPE2, fossil_col=COL_BA_FOSSIL,
        fossil_is_percent=fossil_is_percent,
        label_col=COL_BA_ID, label_top=12,
        title="Figure 4. Grid water intensity vs siting scarcity (BA-level)",
        outpath=os.path.join(OUTDIR, f"MAIN_Fig4_tradeoff_scatter__{TAG}.pdf"),
    )

    return {"Fig1": f1, "Fig2": f2, "Fig3": f3, "Fig4": f4}


In [ ]:
# redefining the function to not have same naming functions not working weel together

def draw_outlines(ax, ba_gdf=None, states_gdf=None, outline_gdf=None):
    """
    Backward/forward compatible: accepts ba_gdf=... (your new code)
    and outline_gdf=... (older variants). Uses whichever is provided.
    """
    g = ba_gdf if ba_gdf is not None else outline_gdf

    # subtle BA boundaries; clearer state lines
    if g is not None:
        g.boundary.plot(ax=ax, linewidth=0.20, color="#f0f0f0", alpha=0.9, zorder=3)
    if states_gdf is not None:
        states_gdf.boundary.plot(ax=ax, linewidth=0.65, color="#2b2b2b", alpha=0.55, zorder=4)

    ax.set_axis_off()


In [ ]:
def resolve_col(df, candidates, required=True, label=""):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"{label} None of these columns found: {candidates}\nAvailable: {list(df.columns)}")
    return None


def fig3_ba_typology(
    ba_polys, sites, *,
    ba_id_col="subregion_id",
    sites_ba_col="subregion_id",
    mw_col="current_mw",
    scope2_col="scope2_GL_yr_base",   # can be missing; repaired from sites
    fossil_col="share_fossil",
    stress_col="mean_bws",            # may be missing; auto-resolve below
    fossil_is_percent=False,
    scope2_quantile=0.75,
    fossil_thresh=0.50,
    stress_thresh=3.0,
    states_gdf=None,
    dot_smin=18, dot_smax=220,
    dot_breaks=(250, 1000, 3000),
    title="Scope 2 hotspot typology (BA)",
    outpath=None
):
    ba_polys, sites, states_gdf = ensure_crs_match(ba_polys, sites, states_gdf)
    df = ba_polys.copy()

    # ---- scope2 repair if missing
    if scope2_col not in df.columns:
        df = attach_ba_scope2_from_sites(df, sites, ba_id_col, sites_ba_col, site_scope2_col="s2_m3_mid")
        scope2_col = "scope2_m3_mid_from_sites"

    # ---- fossil col resolve (in case name changes)
    fossil_col = resolve_col(df, [fossil_col, "share_fossil", "fossil_share"], required=True, label="fossil_col:")

    # ---- stress col resolve (THIS fixes your error)
    stress_col = resolve_col(df, [stress_col, "mean_bws", "baseline_stress", "mean_stress", "bws_01_score"],
                             required=True, label="stress_col:")

    # fossil percent handling
    if fossil_is_percent:
        df["_fossil_frac"] = safe_num(df[fossil_col]) / 100.0
        fossil_use = "_fossil_frac"
    else:
        fossil_use = fossil_col

    df2 = classify_scope2_typology(
        df, scope2_col=scope2_col, fossil_share_col=fossil_use, stress_col=stress_col,
        scope2_quantile=scope2_quantile, fossil_thresh=fossil_thresh, stress_thresh=stress_thresh
    )

    palette = {
        "None": "#e6e6e6",
        "High S2 only": "#b3cde3",
        "Fossil only": "#fbb4ae",
        "High stress only": "#ccebc5",
        "S2+Fossil": "#decbe4",
        "S2+Stress": "#fed9a6",
        "Fossil+Stress": "#ffffcc",
        "Triple: S2+Fossil+Stress": "#fb8072",
    }

    fig, ax = plt.subplots(figsize=(12.5, 7.5))
    ax.set_axis_off()

    for k, col in palette.items():
        sub = df2[df2["typology"] == k]
        if len(sub) > 0:
            sub.plot(ax=ax, color=col, linewidth=0, zorder=1)

    draw_outlines(ax, ba_gdf=df2, states_gdf=states_gdf)

    dots = aggregate_mw_to_polys(df2, sites, ba_id_col, sites_ba_col, mw_col)
    dots = dots[dots["mw_sum"] > 0].copy()
    dots["ms"] = sizes_sqrt(dots["mw_sum"], smin=dot_smin, smax=dot_smax)

    ax.scatter(dots.geometry.x, dots.geometry.y,
               s=dots["ms"], facecolors="white", edgecolors="black",
               linewidths=0.9, alpha=0.92, zorder=6)

    ax.set_title(title, fontsize=13)

    order_for_legend = [
        ("Triple: S2+Fossil+Stress", palette["Triple: S2+Fossil+Stress"]),
        ("S2+Fossil", palette["S2+Fossil"]),
        ("S2+Stress", palette["S2+Stress"]),
        ("Fossil+Stress", palette["Fossil+Stress"]),
        ("High S2 only", palette["High S2 only"]),
        ("Fossil only", palette["Fossil only"]),
        ("High stress only", palette["High stress only"]),
        ("None", palette["None"]),
    ]
    legA = add_discrete_legend(ax, order_for_legend, "Typology (BA)", anchor=(1.02, 0.72))
    ax.add_artist(legA)
    legB = legend_sizes(ax, dot_breaks, dot_smin, dot_smax, "Hyperscale capacity (sum MW)", anchor=(1.02, 0.26))
    ax.add_artist(legB)
    fig.subplots_adjust(right=0.80)

    return savefig(fig, outpath)


def fig4_scatter_tradeoff(
    ba_df, sites=None, *,
    x_col="grid_L_per_kWh",
    y_col="mean_bws",                 # may be missing; auto-resolve below
    mw_col="hyperscale_mw",
    scope2_col="scope2_GL_yr_base",
    fossil_col="share_fossil",
    fossil_is_percent=False,
    scope2_quantile=0.75,
    fossil_thresh=0.50,
    stress_thresh=3.0,
    label_col="subregion_id",
    label_top=12,
    outpath=None,
    title="Grid water intensity vs siting scarcity (BA-level)"
):
    df = ba_df.copy()

    # resolve x / y / fossil robustly
    x_col = resolve_col(df, [x_col, "grid_L_per_kWh", "grid_L_per_kwh"], required=True, label="x_col:")
    y_col = resolve_col(df, [y_col, "mean_bws", "baseline_stress", "mean_stress", "bws_01_score"], required=True, label="y_col:")
    fossil_col = resolve_col(df, [fossil_col, "share_fossil", "fossil_share"], required=True, label="fossil_col:")

    # repair scope2 if missing
    if scope2_col not in df.columns:
        if sites is None:
            raise KeyError(f"'{scope2_col}' missing in ba_df AND sites not provided for repair.")
        df = attach_ba_scope2_from_sites(df, sites, ba_id_col=label_col, sites_ba_col=label_col, site_scope2_col="s2_m3_mid")
        scope2_col = "scope2_m3_mid_from_sites"

    if fossil_is_percent:
        df["_fossil_frac"] = safe_num(df[fossil_col]) / 100.0
        fossil_use = "_fossil_frac"
    else:
        fossil_use = fossil_col

    df2 = classify_scope2_typology(
        df, scope2_col=scope2_col, fossil_share_col=fossil_use, stress_col=y_col,
        scope2_quantile=scope2_quantile, fossil_thresh=fossil_thresh, stress_thresh=stress_thresh
    )

    palette = {
        "None": "#bdbdbd",
        "High S2 only": "#80b1d3",
        "Fossil only": "#fb8072",
        "High stress only": "#8dd3c7",
        "S2+Fossil": "#bebada",
        "S2+Stress": "#fdb462",
        "Fossil+Stress": "#ffffb3",
        "Triple: S2+Fossil+Stress": "#e31a1c",
    }

    fig, ax = plt.subplots(figsize=(10.5, 7.2))

    order = [
        "Triple: S2+Fossil+Stress",
        "S2+Fossil", "S2+Stress", "Fossil+Stress",
        "High S2 only", "Fossil only", "High stress only",
        "None"
    ]
    for k in order:
        sub = df2[df2["typology"] == k]
        if len(sub) == 0:
            continue
        ms = sizes_sqrt(sub[mw_col], smin=28, smax=280)
        ax.scatter(
            safe_num(sub[x_col]), safe_num(sub[y_col]),
            s=ms, c=palette[k],
            edgecolors="black" if k == "Triple: S2+Fossil+Stress" else "#333333",
            linewidths=0.6, alpha=0.85, label=k, zorder=3
        )

    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Grid water intensity (L/kWh)")
    ax.set_ylabel("Mean baseline stress (0–5)")
    ax.grid(True, linewidth=0.4, alpha=0.35)

    top = df2.assign(_mw=safe_num(df2[mw_col]).fillna(0)).sort_values("_mw", ascending=False).head(label_top)
    for _, r in top.iterrows():
        ax.text(safe_num(r[x_col]), safe_num(r[y_col]), str(r[label_col]),
                fontsize=8, ha="left", va="bottom")

    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5),
              frameon=True, framealpha=0.95, fontsize=8, title="Typology")
    fig.subplots_adjust(right=0.72)

    return savefig(fig, outpath)


In [ ]:
def attach_ba_stress_from_sites(
    ba_polys, sites,
    ba_id_col="subregion_id",
    sites_ba_col="subregion_id",
    site_stress_col="bws_01_score",
    weight_col="current_mw",
    out_col="mean_bws_mw"
):
    ba = ba_polys.copy()
    s = sites.copy()

    # keys
    ba[ba_id_col] = ba[ba_id_col].astype(str)
    s[sites_ba_col] = s[sites_ba_col].astype(str)

    if site_stress_col not in s.columns:
        raise KeyError(f"sites missing '{site_stress_col}'. Available: {list(s.columns)}")

    w = safe_num(s[weight_col]).fillna(0).clip(lower=0) if weight_col in s.columns else pd.Series(1.0, index=s.index)
    x = safe_num(s[site_stress_col])

    tmp = pd.DataFrame({
        "ba": s[sites_ba_col],
        "stress": x,
        "w": w
    }).dropna(subset=["stress"])

    if len(tmp) == 0:
        ba[out_col] = np.nan
        return ba

    # MW-weighted mean; fallback to unweighted if all weights are zero in a BA
    tmp["wx"] = tmp["stress"] * tmp["w"]
    g = tmp.groupby("ba", as_index=False).agg(wsum=("w", "sum"), wxsum=("wx", "sum"), n=("stress", "size"))
    g[out_col] = np.where(g["wsum"] > 0, g["wxsum"] / g["wsum"], g["wxsum"] / g["n"])

    g = g.rename(columns={"ba": ba_id_col})[[ba_id_col, out_col]]
    ba = ba.merge(g, on=ba_id_col, how="left")
    return ba


In [ ]:
# 1) add BA stress derived from sites
ba_p = attach_ba_stress_from_sites(
    ba_p, sites_p,
    ba_id_col="subregion_id",
    sites_ba_col="subregion_id",
    site_stress_col="bws_01_score",
    weight_col="current_mw",
    out_col="mean_bws"   # <- name it mean_bws so the rest of your code works
)

# sanity check
print("BA stress column exists:", "mean_bws" in ba_p.columns, " | non-null:", ba_p["mean_bws"].notna().sum())


In [ ]:

OUTDIR = "figures_main"
TAG = "base"

paths = make_main_figures(
    OUTDIR, TAG,
    ba_polys=ba_p,
    basins_polys=basins_p,
    sites_gdf=sites_p,
    states_gdf=states_p,
    COL_BA_ID="subregion_id",
    COL_SITES_BA="subregion_id",
    COL_BASIN_ID="HYBAS_ID",
    COL_SITES_BASIN="pfaf_id",
    COL_MW="current_mw",
    COL_BA_GRID="grid_L_per_kWh",
    COL_BA_STRESS='mean_bws',
    COL_BASIN_STRESS="bws_01_score",
    COL_BA_SCOPE2="scope2_GL_yr_base",   # ok if missing now
    COL_BA_FOSSIL="share_fossil",
    fossil_is_percent=False
)

paths


In [ ]:
# ============================================
# Nature-ready Fig1a/Fig1b (muted palettes, consistent dots, US clipping)
# Works with your existing objects: ba_p, basins_p, sites_p, states_p
# ============================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# -----------------------
# Helpers (keep robust)
# -----------------------
def safe_num(x):
    return pd.to_numeric(x, errors="coerce")

def ensure_crs_match(*gdfs):
    base = None
    for g in gdfs:
        if g is not None and hasattr(g, "crs") and g.crs is not None:
            base = g.crs
            break
    if base is None:
        return gdfs
    out = []
    for g in gdfs:
        if g is None:
            out.append(None)
        else:
            out.append(g.to_crs(base) if g.crs != base else g)
    return out

def representative_points(poly_gdf):
    rp = poly_gdf.geometry.representative_point()
    return gpd.GeoDataFrame(poly_gdf.drop(columns="geometry"), geometry=rp, crs=poly_gdf.crs)

def aggregate_mw_to_polys(poly_gdf, sites_gdf, poly_id_col, sites_poly_id_col, mw_col):
    # coerce keys to same dtype to avoid int/object merge errors
    poly = poly_gdf[[poly_id_col, "geometry"]].copy()
    sites = sites_gdf[[sites_poly_id_col, mw_col]].copy()
    poly[poly_id_col] = poly[poly_id_col].astype(str)
    sites[sites_poly_id_col] = sites[sites_poly_id_col].astype(str)

    agg = (sites.assign(_mw=safe_num(sites[mw_col]).fillna(0).clip(lower=0))
                .groupby(sites_poly_id_col, as_index=False)["_mw"].sum()
                .rename(columns={sites_poly_id_col: poly_id_col, "_mw": "mw_sum"}))

    out = poly.merge(agg, on=poly_id_col, how="left")
    out["mw_sum"] = out["mw_sum"].fillna(0)
    return representative_points(out)

def sizes_sqrt_global(values, vmin_ref, vmax_ref, smin=14, smax=260):
    """
    Sqrt scaling, but using GLOBAL reference min/max so dots are consistent across figs.
    """
    v = safe_num(values).fillna(0).clip(lower=0).to_numpy()
    if vmax_ref <= vmin_ref:
        return np.full_like(v, smin, dtype=float)
    z = np.sqrt(np.clip(v, vmin_ref, vmax_ref))
    z0 = np.sqrt(vmin_ref)
    z1 = np.sqrt(vmax_ref)
    t = (z - z0) / (z1 - z0 + 1e-12)
    return smin + (smax - smin) * np.clip(t, 0, 1)

def truncate_cmap(cmap_name="cividis", lo=0.18, hi=0.88, n=256):
    """
    Muted version of a colormap (less contrast / fewer saturated extremes).
    """
    cmap = plt.get_cmap(cmap_name)
    colors = cmap(np.linspace(lo, hi, n))
    return mpl.colors.ListedColormap(colors)

def apply_quantile_norm(vals, qlow=0.02, qhigh=0.98):
    vals = safe_num(vals)
    if np.isfinite(vals).any():
        vmin = np.nanquantile(vals, qlow)
        vmax = np.nanquantile(vals, qhigh)
        if not np.isfinite(vmin): vmin = np.nanmin(vals)
        if not np.isfinite(vmax): vmax = np.nanmax(vals)
        if vmax <= vmin:
            vmax = vmin + 1e-12
        return mpl.colors.Normalize(vmin=vmin, vmax=vmax), float(vmin), float(vmax)
    return mpl.colors.Normalize(vmin=0, vmax=1), 0.0, 1.0

def rasterize_first_collection(gdf_plot_return):
    try:
        gdf_plot_return.collections[0].set_rasterized(True)
    except Exception:
        pass

def legend_sizes_outside(ax, breaks_mw, smin, smax, vmin_ref, vmax_ref, title, anchor=(1.02, 0.23)):
    svals = sizes_sqrt_global(pd.Series(breaks_mw), vmin_ref, vmax_ref, smin=smin, smax=smax)
    handles = [
        Line2D([0],[0], marker='o', linestyle='None',
               markerfacecolor="white", markeredgecolor="black",
               markeredgewidth=0.9, alpha=0.95,
               markersize=np.sqrt(s), label=f"{b:,} MW")
        for b, s in zip(breaks_mw, svals)
    ]
    leg = ax.legend(handles=handles, title=title, loc="center left",
                    bbox_to_anchor=anchor, frameon=True, framealpha=0.95,
                    fontsize=8.5, title_fontsize=9.5, borderpad=0.6)
    leg.get_frame().set_linewidth(0.6)
    return leg

def draw_states(ax, states_gdf, lw=0.55, color="#2b2b2b", alpha=0.55, z=6):
    if states_gdf is not None:
        states_gdf.boundary.plot(ax=ax, linewidth=lw, color=color, alpha=alpha, zorder=z)

def savefig(fig, outpath):
    if outpath:
        os.makedirs(os.path.dirname(outpath), exist_ok=True)
        fig.savefig(outpath, bbox_inches="tight", dpi=300)
    return outpath

# -----------------------
# Compute global dot scaling (consistent across Fig1 & Fig2)
# -----------------------
def compute_global_dot_reference(
    ba_polys, basins_polys, sites_gdf,
    ba_id_col, sites_ba_col,
    basin_id_col, sites_basin_col,
    mw_col,
    clip_basins_to_us=True,
    states_gdf=None
):
    ba_polys, basins_polys, sites_gdf, states_gdf = ensure_crs_match(ba_polys, basins_polys, sites_gdf, states_gdf)

    # optional clip basins to US footprint for dot aggregation consistency
    if clip_basins_to_us and states_gdf is not None:
        us_geom = states_gdf.unary_union
        try:
            basins_use = gpd.clip(basins_polys, us_geom)
        except Exception:
            basins_use = basins_polys
    else:
        basins_use = basins_polys

    dots_ba = aggregate_mw_to_polys(ba_polys, sites_gdf, ba_id_col, sites_ba_col, mw_col)
    dots_basin = aggregate_mw_to_polys(basins_use, sites_gdf, basin_id_col, sites_basin_col, mw_col)

    all_mw = pd.concat([dots_ba["mw_sum"], dots_basin["mw_sum"]], ignore_index=True)
    all_mw = safe_num(all_mw).fillna(0).clip(lower=0)

    # Use robust upper reference to avoid one giant region dominating
    vmax_ref = float(np.nanquantile(all_mw, 0.98)) if np.isfinite(all_mw).any() else 1.0
    vmin_ref = 0.0
    if vmax_ref <= 0:
        vmax_ref = 1.0
    return vmin_ref, vmax_ref

# -----------------------
# Fig1a: BA choropleth (grid intensity) + one dot per BA
# -----------------------
def fig1a_ba_grid_intensity(
    ba_polys, sites, *,
    value_col,
    ba_id_col="subregion_id",
    sites_ba_col="subregion_id",
    mw_col="current_mw",
    states_gdf=None,
    dot_breaks=(250, 1000, 3000),
    dot_smin=14, dot_smax=260,
    dot_vmin_ref=0.0, dot_vmax_ref=3000.0,
    cmap_name="cividis",
    cmap_lo=0.22, cmap_hi=0.85,
    qlow=0.02, qhigh=0.98,
    title="Figure 1a. Grid water intensity by balancing authority and hyperscale capacity",
    cbar_label="Grid water intensity (L/kWh)",
    outpath=None
):
    ba_polys, sites, states_gdf = ensure_crs_match(ba_polys, sites, states_gdf)

    fig, ax = plt.subplots(figsize=(12.8, 7.4))
    ax.set_axis_off()

    norm, vmin, vmax = apply_quantile_norm(ba_polys[value_col], qlow, qhigh)
    cmap = truncate_cmap(cmap_name, lo=cmap_lo, hi=cmap_hi)

    # choropleth (no hard borders)
    ba_polys.plot(ax=ax, column=value_col, cmap=cmap, linewidth=0, norm=norm, zorder=1)

    # subtle BA outline only if you want; Nature often prefers minimal internal boundaries
    ba_polys.boundary.plot(ax=ax, linewidth=0.18, color="white", alpha=0.35, zorder=2)

    # states: clear US context
    draw_states(ax, states_gdf, lw=0.65, alpha=0.55, z=6)

    # dots
    dots = aggregate_mw_to_polys(ba_polys, sites, ba_id_col, sites_ba_col, mw_col)
    dots = dots[dots["mw_sum"] > 0].copy()
    dots["ms"] = sizes_sqrt_global(dots["mw_sum"], dot_vmin_ref, dot_vmax_ref, smin=dot_smin, smax=dot_smax)

    p = dots.plot(ax=ax, markersize=dots["ms"],
                  facecolor="white", edgecolor="black",
                  linewidth=0.9, alpha=0.93, zorder=8)
    rasterize_first_collection(p)

    ax.set_title(title, fontsize=13)

    # colorbar (kept slim, right side)
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, fraction=0.028, pad=0.012)
    cb.set_label(cbar_label, fontsize=10)

    # dot legend outside, below the colorbar (no overlap)
    legend_sizes_outside(
        ax, dot_breaks, dot_smin, dot_smax, dot_vmin_ref, dot_vmax_ref,
        title="Hyperscale capacity (sum MW)", anchor=(1.02, 0.22)
    )

    fig.subplots_adjust(right=0.83)
    return savefig(fig, outpath)

# -----------------------
# Fig1b: Basin choropleth (stress) + one dot per basin, clipped to US
# -----------------------
def fig1b_basin_stress(
    basins_polys, sites, *,
    stress_col,
    basin_id_col="HYBAS_ID",
    sites_basin_col="pfaf_id",
    mw_col="current_mw",
    states_gdf=None,
    clip_to_us=True,
    dot_breaks=(250, 1000, 3000),
    dot_smin=14, dot_smax=260,
    dot_vmin_ref=0.0, dot_vmax_ref=3000.0,
    cmap_name="cividis",
    cmap_lo=0.22, cmap_hi=0.85,
    qlow=0.02, qhigh=0.98,
    title="Figure 1b. Basin water stress (Aqueduct) and hyperscale capacity",
    cbar_label="Aqueduct baseline water stress (0–5)",
    outpath=None
):
    basins_polys, sites, states_gdf = ensure_crs_match(basins_polys, sites, states_gdf)

    # clip to US footprint to avoid showing Canada/Mexico basin pieces
    basins_use = basins_polys
    if clip_to_us and states_gdf is not None:
        us_geom = states_gdf.unary_union
        try:
            basins_use = gpd.clip(basins_polys, us_geom)
        except Exception:
            basins_use = basins_polys  # fallback

    fig, ax = plt.subplots(figsize=(12.8, 7.4))
    ax.set_axis_off()

    norm, vmin, vmax = apply_quantile_norm(basins_use[stress_col], qlow, qhigh)
    cmap = truncate_cmap(cmap_name, lo=cmap_lo, hi=cmap_hi)

    # choropleth
    basins_use.plot(ax=ax, column=stress_col, cmap=cmap, linewidth=0, norm=norm, zorder=1)

    # faint basin boundaries to show structure without clutter
    basins_use.boundary.plot(ax=ax, linewidth=0.12, color="white", alpha=0.30, zorder=2)

    # states: make US outline unambiguous
    draw_states(ax, states_gdf, lw=0.70, alpha=0.60, z=7)

    # dots
    dots = aggregate_mw_to_polys(basins_use, sites, basin_id_col, sites_basin_col, mw_col)
    dots = dots[dots["mw_sum"] > 0].copy()
    dots["ms"] = sizes_sqrt_global(dots["mw_sum"], dot_vmin_ref, dot_vmax_ref, smin=dot_smin, smax=dot_smax)

    p = dots.plot(ax=ax, markersize=dots["ms"],
                  facecolor="white", edgecolor="black",
                  linewidth=0.9, alpha=0.93, zorder=8)
    rasterize_first_collection(p)

    ax.set_title(title, fontsize=13)

    # colorbar
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, fraction=0.028, pad=0.012)
    cb.set_label(cbar_label, fontsize=10)

    # dot legend outside
    legend_sizes_outside(
        ax, dot_breaks, dot_smin, dot_smax, dot_vmin_ref, dot_vmax_ref,
        title="Hyperscale capacity (sum MW)", anchor=(1.02, 0.22)
    )

    fig.subplots_adjust(right=0.83)
    return savefig(fig, outpath)

# -----------------------
# One-call wrapper for your “Figure 1a / 1b”
# -----------------------
def make_fig1ab_nature(
    OUTDIR, TAG, *,
    ba_polys, basins_polys, sites_gdf, states_gdf,
    COL_BA_ID="subregion_id",
    COL_SITES_BA="subregion_id",
    COL_BASIN_ID="HYBAS_ID",
    COL_SITES_BASIN="pfaf_id",
    COL_MW="current_mw",
    COL_BA_GRID="grid_L_per_kWh",
    COL_BASIN_STRESS="bws_01_score",
):
    os.makedirs(OUTDIR, exist_ok=True)

    # global dot scaling reference so dots match between Fig1a and Fig1b
    dot_vmin_ref, dot_vmax_ref = compute_global_dot_reference(
        ba_polys, basins_polys, sites_gdf,
        ba_id_col=COL_BA_ID, sites_ba_col=COL_SITES_BA,
        basin_id_col=COL_BASIN_ID, sites_basin_col=COL_SITES_BASIN,
        mw_col=COL_MW,
        clip_basins_to_us=True,
        states_gdf=states_gdf
    )

    f1a = os.path.join(OUTDIR, f"MAIN_Fig1a_BA_gridIntensity_plusMW__{TAG}.pdf")
    f1b = os.path.join(OUTDIR, f"MAIN_Fig1b_Basin_stress_plusMW__{TAG}.pdf")

    fig1a_ba_grid_intensity(
        ba_polys, sites_gdf,
        value_col=COL_BA_GRID,
        ba_id_col=COL_BA_ID, sites_ba_col=COL_SITES_BA,
        mw_col=COL_MW,
        states_gdf=states_gdf,
        dot_vmin_ref=dot_vmin_ref, dot_vmax_ref=dot_vmax_ref,
        outpath=f1a
    )

    fig1b_basin_stress(
        basins_polys, sites_gdf,
        stress_col=COL_BASIN_STRESS,
        basin_id_col=COL_BASIN_ID, sites_basin_col=COL_SITES_BASIN,
        mw_col=COL_MW,
        states_gdf=states_gdf,
        clip_to_us=True,
        dot_vmin_ref=dot_vmin_ref, dot_vmax_ref=dot_vmax_ref,
        outpath=f1b
    )

    return {"Fig1a": f1a, "Fig1b": f1b, "dot_vmax_ref_used": dot_vmax_ref}


# =======================
# Example call (your cols)
# =======================
OUTDIR = "figures_main_nature"
TAG = "base"
paths_1ab = make_fig1ab_nature(
     OUTDIR, TAG,
     ba_polys=ba_p,
     basins_polys=basins_p,
     sites_gdf=sites_p,
     states_gdf=states_p,
     COL_BA_ID="subregion_id",
     COL_SITES_BA="subregion_id",
     COL_BASIN_ID="HYBAS_ID",
     COL_SITES_BASIN="pfaf_id",
     COL_MW="current_mw",
     COL_BA_GRID="grid_L_per_kWh",
     COL_BASIN_STRESS="bws_01_score",
 )
paths_1ab


In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# -----------------------
# helpers
# -----------------------
def safe_num(x):
    return pd.to_numeric(x, errors="coerce")

def ensure_crs_match(*gdfs):
    base = None
    for g in gdfs:
        if g is not None and hasattr(g, "crs") and g.crs is not None:
            base = g.crs
            break
    if base is None:
        return gdfs
    out = []
    for g in gdfs:
        if g is None:
            out.append(None)
        else:
            out.append(g.to_crs(base) if g.crs != base else g)
    return out

def truncate_cmap(cmap_name="cividis", lo=0.22, hi=0.85, n=256):
    cmap = plt.get_cmap(cmap_name)
    return mpl.colors.ListedColormap(cmap(np.linspace(lo, hi, n)))

def apply_quantile_norm(vals, qlow=0.02, qhigh=0.98):
    v = safe_num(vals)
    if np.isfinite(v).any():
        vmin = float(np.nanquantile(v, qlow))
        vmax = float(np.nanquantile(v, qhigh))
        if not np.isfinite(vmin): vmin = float(np.nanmin(v))
        if not np.isfinite(vmax): vmax = float(np.nanmax(v))
        if vmax <= vmin: vmax = vmin + 1e-12
        return mpl.colors.Normalize(vmin=vmin, vmax=vmax), vmin, vmax
    return mpl.colors.Normalize(vmin=0, vmax=1), 0.0, 1.0

def sizes_sqrt_global(values, vmin_ref, vmax_ref, smin=6, smax=55):
    """marker AREAS in points^2 (good for GeoPandas markersize / ax.scatter s=)."""
    v = safe_num(values).fillna(0).clip(lower=0).to_numpy()
    vmax_ref = max(float(vmax_ref), 1e-9)
    v = np.clip(v, 0, vmax_ref)
    z = np.sqrt(v)
    z0, z1 = 0.0, np.sqrt(vmax_ref)
    t = (z - z0) / (z1 - z0 + 1e-12)
    return smin + (smax - smin) * np.clip(t, 0, 1)

def add_size_legend(ax, breaks_mw, vmin_ref, vmax_ref, smin, smax,
                    title="Facility load (MW)", anchor=(1.02, 0.22)):
    svals = sizes_sqrt_global(pd.Series(breaks_mw), vmin_ref, vmax_ref, smin=smin, smax=smax)
    handles = [
        Line2D([0],[0], marker='o', linestyle='None',
               markerfacecolor="black", markeredgecolor="black",
               alpha=0.35, markeredgewidth=0.0,
               markersize=np.sqrt(s),  # markersize in points, s is points^2
               label=f"{b:g} MW")
        for b, s in zip(breaks_mw, svals)
    ]
    leg = ax.legend(handles=handles, title=title, loc="center left",
                    bbox_to_anchor=anchor, frameon=True, framealpha=0.95,
                    fontsize=8.5, title_fontsize=9.5, borderpad=0.6)
    leg.get_frame().set_linewidth(0.6)
    return leg

def rasterize_first_collection(gdf_plot_return):
    try:
        gdf_plot_return.collections[0].set_rasterized(True)
    except Exception:
        pass

def draw_states(ax, states_gdf, lw=0.70, color="#2b2b2b", alpha=0.60, z=10):
    if states_gdf is not None:
        states_gdf.boundary.plot(ax=ax, linewidth=lw, color=color, alpha=alpha, zorder=z)

def savefig(fig, outpath):
    if outpath:
        os.makedirs(os.path.dirname(outpath), exist_ok=True)
        fig.savefig(outpath, bbox_inches="tight", dpi=300)
    return outpath

def compute_global_site_mw_ref(sites_gdf, mw_col="current_mw", q=0.99):
    mw = safe_num(sites_gdf[mw_col]).fillna(0).clip(lower=0)
    vmax = float(np.nanquantile(mw, q)) if np.isfinite(mw).any() else 1.0
    return 0.0, max(vmax, 1.0)

# -----------------------
# Fig1a: BA choropleth + MANY site dots
# -----------------------
def fig1a_ba_grid_intensity_sites(
    ba_polys, sites_gdf, *,
    value_col="grid_L_per_kWh",
    mw_col="current_mw",
    states_gdf=None,
    cmap_name="cividis", cmap_lo=0.22, cmap_hi=0.85,
    qlow=0.02, qhigh=0.98,
    dot_smin=6, dot_smax=55,
    dot_breaks=(25, 100, 300),
    dot_vmin_ref=0.0, dot_vmax_ref=300.0,
    title="Figure 1a. Grid water intensity by balancing authority and hyperscale sites",
    cbar_label="Grid water intensity (L/kWh)",
    outpath=None
):
    ba_polys, sites_gdf, states_gdf = ensure_crs_match(ba_polys, sites_gdf, states_gdf)

    fig, ax = plt.subplots(figsize=(12.8, 7.4))
    ax.set_axis_off()

    norm, vmin, vmax = apply_quantile_norm(ba_polys[value_col], qlow, qhigh)
    cmap = truncate_cmap(cmap_name, lo=cmap_lo, hi=cmap_hi)

    ba_polys.plot(ax=ax, column=value_col, cmap=cmap, linewidth=0, norm=norm, zorder=1)
    ba_polys.boundary.plot(ax=ax, linewidth=0.18, color="white", alpha=0.30, zorder=2)
    draw_states(ax, states_gdf)

    ms = sizes_sqrt_global(sites_gdf[mw_col], dot_vmin_ref, dot_vmax_ref, smin=dot_smin, smax=dot_smax)

    # Plot sites (GeoPandas so we can rasterize easily)
    p = sites_gdf.plot(ax=ax, markersize=ms, color="black", alpha=0.30, linewidth=0, zorder=8)
    rasterize_first_collection(p)

    ax.set_title(title, fontsize=13)

    # Colorbar (top-right-ish)
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, fraction=0.028, pad=0.012)
    cb.set_label(cbar_label, fontsize=10)

    # Dot-size legend BELOW colorbar (no overlap)
    add_size_legend(ax, dot_breaks, dot_vmin_ref, dot_vmax_ref, dot_smin, dot_smax,
                    title="Facility load (MW)", anchor=(1.02, 0.22))

    fig.subplots_adjust(right=0.83)
    return savefig(fig, outpath)

# -----------------------
# Fig1b: Basin choropleth + MANY site dots (clip to US)
# -----------------------
def fig1b_basin_stress_sites(
    basins_polys, sites_gdf, *,
    stress_col="bws_01_score",
    mw_col="current_mw",
    states_gdf=None,
    clip_to_us=True,
    cmap_name="cividis", cmap_lo=0.22, cmap_hi=0.85,
    qlow=0.02, qhigh=0.98,
    dot_smin=6, dot_smax=55,
    dot_breaks=(25, 100, 300),
    dot_vmin_ref=0.0, dot_vmax_ref=300.0,
    title="Figure 1b. Basin water stress (Aqueduct) and hyperscale sites",
    cbar_label="Aqueduct baseline water stress (0–5)",
    outpath=None
):
    basins_polys, sites_gdf, states_gdf = ensure_crs_match(basins_polys, sites_gdf, states_gdf)

    basins_use = basins_polys
    if clip_to_us and states_gdf is not None:
        us_geom = states_gdf.unary_union
        try:
            basins_use = gpd.clip(basins_polys, us_geom)
        except Exception:
            basins_use = basins_polys

    fig, ax = plt.subplots(figsize=(12.8, 7.4))
    ax.set_axis_off()

    norm, vmin, vmax = apply_quantile_norm(basins_use[stress_col], qlow, qhigh)
    cmap = truncate_cmap(cmap_name, lo=cmap_lo, hi=cmap_hi)

    basins_use.plot(ax=ax, column=stress_col, cmap=cmap, linewidth=0, norm=norm, zorder=1)
    basins_use.boundary.plot(ax=ax, linewidth=0.12, color="white", alpha=0.28, zorder=2)
    draw_states(ax, states_gdf)

    ms = sizes_sqrt_global(sites_gdf[mw_col], dot_vmin_ref, dot_vmax_ref, smin=dot_smin, smax=dot_smax)
    p = sites_gdf.plot(ax=ax, markersize=ms, color="black", alpha=0.30, linewidth=0, zorder=8)
    rasterize_first_collection(p)

    ax.set_title(title, fontsize=13)

    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, fraction=0.028, pad=0.012)
    cb.set_label(cbar_label, fontsize=10)

    add_size_legend(ax, dot_breaks, dot_vmin_ref, dot_vmax_ref, dot_smin, dot_smax,
                    title="Facility load (MW)", anchor=(1.02, 0.22))

    fig.subplots_adjust(right=0.83)
    return savefig(fig, outpath)

# -----------------------
# one-call wrapper
# -----------------------
def make_fig1ab_sites(
    OUTDIR, TAG, *,
    ba_polys, basins_polys, sites_gdf, states_gdf,
    COL_MW="current_mw",
    COL_BA_GRID="grid_L_per_kWh",
    COL_BASIN_STRESS="bws_01_score",
    dot_smin=6, dot_smax=55,
    dot_breaks=(25, 100, 300),
):
    os.makedirs(OUTDIR, exist_ok=True)

    # global dot scaling reference from *sites* so Fig1a and Fig1b match
    vmin_ref, vmax_ref = compute_global_site_mw_ref(sites_gdf, mw_col=COL_MW, q=0.99)

    f1a = os.path.join(OUTDIR, f"MAIN_Fig1a_BA_gridIntensity_sites__{TAG}.pdf")
    f1b = os.path.join(OUTDIR, f"MAIN_Fig1b_Basin_stress_sites__{TAG}.pdf")

    fig1a_ba_grid_intensity_sites(
        ba_polys, sites_gdf,
        value_col=COL_BA_GRID, mw_col=COL_MW, states_gdf=states_gdf,
        dot_smin=dot_smin, dot_smax=dot_smax,
        dot_breaks=dot_breaks, dot_vmin_ref=vmin_ref, dot_vmax_ref=vmax_ref,
        outpath=f1a
    )

    fig1b_basin_stress_sites(
        basins_polys, sites_gdf,
        stress_col=COL_BASIN_STRESS, mw_col=COL_MW, states_gdf=states_gdf,
        dot_smin=dot_smin, dot_smax=dot_smax,
        dot_breaks=dot_breaks, dot_vmin_ref=vmin_ref, dot_vmax_ref=vmax_ref,
        outpath=f1b
    )

    return {"Fig1a": f1a, "Fig1b": f1b, "dot_vmax_ref_used": vmax_ref}


In [ ]:
OUTDIR = "figures_main_nature"
TAG = "base"

paths = make_fig1ab_sites(
    OUTDIR, TAG,
    ba_polys=ba_p,
    basins_polys=basins_p,
    sites_gdf=sites_p,
    states_gdf=states_p,
    COL_MW="current_mw",
    COL_BA_GRID="grid_L_per_kWh",
    COL_BASIN_STRESS="bws_01_score",
    dot_smin=6, dot_smax=55,
    dot_breaks=(25, 100, 300),
)
paths

In [ ]:
# ============================================
# Nature/Science-ready Fig1a–Fig1b (sites dots)
# - muted palettes
# - identical dot scaling + legend in both figs
# - Fig1b clipped to US extent (no extra world)
# - non-overlapping legends, rasterized points
# ============================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from shapely.geometry import box

# -----------------------
# 0) Helpers (keep yours)
# -----------------------
def safe_num(x):
    return pd.to_numeric(x, errors="coerce")

def ensure_crs_match(*gdfs):
    base = None
    for g in gdfs:
        if g is not None and hasattr(g, "crs") and g.crs is not None:
            base = g.crs
            break
    if base is None:
        return gdfs
    out = []
    for g in gdfs:
        if g is None:
            out.append(None)
        else:
            out.append(g.to_crs(base) if g.crs != base else g)
    return out

def sizes_sqrt(series, smin=10, smax=160):
    x = safe_num(series).fillna(0).clip(lower=0)
    if len(x) == 0 or x.max() <= 0:
        return np.array([])
    z = np.sqrt(x)
    z = (z - z.min()) / (z.max() - z.min() + 1e-12)
    return smin + (smax - smin) * z

def rasterize_first_collection(plot_return):
    try:
        plot_return.collections[0].set_rasterized(True)
    except Exception:
        pass

def savefig(fig, outpath, dpi=300):
    if outpath:
        os.makedirs(os.path.dirname(outpath), exist_ok=True)
        fig.savefig(outpath, bbox_inches="tight", dpi=dpi)
    return outpath

def apply_quantile_norm(vals, qlow=0.05, qhigh=0.95, gamma=1.15):
    """
    Quantile clipping + gentle gamma to reduce contrast (gamma>1 compresses highs a bit).
    """
    v = safe_num(vals)
    if np.isfinite(v).any():
        vmin = np.nanquantile(v, qlow)
        vmax = np.nanquantile(v, qhigh)
        if not np.isfinite(vmin): vmin = np.nanmin(v)
        if not np.isfinite(vmax): vmax = np.nanmax(v)
        if vmax <= vmin:
            vmax = vmin + 1e-12
        norm = mpl.colors.PowerNorm(gamma=gamma, vmin=vmin, vmax=vmax)
        return norm, vmin, vmax
    return mpl.colors.Normalize(vmin=0, vmax=1), 0, 1

def draw_outlines(ax, states_gdf=None, coast_gdf=None):
    """
    Clean, light outlines; avoid heavy borders.
    """
    if coast_gdf is not None:
        coast_gdf.boundary.plot(ax=ax, linewidth=0.5, color="#2b2b2b", alpha=0.35, zorder=4)
    if states_gdf is not None:
        states_gdf.boundary.plot(ax=ax, linewidth=0.55, color="#2b2b2b", alpha=0.55, zorder=5)
    ax.set_axis_off()

def set_extent_to_states(ax, states_gdf, pad_frac=0.03):
    xmin, ymin, xmax, ymax = states_gdf.total_bounds
    dx = xmax - xmin
    dy = ymax - ymin
    ax.set_xlim(xmin - pad_frac*dx, xmax + pad_frac*dx)
    ax.set_ylim(ymin - pad_frac*dy, ymax + pad_frac*dy)

def legend_sizes(ax, breaks, smin, smax, title, anchor=(1.02, 0.18)):
    svals = sizes_sqrt(pd.Series(breaks), smin=smin, smax=smax)
    handles = [
        Line2D([0],[0], marker='o', linestyle='None',
               markerfacecolor="white", markeredgecolor="#1a1a1a",
               markeredgewidth=0.8, alpha=0.95,
               markersize=np.sqrt(s), label=f"{b:,} MW")
        for b, s in zip(breaks, svals)
    ]
    leg = ax.legend(handles=handles, title=title, loc="center left",
                    bbox_to_anchor=anchor, frameon=True, framealpha=0.97,
                    fontsize=8, title_fontsize=9, borderpad=0.6)
    leg.get_frame().set_linewidth(0.6)
    return leg

# -----------------------
# 1) Publication map base
# -----------------------
def _setup_pub_map(figsize=(12.5, 7.2)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_axis_off()
    return fig, ax

def _add_outside_colorbar(fig, ax, mappable, label, y=0.62):
    """
    Places a slim vertical colorbar outside-right.
    y sets its vertical anchor relative to axes box.
    """
    cb = fig.colorbar(mappable, ax=ax, fraction=0.030, pad=0.012)
    cb.set_label(label, fontsize=9)
    cb.ax.tick_params(labelsize=8)
    return cb

# ------------------------------------------
# 2) Fig 1a: BA grid intensity + site dots
# ------------------------------------------
def fig1a_ba_grid_intensity_sites(
    ba_polys, sites, *,
    value_col="grid_L_per_kWh",
    ba_outline_col=None,            # optional; can pass ba_polys itself for faint BA boundaries
    mw_col="current_mw",
    states_gdf=None,
    cmap="cividis",
    qlow=0.05, qhigh=0.95, gamma=1.20,   # muted contrast
    dot_smin=10, dot_smax=160,           # SAME for Fig1a and Fig1b
    dot_breaks=(25, 100, 300),
    title="Figure 1a. Grid water intensity by balancing authority and hyperscale sites",
    cbar_label="Grid water intensity (L/kWh)",
    outpath=None
):
    ba_polys, sites, states_gdf = ensure_crs_match(ba_polys, sites, states_gdf)

    fig, ax = _setup_pub_map()

    # Choropleth (no internal polygon borders)
    norm, vmin, vmax = apply_quantile_norm(ba_polys[value_col], qlow=qlow, qhigh=qhigh, gamma=gamma)
    poly = ba_polys.plot(ax=ax, column=value_col, cmap=cmap, linewidth=0, norm=norm, zorder=1)
    rasterize_first_collection(poly)

    # Outlines
    draw_outlines(ax, states_gdf=states_gdf, coast_gdf=None)

    # Extent locked to US frame
    if states_gdf is not None:
        set_extent_to_states(ax, states_gdf, pad_frac=0.02)

    # Sites as hollow circles (readable over color)
    ms = sizes_sqrt(sites[mw_col], smin=dot_smin, smax=dot_smax)
    pts = sites.plot(
        ax=ax, markersize=ms,
        facecolor="white", edgecolor="#111111",
        linewidth=0.45, alpha=0.70, zorder=6
    )
    rasterize_first_collection(pts)

    ax.set_title(title, fontsize=12, pad=8)

    # Colorbar (outside)
    sm = mpl.cm.ScalarMappable(cmap=plt.get_cmap(cmap), norm=norm)
    sm.set_array([])
    _add_outside_colorbar(fig, ax, sm, cbar_label)

    # Size legend (outside; same breaks for both figs)
    legend_sizes(ax, dot_breaks, dot_smin, dot_smax, "Facility load (MW)", anchor=(1.02, 0.22))

    # leave room on right for cb + legend
    fig.subplots_adjust(right=0.84)

    return savefig(fig, outpath)

# ------------------------------------------
# 3) Fig 1b: Basin stress + site dots (clipped)
# ------------------------------------------
def fig1b_basin_stress_sites(
    basins_polys, sites, *,
    stress_col="bws_01_score",
    mw_col="current_mw",
    states_gdf=None,
    # For Aqueduct stress 0–5, discrete bins read better than continuous gradients
    discrete=True,
    bins=(0, 1, 2, 3, 4, 5),
    cmap_discrete="Blues",      # muted; avoids harsh red
    cmap_cont="cividis",
    qlow=0.05, qhigh=0.95, gamma=1.10,
    dot_smin=10, dot_smax=160,  # MUST MATCH Fig1a
    dot_breaks=(25, 100, 300),
    title="Figure 1b. Basin water stress (Aqueduct) and hyperscale sites",
    cbar_label="Aqueduct baseline water stress (0–5)",
    outpath=None
):
    basins_polys, sites, states_gdf = ensure_crs_match(basins_polys, sites, states_gdf)

    fig, ax = _setup_pub_map()

    # Clip basins to the US frame (prevents “more than US borders” problem)
    if states_gdf is not None:
        xmin, ymin, xmax, ymax = states_gdf.total_bounds
        clip_box = box(xmin, ymin, xmax, ymax)
        basins_polys = basins_polys[basins_polys.intersects(clip_box)].copy()

    # Choropleth
    if discrete:
        b = np.array(bins)
        cmap = plt.get_cmap(cmap_discrete, len(b)-1)
        norm = mpl.colors.BoundaryNorm(b, cmap.N, clip=True)
        poly = basins_polys.plot(ax=ax, column=stress_col, cmap=cmap, norm=norm, linewidth=0, zorder=1)
        rasterize_first_collection(poly)
    else:
        norm, vmin, vmax = apply_quantile_norm(basins_polys[stress_col], qlow=qlow, qhigh=qhigh, gamma=gamma)
        poly = basins_polys.plot(ax=ax, column=stress_col, cmap=cmap_cont, linewidth=0, norm=norm, zorder=1)
        rasterize_first_collection(poly)

    # States outlines + a subtle “US frame” boundary
    if states_gdf is not None:
        # union boundary as a subtle frame
        try:
            gpd.GeoSeries(states_gdf.unary_union).boundary.plot(
                ax=ax, linewidth=0.9, color="#1f1f1f", alpha=0.35, zorder=4
            )
        except Exception:
            pass
    draw_outlines(ax, states_gdf=states_gdf, coast_gdf=None)

    # Extent locked to US frame
    if states_gdf is not None:
        set_extent_to_states(ax, states_gdf, pad_frac=0.02)

    # Sites dots (same scaling + style as Fig1a)
    ms = sizes_sqrt(sites[mw_col], smin=dot_smin, smax=dot_smax)
    pts = sites.plot(
        ax=ax, markersize=ms,
        facecolor="white", edgecolor="#111111",
        linewidth=0.45, alpha=0.70, zorder=6
    )
    rasterize_first_collection(pts)

    ax.set_title(title, fontsize=12, pad=8)

    # Colorbar outside
    if discrete:
        sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cb = _add_outside_colorbar(fig, ax, sm, cbar_label)
        # nicer categorical tick labels centered in bins
        centers = (np.array(bins[:-1]) + np.array(bins[1:])) / 2
        cb.set_ticks(centers)
        cb.set_ticklabels([f"{bins[i]}–{bins[i+1]}" for i in range(len(bins)-1)])
    else:
        sm = mpl.cm.ScalarMappable(cmap=plt.get_cmap(cmap_cont), norm=norm)
        sm.set_array([])
        _add_outside_colorbar(fig, ax, sm, cbar_label)

    # Size legend outside (same breaks as Fig1a)
    legend_sizes(ax, dot_breaks, dot_smin, dot_smax, "Facility load (MW)", anchor=(1.02, 0.22))

    fig.subplots_adjust(right=0.84)

    return savefig(fig, outpath)

# -----------------------
# 4) Wrapper to regenerate both
# -----------------------
def make_fig1ab_publication(
    OUTDIR, TAG, *,
    ba_polys, basins_polys, sites_gdf, states_gdf,
    COL_GRID="grid_L_per_kWh",
    COL_STRESS="bws_01_score",
    COL_MW="current_mw",
    dot_smin=10, dot_smax=160,
    dot_breaks=(25, 100, 300),
):
    os.makedirs(OUTDIR, exist_ok=True)

    f1a = os.path.join(OUTDIR, f"MAIN_Fig1a_BA_gridIntensity_sites__{TAG}.pdf")
    f1b = os.path.join(OUTDIR, f"MAIN_Fig1b_Basin_stress_sites__{TAG}.pdf")

    fig1a_ba_grid_intensity_sites(
        ba_polys, sites_gdf,
        value_col=COL_GRID, mw_col=COL_MW, states_gdf=states_gdf,
        dot_smin=dot_smin, dot_smax=dot_smax, dot_breaks=dot_breaks,
        outpath=f1a
    )

    fig1b_basin_stress_sites(
        basins_polys, sites_gdf,
        stress_col=COL_STRESS, mw_col=COL_MW, states_gdf=states_gdf,
        dot_smin=dot_smin, dot_smax=dot_smax, dot_breaks=dot_breaks,
        outpath=f1b
    )

    return {"Fig1a": f1a, "Fig1b": f1b}

# -----------------------
# Example call (your objects)
# -----------------------
OUTDIR = "figures_main"
TAG = "base"
paths_1ab = make_fig1ab_publication(
     OUTDIR, TAG,
     ba_polys=ba_p,
     basins_polys=basins_p,
     sites_gdf=sites_p,
     states_gdf=states_p,
     COL_GRID="grid_L_per_kWh",     # note the exact capitalization you have
     COL_STRESS="bws_01_score",
     COL_MW="current_mw",
     dot_smin=10, dot_smax=160,
     dot_breaks=(25, 100, 300),
 )
paths_1ab


In [ ]:
# ============================================
# Nature/Science-ready Fig1a–Fig1b (sites dots)
# - muted palettes
# - identical dot scaling + legend in both figs
# - Fig1b clipped to US extent (no extra world)
# - non-overlapping legends, rasterized points
# ============================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from shapely.geometry import box

# -----------------------
# 0) Helpers (keep yours)
# -----------------------
def safe_num(x):
    return pd.to_numeric(x, errors="coerce")

def ensure_crs_match(*gdfs):
    base = None
    for g in gdfs:
        if g is not None and hasattr(g, "crs") and g.crs is not None:
            base = g.crs
            break
    if base is None:
        return gdfs
    out = []
    for g in gdfs:
        if g is None:
            out.append(None)
        else:
            out.append(g.to_crs(base) if g.crs != base else g)
    return out

def sizes_sqrt(series, smin=10, smax=160):
    x = safe_num(series).fillna(0).clip(lower=0)
    if len(x) == 0 or x.max() <= 0:
        return np.array([])
    z = np.sqrt(x)
    z = (z - z.min()) / (z.max() - z.min() + 1e-12)
    return smin + (smax - smin) * z

def rasterize_first_collection(plot_return):
    try:
        plot_return.collections[0].set_rasterized(True)
    except Exception:
        pass

def savefig(fig, outpath, dpi=300):
    if outpath:
        os.makedirs(os.path.dirname(outpath), exist_ok=True)
        fig.savefig(outpath, bbox_inches="tight", dpi=dpi)
    return outpath

def apply_quantile_norm(vals, qlow=0.05, qhigh=0.95, gamma=1.15):
    """
    Quantile clipping + gentle gamma to reduce contrast (gamma>1 compresses highs a bit).
    """
    v = safe_num(vals)
    if np.isfinite(v).any():
        vmin = np.nanquantile(v, qlow)
        vmax = np.nanquantile(v, qhigh)
        if not np.isfinite(vmin): vmin = np.nanmin(v)
        if not np.isfinite(vmax): vmax = np.nanmax(v)
        if vmax <= vmin:
            vmax = vmin + 1e-12
        norm = mpl.colors.PowerNorm(gamma=gamma, vmin=vmin, vmax=vmax)
        return norm, vmin, vmax
    return mpl.colors.Normalize(vmin=0, vmax=1), 0, 1

def draw_outlines(ax, states_gdf=None, coast_gdf=None):
    """
    Clean, light outlines; avoid heavy borders.
    """
    if coast_gdf is not None:
        coast_gdf.boundary.plot(ax=ax, linewidth=0.5, color="#2b2b2b", alpha=0.35, zorder=4)
    if states_gdf is not None:
        states_gdf.boundary.plot(ax=ax, linewidth=0.55, color="#2b2b2b", alpha=0.55, zorder=5)
    ax.set_axis_off()

def set_extent_to_states(ax, states_gdf, pad_frac=0.03):
    xmin, ymin, xmax, ymax = states_gdf.total_bounds
    dx = xmax - xmin
    dy = ymax - ymin
    ax.set_xlim(xmin - pad_frac*dx, xmax + pad_frac*dx)
    ax.set_ylim(ymin - pad_frac*dy, ymax + pad_frac*dy)

def legend_sizes(ax, breaks, smin, smax, title, anchor=(1.02, 0.18)):
    svals = sizes_sqrt(pd.Series(breaks), smin=smin, smax=smax)
    handles = [
        Line2D([0],[0], marker='o', linestyle='None',
               markerfacecolor="white", markeredgecolor="#1a1a1a",
               markeredgewidth=0.8, alpha=0.95,
               markersize=np.sqrt(s), label=f"{b:,} MW")
        for b, s in zip(breaks, svals)
    ]
    leg = ax.legend(handles=handles, title=title, loc="center left",
                    bbox_to_anchor=anchor, frameon=True, framealpha=0.97,
                    fontsize=8, title_fontsize=9, borderpad=0.6)
    leg.get_frame().set_linewidth(0.6)
    return leg

# -----------------------
# 1) Publication map base
# -----------------------
def _setup_pub_map(figsize=(12.5, 7.2)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_axis_off()
    return fig, ax

def _add_outside_colorbar(fig, ax, mappable, label, y=0.62):
    """
    Places a slim vertical colorbar outside-right.
    y sets its vertical anchor relative to axes box.
    """
    cb = fig.colorbar(mappable, ax=ax, fraction=0.030, pad=0.012)
    cb.set_label(label, fontsize=9)
    cb.ax.tick_params(labelsize=8)
    return cb

# ------------------------------------------
# 2) Fig 1a: BA grid intensity + site dots
# ------------------------------------------
def fig1a_ba_grid_intensity_sites(
    ba_polys, sites, *,
    value_col="grid_L_per_kWh",
    ba_outline_col=None,            # optional; can pass ba_polys itself for faint BA boundaries
    mw_col="current_mw",
    states_gdf=None,
    cmap="cividis",
    qlow=0.05, qhigh=0.95, gamma=1.20,   # muted contrast
    dot_smin=10, dot_smax=160,           # SAME for Fig1a and Fig1b
    dot_breaks=(25, 100, 300),
    title="Figure 1a. Grid water intensity by balancing authority and hyperscale sites",
    cbar_label="Grid water intensity (L/kWh)",
    outpath=None
):
    ba_polys, sites, states_gdf = ensure_crs_match(ba_polys, sites, states_gdf)

    fig, ax = _setup_pub_map()

    # Choropleth (no internal polygon borders)
    norm, vmin, vmax = apply_quantile_norm(ba_polys[value_col], qlow=qlow, qhigh=qhigh, gamma=gamma)
    poly = ba_polys.plot(ax=ax, column=value_col, cmap=cmap, linewidth=0, norm=norm, zorder=1)
    rasterize_first_collection(poly)

    # Outlines
    draw_outlines(ax, states_gdf=states_gdf, coast_gdf=None)

    # Extent locked to US frame
    if states_gdf is not None:
        set_extent_to_states(ax, states_gdf, pad_frac=0.02)

    # Sites as hollow circles (readable over color)
    ms = sizes_sqrt(sites[mw_col], smin=dot_smin, smax=dot_smax)
    pts = sites.plot(
        ax=ax, markersize=ms,
        facecolor="white", edgecolor="#111111",
        linewidth=0.45, alpha=0.70, zorder=6
    )
    rasterize_first_collection(pts)

    ax.set_title(title, fontsize=12, pad=8)

    # Colorbar (outside)
    sm = mpl.cm.ScalarMappable(cmap=plt.get_cmap(cmap), norm=norm)
    sm.set_array([])
    _add_outside_colorbar(fig, ax, sm, cbar_label)

    # Size legend (outside; same breaks for both figs)
    legend_sizes(ax, dot_breaks, dot_smin, dot_smax, "Facility load (MW)", anchor=(1.02, 0.22))

    # leave room on right for cb + legend
    fig.subplots_adjust(right=0.84)

    return savefig(fig, outpath)

# ------------------------------------------
# 3) Fig 1b: Basin stress + site dots (clipped)
# ------------------------------------------
def fig1b_basin_stress_sites(
    basins_polys, sites, *,
    stress_col="bws_01_score",
    mw_col="current_mw",
    states_gdf=None,
    # For Aqueduct stress 0–5, discrete bins read better than continuous gradients
    discrete=True,
    bins=(0, 1, 2, 3, 4, 5),
    cmap_discrete="Blues",      # muted; avoids harsh red
    cmap_cont="cividis",
    qlow=0.05, qhigh=0.95, gamma=1.10,
    dot_smin=10, dot_smax=160,  # MUST MATCH Fig1a
    dot_breaks=(25, 100, 300),
    title="Figure 1b. Basin water stress (Aqueduct) and hyperscale sites",
    cbar_label="Aqueduct baseline water stress (0–5)",
    outpath=None
):
    basins_polys, sites, states_gdf = ensure_crs_match(basins_polys, sites, states_gdf)

    fig, ax = _setup_pub_map()

    # Clip basins to the US frame (prevents “more than US borders” problem)
    if states_gdf is not None:
        xmin, ymin, xmax, ymax = states_gdf.total_bounds
        clip_box = box(xmin, ymin, xmax, ymax)
        basins_polys = basins_polys[basins_polys.intersects(clip_box)].copy()

    # Choropleth
    if discrete:
        b = np.array(bins)
        cmap = plt.get_cmap(cmap_discrete, len(b)-1)
        norm = mpl.colors.BoundaryNorm(b, cmap.N, clip=True)
        poly = basins_polys.plot(ax=ax, column=stress_col, cmap=cmap, norm=norm, linewidth=0, zorder=1)
        rasterize_first_collection(poly)
    else:
        norm, vmin, vmax = apply_quantile_norm(basins_polys[stress_col], qlow=qlow, qhigh=qhigh, gamma=gamma)
        poly = basins_polys.plot(ax=ax, column=stress_col, cmap=cmap_cont, linewidth=0, norm=norm, zorder=1)
        rasterize_first_collection(poly)

    # States outlines + a subtle “US frame” boundary
    if states_gdf is not None:
        # union boundary as a subtle frame
        try:
            gpd.GeoSeries(states_gdf.unary_union).boundary.plot(
                ax=ax, linewidth=0.9, color="#1f1f1f", alpha=0.35, zorder=4
            )
        except Exception:
            pass
    draw_outlines(ax, states_gdf=states_gdf, coast_gdf=None)

    # Extent locked to US frame
    if states_gdf is not None:
        set_extent_to_states(ax, states_gdf, pad_frac=0.02)

    # Sites dots (same scaling + style as Fig1a)
    ms = sizes_sqrt(sites[mw_col], smin=dot_smin, smax=dot_smax)
    pts = sites.plot(
        ax=ax, markersize=ms,
        facecolor="white", edgecolor="#111111",
        linewidth=0.45, alpha=0.70, zorder=6
    )
    rasterize_first_collection(pts)

    ax.set_title(title, fontsize=12, pad=8)

    # Colorbar outside
    if discrete:
        sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cb = _add_outside_colorbar(fig, ax, sm, cbar_label)
        # nicer categorical tick labels centered in bins
        centers = (np.array(bins[:-1]) + np.array(bins[1:])) / 2
        cb.set_ticks(centers)
        cb.set_ticklabels([f"{bins[i]}–{bins[i+1]}" for i in range(len(bins)-1)])
    else:
        sm = mpl.cm.ScalarMappable(cmap=plt.get_cmap(cmap_cont), norm=norm)
        sm.set_array([])
        _add_outside_colorbar(fig, ax, sm, cbar_label)

    # Size legend outside (same breaks as Fig1a)
    legend_sizes(ax, dot_breaks, dot_smin, dot_smax, "Facility load (MW)", anchor=(1.02, 0.22))

    fig.subplots_adjust(right=0.84)

    return savefig(fig, outpath)

# -----------------------
# 4) Wrapper to regenerate both
# -----------------------
def make_fig1ab_publication(
    OUTDIR, TAG, *,
    ba_polys, basins_polys, sites_gdf, states_gdf,
    COL_GRID="grid_L_per_kWh",
    COL_STRESS="bws_01_score",
    COL_MW="current_mw",
    dot_smin=10, dot_smax=160,
    dot_breaks=(25, 100, 300),
):
    os.makedirs(OUTDIR, exist_ok=True)

    f1a = os.path.join(OUTDIR, f"MAIN_Fig1a_BA_gridIntensity_sites__{TAG}.pdf")
    f1b = os.path.join(OUTDIR, f"MAIN_Fig1b_Basin_stress_sites__{TAG}.pdf")

    fig1a_ba_grid_intensity_sites(
        ba_polys, sites_gdf,
        value_col=COL_GRID, mw_col=COL_MW, states_gdf=states_gdf,
        dot_smin=dot_smin, dot_smax=dot_smax, dot_breaks=dot_breaks,
        outpath=f1a
    )

    fig1b_basin_stress_sites(
        basins_polys, sites_gdf,
        stress_col=COL_STRESS, mw_col=COL_MW, states_gdf=states_gdf,
        dot_smin=dot_smin, dot_smax=dot_smax, dot_breaks=dot_breaks,
        outpath=f1b
    )

    return {"Fig1a": f1a, "Fig1b": f1b}

# -----------------------
# Example call (your objects)
# -----------------------
OUTDIR = "figures_main"
TAG = "base"
paths_1ab = make_fig1ab_publication(
     OUTDIR, TAG,
     ba_polys=ba_p,
     basins_polys=basins_p,
     sites_gdf=sites_p,
     states_gdf=states_p,
     COL_GRID="grid_L_per_kWh",     # note the exact capitalization you have
     COL_STRESS="bws_01_score",
     COL_MW="current_mw",
     dot_smin=10, dot_smax=160,
     dot_breaks=(25, 100, 300),
 )
paths_1ab


In [ ]:
sites_p.groupby('subregion_id').sum()['current_mw']

# new plots

In [ ]:
import pandas as pd
import geopandas as gpd

def _is_df(x):  return isinstance(x, pd.DataFrame) and not isinstance(x, gpd.GeoDataFrame)
def _is_gdf(x): return isinstance(x, gpd.GeoDataFrame)

# ---- locals(): show df/gdf names ----
dfs, gdfs = [], []
for name, obj in list(globals().items()):
    if name.startswith("_"): 
        continue
    try:
        if _is_gdf(obj):
            gdfs.append(name)
        elif _is_df(obj):
            dfs.append(name)
    except Exception:
        pass

print("GeoDataFrames:", sorted(gdfs))
print("DataFrames:", sorted(dfs))

# ---- CFG keys (if CFG exists) ----
if "CFG" in globals() and isinstance(CFG, dict):
    print("\nCFG keys:", sorted(CFG.keys()))
    for k in ["gdf_ba", "gdf_basin", "gdf_sites"]:
        v = CFG.get(k, None)
        print(f"CFG['{k}']:", type(v), "shape=", getattr(v, "shape", None))


In [ ]:
import pandas as pd
import geopandas as gpd

def quick_profile(name, obj, n=5, key_candidates=None):
    print("\n" + "="*90)
    print(f"{name}: {type(obj)}")
    if hasattr(obj, "shape"):
        print("shape:", obj.shape)
    if isinstance(obj, (pd.DataFrame, gpd.GeoDataFrame)):
        cols = list(obj.columns)
        print(f"ncols={len(cols)}")
        print("columns:", cols)

        # geometry / crs info if GeoDataFrame
        if isinstance(obj, gpd.GeoDataFrame):
            print("crs:", obj.crs)
            print("geom type counts:", obj.geom_type.value_counts().to_dict())

        # show candidate key columns unique samples
        if key_candidates:
            for k in key_candidates:
                if k in obj.columns:
                    s = obj[k]
                    print(f"\n-- sample values for '{k}' (dtype={s.dtype})")
                    print(s.dropna().astype(str).head(n).tolist())
                    print("nunique:", s.nunique(dropna=True))
    print("="*90)

# -------------------------------------------------------------------
# The ONLY objects we need for Fig 1 panel (maps + points)
# -------------------------------------------------------------------
gdf_sites = CFG["gdf_sites"]
gdf_ba    = CFG["gdf_ba"]
gdf_basin = CFG["gdf_basin"]

# If you have a USA / states basemap layer you want behind the polygons
# (optional but useful for "seeing the map"):
maybe_layers = {}
for nm in ["usa", "states48", "states", "world"]:
    if nm in globals():
        maybe_layers[nm] = globals()[nm]

# -------------------------------------------------------------------
# Candidates that might contain join IDs / labels / intensities
# -------------------------------------------------------------------
site_key_candidates = [
    "subregion_id","region_id","region_name","region","region_BA","region_BA_",
    "pfaf_id","STUSPS","state_abbrev","state_name"
]
ba_key_candidates = [
    "subregion_id","region_id","region_name","region","region_BA","region_BA_"
]
basin_key_candidates = ["pfaf_id","PFAF_ID","HYBAS_ID","HYBAS_ID_4","HYBAS_ID_6"]

intensity_candidates = [
    "grid_L_per_kwh","grid_L_per_kWh","I_grid_L_per_kWh","water_intensity_L_per_kWh","L_per_kWh","I_grid"
]
mw_candidates = ["current_mw","total_mw","power_mw_estimated","power_mw"]
scope_candidates = ["s1_m3_mid","s2_m3_mid","total_m3_mid","mwh_fac_A_mid","share_fossil","bws_01_score"]

# -------------------------------------------------------------------
# Print profiles
# -------------------------------------------------------------------
quick_profile("CFG['gdf_sites']", gdf_sites, key_candidates=site_key_candidates + intensity_candidates + mw_candidates + scope_candidates)
quick_profile("CFG['gdf_ba']", gdf_ba, key_candidates=ba_key_candidates)
quick_profile("CFG['gdf_basin']", gdf_basin, key_candidates=basin_key_candidates + ["bws_01_score"])

for nm, layer in maybe_layers.items():
    quick_profile(nm, layer, key_candidates=["NAME","STUSPS","state_abbrev"])


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from pyproj import CRS

# -----------------------------
# Inputs (from your environment)
# -----------------------------
gdf_sites = CFG["gdf_sites"].copy()
gdf_ba    = CFG["gdf_ba"].copy()
gdf_basin = CFG["gdf_basin"].copy()

states48 = globals().get("states48", None)
assert states48 is not None, "Expected a 'states48' GeoDataFrame in your globals()"

# -----------------------------
# Column names (from your dump)
# -----------------------------
BA_KEY_SITES   = "subregion_id"
BA_KEY_BA      = "subregion_id"

BASIN_KEY_SITES  = "pfaf_id"
BASIN_KEY_BASIN  = "PFAF_ID"

MW_COL = "current_mw"
BA_INT_COL_SITE = "grid_L_per_kwh"
STRESS_COL_SITE = "bws_01_score"
STRESS_COL_BAS  = "bws_01_score"

# -----------------------------
# Lambert Conformal Conic (CONUS)
# -----------------------------
LCC_CONUS = CRS.from_proj4(
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 "
    "+x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"
)

def to_lcc(gdf):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(LCC_CONUS)

# Reproject
gdf_sites_l  = to_lcc(gdf_sites)
gdf_ba_l     = to_lcc(gdf_ba)
gdf_basin_l  = to_lcc(gdf_basin)
states48_l   = to_lcc(states48)

# -----------------------------
# Clip everything to states48 extent + geometry
# -----------------------------
conus_geom = states48_l.unary_union
gdf_ba_l    = gdf_ba_l[gdf_ba_l.intersects(conus_geom)].copy()
gdf_basin_l = gdf_basin_l[gdf_basin_l.intersects(conus_geom)].copy()
gdf_sites_l = gdf_sites_l[gdf_sites_l.intersects(conus_geom)].copy()

# -----------------------------
# Normalize join keys to strings (prevents dtype merge errors)
# -----------------------------
gdf_sites_l[BA_KEY_SITES] = gdf_sites_l[BA_KEY_SITES].astype(str)
gdf_ba_l[BA_KEY_BA]       = gdf_ba_l[BA_KEY_BA].astype(str)

gdf_sites_l[BASIN_KEY_SITES] = gdf_sites_l[BASIN_KEY_SITES].astype(str)
gdf_basin_l[BASIN_KEY_BASIN] = gdf_basin_l[BASIN_KEY_BASIN].astype(str)

# -----------------------------
# Aggregate MW to polygons
# -----------------------------
ba_mw = (gdf_sites_l.groupby(BA_KEY_SITES)[MW_COL].sum().reset_index().rename(columns={MW_COL: "mw_sum"}))
bas_mw = (gdf_sites_l.groupby(BASIN_KEY_SITES)[MW_COL].sum().reset_index().rename(columns={MW_COL: "mw_sum"}))

gdf_ba_l = gdf_ba_l.merge(ba_mw, left_on=BA_KEY_BA, right_on=BA_KEY_SITES, how="left")
gdf_ba_l["mw_sum"] = gdf_ba_l["mw_sum"].fillna(0.0)

gdf_basin_l = gdf_basin_l.merge(bas_mw, left_on=BASIN_KEY_BASIN, right_on=BASIN_KEY_SITES, how="left")
gdf_basin_l["mw_sum"] = gdf_basin_l["mw_sum"].fillna(0.0)

# -----------------------------
# BA intensity: MW-weighted mean from sites
# -----------------------------
ba_int = (gdf_sites_l
          .dropna(subset=[BA_INT_COL_SITE, MW_COL])
          .groupby(BA_KEY_SITES)
          .apply(lambda d: np.average(d[BA_INT_COL_SITE].astype(float), weights=d[MW_COL].astype(float)))
          .reset_index(name="ba_grid_L_per_kwh_mw_weighted"))

gdf_ba_l = gdf_ba_l.drop(columns=["ba_grid_L_per_kwh_mw_weighted"], errors="ignore").merge(
    ba_int, left_on=BA_KEY_BA, right_on=BA_KEY_SITES, how="left"
)

# -----------------------------
# Better point sizing
# -----------------------------
mw = gdf_sites_l[MW_COL].fillna(0).astype(float).values
p90 = np.nanpercentile(mw[mw > 0], 90) if np.any(mw > 0) else 100.0
mw_clip = np.clip(mw, 0, p90)
size = 18 + 260 * (np.sqrt(mw_clip) / np.sqrt(p90 + 1e-9))  # 18..~278

# -----------------------------
# Map extent = states48 bounds (tight)
# -----------------------------
minx, miny, maxx, maxy = states48_l.total_bounds
pad_x = 0.02 * (maxx - minx)
pad_y = 0.02 * (maxy - miny)
EXTENT = (minx - pad_x, maxx + pad_x, miny - pad_y, maxy + pad_y)

def draw_base(ax):
    # Only states48 (fill + outline)
    states48_l.plot(ax=ax, color="white", edgecolor="0.55", linewidth=0.7, zorder=0)
    ax.set_xlim(EXTENT[0], EXTENT[1])
    ax.set_ylim(EXTENT[2], EXTENT[3])
    ax.set_axis_off()
    ax.set_aspect("equal")

def add_cbar(ax, mappable, label):
    cax = inset_axes(ax, width="3.2%", height="62%", loc="lower left",
                     bbox_to_anchor=(1.02, 0.16, 1, 1),
                     bbox_transform=ax.transAxes, borderpad=0)
    cb = plt.colorbar(mappable, cax=cax)
    cb.set_label(label, fontsize=10)
    cb.ax.tick_params(labelsize=9)

# -----------------------------
# Figure layout: BIGGER maps
# -----------------------------
fig = plt.figure(figsize=(18, 10))  # larger canvas
gs = fig.add_gridspec(2, 2, wspace=0.05, hspace=0.08)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[1, 1])

# -----------------------------
# Panel A: BA grid intensity + sites
# -----------------------------
draw_base(ax1)
gdf_ba_l.plot(
    ax=ax1,
    column="ba_grid_L_per_kwh_mw_weighted",
    cmap="viridis",
    linewidth=0.35,
    edgecolor="white",
    legend=False,
    zorder=1,
    missing_kwds={"color": "0.92", "edgecolor": "white", "hatch": "///"},
)
gdf_sites_l.plot(ax=ax1, markersize=size, color="black", alpha=0.55, linewidth=0, zorder=3)
ax1.set_title("a  Balancing authority grid water intensity (L/kWh) + hyperscale sites", loc="left", fontsize=13)

vmin1 = np.nanpercentile(gdf_ba_l["ba_grid_L_per_kwh_mw_weighted"], 2)
vmax1 = np.nanpercentile(gdf_ba_l["ba_grid_L_per_kwh_mw_weighted"], 98)
m1 = ScalarMappable(norm=Normalize(vmin=vmin1, vmax=vmax1), cmap="viridis")
m1.set_array([])
add_cbar(ax1, m1, "Grid water intensity (L/kWh)")

# -----------------------------
# Panel B: Basin stress + sites
# -----------------------------
draw_base(ax2)
gdf_basin_l.plot(
    ax=ax2,
    column=STRESS_COL_BAS,
    cmap="plasma",
    linewidth=0.15,
    edgecolor="white",
    legend=False,
    zorder=1,
)
gdf_sites_l.plot(ax=ax2, markersize=size, color="black", alpha=0.55, linewidth=0, zorder=3)
ax2.set_title("b  Basin water stress (Aqueduct baseline) + hyperscale sites", loc="left", fontsize=13)

m2 = ScalarMappable(norm=Normalize(vmin=0, vmax=5), cmap="plasma")
m2.set_array([])
add_cbar(ax2, m2, "Baseline water stress (0–5)")

# -----------------------------
# Panel C: BA capacity (MW sum) + sites
# -----------------------------
draw_base(ax3)
gdf_ba_l.plot(
    ax=ax3,
    column="mw_sum",
    cmap="cividis",
    linewidth=0.35,
    edgecolor="white",
    legend=False,
    zorder=1,
)
gdf_sites_l.plot(ax=ax3, markersize=size, color="black", alpha=0.55, linewidth=0, zorder=3)
ax3.set_title("c  Hyperscale capacity by balancing authority (MW) + sites", loc="left", fontsize=13)

m3 = ScalarMappable(norm=Normalize(vmin=0, vmax=np.nanpercentile(gdf_ba_l["mw_sum"], 98)), cmap="cividis")
m3.set_array([])
add_cbar(ax3, m3, "Capacity (MW)")

# -----------------------------
# Panel D: Basin capacity (MW sum) + sites
# -----------------------------
draw_base(ax4)
gdf_basin_l.plot(
    ax=ax4,
    column="mw_sum",
    cmap="cividis",
    linewidth=0.15,
    edgecolor="white",
    legend=False,
    zorder=1,
)
gdf_sites_l.plot(ax=ax4, markersize=size, color="black", alpha=0.55, linewidth=0, zorder=3)
ax4.set_title("d  Hyperscale capacity by basin (MW) + sites", loc="left", fontsize=13)

m4 = ScalarMappable(norm=Normalize(vmin=0, vmax=np.nanpercentile(gdf_basin_l["mw_sum"], 98)), cmap="cividis")
m4.set_array([])
add_cbar(ax4, m4, "Capacity (MW)")

# -----------------------------
# Output
# -----------------------------
fig.suptitle("Figure 1 | Geographic context for Scope 1–2 water accounting (baseline)", y=0.98, fontsize=15)

out_png = "Fig1_panel__base_Lambert_states48_big.png"
out_pdf = "Fig1_panel__base_Lambert_states48_big.pdf"
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.savefig(out_pdf, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out_png)
print("Saved:", out_pdf)


In [ ]:
# Figure 1 - improved (Lambert CONUS, smaller site markers, top-site labels)
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from pyproj import CRS

# ----------- Inputs from your session (should be present) --------------
gdf_sites = CFG["gdf_sites"].copy()
gdf_ba    = CFG["gdf_ba"].copy()
gdf_basin = CFG["gdf_basin"].copy()
states48  = globals().get("states48")
assert states48 is not None, "states48 must be present"

# Column names (from your env)
MW_COL = "current_mw"
BA_INT_COL_SITE = "grid_L_per_kwh"
STRESS_COL_BAS = "bws_01_score"
BA_KEY_SITES = "subregion_id"
BASIN_KEY_SITES = "pfaf_id"
BA_KEY_BA = "subregion_id"
BASIN_KEY_BAS = "PFAF_ID"

# ---------- Lambert Conformal Conic (CONUS) ----------
LCC_CONUS = CRS.from_proj4(
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 "
    "+x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"
)
def to_lcc(gdf):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(LCC_CONUS)

gdf_sites_l  = to_lcc(gdf_sites)
gdf_ba_l     = to_lcc(gdf_ba)
gdf_basin_l  = to_lcc(gdf_basin)
states48_l   = to_lcc(states48)

# Clip to CONUS extent
conus_geom = states48_l.unary_union
gdf_ba_l    = gdf_ba_l[gdf_ba_l.intersects(conus_geom)].copy()
gdf_basin_l = gdf_basin_l[gdf_basin_l.intersects(conus_geom)].copy()
gdf_sites_l = gdf_sites_l[gdf_sites_l.intersects(conus_geom)].copy()

# Normalize keys to strings to avoid merge dtype issues
gdf_sites_l[BA_KEY_SITES] = gdf_sites_l[BA_KEY_SITES].astype(str)
gdf_ba_l[BA_KEY_BA] = gdf_ba_l[BA_KEY_BA].astype(str)
gdf_sites_l[BASIN_KEY_SITES] = gdf_sites_l[BASIN_KEY_SITES].astype(str)
gdf_basin_l[BASIN_KEY_BAS] = gdf_basin_l[BASIN_KEY_BAS].astype(str)

# Aggregate MW by BA and by basin
ba_mw = (gdf_sites_l.groupby(BA_KEY_SITES)[MW_COL].sum().reset_index().rename(columns={MW_COL: "mw_sum"}))
bas_mw = (gdf_sites_l.groupby(BASIN_KEY_SITES)[MW_COL].sum().reset_index().rename(columns={MW_COL: "mw_sum"}))

gdf_ba_l = gdf_ba_l.merge(ba_mw, left_on=BA_KEY_BA, right_on=BA_KEY_SITES, how="left")
gdf_ba_l["mw_sum"] = gdf_ba_l["mw_sum"].fillna(0.0)

gdf_basin_l = gdf_basin_l.merge(bas_mw, left_on=BASIN_KEY_BAS, right_on=BASIN_KEY_SITES, how="left")
gdf_basin_l["mw_sum"] = gdf_basin_l["mw_sum"].fillna(0.0)

# Compute MW-weighted BA intensity from site column if present
if BA_INT_COL_SITE in gdf_sites_l.columns:
    ba_int = (gdf_sites_l.dropna(subset=[BA_INT_COL_SITE, MW_COL])
              .groupby(BA_KEY_SITES)
              .apply(lambda d: np.average(d[BA_INT_COL_SITE].astype(float), weights=d[MW_COL].astype(float)))
              .reset_index(name="ba_grid_L_per_kwh_mw_weighted"))
    gdf_ba_l = gdf_ba_l.merge(ba_int, left_on=BA_KEY_BA, right_on=BA_KEY_SITES, how="left")

# -------- marker sizing (small, not overpowering) ----------
mw = gdf_sites_l[MW_COL].fillna(0).astype(float).values
p90 = np.nanpercentile(mw[mw>0], 90) if np.any(mw>0) else 100.
mw_clip = np.clip(mw, 0, p90)
# smaller sizes: 6..60
marker_size = 6 + 54 * (np.sqrt(mw_clip) / (np.sqrt(p90)+1e-9))

# Choose how many top sites to annotate (large, numbered)
N_ANNOT = 10
top_sites = gdf_sites_l.sort_values(MW_COL, ascending=False).head(N_ANNOT).copy()
top_sites = top_sites.reset_index(drop=True)

# -------- figure layout (bigger, publication-ready) --------
fig = plt.figure(figsize=(20, 11))
gs = fig.add_gridspec(2, 2, wspace=0.08, hspace=0.08)

axs = [fig.add_subplot(gs[0,0]), fig.add_subplot(gs[0,1]), fig.add_subplot(gs[1,0]), fig.add_subplot(gs[1,1])]

def draw_base(ax):
    # subtle base: white fill, faint state border
    states48_l.plot(ax=ax, color="white", edgecolor="#bbbbbb", linewidth=0.45, zorder=0)
    ax.set_axis_off()
    ax.set_aspect("equal")
    # tight extent
    minx, miny, maxx, maxy = states48_l.total_bounds
    pad_x = 0.01*(maxx-minx); pad_y = 0.01*(maxy-miny)
    ax.set_xlim(minx-pad_x, maxx+pad_x)
    ax.set_ylim(miny-pad_y, maxy+pad_y)

def add_cbar(ax, mappable, label, fontsize=11):
    cax = inset_axes(ax, width="3.0%", height="55%", loc="lower left",
                     bbox_to_anchor=(1.02, 0.16, 1, 1),
                     bbox_transform=ax.transAxes, borderpad=0)
    cb = plt.colorbar(mappable, cax=cax)
    cb.set_label(label, fontsize=fontsize)
    cb.ax.tick_params(labelsize=10)

# Panel A: BA grid intensity + site small markers
ax = axs[0]
draw_base(ax)
vmin = np.nanpercentile(gdf_ba_l["ba_grid_L_per_kwh_mw_weighted"].dropna(), 3)
vmax = np.nanpercentile(gdf_ba_l["ba_grid_L_per_kwh_mw_weighted"].dropna(), 97)
gdf_ba_l.plot(ax=ax, column="ba_grid_L_per_kwh_mw_weighted", cmap="viridis",
              linewidth=0.25, edgecolor="#ffffff", zorder=1,
              missing_kwds={"color":"#f0f0f0", "edgecolor":"#f0f0f0"})
# site markers: small filled circles with white outline
gdf_sites_l.plot(ax=ax, markersize=marker_size, column=None, color="#222222",
                 alpha=0.65, linewidth=0.35, edgecolor="white", zorder=3)
ax.set_title("a  Balancing-authority grid water intensity (L kWh$^{-1}$) + site locations", loc="left", fontsize=14)
m = ScalarMappable(norm=Normalize(vmin=vmin, vmax=vmax), cmap="viridis")
m.set_array([])
add_cbar(ax, m, "Grid water intensity (L/kWh)")

# Annotate only the top N sites (numbers)
for i, row in top_sites.iterrows():
    x, y = row.geometry.x, row.geometry.y
    ax.text(x + 12000, y + 12000, f"{i+1}", fontsize=9, fontweight="bold", color="#111111",
            bbox=dict(facecolor="white", edgecolor="none", pad=1), zorder=5)

# Panel B: Basin water stress + sites (small dots)
ax = axs[1]
draw_base(ax)
gdf_basin_l.plot(ax=ax, column=STRESS_COL_BAS, cmap="plasma", linewidth=0.12, edgecolor="#ffffff", zorder=1)
gdf_sites_l.plot(ax=ax, markersize=marker_size, color="#222222", alpha=0.65, edgecolor="white", linewidth=0.35, zorder=3)
ax.set_title("b  Basin baseline water stress (Aqueduct 0–5) + site locations", loc="left", fontsize=14)
m2 = ScalarMappable(norm=Normalize(vmin=0, vmax=5), cmap="plasma")
m2.set_array([])
add_cbar(ax, m2, "Baseline water stress (0–5)")

# Annotate top sites on this panel too (same numbers)
for i, row in top_sites.iterrows():
    x, y = row.geometry.x, row.geometry.y
    ax.text(x + 12000, y + 12000, f"{i+1}", fontsize=9, fontweight="bold", color="#111111",
            bbox=dict(facecolor="white", edgecolor="none", pad=1), zorder=5)

# Panel C: BA capacity (MW) choropleth
ax = axs[2]
draw_base(ax)
vmin3 = 0
vmax3 = np.nanpercentile(gdf_ba_l["mw_sum"].replace(0, np.nan), 97)
gdf_ba_l.plot(ax=ax, column="mw_sum", cmap="cividis", linewidth=0.25, edgecolor="#ffffff", zorder=1)
gdf_sites_l.plot(ax=ax, markersize=marker_size, color="#222222", alpha=0.65, edgecolor="white", linewidth=0.35, zorder=3)
ax.set_title("c  Hyperscale capacity by BA (MW) + site locations", loc="left", fontsize=14)
m3 = ScalarMappable(norm=Normalize(vmin=vmin3, vmax=vmax3), cmap="cividis")
m3.set_array([])
add_cbar(ax, m3, "Capacity (MW)")

# Panel D: Basin capacity (MW) choropleth
ax = axs[3]
draw_base(ax)
vmin4 = 0
vmax4 = np.nanpercentile(gdf_basin_l["mw_sum"].replace(0, np.nan), 97)
gdf_basin_l.plot(ax=ax, column="mw_sum", cmap="cividis", linewidth=0.08, edgecolor="#ffffff", zorder=1)
gdf_sites_l.plot(ax=ax, markersize=marker_size, color="#222222", alpha=0.65, edgecolor="white", linewidth=0.35, zorder=3)
ax.set_title("d  Hyperscale capacity by basin (MW) + site locations", loc="left", fontsize=14)
m4 = ScalarMappable(norm=Normalize(vmin=vmin4, vmax=vmax4), cmap="cividis")
m4.set_array([])
add_cbar(ax, m4, "Capacity (MW)")

# Super title and footer note
fig.suptitle("Figure 1 | Geographic context for Scope 1–2 water accounting (baseline)", y=0.975, fontsize=16)
# Add a small legend for numbered top sites: print to console or save mapping table
top_mapping = top_sites[[MW_COL, "site_name"]].reset_index().rename(columns={"index":"label_index"})
print("Top site annotation mapping (label number -> site_name, MW):")
for i, row in top_mapping.iterrows():
    print(f"{i+1}: {row['site_name']} ({row[MW_COL]:.0f} MW)")

# Save outputs
out_png = "Fig1_panel__improved_Lambert_states48.png"
out_pdf = "Fig1_panel__improved_Lambert_states48.pdf"
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.savefig(out_pdf, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", out_png, out_pdf)


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, LogNorm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from pyproj import CRS

# ----------- Inputs from your session --------------
gdf_sites = CFG["gdf_sites"].copy()
gdf_ba    = CFG["gdf_ba"].copy()
gdf_basin = CFG["gdf_basin"].copy()
states48  = globals().get("states48")
assert states48 is not None, "states48 must be present"

# Column names
MW_COL = "current_mw"
BA_INT_COL_SITE = "grid_L_per_kwh"
STRESS_COL_BAS = "bws_01_score"
BA_KEY_SITES = "subregion_id"
BASIN_KEY_SITES = "pfaf_id"
BA_KEY_BA = "subregion_id"
BASIN_KEY_BAS = "PFAF_ID"

# ---------- Lambert Conformal Conic (CONUS) ----------
LCC_CONUS = CRS.from_proj4(
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 "
    "+x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"
)

def to_lcc(gdf):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(LCC_CONUS)

def winsor_limits(x, qlo=0.02, qhi=0.98):
    x = pd.Series(x).dropna().astype(float)
    if len(x) == 0:
        return (0, 1)
    return (x.quantile(qlo), x.quantile(qhi))

def clip_series(s, lo, hi):
    return s.clip(lower=lo, upper=hi)

# Project
gdf_sites_l  = to_lcc(gdf_sites)
gdf_ba_l     = to_lcc(gdf_ba)
gdf_basin_l  = to_lcc(gdf_basin)
states48_l   = to_lcc(states48)

# Clip to CONUS
conus_geom = states48_l.unary_union
gdf_ba_l    = gdf_ba_l[gdf_ba_l.intersects(conus_geom)].copy()
gdf_basin_l = gdf_basin_l[gdf_basin_l.intersects(conus_geom)].copy()
gdf_sites_l = gdf_sites_l[gdf_sites_l.intersects(conus_geom)].copy()

# Normalize join keys
gdf_sites_l[BA_KEY_SITES] = gdf_sites_l[BA_KEY_SITES].astype(str)
gdf_ba_l[BA_KEY_BA]       = gdf_ba_l[BA_KEY_BA].astype(str)
gdf_sites_l[BASIN_KEY_SITES] = gdf_sites_l[BASIN_KEY_SITES].astype(str)
gdf_basin_l[BASIN_KEY_BAS]   = gdf_basin_l[BASIN_KEY_BAS].astype(str)

# Aggregate MW by BA / basin
ba_mw = (gdf_sites_l.groupby(BA_KEY_SITES)[MW_COL].sum()
         .reset_index().rename(columns={MW_COL: "mw_sum"}))
bas_mw = (gdf_sites_l.groupby(BASIN_KEY_SITES)[MW_COL].sum()
          .reset_index().rename(columns={MW_COL: "mw_sum"}))

gdf_ba_l = gdf_ba_l.merge(ba_mw, left_on=BA_KEY_BA, right_on=BA_KEY_SITES, how="left")
gdf_ba_l["mw_sum"] = gdf_ba_l["mw_sum"].fillna(0.0)

gdf_basin_l = gdf_basin_l.merge(bas_mw, left_on=BASIN_KEY_BAS, right_on=BASIN_KEY_SITES, how="left")
gdf_basin_l["mw_sum"] = gdf_basin_l["mw_sum"].fillna(0.0)

# MW-weighted BA grid intensity from site col (if present)
if BA_INT_COL_SITE in gdf_sites_l.columns:
    ba_int = (gdf_sites_l.dropna(subset=[BA_INT_COL_SITE, MW_COL])
              .groupby(BA_KEY_SITES)
              .apply(lambda d: np.average(d[BA_INT_COL_SITE].astype(float), weights=d[MW_COL].astype(float)))
              .reset_index(name="ba_grid_L_per_kwh_mw_weighted"))
    gdf_ba_l = gdf_ba_l.merge(ba_int, left_on=BA_KEY_BA, right_on=BA_KEY_SITES, how="left")

# ---- Site marker sizing (smaller + controlled) ----
mw = gdf_sites_l[MW_COL].fillna(0).astype(float).values
p90 = np.nanpercentile(mw[mw > 0], 90) if np.any(mw > 0) else 100.0
mw_clip = np.clip(mw, 0, p90)
marker_size = 5 + 40 * (np.sqrt(mw_clip) / (np.sqrt(p90) + 1e-9))  # smaller than before

# Annotate top sites (optional; keep low)
N_ANNOT = 8
top_sites = gdf_sites_l.sort_values(MW_COL, ascending=False).head(N_ANNOT).reset_index(drop=True)

# ---- Figure layout ----
fig = plt.figure(figsize=(22, 12))  # larger
gs = fig.add_gridspec(2, 2, wspace=0.08, hspace=0.10)
axs = [fig.add_subplot(gs[0,0]), fig.add_subplot(gs[0,1]),
       fig.add_subplot(gs[1,0]), fig.add_subplot(gs[1,1])]

# Extent
minx, miny, maxx, maxy = states48_l.total_bounds
pad_x = 0.01*(maxx-minx); pad_y = 0.01*(maxy-miny)

def draw_base(ax):
    # base fill
    states48_l.plot(ax=ax, color="white", edgecolor="none", zorder=0)
    ax.set_axis_off()
    ax.set_aspect("equal")
    ax.set_xlim(minx-pad_x, maxx+pad_x)
    ax.set_ylim(miny-pad_y, maxy+pad_y)

def draw_state_borders(ax):
    # CONTRASTING borders for orientation
    states48_l.boundary.plot(ax=ax, color="#111111", linewidth=0.9, alpha=0.9, zorder=10)

def add_cbar(ax, mappable, label, fontsize=11):
    cax = inset_axes(ax, width="3.0%", height="55%", loc="lower left",
                     bbox_to_anchor=(1.02, 0.16, 1, 1),
                     bbox_transform=ax.transAxes, borderpad=0)
    cb = plt.colorbar(mappable, cax=cax)
    cb.set_label(label, fontsize=fontsize)
    cb.ax.tick_params(labelsize=10)

# ---------------- Panel A: BA grid intensity map ----------------
ax = axs[0]
draw_base(ax)

colA = "ba_grid_L_per_kwh_mw_weighted"
loA, hiA = winsor_limits(gdf_ba_l[colA], 0.02, 0.98)
gdf_ba_l["_A_clipped"] = clip_series(gdf_ba_l[colA].astype(float), loA, hiA)

gdf_ba_l.plot(ax=ax, column="_A_clipped", cmap="viridis",
              linewidth=0.30, edgecolor="#f6f6f6", zorder=1,
              missing_kwds={"color":"#f0f0f0", "edgecolor":"#f0f0f0"})

# sites (subtle)
gdf_sites_l.plot(ax=ax, markersize=marker_size, color="#222222",
                 alpha=0.50, linewidth=0.35, edgecolor="white", zorder=3)

# state borders on top
draw_state_borders(ax)

ax.set_title("a  BA grid water intensity (L kWh$^{-1}$) with hyperscale sites", loc="left", fontsize=14)

mA = ScalarMappable(norm=Normalize(vmin=loA, vmax=hiA), cmap="viridis")
mA.set_array([])
add_cbar(ax, mA, "Grid water intensity (winsorized)")

# annotate top sites (numbers)
for i, row in top_sites.iterrows():
    x, y = row.geometry.x, row.geometry.y
    ax.text(x + 12000, y + 12000, f"{i+1}", fontsize=9, fontweight="bold",
            bbox=dict(facecolor="white", edgecolor="none", pad=1), zorder=20)

# ---------------- Panel B: Basin stress map ----------------
ax = axs[1]
draw_base(ax)

loB, hiB = winsor_limits(gdf_basin_l[STRESS_COL_BAS], 0.00, 0.995)  # stress bounded but keep near-max
gdf_basin_l["_B_clipped"] = clip_series(gdf_basin_l[STRESS_COL_BAS].astype(float), loB, hiB)

gdf_basin_l.plot(ax=ax, column="_B_clipped", cmap="plasma",
                 linewidth=0.10, edgecolor="#f8f8f8", zorder=1)

gdf_sites_l.plot(ax=ax, markersize=marker_size, color="#222222",
                 alpha=0.50, edgecolor="white", linewidth=0.35, zorder=3)

draw_state_borders(ax)

ax.set_title("b  Basin baseline water stress (Aqueduct 0–5) with hyperscale sites", loc="left", fontsize=14)

mB = ScalarMappable(norm=Normalize(vmin=0, vmax=5), cmap="plasma")
mB.set_array([])
add_cbar(ax, mB, "Baseline water stress (0–5)")

for i, row in top_sites.iterrows():
    x, y = row.geometry.x, row.geometry.y
    ax.text(x + 12000, y + 12000, f"{i+1}", fontsize=9, fontweight="bold",
            bbox=dict(facecolor="white", edgecolor="none", pad=1), zorder=20)

# ---------------- Panel C: BA capacity map (log norm) ----------------
ax = axs[2]
draw_base(ax)

# LogNorm handles heavy tails nicely for capacity
cap = gdf_ba_l["mw_sum"].astype(float)
cap_nonzero = cap[cap > 0]
vminC = max(cap_nonzero.min(), 1.0) if len(cap_nonzero) else 1.0
vmaxC = cap_nonzero.quantile(0.98) if len(cap_nonzero) else 10.0

gdf_ba_l.plot(ax=ax, column="mw_sum", cmap="cividis",
              linewidth=0.30, edgecolor="#f6f6f6", zorder=1,
              norm=LogNorm(vmin=vminC, vmax=vmaxC))

gdf_sites_l.plot(ax=ax, markersize=marker_size, color="#222222",
                 alpha=0.45, edgecolor="white", linewidth=0.35, zorder=3)

draw_state_borders(ax)

ax.set_title("c  Hyperscale capacity by BA (MW, log-scaled) with sites", loc="left", fontsize=14)

mC = ScalarMappable(norm=LogNorm(vmin=vminC, vmax=vmaxC), cmap="cividis")
mC.set_array([])
add_cbar(ax, mC, "Capacity (MW, log)")

# ---------------- Panel D: NOT a map — basin Pareto concentration ----------------
ax = axs[3]
ax.set_axis_on()

# Build Pareto curve: rank basins by MW and plot cumulative share
bas_cap = (gdf_basin_l[[BASIN_KEY_BAS, "mw_sum"]]
           .dropna()
           .assign(mw_sum=lambda d: d["mw_sum"].astype(float)))
bas_cap = bas_cap[bas_cap["mw_sum"] > 0].sort_values("mw_sum", ascending=False).reset_index(drop=True)

if len(bas_cap) > 0:
    bas_cap["rank"] = np.arange(1, len(bas_cap)+1)
    bas_cap["cum_mw"] = bas_cap["mw_sum"].cumsum()
    total = bas_cap["mw_sum"].sum()
    bas_cap["cum_share"] = bas_cap["cum_mw"] / total

    ax.plot(bas_cap["rank"], bas_cap["cum_share"], linewidth=2.2)
    ax.set_ylim(0, 1.01)
    ax.set_xlim(1, len(bas_cap))
    ax.set_xlabel("Basin rank (by hyperscale capacity)", fontsize=12)
    ax.set_ylabel("Cumulative share of total hyperscale capacity", fontsize=12)
    ax.grid(True, alpha=0.25)

    # annotate concentration points
    for k in [5, 10, 20, 50, 100]:
        if k <= len(bas_cap):
            y = bas_cap.loc[k-1, "cum_share"]
            ax.scatter([k], [y], s=35)
            ax.text(k, y + 0.03, f"{k}", fontsize=10)

    # Optional: share in top N
    top10 = bas_cap.loc[min(9, len(bas_cap)-1), "cum_share"]
    ax.text(0.02, 0.10, f"Top 10 basins = {top10:.0%} of total MW",
            transform=ax.transAxes, fontsize=12)

else:
    ax.text(0.5, 0.5, "No basin capacity data available", ha="center", va="center")

ax.set_title("d  Concentration of hyperscale capacity across basins (Pareto curve)", loc="left", fontsize=14)

# Suptitle and save
fig.suptitle("Figure 1 | Geographic context and concentration for Scope 1–2 water accounting (baseline)", y=0.98, fontsize=16)

out_png = "Fig1_panel__improved_v3_states_borders_outliers_pareto.png"
out_pdf = "Fig1_panel__improved_v3_states_borders_outliers_pareto.pdf"
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.savefig(out_pdf, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out_png, out_pdf)

print("\nTop-site annotation mapping (label number -> site_name, MW):")
for i, row in top_sites.iterrows():
    print(f"{i+1}: {row.get('site_name','(no name)')} ({float(row[MW_COL]):.0f} MW)")


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, LogNorm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from pyproj import CRS

# ---------------- Inputs ----------------
gdf_sites  = CFG["gdf_sites"].copy()
gdf_basin  = CFG["gdf_basin"].copy()
states48   = globals().get("states48")
assert states48 is not None, "states48 must exist in your session"

MW_COL = "current_mw"
BASIN_KEY_SITES = "pfaf_id"
BASIN_KEY_BAS   = "PFAF_ID"

# ------------- Projection (Lambert CONUS) -------------
LCC_CONUS = CRS.from_proj4(
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 "
    "+x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"
)

def to_lcc(gdf):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(LCC_CONUS)

def winsor_limits(x, qlo=0.02, qhi=0.98):
    x = pd.Series(x).dropna().astype(float)
    if len(x) == 0:
        return (0, 1)
    return (x.quantile(qlo), x.quantile(qhi))

# Project + clip to CONUS
gdf_sites_l  = to_lcc(gdf_sites)
gdf_basin_l  = to_lcc(gdf_basin)
states48_l   = to_lcc(states48)

conus_geom = states48_l.unary_union
gdf_basin_l = gdf_basin_l[gdf_basin_l.intersects(conus_geom)].copy()
gdf_sites_l = gdf_sites_l[gdf_sites_l.intersects(conus_geom)].copy()

# Normalize keys
gdf_sites_l[BASIN_KEY_SITES] = gdf_sites_l[BASIN_KEY_SITES].astype(str)
gdf_basin_l[BASIN_KEY_BAS]   = gdf_basin_l[BASIN_KEY_BAS].astype(str)

# -------- Aggregate hyperscale MW to basins --------
bas_mw = (gdf_sites_l.groupby(BASIN_KEY_SITES)[MW_COL].sum()
          .reset_index().rename(columns={MW_COL: "mw_sum"}))

gdf_basin_l = gdf_basin_l.merge(
    bas_mw, left_on=BASIN_KEY_BAS, right_on=BASIN_KEY_SITES, how="left"
)
gdf_basin_l["mw_sum"] = gdf_basin_l["mw_sum"].fillna(0.0)

# Basin area (km2) in Lambert meters -> km2
gdf_basin_l["area_km2"] = gdf_basin_l.geometry.area / 1e6

# MW density option (MW per 1000 km2 is usually nicer magnitude)
gdf_basin_l["mw_per_1000_km2"] = np.where(
    gdf_basin_l["area_km2"] > 0,
    gdf_basin_l["mw_sum"] / (gdf_basin_l["area_km2"] / 1000.0),
    0.0
)

# ----------------- Choose metric + scaling -----------------
USE_DENSITY = True   # True: MW density; False: total MW
USE_LOG     = True   # True: LogNorm; False: winsorized linear

value_col = "mw_per_1000_km2" if USE_DENSITY else "mw_sum"
vals = gdf_basin_l[value_col].astype(float)

# ignore zeros for scaling limits
vals_nz = vals[vals > 0]

if USE_LOG:
    vmin = max(vals_nz.min(), 1e-3) if len(vals_nz) else 1e-3
    vmax = vals_nz.quantile(0.98) if len(vals_nz) else 1.0
    norm = LogNorm(vmin=vmin, vmax=vmax)
    cbar_label = f"{'MW per 1,000 km$^2$' if USE_DENSITY else 'MW'} (log-scaled, p98 max)"
else:
    lo, hi = winsor_limits(vals_nz, 0.02, 0.98) if len(vals_nz) else (0, 1)
    # keep zeros as zeros; clip positives to [lo, hi]
    gdf_basin_l["_clip"] = vals.clip(lower=0, upper=hi)
    norm = Normalize(vmin=0, vmax=hi)
    value_col = "_clip"
    cbar_label = f"{'MW per 1,000 km$^2$' if USE_DENSITY else 'MW'} (winsorized p98)"

# ----------------- Top basins outline (no dots) -----------------
TOP_N = 15
top_basins = gdf_basin_l.sort_values("mw_sum", ascending=False).head(TOP_N)

# ----------------- Plot -----------------
fig, ax = plt.subplots(figsize=(18, 10))

# base
states48_l.plot(ax=ax, color="white", edgecolor="none", zorder=0)

# basin fill (no sites)
gdf_basin_l.plot(
    ax=ax, column=value_col, cmap="cividis",
    linewidth=0.08, edgecolor="#f2f2f2", zorder=1,
    norm=norm,
    missing_kwds={"color":"#f0f0f0", "edgecolor":"#f0f0f0"}
)

# states borders for orientation (contrasting)
states48_l.boundary.plot(ax=ax, color="#111111", linewidth=0.9, alpha=0.9, zorder=10)

# outline top basins
top_basins.boundary.plot(ax=ax, color="#ffcc00", linewidth=1.6, alpha=0.95, zorder=20)

# extent
minx, miny, maxx, maxy = states48_l.total_bounds
pad_x = 0.01*(maxx-minx); pad_y = 0.01*(maxy-miny)
ax.set_xlim(minx-pad_x, maxx+pad_x)
ax.set_ylim(miny-pad_y, maxy+pad_y)
ax.set_axis_off()
ax.set_aspect("equal")

title_metric = "MW density (MW per 1,000 km$^2$)" if USE_DENSITY else "total MW"
ax.set_title(
    f"Hyperscale capacity by basin ({title_metric}) — top {TOP_N} basins outlined",
    loc="left", fontsize=16
)

# colorbar
cax = inset_axes(ax, width="2.8%", height="55%", loc="lower left",
                 bbox_to_anchor=(1.02, 0.18, 1, 1),
                 bbox_transform=ax.transAxes, borderpad=0)
sm = ScalarMappable(norm=norm, cmap="cividis")
sm.set_array([])
cb = plt.colorbar(sm, cax=cax)
cb.set_label(cbar_label, fontsize=12)
cb.ax.tick_params(labelsize=10)

out_png = "Fig_basin_capacity_density_no_dots.png"
out_pdf = "Fig_basin_capacity_density_no_dots.pdf"
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.savefig(out_pdf, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out_png, out_pdf)

print("\nTop basins by MW (for caption / callouts):")
print(top_basins[[BASIN_KEY_BAS, "mw_sum", "mw_per_1000_km2"]].head(TOP_N).to_string(index=False))


#### tweaks for this map
Use total MW instead of density (sometimes reviewers prefer absolute burden): set USE_DENSITY=False.

Make the outlines less “loud”: change TOP_BORDER to a dark neutral (e.g., "#111111") and keep linewidth 1.2.

Make it consistent across scenarios (same color scale for eff/base/high): I’ll give you a tiny helper that computes a global vmax from all three scenario tables and pins it for all panels.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, LogNorm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from pyproj import CRS

# -------------------------
# Panel-ready basin choropleth (no dots)
# -------------------------

# ---- Required in your session:
# CFG["gdf_sites"], CFG["gdf_basin"], states48

# -------------------------
# Settings (edit these)
# -------------------------
PANEL_LETTER   = "d"
TITLE          = "Hyperscale capacity by basin"
SUBTITLE       = "Basins colored by capacity density; top basins outlined"

MW_COL         = "current_mw"   # use "current_mw" for MW aggregation
BASIN_KEY_SITES = "pfaf_id"
BASIN_KEY_BAS   = "PFAF_ID"

USE_DENSITY    = True   # True: MW per 1,000 km^2 ; False: total MW
SCALE_MODE     = "log"  # "log" or "winsor" (linear with p98 cap)
WINSOR_QHI     = 0.98
WINSOR_QLO     = 0.02   # only used for diagnostics
TOP_N_OUTLINE  = 15

# figure aesthetics
FIGSIZE        = (10.5, 6.2)   # single panel size (tune for your multipanel layout)
DPI            = 450
CMAP           = "cividis"     # colorblind-friendly + good print
BASIN_EDGE     = "#f2f2f2"
BASIN_EDGE_LW  = 0.08
STATE_BORDER   = "#111111"
STATE_LW       = 0.85
TOP_BORDER     = "#ffcc00"
TOP_LW         = 1.6

# output names
OUT_TAG        = "base"  # change to "efficiency" / "high_load" if needed
OUT_PNG        = f"Fig1_{PANEL_LETTER}_basin_capacityDensity_noDots_{OUT_TAG}.png"
OUT_PDF        = f"Fig1_{PANEL_LETTER}_basin_capacityDensity_noDots_{OUT_TAG}.pdf"

# -------------------------
# Projection: Lambert Conformal (CONUS)
# -------------------------
LCC_CONUS = CRS.from_proj4(
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 "
    "+x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"
)

def _to_crs(gdf, crs=LCC_CONUS):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(crs)

def _winsor_limits(x, qlo=0.02, qhi=0.98):
    x = pd.Series(x).dropna().astype(float)
    if len(x) == 0:
        return (0.0, 1.0)
    return (float(x.quantile(qlo)), float(x.quantile(qhi)))

# -------------------------
# Load & project
# -------------------------
assert "gdf_sites" in CFG and CFG["gdf_sites"] is not None, "CFG['gdf_sites'] missing"
assert "gdf_basin" in CFG and CFG["gdf_basin"] is not None, "CFG['gdf_basin'] missing"
assert "states48" in globals(), "states48 missing in environment"

gdf_sites  = _to_crs(CFG["gdf_sites"].copy())
gdf_basin  = _to_crs(CFG["gdf_basin"].copy())
states48_l = _to_crs(states48.copy())

# clip to CONUS
conus_geom = states48_l.unary_union
gdf_basin = gdf_basin[gdf_basin.intersects(conus_geom)].copy()
gdf_sites = gdf_sites[gdf_sites.intersects(conus_geom)].copy()

# normalize join keys
gdf_sites[BASIN_KEY_SITES] = gdf_sites[BASIN_KEY_SITES].astype(str)
gdf_basin[BASIN_KEY_BAS]   = gdf_basin[BASIN_KEY_BAS].astype(str)

# -------------------------
# Aggregate MW to basins
# -------------------------
bas_mw = (
    gdf_sites.groupby(BASIN_KEY_SITES)[MW_COL]
    .sum()
    .reset_index()
    .rename(columns={MW_COL: "mw_sum"})
)

gdf_basin = gdf_basin.merge(
    bas_mw, left_on=BASIN_KEY_BAS, right_on=BASIN_KEY_SITES, how="left"
)
gdf_basin["mw_sum"] = gdf_basin["mw_sum"].fillna(0.0)

# area km^2 (Lambert is meters)
gdf_basin["area_km2"] = gdf_basin.geometry.area / 1e6

# MW density (MW per 1,000 km2)
gdf_basin["mw_per_1000_km2"] = np.where(
    gdf_basin["area_km2"] > 0,
    gdf_basin["mw_sum"] / (gdf_basin["area_km2"] / 1000.0),
    0.0
)

value_col = "mw_per_1000_km2" if USE_DENSITY else "mw_sum"
vals = gdf_basin[value_col].astype(float)
vals_nz = vals[vals > 0]

# -------------------------
# Scaling / outlier control
# -------------------------
if SCALE_MODE.lower() == "log":
    vmin = max(float(vals_nz.min()), 1e-3) if len(vals_nz) else 1e-3
    vmax = float(vals_nz.quantile(WINSOR_QHI)) if len(vals_nz) else 1.0
    norm = LogNorm(vmin=vmin, vmax=vmax)
    scale_note = f"log scale (cap at p{int(WINSOR_QHI*100)})"
else:
    lo, hi = _winsor_limits(vals_nz, WINSOR_QLO, WINSOR_QHI) if len(vals_nz) else (0.0, 1.0)
    gdf_basin["_clip"] = vals.clip(lower=0.0, upper=hi)
    value_col = "_clip"
    norm = Normalize(vmin=0.0, vmax=hi)
    scale_note = f"linear scale (winsorized to p{int(WINSOR_QHI*100)})"

unit_label = "MW per 1,000 km$^2$" if USE_DENSITY else "MW"
cbar_label = f"{unit_label} ({scale_note})"

# -------------------------
# Top basins outline (by total MW, not density)
# -------------------------
top_basins = gdf_basin.sort_values("mw_sum", ascending=False).head(TOP_N_OUTLINE)

# -------------------------
# Plot
# -------------------------
fig, ax = plt.subplots(figsize=FIGSIZE)

# base canvas
states48_l.plot(ax=ax, color="white", edgecolor="none", zorder=0)

# basin choropleth
gdf_basin.plot(
    ax=ax,
    column=value_col,
    cmap=CMAP,
    norm=norm,
    linewidth=BASIN_EDGE_LW,
    edgecolor=BASIN_EDGE,
    zorder=1,
    missing_kwds={"color": "#f0f0f0", "edgecolor": "#f0f0f0"},
)

# state borders (contrast)
states48_l.boundary.plot(ax=ax, color=STATE_BORDER, linewidth=STATE_LW, alpha=0.95, zorder=10)

# highlight top basins with outline only
top_basins.boundary.plot(ax=ax, color=TOP_BORDER, linewidth=TOP_LW, alpha=0.95, zorder=20)

# extent with small padding
minx, miny, maxx, maxy = states48_l.total_bounds
pad_x = 0.01 * (maxx - minx)
pad_y = 0.01 * (maxy - miny)
ax.set_xlim(minx - pad_x, maxx + pad_x)
ax.set_ylim(miny - pad_y, maxy + pad_y)
ax.set_aspect("equal")
ax.set_axis_off()

# panel letter + title (Nature-ish)
ax.text(
    0.0, 1.02, PANEL_LETTER,
    transform=ax.transAxes, fontsize=14, fontweight="bold", va="bottom", ha="left"
)
ax.text(
    0.04, 1.02, f"{TITLE}",
    transform=ax.transAxes, fontsize=13, va="bottom", ha="left"
)
ax.text(
    0.04, 0.99, f"{SUBTITLE}",
    transform=ax.transAxes, fontsize=10.5, va="top", ha="left", color="#333333"
)

# colorbar (inset on right)
cax = inset_axes(
    ax, width="3.2%", height="62%",
    loc="lower left",
    bbox_to_anchor=(1.02, 0.17, 1, 1),
    bbox_transform=ax.transAxes,
    borderpad=0
)
sm = ScalarMappable(norm=norm, cmap=CMAP)
sm.set_array([])
cb = plt.colorbar(sm, cax=cax)
cb.set_label(cbar_label, fontsize=10.5)
cb.ax.tick_params(labelsize=9)

# save
plt.savefig(OUT_PNG, dpi=DPI, bbox_inches="tight")
plt.savefig(OUT_PDF, dpi=DPI, bbox_inches="tight")
plt.show()

print("Saved:", OUT_PNG, OUT_PDF)

# handy caption support: list top basins
print("\nTop basins by total MW (outline set):")
print(top_basins[[BASIN_KEY_BAS, "mw_sum", "mw_per_1000_km2"]].to_string(index=False))


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, LogNorm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from pyproj import CRS

# ============================================================
# Scope 1 basin context: (Left) MW density, (Right) scarcity
# Top-N (by total MW) outlined in RED on BOTH maps.
# ============================================================

# -------------------------
# Settings
# -------------------------
TOP_N_OUTLINE  = 15
MW_COL         = "current_mw"
BASIN_KEY_SITES = "pfaf_id"
BASIN_KEY_BAS   = "PFAF_ID"

# Left map (MW density)
USE_DENSITY    = True          # MW per 1,000 km2
SCALE_MODE_MW  = "log"         # "log" or "winsor"
WINSOR_QHI_MW  = 0.98          # cap outliers for colorbar
CMAP_MW        = "cividis"

# Right map (scarcity)
SCARCITY_COL   = "bws_01_score"  # Aqueduct baseline score (0–5)
CMAP_SCARCITY  = "viridis"        # vivid, works well for 0–5

# Styling
FIGSIZE        = (16.5, 7.2)   # big, Nature-ish
DPI            = 450
BASIN_EDGE     = "#f2f2f2"
BASIN_EDGE_LW  = 0.08
STATE_BORDER   = "#111111"
STATE_LW       = 0.85
TOP_BORDER     = "#cc0000"     # RED outline
TOP_LW         = 1.8

SUPTITLE       = "Scope 1 siting context (basins): capacity density vs scarcity"
LEFT_TITLE     = "Hyperscale capacity density by basin"
RIGHT_TITLE    = "Baseline water scarcity by basin (Aqueduct)"
LEFT_SUB       = "Basins colored by MW per 1,000 km$^2$; top basins (by total MW) outlined"
RIGHT_SUB      = "Basins colored by Aqueduct baseline stress score (0–5); same top basins outlined"

OUT_PNG        = "S1_Basins_MWdensity_vs_Scarcity_top15_red.png"
OUT_PDF        = "S1_Basins_MWdensity_vs_Scarcity_top15_red.pdf"

# -------------------------
# Projection: Lambert Conformal (CONUS)
# -------------------------
LCC_CONUS = CRS.from_proj4(
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 "
    "+x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"
)

def _to_crs(gdf, crs=LCC_CONUS):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(crs)

def _winsor_limits(x, qhi=0.98):
    x = pd.Series(x).dropna().astype(float)
    if len(x) == 0:
        return 1.0
    return float(x.quantile(qhi))

# -------------------------
# Load & project
# -------------------------
assert "gdf_sites" in CFG and CFG["gdf_sites"] is not None, "CFG['gdf_sites'] missing"
assert "gdf_basin" in CFG and CFG["gdf_basin"] is not None, "CFG['gdf_basin'] missing"
assert "states48" in globals(), "states48 missing in environment"

gdf_sites  = _to_crs(CFG["gdf_sites"].copy())
gdf_basin  = _to_crs(CFG["gdf_basin"].copy())
states48_l = _to_crs(states48.copy())

# clip to CONUS
conus_geom = states48_l.unary_union
gdf_basin = gdf_basin[gdf_basin.intersects(conus_geom)].copy()
gdf_sites = gdf_sites[gdf_sites.intersects(conus_geom)].copy()

# normalize join keys
gdf_sites[BASIN_KEY_SITES] = gdf_sites[BASIN_KEY_SITES].astype(str)
gdf_basin[BASIN_KEY_BAS]   = gdf_basin[BASIN_KEY_BAS].astype(str)

# -------------------------
# Aggregate MW to basins
# -------------------------
bas_mw = (
    gdf_sites.groupby(BASIN_KEY_SITES)[MW_COL]
    .sum()
    .reset_index()
    .rename(columns={MW_COL: "mw_sum"})
)

gdf_basin = gdf_basin.merge(
    bas_mw, left_on=BASIN_KEY_BAS, right_on=BASIN_KEY_SITES, how="left"
)
gdf_basin["mw_sum"] = gdf_basin["mw_sum"].fillna(0.0)

# area km^2 (Lambert is meters)
gdf_basin["area_km2"] = gdf_basin.geometry.area / 1e6
gdf_basin["mw_per_1000_km2"] = np.where(
    gdf_basin["area_km2"] > 0,
    gdf_basin["mw_sum"] / (gdf_basin["area_km2"] / 1000.0),
    0.0
)

# pick value for left map
mw_value_col = "mw_per_1000_km2" if USE_DENSITY else "mw_sum"
mw_vals = gdf_basin[mw_value_col].astype(float)
mw_vals_nz = mw_vals[mw_vals > 0]

# MW scaling
if SCALE_MODE_MW.lower() == "log":
    mw_vmin = max(float(mw_vals_nz.min()), 1e-3) if len(mw_vals_nz) else 1e-3
    mw_vmax = float(mw_vals_nz.quantile(WINSOR_QHI_MW)) if len(mw_vals_nz) else 1.0
    mw_norm = LogNorm(vmin=mw_vmin, vmax=mw_vmax)
    mw_note = f"log scale (cap at p{int(WINSOR_QHI_MW*100)})"
else:
    mw_cap = _winsor_limits(mw_vals_nz, qhi=WINSOR_QHI_MW) if len(mw_vals_nz) else 1.0
    gdf_basin["_mw_clip"] = mw_vals.clip(lower=0.0, upper=mw_cap)
    mw_value_col = "_mw_clip"
    mw_norm = Normalize(vmin=0.0, vmax=mw_cap)
    mw_note = f"linear scale (winsorized to p{int(WINSOR_QHI_MW*100)})"

mw_unit_label = "MW per 1,000 km$^2$" if USE_DENSITY else "MW"
mw_cbar_label = f"{mw_unit_label} ({mw_note})"

# Scarcity scaling: Aqueduct baseline score is already 0–5, so keep linear 0–5
assert SCARCITY_COL in gdf_basin.columns, f"Missing {SCARCITY_COL} on gdf_basin"
scar_vals = gdf_basin[SCARCITY_COL].astype(float)
scar_norm = Normalize(vmin=0.0, vmax=5.0)
scar_cbar_label = "Baseline water stress (0–5)"

# -------------------------
# Top basins: by total MW (outline set)
# -------------------------
top_basins = gdf_basin.sort_values("mw_sum", ascending=False).head(TOP_N_OUTLINE)

# -------------------------
# Plot (side-by-side)
# -------------------------
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)

for ax in axes:
    states48_l.plot(ax=ax, color="white", edgecolor="none", zorder=0)

# Left: MW density
ax0 = axes[0]
gdf_basin.plot(
    ax=ax0,
    column=mw_value_col,
    cmap=CMAP_MW,
    norm=mw_norm,
    linewidth=BASIN_EDGE_LW,
    edgecolor=BASIN_EDGE,
    zorder=1,
    missing_kwds={"color": "#f0f0f0", "edgecolor": "#f0f0f0"},
)
states48_l.boundary.plot(ax=ax0, color=STATE_BORDER, linewidth=STATE_LW, alpha=0.95, zorder=10)
top_basins.boundary.plot(ax=ax0, color=TOP_BORDER, linewidth=TOP_LW, alpha=0.95, zorder=20)

# Right: Scarcity
ax1 = axes[1]
gdf_basin.plot(
    ax=ax1,
    column=SCARCITY_COL,
    cmap=CMAP_SCARCITY,
    norm=scar_norm,
    linewidth=BASIN_EDGE_LW,
    edgecolor=BASIN_EDGE,
    zorder=1,
    missing_kwds={"color": "#f0f0f0", "edgecolor": "#f0f0f0"},
)
states48_l.boundary.plot(ax=ax1, color=STATE_BORDER, linewidth=STATE_LW, alpha=0.95, zorder=10)
top_basins.boundary.plot(ax=ax1, color=TOP_BORDER, linewidth=TOP_LW, alpha=0.95, zorder=20)

# Shared extent
minx, miny, maxx, maxy = states48_l.total_bounds
pad_x = 0.01 * (maxx - minx)
pad_y = 0.01 * (maxy - miny)
for ax in axes:
    ax.set_xlim(minx - pad_x, maxx + pad_x)
    ax.set_ylim(miny - pad_y, maxy + pad_y)
    ax.set_aspect("equal")
    ax.set_axis_off()

# Titles
ax0.text(0.0, 1.02, "a", transform=ax0.transAxes, fontsize=14, fontweight="bold", va="bottom")
ax0.text(0.045, 1.02, LEFT_TITLE, transform=ax0.transAxes, fontsize=13, va="bottom")
ax0.text(0.045, 0.99, LEFT_SUB, transform=ax0.transAxes, fontsize=10.5, va="top", color="#333333")

ax1.text(0.0, 1.02, "b", transform=ax1.transAxes, fontsize=14, fontweight="bold", va="bottom")
ax1.text(0.045, 1.02, RIGHT_TITLE, transform=ax1.transAxes, fontsize=13, va="bottom")
ax1.text(0.045, 0.99, RIGHT_SUB, transform=ax1.transAxes, fontsize=10.5, va="top", color="#333333")

fig.suptitle(SUPTITLE, fontsize=14, y=1.02)

# Colorbars (one per axis)
cax0 = inset_axes(ax0, width="3.2%", height="62%", loc="lower left",
                  bbox_to_anchor=(1.02, 0.17, 1, 1), bbox_transform=ax0.transAxes, borderpad=0)
sm0 = ScalarMappable(norm=mw_norm, cmap=CMAP_MW); sm0.set_array([])
cb0 = plt.colorbar(sm0, cax=cax0)
cb0.set_label(mw_cbar_label, fontsize=10.5)
cb0.ax.tick_params(labelsize=9)

cax1 = inset_axes(ax1, width="3.2%", height="62%", loc="lower left",
                  bbox_to_anchor=(1.02, 0.17, 1, 1), bbox_transform=ax1.transAxes, borderpad=0)
sm1 = ScalarMappable(norm=scar_norm, cmap=CMAP_SCARCITY); sm1.set_array([])
cb1 = plt.colorbar(sm1, cax=cax1)
cb1.set_label(scar_cbar_label, fontsize=10.5)
cb1.ax.tick_params(labelsize=9)

plt.tight_layout()
plt.savefig(OUT_PNG, dpi=DPI, bbox_inches="tight")
plt.savefig(OUT_PDF, dpi=DPI, bbox_inches="tight")
plt.show()

print("Saved:", OUT_PNG, OUT_PDF)

# -------------------------
# Optional: ranking readout table for the top basins
# -------------------------
rank_tbl = top_basins[[BASIN_KEY_BAS, "mw_sum", "mw_per_1000_km2", SCARCITY_COL]].copy()
rank_tbl["mw_rank"] = rank_tbl["mw_sum"].rank(ascending=False, method="dense").astype(int)
rank_tbl["scarcity_rank_among_all_basins"] = gdf_basin[SCARCITY_COL].rank(ascending=False, method="dense").loc[rank_tbl.index].astype(int).values
rank_tbl = rank_tbl.sort_values("mw_sum", ascending=False)
print("\nTop basins by MW (with scarcity):")
print(rank_tbl.to_string(index=False))


In [ ]:
# === Side-by-side focus maps: top-30 basins highlighted, others muted ===
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, LogNorm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from pyproj import CRS

# --------------------------------
# Settings (tweak as desired)
# --------------------------------
TOP_N = 30
MW_COL = "current_mw"
SITE_BASIN_KEY = "pfaf_id"   # in gdf_sites
BASIN_KEY = "PFAF_ID"        # in gdf_basin
SCARCITY_COL = "bws_01_score"  # 0-5

FIGSIZE = (17.0, 7.5)
DPI = 300

LCC_CONUS = CRS.from_proj4(
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 +datum=WGS84 +units=m +no_defs"
)

MW_USE_DENSITY = True   # True -> MW per 1,000 km^2 else total MW
MW_CMAP = "cividis"
SCARCITY_CMAP = "plasma"
MUTED_COLOR = "#dcdcdc"   # fill for muted basins
MUTED_ALPHA = 0.33
BASIN_EDGE = "#efefef"
BASIN_EDGE_LW = 0.06
STATE_BORDER = "#111111"
STATE_LW = 0.8
TOP_EDGE_COLOR = "#cc0000"
TOP_EDGE_LW = 1.6

OUT_PNG = "S1_Basins_top30_focus_red.png"
OUT_PDF = "S1_Basins_top30_focus_red.pdf"

# --------------------------------
# Helper: to CRs (Lambert)
# --------------------------------
def to_lcc(gdf, crs=LCC_CONUS):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(crs)

# --------------------------------
# Load & sanity checks (use CFG gdfs)
# --------------------------------
assert "gdf_sites" in CFG and CFG["gdf_sites"] is not None, "CFG['gdf_sites'] missing"
assert "gdf_basin" in CFG and CFG["gdf_basin"] is not None, "CFG['gdf_basin'] missing"
assert "states48" in globals(), "states48 missing"

gdf_sites = CFG["gdf_sites"].copy()
gdf_basin = CFG["gdf_basin"].copy()
states48_g = states48.copy()

# project
gdf_sites = to_lcc(gdf_sites)
gdf_basin = to_lcc(gdf_basin)
states48_l = to_lcc(states48_g)

# clip to CONUS (using states48)
conus_geom = states48_l.unary_union
gdf_basin = gdf_basin[gdf_basin.intersects(conus_geom)].copy()
gdf_sites = gdf_sites[gdf_sites.intersects(conus_geom)].copy()

# normalize basin id keys to str
if SITE_BASIN_KEY not in gdf_sites.columns:
    raise KeyError(f"Missing {SITE_BASIN_KEY} in sites")
if BASIN_KEY not in gdf_basin.columns:
    raise KeyError(f"Missing {BASIN_KEY} in basins")
gdf_sites[SITE_BASIN_KEY] = gdf_sites[SITE_BASIN_KEY].astype(str)
gdf_basin[BASIN_KEY] = gdf_basin[BASIN_KEY].astype(str)

# --------------------------------
# Aggregate MW per basin
# --------------------------------
bas_mw = gdf_sites.groupby(SITE_BASIN_KEY)[MW_COL].sum().reset_index().rename(columns={MW_COL: "mw_sum"})
gdf_basin = gdf_basin.merge(bas_mw, left_on=BASIN_KEY, right_on=SITE_BASIN_KEY, how="left")
gdf_basin["mw_sum"] = gdf_basin["mw_sum"].fillna(0.0)

# compute area and density
gdf_basin["area_km2"] = gdf_basin.geometry.area / 1e6
gdf_basin["mw_per_1000_km2"] = np.where(
    gdf_basin["area_km2"] > 0,
    gdf_basin["mw_sum"] / (gdf_basin["area_km2"] / 1000.0),
    0.0
)

left_col = "mw_per_1000_km2" if MW_USE_DENSITY else "mw_sum"

# prepare color normalization for MW: lognorm with winsor cap at p98 to control outliers
mw_vals = gdf_basin[left_col].replace(0, np.nan).dropna()
if len(mw_vals):
    vmax = float(mw_vals.quantile(0.98))
    vmin = max(float(mw_vals.min()), 1e-3)
    from matplotlib.colors import LogNorm
    mw_norm = LogNorm(vmin=vmin, vmax=vmax)
else:
    from matplotlib.colors import Normalize
    mw_norm = Normalize(vmin=0, vmax=1)

# scarcity normalization (0--5)
if SCARCITY_COL not in gdf_basin.columns:
    raise KeyError(f"{SCARCITY_COL} missing in basin geodataframe")
scar_norm = Normalize(vmin=0.0, vmax=5.0)

# --------------------------------
# select top-N basins by total MW
# --------------------------------
top_basins = gdf_basin.sort_values("mw_sum", ascending=False).head(TOP_N)
top_ids = set(top_basins[BASIN_KEY].tolist())

# create masks
gdf_basin["_is_top"] = gdf_basin[BASIN_KEY].isin(top_ids)

# --------------------------------
# Plotting: side-by-side
# --------------------------------
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=FIGSIZE, constrained_layout=True)

# --- left: MW density ---
# muted (other basins): draw with muted color / low alpha
others = gdf_basin[~gdf_basin["_is_top"]].copy()
if len(others):
    others.plot(ax=ax0, color=MUTED_COLOR, edgecolor=BASIN_EDGE, linewidth=BASIN_EDGE_LW, alpha=MUTED_ALPHA, zorder=1)
# top basins: full color by density
tops = gdf_basin[gdf_basin["_is_top"]].copy()
if len(tops):
    tops.plot(ax=ax0, column=left_col, cmap=MW_CMAP, norm=mw_norm,
              edgecolor=TOP_EDGE_COLOR, linewidth=TOP_EDGE_LW, alpha=1.0, zorder=5)
# states borders on top
states48_l.boundary.plot(ax=ax0, color=STATE_BORDER, linewidth=STATE_LW, zorder=10)
ax0.set_title("a. Hyperscale capacity density by basin (top-30 highlighted)", fontsize=12)
ax0.set_axis_off()

# colorbar for left
from matplotlib.cm import ScalarMappable
sm_mw = ScalarMappable(norm=mw_norm, cmap=MW_CMAP)
sm_mw.set_array([])
cax0 = inset_axes(ax0, width="3.0%", height="55%", loc="lower left",
                  bbox_to_anchor=(1.02, 0.18, 1, 1), bbox_transform=ax0.transAxes, borderpad=0)
cb0 = fig.colorbar(sm_mw, cax=cax0)
unit = "MW per 1,000 km²" if MW_USE_DENSITY else "MW"
cb0.set_label(unit + " (log-scale, p98 cap)", fontsize=9)
cb0.ax.tick_params(labelsize=8)

# --- right: scarcity, but emphasize top basins ---
others = gdf_basin[~gdf_basin["_is_top"]].copy()
if len(others):
    # muted basins in very light gray to avoid competing with top colors
    others.plot(ax=ax1, color="#f2f2f2", edgecolor=BASIN_EDGE, linewidth=BASIN_EDGE_LW, alpha=MUTED_ALPHA, zorder=1)
# top basins colored by scarcity
tops = gdf_basin[gdf_basin["_is_top"]].copy()
if len(tops):
    tops.plot(ax=ax1, column=SCARCITY_COL, cmap=SCARCITY_CMAP, norm=scar_norm,
              edgecolor=TOP_EDGE_COLOR, linewidth=TOP_EDGE_LW, alpha=1.0, zorder=5)
# states borders
states48_l.boundary.plot(ax=ax1, color=STATE_BORDER, linewidth=STATE_LW, zorder=10)
ax1.set_title("b. Baseline water stress (Aqueduct) for top-30 basins", fontsize=12)
ax1.set_axis_off()

# colorbar for right
sm_scar = ScalarMappable(norm=scar_norm, cmap=SCARCITY_CMAP)
sm_scar.set_array([])
cax1 = inset_axes(ax1, width="3.0%", height="55%", loc="lower left",
                  bbox_to_anchor=(1.02, 0.18, 1, 1), bbox_transform=ax1.transAxes, borderpad=0)
cb1 = fig.colorbar(sm_scar, cax=cax1)
cb1.set_label("Aqueduct baseline stress (0–5)", fontsize=9)
cb1.ax.tick_params(labelsize=8)

# shared extent & aspect
minx, miny, maxx, maxy = states48_l.total_bounds
pad_x = 0.02 * (maxx - minx)
pad_y = 0.02 * (maxy - miny)
for ax in (ax0, ax1):
    ax.set_xlim(minx - pad_x, maxx + pad_x)
    ax.set_ylim(miny - pad_y, maxy + pad_y)
    ax.set_aspect("equal")

# annotate top N count
fig.suptitle(f"Scope 1 basins — focus on top {TOP_N} by capacity (red border)", fontsize=14, y=0.98)

# save & show
plt.savefig(OUT_PNG, dpi=DPI, bbox_inches="tight")
plt.savefig(OUT_PDF, dpi=DPI, bbox_inches="tight")
plt.show()

# --------------------------------
# Print a compact table of the top basins for reference
# --------------------------------
tbl = gdf_basin[gdf_basin["_is_top"]][[BASIN_KEY, "mw_sum", "mw_per_1000_km2", SCARCITY_COL]].copy()
tbl = tbl.sort_values("mw_sum", ascending=False)
tbl["mw_rank"] = range(1, len(tbl)+1)
print("\nTop basins (by total MW):")
display(tbl.reset_index(drop=True))


In [ ]:
# === 3-panel focus maps: ALL basins colored; top-15 crisp w/ red border ===
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, LogNorm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from pyproj import CRS

# -----------------------------
# CONFIG
# -----------------------------
TOP_N = 15

MW_COL = "current_mw"
SITE_BASIN_KEY = "pfaf_id"     # in CFG['gdf_sites']
BASIN_KEY = "PFAF_ID"          # in CFG['gdf_basin']
SCARCITY_COL = "bws_01_score"  # in basin polygons (0-5)

FIGSIZE = (20.5, 7.4)  # wide, Nature-friendly
DPI = 300

# Lambert Conformal Conic for CONUS
LCC_CONUS = CRS.from_proj4(
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 +datum=WGS84 +units=m +no_defs"
)

# Colormaps (feel free to swap)
CMAP_MW_SUM = "viridis"
CMAP_MW_DENS = "cividis"
CMAP_SCARC = "plasma"

# Visual emphasis
ALPHA_OTHER = 0.25      # muted basins
ALPHA_TOP   = 0.95      # crisp basins
EDGE_OTHER  = "#f2f2f2" # very light edges so polygons still readable
LW_OTHER    = 0.05
EDGE_TOP    = "#cc0000" # RED outline
LW_TOP      = 1.6

STATE_BORDER = "#111111"
STATE_LW = 0.75

# Outlier control (winsorize at p98; use log scaling)
PCT_CAP = 0.98
MIN_POS = 1e-3

OUT_PNG = "S1_Basins_top15_sum_density_scarcity.png"
OUT_PDF = "S1_Basins_top15_sum_density_scarcity.pdf"

# -----------------------------
# Helpers
# -----------------------------
def to_lcc(gdf, crs=LCC_CONUS):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(crs)

def _log_norm_with_cap(vals, pct=PCT_CAP):
    """LogNorm using vmin=min positive, vmax=quantile cap."""
    v = pd.Series(vals).replace([np.inf, -np.inf], np.nan)
    v = v[v > 0].dropna()
    if len(v) == 0:
        return Normalize(vmin=0, vmax=1), 1.0
    vmax = float(v.quantile(pct))
    vmin = max(float(v.min()), MIN_POS)
    if vmax <= vmin:
        vmax = vmin * 10
    return LogNorm(vmin=vmin, vmax=vmax), vmax

# -----------------------------
# Inputs
# -----------------------------
assert "gdf_sites" in CFG and CFG["gdf_sites"] is not None, "CFG['gdf_sites'] missing"
assert "gdf_basin" in CFG and CFG["gdf_basin"] is not None, "CFG['gdf_basin'] missing"
assert "states48" in globals(), "states48 missing"

gdf_sites = CFG["gdf_sites"].copy()
gdf_basin = CFG["gdf_basin"].copy()
states48_g = states48.copy()

# Project
gdf_sites = to_lcc(gdf_sites)
gdf_basin = to_lcc(gdf_basin)
states48_l = to_lcc(states48_g)

# Clip to CONUS using states48 geometry
conus_geom = states48_l.unary_union
gdf_basin = gdf_basin[gdf_basin.intersects(conus_geom)].copy()
gdf_sites = gdf_sites[gdf_sites.intersects(conus_geom)].copy()

# Key cleanup
for col, name in [(SITE_BASIN_KEY, "SITE_BASIN_KEY"), (MW_COL, "MW_COL")]:
    if col not in gdf_sites.columns:
        raise KeyError(f"{name} '{col}' not found in gdf_sites columns")
if BASIN_KEY not in gdf_basin.columns:
    raise KeyError(f"BASIN_KEY '{BASIN_KEY}' not found in gdf_basin columns")
if SCARCITY_COL not in gdf_basin.columns:
    raise KeyError(f"SCARCITY_COL '{SCARCITY_COL}' not found in gdf_basin columns")

gdf_sites[SITE_BASIN_KEY] = gdf_sites[SITE_BASIN_KEY].astype(str)
gdf_basin[BASIN_KEY] = gdf_basin[BASIN_KEY].astype(str)

# -----------------------------
# Aggregate MW by basin
# -----------------------------
bas_mw = (
    gdf_sites.groupby(SITE_BASIN_KEY, dropna=False)[MW_COL]
    .sum()
    .reset_index()
    .rename(columns={MW_COL: "mw_sum"})
)
gdf_basin = gdf_basin.merge(bas_mw, left_on=BASIN_KEY, right_on=SITE_BASIN_KEY, how="left")
gdf_basin["mw_sum"] = gdf_basin["mw_sum"].fillna(0.0)

# Area & density
gdf_basin["area_km2"] = gdf_basin.geometry.area / 1e6
gdf_basin["mw_per_1000_km2"] = np.where(
    gdf_basin["area_km2"] > 0,
    gdf_basin["mw_sum"] / (gdf_basin["area_km2"] / 1000.0),
    0.0
)

# -----------------------------
# Top-15 basins by MW SUM
# -----------------------------
top_ids = set(
    gdf_basin.sort_values("mw_sum", ascending=False)
    .head(TOP_N)[BASIN_KEY]
    .tolist()
)
gdf_basin["_is_top"] = gdf_basin[BASIN_KEY].isin(top_ids)

# -----------------------------
# Normalizations (log + p98 cap)
# -----------------------------
norm_mw_sum, vmax_sum = _log_norm_with_cap(gdf_basin["mw_sum"])
norm_mw_den, vmax_den = _log_norm_with_cap(gdf_basin["mw_per_1000_km2"])
norm_scar = Normalize(vmin=0.0, vmax=5.0)

# -----------------------------
# Plot: 3 panels
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=FIGSIZE, constrained_layout=True)
ax_sum, ax_den, ax_scar = axes

# Shared bounds
minx, miny, maxx, maxy = states48_l.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

def draw_panel(ax, col, cmap, norm, title, cbar_label, cbar_note=None):
    # 1) Draw ALL basins colored (muted alpha)
    gdf_basin.plot(
        ax=ax, column=col, cmap=cmap, norm=norm,
        alpha=ALPHA_OTHER, edgecolor=EDGE_OTHER, linewidth=LW_OTHER, zorder=1
    )
    # 2) Draw TOP basins again (crisp) + red border
    gdf_basin[gdf_basin["_is_top"]].plot(
        ax=ax, column=col, cmap=cmap, norm=norm,
        alpha=ALPHA_TOP, edgecolor=EDGE_TOP, linewidth=LW_TOP, zorder=5
    )
    # 3) State borders on top
    states48_l.boundary.plot(ax=ax, color=STATE_BORDER, linewidth=STATE_LW, zorder=10)

    ax.set_title(title, fontsize=12)
    ax.set_axis_off()
    ax.set_xlim(minx - padx, maxx + padx)
    ax.set_ylim(miny - pady, maxy + pady)
    ax.set_aspect("equal")

    # Colorbar
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cax = inset_axes(ax, width="3.2%", height="58%", loc="lower left",
                     bbox_to_anchor=(1.02, 0.18, 1, 1), bbox_transform=ax.transAxes, borderpad=0)
    cb = fig.colorbar(sm, cax=cax)
    cb.set_label(cbar_label, fontsize=9)
    cb.ax.tick_params(labelsize=8)
    if cbar_note:
        cb.ax.set_title(cbar_note, fontsize=8, pad=6)

# Panel A: MW sum
draw_panel(
    ax_sum,
    col="mw_sum",
    cmap=CMAP_MW_SUM,
    norm=norm_mw_sum,
    title="a. Total hyperscale capacity by basin (top-15 outlined)",
    cbar_label="MW (log scale)",
    cbar_note=f"cap at p{int(PCT_CAP*100)}"
)

# Panel B: MW density
draw_panel(
    ax_den,
    col="mw_per_1000_km2",
    cmap=CMAP_MW_DENS,
    norm=norm_mw_den,
    title="b. Hyperscale capacity density by basin (top-15 outlined)",
    cbar_label="MW per 1,000 km² (log scale)",
    cbar_note=f"cap at p{int(PCT_CAP*100)}"
)

# Panel C: Scarcity
draw_panel(
    ax_scar,
    col=SCARCITY_COL,
    cmap=CMAP_SCARC,
    norm=norm_scar,
    title="c. Baseline water stress by basin (Aqueduct, 0–5)",
    cbar_label="Aqueduct baseline stress (0–5)"
)

fig.suptitle(f"Scope 1 basins — all basins colored; top {TOP_N} emphasized (red border)", fontsize=14, y=0.99)

plt.savefig(OUT_PNG, dpi=DPI, bbox_inches="tight")
plt.savefig(OUT_PDF, dpi=DPI, bbox_inches="tight")
plt.show()

# -----------------------------
# Optional: quick rank table (top-15)
# -----------------------------
tbl = (
    gdf_basin[gdf_basin["_is_top"]][[BASIN_KEY, "mw_sum", "mw_per_1000_km2", SCARCITY_COL]]
    .sort_values("mw_sum", ascending=False)
    .reset_index(drop=True)
)
tbl["rank_mw_sum"] = np.arange(1, len(tbl) + 1)
display(tbl)
print(f"Saved: {OUT_PNG} and {OUT_PDF}")


In [ ]:
# === 3-panel focus maps: ALL basins colored; top-15 crisp w/ red border ===
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, LogNorm
from matplotlib.gridspec import GridSpec
from pyproj import CRS

# -----------------------------
# CONFIG
# -----------------------------
TOP_N = 15

MW_COL = "current_mw"
SITE_BASIN_KEY = "pfaf_id"     # in CFG['gdf_sites']
BASIN_KEY = "PFAF_ID"          # in CFG['gdf_basin']
SCARCITY_COL = "bws_01_score"  # in basin polygons (0-5)

FIGSIZE = (20.5, 7.4)
DPI = 300

# Lambert Conformal Conic for CONUS
LCC_CONUS = CRS.from_proj4(
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 +datum=WGS84 +units=m +no_defs"
)

CMAP_MW_SUM = "viridis"
CMAP_MW_DENS = "cividis"
CMAP_SCARC   = "plasma"

# Visual emphasis
ALPHA_OTHER = 0.28      # muted basins
ALPHA_TOP   = 0.95      # crisp basins
EDGE_OTHER  = "#cfcfcf" # light-gray basin borders (requested)
LW_OTHER    = 0.18

EDGE_TOP    = "#cc0000" # RED outline
LW_TOP      = 1.8

STATE_BORDER = "#111111"
STATE_LW = 0.8

# Outlier control (winsorize at p98; use log scaling)
PCT_CAP = 0.98
MIN_POS = 1e-3

OUT_PNG = "S1_Basins_top15_sum_density_scarcity.png"
OUT_PDF = "S1_Basins_top15_sum_density_scarcity.pdf"

# -----------------------------
# Helpers
# -----------------------------
def to_lcc(gdf, crs=LCC_CONUS):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(crs)

def _log_norm_with_cap(vals, pct=PCT_CAP):
    """LogNorm using vmin=min positive, vmax=quantile cap."""
    v = pd.Series(vals).replace([np.inf, -np.inf], np.nan)
    v = v[v > 0].dropna()
    if len(v) == 0:
        return Normalize(vmin=0, vmax=1), 1.0
    vmax = float(v.quantile(pct))
    vmin = max(float(v.min()), MIN_POS)
    if vmax <= vmin:
        vmax = vmin * 10
    return LogNorm(vmin=vmin, vmax=vmax), vmax

def add_cb(ax_cbar, cmap, norm, label, ticks=None, ticklabels=None, note=None):
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cb = plt.colorbar(sm, cax=ax_cbar)
    cb.set_label(label, fontsize=9)
    cb.ax.tick_params(labelsize=8, length=3)
    if ticks is not None:
        cb.set_ticks(ticks)
    if ticklabels is not None:
        cb.set_ticklabels(ticklabels)
    if note:
        cb.ax.set_title(note, fontsize=8, pad=6)
    return cb

# -----------------------------
# Inputs
# -----------------------------
assert "gdf_sites" in CFG and CFG["gdf_sites"] is not None, "CFG['gdf_sites'] missing"
assert "gdf_basin" in CFG and CFG["gdf_basin"] is not None, "CFG['gdf_basin'] missing"
assert "states48" in globals(), "states48 missing"

gdf_sites = CFG["gdf_sites"].copy()
gdf_basin = CFG["gdf_basin"].copy()
states48_g = states48.copy()

# Project to LCC
gdf_sites  = to_lcc(gdf_sites)
gdf_basin  = to_lcc(gdf_basin)
states48_l = to_lcc(states48_g)

# Clip to CONUS using states48 geometry
conus_geom = states48_l.unary_union
gdf_basin = gdf_basin[gdf_basin.intersects(conus_geom)].copy()
gdf_sites = gdf_sites[gdf_sites.intersects(conus_geom)].copy()

# Key cleanup
for col, name in [(SITE_BASIN_KEY, "SITE_BASIN_KEY"), (MW_COL, "MW_COL")]:
    if col not in gdf_sites.columns:
        raise KeyError(f"{name} '{col}' not found in gdf_sites columns")
if BASIN_KEY not in gdf_basin.columns:
    raise KeyError(f"BASIN_KEY '{BASIN_KEY}' not found in gdf_basin columns")
if SCARCITY_COL not in gdf_basin.columns:
    raise KeyError(f"SCARCITY_COL '{SCARCITY_COL}' not found in gdf_basin columns")

gdf_sites[SITE_BASIN_KEY] = gdf_sites[SITE_BASIN_KEY].astype(str)
gdf_basin[BASIN_KEY]      = gdf_basin[BASIN_KEY].astype(str)

# -----------------------------
# Aggregate MW by basin
# -----------------------------
bas_mw = (
    gdf_sites.groupby(SITE_BASIN_KEY, dropna=False)[MW_COL]
    .sum()
    .reset_index()
    .rename(columns={MW_COL: "mw_sum"})
)

gdf_basin = gdf_basin.merge(bas_mw, left_on=BASIN_KEY, right_on=SITE_BASIN_KEY, how="left")
gdf_basin["mw_sum"] = gdf_basin["mw_sum"].fillna(0.0)

# Area & density
gdf_basin["area_km2"] = gdf_basin.geometry.area / 1e6
gdf_basin["mw_per_1000_km2"] = np.where(
    gdf_basin["area_km2"] > 0,
    gdf_basin["mw_sum"] / (gdf_basin["area_km2"] / 1000.0),
    0.0
)

# -----------------------------
# Top-15 basins by MW SUM
# -----------------------------
top_ids = set(
    gdf_basin.sort_values("mw_sum", ascending=False)
    .head(TOP_N)[BASIN_KEY].tolist()
)
gdf_basin["_is_top"] = gdf_basin[BASIN_KEY].isin(top_ids)

# -----------------------------
# Normalizations
# -----------------------------
norm_mw_sum, _ = _log_norm_with_cap(gdf_basin["mw_sum"])
norm_mw_den, _ = _log_norm_with_cap(gdf_basin["mw_per_1000_km2"])
norm_scar = Normalize(vmin=0.0, vmax=5.0)

# -----------------------------
# Figure layout: map + dedicated colorbar axis per panel (NO OVERLAP)
# -----------------------------
fig = plt.figure(figsize=FIGSIZE, dpi=DPI)

# 3 blocks, each block = (map axis, cbar axis)
gs_outer = GridSpec(
    1, 3, figure=fig,
    width_ratios=[1, 1, 1],
    wspace=0.22
)

def make_block(col_index):
    # inside each column: map takes most width, colorbar gets its own lane
    gs = gs_outer[0, col_index].subgridspec(1, 2, width_ratios=[1.0, 0.05], wspace=0.05)
    ax_map = fig.add_subplot(gs[0, 0])
    ax_cbar = fig.add_subplot(gs[0, 1])
    return ax_map, ax_cbar

ax_sum,  cax_sum  = make_block(0)
ax_den,  cax_den  = make_block(1)
ax_scar, cax_scar = make_block(2)

# Map bounds
minx, miny, maxx, maxy = states48_l.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

def draw_panel(ax, cax, col, cmap, norm, title, cbar_label, cbar_note=None,
               add_stress_thresholds=False):
    # ALL basins (muted but still colored with same cmap)
    gdf_basin.plot(
        ax=ax, column=col, cmap=cmap, norm=norm,
        alpha=ALPHA_OTHER, edgecolor=EDGE_OTHER, linewidth=LW_OTHER, zorder=1
    )
    # TOP basins (crisp) + red outline
    gdf_basin[gdf_basin["_is_top"]].plot(
        ax=ax, column=col, cmap=cmap, norm=norm,
        alpha=ALPHA_TOP, edgecolor=EDGE_TOP, linewidth=LW_TOP, zorder=5
    )

    # State borders
    states48_l.boundary.plot(ax=ax, color=STATE_BORDER, linewidth=STATE_LW, zorder=10)

    ax.set_title(title, fontsize=12)
    ax.set_axis_off()
    ax.set_xlim(minx - padx, maxx + padx)
    ax.set_ylim(miny - pady, maxy + pady)
    ax.set_aspect("equal")

    # Colorbar (in dedicated axis -> never overlaps neighbors)
    cb = add_cb(cax, cmap=cmap, norm=norm, label=cbar_label, note=cbar_note)

    if add_stress_thresholds:
        # Mark the policy-relevant stress thresholds on the colorbar
        for t, lab in [(3, "High (≥3)"), (4, "Extremely high (≥4)")]:
            cb.ax.axhline(t, linewidth=1.2)
        # Place one clear label at 3
        cb.ax.text(
            1.55, 3, "High stress ≥ 3",
            va="center", ha="left", fontsize=8,
            transform=cb.ax.get_yaxis_transform()
        )

    return cb

# Panel A: MW sum
draw_panel(
    ax_sum, cax_sum,
    col="mw_sum",
    cmap=CMAP_MW_SUM,
    norm=norm_mw_sum,
    title="a. Total hyperscale capacity by basin (top-15 outlined)",
    cbar_label="MW (log scale)",
    cbar_note=f"cap at p{int(PCT_CAP*100)}"
)

# Panel B: MW density
draw_panel(
    ax_den, cax_den,
    col="mw_per_1000_km2",
    cmap=CMAP_MW_DENS,
    norm=norm_mw_den,
    title="b. Hyperscale capacity density by basin (top-15 outlined)",
    cbar_label="MW per 1,000 km² (log scale)",
    cbar_note=f"cap at p{int(PCT_CAP*100)}"
)

# Panel C: Scarcity
draw_panel(
    ax_scar, cax_scar,
    col=SCARCITY_COL,
    cmap=CMAP_SCARC,
    norm=norm_scar,
    title="c. Baseline water stress by basin (Aqueduct, 0–5)",
    cbar_label="Aqueduct baseline stress (0–5)",
    add_stress_thresholds=True
)

fig.suptitle(f"Scope 1 basins — all basins colored; top {TOP_N} emphasized (red border)", fontsize=14, y=0.98)

plt.savefig(OUT_PNG, dpi=DPI, bbox_inches="tight")
plt.savefig(OUT_PDF, dpi=DPI, bbox_inches="tight")
plt.show()

# Optional top table
tbl = (
    gdf_basin[gdf_basin["_is_top"]][[BASIN_KEY, "mw_sum", "mw_per_1000_km2", SCARCITY_COL]]
    .sort_values("mw_sum", ascending=False)
    .reset_index(drop=True)
)
tbl["rank_mw_sum"] = np.arange(1, len(tbl) + 1)
display(tbl)

print(f"Saved: {OUT_PNG} and {OUT_PDF}")

In [ ]:
# === 3-panel BA maps: ALL BAs colored; top-15 crisp w/ red border ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LogNorm
from matplotlib.cm import ScalarMappable
from matplotlib.gridspec import GridSpec

TOP_N = 15
BA_KEY = "subregion_id"     # in both gdf_sites and gdf_ba
MW_COL = "current_mw"
SITE_SCARCITY_COL = "bws_01_score"   # on sites

FIGSIZE = (20.5, 7.4)
DPI = 300

CMAP_MW_SUM = "viridis"
CMAP_MW_DENS = "cividis"
CMAP_SCARC   = "plasma"

ALPHA_OTHER = 0.28
ALPHA_TOP   = 0.95
EDGE_OTHER  = "#cfcfcf"
LW_OTHER    = 0.18
EDGE_TOP    = "#cc0000"
LW_TOP      = 1.8

STATE_BORDER = "#111111"
STATE_LW = 0.8

PCT_CAP = 0.98
MIN_POS = 1e-3

OUT_PNG = "S2_BA_top15_sum_density_scarcity.png"
OUT_PDF = "S2_BA_top15_sum_density_scarcity.pdf"

def _log_norm_with_cap(vals, pct=PCT_CAP):
    v = pd.Series(vals).replace([np.inf, -np.inf], np.nan)
    v = v[v > 0].dropna()
    if len(v) == 0:
        return Normalize(vmin=0, vmax=1), 1.0
    vmax = float(v.quantile(pct))
    vmin = max(float(v.min()), MIN_POS)
    if vmax <= vmin:
        vmax = vmin * 10
    return LogNorm(vmin=vmin, vmax=vmax), vmax

def add_cb(fig, cax, cmap, norm, label, note=None):
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cb = fig.colorbar(sm, cax=cax)
    cb.set_label(label, fontsize=9)
    cb.ax.tick_params(labelsize=8, length=3)
    if note:
        cb.ax.set_title(note, fontsize=8, pad=6)
    return cb

# Inputs
gdf_sites = CFG["gdf_sites"].copy()
gdf_ba    = CFG["gdf_ba"].copy()
states48_g = states48.copy()

# Project to same LCC
gdf_sites  = to_lcc(gdf_sites)
gdf_ba     = to_lcc(gdf_ba)
states48_l = to_lcc(states48_g)

# Clip to CONUS
conus_geom = states48_l.unary_union
gdf_ba    = gdf_ba[gdf_ba.intersects(conus_geom)].copy()
gdf_sites = gdf_sites[gdf_sites.intersects(conus_geom)].copy()

# Key cleanup
for col in [BA_KEY, MW_COL, SITE_SCARCITY_COL]:
    if col not in gdf_sites.columns:
        raise KeyError(f"'{col}' missing in gdf_sites")
if BA_KEY not in gdf_ba.columns:
    raise KeyError(f"'{BA_KEY}' missing in gdf_ba")

gdf_sites[BA_KEY] = gdf_sites[BA_KEY].astype(str)
gdf_ba[BA_KEY]    = gdf_ba[BA_KEY].astype(str)

# Aggregate MW and scarcity exposure by BA
ba_stats = (
    gdf_sites.groupby(BA_KEY, dropna=False)
    .agg(
        mw_sum=(MW_COL, "sum"),
        # MW-weighted stress is usually better than plain mean; keep both if you want.
        stress_mean=(SITE_SCARCITY_COL, "mean"),
        stress_mw_wt=(SITE_SCARCITY_COL, lambda s: np.average(s, weights=gdf_sites.loc[s.index, MW_COL]))
    )
    .reset_index()
)

gdf_ba = gdf_ba.merge(ba_stats, on=BA_KEY, how="left")
gdf_ba["mw_sum"] = gdf_ba["mw_sum"].fillna(0.0)

# Density
gdf_ba["area_km2"] = gdf_ba.geometry.area / 1e6
gdf_ba["mw_per_1000_km2"] = np.where(
    gdf_ba["area_km2"] > 0,
    gdf_ba["mw_sum"] / (gdf_ba["area_km2"] / 1000.0),
    0.0
)

# Top-15 BAs by MW sum
top_ids = set(gdf_ba.sort_values("mw_sum", ascending=False).head(TOP_N)[BA_KEY].tolist())
gdf_ba["_is_top"] = gdf_ba[BA_KEY].isin(top_ids)

# Norms
norm_mw_sum, _ = _log_norm_with_cap(gdf_ba["mw_sum"])
norm_mw_den, _ = _log_norm_with_cap(gdf_ba["mw_per_1000_km2"])
norm_scar = Normalize(vmin=0.0, vmax=5.0)

# Layout (no overlap)
fig = plt.figure(figsize=FIGSIZE, dpi=DPI)
gs_outer = GridSpec(1, 3, figure=fig, width_ratios=[1, 1, 1], wspace=0.22)

def make_block(col_index):
    gs = gs_outer[0, col_index].subgridspec(1, 2, width_ratios=[1.0, 0.05], wspace=0.05)
    return fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])

ax_sum,  cax_sum  = make_block(0)
ax_den,  cax_den  = make_block(1)
ax_scar, cax_scar = make_block(2)

minx, miny, maxx, maxy = states48_l.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

def draw_panel(ax, cax, col, cmap, norm, title, cbar_label, cbar_note=None, add_stress_thresholds=False):
    gdf_ba.plot(ax=ax, column=col, cmap=cmap, norm=norm,
                alpha=ALPHA_OTHER, edgecolor=EDGE_OTHER, linewidth=LW_OTHER, zorder=1)
    gdf_ba[gdf_ba["_is_top"]].plot(ax=ax, column=col, cmap=cmap, norm=norm,
                                  alpha=ALPHA_TOP, edgecolor=EDGE_TOP, linewidth=LW_TOP, zorder=5)

    states48_l.boundary.plot(ax=ax, color=STATE_BORDER, linewidth=STATE_LW, zorder=10)

    ax.set_title(title, fontsize=12)
    ax.set_axis_off()
    ax.set_xlim(minx - padx, maxx + padx)
    ax.set_ylim(miny - pady, maxy + pady)
    ax.set_aspect("equal")

    cb = add_cb(fig, cax, cmap=cmap, norm=norm, label=cbar_label, note=cbar_note)

    if add_stress_thresholds:
        for t in [3, 4]:
            cb.ax.axhline(t, linewidth=1.2)
        cb.ax.text(
            1.55, 3, "High stress ≥ 3",
            va="center", ha="left", fontsize=8,
            transform=cb.ax.get_yaxis_transform()
        )

draw_panel(
    ax_sum, cax_sum,
    col="mw_sum", cmap=CMAP_MW_SUM, norm=norm_mw_sum,
    title="a. Total hyperscale capacity by BA (top-15 outlined)",
    cbar_label="MW (log scale)", cbar_note=f"cap at p{int(PCT_CAP*100)}"
)

draw_panel(
    ax_den, cax_den,
    col="mw_per_1000_km2", cmap=CMAP_MW_DENS, norm=norm_mw_den,
    title="b. Hyperscale capacity density by BA (top-15 outlined)",
    cbar_label="MW per 1,000 km² (log scale)", cbar_note=f"cap at p{int(PCT_CAP*100)}"
)

# choose stress_mean or stress_mw_wt (I recommend MW-weighted for “exposure where load is”)
draw_panel(
    ax_scar, cax_scar,
    col="stress_mw_wt", cmap=CMAP_SCARC, norm=norm_scar,
    title="c. Baseline water stress exposure by BA (Aqueduct, 0–5)",
    cbar_label="Aqueduct baseline stress (0–5)",
    add_stress_thresholds=True
)

fig.suptitle(f"BAs — all regions colored; top {TOP_N} emphasized (red border)", fontsize=14, y=0.98)
plt.savefig(OUT_PNG, dpi=DPI, bbox_inches="tight")
plt.savefig(OUT_PDF, dpi=DPI, bbox_inches="tight")
plt.show()

print(f"Saved: {OUT_PNG} and {OUT_PDF}")

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, LogNorm
from pyproj import CRS

# -----------------------------
# Shared helpers
# -----------------------------
def to_lcc(gdf, crs):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(crs)

def log_norm_with_cap(vals, pct=0.98, min_pos=1e-3):
    v = pd.Series(vals).replace([np.inf, -np.inf], np.nan)
    v = v[v > 0].dropna()
    if len(v) == 0:
        return Normalize(vmin=0, vmax=1), 1.0
    vmax = float(v.quantile(pct))
    vmin = max(float(v.min()), min_pos)
    if vmax <= vmin:
        vmax = vmin * 10
    return LogNorm(vmin=vmin, vmax=vmax), vmax

def add_hcolorbar(fig, ax, sm, label, note=None, pad=0.08, fraction=0.05, shrink=0.92):
    """
    Horizontal colorbar beneath the axis (prevents overlap and keeps maps large).
    pad is relative to axis height; increase if labels collide.
    """
    cb = fig.colorbar(sm, ax=ax, orientation="horizontal",
                      pad=pad, fraction=fraction, shrink=shrink)
    cb.set_label(label, fontsize=9)
    cb.ax.tick_params(labelsize=8)
    if note:
        cb.ax.set_title(note, fontsize=8, pad=6)
    return cb

def draw_focus_panel(
    fig, ax,
    gdf_all, gdf_top,
    col, cmap, norm,
    states48_l,
    title,
    cbar_label,
    cbar_note=None,
    # styling
    alpha_other=0.22,
    alpha_top=0.95,
    edge_other="#cfcfcf",   # light gray basin/BA borders
    lw_other=0.18,
    edge_top="#cc0000",
    lw_top=1.7,
    state_border="#111111",
    state_lw=0.8,
    bounds=None
):
    # All polygons (muted)
    gdf_all.plot(
        ax=ax, column=col, cmap=cmap, norm=norm,
        alpha=alpha_other, edgecolor=edge_other, linewidth=lw_other, zorder=1
    )
    # Top polygons (crisp + red border)
    gdf_top.plot(
        ax=ax, column=col, cmap=cmap, norm=norm,
        alpha=alpha_top, edgecolor=edge_top, linewidth=lw_top, zorder=5
    )
    # State borders
    states48_l.boundary.plot(ax=ax, color=state_border, linewidth=state_lw, zorder=10, alpha =0.5)

    ax.set_title(title, fontsize=12)
    ax.set_axis_off()
    ax.set_aspect("equal")

    if bounds is not None:
        minx, miny, maxx, maxy = bounds
        padx = 0.02 * (maxx - minx)
        pady = 0.02 * (maxy - miny)
        ax.set_xlim(minx - padx, maxx + padx)
        ax.set_ylim(miny - pady, maxy + pady)

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    add_hcolorbar(fig, ax, sm, cbar_label, note=cbar_note, pad=0.06, fraction=0.05, shrink=0.92)
    return sm

In [ ]:
# =============================
# BASINS 3-PANEL (Scope 1)
# =============================
TOP_N = 15
PCT_CAP = 0.98
MIN_POS = 1e-3

MW_COL = "current_mw"
SITE_BASIN_KEY = "pfaf_id"
BASIN_KEY = "PFAF_ID"
SCARCITY_COL = "bws_01_score"   # continuous 0–5

CMAP_MW_SUM = "viridis"
CMAP_MW_DENS = "cividis"
CMAP_SCARC  = "plasma"

LCC_CONUS = CRS.from_proj4(
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 +datum=WGS84 +units=m +no_defs"
)

# Inputs
gdf_sites = CFG["gdf_sites"].copy()
gdf_basin = CFG["gdf_basin"].copy()
states48_g = states48.copy()

# Project
gdf_sites  = to_lcc(gdf_sites, LCC_CONUS)
gdf_basin  = to_lcc(gdf_basin, LCC_CONUS)
states48_l = to_lcc(states48_g, LCC_CONUS)

# Clip to CONUS
conus_geom = states48_l.unary_union
gdf_basin = gdf_basin[gdf_basin.intersects(conus_geom)].copy()
gdf_sites = gdf_sites[gdf_sites.intersects(conus_geom)].copy()

# Keys
gdf_sites[SITE_BASIN_KEY] = gdf_sites[SITE_BASIN_KEY].astype(str)
gdf_basin[BASIN_KEY]      = gdf_basin[BASIN_KEY].astype(str)

# Aggregate MW by basin
bas_mw = (
    gdf_sites.groupby(SITE_BASIN_KEY, dropna=False)[MW_COL]
    .sum().reset_index()
    .rename(columns={MW_COL: "mw_sum"})
)
gdf_basin = gdf_basin.merge(bas_mw, left_on=BASIN_KEY, right_on=SITE_BASIN_KEY, how="left")
gdf_basin["mw_sum"] = gdf_basin["mw_sum"].fillna(0.0)

# Area + density
gdf_basin["area_km2"] = gdf_basin.geometry.area / 1e6
gdf_basin["mw_per_1000_km2"] = np.where(
    gdf_basin["area_km2"] > 0,
    gdf_basin["mw_sum"] / (gdf_basin["area_km2"] / 1000.0),
    0.0
)

# Top-N by MW sum
top_ids = set(gdf_basin.sort_values("mw_sum", ascending=False).head(TOP_N)[BASIN_KEY])
gdf_basin["_is_top"] = gdf_basin[BASIN_KEY].isin(top_ids)

# Norms
norm_sum, _ = log_norm_with_cap(gdf_basin["mw_sum"], pct=PCT_CAP, min_pos=MIN_POS)
norm_den, _ = log_norm_with_cap(gdf_basin["mw_per_1000_km2"], pct=PCT_CAP, min_pos=MIN_POS)
norm_scar   = Normalize(vmin=0.0, vmax=5.0)

# Bounds for consistent framing
bounds = states48_l.total_bounds

# --- Layout: make maps bigger, minimize whitespace
fig, axes = plt.subplots(
    1, 3,
    figsize=(22, 7.2),
    gridspec_kw=dict(wspace=0.06)
)
ax_sum, ax_den, ax_scar = axes

g_all = gdf_basin
g_top = gdf_basin[gdf_basin["_is_top"]].copy()

draw_focus_panel(
    fig, ax_sum,
    g_all, g_top,
    col="mw_sum", cmap=CMAP_MW_SUM, norm=norm_sum,
    states48_l=states48_l,
    title="a. Total hyperscale capacity by basin (top-15 outlined)",
    cbar_label="MW (log scale)",
    cbar_note=f"winsorized at p{int(PCT_CAP*100)}",
    edge_other="#d0d0d0", lw_other=0.18,   # basin borders visible
    bounds=bounds
)

draw_focus_panel(
    fig, ax_den,
    g_all, g_top,
    col="mw_per_1000_km2", cmap=CMAP_MW_DENS, norm=norm_den,
    states48_l=states48_l,
    title="b. Hyperscale capacity density by basin (top-15 outlined)",
    cbar_label="MW per 1,000 km² (log scale)",
    cbar_note=f"winsorized at p{int(PCT_CAP*100)}",
    edge_other="#d0d0d0", lw_other=0.18,
    bounds=bounds
)

draw_focus_panel(
    fig, ax_scar,
    g_all, g_top,
    col=SCARCITY_COL, cmap=CMAP_SCARC, norm=norm_scar,
    states48_l=states48_l,
    title="c. Baseline water stress by basin (Aqueduct, 0–5)",
    cbar_label="Aqueduct baseline stress (0–5)",
    edge_other="#e0e0e0", lw_other=0.15,
    bounds=bounds
)

# Make “high stress >= 3” explicit (no colormap change)
ax_scar.text(
    0.02, 0.02, "High stress threshold: ≥ 3",
    transform=ax_scar.transAxes,
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.6", alpha=0.9)
)

#fig.suptitle(f"Scope 1 basins — all basins colored; top {TOP_N} emphasized (red border)", fontsize=14, y=0.98)
fig.subplots_adjust(left=0.01, right=0.995, top=0.88, bottom=0.10)

OUT_PNG = "S1_Basins_top15_sum_density_scarcity_v2.png"
OUT_PDF = "S1_Basins_top15_sum_density_scarcity_v2.pdf"
plt.savefig(OUT_PNG, dpi=300, bbox_inches="tight")
plt.savefig(OUT_PDF, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", OUT_PNG, "and", OUT_PDF)

In [ ]:
# =============================
# BA 3-PANEL (Scope 2 context / exposure)
# =============================
TOP_N = 15
PCT_CAP = 0.98
MIN_POS = 1e-3

BA_KEY = "subregion_id"       # in CFG['gdf_ba'] and CFG['gdf_sites']
MW_COL = "current_mw"
SITE_STRESS_COL = "bws_01_score"  # on sites; use to compute BA exposure

CMAP_MW_SUM = "viridis"
CMAP_MW_DENS = "cividis"
CMAP_SCARC  = "plasma"

# Inputs
gdf_sites = CFG["gdf_sites"].copy()
gdf_ba    = CFG["gdf_ba"].copy()
states48_g = states48.copy()

# Project
gdf_sites  = to_lcc(gdf_sites, LCC_CONUS)
gdf_ba     = to_lcc(gdf_ba,    LCC_CONUS)
states48_l = to_lcc(states48_g, LCC_CONUS)

# Clip to CONUS using states48
conus_geom = states48_l.unary_union
gdf_ba    = gdf_ba[gdf_ba.intersects(conus_geom)].copy()
gdf_sites = gdf_sites[gdf_sites.intersects(conus_geom)].copy()

# Keys
gdf_sites[BA_KEY] = gdf_sites[BA_KEY].astype(str)
gdf_ba[BA_KEY]    = gdf_ba[BA_KEY].astype(str)

# --- Aggregate MW by BA
ba_mw = (
    gdf_sites.groupby(BA_KEY, dropna=False)[MW_COL]
    .sum().reset_index()
    .rename(columns={MW_COL: "mw_sum"})
)

# --- Capacity-weighted mean stress by BA (exposure proxy)
tmp = gdf_sites[[BA_KEY, MW_COL, SITE_STRESS_COL]].copy()
tmp["_w"] = tmp[MW_COL].clip(lower=0).fillna(0.0)
tmp["_xw"] = tmp[SITE_STRESS_COL].fillna(0.0) * tmp["_w"]

ba_stress = tmp.groupby(BA_KEY, dropna=False).agg(
    mw_sum_check=("_w","sum"),
    stress_xw=("_xw","sum"),
).reset_index()
ba_stress["stress_wmean"] = np.where(
    ba_stress["mw_sum_check"] > 0,
    ba_stress["stress_xw"] / ba_stress["mw_sum_check"],
    np.nan
)

# Merge BA stats onto BA polygons
gdf_ba = gdf_ba.merge(ba_mw, on=BA_KEY, how="left")
gdf_ba = gdf_ba.merge(ba_stress[[BA_KEY, "stress_wmean"]], on=BA_KEY, how="left")

gdf_ba["mw_sum"] = gdf_ba["mw_sum"].fillna(0.0)
gdf_ba["area_km2"] = gdf_ba.geometry.area / 1e6
gdf_ba["mw_per_1000_km2"] = np.where(
    gdf_ba["area_km2"] > 0,
    gdf_ba["mw_sum"] / (gdf_ba["area_km2"] / 1000.0),
    0.0
)

# Top-N BAs by MW sum
top_ids = set(gdf_ba.sort_values("mw_sum", ascending=False).head(TOP_N)[BA_KEY])
gdf_ba["_is_top"] = gdf_ba[BA_KEY].isin(top_ids)

# Norms
norm_sum, _ = log_norm_with_cap(gdf_ba["mw_sum"], pct=PCT_CAP, min_pos=MIN_POS)
norm_den, _ = log_norm_with_cap(gdf_ba["mw_per_1000_km2"], pct=PCT_CAP, min_pos=MIN_POS)
norm_scar   = Normalize(vmin=0.0, vmax=5.0)

bounds = states48_l.total_bounds

# Plot
fig, axes = plt.subplots(1, 3, figsize=(22, 7.2), gridspec_kw=dict(wspace=0.06))
ax_sum, ax_den, ax_scar = axes

g_all = gdf_ba
g_top = gdf_ba[gdf_ba["_is_top"]].copy()

draw_focus_panel(
    fig, ax_sum,
    g_all, g_top,
    col="mw_sum", cmap=CMAP_MW_SUM, norm=norm_sum,
    states48_l=states48_l,
    title="a. Total hyperscale capacity by BA (top-15 outlined)",
    cbar_label="MW (log scale)",
    cbar_note=f"winsorized at p{int(PCT_CAP*100)}",
    edge_other="#d0d0d0", lw_other=0.30,   # BA borders a bit thicker than basins
    bounds=bounds
)

draw_focus_panel(
    fig, ax_den,
    g_all, g_top,
    col="mw_per_1000_km2", cmap=CMAP_MW_DENS, norm=norm_den,
    states48_l=states48_l,
    title="b. Hyperscale capacity density by BA (top-15 outlined)",
    cbar_label="MW per 1,000 km² (log scale)",
    cbar_note=f"winsorized at p{int(PCT_CAP*100)}",
    edge_other="#d0d0d0", lw_other=0.30,
    bounds=bounds
)

draw_focus_panel(
    fig, ax_scar,
    g_all, g_top,
    col="stress_wmean", cmap=CMAP_SCARC, norm=norm_scar,
    states48_l=states48_l,
    title="c. Baseline water stress exposure by BA (Aqueduct, 0–5)",
    cbar_label="Capacity-weighted Aqueduct stress (0–5)",
    edge_other="#d0d0d0", lw_other=0.30,
    bounds=bounds
)

ax_scar.text(
    0.02, 0.02, "High stress threshold: ≥ 3",
    transform=ax_scar.transAxes,
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.6", alpha=0.9)
)

#fig.suptitle(f"BAs — all regions colored; top {TOP_N} emphasized (red border)", fontsize=14, y=0.98)
fig.subplots_adjust(left=0.01, right=0.995, top=0.88, bottom=0.10)

OUT_PNG = "S2_BA_top15_sum_density_scarcity_v2.png"
OUT_PDF = "S2_BA_top15_sum_density_scarcity_v2.pdf"
plt.savefig(OUT_PNG, dpi=300, bbox_inches="tight")
plt.savefig(OUT_PDF, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", OUT_PNG, "and", OUT_PDF)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, LogNorm
from pyproj import CRS

# -----------------------------
# CONFIG
# -----------------------------
TOP_N = 15
PCT_CAP = .98

MW_COL = "current_mw"
SITE_BASIN_KEY = "pfaf_id"
BASIN_KEY = "PFAF_ID"

S1_COL = "s1_m3_mid"          # site-level Scope 1 water (m3/yr), baseline/mid
SCARCITY_COL = "bws_01_score" # basin-level Aqueduct baseline stress (0–5)

CMAP_MW_SUM = "viridis"
CMAP_MW_DENS = "cividis"
CMAP_SCARC  = "plasma"

# Toggle: set SCALE_MODE = "log" or "linear"
SCALE_MODE = "linear"   # <-- change to "linear" to remove log scales

# Outline criterion for top polygons
TOP_BY = "s1_m3"      # "s1_m3" (recommended) or "mw_sum"

OUT_PREFIX = f"S1_Basins_top{TOP_N}_by_{TOP_BY}_{SCALE_MODE}"

LCC_CONUS = CRS.from_proj4(
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 +datum=WGS84 +units=m +no_defs"
)

# -----------------------------
# Helpers
# -----------------------------
def to_lcc(gdf, crs=LCC_CONUS):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(crs)

def norm_with_cap(vals, pct=PCT_CAP, mode="log", min_pos=1e-3):
    v = pd.Series(vals).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    vmax = float(v.quantile(pct))
    if mode == "log":
        vpos = v[v > 0]
        vmin = max(float(vpos.min()) if len(vpos) else min_pos, min_pos)
        if vmax <= vmin:
            vmax = vmin * 10
        return LogNorm(vmin=vmin, vmax=max(vmax, vmin*1.01)), vmax
    else:
        return Normalize(vmin=0.0, vmax=max(vmax, 1e-12)), vmax

def add_hcolorbar(fig, ax, sm, label, note=None, pad=0.06, fraction=0.05, shrink=0.92):
    cb = fig.colorbar(sm, ax=ax, orientation="horizontal",
                      pad=pad, fraction=fraction, shrink=shrink)
    cb.set_label(label, fontsize=9)
    cb.ax.tick_params(labelsize=8)
    if note:
        cb.ax.set_title(note, fontsize=8, pad=6)
    return cb

def draw_panel(fig, ax, g_all, g_top, col, cmap, norm, states48_l, bounds,
               title, cbar_label, cbar_note=None,
               alpha_other=0.5, alpha_top=0.95,
               edge_other="#d0d0d0", lw_other=0.18,
               edge_top="#cc0000", lw_top=1.7,
               state_border="#111111", state_lw=0.8):
    g_all.plot(ax=ax, column=col, cmap=cmap, norm=norm,
               alpha=alpha_other, edgecolor=edge_other, linewidth=lw_other, zorder=1)
    g_top.plot(ax=ax, column=col, cmap=cmap, norm=norm,
               alpha=alpha_top, edgecolor=edge_top, linewidth=lw_top, zorder=5)
    states48_l.boundary.plot(ax=ax, color=state_border, linewidth=state_lw, zorder=10)

    ax.set_title(title, fontsize=12)
    ax.set_axis_off()
    ax.set_aspect("equal")

    minx, miny, maxx, maxy = bounds
    padx, pady = 0.02*(maxx-minx), 0.02*(maxy-miny)
    ax.set_xlim(minx-padx, maxx+padx)
    ax.set_ylim(miny-pady, maxy+pady)

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    add_hcolorbar(fig, ax, sm, cbar_label, note=cbar_note)
    return sm

# -----------------------------
# Inputs
# -----------------------------
gdf_sites  = CFG["gdf_sites"].copy()
gdf_basin  = CFG["gdf_basin"].copy()
states48_g = states48.copy()

# Project
gdf_sites  = to_lcc(gdf_sites)
gdf_basin  = to_lcc(gdf_basin)
states48_l = to_lcc(states48_g)

# Clip to CONUS
conus_geom = states48_l.unary_union
gdf_basin = gdf_basin[gdf_basin.intersects(conus_geom)].copy()
gdf_sites = gdf_sites[gdf_sites.intersects(conus_geom)].copy()

# Keys
gdf_sites[SITE_BASIN_KEY] = gdf_sites[SITE_BASIN_KEY].astype(str)
gdf_basin[BASIN_KEY]      = gdf_basin[BASIN_KEY].astype(str)

# -----------------------------
# Aggregate MW + Scope1 by basin
# -----------------------------
bas_agg = (
    gdf_sites.groupby(SITE_BASIN_KEY, dropna=False)
    .agg(mw_sum=(MW_COL, "sum"), s1_m3=(S1_COL, "sum"))
    .reset_index()
)

gdf_basin = gdf_basin.merge(bas_agg, left_on=BASIN_KEY, right_on=SITE_BASIN_KEY, how="left")
gdf_basin["mw_sum"] = gdf_basin["mw_sum"].fillna(0.0)
gdf_basin["s1_m3"]  = gdf_basin["s1_m3"].fillna(0.0)

# Density
gdf_basin["area_km2"] = gdf_basin.geometry.area / 1e6
gdf_basin["mw_per_1000_km2"] = np.where(
    gdf_basin["area_km2"] > 0,
    gdf_basin["mw_sum"] / (gdf_basin["area_km2"]/1000.0),
    0.0
)

# Choose top-15 outline criterion
rank_col = "s1_m3" if TOP_BY == "s1_m3" else "mw_sum"
top_ids = set(gdf_basin.sort_values(rank_col, ascending=False).head(TOP_N)[BASIN_KEY])
gdf_basin["_is_top"] = gdf_basin[BASIN_KEY].isin(top_ids)

# Norms (log or linear)
norm_sum, _ = norm_with_cap(gdf_basin["mw_sum"], pct=PCT_CAP, mode=SCALE_MODE)
norm_den, _ = norm_with_cap(gdf_basin["mw_per_1000_km2"], pct=PCT_CAP, mode=SCALE_MODE)
norm_scar   = Normalize(vmin=0.0, vmax=5.0)

bounds = states48_l.total_bounds

# -----------------------------
# Plot
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(22, 7.2), gridspec_kw=dict(wspace=0.06))
ax_sum, ax_den, ax_scar = axes

g_all = gdf_basin
g_top = gdf_basin[gdf_basin["_is_top"]].copy()

scale_label = "log scale" if SCALE_MODE == "log" else "linear scale"
cap_note = f"winsorized at p{int(PCT_CAP*100)}"

draw_panel(
    fig, ax_sum, g_all, g_top,
    col="mw_sum", cmap=CMAP_MW_SUM, norm=norm_sum,
    states48_l=states48_l, bounds=bounds,
    title=f"a. Total hyperscale capacity by basin (top-{TOP_N} by {TOP_BY} outlined)",
    cbar_label=f"MW ({scale_label})",
    cbar_note=cap_note
)

draw_panel(
    fig, ax_den, g_all, g_top,
    col="mw_per_1000_km2", cmap=CMAP_MW_DENS, norm=norm_den,
    states48_l=states48_l, bounds=bounds,
    title=f"b. Hyperscale capacity density by basin (top-{TOP_N} by {TOP_BY} outlined)",
    cbar_label=f"MW per 1,000 km² ({scale_label})",
    cbar_note=cap_note
)

draw_panel(
    fig, ax_scar, g_all, g_top,
    col=SCARCITY_COL, cmap=CMAP_SCARC, norm=norm_scar,
    states48_l=states48_l, bounds=bounds,
    title="c. Baseline water stress by basin (Aqueduct, 0–5)",
    cbar_label="Aqueduct baseline stress (0–5)"
)

# Make "high stress ≥ 3" unambiguous (no colormap change)
ax_scar.text(
    0.02, 0.02, "High stress threshold: ≥ 3",
    transform=ax_scar.transAxes,
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.6", alpha=0.9)
)

#fig.suptitle(
#    f"Scope 1 basins — all basins colored; top {TOP_N} emphasized (red border; ranked by {TOP_BY})",
#    fontsize=14, y=0.98
#)
fig.subplots_adjust(left=0.01, right=0.995, top=0.88, bottom=0.10)

out_png = f"{OUT_PREFIX}.png"
out_pdf = f"{OUT_PREFIX}.pdf"
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.savefig(out_pdf, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out_png, "and", out_pdf)

# -----------------------------
# Export "Top-15" rank table (for LaTeX + CSV)
# -----------------------------
tbl = (
    gdf_basin[gdf_basin["_is_top"]][[BASIN_KEY, "mw_sum", "mw_per_1000_km2", "s1_m3", SCARCITY_COL]]
    .sort_values(rank_col, ascending=False)
    .reset_index(drop=True)
)
tbl["rank"] = np.arange(1, len(tbl)+1)
tbl["s1_GL"] = tbl["s1_m3"] / 1e6

# nicer rounding for manuscript
tbl_out = tbl[["rank", BASIN_KEY, "mw_sum", "mw_per_1000_km2", "s1_GL", SCARCITY_COL]].copy()
tbl_out["mw_sum"] = tbl_out["mw_sum"].round(1)
tbl_out["mw_per_1000_km2"] = tbl_out["mw_per_1000_km2"].round(2)
tbl_out["s1_GL"] = tbl_out["s1_GL"].round(2)
tbl_out[SCARCITY_COL] = tbl_out[SCARCITY_COL].round(2)

csv_path = f"{OUT_PREFIX}_top{TOP_N}_table.csv"
tex_path = f"{OUT_PREFIX}_top{TOP_N}_table.tex"
tbl_out.to_csv(csv_path, index=False)

latex = tbl_out.to_latex(index=False, escape=False,
                         caption=f"Top {TOP_N} basins ranked by {TOP_BY} (baseline/mid).",
                         label=f"tab:top{TOP_N}_basins_{TOP_BY}",
                         column_format="r l r r r r")
with open(tex_path, "w") as f:
    f.write(latex)

print("Wrote:", csv_path)
print("Wrote:", tex_path)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, LogNorm
from pyproj import CRS

# -----------------------------
# CONFIG
# -----------------------------
TOP_N = 5
PCT_CAP = 0.99

# Site columns
MW_COL = "current_mw"
S2_COL = "s2_m3_mid"          # site-level Scope 2 water (m3/yr), baseline/mid
SITE_BA_KEY = "subregion_id"  # in CFG['gdf_sites'] (matches your BA polygons)

# BA polygon key
BA_KEY = "subregion_id"       # in CFG['gdf_ba']

# Scarcity exposure:
# We'll compute BA mean stress (capacity-weighted by MW) from site bws_01_score
SITE_STRESS_COL = "bws_01_score"   # on sites (0–5)
SCARCITY_VMIN, SCARCITY_VMAX = 0.0, 5.0

# Toggle: "log" or "linear"
SCALE_MODE = "linear"  # <-- set to "linear" for non-log version

# Colormaps
CMAP_MW_SUM = "viridis"
CMAP_MW_DENS = "cividis"
CMAP_SCARC  = "plasma"

OUT_PREFIX = f"S2_BA_top{TOP_N}_by_s2_{SCALE_MODE}"

# Lambert Conformal Conic for CONUS
LCC_CONUS = CRS.from_proj4(
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 +datum=WGS84 +units=m +no_defs"
)

# Visual styling
ALPHA_OTHER = 0.6
ALPHA_TOP   = 0.95
EDGE_OTHER  = "#d0d0d0"  # light gray BA borders (fix for A/B)
LW_OTHER    = 0.25
EDGE_TOP    = "#cc0000"  # red outline
LW_TOP      = 2.0

STATE_BORDER = "#111111"
STATE_LW = 0.8

# -----------------------------
# Helpers
# -----------------------------
def to_lcc(gdf, crs=LCC_CONUS):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(crs)

def norm_with_cap(vals, pct=PCT_CAP, mode="log", min_pos=1e-3):
    v = pd.Series(vals).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    vmax = float(v.quantile(pct))
    if mode == "log":
        vpos = v[v > 0]
        vmin = max(float(vpos.min()) if len(vpos) else min_pos, min_pos)
        if vmax <= vmin:
            vmax = vmin * 10
        return LogNorm(vmin=vmin, vmax=max(vmax, vmin*1.01)), vmax
    else:
        return Normalize(vmin=0.0, vmax=max(vmax, 1e-12)), vmax

def add_hcolorbar(fig, ax, sm, label, note=None, pad=0.06, fraction=0.05, shrink=0.92):
    cb = fig.colorbar(sm, ax=ax, orientation="horizontal",
                      pad=pad, fraction=fraction, shrink=shrink)
    cb.set_label(label, fontsize=9)
    cb.ax.tick_params(labelsize=8)
    if note:
        cb.ax.set_title(note, fontsize=8, pad=6)
    return cb

def draw_panel(fig, ax, g_all, g_top, col, cmap, norm, states48_l, bounds,
               title, cbar_label, cbar_note=None):
    g_all.plot(
        ax=ax, column=col, cmap=cmap, norm=norm,
        alpha=ALPHA_OTHER, edgecolor=EDGE_OTHER, linewidth=LW_OTHER, zorder=1
    )
    g_top.plot(
        ax=ax, column=col, cmap=cmap, norm=norm,
        alpha=ALPHA_TOP, edgecolor=EDGE_TOP, linewidth=LW_TOP, zorder=5
    )
    states48_l.boundary.plot(ax=ax, color=STATE_BORDER, linewidth=STATE_LW, zorder=10)

    ax.set_title(title, fontsize=12)
    ax.set_axis_off()
    ax.set_aspect("equal")

    minx, miny, maxx, maxy = bounds
    padx, pady = 0.02*(maxx-minx), 0.02*(maxy-miny)
    ax.set_xlim(minx-padx, maxx+padx)
    ax.set_ylim(miny-pady, maxy+pady)

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    add_hcolorbar(fig, ax, sm, cbar_label, note=cbar_note)
    return sm

# -----------------------------
# Inputs
# -----------------------------
assert CFG.get("gdf_sites") is not None, "CFG['gdf_sites'] missing"
assert CFG.get("gdf_ba") is not None, "CFG['gdf_ba'] missing"
assert "states48" in globals(), "states48 missing"

gdf_sites = CFG["gdf_sites"].copy()
gdf_ba    = CFG["gdf_ba"].copy()
states48_g = states48.copy()

# Project to Lambert
gdf_sites  = to_lcc(gdf_sites)
gdf_ba     = to_lcc(gdf_ba)
states48_l = to_lcc(states48_g)

# Clip to CONUS
conus_geom = states48_l.unary_union
gdf_ba = gdf_ba[gdf_ba.intersects(conus_geom)].copy()
gdf_sites = gdf_sites[gdf_sites.intersects(conus_geom)].copy()

# Key cleanup
for c in [SITE_BA_KEY, MW_COL, S2_COL, SITE_STRESS_COL]:
    if c not in gdf_sites.columns:
        raise KeyError(f"Missing '{c}' in gdf_sites columns")

if BA_KEY not in gdf_ba.columns:
    raise KeyError(f"Missing '{BA_KEY}' in gdf_ba columns")

gdf_sites[SITE_BA_KEY] = gdf_sites[SITE_BA_KEY].astype(str)
gdf_ba[BA_KEY]         = gdf_ba[BA_KEY].astype(str)

# -----------------------------
# Aggregate to BA: MW sum, MW density, Scope-2 sum, stress exposure
# -----------------------------
# 1) MW sum + S2 sum
ba_sum = (
    gdf_sites.groupby(SITE_BA_KEY, dropna=False)
    .agg(
        mw_sum=(MW_COL, "sum"),
        s2_m3=(S2_COL, "sum")
    )
    .reset_index()
    .rename(columns={SITE_BA_KEY: BA_KEY})
)

# 2) BA area for density
gdf_ba["area_km2"] = gdf_ba.geometry.area / 1e6

# Merge sums onto polygons
gdf_ba = gdf_ba.merge(ba_sum, on=BA_KEY, how="left")
gdf_ba["mw_sum"] = gdf_ba["mw_sum"].fillna(0.0)
gdf_ba["s2_m3"]  = gdf_ba["s2_m3"].fillna(0.0)

gdf_ba["mw_per_1000_km2"] = np.where(
    gdf_ba["area_km2"] > 0,
    gdf_ba["mw_sum"] / (gdf_ba["area_km2"]/1000.0),
    0.0
)

# 3) Stress exposure by BA: MW-weighted mean site stress (0–5)
tmp = gdf_sites[[SITE_BA_KEY, MW_COL, SITE_STRESS_COL]].copy()
tmp = tmp.rename(columns={SITE_BA_KEY: BA_KEY})
tmp["w"] = tmp[MW_COL].clip(lower=0.0)

# Avoid division by zero
ba_stress = (
    tmp.groupby(BA_KEY)
    .apply(lambda d: np.nan if d["w"].sum() == 0 else np.average(d[SITE_STRESS_COL], weights=d["w"]))
    .reset_index(name="stress_mw_wtd")
)

gdf_ba = gdf_ba.merge(ba_stress, on=BA_KEY, how="left")
gdf_ba["stress_mw_wtd"] = gdf_ba["stress_mw_wtd"].fillna(np.nan)

# -----------------------------
# Top-N by Scope-2 water (S2)
# -----------------------------
top_ids = set(
    gdf_ba.sort_values("s2_m3", ascending=False)
    .head(TOP_N)[BA_KEY]
    .tolist()
)
gdf_ba["_is_top"] = gdf_ba[BA_KEY].isin(top_ids)

# -----------------------------
# Norms (log or linear) + caps
# -----------------------------
norm_mw_sum, _ = norm_with_cap(gdf_ba["mw_sum"], pct=PCT_CAP, mode=SCALE_MODE)
norm_mw_den, _ = norm_with_cap(gdf_ba["mw_per_1000_km2"], pct=PCT_CAP, mode=SCALE_MODE)
norm_scar = Normalize(vmin=SCARCITY_VMIN, vmax=SCARCITY_VMAX)

bounds = states48_l.total_bounds
cap_note = f"winsorized at p{int(PCT_CAP*100)}"
scale_label = "log scale" if SCALE_MODE == "log" else "linear scale"

# -----------------------------
# Plot
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(22, 7.2), gridspec_kw=dict(wspace=0.06))
ax_sum, ax_den, ax_scar = axes

g_all = gdf_ba
g_top = gdf_ba[gdf_ba["_is_top"]].copy()

draw_panel(
    fig, ax_sum, g_all, g_top,
    col="mw_sum", cmap=CMAP_MW_SUM, norm=norm_mw_sum,
    states48_l=states48_l, bounds=bounds,
    title=f"a. Total hyperscale capacity by BA (top-{TOP_N} by Scope 2 outlined)",
    cbar_label=f"MW ({scale_label})",
    cbar_note=cap_note
)

draw_panel(
    fig, ax_den, g_all, g_top,
    col="mw_per_1000_km2", cmap=CMAP_MW_DENS, norm=norm_mw_den,
    states48_l=states48_l, bounds=bounds,
    title=f"b. Hyperscale capacity density by BA (top-{TOP_N} by Scope 2 outlined)",
    cbar_label=f"MW per 1,000 km² ({scale_label})",
    cbar_note=cap_note
)

draw_panel(
    fig, ax_scar, g_all, g_top,
    col="stress_mw_wtd", cmap=CMAP_SCARC, norm=norm_scar,
    states48_l=states48_l, bounds=bounds,
    title="c. Baseline water-stress exposure by BA (MW-weighted mean site stress, 0–5)",
    cbar_label="MW-weighted Aqueduct baseline stress (0–5)"
)

# Make high stress threshold explicit
ax_scar.text(
    0.02, 0.02, "High stress threshold: ≥ 3",
    transform=ax_scar.transAxes,
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.6", alpha=0.9)
)

#fig.suptitle(f"BAs — all regions colored; top {TOP_N} by Scope 2 emphasized (red border)", fontsize=14, y=0.98)
fig.subplots_adjust(left=0.01, right=0.995, top=0.88, bottom=0.10)

out_png = f"{OUT_PREFIX}.png"
out_pdf = f"{OUT_PREFIX}.pdf"
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.savefig(out_pdf, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out_png, "and", out_pdf)

# -----------------------------
# Export top-N rank table (Scope 2)
# -----------------------------
tbl = (
    gdf_ba[gdf_ba["_is_top"]][[BA_KEY, "mw_sum", "mw_per_1000_km2", "s2_m3", "stress_mw_wtd"]]
    .sort_values("s2_m3", ascending=False)
    .reset_index(drop=True)
)
tbl["rank"] = np.arange(1, len(tbl)+1)
tbl["s2_GL"] = tbl["s2_m3"] / 1e6

tbl_out = tbl[["rank", BA_KEY, "mw_sum", "mw_per_1000_km2", "s2_GL", "stress_mw_wtd"]].copy()
tbl_out["mw_sum"] = tbl_out["mw_sum"].round(1)
tbl_out["mw_per_1000_km2"] = tbl_out["mw_per_1000_km2"].round(2)
tbl_out["s2_GL"] = tbl_out["s2_GL"].round(2)
tbl_out["stress_mw_wtd"] = tbl_out["stress_mw_wtd"].round(2)

csv_path = f"{OUT_PREFIX}_top{TOP_N}_table.csv"
tex_path = f"{OUT_PREFIX}_top{TOP_N}_table.tex"
tbl_out.to_csv(csv_path, index=False)

latex = tbl_out.to_latex(index=False, escape=False,
                         caption=f"Top {TOP_N} balancing authorities ranked by Scope 2 water (baseline/mid).",
                         label=f"tab:top{TOP_N}_ba_scope2",
                         column_format="r l r r r r")
with open(tex_path, "w") as f:
    f.write(latex)

print("Wrote:", csv_path)
print("Wrote:", tex_path)

In [ ]:
# ==============================
# FIGURE 1 (IMPACT OPENER): a–c
# National split + S1 basin exposure + S2 BA exposure
# ==============================
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from pyproj import CRS

# -----------------------------
# CONFIG
# -----------------------------
DPI = 300
FIGSIZE = (16.5, 7.2)  # Nature-ish wide figure

# Use Lambert for CONUS
LCC_CONUS = CRS.from_proj4(
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 +datum=WGS84 +units=m +no_defs"
)

# Key columns (from your printed columns)
MW_COL = "current_mw"
SITE_BASIN_KEY = "pfaf_id"
BASIN_KEY = "PFAF_ID"
BA_KEY = "subregion_id"

S1_COL = "s1_m3_mid"     # facility-level Scope 1 (m3/yr), base
S2_COL = "s2_m3_mid"     # facility-level Scope 2 (m3/yr), base
TOT_COL = "total_m3_mid" # facility-level total (m3/yr), base

SCARCITY_COL = "bws_01_score"  # continuous (0–5)

TOP_BASINS = 15
TOP_BAS_OUTLINE = "#cc0000"   # red
TOP_BAS_LW = 1.6

TOP_BAS_TO_SHOW_STRESS = 3.0  # threshold in caption + annotation

TOP_BAS_COLOR = "viridis"
TOP_BA_COLOR  = "cividis"

# For Scope-2 "top 5 only" emphasis
TOP_BAS_FOR_S2 = 5

# Visual
BASIN_EDGE = "#cfcfcf"  # light gray basin boundaries for A/B
BASIN_LW = 0.12
STATE_EDGE = "#111111"
STATE_LW = 0.7

CBAR_HEIGHT = "5%"   # horizontal bars under maps
CBAR_PAD = 0.08

OUT_PNG = "FIG1_impact_opener.png"
OUT_PDF = "FIG1_impact_opener.pdf"

# -----------------------------
# Helpers
# -----------------------------
def to_lcc(gdf, crs=LCC_CONUS):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(crs)

def _cap(vals, q=0.98):
    """Winsorize at q to control outliers for color scaling (not data)."""
    v = pd.Series(vals).replace([np.inf, -np.inf], np.nan).fillna(0)
    vmax = float(v.quantile(q))
    return vmax if vmax > 0 else float(v.max() if len(v) else 1.0)

def _add_h_cbar(fig, ax, sm, label, ticks=None, ticklabels=None):
    """Small horizontal colorbar under the axis, does not steal map area."""
    cax = inset_axes(ax, width="78%", height=CBAR_HEIGHT, loc="lower center",
                     bbox_to_anchor=(0.5, -CBAR_PAD, 1, 1),
                     bbox_transform=ax.transAxes, borderpad=0)
    cb = fig.colorbar(sm, cax=cax, orientation="horizontal")
    cb.set_label(label, fontsize=9)
    cb.ax.tick_params(labelsize=8)
    if ticks is not None:
        cb.set_ticks(ticks)
    if ticklabels is not None:
        cb.set_ticklabels(ticklabels)
    return cb

# -----------------------------
# Inputs
# -----------------------------
assert CFG.get("gdf_sites") is not None, "CFG['gdf_sites'] missing"
assert CFG.get("gdf_basin") is not None, "CFG['gdf_basin'] missing"
assert CFG.get("gdf_ba") is not None, "CFG['gdf_ba'] missing"
assert "states48" in globals(), "states48 missing"

gdf_sites = CFG["gdf_sites"].copy()
gdf_basin = CFG["gdf_basin"].copy()
gdf_ba    = CFG["gdf_ba"].copy()
states48_g = states48.copy()

# Project
gdf_sites = to_lcc(gdf_sites)
gdf_basin = to_lcc(gdf_basin)
gdf_ba    = to_lcc(gdf_ba)
states48_l = to_lcc(states48_g)

# Clip to CONUS
conus_geom = states48_l.unary_union
gdf_sites = gdf_sites[gdf_sites.intersects(conus_geom)].copy()
gdf_basin = gdf_basin[gdf_basin.intersects(conus_geom)].copy()
gdf_ba    = gdf_ba[gdf_ba.intersects(conus_geom)].copy()

# Clean keys
gdf_sites[SITE_BASIN_KEY] = gdf_sites[SITE_BASIN_KEY].astype(str)
gdf_basin[BASIN_KEY] = gdf_basin[BASIN_KEY].astype(str)
gdf_sites[BA_KEY] = gdf_sites[BA_KEY].astype(str)
gdf_ba[BA_KEY] = gdf_ba[BA_KEY].astype(str)

# -----------------------------
# Panel (a): national totals (Base) + placeholder scenario bars
# -----------------------------
# Base totals from facility sums (convert m3 -> GL: 1 GL = 1e6 m3)
s1_gl = gdf_sites[S1_COL].sum() / 1e6
s2_gl = gdf_sites[S2_COL].sum() / 1e6
tot_gl = gdf_sites[TOT_COL].sum() / 1e6

# If you have scenario totals, put them here (GL/yr)
# Format: (eff, base, high)
# If not, we show base only with a faint interval placeholder.
S1_SCEN = None  # e.g. (X1, X2, X3)
S2_SCEN = None  # e.g. (Y1, Y2, Y3)

# -----------------------------
# Panel (b): Scope 1 basin exposure (MW sum) + stress overlay + top basins outline
# -----------------------------
bas_mw = (
    gdf_sites.groupby(SITE_BASIN_KEY, dropna=False)[MW_COL]
    .sum().reset_index()
    .rename(columns={MW_COL: "mw_sum"})
)
gdf_basin2 = gdf_basin.merge(bas_mw, left_on=BASIN_KEY, right_on=SITE_BASIN_KEY, how="left")
gdf_basin2["mw_sum"] = gdf_basin2["mw_sum"].fillna(0.0)

# Top basins by MW sum
top_basin_ids = set(
    gdf_basin2.sort_values("mw_sum", ascending=False).head(TOP_BASINS)[BASIN_KEY].tolist()
)
gdf_basin2["_top"] = gdf_basin2[BASIN_KEY].isin(top_basin_ids)

# Stress mask (>=3)
gdf_basin2["_stress_hi"] = (gdf_basin2[SCARCITY_COL] >= TOP_BAS_TO_SHOW_STRESS)

# Color scaling
vmax_mw = _cap(gdf_basin2["mw_sum"], q=0.98)
norm_mw = Normalize(vmin=0, vmax=vmax_mw)

# -----------------------------
# Panel (c): Scope 2 BA exposure (Scope 2 water summed to BA) + top-5 BA outline
# -----------------------------
ba_s2 = (
    gdf_sites.groupby(BA_KEY, dropna=False)[S2_COL]
    .sum().reset_index()
    .rename(columns={S2_COL: "s2_m3_sum"})
)
gdf_ba2 = gdf_ba.merge(ba_s2, on=BA_KEY, how="left")
gdf_ba2["s2_m3_sum"] = gdf_ba2["s2_m3_sum"].fillna(0.0)
gdf_ba2["s2_GL"] = gdf_ba2["s2_m3_sum"] / 1e6

top_ba_ids = set(
    gdf_ba2.sort_values("s2_GL", ascending=False).head(TOP_BAS_FOR_S2)[BA_KEY].tolist()
)
gdf_ba2["_top"] = gdf_ba2[BA_KEY].isin(top_ba_ids)

vmax_s2 = _cap(gdf_ba2["s2_GL"], q=0.98)
norm_s2 = Normalize(vmin=0, vmax=vmax_s2)

# -----------------------------
# Plot layout
# -----------------------------
fig = plt.figure(figsize=FIGSIZE, constrained_layout=False)

# Grid: (a) tall left, (b,c) two maps on right
# Manual axes for precise control (avoid colorbar overlap)
ax_a  = fig.add_axes([0.05, 0.12, 0.30, 0.78])  # bar + annotations
ax_b  = fig.add_axes([0.40, 0.55, 0.55, 0.35])  # basin map
ax_c  = fig.add_axes([0.40, 0.10, 0.55, 0.35])  # BA map

# -----------------------------
# (a) National split
# -----------------------------
ax_a.set_title("a. National operational water consumption (base case)", loc="left", fontsize=12)

# Two bars: Scope 1, Scope 2; plus total label
vals = [s1_gl, s2_gl]
labels = ["Scope 1\n(direct cooling)", "Scope 2\n(electricity-related)"]

bars = ax_a.bar([0, 1], vals)

# Add base labels
for i, v in enumerate(vals):
    ax_a.text(i, v * 1.01, f"{v:,.1f} GL/yr", ha="center", va="bottom", fontsize=10)

ax_a.text(0.5, max(vals) * 1.18, f"Total: {tot_gl:,.1f} GL/yr", ha="center", fontsize=11, fontweight="bold")

# Optional scenario intervals (if you plug them in)
def _draw_interval(x, triple, color="#444444"):
    eff, base, high = triple
    ax_a.plot([x, x], [eff, high], color=color, lw=2)
    ax_a.plot([x-0.12, x+0.12], [eff, eff], color=color, lw=2)
    ax_a.plot([x-0.12, x+0.12], [high, high], color=color, lw=2)
    ax_a.plot([x-0.12, x+0.12], [base, base], color=color, lw=4)

if S1_SCEN is not None:
    _draw_interval(0, S1_SCEN)
if S2_SCEN is not None:
    _draw_interval(1, S2_SCEN)

ax_a.set_xticks([0, 1])
ax_a.set_xticklabels(labels, fontsize=9)
ax_a.set_ylabel("Water consumption (GL/yr)", fontsize=10)
ax_a.spines["top"].set_visible(False)
ax_a.spines["right"].set_visible(False)

# A short “Nature-style” takeaway annotation
ax_a.text(
    0.0, -0.18,
    "Key message: the national footprint splits across\n"
    "two pathways that concentrate in different geographies.",
    transform=ax_a.transAxes, fontsize=9
)

# -----------------------------
# (b) Scope 1 basin exposure map
# -----------------------------
ax_b.set_title("b. Scope 1 exposure: hyperscale capacity by basin\n(top basins outlined; stress ≥3 hatched)", loc="left", fontsize=12)

# All basins colored by MW sum
gdf_basin2.plot(
    ax=ax_b, column="mw_sum", cmap=TOP_BAS_COLOR, norm=norm_mw,
    edgecolor=BASIN_EDGE, linewidth=BASIN_LW
)

# Overlay high-stress basins with subtle hatch (keeps color visible)
gdf_basin2[gdf_basin2["_stress_hi"]].plot(
    ax=ax_b, facecolor="none", edgecolor="none", hatch="////", linewidth=0.0, zorder=5
)

# Top basins red outline
gdf_basin2[gdf_basin2["_top"]].boundary.plot(
    ax=ax_b, color=TOP_BAS_OUTLINE, linewidth=TOP_BAS_LW, zorder=10
)

# State borders
states48_l.boundary.plot(ax=ax_b, color=STATE_EDGE, linewidth=STATE_LW, zorder=20)

ax_b.set_axis_off()
ax_b.set_aspect("equal")

sm_mw = ScalarMappable(norm=norm_mw, cmap=TOP_BAS_COLOR)
sm_mw.set_array([])
_add_h_cbar(fig, ax_b, sm_mw, "Hyperscale capacity by basin (MW; capped at p98)")

# -----------------------------
# (c) Scope 2 BA exposure map
# -----------------------------
ax_c.set_title("c. Scope 2 exposure: electricity-related water by balancing authority\n(top 5 outlined)", loc="left", fontsize=12)

gdf_ba2.plot(
    ax=ax_c, column="s2_GL", cmap=TOP_BA_COLOR, norm=norm_s2,
    edgecolor="#e6e6e6", linewidth=0.25
)

# Top 5 red outline
gdf_ba2[gdf_ba2["_top"]].boundary.plot(
    ax=ax_c, color=TOP_BAS_OUTLINE, linewidth=1.8, zorder=10
)

states48_l.boundary.plot(ax=ax_c, color=STATE_EDGE, linewidth=STATE_LW, zorder=20)

ax_c.set_axis_off()
ax_c.set_aspect("equal")

sm_s2 = ScalarMappable(norm=norm_s2, cmap=TOP_BA_COLOR)
sm_s2.set_array([])
_add_h_cbar(fig, ax_c, sm_s2, "Scope 2 water by balancing authority (GL/yr; capped at p98)")

# -----------------------------
# Final figure title
# -----------------------------
fig.suptitle("Figure 1 | National magnitude and spatial concentration of operational water use by U.S. hyperscale data centers",
             fontsize=14, y=0.98)

plt.savefig(OUT_PNG, dpi=DPI, bbox_inches="tight")
plt.savefig(OUT_PDF, dpi=DPI, bbox_inches="tight")
plt.show()

print("Saved:", OUT_PNG, "and", OUT_PDF)

In [ ]:
# === IMPACT FIGURE 1 (Nature-style opener): big maps + compact scenario strip ===
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from pyproj import CRS

# -----------------------------
# CONFIG
# -----------------------------
# Map focus
TOP_BASINS_N = 15     # Scope-1 basins: top 15
TOP_BA_N     = 5      # Scope-2 BAs: top 5 (per your latest request)

# Keys / columns (from your printed schema)
SITE_BASIN_KEY = "pfaf_id"      # in CFG['gdf_sites']
BASIN_KEY      = "PFAF_ID"      # in CFG['gdf_basin']
SITE_BA_KEY    = "subregion_id" # in CFG['gdf_sites']
BA_KEY         = "subregion_id" # in CFG['gdf_ba'] or merge target

MW_COL         = "current_mw"
S1_COL_M3      = "s1_m3_mid"
S2_COL_M3      = "s2_m3_mid"
SCARCITY_COL   = "bws_01_score" # continuous (0–5)

# Styling
FIG_W, FIG_H = 17.5, 9.5   # Nature-friendly 2-col figure size
DPI = 350

EDGE_OTHER = "#cfcfcf"     # light gray basin/BA outlines
LW_OTHER   = 0.25

EDGE_TOP   = "#b00000"     # red outlines for emphasized regions
LW_TOP     = 1.8

STATE_BORDER = "#111111"
STATE_LW     = 0.65

ALPHA_OTHER = 0.35         # muted background polygons
ALPHA_TOP   = 0.95         # crisp highlighted polygons

# Color maps: keep perceptually uniform / publication-safe
CMAP_S1 = "viridis"        # Scope-1 water
CMAP_S2 = "cividis"        # Scope-2 water

# Horizontal colorbar settings (under maps)
CBAR_H = "3.2%"
CBAR_W = "78%"
CBAR_PAD_Y = 0.06

# Lambert Conformal Conic for CONUS (as you prefer)
LCC_CONUS = CRS.from_proj4(
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 +datum=WGS84 +units=m +no_defs"
)

# Optional outlier control (winsorize at p98 for map normalization)
CAP_PCT = 0.98

# Inputs
assert "gdf_sites" in CFG and CFG["gdf_sites"] is not None
assert "gdf_basin" in CFG and CFG["gdf_basin"] is not None
assert "gdf_ba" in CFG and CFG["gdf_ba"] is not None
assert "states48" in globals(), "states48 GeoDataFrame must exist"

gdf_sites = CFG["gdf_sites"].copy()
gdf_basin = CFG["gdf_basin"].copy()
gdf_ba    = CFG["gdf_ba"].copy()
states48_g = states48.copy()

# -----------------------------
# Helpers
# -----------------------------
def to_lcc(gdf, crs=LCC_CONUS):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(crs)

def cap_norm(series, pct=CAP_PCT):
    s = pd.Series(series).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    vmax = float(s.quantile(pct)) if (s > 0).any() else float(s.max() if len(s) else 1.0)
    vmax = max(vmax, 1e-12)
    return Normalize(vmin=0.0, vmax=vmax), vmax

def add_horiz_cbar(fig, ax, cmap, norm, label, tick_fmt=None):
    # horizontal colorbar beneath the axis (not stealing map area)
    cax = inset_axes(
        ax, width=CBAR_W, height=CBAR_H, loc="lower center",
        bbox_to_anchor=(0.5, -CBAR_PAD_Y, 1, 1),
        bbox_transform=ax.transAxes, borderpad=0
    )
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cb = fig.colorbar(sm, cax=cax, orientation="horizontal")
    cb.set_label(label, fontsize=9)
    cb.ax.tick_params(labelsize=8)
    if tick_fmt is not None:
        ticks = cb.get_ticks()
        cb.set_ticks(ticks)
        cb.set_ticklabels([tick_fmt(t) for t in ticks])
    return cb

def fmt_GL(x_m3):
    # m3 -> GL (1 GL = 1e6 m3)
    return x_m3 / 1e6

# -----------------------------
# Project + clip to CONUS
# -----------------------------
gdf_sites   = to_lcc(gdf_sites)
gdf_basin   = to_lcc(gdf_basin)
gdf_ba      = to_lcc(gdf_ba)
states48_l  = to_lcc(states48_g)

conus_geom = states48_l.unary_union
gdf_sites  = gdf_sites[gdf_sites.intersects(conus_geom)].copy()
gdf_basin  = gdf_basin[gdf_basin.intersects(conus_geom)].copy()
gdf_ba     = gdf_ba[gdf_ba.intersects(conus_geom)].copy()

# Key cleanup
for col in [SITE_BASIN_KEY, SITE_BA_KEY, MW_COL, S1_COL_M3, S2_COL_M3]:
    if col not in gdf_sites.columns:
        raise KeyError(f"Missing {col} in CFG['gdf_sites']")

for col in [BASIN_KEY, SCARCITY_COL]:
    if col not in gdf_basin.columns:
        raise KeyError(f"Missing {col} in CFG['gdf_basin']")

if BA_KEY not in gdf_ba.columns:
    raise KeyError(f"Missing {BA_KEY} in CFG['gdf_ba'] (your BA polygons)")

gdf_sites[SITE_BASIN_KEY] = gdf_sites[SITE_BASIN_KEY].astype(str)
gdf_sites[SITE_BA_KEY]    = gdf_sites[SITE_BA_KEY].astype(str)
gdf_basin[BASIN_KEY]      = gdf_basin[BASIN_KEY].astype(str)
gdf_ba[BA_KEY]            = gdf_ba[BA_KEY].astype(str)

# -----------------------------
# Aggregate Scope 1 by basin
# -----------------------------
bas_agg = (
    gdf_sites.groupby(SITE_BASIN_KEY, dropna=False)
    .agg(
        s1_m3_sum=(S1_COL_M3, "sum"),
        mw_sum=(MW_COL, "sum")
    )
    .reset_index()
    .rename(columns={SITE_BASIN_KEY: BASIN_KEY})
)

gdf_basin = gdf_basin.merge(bas_agg, on=BASIN_KEY, how="left")
gdf_basin["s1_m3_sum"] = gdf_basin["s1_m3_sum"].fillna(0.0)
gdf_basin["mw_sum"]    = gdf_basin["mw_sum"].fillna(0.0)

# Basins: top-N by Scope-1 water (NOT MW) for an "impact" opener
top_basins = set(
    gdf_basin.sort_values("s1_m3_sum", ascending=False)
    .head(TOP_BASINS_N)[BASIN_KEY].tolist()
)
gdf_basin["_is_top"] = gdf_basin[BASIN_KEY].isin(top_basins)

# Stress highlight (>=3 = high stress)
gdf_basin["_high_stress"] = gdf_basin[SCARCITY_COL].fillna(0.0) >= 3.0

# -----------------------------
# Aggregate Scope 2 by BA
# -----------------------------
ba_agg = (
    gdf_sites.groupby(SITE_BA_KEY, dropna=False)
    .agg(
        s2_m3_sum=(S2_COL_M3, "sum"),
        mw_sum=(MW_COL, "sum")
    )
    .reset_index()
    .rename(columns={SITE_BA_KEY: BA_KEY})
)

gdf_ba = gdf_ba.merge(ba_agg, on=BA_KEY, how="left")
gdf_ba["s2_m3_sum"] = gdf_ba["s2_m3_sum"].fillna(0.0)
gdf_ba["mw_sum"]    = gdf_ba["mw_sum"].fillna(0.0)

top_bas = set(
    gdf_ba.sort_values("s2_m3_sum", ascending=False)
    .head(TOP_BA_N)[BA_KEY].tolist()
)
gdf_ba["_is_top"] = gdf_ba[BA_KEY].isin(top_bas)

# -----------------------------
# Normalizations (winsorized)
# -----------------------------
norm_s1, vmax_s1 = cap_norm(gdf_basin["s1_m3_sum"])
norm_s2, vmax_s2 = cap_norm(gdf_ba["s2_m3_sum"])

# Common bounds
minx, miny, maxx, maxy = states48_l.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

# -----------------------------
# Scenario strip (Eff / Base / High) from national CSV (or computed)
# -----------------------------
nat_path = "table_national_water_totals_mid.csv"
try:
    nat = pd.read_csv(nat_path)
except FileNotFoundError:
    # fallback if you run from a different CWD
    nat = pd.read_csv("table_national_water_totals_mid.csv")

# Base (mid) totals in m3
base_s1 = float(nat.loc[0, "s1_m3_total"])
base_s2 = float(nat.loc[0, "s2_m3_total"])
base_tt = float(nat.loc[0, "total_m3_total"])

# If you DO have a low/high table elsewhere, replace this with those.
# Otherwise, compute low/high via your scenario windows (transparent scaling):
# Scope2 ~ u ; Scope1 ~ u*(WUE/PUE)
u_mid  = float(nat.loc[0, "utilization_u"])
pue_mid = float(nat.loc[0, "PUE"])
wue_mid = float(nat.loc[0, "WUE_L_per_kWh_IT"])

# Define your windows (edit if needed; these match your Methods table)
u_low, u_high   = 0.55, 0.85
pue_low, pue_high = 1.15, 1.40
wue_low, wue_high = 0.2, 1.5

# Scaling factors
s2_low  = base_s2 * (u_low / u_mid)
s2_high = base_s2 * (u_high / u_mid)

s1_low  = base_s1 * (u_low/u_mid) * ((wue_low/pue_low) / (wue_mid/pue_mid))
s1_high = base_s1 * (u_high/u_mid) * ((wue_high/pue_high) / (wue_mid/pue_mid))

tt_low  = s1_low + s2_low
tt_high = s1_high + s2_high

# Convert to GL/yr for plotting
vals_GL = {
    "Scope 1 (direct cooling)": (fmt_GL(s1_low), fmt_GL(base_s1), fmt_GL(s1_high)),
    "Scope 2 (electricity-related)": (fmt_GL(s2_low), fmt_GL(base_s2), fmt_GL(s2_high)),
    "Total (Scope 1 + 2)": (fmt_GL(tt_low), fmt_GL(base_tt), fmt_GL(tt_high)),
}

# -----------------------------
# FIGURE LAYOUT (maps dominate)
# -----------------------------
fig = plt.figure(figsize=(FIG_W, FIG_H), constrained_layout=True)
gs = fig.add_gridspec(
    nrows=2, ncols=2,
    height_ratios=[1.0, 0.28],  # maps big; strip small
    width_ratios=[1.0, 1.0]
)

axA = fig.add_subplot(gs[0, 0])  # Basin Scope-1
axB = fig.add_subplot(gs[0, 1])  # BA Scope-2
axC = fig.add_subplot(gs[1, :])  # Scenario strip spanning width

# -----------------------------
# Panel A: Scope 1 by basin (m3) + high stress hatch + top outlined red
# -----------------------------
gdf_basin.plot(
    ax=axA, column="s1_m3_sum", cmap=CMAP_S1, norm=norm_s1,
    alpha=ALPHA_OTHER, edgecolor=EDGE_OTHER, linewidth=LW_OTHER, zorder=1
)

# Hatch overlay for high-stress basins (>=3) to “make clear” without changing cmap
# Keep subtle (reviewer-friendly)
gdf_basin[gdf_basin["_high_stress"]].plot(
    ax=axA, facecolor="none", edgecolor="none",
    hatch="////", linewidth=0.0, alpha=0.0, zorder=2
)

gdf_basin[gdf_basin["_is_top"]].plot(
    ax=axA, column="s1_m3_sum", cmap=CMAP_S1, norm=norm_s1,
    alpha=ALPHA_TOP, edgecolor=EDGE_TOP, linewidth=LW_TOP, zorder=4
)

states48_l.boundary.plot(ax=axA, color=STATE_BORDER, linewidth=STATE_LW, zorder=10)

axA.set_title(
    f"a. Scope 1 water consumption by basin (Base), top {TOP_BASINS_N} outlined; hatched = high stress (≥3)",
    fontsize=11
)
axA.set_axis_off()
axA.set_xlim(minx - padx, maxx + padx)
axA.set_ylim(miny - pady, maxy + pady)
axA.set_aspect("equal")

add_horiz_cbar(
    fig, axA, cmap=CMAP_S1, norm=norm_s1,
    label="Scope 1 water consumption (m³/yr), winsorized",
    tick_fmt=lambda t: f"{t/1e6:.0f} GL" if t > 0 else "0"
)

# -----------------------------
# Panel B: Scope 2 by BA (m3) + top outlined red (top 5)
# -----------------------------
gdf_ba.plot(
    ax=axB, column="s2_m3_sum", cmap=CMAP_S2, norm=norm_s2,
    alpha=ALPHA_OTHER, edgecolor=EDGE_OTHER, linewidth=LW_OTHER, zorder=1
)
gdf_ba[gdf_ba["_is_top"]].plot(
    ax=axB, column="s2_m3_sum", cmap=CMAP_S2, norm=norm_s2,
    alpha=ALPHA_TOP, edgecolor=EDGE_TOP, linewidth=LW_TOP, zorder=4
)
states48_l.boundary.plot(ax=axB, color=STATE_BORDER, linewidth=STATE_LW, zorder=10)

axB.set_title(
    f"b. Scope 2 water consumption by balancing authority (Base), top {TOP_BA_N} outlined",
    fontsize=11
)
axB.set_axis_off()
axB.set_xlim(minx - padx, maxx + padx)
axB.set_ylim(miny - pady, maxy + pady)
axB.set_aspect("equal")

add_horiz_cbar(
    fig, axB, cmap=CMAP_S2, norm=norm_s2,
    label="Scope 2 water consumption (m³/yr), winsorized",
    tick_fmt=lambda t: f"{t/1e6:.0f} GL" if t > 0 else "0"
)

# -----------------------------
# Panel C: Scenario strip (Eff / Base / High) — compact, “Nature-like”
# -----------------------------
axC.set_axis_off()

labels = list(vals_GL.keys())
y = np.arange(len(labels))[::-1]  # top to bottom

# Draw interval bars and base marker
for i, lab in enumerate(labels):
    lo, mid, hi = vals_GL[lab]
    yy = y[i]
    # interval
    axC.plot([lo, hi], [yy, yy], lw=6, solid_capstyle="round")
    # base marker
    axC.plot([mid], [yy], marker="o", ms=8)
    # text labels
    axC.text(lo, yy + 0.18, "Eff", fontsize=9, ha="center", va="bottom")
    axC.text(mid, yy + 0.18, "Base", fontsize=9, ha="center", va="bottom", fontweight="bold")
    axC.text(hi, yy + 0.18, "High", fontsize=9, ha="center", va="bottom")

# Left-side labels
for i, lab in enumerate(labels):
    axC.text(axC.get_xlim()[0] if axC.get_xlim()[0] != 0 else 0, y[i], "", alpha=0)  # ensure limits exist

# Set sensible x-limits
all_vals = np.array([v for trip in vals_GL.values() for v in trip])
xmin, xmax = all_vals.min(), all_vals.max()
pad = 0.06 * (xmax - xmin)
axC.set_xlim(xmin - pad, xmax + pad)
axC.set_ylim(-0.8, len(labels)-0.2)

# Write labels at left with a fixed x anchor
x_lab = xmin - pad*0.85
for i, lab in enumerate(labels):
    axC.text(x_lab, y[i], lab, fontsize=10, ha="left", va="center")

axC.text(
    xmin - pad*0.85, len(labels)-0.05,
    "c. National totals (GL/yr) — scenario intervals (Eff / Base / High)",
    fontsize=11, ha="left", va="bottom"
)

# -----------------------------
# Suptitle + save
# -----------------------------
fig.suptitle(
    "Figure 1 | Where hyperscale water consumption concentrates (Scope 1 vs Scope 2) and how totals vary across scenarios",
    fontsize=13, y=1.02
)

out_png = "Fig1_Impact_opener_maps_plus_scenarios.png"
out_pdf = "Fig1_Impact_opener_maps_plus_scenarios.pdf"
plt.savefig(out_png, dpi=DPI, bbox_inches="tight")
plt.savefig(out_pdf, dpi=DPI, bbox_inches="tight")
plt.show()

print(f"Saved: {out_png}\nSaved: {out_pdf}")

In [ ]:
# === FIGURE 1 (improved layout): aligned colorbars + Panel C with axis + printed values ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from pyproj import CRS

# -----------------------------
# CONFIG
# -----------------------------
TOP_BASINS_N = 15
TOP_BA_N     = 5

SITE_BASIN_KEY = "pfaf_id"
BASIN_KEY      = "PFAF_ID"
SITE_BA_KEY    = "subregion_id"
BA_KEY         = "subregion_id"

S1_COL_M3      = "s1_m3_mid"
S2_COL_M3      = "s2_m3_mid"
MW_COL         = "current_mw"
SCARCITY_COL   = "bws_01_score"

CMAP_S1 = "viridis"
CMAP_S2 = "viridis" #"cividis"

EDGE_OTHER = "#cfcfcf"
LW_OTHER   = 0.25
EDGE_TOP   = "#b00000"
LW_TOP     = 1.8
STATE_BORDER = "#111111"
STATE_LW     = 0.65
ALPHA_OTHER  = 0.35
ALPHA_TOP    = 0.95

CAP_PCT = 0.98

FIG_W, FIG_H = 17.5, 9.8
DPI = 350

LCC_CONUS = CRS.from_proj4(
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 +datum=WGS84 +units=m +no_defs"
)

def to_lcc(gdf, crs=LCC_CONUS):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(crs)

def cap_norm(series, pct=CAP_PCT):
    s = pd.Series(series).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    vmax = float(s.quantile(pct)) if (s > 0).any() else float(s.max() if len(s) else 1.0)
    vmax = max(vmax, 1e-12)
    return Normalize(vmin=0.0, vmax=vmax), vmax

def m3_to_GL(x):  # 1 GL = 1e6 m3
    return np.asarray(x) / 1e6

# -----------------------------
# Inputs (must exist in your env)
# -----------------------------
assert "gdf_sites" in CFG and CFG["gdf_sites"] is not None
assert "gdf_basin" in CFG and CFG["gdf_basin"] is not None
assert "gdf_ba" in CFG and CFG["gdf_ba"] is not None
assert "states48" in globals()

gdf_sites = to_lcc(CFG["gdf_sites"].copy())
gdf_basin = to_lcc(CFG["gdf_basin"].copy())
gdf_ba    = to_lcc(CFG["gdf_ba"].copy())
states48_l = to_lcc(states48.copy())

conus_geom = states48_l.unary_union
gdf_sites = gdf_sites[gdf_sites.intersects(conus_geom)].copy()
gdf_basin = gdf_basin[gdf_basin.intersects(conus_geom)].copy()
gdf_ba    = gdf_ba[gdf_ba.intersects(conus_geom)].copy()

# Key cleanup
gdf_sites[SITE_BASIN_KEY] = gdf_sites[SITE_BASIN_KEY].astype(str)
gdf_sites[SITE_BA_KEY]    = gdf_sites[SITE_BA_KEY].astype(str)
gdf_basin[BASIN_KEY]      = gdf_basin[BASIN_KEY].astype(str)
gdf_ba[BA_KEY]            = gdf_ba[BA_KEY].astype(str)

# -----------------------------
# Aggregate Scope 1 by basin
# -----------------------------
bas_agg = (
    gdf_sites.groupby(SITE_BASIN_KEY, dropna=False)
    .agg(s1_m3_sum=(S1_COL_M3, "sum"), mw_sum=(MW_COL, "sum"))
    .reset_index()
    .rename(columns={SITE_BASIN_KEY: BASIN_KEY})
)
gdf_basin = gdf_basin.merge(bas_agg, on=BASIN_KEY, how="left")
gdf_basin["s1_m3_sum"] = gdf_basin["s1_m3_sum"].fillna(0.0)
gdf_basin["_is_top"] = gdf_basin[BASIN_KEY].isin(
    gdf_basin.sort_values("s1_m3_sum", ascending=False).head(TOP_BASINS_N)[BASIN_KEY]
)
gdf_basin["_high_stress"] = gdf_basin[SCARCITY_COL].fillna(0.0) >= 3.0

# -----------------------------
# Aggregate Scope 2 by BA
# -----------------------------
ba_agg = (
    gdf_sites.groupby(SITE_BA_KEY, dropna=False)
    .agg(s2_m3_sum=(S2_COL_M3, "sum"), mw_sum=(MW_COL, "sum"))
    .reset_index()
    .rename(columns={SITE_BA_KEY: BA_KEY})
)
gdf_ba = gdf_ba.merge(ba_agg, on=BA_KEY, how="left")
gdf_ba["s2_m3_sum"] = gdf_ba["s2_m3_sum"].fillna(0.0)
gdf_ba["_is_top"] = gdf_ba[BA_KEY].isin(
    gdf_ba.sort_values("s2_m3_sum", ascending=False).head(TOP_BA_N)[BA_KEY]
)

# -----------------------------
# Norms (winsorized)
# -----------------------------
norm_s1, _ = cap_norm(gdf_basin["s1_m3_sum"])
norm_s2, _ = cap_norm(gdf_ba["s2_m3_sum"])

# Common bounds for maps
minx, miny, maxx, maxy = states48_l.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

# -----------------------------
# Panel C values (from your national table; low/high computed transparently)
# -----------------------------
nat = pd.read_csv("table_national_water_totals_mid.csv")

base_s1 = float(nat.loc[0, "s1_m3_total"])
base_s2 = float(nat.loc[0, "s2_m3_total"])
base_tt = float(nat.loc[0, "total_m3_total"])

u_mid   = float(nat.loc[0, "utilization_u"])
pue_mid = float(nat.loc[0, "PUE"])
wue_mid = float(nat.loc[0, "WUE_L_per_kWh_IT"])

# scenario windows (edit if you change the Methods table)
u_low, u_high     = 0.55, 0.85
pue_low, pue_high = 1.15, 1.40
wue_low, wue_high = 0.2, 1.5

s2_low  = base_s2 * (u_low / u_mid)
s2_high = base_s2 * (u_high / u_mid)
s1_low  = base_s1 * (u_low/u_mid) * ((wue_low/pue_low) / (wue_mid/pue_mid))
s1_high = base_s1 * (u_high/u_mid) * ((wue_high/pue_high) / (wue_mid/pue_mid))
tt_low  = s1_low + s2_low
tt_high = s1_high + s2_high

vals_GL = {
    "Scope 1 (direct cooling)": (m3_to_GL(s1_low), m3_to_GL(base_s1), m3_to_GL(s1_high)),
    "Scope 2 (electricity-related)": (m3_to_GL(s2_low), m3_to_GL(base_s2), m3_to_GL(s2_high)),
    "Total (Scope 1 + 2)": (m3_to_GL(tt_low), m3_to_GL(base_tt), m3_to_GL(tt_high)),
}

# -----------------------------
# Figure layout: maps row + aligned cbar row + panel C row
# -----------------------------
fig = plt.figure(figsize=(FIG_W, FIG_H), constrained_layout=False)

# 3 rows: maps, colorbars, scenario strip
gs = fig.add_gridspec(
    nrows=3, ncols=2,
    height_ratios=[1.0, 0.055, 0.33],
    width_ratios=[1.0, 1.0],
    left=0.045, right=0.985, top=0.93, bottom=0.07,
    wspace=0.06, hspace=0.10
)

axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])
caxA = fig.add_subplot(gs[1, 0])
caxB = fig.add_subplot(gs[1, 1])
axC = fig.add_subplot(gs[2, :])

# -----------------------------
# Panel A: Scope 1 basin map
# -----------------------------
gdf_basin.plot(ax=axA, column="s1_m3_sum", cmap=CMAP_S1, norm=norm_s1,
              alpha=ALPHA_OTHER, edgecolor=EDGE_OTHER, linewidth=LW_OTHER, zorder=1)

# stress hatch overlay (subtle)
gdf_basin[gdf_basin["_high_stress"]].plot(
    ax=axA, facecolor="none", edgecolor="none", hatch="////", linewidth=0, zorder=2
)

gdf_basin[gdf_basin["_is_top"]].plot(ax=axA, column="s1_m3_sum", cmap=CMAP_S1, norm=norm_s1,
                                    alpha=ALPHA_TOP, edgecolor=EDGE_TOP, linewidth=LW_TOP, zorder=4)

states48_l.boundary.plot(ax=axA, color=STATE_BORDER, linewidth=STATE_LW, zorder=10)

axA.set_title(f"a. Scope 1 water consumption by basin (Base), top {TOP_BASINS_N} outlined; hatched = high stress (≥3)",
              fontsize=11)
axA.set_axis_off()
axA.set_xlim(minx - padx, maxx + padx); axA.set_ylim(miny - pady, maxy + pady)
axA.set_aspect("equal")

# -----------------------------
# Panel B: Scope 2 BA map
# -----------------------------
gdf_ba.plot(ax=axB, column="s2_m3_sum", cmap=CMAP_S2, norm=norm_s2,
            alpha=ALPHA_OTHER, edgecolor=EDGE_OTHER, linewidth=LW_OTHER, zorder=1)

gdf_ba[gdf_ba["_is_top"]].plot(ax=axB, column="s2_m3_sum", cmap=CMAP_S2, norm=norm_s2,
                               alpha=ALPHA_TOP, edgecolor=EDGE_TOP, linewidth=LW_TOP, zorder=4)

states48_l.boundary.plot(ax=axB, color=STATE_BORDER, linewidth=STATE_LW, zorder=10)

axB.set_title(f"b. Scope 2 water consumption by balancing authority (Base), top {TOP_BA_N} outlined",
              fontsize=11)
axB.set_axis_off()
axB.set_xlim(minx - padx, maxx + padx); axB.set_ylim(miny - pady, maxy + pady)
axB.set_aspect("equal")

# -----------------------------
# Aligned horizontal colorbars (dedicated axes)
# -----------------------------
smA = ScalarMappable(norm=norm_s1, cmap=CMAP_S1); smA.set_array([])
cbA = fig.colorbar(smA, cax=caxA, orientation="horizontal")
cbA.set_label("Scope 1 water consumption (GL/yr), winsorized", fontsize=15)
cbA.ax.tick_params(labelsize=8)
# convert ticks m3 -> GL for labels
ticksA = cbA.get_ticks()
cbA.set_ticklabels([f"{t/1e6:.0f}" for t in ticksA])

smB = ScalarMappable(norm=norm_s2, cmap=CMAP_S2); smB.set_array([])
cbB = fig.colorbar(smB, cax=caxB, orientation="horizontal")
cbB.set_label("Scope 2 water consumption (GL/yr), winsorized", fontsize=15)
cbB.ax.tick_params(labelsize=8)
ticksB = cbB.get_ticks()
cbB.set_ticklabels([f"{t/1e6:.0f}" for t in ticksB])

# Clean colorbar axes spines
for cax in (caxA, caxB):
    for sp in cax.spines.values():
        sp.set_visible(False)

# -----------------------------
# Panel C: scenario interval strip WITH axis + printed values
# -----------------------------
axC.set_title("c. National totals (GL/yr) — scenario intervals (Eff / Base / High)", fontsize=11, loc="center")

labels = list(vals_GL.keys())
y = np.arange(len(labels))[::-1]

# Axis scale based on all values
all_vals = np.array([v for trip in vals_GL.values() for v in trip], dtype=float)
xmin, xmax = float(all_vals.min()), float(all_vals.max())
pad = 0.09 * (xmax - xmin if xmax > xmin else 1.0)
axC.set_xlim(xmin - pad, xmax + pad)
axC.set_ylim(-0.8, len(labels)-0.2)

# Make it readable
axC.grid(axis="x", alpha=0.25)
axC.set_yticks(y)
axC.set_yticklabels(labels, fontsize=10)
axC.tick_params(axis="x", labelsize=9)
axC.set_xlabel("GL per year (GL/yr)", fontsize=10)

def fmt_trip(lo, mid, hi):
    return f"{lo:.0f} / {mid:.0f} / {hi:.0f}"

# Draw each interval on the same axis
for i, lab in enumerate(labels):
    lo, mid, hi = vals_GL[lab]
    yy = y[i]
    axC.plot([lo, hi], [yy, yy], lw=6, solid_capstyle="round")
    axC.plot([mid], [yy], marker="o", ms=7)

    # annotate values at right of each line (Eff/Base/High)
    axC.text(
        xmax + pad*0.2, yy,
        fmt_trip(lo, mid, hi),
        fontsize=9, va="center", ha="left"
    )

    # small Eff/Base/High tags above the line
    axC.text(lo,  yy + 0.18, "Eff",  fontsize=8, ha="center", va="bottom")
    axC.text(mid, yy + 0.18, "Base", fontsize=8, ha="center", va="bottom", fontweight="bold")
    axC.text(hi,  yy + 0.18, "High", fontsize=8, ha="center", va="bottom")

# Give extra space on the right for the printed triplets
axC.set_xlim(xmin - pad, xmax + pad*0.45)

# Clean spines
axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)
axC.spines["left"].set_visible(False)

# -----------------------------
# Suptitle + save
# -----------------------------
#fig.suptitle(
#    "Figure 1 | Where hyperscale water consumption concentrates (Scope 1 vs Scope 2) and how totals vary across scenarios",
#    fontsize=13, y=0.98
#)

out_png = "Fig1_Impact_opener_maps_alignedCbar_panelCaxis.png"
out_pdf = "Fig1_Impact_opener_maps_alignedCbar_panelCaxis.pdf"
plt.savefig(out_png, dpi=DPI, bbox_inches="tight")
plt.savefig(out_pdf, dpi=DPI, bbox_inches="tight")
plt.show()

print(f"Saved: {out_png}\nSaved: {out_pdf}")

In [ ]:
# === FIGURE 2 (Nature-style): Mechanism + hotspots (Scope 2 BA typology + Scope 1 basin hotspots) ===
# Uses ONLY: CFG['gdf_sites'], CFG['gdf_ba'], CFG['gdf_basin'], states48
#
# Panels:
# (a) BA driver-space scatter: grid water intensity vs stress; size=MW; color=Scope2 water (base)
# (b) Scope 2 BA hotspot typology map (base): typology classes; top-5 outlined
# (c) Scope 1 basin hotspot map (base): basins colored by Scope1 (winsorized); top-15 outlined; hatched stress>=3

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib import ticker
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from pyproj import CRS

# -----------------------------
# CONFIG
# -----------------------------
DPI = 300
FIGSIZE = (15.8, 9.2)  # balanced for Nature 2-col figure
OUT_PNG = "Fig2_mechanism_hotspots.png"
OUT_PDF = "Fig2_mechanism_hotspots.pdf"

# Keys / columns (confirmed from your prints)
BA_KEY = "subregion_id"
BASIN_KEY_POLY = "PFAF_ID"
BASIN_KEY_SITE = "pfaf_id"

MW_COL = "current_mw"
GRID_COL_SITE = "grid_L_per_kwh"     # per-site BA grid intensity attached already
FOSSIL_COL_SITE = "share_fossil"     # per-site fossil share attached already
STRESS_COL_SITE = "bws_01_score"     # per-site stress attached already
STRESS_COL_BASIN = "bws_01_score"    # basin polygon stress column (continuous 0–5)
S1_M3_COL = "s1_m3_mid"
S2_M3_COL = "s2_m3_mid"

# Hotspot thresholds (edit here if you want different percentiles)
PCTL_S2 = 0.75         # high Scope2 = top quartile
FOSSIL_THR = 0.50      # fossil-heavy > 0.5
STRESS_THR = 3.0       # high stress exposure >= 3

# Highlight counts
TOP5_BA = 5
TOP15_BASINS = 15

# Winsorize map scales to control outliers
WIN_P = 0.98

# Style
STATE_BORDER = "#1a1a1a"
STATE_LW = 0.60
EDGE_LIGHT = "#d0d0d0"
EDGE_LW = 0.20

OUTLINE_RED = "#cc0000"
OUTLINE_LW = 1.7

# Projections
LCC_CONUS = CRS.from_proj4(
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 +datum=WGS84 +units=m +no_defs"
)

# Colormaps (do not hardcode colors in plots beyond outlines/legend; cmap names are fine)
CMAP_S2 = "cividis"
CMAP_S1 = "viridis"

# -----------------------------
# Helpers
# -----------------------------
def _assert_exists(obj, name):
    if obj is None:
        raise ValueError(f"{name} is None. Make sure it exists in the environment.")
    return obj

def to_lcc(gdf, crs=LCC_CONUS):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(crs)

def winsorize(series, p=WIN_P):
    s = pd.Series(series).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    cap = float(s.quantile(p))
    return np.clip(s, 0, cap), cap

def fmt_gl(x, pos=None):
    # x assumed GL/yr
    if x >= 100:
        return f"{x:.0f}"
    if x >= 10:
        return f"{x:.1f}"
    return f"{x:.2f}"

def fmt_mw(x, pos=None):
    if x >= 1000:
        return f"{x/1000:.1f}k"
    return f"{x:.0f}"

def mw_size_sqrt(mw, smin=20, smax=260):
    mw = np.asarray(mw, dtype=float)
    mw = np.clip(mw, 0, None)
    if mw.max() <= 0:
        return np.full_like(mw, (smin+smax)/2)
    r = np.sqrt(mw / mw.max())
    return smin + (smax - smin) * r

# -----------------------------
# Inputs
# -----------------------------
_assert_exists(CFG.get("gdf_sites", None), "CFG['gdf_sites']")
_assert_exists(CFG.get("gdf_ba", None), "CFG['gdf_ba']")
_assert_exists(CFG.get("gdf_basin", None), "CFG['gdf_basin']")
_assert_exists(globals().get("states48", None), "states48")

gdf_sites = CFG["gdf_sites"].copy()
gdf_ba = CFG["gdf_ba"].copy()
gdf_basin = CFG["gdf_basin"].copy()
states48_g = states48.copy()

# Sanity checks
need_sites = [BA_KEY, BASIN_KEY_SITE, MW_COL, GRID_COL_SITE, FOSSIL_COL_SITE, STRESS_COL_SITE, S1_M3_COL, S2_M3_COL]
missing = [c for c in need_sites if c not in gdf_sites.columns]
if missing:
    raise KeyError(f"Missing required columns in CFG['gdf_sites']: {missing}")

need_ba = [BA_KEY, "geometry"]
missing = [c for c in need_ba if c not in gdf_ba.columns]
if missing:
    raise KeyError(f"Missing required columns in CFG['gdf_ba']: {missing}")

need_basin = [BASIN_KEY_POLY, STRESS_COL_BASIN, "geometry"]
missing = [c for c in need_basin if c not in gdf_basin.columns]
if missing:
    raise KeyError(f"Missing required columns in CFG['gdf_basin']: {missing}")

# Normalize key dtypes
gdf_sites[BA_KEY] = gdf_sites[BA_KEY].astype(str)
gdf_ba[BA_KEY] = gdf_ba[BA_KEY].astype(str)
gdf_sites[BASIN_KEY_SITE] = gdf_sites[BASIN_KEY_SITE].astype(str)
gdf_basin[BASIN_KEY_POLY] = gdf_basin[BASIN_KEY_POLY].astype(str)

# Project everything
gdf_sites = to_lcc(gdf_sites)
gdf_ba = to_lcc(gdf_ba)
gdf_basin = to_lcc(gdf_basin)
states48_l = to_lcc(states48_g)

# Clip to CONUS
conus_geom = states48_l.unary_union
gdf_sites = gdf_sites[gdf_sites.intersects(conus_geom)].copy()
gdf_ba = gdf_ba[gdf_ba.intersects(conus_geom)].copy()
gdf_basin = gdf_basin[gdf_basin.intersects(conus_geom)].copy()

# -----------------------------
# Build BA aggregates from sites (base scenario)
# -----------------------------
# MW totals
ba_mw = gdf_sites.groupby(BA_KEY, dropna=False)[MW_COL].sum().rename("mw_sum")

# Scope 2 base totals (m3 -> GL)
ba_s2_gl = (gdf_sites.groupby(BA_KEY, dropna=False)[S2_M3_COL].sum() / 1e6).rename("s2_gl_base")

# Stress exposure (mean; MW-weighted is also defensible. We'll use MW-weighted by default.)
def wavg(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    m = np.isfinite(x) & np.isfinite(w) & (w > 0)
    if m.sum() == 0:
        return np.nan
    return float(np.sum(x[m] * w[m]) / np.sum(w[m]))

ba_stress = gdf_sites.groupby(BA_KEY).apply(lambda d: wavg(d[STRESS_COL_SITE], d[MW_COL])).rename("stress_wavg")
ba_fossil = gdf_sites.groupby(BA_KEY).apply(lambda d: wavg(d[FOSSIL_COL_SITE], d[MW_COL])).rename("fossil_wavg")
ba_grid = gdf_sites.groupby(BA_KEY).apply(lambda d: wavg(d[GRID_COL_SITE], d[MW_COL])).rename("grid_L_per_kwh_wavg")

ba_stats = pd.concat([ba_mw, ba_s2_gl, ba_stress, ba_fossil, ba_grid], axis=1).reset_index()

# Merge onto BA polygons
gdf_ba2 = gdf_ba.merge(ba_stats, on=BA_KEY, how="left")
for c in ["mw_sum", "s2_gl_base", "stress_wavg", "fossil_wavg", "grid_L_per_kwh_wavg"]:
    gdf_ba2[c] = gdf_ba2[c].fillna(0.0)

# -----------------------------
# Scope 2 hotspot typology (base)
# -----------------------------
s2_thr = float(pd.Series(gdf_ba2["s2_gl_base"]).quantile(PCTL_S2))
gdf_ba2["_is_high_s2"] = gdf_ba2["s2_gl_base"] >= s2_thr
gdf_ba2["_is_fossil"]  = gdf_ba2["fossil_wavg"] > FOSSIL_THR
gdf_ba2["_is_stress"]  = gdf_ba2["stress_wavg"] >= STRESS_THR

# 5-class typology for readability (Nature-friendly)
# 0 none, 1 high S2 only, 2 high S2+fossil, 3 high S2+stress, 4 hotspot triple
def typology(row):
    hs2, fos, strx = row["_is_high_s2"], row["_is_fossil"], row["_is_stress"]
    if hs2 and fos and strx: return 4
    if hs2 and fos and (not strx): return 2
    if hs2 and (not fos) and strx: return 3
    if hs2: return 1
    return 0

gdf_ba2["typology"] = gdf_ba2.apply(typology, axis=1)

# Top-5 BA by Scope 2 base
top5_ba_ids = (
    gdf_ba2.sort_values("s2_gl_base", ascending=False)
    .head(TOP5_BA)[BA_KEY].astype(str).tolist()
)
gdf_ba2["_top5"] = gdf_ba2[BA_KEY].isin(set(top5_ba_ids))

# -----------------------------
# Basin aggregates for Scope 1 (base)
# -----------------------------
bas_s1_gl = (gdf_sites.groupby(BASIN_KEY_SITE)[S1_M3_COL].sum() / 1e6).rename("s1_gl_base")
bas_mw = gdf_sites.groupby(BASIN_KEY_SITE)[MW_COL].sum().rename("mw_sum_basin")

bas_stats = pd.concat([bas_s1_gl, bas_mw], axis=1).reset_index().rename(columns={BASIN_KEY_SITE: BASIN_KEY_POLY})
gdf_basin2 = gdf_basin.merge(bas_stats, on=BASIN_KEY_POLY, how="left")
gdf_basin2["s1_gl_base"] = gdf_basin2["s1_gl_base"].fillna(0.0)
gdf_basin2["mw_sum_basin"] = gdf_basin2["mw_sum_basin"].fillna(0.0)

# Top-15 basins by Scope 1 base (GL)
top15_basin_ids = (
    gdf_basin2.sort_values("s1_gl_base", ascending=False)
    .head(TOP15_BASINS)[BASIN_KEY_POLY].astype(str).tolist()
)
gdf_basin2["_top15"] = gdf_basin2[BASIN_KEY_POLY].isin(set(top15_basin_ids))
gdf_basin2["_stress_hi"] = gdf_basin2[STRESS_COL_BASIN] >= STRESS_THR

# Winsorize for map color scales (Scope1, Scope2)
gdf_basin2["s1_gl_w"], s1_cap = winsorize(gdf_basin2["s1_gl_base"], p=WIN_P)
gdf_ba2["s2_gl_w"], s2_cap = winsorize(gdf_ba2["s2_gl_base"], p=WIN_P)

# -----------------------------
# Figure layout
# -----------------------------
# Use GridSpec for tight alignment and to control map sizes
fig = plt.figure(figsize=FIGSIZE, dpi=DPI, constrained_layout=False)
gs = fig.add_gridspec(
    nrows=2, ncols=2,
    height_ratios=[1.05, 1.0],
    width_ratios=[1.05, 1.0],
    left=0.04, right=0.985, top=0.93, bottom=0.07,
    wspace=0.14, hspace=0.22
)

ax_a = fig.add_subplot(gs[0, 0])  # scatter
ax_b = fig.add_subplot(gs[0, 1])  # BA typology map
ax_c = fig.add_subplot(gs[1, :])  # Basin hotspot map (wide)

# Common bounds from states48
minx, miny, maxx, maxy = states48_l.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

# -----------------------------
# Panel (a): driver-space scatter (BA)
# -----------------------------
# x = grid intensity, y = stress, color = Scope2 water, size = MW
df_sc = gdf_ba2.copy()
df_sc = df_sc[(df_sc["mw_sum"] > 0) & np.isfinite(df_sc["grid_L_per_kwh_wavg"])]

# winsorize color values for scatter
cvals, ccap = winsorize(df_sc["s2_gl_base"], p=WIN_P)
df_sc["_s2_gl_c"] = cvals

sizes = mw_size_sqrt(df_sc["mw_sum"].values, smin=18, smax=240)

sc = ax_a.scatter(
    df_sc["grid_L_per_kwh_wavg"],
    df_sc["stress_wavg"],
    s=sizes,
    c=df_sc["_s2_gl_c"],
    cmap=CMAP_S2,
    norm=Normalize(vmin=0, vmax=ccap),
    alpha=0.90,
    linewidths=0.3
)

# highlight triple-hotspot BAs with red edge
hot = df_sc[(df_sc["typology"] == 4)]
ax_a.scatter(
    hot["grid_L_per_kwh_wavg"], hot["stress_wavg"],
    s=mw_size_sqrt(hot["mw_sum"].values, smin=36, smax=320),
    facecolors="none", edgecolors=OUTLINE_RED, linewidths=1.6, zorder=5
)

# annotate top-5 Scope2 by label (optional but useful)
lab = df_sc.sort_values("s2_gl_base", ascending=False).head(TOP5_BA)
for _, r in lab.iterrows():
    ax_a.text(
        r["grid_L_per_kwh_wavg"], r["stress_wavg"],
        f" {r[BA_KEY]}",
        fontsize=9, weight="bold",
        va="center", ha="left"
    )

ax_a.set_title("a. What drives Scope 2 hotspots (balancing authorities, base)", fontsize=12, loc="left")
ax_a.set_xlabel("BA grid water intensity (L/kWh)")
ax_a.set_ylabel("Mean water-stress exposure (Aqueduct 0–5)")
ax_a.set_ylim(-0.1, 5.1)
ax_a.grid(True, linewidth=0.35, alpha=0.35)

# “High stress ≥3” reference line
ax_a.axhline(STRESS_THR, linewidth=0.9, alpha=0.8)
ax_a.text(ax_a.get_xlim()[0], STRESS_THR + 0.07, "high stress ≥ 3", fontsize=9, va="bottom")

# Colorbar for scatter (small, inside)
cax_a = inset_axes(ax_a, width="2.8%", height="60%", loc="lower right", borderpad=1.2)
cb_a = fig.colorbar(sc, cax=cax_a)
cb_a.set_label("Scope 2 water (GL/yr), winsorized", fontsize=9)
cb_a.ax.tick_params(labelsize=8)
cb_a.formatter = ticker.FuncFormatter(fmt_gl)
cb_a.update_ticks()

# Size legend for MW
# pick three reference points
mw_refs = np.array([df_sc["mw_sum"].quantile(q) for q in [0.25, 0.5, 0.9]])
mw_refs = np.unique(np.clip(mw_refs, 1, None).astype(int))
if len(mw_refs) < 3:
    mw_refs = np.array([50, 300, 1000])
handles = [
    Line2D([0], [0], marker='o', color='none', markerfacecolor='none',
           markeredgecolor='black', markersize=np.sqrt(mw_size_sqrt([m])[0])/1.3,
           label=f"{int(m)} MW")
    for m in mw_refs[:3]
]
leg = ax_a.legend(handles=handles, title="Hyperscale capacity", loc="upper left",
                  frameon=True, fontsize=8, title_fontsize=9, borderpad=0.6)
leg.get_frame().set_linewidth(0.6)

# -----------------------------
# Panel (b): BA typology map (Scope 2)
# -----------------------------
ax_b.set_title("b. Scope 2 hotspot typology (base) — top 5 outlined", fontsize=12, loc="left")

# States background
states48_l.plot(ax=ax_b, color="white", edgecolor=STATE_BORDER, linewidth=STATE_LW, zorder=10)

# Typology colors via categorical mapping (kept minimal; legend handles show meaning)
# (Using a ListedColormap is fine; this is not “hardcoding plot colors” in the data sense.)
from matplotlib.colors import ListedColormap, BoundaryNorm
typ_colors = ListedColormap(["#f0f0f0", "#c6dbef", "#9ecae1", "#fdd0a2", "#fc9272"])
typ_norm = BoundaryNorm([-0.5,0.5,1.5,2.5,3.5,4.5], typ_colors.N)

gdf_ba2.plot(
    ax=ax_b, column="typology",
    cmap=typ_colors, norm=typ_norm,
    edgecolor=EDGE_LIGHT, linewidth=EDGE_LW, zorder=2
)

# Outline top-5
gdf_ba2[gdf_ba2["_top5"]].boundary.plot(ax=ax_b, color=OUTLINE_RED, linewidth=OUTLINE_LW, zorder=20)

# State borders on top
states48_l.boundary.plot(ax=ax_b, color=STATE_BORDER, linewidth=STATE_LW, zorder=30)

ax_b.set_axis_off()
ax_b.set_xlim(minx - padx, maxx + padx)
ax_b.set_ylim(miny - pady, maxy + pady)
ax_b.set_aspect("equal")

# Legend (categorical)
legend_elems = [
    Patch(facecolor=typ_colors(0), edgecolor="#999999", label="Not high Scope 2"),
    Patch(facecolor=typ_colors(1), edgecolor="#999999", label=f"High Scope 2 (≥P{int(PCTL_S2*100)})"),
    Patch(facecolor=typ_colors(2), edgecolor="#999999", label="High Scope 2 + fossil-heavy"),
    Patch(facecolor=typ_colors(3), edgecolor="#999999", label="High Scope 2 + high stress"),
    Patch(facecolor=typ_colors(4), edgecolor="#999999", label="Scope 2 hotspot (triple intersection)"),
    Line2D([0],[0], color=OUTLINE_RED, lw=OUTLINE_LW, label=f"Top {TOP5_BA} BAs by Scope 2"),
]
ax_b.legend(handles=legend_elems, loc="lower left", fontsize=8, frameon=True, borderpad=0.6)

# -----------------------------
# Panel (c): Basin map (Scope 1 base) with stress hatch + top-15 outline
# -----------------------------
ax_c.set_title("c. Scope 1 basin hotspots (base) — top 15 outlined; hatched = high stress (≥3)", fontsize=12, loc="left")

# Background
states48_l.plot(ax=ax_c, color="white", edgecolor=STATE_BORDER, linewidth=STATE_LW, zorder=10)

# Basin colored by winsorized Scope1 (GL)
norm_s1 = Normalize(vmin=0, vmax=s1_cap)
gdf_basin2.plot(
    ax=ax_c, column="s1_gl_w",
    cmap=CMAP_S1, norm=norm_s1,
    edgecolor=EDGE_LIGHT, linewidth=0.08,
    alpha=0.95, zorder=2
)

# Hatch high-stress basins (keep subtle)
gdf_basin2[gdf_basin2["_stress_hi"]].plot(
    ax=ax_c, facecolor="none", edgecolor="none",
    hatch="///", linewidth=0.0, zorder=5
)

# Outline top-15 basins
gdf_basin2[gdf_basin2["_top15"]].boundary.plot(
    ax=ax_c, color=OUTLINE_RED, linewidth=OUTLINE_LW, zorder=20
)

# State borders on top
states48_l.boundary.plot(ax=ax_c, color=STATE_BORDER, linewidth=STATE_LW, zorder=30)

ax_c.set_axis_off()
ax_c.set_xlim(minx - padx, maxx + padx)
ax_c.set_ylim(miny - pady, maxy + pady)
ax_c.set_aspect("equal")

# Horizontal colorbar under Panel (c)
sm_s1 = ScalarMappable(norm=norm_s1, cmap=CMAP_S1)
sm_s1.set_array([])
cax_c = inset_axes(ax_c, width="62%", height="4.5%", loc="lower center", borderpad=2.0)
cb_c = fig.colorbar(sm_s1, cax=cax_c, orientation="horizontal")
cb_c.set_label("Scope 1 water consumption (GL/yr), winsorized", fontsize=9)
cb_c.ax.tick_params(labelsize=8)
cb_c.ax.xaxis.set_major_formatter(ticker.FuncFormatter(fmt_gl))
cb_c.update_ticks()

# Add hatch legend + outline legend
ax_c.legend(
    handles=[
        Patch(facecolor="white", edgecolor="#666666", hatch="///", label="High stress (Aqueduct ≥ 3)"),
        Line2D([0],[0], color=OUTLINE_RED, lw=OUTLINE_LW, label=f"Top {TOP15_BASINS} basins by Scope 1"),
    ],
    loc="lower left", fontsize=8, frameon=True, borderpad=0.6
)

# -----------------------------
# Figure title + notes
# -----------------------------
fig.suptitle(
    "Figure 2 | Why Scope 1 and Scope 2 hotspots differ: drivers (a) and hotspot maps (b–c)",
    fontsize=14, y=0.985
)

# Save
plt.savefig(OUT_PNG, dpi=DPI, bbox_inches="tight")
plt.savefig(OUT_PDF, dpi=DPI, bbox_inches="tight")
plt.show()

print(f"Saved: {OUT_PNG} and {OUT_PDF}")
print(f"Scope 2 high threshold (P{int(PCTL_S2*100)}): {s2_thr:.2f} GL/yr (base)")
print(f"Scope 1 winsor cap (p{int(WIN_P*100)}): {s1_cap:.2f} GL/yr (base)")
print(f"Scope 2 winsor cap (p{int(WIN_P*100)}): {s2_cap:.2f} GL/yr (base)")

In [ ]:
# === FIGURE 2 (fixed layout + correct layering) ===
# Top row: (a) Scope 2 BA typology map, (b) Scope 1 basin hotspots map
# Bottom:  (c) Driver-space scatter (BA)

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from matplotlib.colors import Normalize, ListedColormap, BoundaryNorm
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib import ticker
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from pyproj import CRS

# -----------------------------
# CONFIG
# -----------------------------
DPI = 300
FIGSIZE = (15.8, 9.2)
OUT_PNG = "Fig2_mechanism_hotspots_v2.png"
OUT_PDF = "Fig2_mechanism_hotspots_v2.pdf"

BA_KEY = "subregion_id"
BASIN_KEY_POLY = "PFAF_ID"
BASIN_KEY_SITE = "pfaf_id"

MW_COL = "current_mw"
GRID_COL_SITE = "grid_L_per_kwh"
FOSSIL_COL_SITE = "share_fossil"
STRESS_COL_SITE = "bws_01_score"
STRESS_COL_BASIN = "bws_01_score"
S1_M3_COL = "s1_m3_mid"
S2_M3_COL = "s2_m3_mid"

PCTL_S2 = 0.75
FOSSIL_THR = 0.50
STRESS_THR = 3.0

TOP5_BA = 5
TOP15_BASINS = 15
WIN_P = 0.98

# Map styling
STATE_BORDER = "#1a1a1a"
STATE_LW = 0.65
BA_EDGE = "#d0d0d0"
BA_EDGE_LW = 0.25
BAS_EDGE = "#d9d9d9"
BAS_EDGE_LW = 0.12

OUTLINE_RED = "#cc0000"
OUTLINE_LW = 1.7

CMAP_S2 = "cividis"
CMAP_S1 = "viridis"

LCC_CONUS = CRS.from_proj4(
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 +datum=WGS84 +units=m +no_defs"
)

# -----------------------------
# Helpers
# -----------------------------
def to_lcc(gdf, crs=LCC_CONUS):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(crs)

def winsorize(series, p=WIN_P):
    s = pd.Series(series).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    cap = float(s.quantile(p))
    return np.clip(s, 0, cap), cap

def fmt_gl(x, pos=None):
    if x >= 100:
        return f"{x:.0f}"
    if x >= 10:
        return f"{x:.1f}"
    return f"{x:.2f}"

def mw_size_sqrt(mw, smin=22, smax=260):
    mw = np.asarray(mw, dtype=float)
    mw = np.clip(mw, 0, None)
    if mw.max() <= 0:
        return np.full_like(mw, (smin+smax)/2)
    r = np.sqrt(mw / mw.max())
    return smin + (smax - smin) * r

def wavg(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    m = np.isfinite(x) & np.isfinite(w) & (w > 0)
    if m.sum() == 0:
        return np.nan
    return float(np.sum(x[m] * w[m]) / np.sum(w[m]))

# -----------------------------
# Inputs
# -----------------------------
gdf_sites = CFG["gdf_sites"].copy()
gdf_ba = CFG["gdf_ba"].copy()
gdf_basin = CFG["gdf_basin"].copy()
states48_g = states48.copy()

# Normalize key dtypes
gdf_sites[BA_KEY] = gdf_sites[BA_KEY].astype(str)
gdf_ba[BA_KEY] = gdf_ba[BA_KEY].astype(str)
gdf_sites[BASIN_KEY_SITE] = gdf_sites[BASIN_KEY_SITE].astype(str)
gdf_basin[BASIN_KEY_POLY] = gdf_basin[BASIN_KEY_POLY].astype(str)

# Project + clip
gdf_sites = to_lcc(gdf_sites)
gdf_ba = to_lcc(gdf_ba)
gdf_basin = to_lcc(gdf_basin)
states48_l = to_lcc(states48_g)

conus_geom = states48_l.unary_union
gdf_sites = gdf_sites[gdf_sites.intersects(conus_geom)].copy()
gdf_ba = gdf_ba[gdf_ba.intersects(conus_geom)].copy()
gdf_basin = gdf_basin[gdf_basin.intersects(conus_geom)].copy()

minx, miny, maxx, maxy = states48_l.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

# -----------------------------
# BA aggregates from sites
# -----------------------------
ba_mw = gdf_sites.groupby(BA_KEY)[MW_COL].sum().rename("mw_sum")
ba_s2_gl = (gdf_sites.groupby(BA_KEY)[S2_M3_COL].sum() / 1e6).rename("s2_gl_base")
ba_stress = gdf_sites.groupby(BA_KEY).apply(lambda d: wavg(d[STRESS_COL_SITE], d[MW_COL])).rename("stress_wavg")
ba_fossil = gdf_sites.groupby(BA_KEY).apply(lambda d: wavg(d[FOSSIL_COL_SITE], d[MW_COL])).rename("fossil_wavg")
ba_grid = gdf_sites.groupby(BA_KEY).apply(lambda d: wavg(d[GRID_COL_SITE], d[MW_COL])).rename("grid_L_per_kwh_wavg")

ba_stats = pd.concat([ba_mw, ba_s2_gl, ba_stress, ba_fossil, ba_grid], axis=1).reset_index()
gdf_ba2 = gdf_ba.merge(ba_stats, on=BA_KEY, how="left").fillna(0.0)

# typology threshold
s2_thr = float(pd.Series(gdf_ba2["s2_gl_base"]).quantile(PCTL_S2))
gdf_ba2["_is_high_s2"] = gdf_ba2["s2_gl_base"] >= s2_thr
gdf_ba2["_is_fossil"]  = gdf_ba2["fossil_wavg"] > FOSSIL_THR
gdf_ba2["_is_stress"]  = gdf_ba2["stress_wavg"] >= STRESS_THR

def typology(row):
    hs2, fos, strx = row["_is_high_s2"], row["_is_fossil"], row["_is_stress"]
    if hs2 and fos and strx: return 4
    if hs2 and fos and (not strx): return 2
    if hs2 and (not fos) and strx: return 3
    if hs2: return 1
    return 0

gdf_ba2["typology"] = gdf_ba2.apply(typology, axis=1)

top5_ba_ids = (
    gdf_ba2.sort_values("s2_gl_base", ascending=False)
    .head(TOP5_BA)[BA_KEY].tolist()
)
gdf_ba2["_top5"] = gdf_ba2[BA_KEY].isin(set(top5_ba_ids))

# winsor for map scale (optional)
gdf_ba2["s2_gl_w"], s2_cap = winsorize(gdf_ba2["s2_gl_base"], p=WIN_P)

# -----------------------------
# Basin aggregates (Scope 1)
# -----------------------------
bas_s1_gl = (gdf_sites.groupby(BASIN_KEY_SITE)[S1_M3_COL].sum() / 1e6).rename("s1_gl_base")
bas_stats = bas_s1_gl.reset_index().rename(columns={BASIN_KEY_SITE: BASIN_KEY_POLY})
gdf_basin2 = gdf_basin.merge(bas_stats, on=BASIN_KEY_POLY, how="left")
gdf_basin2["s1_gl_base"] = gdf_basin2["s1_gl_base"].fillna(0.0)

top15_basin_ids = (
    gdf_basin2.sort_values("s1_gl_base", ascending=False)
    .head(TOP15_BASINS)[BASIN_KEY_POLY].tolist()
)
gdf_basin2["_top15"] = gdf_basin2[BASIN_KEY_POLY].isin(set(top15_basin_ids))
gdf_basin2["_stress_hi"] = gdf_basin2[STRESS_COL_BASIN] >= STRESS_THR

gdf_basin2["s1_gl_w"], s1_cap = winsorize(gdf_basin2["s1_gl_base"], p=WIN_P)

# -----------------------------
# FIG LAYOUT: two maps on top, scatter bottom
# -----------------------------
fig = plt.figure(figsize=FIGSIZE, dpi=DPI)
gs = fig.add_gridspec(
    nrows=2, ncols=2,
    height_ratios=[1.05, 0.95],
    width_ratios=[1.0, 1.0],
    left=0.05, right=0.98, top=0.93, bottom=0.08,
    wspace=0.10, hspace=0.22
)

ax_map_ba = fig.add_subplot(gs[0, 0])
ax_map_bas = fig.add_subplot(gs[0, 1])
ax_sc = fig.add_subplot(gs[1, :])

# -----------------------------
# (a) BA typology map
# -----------------------------
ax_map_ba.set_title("a. Scope 2 hotspot typology (base) — top 5 outlined", fontsize=12, loc="left")

typ_colors = ListedColormap(["#f0f0f0", "#c6dbef", "#9ecae1", "#fdd0a2", "#fc9272"])
typ_norm = BoundaryNorm([-0.5,0.5,1.5,2.5,3.5,4.5], typ_colors.N)

# Plot BAs FIRST (no state fill!)
gdf_ba2.plot(
    ax=ax_map_ba, column="typology",
    cmap=typ_colors, norm=typ_norm,
    edgecolor=BA_EDGE, linewidth=BA_EDGE_LW,
    zorder=2
)

# Outline top-5
gdf_ba2[gdf_ba2["_top5"]].boundary.plot(
    ax=ax_map_ba, color=OUTLINE_RED, linewidth=OUTLINE_LW, zorder=5
)

# States LAST, boundary only
states48_l.boundary.plot(ax=ax_map_ba, color=STATE_BORDER, linewidth=STATE_LW, zorder=10)

ax_map_ba.set_axis_off()
ax_map_ba.set_xlim(minx - padx, maxx + padx)
ax_map_ba.set_ylim(miny - pady, maxy + pady)
ax_map_ba.set_aspect("equal")

# Categorical legend (move to whitespace)
legend_elems = [
    Patch(facecolor=typ_colors(0), edgecolor="#999999", label="Not high Scope 2"),
    Patch(facecolor=typ_colors(1), edgecolor="#999999", label=f"High Scope 2 (≥P{int(PCTL_S2*100)})"),
    Patch(facecolor=typ_colors(2), edgecolor="#999999", label="High Scope 2 + fossil-heavy"),
    Patch(facecolor=typ_colors(3), edgecolor="#999999", label="High Scope 2 + high stress"),
    Patch(facecolor=typ_colors(4), edgecolor="#999999", label="Scope 2 hotspot (triple)"),
    Line2D([0],[0], color=OUTLINE_RED, lw=OUTLINE_LW, label=f"Top {TOP5_BA} by Scope 2"),
]
ax_map_ba.legend(handles=legend_elems, loc="lower left", fontsize=8, frameon=True, borderpad=0.6)

# -----------------------------
# (b) Basin Scope 1 hotspot map
# -----------------------------
ax_map_bas.set_title("b. Scope 1 basin hotspots (base) — top 15 outlined; hatched = high stress (≥3)", fontsize=12, loc="left")

norm_s1 = Normalize(vmin=0, vmax=s1_cap)

gdf_basin2.plot(
    ax=ax_map_bas, column="s1_gl_w",
    cmap=CMAP_S1, norm=norm_s1,
    edgecolor=BAS_EDGE, linewidth=BAS_EDGE_LW,
    zorder=2
)

# Hatch high stress, without wiping fill color
gdf_basin2[gdf_basin2["_stress_hi"]].plot(
    ax=ax_map_bas, facecolor="none", edgecolor="#666666",
    linewidth=0.0, hatch="///", zorder=4
)

# Outline top basins
gdf_basin2[gdf_basin2["_top15"]].boundary.plot(
    ax=ax_map_bas, color=OUTLINE_RED, linewidth=OUTLINE_LW, zorder=6
)

# States boundary last
states48_l.boundary.plot(ax=ax_map_bas, color=STATE_BORDER, linewidth=STATE_LW, zorder=10)

ax_map_bas.set_axis_off()
ax_map_bas.set_xlim(minx - padx, maxx + padx)
ax_map_bas.set_ylim(miny - pady, maxy + pady)
ax_map_bas.set_aspect("equal")

# Horizontal colorbar under basin map (compact)
sm_s1 = ScalarMappable(norm=norm_s1, cmap=CMAP_S1)
sm_s1.set_array([])
cax_b = inset_axes(ax_map_bas, width="70%", height="5%", loc="lower center", borderpad=1.2)
cb_b = fig.colorbar(sm_s1, cax=cax_b, orientation="horizontal")
cb_b.set_label("Scope 1 water (GL/yr), winsorized", fontsize=9)
cb_b.ax.tick_params(labelsize=8)
cb_b.ax.xaxis.set_major_formatter(ticker.FuncFormatter(fmt_gl))
cb_b.update_ticks()

ax_map_bas.legend(
    handles=[
        Patch(facecolor="white", edgecolor="#666666", hatch="///", label="High stress (Aqueduct ≥ 3)"),
        Line2D([0],[0], color=OUTLINE_RED, lw=OUTLINE_LW, label=f"Top {TOP15_BASINS} basins by Scope 1"),
    ],
    loc="lower left", fontsize=8, frameon=True, borderpad=0.6
)

# -----------------------------
# (c) Scatter BELOW
# -----------------------------
ax_sc.set_title("c. Drivers of Scope 2 hotspots across balancing authorities (base)", fontsize=12, loc="left")

df_sc = gdf_ba2.copy()
df_sc = df_sc[(df_sc["mw_sum"] > 0) & np.isfinite(df_sc["grid_L_per_kwh_wavg"])].copy()

# winsorize color for scatter
cvals, ccap = winsorize(df_sc["s2_gl_base"], p=WIN_P)
df_sc["_s2_gl_c"] = cvals

sizes = mw_size_sqrt(df_sc["mw_sum"].values, smin=20, smax=260)

sc = ax_sc.scatter(
    df_sc["grid_L_per_kwh_wavg"],
    df_sc["stress_wavg"],
    s=sizes,
    c=df_sc["_s2_gl_c"],
    cmap=CMAP_S2,
    norm=Normalize(vmin=0, vmax=ccap),
    alpha=0.90,
    linewidths=0.25
)

# highlight triple intersection
hot = df_sc[df_sc["typology"] == 4]
ax_sc.scatter(
    hot["grid_L_per_kwh_wavg"], hot["stress_wavg"],
    s=mw_size_sqrt(hot["mw_sum"].values, smin=50, smax=340),
    facecolors="none", edgecolors=OUTLINE_RED, linewidths=1.7, zorder=5
)

# stress threshold line
ax_sc.axhline(STRESS_THR, linewidth=0.9, alpha=0.85)
ax_sc.text(ax_sc.get_xlim()[0], STRESS_THR + 0.08, "high stress ≥ 3", fontsize=9, va="bottom")

# annotate top-5 by Scope2 (optional)
lab = df_sc.sort_values("s2_gl_base", ascending=False).head(TOP5_BA)
for _, r in lab.iterrows():
    ax_sc.text(r["grid_L_per_kwh_wavg"], r["stress_wavg"], f" {r[BA_KEY]}",
               fontsize=9, weight="bold", va="center", ha="left")

ax_sc.set_xlabel("BA grid water intensity (L/kWh)")
ax_sc.set_ylabel("Mean water-stress exposure (Aqueduct 0–5)")
ax_sc.set_ylim(-0.1, 5.1)
ax_sc.grid(True, linewidth=0.35, alpha=0.30)

# --- move BOTH legends outside (no overlap) ---
# Size legend
mw_refs = np.array([df_sc["mw_sum"].quantile(q) for q in [0.25, 0.5, 0.9]])
mw_refs = np.unique(np.clip(mw_refs, 1, None).astype(int))
if len(mw_refs) < 3:
    mw_refs = np.array([50, 300, 1000])

handles = [
    Line2D([0], [0], marker='o', color='none',
           markerfacecolor='none', markeredgecolor='black',
           markersize=np.sqrt(mw_size_sqrt([m], smin=20, smax=260)[0])/1.3,
           label=f"{int(m)} MW")
    for m in mw_refs[:3]
]
leg1 = ax_sc.legend(handles=handles, title="Hyperscale capacity", loc="upper left",
                    bbox_to_anchor=(1.01, 1.00), frameon=True,
                    fontsize=8, title_fontsize=9, borderpad=0.6)
leg1.get_frame().set_linewidth(0.6)

# Colorbar outside (right)
cax_sc = inset_axes(ax_sc, width="1.8%", height="70%", loc="center left",
                    bbox_to_anchor=(1.18, 0.05, 1, 1), bbox_transform=ax_sc.transAxes, borderpad=0)
cb_sc = fig.colorbar(sc, cax=cax_sc)
cb_sc.set_label("Scope 2 water (GL/yr), winsorized", fontsize=9)
cb_sc.ax.tick_params(labelsize=8)
cb_sc.formatter = ticker.FuncFormatter(fmt_gl)
cb_sc.update_ticks()

# -----------------------------
# Title + save
# -----------------------------
fig.suptitle(
    "Figure 2 | Why Scope 1 and Scope 2 hotspots differ: maps (a–b) and drivers (c)",
    fontsize=14, y=0.985
)

plt.savefig(OUT_PNG, dpi=DPI, bbox_inches="tight")
plt.savefig(OUT_PDF, dpi=DPI, bbox_inches="tight")
plt.show()

print(f"Saved: {OUT_PNG} and {OUT_PDF}")
print(f"Scope 2 high threshold (P{int(PCTL_S2*100)}): {s2_thr:.2f} GL/yr (base)")
print(f"Scope 1 winsor cap (p{int(WIN_P*100)}): {s1_cap:.2f} GL/yr (base)")
print(f"Scope 2 winsor cap (p{int(WIN_P*100)}): {ccap:.2f} GL/yr (base)")

In [ ]:
# === FIGURE 2 (Nature polish pass): better proportions + no legend overlap + new outline color ===

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from matplotlib.colors import Normalize, ListedColormap, BoundaryNorm
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib import ticker
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from pyproj import CRS

# -----------------------------
# CONFIG
# -----------------------------
DPI = 300
FIGSIZE = (16.0, 9.5)

OUT_PNG = "Fig2_mechanism_hotspots_v3.png"
OUT_PDF = "Fig2_mechanism_hotspots_v3.pdf"

BA_KEY = "subregion_id"
BASIN_KEY_POLY = "PFAF_ID"
BASIN_KEY_SITE = "pfaf_id"

MW_COL = "current_mw"
GRID_COL_SITE = "grid_L_per_kwh"
FOSSIL_COL_SITE = "share_fossil"
STRESS_COL_SITE = "bws_01_score"
STRESS_COL_BASIN = "bws_01_score"
S1_M3_COL = "s1_m3_mid"
S2_M3_COL = "s2_m3_mid"

PCTL_S2 = 0.75
FOSSIL_THR = 0.50
STRESS_THR = 3.0

TOP5_BA = 5
TOP15_BASINS = 15
WIN_P = 0.98

# ---- STYLE ----
STATE_BORDER = "#1a1a1a"
STATE_LW = 0.55

BA_EDGE = "#cfcfcf"
BA_EDGE_LW = 0.25

BAS_EDGE = "#d6d6d6"
BAS_EDGE_LW = 0.10

# Alternative to red (better in print, still high contrast)
OUTLINE = "#7a0177"      # dark magenta
OUTLINE_LW = 1.6

STRESS_EDGE = "#4d4d4d"
STRESS_EDGE_LW = 0.35
STRESS_HATCH = "///"

CMAP_S1 = "viridis"
CMAP_S2 = "cividis"

# Muted typology palette (print friendly)
# 0: not high, 1: high s2, 2: high s2+fossil, 3: high s2+stress, 4: triple
TYPO_COLORS = ListedColormap([
    "#f2f2f2",  # 0
    "#c6dbef",  # 1
    "#9ecae1",  # 2
    "#fdd0a2",  # 3
    "#fb6a4a",  # 4 (strong)
])
TYPO_NORM = BoundaryNorm([-0.5,0.5,1.5,2.5,3.5,4.5], TYPO_COLORS.N)

LCC_CONUS = CRS.from_proj4(
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 +datum=WGS84 +units=m +no_defs"
)

# -----------------------------
# Helpers
# -----------------------------
def to_lcc(gdf, crs=LCC_CONUS):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(crs)

def winsorize(series, p=WIN_P):
    s = pd.Series(series).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    cap = float(s.quantile(p))
    return np.clip(s, 0, cap), cap

def fmt_gl(x, pos=None):
    if x >= 100: return f"{x:.0f}"
    if x >= 10:  return f"{x:.1f}"
    return f"{x:.2f}"

def mw_size_sqrt(mw, smin=18, smax=240):
    mw = np.asarray(mw, dtype=float)
    mw = np.clip(mw, 0, None)
    if mw.max() <= 0:
        return np.full_like(mw, (smin+smax)/2)
    r = np.sqrt(mw / mw.max())
    return smin + (smax - smin) * r

def wavg(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    m = np.isfinite(x) & np.isfinite(w) & (w > 0)
    if m.sum() == 0:
        return np.nan
    return float(np.sum(x[m] * w[m]) / np.sum(w[m]))

# -----------------------------
# Inputs
# -----------------------------
gdf_sites  = CFG["gdf_sites"].copy()
gdf_ba     = CFG["gdf_ba"].copy()
gdf_basin  = CFG["gdf_basin"].copy()
states48_g = states48.copy()

# normalize keys
gdf_sites[BA_KEY] = gdf_sites[BA_KEY].astype(str)
gdf_ba[BA_KEY] = gdf_ba[BA_KEY].astype(str)
gdf_sites[BASIN_KEY_SITE] = gdf_sites[BASIN_KEY_SITE].astype(str)
gdf_basin[BASIN_KEY_POLY] = gdf_basin[BASIN_KEY_POLY].astype(str)

# project + clip
gdf_sites  = to_lcc(gdf_sites)
gdf_ba     = to_lcc(gdf_ba)
gdf_basin  = to_lcc(gdf_basin)
states48_l = to_lcc(states48_g)

conus_geom = states48_l.unary_union
gdf_sites  = gdf_sites[gdf_sites.intersects(conus_geom)].copy()
gdf_ba     = gdf_ba[gdf_ba.intersects(conus_geom)].copy()
gdf_basin  = gdf_basin[gdf_basin.intersects(conus_geom)].copy()

minx, miny, maxx, maxy = states48_l.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

# -----------------------------
# BA aggregates
# -----------------------------
ba_mw = gdf_sites.groupby(BA_KEY)[MW_COL].sum().rename("mw_sum")
ba_s2_gl = (gdf_sites.groupby(BA_KEY)[S2_M3_COL].sum() / 1e6).rename("s2_gl_base")
ba_stress = gdf_sites.groupby(BA_KEY).apply(lambda d: wavg(d[STRESS_COL_SITE], d[MW_COL])).rename("stress_wavg")
ba_fossil = gdf_sites.groupby(BA_KEY).apply(lambda d: wavg(d[FOSSIL_COL_SITE], d[MW_COL])).rename("fossil_wavg")
ba_grid = gdf_sites.groupby(BA_KEY).apply(lambda d: wavg(d[GRID_COL_SITE], d[MW_COL])).rename("grid_L_per_kwh_wavg")

ba_stats = pd.concat([ba_mw, ba_s2_gl, ba_stress, ba_fossil, ba_grid], axis=1).reset_index()
gdf_ba2 = gdf_ba.merge(ba_stats, on=BA_KEY, how="left").fillna(0.0)

s2_thr = float(pd.Series(gdf_ba2["s2_gl_base"]).quantile(PCTL_S2))
gdf_ba2["_is_high_s2"] = gdf_ba2["s2_gl_base"] >= s2_thr
gdf_ba2["_is_fossil"]  = gdf_ba2["fossil_wavg"] > FOSSIL_THR
gdf_ba2["_is_stress"]  = gdf_ba2["stress_wavg"] >= STRESS_THR

def typology(row):
    hs2, fos, st = row["_is_high_s2"], row["_is_fossil"], row["_is_stress"]
    if hs2 and fos and st: return 4
    if hs2 and fos:        return 2
    if hs2 and st:         return 3
    if hs2:                return 1
    return 0

gdf_ba2["typology"] = gdf_ba2.apply(typology, axis=1)

top5_ba_ids = gdf_ba2.sort_values("s2_gl_base", ascending=False).head(TOP5_BA)[BA_KEY].tolist()
gdf_ba2["_top5"] = gdf_ba2[BA_KEY].isin(set(top5_ba_ids))

# -----------------------------
# Basin aggregates (Scope 1)
# -----------------------------
bas_s1_gl = (gdf_sites.groupby(BASIN_KEY_SITE)[S1_M3_COL].sum() / 1e6).rename("s1_gl_base")
bas_stats = bas_s1_gl.reset_index().rename(columns={BASIN_KEY_SITE: BASIN_KEY_POLY})
gdf_basin2 = gdf_basin.merge(bas_stats, on=BASIN_KEY_POLY, how="left")
gdf_basin2["s1_gl_base"] = gdf_basin2["s1_gl_base"].fillna(0.0)

top15_basin_ids = gdf_basin2.sort_values("s1_gl_base", ascending=False).head(TOP15_BASINS)[BASIN_KEY_POLY].tolist()
gdf_basin2["_top15"] = gdf_basin2[BASIN_KEY_POLY].isin(set(top15_basin_ids))
gdf_basin2["_stress_hi"] = gdf_basin2[STRESS_COL_BASIN] >= STRESS_THR

gdf_basin2["s1_gl_w"], s1_cap = winsorize(gdf_basin2["s1_gl_base"], p=WIN_P)
norm_s1 = Normalize(vmin=0, vmax=s1_cap)

# -----------------------------
# Layout (more room for maps, legends outside)
# -----------------------------
fig = plt.figure(figsize=FIGSIZE, dpi=DPI)
gs = fig.add_gridspec(
    2, 2,
    height_ratios=[1.25, 0.85],
    width_ratios=[1.0, 1.0],
    left=0.05, right=0.86, top=0.93, bottom=0.08,
    wspace=0.10, hspace=0.26
)

axA = fig.add_subplot(gs[0, 0])   # BA typology
axB = fig.add_subplot(gs[0, 1])   # Basin scope1
axC = fig.add_subplot(gs[1, :])   # Scatter

# -----------------------------
# Panel a: BA typology map (legend OUTSIDE)
# -----------------------------
axA.set_title("a. Scope 2 hotspot typology (base) — top 5 outlined", fontsize=12, loc="left")

gdf_ba2.plot(ax=axA, column="typology", cmap=TYPO_COLORS, norm=TYPO_NORM,
            edgecolor=BA_EDGE, linewidth=BA_EDGE_LW, zorder=2)
gdf_ba2[gdf_ba2["_top5"]].boundary.plot(ax=axA, color=OUTLINE, linewidth=OUTLINE_LW, zorder=6)
states48_l.boundary.plot(ax=axA, color=STATE_BORDER, linewidth=STATE_LW, zorder=10)

axA.set_axis_off()
axA.set_xlim(minx - padx, maxx + padx)
axA.set_ylim(miny - pady, maxy + pady)
axA.set_aspect("equal")

# legend OUTSIDE to right margin (figure-level)
legA = [
    Patch(facecolor=TYPO_COLORS(0), edgecolor="#999999", label="Not high Scope 2"),
    Patch(facecolor=TYPO_COLORS(1), edgecolor="#999999", label=f"High Scope 2 (≥P{int(PCTL_S2*100)})"),
    Patch(facecolor=TYPO_COLORS(2), edgecolor="#999999", label="High Scope 2 + fossil-heavy"),
    Patch(facecolor=TYPO_COLORS(3), edgecolor="#999999", label="High Scope 2 + high stress"),
    Patch(facecolor=TYPO_COLORS(4), edgecolor="#999999", label="Scope 2 hotspot (triple)"),
    Line2D([0],[0], color=OUTLINE, lw=OUTLINE_LW, label=f"Top {TOP5_BA} by Scope 2"),
]
axA.legend(handles=legA, loc="upper left", bbox_to_anchor=(1.02, 1.02),
           fontsize=8, frameon=True, borderpad=0.6)

# -----------------------------
# Panel b: Basin scope 1 + hatch + outline (legend OUTSIDE, colorbar compact)
# -----------------------------
axB.set_title("b. Scope 1 basin hotspots (base) — top 15 outlined; hatched = high stress (≥3)", fontsize=12, loc="left")

gdf_basin2.plot(ax=axB, column="s1_gl_w", cmap=CMAP_S1, norm=norm_s1,
               edgecolor=BAS_EDGE, linewidth=BAS_EDGE_LW, zorder=2)

# hatch + thin boundary to increase readability
gdf_basin2[gdf_basin2["_stress_hi"]].plot(
    ax=axB, facecolor="none", edgecolor=STRESS_EDGE,
    linewidth=STRESS_EDGE_LW, hatch=STRESS_HATCH, zorder=4
)

gdf_basin2[gdf_basin2["_top15"]].boundary.plot(ax=axB, color=OUTLINE, linewidth=OUTLINE_LW, zorder=6)
states48_l.boundary.plot(ax=axB, color=STATE_BORDER, linewidth=STATE_LW, zorder=10)

axB.set_axis_off()
axB.set_xlim(minx - padx, maxx + padx)
axB.set_ylim(miny - pady, maxy + pady)
axB.set_aspect("equal")

# compact horizontal colorbar under panel b (centered)
sm_s1 = ScalarMappable(norm=norm_s1, cmap=CMAP_S1); sm_s1.set_array([])
caxB = inset_axes(axB, width="72%", height="5.5%", loc="lower center", borderpad=1.1)
cbB = fig.colorbar(sm_s1, cax=caxB, orientation="horizontal")
cbB.set_label("Scope 1 water (GL/yr), winsorized", fontsize=9)
cbB.ax.tick_params(labelsize=8)
cbB.ax.xaxis.set_major_formatter(ticker.FuncFormatter(fmt_gl))
cbB.update_ticks()

# legend OUTSIDE
legB = [
    Patch(facecolor="white", edgecolor=STRESS_EDGE, hatch=STRESS_HATCH, label="High stress (Aqueduct ≥ 3)"),
    Line2D([0],[0], color=OUTLINE, lw=OUTLINE_LW, label=f"Top {TOP15_BASINS} basins by Scope 1"),
]
axB.legend(handles=legB, loc="lower left", bbox_to_anchor=(1.02, 0.05),
           fontsize=8, frameon=True, borderpad=0.6)

# -----------------------------
# Panel c: Driver scatter (legends OUTSIDE + no overlap)
# -----------------------------
axC.set_title("c. Drivers of Scope 2 hotspots across balancing authorities (base)", fontsize=12, loc="left")

df_sc = gdf_ba2[(gdf_ba2["mw_sum"] > 0) & np.isfinite(gdf_ba2["grid_L_per_kwh_wavg"])].copy()
cvals, ccap = winsorize(df_sc["s2_gl_base"], p=WIN_P)
df_sc["_s2_gl_c"] = cvals
sizes = mw_size_sqrt(df_sc["mw_sum"].values, smin=18, smax=240)

sc = axC.scatter(
    df_sc["grid_L_per_kwh_wavg"], df_sc["stress_wavg"],
    s=sizes, c=df_sc["_s2_gl_c"],
    cmap=CMAP_S2, norm=Normalize(vmin=0, vmax=ccap),
    alpha=0.90, linewidths=0.25
)

# highlight triple
hot = df_sc[df_sc["typology"] == 4]
axC.scatter(
    hot["grid_L_per_kwh_wavg"], hot["stress_wavg"],
    s=mw_size_sqrt(hot["mw_sum"].values, smin=50, smax=320),
    facecolors="none", edgecolors=OUTLINE, linewidths=1.8, zorder=6
)

axC.axhline(STRESS_THR, linewidth=0.9, alpha=0.85)
axC.text(axC.get_xlim()[0], STRESS_THR + 0.08, "high stress ≥ 3", fontsize=9, va="bottom")

# annotate top 5 scope2
lab = df_sc.sort_values("s2_gl_base", ascending=False).head(TOP5_BA)
for _, r in lab.iterrows():
    axC.text(r["grid_L_per_kwh_wavg"], r["stress_wavg"], f" {r[BA_KEY]}",
             fontsize=9, weight="bold", va="center", ha="left")

axC.set_xlabel("BA grid water intensity (L/kWh)")
axC.set_ylabel("Mean water-stress exposure (Aqueduct 0–5)")
axC.set_ylim(-0.1, 5.1)
axC.grid(True, linewidth=0.35, alpha=0.30)

# bubble-size legend OUTSIDE (right margin)
mw_refs = np.array([df_sc["mw_sum"].quantile(q) for q in [0.25, 0.5, 0.9]])
mw_refs = np.unique(np.clip(mw_refs, 1, None).astype(int))
if len(mw_refs) < 3:
    mw_refs = np.array([200, 600, 1500])

handles = [
    Line2D([0], [0], marker='o', color='none',
           markerfacecolor='none', markeredgecolor='black',
           markersize=np.sqrt(mw_size_sqrt([m], smin=18, smax=240)[0])/1.35,
           label=f"{int(m)} MW")
    for m in mw_refs[:3]
]
legC = axC.legend(handles=handles, title="Hyperscale capacity", loc="upper left",
                  bbox_to_anchor=(1.01, 1.00), frameon=True,
                  fontsize=8, title_fontsize=9, borderpad=0.6)
legC.get_frame().set_linewidth(0.6)

# colorbar OUTSIDE (right margin)
caxC = inset_axes(axC, width="1.8%", height="68%", loc="center left",
                  bbox_to_anchor=(1.18, 0.12, 1, 1), bbox_transform=axC.transAxes, borderpad=0)
cbC = fig.colorbar(sc, cax=caxC)
cbC.set_label("Scope 2 water (GL/yr), winsorized", fontsize=9)
cbC.ax.tick_params(labelsize=8)
cbC.formatter = ticker.FuncFormatter(fmt_gl)
cbC.update_ticks()

# -----------------------------
# Title + save
# -----------------------------
fig.suptitle(
    "Figure 2 | Why Scope 1 and Scope 2 hotspots differ: maps (a–b) and drivers (c)",
    fontsize=14, y=0.985
)

plt.savefig(OUT_PNG, dpi=DPI, bbox_inches="tight")
plt.savefig(OUT_PDF, dpi=DPI, bbox_inches="tight")
plt.show()

print(f"Saved: {OUT_PNG} and {OUT_PDF}")

In [ ]:
# FIGURE A: Two maps side-by-side (BA typology | Basin Scope1)
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, ListedColormap, BoundaryNorm
from matplotlib.cm import ScalarMappable
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from pyproj import CRS

# ---- CONFIG ----
FIGSIZE = (14, 6.6)
DPI = 300
LCC_CONUS = CRS.from_proj4(
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 +datum=WGS84 +units=m +no_defs"
)

BA_KEY = "subregion_id"        # on gdf_ba
BASIN_KEY_POLY = "PFAF_ID"     # on gdf_basin
BASIN_KEY_SITE = "pfaf_id"     # on gdf_sites
MW_COL = "current_mw"
S1_M3_COL = "s1_m3_mid"
S2_M3_COL = "s2_m3_mid"
GRID_COL_SITE = "grid_L_per_kwh"
FOSSIL_COL_SITE = "share_fossil"
STRESS_COL_BASIN = "bws_01_score"
STRESS_COL_SITE = "bws_01_score"

TOP5_BA = 5
TOP15_BASINS = 15
PCTL_S2 = 0.75
STRESS_THR = 3.0
WIN_P = 0.98

# Style knobs
STATE_BORDER = "#111111"; STATE_LW = 0.55
BA_EDGE = "#dcdcdc"; BA_EDGE_LW = 0.20
BAS_EDGE = "#e6e6e6"; BAS_EDGE_LW = 0.08
OUTLINE = "#7a0177"   # magenta-ish outline for emphasis
OUTLINE_LW = 1.6
STRESS_EDGE = "#444444"; STRESS_EDGE_LW = 0.35
STRESS_HATCH = "///"

CMAP_S1 = "viridis"
CMAP_S2 = "cividis"

TYPO_COLORS = ListedColormap([
    "#f2f2f2", "#c6dbef", "#9ecae1", "#fdd0a2", "#fb6a4a"
])
TYPO_NORM = BoundaryNorm([-0.5,0.5,1.5,2.5,3.5,4.5], TYPO_COLORS.N)

# ---- helpers ----
def to_lcc(gdf):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(LCC_CONUS)

def winsorize(s, p=WIN_P):
    v = pd.Series(s).replace([np.inf, -np.inf], np.nan).fillna(0)
    cap = float(v.quantile(p))
    return np.clip(v, 0, cap), cap

# ---- load from CFG (you already have these) ----
gdf_sites = CFG["gdf_sites"].copy()
gdf_ba = CFG["gdf_ba"].copy()
gdf_basin = CFG["gdf_basin"].copy()
states48_g = states48.copy()

# normalize keys & project to LCC
gdf_sites[BA_KEY] = gdf_sites[BA_KEY].astype(str)
gdf_sites[BASIN_KEY_SITE] = gdf_sites[BASIN_KEY_SITE].astype(str)
gdf_ba[BA_KEY] = gdf_ba[BA_KEY].astype(str)
gdf_basin[BASIN_KEY_POLY] = gdf_basin[BASIN_KEY_POLY].astype(str)

gdf_sites = to_lcc(gdf_sites)
gdf_ba = to_lcc(gdf_ba)
gdf_basin = to_lcc(gdf_basin)
states48_l = to_lcc(states48_g)

# clip to CONUS
conus_geom = states48_l.unary_union
gdf_sites = gdf_sites[gdf_sites.intersects(conus_geom)].copy()
gdf_ba = gdf_ba[gdf_ba.intersects(conus_geom)].copy()
gdf_basin = gdf_basin[gdf_basin.intersects(conus_geom)].copy()

minx, miny, maxx, maxy = states48_l.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

# ---- BA aggregates (for typology & drivers) ----
def wavg(series, weight):
    series = np.asarray(series, dtype=float)
    weight = np.asarray(weight, dtype=float)
    mask = np.isfinite(series) & np.isfinite(weight) & (weight > 0)
    if mask.sum() == 0:
        return np.nan
    return float((series[mask] * weight[mask]).sum() / weight[mask].sum())

ba_mw = gdf_sites.groupby(BA_KEY)[MW_COL].sum().rename("mw_sum")
ba_s2_gl = (gdf_sites.groupby(BA_KEY)[S2_M3_COL].sum() / 1e6).rename("s2_gl_base")
ba_stress = gdf_sites.groupby(BA_KEY).apply(lambda d: wavg(d[STRESS_COL_SITE], d[MW_COL])).rename("stress_wavg")
ba_fossil = gdf_sites.groupby(BA_KEY).apply(lambda d: wavg(d[FOSSIL_COL_SITE], d[MW_COL])).rename("fossil_wavg")
ba_grid = gdf_sites.groupby(BA_KEY).apply(lambda d: wavg(d[GRID_COL_SITE], d[MW_COL])).rename("grid_L_per_kwh_wavg")

ba_stats = pd.concat([ba_mw, ba_s2_gl, ba_stress, ba_fossil, ba_grid], axis=1).reset_index()
gdf_ba2 = gdf_ba.merge(ba_stats, on=BA_KEY, how="left").fillna(0.0)

s2_thr = float(gdf_ba2["s2_gl_base"].quantile(PCTL_S2))
gdf_ba2["_is_high_s2"] = gdf_ba2["s2_gl_base"] >= s2_thr
gdf_ba2["_is_fossil"]  = gdf_ba2["fossil_wavg"] > 0.5
gdf_ba2["_is_stress"]  = gdf_ba2["stress_wavg"] >= STRESS_THR

def typology(row):
    hs2, fos, st = row["_is_high_s2"], row["_is_fossil"], row["_is_stress"]
    if hs2 and fos and st: return 4
    if hs2 and fos:        return 2
    if hs2 and st:         return 3
    if hs2:                return 1
    return 0
gdf_ba2["typology"] = gdf_ba2.apply(typology, axis=1)
top5_ba_ids = gdf_ba2.sort_values("s2_gl_base", ascending=False).head(TOP5_BA)[BA_KEY].tolist()
gdf_ba2["_top5"] = gdf_ba2[BA_KEY].isin(set(top5_ba_ids))

# ---- Basin aggregates (Scope 1) ----
bas_s1_gl = (gdf_sites.groupby(BASIN_KEY_SITE)[S1_M3_COL].sum() / 1e6).rename("s1_gl_base")
gdf_basin2 = gdf_basin.merge(bas_s1_gl.reset_index().rename(columns={BASIN_KEY_SITE: BASIN_KEY_POLY}), on=BASIN_KEY_POLY, how="left")
gdf_basin2["s1_gl_base"] = gdf_basin2["s1_gl_base"].fillna(0.0)
top15_basin_ids = gdf_basin2.sort_values("s1_gl_base", ascending=False).head(TOP15_BASINS)[BASIN_KEY_POLY].tolist()
gdf_basin2["_top15"] = gdf_basin2[BASIN_KEY_POLY].isin(set(top15_basin_ids))
gdf_basin2["_stress_hi"] = gdf_basin2[STRESS_COL_BASIN] >= STRESS_THR

gdf_basin2["s1_gl_w"], s1_cap = winsorize(gdf_basin2["s1_gl_base"], p=WIN_P)
norm_s1 = Normalize(vmin=0.0, vmax=max(1e-6, s1_cap))

# ---- make figure (two maps) ----
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE, dpi=DPI, constrained_layout=True)
axA, axB = axes

# a) BA typology
axA.set_title(f"a. Scope 2 hotspot typology (base) — top {TOP5_BA} outlined", fontsize=11, loc="left")
gdf_ba2.plot(ax=axA, column="typology", cmap=TYPO_COLORS, norm=TYPO_NORM,
             edgecolor=BA_EDGE, linewidth=BA_EDGE_LW, zorder=2)
gdf_ba2[gdf_ba2["_top5"]].boundary.plot(ax=axA, color=OUTLINE, linewidth=OUTLINE_LW, zorder=6)
states48_l.boundary.plot(ax=axA, color=STATE_BORDER, linewidth=STATE_LW, zorder=10)
axA.set_axis_off(); axA.set_xlim(minx - padx, maxx + padx); axA.set_ylim(miny - pady, maxy + pady); axA.set_aspect("equal")

# legend for typology (outside to the right of axA)
legend_elements = [
    Patch(facecolor=TYPO_COLORS(0), edgecolor="#999999", label="Not high Scope 2"),
    Patch(facecolor=TYPO_COLORS(1), edgecolor="#999999", label=f"High Scope 2 (≥ P{int(PCTL_S2*100)})"),
    Patch(facecolor=TYPO_COLORS(2), edgecolor="#999999", label="High Scope 2 + fossil-heavy"),
    Patch(facecolor=TYPO_COLORS(3), edgecolor="#999999", label="High Scope 2 + high stress"),
    Patch(facecolor=TYPO_COLORS(4), edgecolor="#999999", label="Scope 2 hotspot (triple)"),
    Line2D([0],[0], color=OUTLINE, lw=OUTLINE_LW, label=f"Top {TOP5_BA} by Scope 2")
]
axA.legend(handles=legend_elements, frameon=True, fontsize=8, loc="lower left",
           bbox_to_anchor=(1.02, 0.02))

# b) Basin scope1
axB.set_title(f"b. Scope 1 water by basin (base) — top {TOP15_BASINS} outlined; hatched = high stress (≥{STRESS_THR})", fontsize=11, loc="left")
gdf_basin2.plot(ax=axB, column="s1_gl_w", cmap=CMAP_S1, norm=norm_s1, edgecolor=BAS_EDGE, linewidth=BAS_EDGE_LW, zorder=2)
# hatched high-stress basins
gdf_basin2[gdf_basin2["_stress_hi"]].plot(ax=axB, facecolor="none", edgecolor=STRESS_EDGE, linewidth=STRESS_EDGE_LW, hatch=STRESS_HATCH, zorder=4)
# outlined top15
gdf_basin2[gdf_basin2["_top15"]].boundary.plot(ax=axB, color=OUTLINE, linewidth=OUTLINE_LW, zorder=6)
states48_l.boundary.plot(ax=axB, color=STATE_BORDER, linewidth=STATE_LW, zorder=10)
axB.set_axis_off(); axB.set_xlim(minx - padx, maxx + padx); axB.set_ylim(miny - pady, maxy + pady); axB.set_aspect("equal")

# horizontal colorbar under axB
sm = ScalarMappable(norm=norm_s1, cmap=CMAP_S1); sm.set_array([])
cax = inset_axes(axB, width="72%", height="6%", loc="lower center", borderpad=1.0)
cb = fig.colorbar(sm, cax=cax, orientation="horizontal")
cb.set_label("Scope 1 water (GL/yr), winsorized", fontsize=9)
cb.ax.tick_params(labelsize=8)

# smaller legend for stress + outline to right of axB
legB = [
    Patch(facecolor="white", edgecolor=STRESS_EDGE, hatch=STRESS_HATCH, label=f"High stress (Aqueduct ≥ {STRESS_THR})"),
    Line2D([0],[0], color=OUTLINE, lw=OUTLINE_LW, label=f"Top {TOP15_BASINS} basins by Scope 1")
]
axB.legend(handles=legB, frameon=True, fontsize=8, loc="lower left", bbox_to_anchor=(1.02, 0.02))

fig.suptitle("Where hyperscale water consumption concentrates (Scope 2 typology and Scope 1 basins)", fontsize=13, y=0.995)
plt.savefig("FigA_two_maps_scope1_scope2.png", dpi=DPI, bbox_inches="tight")
plt.show()

In [ ]:
# FIGURE B: Scatter (BA grid intensity vs mean stress) with bubble size = MW, color = Scope2 water
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

FIGSIZE = (10.5, 5.2)
DPI = 300

# Use gdf_ba2 produced above (if run the previous cell). If not, recreate BA aggregates as above.
df_sc = gdf_ba2.copy()
# winsorize scope2 for color
df_sc["_s2_gl_w"], cap_s2 = winsorize(df_sc["s2_gl_base"], p=WIN_P)

# marker sizes (sqrt scaling)
def marker_sizes(mw, smin=30, smax=260):
    mw = np.array(mw, dtype=float)
    if mw.max() <= 0:
        return np.full_like(mw, (smin+smax)/2)
    r = np.sqrt(mw / mw.max())
    return smin + (smax - smin) * r

sizes = marker_sizes(df_sc["mw_sum"].values)

fig, ax = plt.subplots(1, 1, figsize=FIGSIZE, dpi=DPI)
sc = ax.scatter(df_sc["grid_L_per_kwh_wavg"], df_sc["stress_wavg"],
                s=sizes, c=df_sc["_s2_gl_w"], cmap=CMAP_S2, norm=plt.Normalize(vmin=0, vmax=cap_s2),
                edgecolors="k", linewidths=0.2, alpha=0.9, zorder=3)

ax.axhline(STRESS_THR, color="gray", linewidth=0.8, linestyle="--", alpha=0.6)
ax.text(ax.get_xlim()[0], STRESS_THR + 0.08, "high stress ≥ 3", fontsize=9, va="bottom")

# annotate top 5 by scope2
top5 = df_sc.sort_values("s2_gl_base", ascending=False).head(TOP5_BA)
for _, r in top5.iterrows():
    ax.text(r["grid_L_per_kwh_wavg"], r["stress_wavg"], f" {r[BA_KEY]}", fontsize=9, weight="bold", va="center")

ax.set_xlabel("BA grid water intensity (L/kWh)")
ax.set_ylabel("Mean water-stress exposure (Aqueduct 0–5)")
ax.set_ylim(-0.1, 5.1)
ax.grid(True, alpha=0.25, linewidth=0.35)

# colorbar (right)
cax = inset_axes(ax, width="2.2%", height="60%", loc="center left", bbox_to_anchor=(1.05, 0.12, 1, 1), bbox_transform=ax.transAxes, borderpad=0)
cb = plt.colorbar(ScalarMappable(cmap=CMAP_S2, norm=plt.Normalize(vmin=0, vmax=cap_s2)), cax=cax)
cb.set_label("Scope 2 water (GL/yr), winsorized", fontsize=9)
cb.ax.tick_params(labelsize=8)

# bubble size legend (right margin)
# pick three reference MW values
refs = [int(df_sc["mw_sum"].quantile(q)) for q in [0.25, 0.5, 0.9]]
refs = [r for r in refs if r > 0]
if len(refs) < 3:
    refs = [200, 600, 1500]
handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=np.sqrt(marker_sizes([r])[0])/1.4, label=f"{r} MW", markeredgecolor='k') for r in refs[:3]]
ax.legend(handles=handles, title="Hyperscale capacity", loc='upper left', bbox_to_anchor=(1.02, 0.95), frameon=True, fontsize=9)

fig.suptitle("Drivers of Scope 2 hotspots: BA grid intensity vs mean stress (bubble size = MW)", fontsize=13, y=0.98)
plt.savefig("FigB_scatter_drivers.png", dpi=DPI, bbox_inches="tight")
plt.show()

In [ ]:
!pip install contextily xyzservices

In [ ]:
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

def plot_nytish_polygons(
    gdf_poly,
    states48_l,
    col,
    *,
    title="",
    cmap="viridis",
    vmin=None,
    vmax=None,
    alpha=0.55,
    edgecolor="white",
    lw=0.35,
    outline_gdf=None,
    outline_color="#b30000",
    outline_lw=1.4,
    basemap="terrain",   # "terrain" | "light"
    add_colorbar=True,
    cbar_label="",
    figsize=(7.2, 4.6),
    dpi=250
):
    """
    NYT-ish map: basemap + translucent thematic polygons + crisp boundaries.
    Assumes inputs already in *Web Mercator* (EPSG:3857) or projects them.
    """
    # Ensure Web Mercator for basemap tiles
    if gdf_poly.crs is None:
        raise ValueError("gdf_poly needs a CRS")
    if gdf_poly.crs.to_epsg() != 3857:
        gdf_poly = gdf_poly.to_crs(3857)
    if states48_l.crs.to_epsg() != 3857:
        states48_wm = states48_l.to_crs(3857)
    else:
        states48_wm = states48_l

    if outline_gdf is not None:
        outline_wm = outline_gdf.to_crs(3857) if outline_gdf.crs.to_epsg() != 3857 else outline_gdf
    else:
        outline_wm = None

    # bounds
    minx, miny, maxx, maxy = states48_wm.total_bounds
    padx = 0.02 * (maxx - minx)
    pady = 0.02 * (maxy - miny)

    # normalization
    vals = gdf_poly[col].replace([np.inf, -np.inf], np.nan).fillna(0)
    if vmin is None: vmin = float(vals.min())
    if vmax is None: vmax = float(vals.max()) if float(vals.max()) > float(vals.min()) else float(vals.min()) + 1.0
    norm = Normalize(vmin=vmin, vmax=vmax)

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

    # Basemap tiles (pick the vibe)
    if basemap == "terrain":
        source = cx.providers.Stadia.StamenTerrainBackground
    else:
        source = cx.providers.CartoDB.PositronNoLabels

    cx.add_basemap(ax, source=source, zoom="auto", attribution=False)

    # Thematic layer (translucent)
    gdf_poly.plot(
        ax=ax, column=col, cmap=cmap, norm=norm,
        alpha=alpha, edgecolor=edgecolor, linewidth=lw, zorder=5
    )

    # State borders on top (NYT-ish: thin, light)
    states48_wm.boundary.plot(ax=ax, color="white", linewidth=0.9, alpha=0.9, zorder=8)
    states48_wm.boundary.plot(ax=ax, color="#2b2b2b", linewidth=0.25, alpha=0.35, zorder=9)

    # Optional outline (top basins, top BAs, etc.)
    if outline_wm is not None:
        outline_wm.boundary.plot(ax=ax, color=outline_color, linewidth=outline_lw, zorder=12)

    ax.set_title(title, loc="left", fontsize=11)
    ax.set_xlim(minx - padx, maxx + padx)
    ax.set_ylim(miny - pady, maxy + pady)
    ax.set_axis_off()
    ax.set_aspect("equal")

    # Compact horizontal colorbar under map
    if add_colorbar:
        sm = ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cax = inset_axes(ax, width="68%", height="6%", loc="lower center", borderpad=1.2)
        cb = fig.colorbar(sm, cax=cax, orientation="horizontal")
        cb.set_label(cbar_label, fontsize=9)
        cb.ax.tick_params(labelsize=8)

    return fig, ax

In [ ]:
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

def plot_nytish_polygons(
    gdf_poly,
    states48_l,
    col,
    *,
    title="",
    cmap="viridis",
    vmin=None,
    vmax=None,
    alpha=0.55,
    edgecolor="white",
    lw=0.35,
    outline_gdf=None,
    outline_color="#b30000",
    outline_lw=1.4,
    basemap="terrain",   # "terrain" | "light"
    add_colorbar=True,
    cbar_label="",
    figsize=(7.2, 4.6),
    dpi=250
):
    """
    NYT-ish map: basemap + translucent thematic polygons + crisp boundaries.
    Assumes inputs already in *Web Mercator* (EPSG:3857) or projects them.
    """
    # Ensure Web Mercator for basemap tiles
    if gdf_poly.crs is None:
        raise ValueError("gdf_poly needs a CRS")
    if gdf_poly.crs.to_epsg() != 3857:
        gdf_poly = gdf_poly.to_crs(3857)
    if states48_l.crs.to_epsg() != 3857:
        states48_wm = states48_l.to_crs(3857)
    else:
        states48_wm = states48_l

    if outline_gdf is not None:
        outline_wm = outline_gdf.to_crs(3857) if outline_gdf.crs.to_epsg() != 3857 else outline_gdf
    else:
        outline_wm = None

    # bounds
    minx, miny, maxx, maxy = states48_wm.total_bounds
    padx = 0.02 * (maxx - minx)
    pady = 0.02 * (maxy - miny)

    # normalization
    vals = gdf_poly[col].replace([np.inf, -np.inf], np.nan).fillna(0)
    if vmin is None: vmin = float(vals.min())
    if vmax is None: vmax = float(vals.max()) if float(vals.max()) > float(vals.min()) else float(vals.min()) + 1.0
    norm = Normalize(vmin=vmin, vmax=vmax)

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

    # Basemap tiles (pick the vibe)
    if basemap == "terrain":
        source = cx.providers.Stadia.StamenTerrainBackground
    else:
        source = cx.providers.CartoDB.PositronNoLabels

    cx.add_basemap(ax, source=source, zoom="auto", attribution=False)

    # Thematic layer (translucent)
    gdf_poly.plot(
        ax=ax, column=col, cmap=cmap, norm=norm,
        alpha=alpha, edgecolor=edgecolor, linewidth=lw, zorder=5
    )

    # State borders on top (NYT-ish: thin, light)
    states48_wm.boundary.plot(ax=ax, color="white", linewidth=0.9, alpha=0.9, zorder=8)
    states48_wm.boundary.plot(ax=ax, color="#2b2b2b", linewidth=0.25, alpha=0.35, zorder=9)

    # Optional outline (top basins, top BAs, etc.)
    if outline_wm is not None:
        outline_wm.boundary.plot(ax=ax, color=outline_color, linewidth=outline_lw, zorder=12)

    ax.set_title(title, loc="left", fontsize=11)
    ax.set_xlim(minx - padx, maxx + padx)
    ax.set_ylim(miny - pady, maxy + pady)
    ax.set_axis_off()
    ax.set_aspect("equal")

    # Compact horizontal colorbar under map
    if add_colorbar:
        sm = ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cax = inset_axes(ax, width="68%", height="6%", loc="lower center", borderpad=1.2)
        cb = fig.colorbar(sm, cax=cax, orientation="horizontal")
        cb.set_label(cbar_label, fontsize=9)
        cb.ax.tick_params(labelsize=8)

    return fig, ax

## new figures

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import FuncFormatter, MaxNLocator
from pyproj import CRS

# -----------------------------
# CONFIG
# -----------------------------
TOP_BASINS_N = 15
TOP_BA_N     = 5

SITE_BASIN_KEY = "pfaf_id"
BASIN_KEY      = "PFAF_ID"
SITE_BA_KEY    = "subregion_id"
BA_KEY         = "subregion_id"

S1_COL_M3      = "s1_m3_mid"
S2_COL_M3      = "s2_m3_mid"
MW_COL         = "current_mw"
SCARCITY_COL   = "bws_01_score"

CMAP_S1 = "viridis"
CMAP_S2 = "viridis"

EDGE_OTHER = "#cfcfcf"
LW_OTHER   = 0.25
EDGE_TOP   = "#b00000"
LW_TOP     = 1.8
STATE_BORDER = "#111111"
STATE_LW     = 0.65
ALPHA_OTHER  = 0.35
ALPHA_TOP    = 0.95

CAP_PCT = 0.99
DPI = 350

LCC_CONUS = CRS.from_proj4(
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 +datum=WGS84 +units=m +no_defs"
)

def to_lcc(gdf, crs=LCC_CONUS):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(crs)

def cap_norm(series, pct=CAP_PCT):
    s = pd.Series(series).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    vmax = float(s.quantile(pct)) if (s > 0).any() else float(s.max() if len(s) else 1.0)
    vmax = max(vmax, 1e-12)
    return Normalize(vmin=0.0, vmax=vmax), vmax

def m3_to_GL(x):  # 1 GL = 1e6 m3
    return np.asarray(x) / 1e6

# -----------------------------
# Inputs (must exist in your env)
# -----------------------------
assert "gdf_sites" in CFG and CFG["gdf_sites"] is not None
assert "gdf_basin" in CFG and CFG["gdf_basin"] is not None
assert "gdf_ba" in CFG and CFG["gdf_ba"] is not None
assert "states48" in globals()

gdf_sites = to_lcc(CFG["gdf_sites"].copy())
gdf_basin = to_lcc(CFG["gdf_basin"].copy())
gdf_ba    = to_lcc(CFG["gdf_ba"].copy())
states48_l = to_lcc(states48.copy())

conus_geom = states48_l.unary_union
gdf_sites = gdf_sites[gdf_sites.intersects(conus_geom)].copy()
gdf_basin = gdf_basin[gdf_basin.intersects(conus_geom)].copy()
gdf_ba    = gdf_ba[gdf_ba.intersects(conus_geom)].copy()

# Key cleanup
gdf_sites[SITE_BASIN_KEY] = gdf_sites[SITE_BASIN_KEY].astype(str)
gdf_sites[SITE_BA_KEY]    = gdf_sites[SITE_BA_KEY].astype(str)
gdf_basin[BASIN_KEY]      = gdf_basin[BASIN_KEY].astype(str)
gdf_ba[BA_KEY]            = gdf_ba[BA_KEY].astype(str)

# -----------------------------
# Aggregate Scope 1 by basin
# -----------------------------
bas_agg = (
    gdf_sites.groupby(SITE_BASIN_KEY, dropna=False)
    .agg(s1_m3_sum=(S1_COL_M3, "sum"), mw_sum=(MW_COL, "sum"))
    .reset_index()
    .rename(columns={SITE_BASIN_KEY: BASIN_KEY})
)
gdf_basin = gdf_basin.merge(bas_agg, on=BASIN_KEY, how="left")
gdf_basin["s1_m3_sum"] = gdf_basin["s1_m3_sum"].fillna(0.0)
gdf_basin["_is_top"] = gdf_basin[BASIN_KEY].isin(
    gdf_basin.sort_values("s1_m3_sum", ascending=False).head(TOP_BASINS_N)[BASIN_KEY]
)
gdf_basin["_high_stress"] = gdf_basin[SCARCITY_COL].fillna(0.0) >= 3.0

# -----------------------------
# Aggregate Scope 2 by BA
# -----------------------------
ba_agg = (
    gdf_sites.groupby(SITE_BA_KEY, dropna=False)
    .agg(s2_m3_sum=(S2_COL_M3, "sum"), mw_sum=(MW_COL, "sum"))
    .reset_index()
    .rename(columns={SITE_BA_KEY: BA_KEY})
)
gdf_ba = gdf_ba.merge(ba_agg, on=BA_KEY, how="left")
gdf_ba["s2_m3_sum"] = gdf_ba["s2_m3_sum"].fillna(0.0)
gdf_ba["_is_top"] = gdf_ba[BA_KEY].isin(
    gdf_ba.sort_values("s2_m3_sum", ascending=False).head(TOP_BA_N)[BA_KEY]
)

# -----------------------------
# Norms (winsorized)
# -----------------------------
norm_s1, _ = cap_norm(gdf_basin["s1_m3_sum"])
norm_s2, _ = cap_norm(gdf_ba["s2_m3_sum"])

# Common bounds for maps
minx, miny, maxx, maxy = states48_l.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

# -----------------------------
# Panel C values (anchored to table row; low/high computed)
# -----------------------------
nat = pd.read_csv("table_national_water_totals_mid.csv")

base_s1 = float(nat.loc[0, "s1_m3_total"])
base_s2 = float(nat.loc[0, "s2_m3_total"])
base_tt = float(nat.loc[0, "total_m3_total"])

u_mid   = float(nat.loc[0, "utilization_u"])
pue_mid = float(nat.loc[0, "PUE"])
wue_mid = float(nat.loc[0, "WUE_L_per_kWh_IT"])

u_low, u_high     = 0.55, 0.85
pue_low, pue_high = 1.15, 1.40
wue_low, wue_high = 0.2, 1.5

s2_low  = base_s2 * (u_low / u_mid)
s2_high = base_s2 * (u_high / u_mid)
s1_low  = base_s1 * (u_low/u_mid) * ((wue_low/pue_low) / (wue_mid/pue_mid))
s1_high = base_s1 * (u_high/u_mid) * ((wue_high/pue_high) / (wue_mid/pue_mid))
tt_low  = s1_low + s2_low
tt_high = s1_high + s2_high

vals_GL = {
    "Scope 1 (direct cooling)": (m3_to_GL(s1_low), m3_to_GL(base_s1), m3_to_GL(s1_high)),
    "Scope 2 (electricity-related)": (m3_to_GL(s2_low), m3_to_GL(base_s2), m3_to_GL(s2_high)),
    "Total (Scope 1 + 2)": (m3_to_GL(tt_low), m3_to_GL(base_tt), m3_to_GL(tt_high)),
}

# =====================================================================
# FIGURE A: MAPS ONLY (Panels a-b) + aligned colorbars
# =====================================================================
FIG_W_MAPS, FIG_H_MAPS = 17.5, 7.6
fig_maps = plt.figure(figsize=(FIG_W_MAPS, FIG_H_MAPS), constrained_layout=False)

gs = fig_maps.add_gridspec(
    nrows=2, ncols=2,
    height_ratios=[1.0, 0.07],
    width_ratios=[1.0, 1.0],
    left=0.045, right=0.985, top=0.92, bottom=0.10,
    wspace=0.06, hspace=0.10
)

axA  = fig_maps.add_subplot(gs[0, 0])
axB  = fig_maps.add_subplot(gs[0, 1])
caxA = fig_maps.add_subplot(gs[1, 0])
caxB = fig_maps.add_subplot(gs[1, 1])

# Panel a
gdf_basin.plot(ax=axA, column="s1_m3_sum", cmap=CMAP_S1, norm=norm_s1,
              alpha=ALPHA_OTHER, edgecolor=EDGE_OTHER, linewidth=LW_OTHER, zorder=1)
gdf_basin[gdf_basin["_high_stress"]].plot(
    ax=axA, facecolor="none", edgecolor="none", hatch="////", linewidth=0, zorder=2
)
gdf_basin[gdf_basin["_is_top"]].plot(ax=axA, column="s1_m3_sum", cmap=CMAP_S1, norm=norm_s1,
                                    alpha=ALPHA_TOP, edgecolor=EDGE_TOP, linewidth=LW_TOP, zorder=4)
states48_l.boundary.plot(ax=axA, color=STATE_BORDER, linewidth=STATE_LW, zorder=10)

axA.set_title(
    f"a. Scope 1 water consumption by basin (Base)\nTop {TOP_BASINS_N} outlined; hatched = high water stress (Aqueduct ≥3)",
    fontsize=11
)
axA.set_axis_off()
axA.set_xlim(minx - padx, maxx + padx); axA.set_ylim(miny - pady, maxy + pady)
axA.set_aspect("equal")

# Panel b
gdf_ba.plot(ax=axB, column="s2_m3_sum", cmap=CMAP_S2, norm=norm_s2,
            alpha=ALPHA_OTHER, edgecolor=EDGE_OTHER, linewidth=LW_OTHER, zorder=1)
gdf_ba[gdf_ba["_is_top"]].plot(ax=axB, column="s2_m3_sum", cmap=CMAP_S2, norm=norm_s2,
                               alpha=ALPHA_TOP, edgecolor=EDGE_TOP, linewidth=LW_TOP, zorder=4)
states48_l.boundary.plot(ax=axB, color=STATE_BORDER, linewidth=STATE_LW, zorder=10)

axB.set_title(
    f"b. Scope 2 water consumption by balancing authority (Base)\nTop {TOP_BA_N} outlined",
    fontsize=11
)
axB.set_axis_off()
axB.set_xlim(minx - padx, maxx + padx); axB.set_ylim(miny - pady, maxy + pady)
axB.set_aspect("equal")

# Colorbars (ticks labeled in GL)
smA = ScalarMappable(norm=norm_s1, cmap=CMAP_S1); smA.set_array([])
cbA = fig_maps.colorbar(smA, cax=caxA, orientation="horizontal")
cbA.set_label("Scope 1 water consumption (GL/yr), winsorized", fontsize=11)
cbA.ax.tick_params(labelsize=9)
cbA.locator = MaxNLocator(nbins=5)
cbA.formatter = FuncFormatter(lambda t, pos: f"{t/1e6:.0f}")
cbA.update_ticks()

smB = ScalarMappable(norm=norm_s2, cmap=CMAP_S2); smB.set_array([])
cbB = fig_maps.colorbar(smB, cax=caxB, orientation="horizontal")
cbB.set_label("Scope 2 water consumption (GL/yr), winsorized", fontsize=11)
cbB.ax.tick_params(labelsize=9)
cbB.locator = MaxNLocator(nbins=5)
cbB.formatter = FuncFormatter(lambda t, pos: f"{t/1e6:.0f}")
cbB.update_ticks()

for cax in (caxA, caxB):
    for sp in cax.spines.values():
        sp.set_visible(False)

out_pdf_maps = "Fig1a_maps_scope1_scope2.pdf"
fig_maps.savefig(out_pdf_maps, dpi=DPI, bbox_inches="tight")
plt.show()
print(f"Saved: {out_pdf_maps}")

# =====================================================================
# FIGURE B: BARS ONLY (Panel c) — scenario intervals
# =====================================================================
FIG_W_BARS, FIG_H_BARS = 12.8, 3.6
fig_bars = plt.figure(figsize=(FIG_W_BARS, FIG_H_BARS), constrained_layout=False)
axC = fig_bars.add_axes([0.08, 0.22, 0.86, 0.68])  # generous right side built-in

axC.set_title(
    "Figure 1c. National water consumption totals — scenario intervals (Eff / Base / High)",
    fontsize=12, loc="center"
)

labels = list(vals_GL.keys())
y = np.arange(len(labels))[::-1]

all_vals = np.array([v for trip in vals_GL.values() for v in trip], dtype=float)
xmin, xmax = float(all_vals.min()), float(all_vals.max())
pad = 0.10 * (xmax - xmin if xmax > xmin else 1.0)

axC.set_ylim(-0.8, len(labels) - 0.2)
axC.set_xlim(xmin - pad, xmax + pad*0.55)  # room for printed triplets
axC.grid(axis="x", alpha=0.25)
axC.set_yticks(y)
axC.set_yticklabels(labels, fontsize=10)
axC.tick_params(axis="x", labelsize=9)
axC.set_xlabel("GL per year (GL/yr)", fontsize=10)

def fmt_trip(lo, mid, hi):
    return f"{lo:.0f} / {mid:.0f} / {hi:.0f}"

for i, lab in enumerate(labels):
    lo, mid, hi = vals_GL[lab]
    yy = y[i]
    axC.plot([lo, hi], [yy, yy], lw=6, solid_capstyle="round")
    axC.plot([mid], [yy], marker="o", ms=7)

    axC.text(xmax + pad*0.18, yy, fmt_trip(lo, mid, hi), fontsize=9, va="center", ha="left")

    axC.text(lo,  yy + 0.18, "Eff",  fontsize=8, ha="center", va="bottom")
    axC.text(mid, yy + 0.18, "Base", fontsize=8, ha="center", va="bottom", fontweight="bold")
    axC.text(hi,  yy + 0.18, "High", fontsize=8, ha="center", va="bottom")

axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)
axC.spines["left"].set_visible(False)

out_pdf_bars = "Fig1c_national_totals_scenarios.pdf"
fig_bars.savefig(out_pdf_bars, dpi=DPI, bbox_inches="tight")
plt.show()
print(f"Saved: {out_pdf_bars}")

In [ ]:
#environment_manifest(globals())

In [ ]:
# === FIGURE 2: Hotspot typologies consistent with paper (Scope 1 basins; Scope 2 BAs) ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pyproj import CRS

# -----------------------------
# CONFIG
# -----------------------------
SITE_BASIN_KEY = "pfaf_id"
BASIN_KEY      = "PFAF_ID"

SITE_BA_KEY    = "subregion_id"
BA_KEY         = "subregion_id"

S1_COL_M3      = "s1_m3_mid"
S2_COL_M3      = "s2_m3_mid"
MW_COL         = "current_mw"

SCARCITY_COL   = "bws_01_score"  # Aqueduct baseline stress score (0-5)

# Hotspot thresholds (per paper)
S1_HIGH_QUANTILE = 0.50  # median among basins with hyperscale activity
S2_HIGH_QUANTILE = 0.75  # P75 among BAs with hyperscale activity
STRESS_THRESHOLD = 3.0
FOSSIL_THRESHOLD = 0.50  # coal+gas share > 0.5

# Figure style
FIG_W, FIG_H = 16.5, 7.8
DPI = 350

# Projection
LCC_CONUS = CRS.from_proj4(
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 +datum=WGS84 +units=m +no_defs"
)

def to_lcc(gdf, crs=LCC_CONUS):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(crs)

def _safe_geom(gdf):
    # Try to repair invalid geometries (common in polygon layers)
    gdf = gdf.copy()
    try:
        gdf["geometry"] = gdf.geometry.make_valid()
    except Exception:
        # fallback
        gdf["geometry"] = gdf.geometry.buffer(0)
    return gdf

def resolve_fossil_share_col(gdf_ba):
    # Add your exact column name here if different
    FOSSIL_SHARE_CANDIDATES = [
        "coal_gas_share",
        "fossil_share",
        "fossil_frac",
        "fossil_pct",
        "coal_gas_frac",
        "coal_gas_pct",
        "share_coal_gas",
        "coal_gas",
    ]
    cols = set(gdf_ba.columns)
    for c in FOSSIL_SHARE_CANDIDATES:
        if c in cols:
            return c
    raise KeyError(
        "Could not find a coal+gas fossil share column in gdf_ba. "
        "Add your column name to FOSSIL_SHARE_CANDIDATES in resolve_fossil_share_col(). "
        f"Available columns: {sorted(list(cols))[:50]} ..."
    )

# -----------------------------
# Inputs (must exist in your env)
# -----------------------------
assert "gdf_sites" in CFG and CFG["gdf_sites"] is not None
assert "gdf_basin" in CFG and CFG["gdf_basin"] is not None
assert "gdf_ba" in CFG and CFG["gdf_ba"] is not None
assert "states48" in globals()

gdf_sites = to_lcc(CFG["gdf_sites"].copy())
gdf_basin = to_lcc(CFG["gdf_basin"].copy())
gdf_ba    = to_lcc(CFG["gdf_ba"].copy())
states48_l = to_lcc(states48.copy())

gdf_basin = _safe_geom(gdf_basin)
gdf_ba    = _safe_geom(gdf_ba)
states48_l = _safe_geom(states48_l)

# CONUS clip by intersection with states union
conus_geom = states48_l.unary_union
gdf_sites = gdf_sites[gdf_sites.intersects(conus_geom)].copy()
gdf_basin = gdf_basin[gdf_basin.intersects(conus_geom)].copy()
gdf_ba    = gdf_ba[gdf_ba.intersects(conus_geom)].copy()

# Key cleanup
gdf_sites[SITE_BASIN_KEY] = gdf_sites[SITE_BASIN_KEY].astype(str)
gdf_sites[SITE_BA_KEY]    = gdf_sites[SITE_BA_KEY].astype(str)
gdf_basin[BASIN_KEY]      = gdf_basin[BASIN_KEY].astype(str)
gdf_ba[BA_KEY]            = gdf_ba[BA_KEY].astype(str)

# Bounds
minx, miny, maxx, maxy = states48_l.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

# =========================================================
# PANEL A: Scope 1 basin hotspot typology (4-class)
# =========================================================
# Aggregate Scope 1 to basins
bas_agg = (
    gdf_sites.groupby(SITE_BASIN_KEY, dropna=False)
    .agg(s1_m3_sum=(S1_COL_M3, "sum"),
         mw_sum=(MW_COL, "sum"),
         n_sites=(MW_COL, "size"))
    .reset_index()
    .rename(columns={SITE_BASIN_KEY: BASIN_KEY})
)
gdf_basin = gdf_basin.merge(bas_agg, on=BASIN_KEY, how="left")
gdf_basin[["s1_m3_sum","mw_sum","n_sites"]] = gdf_basin[["s1_m3_sum","mw_sum","n_sites"]].fillna(0.0)

# "Basins with hyperscale activity": at least one site (or MW>0)
active_basins = gdf_basin["n_sites"] > 0

# High Scope 1 threshold among active basins
s1_thr = float(gdf_basin.loc[active_basins, "s1_m3_sum"].quantile(S1_HIGH_QUANTILE)) if active_basins.any() else 0.0
gdf_basin["_high_s1"] = active_basins & (gdf_basin["s1_m3_sum"] > s1_thr)

# Stress in basin
gdf_basin["_stress_here"] = gdf_basin[SCARCITY_COL].fillna(0.0) >= STRESS_THRESHOLD

# Neighbor-aware stress: stress in basin OR any touching neighbor stressed
# Use spatial index to speed up
sindex = gdf_basin.sindex
stress_neighbor = np.zeros(len(gdf_basin), dtype=bool)

for i, geom in enumerate(gdf_basin.geometry):
    if geom is None or geom.is_empty:
        continue
    # candidates by bbox
    cand_idx = list(sindex.intersection(geom.bounds))
    if not cand_idx:
        continue
    # any touching stressed candidate?
    # (exclude self; require touches)
    for j in cand_idx:
        if j == i:
            continue
        if not gdf_basin.iloc[j]["_stress_here"]:
            continue
        gj = gdf_basin.geometry.iloc[j]
        if gj is None or gj.is_empty:
            continue
        if geom.touches(gj):
            stress_neighbor[i] = True
            break

gdf_basin["_stress_exposed"] = gdf_basin["_stress_here"] | stress_neighbor

# 4-class typology per supplement captions
# neither / stressed only / high s1 only / hotspot
def s1_class(row):
    hs1 = bool(row["_high_s1"])
    st  = bool(row["_stress_exposed"])
    if hs1 and st:
        return "Hotspot (High S1 × Stressed)"
    if hs1 and (not st):
        return "High Scope 1 only"
    if (not hs1) and st:
        return "Stressed only"
    return "Neither"

gdf_basin["_s1_class"] = gdf_basin.apply(s1_class, axis=1)

S1_CLASS_ORDER = [
    "Neither",
    "Stressed only",
    "High Scope 1 only",
    "Hotspot (High S1 × Stressed)"
]
S1_COLORS = {
    "Neither": "#d9d9d9",
    "Stressed only": "#9e9e9e",
    "High Scope 1 only": "#6baed6",
    "Hotspot (High S1 × Stressed)": "#b2182b"
}

# =========================================================
# PANEL B: Scope 2 BA hotspot typology (decision-oriented)
# =========================================================
# Aggregate Scope 2 to BAs
ba_agg = (
    gdf_sites.groupby(SITE_BA_KEY, dropna=False)
    .agg(s2_m3_sum=(S2_COL_M3, "sum"),
         mw_sum=(MW_COL, "sum"),
         n_sites=(MW_COL, "size"),
         mean_stress=(SCARCITY_COL, "mean"))
    .reset_index()
    .rename(columns={SITE_BA_KEY: BA_KEY})
)
gdf_ba = gdf_ba.merge(ba_agg, on=BA_KEY, how="left")
gdf_ba[["s2_m3_sum","mw_sum","n_sites","mean_stress"]] = gdf_ba[["s2_m3_sum","mw_sum","n_sites","mean_stress"]].fillna(0.0)

active_bas = gdf_ba["n_sites"] > 0
s2_thr = float(gdf_ba.loc[active_bas, "s2_m3_sum"].quantile(S2_HIGH_QUANTILE)) if active_bas.any() else 0.0
gdf_ba["_high_s2"] = active_bas & (gdf_ba["s2_m3_sum"] > s2_thr)

# Fossil share column (coal+gas)
FOSSIL_COL = resolve_fossil_share_col(gdf_ba)
gdf_ba["_fossil_heavy"] = gdf_ba[FOSSIL_COL].fillna(0.0) > FOSSIL_THRESHOLD

# Mean stress exposure (mean facility baseline stress within BA)
gdf_ba["_high_stress_mean"] = gdf_ba["mean_stress"].fillna(0.0) >= STRESS_THRESHOLD

# Typology: keep it interpretable (5 classes)
# - Triple intersection = hotspot
# - High S2 + fossil
# - High S2 + stress
# - High S2 only
# - Other / low S2
def s2_class(row):
    hs2 = bool(row["_high_s2"])
    fos = bool(row["_fossil_heavy"])
    st  = bool(row["_high_stress_mean"])
    if hs2 and fos and st:
        return "Hotspot (High S2 × Fossil × Stress)"
    if hs2 and fos and (not st):
        return "High S2 × Fossil"
    if hs2 and (not fos) and st:
        return "High S2 × Stress"
    if hs2 and (not fos) and (not st):
        return "High Scope 2 only"
    return "Other / low Scope 2"

gdf_ba["_s2_class"] = gdf_ba.apply(s2_class, axis=1)

S2_CLASS_ORDER = [
    "Other / low Scope 2",
    "High Scope 2 only",
    "High S2 × Fossil",
    "High S2 × Stress",
    "Hotspot (High S2 × Fossil × Stress)"
]
S2_COLORS = {
    "Other / low Scope 2": "#d9d9d9",
    "High Scope 2 only": "#9ecae1",
    "High S2 × Fossil": "#fdae61",
    "High S2 × Stress": "#abd9e9",
    "Hotspot (High S2 × Fossil × Stress)": "#b2182b"
}

# -----------------------------
# Plot
# -----------------------------
fig, (axA, axB) = plt.subplots(1, 2, figsize=(FIG_W, FIG_H), constrained_layout=True)

# Panel A (basins)
for cls in S1_CLASS_ORDER:
    sub = gdf_basin[gdf_basin["_s1_class"] == cls]
    if len(sub) == 0:
        continue
    sub.plot(
        ax=axA,
        color=S1_COLORS[cls],
        edgecolor="white",
        linewidth=0.20,
        zorder=1
    )

states48_l.boundary.plot(ax=axA, color="#111111", linewidth=0.6, zorder=10)
axA.set_title(
    "a. Scope 1 basin hotspot typology (baseline)\n"
    f"High Scope 1: basin total > P{int(S1_HIGH_QUANTILE*100)} among basins with hyperscale activity; "
    f"Stressed: Aqueduct ≥ {STRESS_THRESHOLD} in basin or touching neighbor",
    fontsize=10
)
axA.set_axis_off()
axA.set_xlim(minx - padx, maxx + padx); axA.set_ylim(miny - pady, maxy + pady)
axA.set_aspect("equal")

handlesA = [Patch(facecolor=S1_COLORS[c], edgecolor="none", label=c) for c in S1_CLASS_ORDER]
axA.legend(handles=handlesA, loc="lower left", frameon=True, fontsize=9, title=None)

# Panel B (BAs)
for cls in S2_CLASS_ORDER:
    sub = gdf_ba[gdf_ba["_s2_class"] == cls]
    if len(sub) == 0:
        continue
    sub.plot(
        ax=axB,
        color=S2_COLORS[cls],
        edgecolor="white",
        linewidth=0.20,
        zorder=1
    )

states48_l.boundary.plot(ax=axB, color="#111111", linewidth=0.6, zorder=10)
axB.set_title(
    "b. Scope 2 BA hotspot typology (baseline)\n"
    f"High Scope 2: BA total > P{int(S2_HIGH_QUANTILE*100)} among BAs with hyperscale activity; "
    f"Fossil-heavy: coal+gas > {int(FOSSIL_THRESHOLD*100)}%; Mean stress ≥ {STRESS_THRESHOLD}",
    fontsize=10
)
axB.set_axis_off()
axB.set_xlim(minx - padx, maxx + padx); axB.set_ylim(miny - pady, maxy + pady)
axB.set_aspect("equal")

handlesB = [Patch(facecolor=S2_COLORS[c], edgecolor="none", label=c) for c in S2_CLASS_ORDER]
axB.legend(handles=handlesB, loc="lower left", frameon=True, fontsize=9, title=None)

# --- Save PDF only + show in notebook
out_pdf = "Fig2_hotspot_typologies_baseline.pdf"
plt.savefig(out_pdf, dpi=DPI, bbox_inches="tight")
plt.show()

print(f"Saved PDF: {out_pdf}")
print(f"Scope 1 high threshold (P50 among active basins): {s1_thr:.3e} m3/yr")
print(f"Scope 2 high threshold (P75 among active BAs):    {s2_thr:.3e} m3/yr")
print(f"Using fossil share column: {FOSSIL_COL}")

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np

def _is_gdf(x):
    return isinstance(x, gpd.GeoDataFrame)

def _pick_first_gdf_from(obj, keys_priority=None, name="obj"):
    """
    obj can be dict-like (CFG) or globals() or locals().
    Returns (key, gdf) or (None, None)
    """
    if keys_priority is None:
        keys_priority = []
    if isinstance(obj, dict):
        # priority keys first
        for k in keys_priority:
            if k in obj and _is_gdf(obj[k]):
                return k, obj[k]
        # then any gdf
        for k, v in obj.items():
            if _is_gdf(v):
                return k, v
    return None, None

def _normalize_colname(s):
    return str(s).strip().lower().replace(" ", "_")

def resolve_column(df, candidates, fuzzy_contains=None, required=True, label=""):
    """
    candidates: exact names to try (case-insensitive)
    fuzzy_contains: list of substrings that must be contained (case-insensitive) as fallback
    """
    cols = list(df.columns)
    cols_norm = {_normalize_colname(c): c for c in cols}  # normalized -> original

    # try exact by normalized
    for c in candidates:
        cn = _normalize_colname(c)
        if cn in cols_norm:
            return cols_norm[cn]

    # try fuzzy contains
    if fuzzy_contains:
        fc = [f.lower() for f in fuzzy_contains]
        for c in cols:
            cl = str(c).lower()
            if all(f in cl for f in fc):
                return c

    if required:
        raise KeyError(
            f"[{label}] Could not resolve column. Tried candidates={candidates}, fuzzy_contains={fuzzy_contains}. "
            f"Available columns (first 60): {cols[:60]}"
        )
    return None

def summarize_gdf(gdf, name):
    print(f"\n=== {name} ===")
    print("type:", type(gdf))
    print("rows:", len(gdf), "cols:", len(gdf.columns))
    print("crs:", gdf.crs)
    print("columns (first 40):", list(gdf.columns)[:40])
    if "geometry" in gdf.columns:
        print("geometry types:", gdf.geometry.geom_type.value_counts().head(5).to_dict())

# ------------------------------------------------------------
# 1) Find your layers robustly
# ------------------------------------------------------------

# Try CFG first if it exists
CFG_exists = "CFG" in globals() and isinstance(globals()["CFG"], dict)
CFG_obj = globals()["CFG"] if CFG_exists else {}

# Candidate keys you *might* be using
SITE_KEYS  = ["gdf_sites", "sites", "facilities", "gdf_facilities", "gdf_site"]
BASIN_KEYS = ["gdf_basin", "basin", "gdf_basins", "hydrobasins", "gdf_hydrobasins"]
BA_KEYS    = ["gdf_ba", "ba", "gdf_bas", "balancing_authority", "gdf_balancing_authority"]
ST_KEYS    = ["states48", "states_48", "states", "us_states", "states_conus", "gdf_states"]

# Pull from CFG if present, otherwise from globals
site_k,  gdf_sites  = _pick_first_gdf_from(CFG_obj, SITE_KEYS,  name="CFG(sites)")
basin_k, gdf_basin  = _pick_first_gdf_from(CFG_obj, BASIN_KEYS, name="CFG(basin)")
ba_k,    gdf_ba     = _pick_first_gdf_from(CFG_obj, BA_KEYS,    name="CFG(ba)")

if gdf_sites is None:
    site_k, gdf_sites = _pick_first_gdf_from(globals(), SITE_KEYS, name="globals(sites)")
if gdf_basin is None:
    basin_k, gdf_basin = _pick_first_gdf_from(globals(), BASIN_KEYS, name="globals(basin)")
if gdf_ba is None:
    ba_k, gdf_ba = _pick_first_gdf_from(globals(), BA_KEYS, name="globals(ba)")

# states48 is often a variable (not in CFG)
states48 = globals().get("states48", None)
if not _is_gdf(states48):
    # try other keys in globals
    for k in ST_KEYS:
        v = globals().get(k, None)
        if _is_gdf(v):
            states48 = v
            break

assert _is_gdf(gdf_sites), "Could not find gdf_sites (GeoDataFrame) in CFG or globals()."
assert _is_gdf(gdf_basin), "Could not find gdf_basin (GeoDataFrame) in CFG or globals()."
assert _is_gdf(gdf_ba),    "Could not find gdf_ba (GeoDataFrame) in CFG or globals()."
assert _is_gdf(states48),  "Could not find states48 (GeoDataFrame) in globals()."

summarize_gdf(gdf_sites,  f"gdf_sites  (found as: {site_k})")
summarize_gdf(gdf_basin,  f"gdf_basin  (found as: {basin_k})")
summarize_gdf(gdf_ba,     f"gdf_ba     (found as: {ba_k})")
summarize_gdf(states48,   "states48")

# ------------------------------------------------------------
# 2) Resolve the key columns automatically (no guessing)
# ------------------------------------------------------------

# Basin join keys
SITE_BASIN_KEY = resolve_column(
    gdf_sites,
    candidates=["pfaf_id", "PFAF_ID", "HYBAS_ID", "hybas_id", "basin_id"],
    fuzzy_contains=["pfaf"],
    required=True,
    label="gdf_sites basin key"
)
BASIN_KEY = resolve_column(
    gdf_basin,
    candidates=["PFAF_ID", "pfaf_id", "HYBAS_ID", "hybas_id", "basin_id"],
    fuzzy_contains=["pfaf"],
    required=True,
    label="gdf_basin basin key"
)

# BA join keys
SITE_BA_KEY = resolve_column(
    gdf_sites,
    candidates=["subregion_id", "SUBREGION_ID", "ba_id", "BA_ID", "egrid_ba", "egrid_subregion"],
    fuzzy_contains=["subregion"],
    required=True,
    label="gdf_sites BA key"
)
BA_KEY = resolve_column(
    gdf_ba,
    candidates=["subregion_id", "SUBREGION_ID", "ba_id", "BA_ID", "egrid_ba", "egrid_subregion"],
    fuzzy_contains=["subregion"],
    required=True,
    label="gdf_ba BA key"
)

# Water and MW columns
S1_COL_M3 = resolve_column(
    gdf_sites,
    candidates=["s1_m3_mid", "s1_m3", "scope1_m3", "W1_m3", "s1_m3_base"],
    fuzzy_contains=["s1", "m3"],
    required=True,
    label="gdf_sites Scope 1 m3"
)
S2_COL_M3 = resolve_column(
    gdf_sites,
    candidates=["s2_m3_mid", "s2_m3", "scope2_m3", "W2_m3", "s2_m3_base"],
    fuzzy_contains=["s2", "m3"],
    required=True,
    label="gdf_sites Scope 2 m3"
)
MW_COL = resolve_column(
    gdf_sites,
    candidates=["current_mw", "mw", "nameplate_mw", "capacity_mw", "P_mw"],
    fuzzy_contains=["mw"],
    required=True,
    label="gdf_sites MW"
)

# Scarcity score
SCARCITY_COL_SITES = resolve_column(
    gdf_sites,
    candidates=["bws_01_score", "baseline_water_stress", "stress_score", "bws_score"],
    fuzzy_contains=["stress"],
    required=False,
    label="gdf_sites stress"
)
SCARCITY_COL_BASIN = resolve_column(
    gdf_basin,
    candidates=["bws_01_score", "baseline_water_stress", "stress_score", "bws_score"],
    fuzzy_contains=["stress"],
    required=False,
    label="gdf_basin stress"
)

print("\n--- RESOLVED COLUMN MAP ---")
print("SITE_BASIN_KEY:", SITE_BASIN_KEY)
print("BASIN_KEY     :", BASIN_KEY)
print("SITE_BA_KEY   :", SITE_BA_KEY)
print("BA_KEY        :", BA_KEY)
print("S1_COL_M3     :", S1_COL_M3)
print("S2_COL_M3     :", S2_COL_M3)
print("MW_COL        :", MW_COL)
print("SCARCITY (sites):", SCARCITY_COL_SITES)
print("SCARCITY (basin):", SCARCITY_COL_BASIN)

print("\nIf any of these are wrong, tell me the correct column names and I’ll hard-wire them.")

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pyproj import CRS

# -----------------------------
# CONFIG (from paper)
# -----------------------------
S1_HIGH_QUANTILE = 0.50   # "above median among active basins"
S2_HIGH_QUANTILE = 0.75   # "top quartile among active BAs"
STRESS_THRESHOLD = 3.0
FOSSIL_THRESHOLD = 0.50   # coal+gas share > 50%

FIG_W, FIG_H = 16.5, 7.8
DPI = 350

LCC_CONUS = CRS.from_proj4(
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 +datum=WGS84 +units=m +no_defs"
)

def to_lcc(gdf, crs=LCC_CONUS):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(crs)

def normcol(s): 
    return str(s).strip().lower().replace(" ", "_")

def find_ba_mix_df(glob):
    """
    Look for a DataFrame that likely contains BA generation mix with:
    - a BA id column: subregion_id / BA / egrid id
    - fuel columns containing 'coal' and 'gas' (or 'fossil')
    Returns (df_name, df) or (None, None)
    """
    candidates = []
    for name, obj in glob.items():
        if not isinstance(obj, (pd.DataFrame, gpd.GeoDataFrame)):
            continue
        if isinstance(obj, gpd.GeoDataFrame):
            df = pd.DataFrame(obj.drop(columns="geometry", errors="ignore"))
        else:
            df = obj

        cols = [normcol(c) for c in df.columns]
        has_ba = any(c in cols for c in ["subregion_id", "ba_id", "egrid_ba", "region_ba", "ba"])
        has_coal = any("coal" in c for c in cols)
        has_gas  = any("gas"  in c for c in cols)
        has_fossil = any("fossil" in c for c in cols)

        if has_ba and ((has_coal and has_gas) or has_fossil):
            candidates.append((name, df))

    # Prefer one that has both coal and gas explicitly
    for name, df in candidates:
        cols = [normcol(c) for c in df.columns]
        if any("coal" in c for c in cols) and any("gas" in c for c in cols):
            return name, df

    return (candidates[0] if candidates else (None, None))

def resolve_one(df, wants, required=True, label=""):
    cols = list(df.columns)
    cols_norm = {normcol(c): c for c in cols}
    for w in wants:
        wn = normcol(w)
        if wn in cols_norm:
            return cols_norm[wn]
    if required:
        raise KeyError(f"[{label}] Could not resolve among {wants}. Available: {cols[:60]}")
    return None

# -----------------------------
# EXPECT: gdf_sites, gdf_basin, gdf_ba, states48 exist (you confirmed)
# -----------------------------
assert isinstance(gdf_sites, gpd.GeoDataFrame)
assert isinstance(gdf_basin, gpd.GeoDataFrame)
assert isinstance(gdf_ba, gpd.GeoDataFrame)
assert isinstance(states48, gpd.GeoDataFrame)

# Resolved from your printout
SITE_BASIN_KEY = "pfaf_id"
BASIN_KEY      = "PFAF_ID"
SITE_BA_KEY    = "subregion_id"
BA_KEY         = "subregion_id"
S1_COL_M3      = "s1_m3_mid"
S2_COL_M3      = "s2_m3_mid"
MW_COL         = "current_mw"
SCARCITY_COL_SITES = "bws_01_score"
SCARCITY_COL_BASIN = "bws_01_score"

# -----------------------------
# Reproject + clip
# -----------------------------
gdf_sites_l = to_lcc(gdf_sites.copy())
gdf_basin_l = to_lcc(gdf_basin.copy())
gdf_ba_l    = to_lcc(gdf_ba.copy())
states48_l  = to_lcc(states48.copy())

conus_geom = states48_l.unary_union
gdf_sites_l = gdf_sites_l[gdf_sites_l.intersects(conus_geom)].copy()
gdf_basin_l = gdf_basin_l[gdf_basin_l.intersects(conus_geom)].copy()
gdf_ba_l    = gdf_ba_l[gdf_ba_l.intersects(conus_geom)].copy()

# Cast join keys
gdf_sites_l[SITE_BASIN_KEY] = gdf_sites_l[SITE_BASIN_KEY].astype(str)
gdf_basin_l[BASIN_KEY]      = gdf_basin_l[BASIN_KEY].astype(str)
gdf_sites_l[SITE_BA_KEY]    = gdf_sites_l[SITE_BA_KEY].astype(str)
gdf_ba_l[BA_KEY]            = gdf_ba_l[BA_KEY].astype(str)

# -----------------------------
# (A) Scope 1 basin hotspot typology (paper definition)
# high S1: > P50 among basins with hyperscale activity
# stressed: basin OR touching neighbor has Aqueduct stress >= 3
# -----------------------------
bas_agg = (
    gdf_sites_l.groupby(SITE_BASIN_KEY, dropna=False)
    .agg(s1_m3_sum=(S1_COL_M3, "sum"),
         mw_sum=(MW_COL, "sum"),
         n_sites=(MW_COL, "size"))
    .reset_index()
    .rename(columns={SITE_BASIN_KEY: BASIN_KEY})
)
gdf_basin_l = gdf_basin_l.merge(bas_agg, on=BASIN_KEY, how="left")
gdf_basin_l[["s1_m3_sum","mw_sum","n_sites"]] = gdf_basin_l[["s1_m3_sum","mw_sum","n_sites"]].fillna(0.0)

active_basins = gdf_basin_l["n_sites"] > 0
s1_thr = float(gdf_basin_l.loc[active_basins, "s1_m3_sum"].quantile(S1_HIGH_QUANTILE)) if active_basins.any() else 0.0
gdf_basin_l["_high_s1"] = active_basins & (gdf_basin_l["s1_m3_sum"] > s1_thr)

gdf_basin_l["_stress_here"] = gdf_basin_l[SCARCITY_COL_BASIN].fillna(0.0) >= STRESS_THRESHOLD

# neighbor-aware stress: basin OR touching stressed neighbor
sindex = gdf_basin_l.sindex
stress_neighbor = np.zeros(len(gdf_basin_l), dtype=bool)
for i, geom in enumerate(gdf_basin_l.geometry):
    if geom is None or geom.is_empty:
        continue
    cand_idx = list(sindex.intersection(geom.bounds))
    for j in cand_idx:
        if j == i:
            continue
        if not bool(gdf_basin_l.iloc[j]["_stress_here"]):
            continue
        gj = gdf_basin_l.geometry.iloc[j]
        if gj is None or gj.is_empty:
            continue
        if geom.touches(gj):
            stress_neighbor[i] = True
            break

gdf_basin_l["_stress_exposed"] = gdf_basin_l["_stress_here"] | stress_neighbor

def s1_class(row):
    hs1 = bool(row["_high_s1"])
    st  = bool(row["_stress_exposed"])
    if hs1 and st: return "Hotspot (High S1 × Stress)"
    if hs1 and not st: return "High Scope 1 only"
    if (not hs1) and st: return "Stress only"
    return "Neither"

gdf_basin_l["_s1_class"] = gdf_basin_l.apply(s1_class, axis=1)

S1_CLASS_ORDER = ["Neither", "Stress only", "High Scope 1 only", "Hotspot (High S1 × Stress)"]
S1_COLORS = {
    "Neither": "#e0e0e0",
    "Stress only": "#9e9e9e",
    "High Scope 1 only": "#74a9cf",
    "Hotspot (High S1 × Stress)": "#b2182b",
}

# -----------------------------
# (B) Scope 2 BA hotspot typology (paper definition)
# high S2: > P75 among BAs with hyperscale activity
# AND fossil-heavy: coal+gas share > 0.5
# AND mean site stress >= 3
# -----------------------------
ba_agg = (
    gdf_sites_l.groupby(SITE_BA_KEY, dropna=False)
    .agg(s2_m3_sum=(S2_COL_M3, "sum"),
         mw_sum=(MW_COL, "sum"),
         n_sites=(MW_COL, "size"),
         mean_stress=(SCARCITY_COL_SITES, "mean"))
    .reset_index()
    .rename(columns={SITE_BA_KEY: BA_KEY})
)
gdf_ba_l = gdf_ba_l.merge(ba_agg, on=BA_KEY, how="left")
for c in ["s2_m3_sum","mw_sum","n_sites","mean_stress"]:
    gdf_ba_l[c] = gdf_ba_l[c].fillna(0.0)

active_bas = gdf_ba_l["n_sites"] > 0
s2_thr = float(gdf_ba_l.loc[active_bas, "s2_m3_sum"].quantile(S2_HIGH_QUANTILE)) if active_bas.any() else 0.0
gdf_ba_l["_high_s2"] = active_bas & (gdf_ba_l["s2_m3_sum"] > s2_thr)
gdf_ba_l["_high_stress_mean"] = gdf_ba_l["mean_stress"] >= STRESS_THRESHOLD

# --- AUTO-FIND & MERGE BA MIX DF ---
mix_name, df_mix = find_ba_mix_df(globals())
if df_mix is None:
    raise RuntimeError(
        "I found your BA geometry (gdf_ba) but NOT the BA generation mix table needed for coal+gas share.\n"
        "Expected something in memory with columns like: subregion_id AND (coal, gas) shares.\n"
        "Load it (csv/parquet) or tell me its variable name; then rerun."
    )

mix_ba_key = resolve_one(df_mix, ["subregion_id","ba_id","egrid_ba","region_ba","ba"], label="mix BA id")
# Prefer explicit coal + gas columns; else fossil_share
coal_col = None
gas_col  = None
try:
    coal_col = resolve_one(df_mix, ["coal_share","coal_frac","coal","share_coal","coal_pct"], required=False)
    gas_col  = resolve_one(df_mix, ["gas_share","gas_frac","gas","share_gas","gas_pct"], required=False)
except Exception:
    pass
fossil_col = resolve_one(df_mix, ["fossil_share","coal_gas_share","fossil_frac","coal+gas_share"], required=False)

df_mix2 = df_mix.copy()
df_mix2[mix_ba_key] = df_mix2[mix_ba_key].astype(str)

if coal_col is not None and gas_col is not None:
    xcoal = df_mix2[coal_col].astype(float)
    xgas  = df_mix2[gas_col].astype(float)
    # if in % convert
    if max(xcoal.max(), xgas.max()) > 1.5:
        fossil_share = (xcoal + xgas) / 100.0
    else:
        fossil_share = (xcoal + xgas)
    df_mix2["_fossil_share"] = fossil_share
elif fossil_col is not None:
    xf = df_mix2[fossil_col].astype(float)
    df_mix2["_fossil_share"] = xf/100.0 if xf.max() > 1.5 else xf
else:
    raise RuntimeError(
        f"Found BA mix table '{mix_name}', but couldn't find coal+gas OR fossil_share columns.\n"
        f"Columns are: {list(df_mix2.columns)[:60]}"
    )

gdf_ba_l = gdf_ba_l.merge(df_mix2[[mix_ba_key, "_fossil_share"]].drop_duplicates(mix_ba_key),
                          left_on=BA_KEY, right_on=mix_ba_key, how="left")
gdf_ba_l["_fossil_share"] = gdf_ba_l["_fossil_share"].fillna(0.0)
gdf_ba_l["_fossil_heavy"] = gdf_ba_l["_fossil_share"] > FOSSIL_THRESHOLD

def s2_class(row):
    hs2 = bool(row["_high_s2"])
    fos = bool(row["_fossil_heavy"])
    st  = bool(row["_high_stress_mean"])
    if hs2 and fos and st: return "Hotspot (High S2 × Fossil × Stress)"
    if hs2 and fos and (not st): return "High S2 × Fossil"
    if hs2 and (not fos) and st: return "High S2 × Stress"
    if hs2 and (not fos) and (not st): return "High Scope 2 only"
    return "Other / low Scope 2"

gdf_ba_l["_s2_class"] = gdf_ba_l.apply(s2_class, axis=1)

S2_CLASS_ORDER = [
    "Other / low Scope 2",
    "High Scope 2 only",
    "High S2 × Fossil",
    "High S2 × Stress",
    "Hotspot (High S2 × Fossil × Stress)"
]
S2_COLORS = {
    "Other / low Scope 2": "#e0e0e0",
    "High Scope 2 only": "#9ecae1",
    "High S2 × Fossil": "#fdae61",
    "High S2 × Stress": "#74a9cf",
    "Hotspot (High S2 × Fossil × Stress)": "#b2182b"
}

# -----------------------------
# Plot (show in notebook + export PDF only)
# -----------------------------
minx, miny, maxx, maxy = states48_l.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

fig, (axA, axB) = plt.subplots(1, 2, figsize=(FIG_W, FIG_H), constrained_layout=True)

# Panel A
for cls in S1_CLASS_ORDER:
    sub = gdf_basin_l[gdf_basin_l["_s1_class"] == cls]
    if len(sub):
        sub.plot(ax=axA, color=S1_COLORS[cls], edgecolor="white", linewidth=0.25, zorder=1)

# Make hotspots pop: strong outline + hatch overlay ONLY on hotspot basins
hotA = gdf_basin_l[gdf_basin_l["_s1_class"] == "Hotspot (High S1 × Stress)"]
if len(hotA):
    hotA.boundary.plot(ax=axA, color="#111111", linewidth=1.2, zorder=6)
    hotA.plot(ax=axA, facecolor="none", edgecolor="#111111", linewidth=0.0, hatch="//////", zorder=7)

states48_l.boundary.plot(ax=axA, color="#111111", linewidth=0.6, zorder=10)
axA.set_axis_off()
axA.set_xlim(minx - padx, maxx + padx); axA.set_ylim(miny - pady, maxy + pady)
axA.set_title(
    "a. Scope 1 basin hotspots\nHigh S1 = above median among basins with hyperscale activity; Stress = basin or touching neighbor has Aqueduct ≥ 3",
    fontsize=10
)
axA.legend([Patch(facecolor=S1_COLORS[c], edgecolor="none") for c in S1_CLASS_ORDER],
           S1_CLASS_ORDER, loc="lower left", frameon=True, fontsize=9)

# Panel B
for cls in S2_CLASS_ORDER:
    sub = gdf_ba_l[gdf_ba_l["_s2_class"] == cls]
    if len(sub):
        sub.plot(ax=axB, color=S2_COLORS[cls], edgecolor="white", linewidth=0.25, zorder=1)

# Make hotspots pop: strong outline
hotB = gdf_ba_l[gdf_ba_l["_s2_class"] == "Hotspot (High S2 × Fossil × Stress)"]
if len(hotB):
    # 1) white halo (drawn first, wider)
    hotB.boundary.plot(ax=axB, color="white", linewidth=4.2, zorder=20)

    # 2) main dark outline (drawn on top)
    hotB.boundary.plot(ax=axB, color="#111111", linewidth=2.4, zorder=21)

    # 3) optional hatch to further emphasize the hotspot polygons
    hotB.plot(ax=axB, facecolor="none", edgecolor="#111111", linewidth=0.1,
              hatch="xx", zorder=19)
    
states48_l.boundary.plot(ax=axB, color="#111111", linewidth=0.6, zorder=10)
axB.set_axis_off()
axB.set_xlim(minx - padx, maxx + padx); axB.set_ylim(miny - pady, maxy + pady)
axB.set_title(
    "b. Scope 2 BA hotspots\nHigh S2 = top quartile among BAs with hyperscale activity; Fossil = coal+gas > 50%; Stress = mean site Aqueduct ≥ 3",
    fontsize=10
)

axB.legend([Patch(facecolor=S2_COLORS[c], edgecolor="none") for c in S2_CLASS_ORDER],
           S2_CLASS_ORDER, loc="lower left", frameon=True, fontsize=9)

out_pdf = "Fig2_hotspot_typologies_baseline.pdf"
plt.savefig(out_pdf, dpi=DPI, bbox_inches="tight")
plt.show()

print("Saved PDF:", out_pdf)
print(f"Using BA mix table: {mix_name}")
print(f"Scope 1 threshold (P50 active basins): {s1_thr:.3e} m3/yr")
print(f"Scope 2 threshold (P75 active BAs):   {s2_thr:.3e} m3/yr")
print("Fossil share stats (fraction):", gdf_ba_l["_fossil_share"].min(), gdf_ba_l["_fossil_share"].median(), gdf_ba_l["_fossil_share"].max())

In [ ]:
# ============================================================
# FIGURES 2 / 3 / 4 — publication-style plotting (PDF export only)
# Uses: gdf_sites, gdf_basin, gdf_ba, states48
# ============================================================

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from pyproj import CRS

# -----------------------------
# CONFIG (edit if you want)
# -----------------------------
OUT_FIG2_PDF = "Fig2_HotspotMaps.pdf"
OUT_FIG3_PDF = "Fig3_Mechanisms.pdf"
OUT_FIG4_PDF = "Fig4_ConcentrationCurves.pdf"

DPI = 350

# Column map (confirmed)
SITE_BASIN_KEY = "pfaf_id"
BASIN_KEY      = "PFAF_ID"
SITE_BA_KEY    = "subregion_id"
BA_KEY         = "subregion_id"

S1_COL_M3      = "s1_m3_mid"
S2_COL_M3      = "s2_m3_mid"
MW_COL         = "current_mw"
STRESS_COL     = "bws_01_score"   # exists in sites and basin

# thresholds (paper)
STRESS_TH = 3.0

# Scope 2 thresholds (paper)
S2_Q = 0.75            # top quartile
FOSSIL_TH = 0.50       # coal+gas > 0.5
MEAN_STRESS_TH = 3.0

# projection (CONUS LCC)
LCC_CONUS = CRS.from_proj4(
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 +datum=WGS84 +units=m +no_defs"
)

def to_lcc(gdf, crs=LCC_CONUS):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(crs)

def safe_str(s):
    return s.astype(str).fillna("")

def m3_to_GL(x):
    return np.asarray(x, dtype=float) / 1e6

def m3_to_ML(x):
    # 1 ML = 1e6 L = 1000 m3
    return np.asarray(x, dtype=float) / 1000.0

def _first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

# -----------------------------
# Validate inputs
# -----------------------------
for name in ["gdf_sites", "gdf_basin", "gdf_ba", "states48"]:
    if name not in globals():
        raise NameError(f"Missing required object: {name}")

# Copy & project
sites  = to_lcc(gdf_sites.copy())
basins = to_lcc(gdf_basin.copy())
bas    = to_lcc(gdf_ba.copy())
st48   = to_lcc(states48.copy())

# Clip to CONUS (safe)
conus = st48.unary_union
sites  = sites[sites.intersects(conus)].copy()
basins = basins[basins.intersects(conus)].copy()
bas    = bas[bas.intersects(conus)].copy()

# Key cleanup
sites[SITE_BASIN_KEY] = safe_str(sites[SITE_BASIN_KEY])
sites[SITE_BA_KEY]    = safe_str(sites[SITE_BA_KEY])
basins[BASIN_KEY]     = safe_str(basins[BASIN_KEY])
bas[BA_KEY]           = safe_str(bas[BA_KEY])

# Bounds
minx, miny, maxx, maxy = st48.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

# ============================================================
# FIGURE 2 — Hotspot maps (Scope 1 basins + Scope 2 BAs)
# ============================================================

# ---- Scope 1: aggregate to basins
bas_agg = (sites
    .groupby(SITE_BASIN_KEY, dropna=False)
    .agg(
        s1_m3_sum=(S1_COL_M3, "sum"),
        mw_sum=(MW_COL, "sum"),
        stress_max_site=(STRESS_COL, "max"),
    )
    .reset_index()
    .rename(columns={SITE_BASIN_KEY: BASIN_KEY})
)

basins2 = basins.merge(bas_agg, on=BASIN_KEY, how="left")
basins2["s1_m3_sum"] = basins2["s1_m3_sum"].fillna(0.0)
basins2["mw_sum"]    = basins2["mw_sum"].fillna(0.0)

# Active basins = have any hyperscale activity (mw_sum > 0 or s1 > 0)
active_basins = basins2["mw_sum"] > 0
s1_median = float(basins2.loc[active_basins, "s1_m3_sum"].median()) if active_basins.any() else 0.0

basins2["_high_s1"] = basins2["s1_m3_sum"] > s1_median

# Stress in basin OR touching neighbor:
# - build neighbor graph using touches() (OK for 1359 polygons)
# - compute max stress among self + neighbors
basins2["_stress_self"] = basins2[STRESS_COL].fillna(0.0) >= STRESS_TH

# neighbor-aware stress: conservative screening
# (We compute for all polygons; if performance issues, we can restrict to active basins)
stress_vals = basins2[STRESS_COL].fillna(0.0).to_numpy()
geoms = basins2.geometry

neighbor_stress_max = np.zeros(len(basins2), dtype=float)
for i in range(len(basins2)):
    # touches can be expensive, but 1359 is manageable
    touched = geoms.touches(geoms.iloc[i])
    # include self
    idx = touched.to_numpy()
    # max among neighbors + self
    m = max(stress_vals[i], stress_vals[idx].max() if idx.any() else 0.0)
    neighbor_stress_max[i] = m

basins2["_stress_neighbor_max"] = neighbor_stress_max
basins2["_stress_any"] = basins2["_stress_neighbor_max"] >= STRESS_TH

basins2["_s1_hotspot"] = basins2["_high_s1"] & basins2["_stress_any"]

# Categorical class for plotting (Scope 1)
def _s1_class(row):
    if row["_s1_hotspot"]:
        return "Hotspot (High S1 × Stress)"
    if row["_high_s1"] and (not row["_stress_any"]):
        return "High Scope 1 only"
    if (not row["_high_s1"]) and row["_stress_any"]:
        return "Stress only"
    return "Neither"

basins2["_class_s1"] = basins2.apply(_s1_class, axis=1)

S1_COLORS = {
    "Neither": "#e0e0e0",
    "Stress only": "#9e9e9e",
    "High Scope 1 only": "#74a9cf",
    "Hotspot (High S1 × Stress)": "#b2182b",
}

# ---- Scope 2: aggregate to BA
ba_agg = (sites
    .groupby(SITE_BA_KEY, dropna=False)
    .agg(
        s2_m3_sum=(S2_COL_M3, "sum"),
        mw_sum=(MW_COL, "sum"),
        mean_stress=(STRESS_COL, "mean"),
    )
    .reset_index()
    .rename(columns={SITE_BA_KEY: BA_KEY})
)

bas2 = bas.merge(ba_agg, on=BA_KEY, how="left")
bas2["s2_m3_sum"]   = bas2["s2_m3_sum"].fillna(0.0)
bas2["mw_sum"]      = bas2["mw_sum"].fillna(0.0)
bas2["mean_stress"] = bas2["mean_stress"].fillna(np.nan)

active_bas = bas2["mw_sum"] > 0
s2_q = float(bas2.loc[active_bas, "s2_m3_sum"].quantile(S2_Q)) if active_bas.any() else 0.0
bas2["_high_s2"] = bas2["s2_m3_sum"] > s2_q
bas2["_stress_ba"] = bas2["mean_stress"] >= MEAN_STRESS_TH

# Fossil share column detection (optional)
fossil_col = _first_existing_col(
    bas2,
    candidates=[
        "fossil_share", "coal_gas_share", "coal_gas_frac", "fossil_frac",
        "share_coal_gas", "coal_gas", "fossil"
    ],
)

if fossil_col is None:
    print("⚠️  Scope 2 typology: fossil-share column not found in gdf_ba. "
          "Falling back to High S2 × Stress (2-way). "
          "If you have it, merge it into gdf_ba as 'fossil_share' (0–1).")
    bas2["_fossil_heavy"] = False
else:
    bas2["_fossil_heavy"] = bas2[fossil_col].astype(float) > FOSSIL_TH

# 5-class typology (as in blueprint; full triple if available)
def _s2_class(row):
    if row["_high_s2"] and row["_stress_ba"] and row["_fossil_heavy"]:
        return "Hotspot (High S2 × Fossil × Stress)"
    if row["_high_s2"] and row["_fossil_heavy"]:
        return "High S2 × Fossil"
    if row["_high_s2"] and row["_stress_ba"]:
        return "High S2 × Stress"
    if row["_high_s2"]:
        return "High Scope 2 only"
    return "Other / low Scope 2"

bas2["_class_s2"] = bas2.apply(_s2_class, axis=1)

S2_COLORS = {
    "Other / low Scope 2": "#e0e0e0",
    "High Scope 2 only": "#9ecae1",
    "High S2 × Fossil": "#fdae61",
    "High S2 × Stress": "#74a9cf",
    "Hotspot (High S2 × Fossil × Stress)": "#b2182b",
}

# ---- Plot Figure 2
fig = plt.figure(figsize=(11.5, 12.5), constrained_layout=False)
gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[1, 1], left=0.04, right=0.995, top=0.965, bottom=0.05, hspace=0.08)
axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[1, 0])

# Panel A
for k, color in S1_COLORS.items():
    basins2.loc[basins2["_class_s1"] == k].plot(ax=axA, color=color, edgecolor="white", linewidth=0.25, zorder=1)

# Hotspot outline (very visible)
basins2.loc[basins2["_s1_hotspot"]].boundary.plot(ax=axA, color="black", linewidth=1.4, zorder=10)

# Optional: hatch overlay for hotspots (stronger than before)
# (Hatching can be subtle depending on backend; keep + outline to guarantee visibility)
basins2.loc[basins2["_s1_hotspot"]].plot(ax=axA, facecolor="none", edgecolor="none", hatch="////", linewidth=0, zorder=9)

st48.boundary.plot(ax=axA, color="#111111", linewidth=0.6, zorder=20)
axA.set_title("a. Scope 1 basin hotspots (baseline)", fontsize=12)
axA.set_axis_off()
axA.set_xlim(minx - padx, maxx + padx); axA.set_ylim(miny - pady, maxy + pady)
axA.set_aspect("equal")

# Legend A
legA = [
    Patch(facecolor=S1_COLORS["Hotspot (High S1 × Stress)"], edgecolor="black", label="Hotspot (High S1 × Stress)"),
    Patch(facecolor=S1_COLORS["High Scope 1 only"], edgecolor="white", label="High Scope 1 only"),
    Patch(facecolor=S1_COLORS["Stress only"], edgecolor="white", label="Stress only"),
    Patch(facecolor=S1_COLORS["Neither"], edgecolor="white", label="Neither"),
]
axA.legend(handles=legA, title="Scope 1 hotspot classification",
           loc="lower left", frameon=True, framealpha=0.95, fontsize=9, title_fontsize=10)

axA.text(0.01, 0.02,
         f"High S1: basin total > median among active basins (P50 = {m3_to_GL(s1_median):.1f} GL/yr)\n"
         f"Stress: Aqueduct ≥ {STRESS_TH:.0f} in basin OR touching neighbor",
         transform=axA.transAxes, fontsize=8, va="bottom")

# Panel B
for k, color in S2_COLORS.items():
    bas2.loc[bas2["_class_s2"] == k].plot(ax=axB, color=color, edgecolor="white", linewidth=0.25, zorder=1)

# Hotspot outline (very evident) — requested
bas2.loc[bas2["_class_s2"] == "Hotspot (High S2 × Fossil × Stress)"].boundary.plot(
    ax=axB, color="black", linewidth=1.8, zorder=10
)
# If fossil not available, still outline High S2 × Stress as "hotspot-like"
if fossil_col is None:
    bas2.loc[bas2["_high_s2"] & bas2["_stress_ba"]].boundary.plot(
        ax=axB, color="black", linewidth=1.8, zorder=10
    )

st48.boundary.plot(ax=axB, color="#111111", linewidth=0.6, zorder=20)
axB.set_title("b. Scope 2 balancing-authority hotspots (baseline)", fontsize=12)
axB.set_axis_off()
axB.set_xlim(minx - padx, maxx + padx); axB.set_ylim(miny - pady, maxy + pady)
axB.set_aspect("equal")

# Legend B
legB = [
    Patch(facecolor=S2_COLORS["Hotspot (High S2 × Fossil × Stress)"], edgecolor="black", label="Hotspot (High S2 × Fossil × Stress)"),
    Patch(facecolor=S2_COLORS["High S2 × Fossil"], edgecolor="white", label="High S2 × Fossil"),
    Patch(facecolor=S2_COLORS["High S2 × Stress"], edgecolor="white", label="High S2 × Stress"),
    Patch(facecolor=S2_COLORS["High Scope 2 only"], edgecolor="white", label="High Scope 2 only"),
    Patch(facecolor=S2_COLORS["Other / low Scope 2"], edgecolor="white", label="Other / low Scope 2"),
]
axB.legend(handles=legB, title="Scope 2 hotspot typology",
           loc="lower left", frameon=True, framealpha=0.95, fontsize=9, title_fontsize=10)

axB.text(0.01, 0.02,
         f"High S2: BA total > P75 among active BAs (P75 = {m3_to_GL(s2_q):.1f} GL/yr)\n"
         f"Stress: mean facility Aqueduct ≥ {MEAN_STRESS_TH:.0f}"
         + ("" if fossil_col is None else f"\nFossil-heavy: coal+gas > {int(FOSSIL_TH*100)}%"),
         transform=axB.transAxes, fontsize=8, va="bottom")

# Save PDF only + show
fig.savefig(OUT_FIG2_PDF, dpi=DPI, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT_FIG2_PDF}")


# ============================================================
# FIGURE 3 — Mechanisms (scatter panels)
# ============================================================

# Basin-level intensity (Scope 1)
bas_tbl = basins2.loc[active_basins].copy()
bas_tbl["s1_GL"] = m3_to_GL(bas_tbl["s1_m3_sum"])
bas_tbl["mw"]    = bas_tbl["mw_sum"]
bas_tbl["s1_intensity_ML_per_MWyr"] = m3_to_ML(bas_tbl["s1_m3_sum"]) / bas_tbl["mw"].replace(0, np.nan)
bas_tbl["stress"] = bas_tbl[STRESS_COL].fillna(np.nan)

# BA-level: we prefer grid intensity if present; otherwise use Scope2 intensity
grid_intensity_col = _first_existing_col(
    bas2,
    candidates=["grid_L_per_kWh", "I_grid_L_per_kWh", "grid_intensity_L_per_kWh", "grid_l_per_kwh", "grid_LkWh"]
)

ba_tbl = bas2.loc[active_bas].copy()
ba_tbl["s2_GL"] = m3_to_GL(ba_tbl["s2_m3_sum"])
ba_tbl["mw"]    = ba_tbl["mw_sum"]
ba_tbl["s2_intensity_ML_per_MWyr"] = m3_to_ML(ba_tbl["s2_m3_sum"]) / ba_tbl["mw"].replace(0, np.nan)
ba_tbl["mean_stress"] = ba_tbl["mean_stress"].fillna(np.nan)

if grid_intensity_col is not None:
    ba_x = ba_tbl[grid_intensity_col].astype(float)
    ba_x_label = "Grid water intensity (L/kWh)"
    x_line = ba_x.quantile(0.75)  # optional
else:
    ba_x = ba_tbl["s2_intensity_ML_per_MWyr"]
    ba_x_label = "Scope 2 intensity (ML/MW-yr)"
    x_line = ba_x.quantile(0.75)

# fossil share (optional) to color
if fossil_col is not None:
    ba_color = ba_tbl[fossil_col].astype(float)
    color_label = "Fossil share (coal+gas)"
else:
    ba_color = ba_tbl["s2_GL"]
    color_label = "Scope 2 total (GL/yr)"

# Plot
fig = plt.figure(figsize=(12.0, 9.0), constrained_layout=False)
gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[1, 1], left=0.08, right=0.98, top=0.95, bottom=0.10, hspace=0.34)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[1, 0])

# Panel A scatter
sizes1 = 30 + 220 * (bas_tbl["s1_GL"] / bas_tbl["s1_GL"].max()).fillna(0.0).to_numpy()
sc1 = ax1.scatter(
    bas_tbl["mw"], bas_tbl["s1_intensity_ML_per_MWyr"],
    s=sizes1, c=bas_tbl["stress"], alpha=0.85
)
ax1.axvline(bas_tbl["mw"].median(), linewidth=1.0)
ax1.axhline(np.nanmedian(bas_tbl["s1_intensity_ML_per_MWyr"]), linewidth=1.0)
ax1.grid(True, alpha=0.25)
ax1.set_title("a. Basin-scale drivers of Scope 1 water consumption", fontsize=12)
ax1.set_xlabel("Total hyperscale capacity in basin (MW)", fontsize=10)
ax1.set_ylabel("Scope 1 intensity (ML/MW-yr)", fontsize=10)
cbar1 = fig.colorbar(sc1, ax=ax1, orientation="vertical", fraction=0.035, pad=0.02)
cbar1.set_label("Aqueduct baseline stress score", fontsize=9)

# Panel B scatter
sizes2 = 30 + 220 * (ba_tbl["s2_GL"] / ba_tbl["s2_GL"].max()).fillna(0.0).to_numpy()
sc2 = ax2.scatter(
    ba_x, ba_tbl["mean_stress"],
    s=sizes2, c=ba_color, alpha=0.85
)
ax2.axvline(x_line, linewidth=1.0)
ax2.axhline(MEAN_STRESS_TH, linewidth=1.0)
ax2.grid(True, alpha=0.25)
ax2.set_title("b. Grid intensity / intensity vs stress exposure among balancing authorities", fontsize=12)
ax2.set_xlabel(ba_x_label, fontsize=10)
ax2.set_ylabel("Mean facility baseline stress (Aqueduct)", fontsize=10)
cbar2 = fig.colorbar(sc2, ax=ax2, orientation="vertical", fraction=0.035, pad=0.02)
cbar2.set_label(color_label, fontsize=9)

fig.savefig(OUT_FIG3_PDF, dpi=DPI, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT_FIG3_PDF}")


# ============================================================
# FIGURE 4 — Concentration / “leverage” curves (actionable)
# How much of national S1/S2 is captured by the top N basins/BAs?
# ============================================================

# Basin concentration curve (Scope 1)
bas_rank = bas_tbl[["s1_m3_sum"]].copy()
bas_rank["s1_m3_sum"] = bas_rank["s1_m3_sum"].fillna(0.0)
bas_rank = bas_rank.sort_values("s1_m3_sum", ascending=False)
bas_rank["cum_share"] = bas_rank["s1_m3_sum"].cumsum() / max(bas_rank["s1_m3_sum"].sum(), 1e-9)
bas_rank["rank"] = np.arange(1, len(bas_rank) + 1)

# BA concentration curve (Scope 2)
ba_rank = ba_tbl[["s2_m3_sum"]].copy()
ba_rank["s2_m3_sum"] = ba_rank["s2_m3_sum"].fillna(0.0)
ba_rank = ba_rank.sort_values("s2_m3_sum", ascending=False)
ba_rank["cum_share"] = ba_rank["s2_m3_sum"].cumsum() / max(ba_rank["s2_m3_sum"].sum(), 1e-9)
ba_rank["rank"] = np.arange(1, len(ba_rank) + 1)

fig = plt.figure(figsize=(12.0, 5.2), constrained_layout=False)
gs = fig.add_gridspec(nrows=1, ncols=2, left=0.07, right=0.98, top=0.92, bottom=0.18, wspace=0.18)
axL = fig.add_subplot(gs[0, 0])
axR = fig.add_subplot(gs[0, 1])

axL.plot(bas_rank["rank"], bas_rank["cum_share"], linewidth=2.5)
axL.set_title("a. Scope 1 concentration across basins", fontsize=12)
axL.set_xlabel("Top N basins (ranked by Scope 1 total)", fontsize=10)
axL.set_ylabel("Cumulative share of national Scope 1", fontsize=10)
axL.grid(True, alpha=0.25)

axR.plot(ba_rank["rank"], ba_rank["cum_share"], linewidth=2.5)
axR.set_title("b. Scope 2 concentration across balancing authorities", fontsize=12)
axR.set_xlabel("Top N BAs (ranked by Scope 2 total)", fontsize=10)
axR.set_ylabel("Cumulative share of national Scope 2", fontsize=10)
axR.grid(True, alpha=0.25)

# Helpful reference lines (optional)
for ax in (axL, axR):
    ax.axhline(0.5, linewidth=1.0, alpha=0.7)
    ax.axhline(0.8, linewidth=1.0, alpha=0.7)

fig.savefig(OUT_FIG4_PDF, dpi=DPI, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT_FIG4_PDF}")

# final plots

In [ ]:
gdf_ba_l.columns

In [ ]:
def pick_ba_layer(gdf_ba, gdf_ba_l=None):
    """
    Prefer gdf_ba_l if provided (contains fossil share / typology fields),
    else fall back to gdf_ba.
    """
    if gdf_ba_l is not None:
        return gdf_ba_l
    # also support if it's defined in globals()
    if "gdf_ba_l" in globals() and globals()["gdf_ba_l"] is not None:
        return globals()["gdf_ba_l"]
    return gdf_ba

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pyproj import CRS

# -----------------------------
# OUTPUT DIR
# -----------------------------
OUTDIR = "images"
os.makedirs(OUTDIR, exist_ok=True)

# -----------------------------
# COLUMN KEYS (from your environment)
# -----------------------------
SITE_BASIN_KEY = "pfaf_id"
BASIN_KEY      = "PFAF_ID"
SITE_BA_KEY    = "subregion_id"
BA_KEY         = "subregion_id"

S1_COL_M3 = "s1_m3_mid"
S2_COL_M3 = "s2_m3_mid"
MW_COL    = "current_mw"
STRESS_COL_SITE  = "bws_01_score"
STRESS_COL_BASIN = "bws_01_score"

# -----------------------------
# SCENARIO DEFAULTS (match your SI Table S1)
# -----------------------------
u_eff, u_base, u_high = 0.55, 0.66, 0.85
pue_eff, pue_base, pue_high = 1.15, 1.25, 1.40
wue_eff, wue_base, wue_high = 0.2, 0.8, 1.5

# -----------------------------
# MAP PROJECTION (CONUS-friendly)
# -----------------------------
LCC_CONUS = CRS.from_proj4(
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 +datum=WGS84 +units=m +no_defs"
)

def _assert_has(df, cols, name="df"):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing required columns: {missing}")

def to_lcc(gdf, crs=LCC_CONUS):
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    return gdf.to_crs(crs)

def m3_to_GL(x):
    # 1 GL = 1e9 L = 1e6 m3
    return np.asarray(x, dtype=float) / 1e6

def _prep_conus(gdf_sites, gdf_basin, gdf_ba, states48):
    # CRS and CONUS clip (intersects union)
    sites = to_lcc(gdf_sites.copy())
    basin = to_lcc(gdf_basin.copy())
    ba    = to_lcc(gdf_ba.copy())
    st    = to_lcc(states48.copy())
    conus = st.unary_union

    sites = sites[sites.intersects(conus)].copy()
    basin = basin[basin.intersects(conus)].copy()
    ba    = ba[ba.intersects(conus)].copy()

    # key cleanup
    sites[SITE_BASIN_KEY] = sites[SITE_BASIN_KEY].astype(str)
    sites[SITE_BA_KEY]    = sites[SITE_BA_KEY].astype(str)
    basin[BASIN_KEY]      = basin[BASIN_KEY].astype(str)
    ba[BA_KEY]            = ba[BA_KEY].astype(str)

    return sites, basin, ba, st

def _neighbor_aware_high_stress(gdf_poly, stress_col, thr=3.0):
    """
    Returns boolean Series: True if basin itself is high stress OR touches any high-stress basin.
    Uses spatial index on stressed polygons for speed.
    """
    s = gdf_poly[stress_col].astype(float).fillna(0.0)
    hi = (s >= thr).values
    geom = gdf_poly.geometry

    stressed = gdf_poly.loc[hi, ["geometry"]].copy()
    if stressed.empty:
        return pd.Series(False, index=gdf_poly.index)

    sidx = stressed.sindex

    out = np.zeros(len(gdf_poly), dtype=bool)
    out[hi] = True

    stressed_geoms = stressed.geometry.values
    stressed_index = stressed.index.values  # not used but kept for clarity

    for i in np.where(~hi)[0]:
        g = geom.iloc[i]
        cand_pos = list(sidx.intersection(g.bounds))
        # cand_pos are positional indices into `stressed`, not original
        touched = False
        for pos in cand_pos:
            if g.touches(stressed_geoms[pos]):
                touched = True
                break
        out[i] = touched

    return pd.Series(out, index=gdf_poly.index)

def _find_fossil_share_series(df):
    """
    Try to find a fossil share (coal+gas) column in df.
    Returns (series, description) or (None, None).
    Accepts either fraction [0,1] or percent [0,100].
    """
    direct_candidates = [
        "coal_gas_share", "coal_gas_frac", "coal_gas_fraction",
        "fossil_share", "fossil_frac", "fossil_fraction",
        "fossil_pct", "coal_gas_pct"
    ]
    for c in direct_candidates:
        if c in df.columns:
            s = pd.to_numeric(df[c], errors="coerce")
            # normalize percent to fraction
            if s.max(skipna=True) > 1.5:
                s = s / 100.0
            return s, f"direct column `{c}`"

    # try coal + gas columns
    coal_candidates = ["coal_share", "share_coal", "coal_frac", "coal_fraction", "coal_pct"]
    gas_candidates  = ["gas_share", "share_gas", "gas_frac", "gas_fraction", "gas_pct",
                       "ng_share", "natural_gas_share", "naturalgas_share"]

    coal_col = next((c for c in coal_candidates if c in df.columns), None)
    gas_col  = next((c for c in gas_candidates if c in df.columns), None)
    if coal_col and gas_col:
        coal = pd.to_numeric(df[coal_col], errors="coerce")
        gas  = pd.to_numeric(df[gas_col], errors="coerce")
        s = coal + gas
        if s.max(skipna=True) > 1.5:
            s = s / 100.0
        return s, f"sum of `{coal_col}` + `{gas_col}`"

    return None, None

def _style_pub():
    plt.rcParams.update({
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "legend.fontsize": 9,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
    })

_style_pub()
print("Helpers loaded. Output dir:", OUTDIR)

## facility and area weighted hotspots

In [ ]:
# =========================
# BA: capacity-weighted mean facility stress
# =========================
import numpy as np
import pandas as pd

# columns (from your resolved map)
SITE_BA_KEY = "subregion_id"
MW_COL = "current_mw"
STRESS_COL_SITE = "bws_01_score"   # site stress column

# clean numeric
sites = gdf_sites.copy()
sites[MW_COL] = pd.to_numeric(sites[MW_COL], errors="coerce").fillna(0.0)
sites[STRESS_COL_SITE] = pd.to_numeric(sites[STRESS_COL_SITE], errors="coerce")

# drop missing stress (don’t let NaN pollute mean)
s = sites.dropna(subset=[STRESS_COL_SITE]).copy()
s["_w"] = s[MW_COL].clip(lower=0.0)

# avoid zero-weight groups: if all MW=0 in a BA, fallback to simple mean
agg = (
    s.groupby(SITE_BA_KEY, dropna=False)
     .apply(lambda df: pd.Series({
         "n_sites_stress": len(df),
         "mw_sum_stress": float(df["_w"].sum()),
         "mean_stress_simple": float(df[STRESS_COL_SITE].mean()),
         "mean_stress_mw": float(
             np.average(df[STRESS_COL_SITE], weights=df["_w"])
         ) if df["_w"].sum() > 0 else float(df[STRESS_COL_SITE].mean())
     }))
     .reset_index()
     .rename(columns={SITE_BA_KEY: "subregion_id"})
)

# join into your BA layer (use gdf_ba_l if you want)
gdf_ba_l = gdf_ba_l.merge(agg[["subregion_id","mean_stress_mw","mean_stress_simple","n_sites_stress","mw_sum_stress"]],
                          on="subregion_id", how="left")

# optional: create the boolean your hotspot logic expects
MEAN_STRESS_THR = 3.0
gdf_ba_l["_high_stress_mean"] = gdf_ba_l["mean_stress_mw"].fillna(0.0) >= MEAN_STRESS_THR

print(gdf_ba_l[["subregion_id","mean_stress_mw","n_sites_stress","mw_sum_stress","_high_stress_mean"]].head())

In [ ]:
# =========================
# BA: area-weighted mean stress from basin polygons
# =========================
import geopandas as gpd
import numpy as np
import pandas as pd
from pyproj import CRS

# Inputs
STRESS_COL_BASIN = "bws_01_score"
BA_KEY = "subregion_id"
BASIN_KEY = "PFAF_ID"

# Choose an equal-area or at least area-meaningful CRS for CONUS.
# Your LCC is okay; equal-area (Albers) is also fine.
LCC_CONUS = CRS.from_proj4(
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 +lon_0=-96 +datum=WGS84 +units=m +no_defs"
)

def to_crs_safe(gdf, crs):
    g = gdf.copy()
    if g.crs is None:
        g = g.set_crs("EPSG:4326", allow_override=True)
    return g.to_crs(crs)

# Use whichever BA layer you want to annotate (gdf_ba_l preferred)
ba = gdf_ba_l.copy()
bas = gdf_basin.copy()

# Keep only columns we need
ba = ba[[BA_KEY, "geometry"]].copy()
bas = bas[[BASIN_KEY, STRESS_COL_BASIN, "geometry"]].copy()

# Clean stress
bas[STRESS_COL_BASIN] = pd.to_numeric(bas[STRESS_COL_BASIN], errors="coerce")
bas = bas.dropna(subset=[STRESS_COL_BASIN]).copy()

# Project for area calculations
ba_p  = to_crs_safe(ba,  LCC_CONUS)
bas_p = to_crs_safe(bas, LCC_CONUS)

# Intersection overlay
# (This can be a bit heavy; with 53 BAs x ~1359 basins it's usually OK.)
inter = gpd.overlay(
    bas_p,
    ba_p,
    how="intersection",
    keep_geom_type=False
)

# Compute areas and area-weighted stress
inter["_area"] = inter.geometry.area
inter = inter[inter["_area"] > 0].copy()

aw = (
    inter.groupby(BA_KEY, dropna=False)
         .apply(lambda df: pd.Series({
             "area_km2": float(df["_area"].sum() / 1e6),
             "mean_stress_area": float(np.average(df[STRESS_COL_BASIN], weights=df["_area"]))
         }))
         .reset_index()
)

# Join back
gdf_ba_l = gdf_ba_l.merge(aw[[BA_KEY, "area_km2", "mean_stress_area"]], on=BA_KEY, how="left")

# Optional boolean
MEAN_STRESS_THR = 3.0
gdf_ba_l["_high_stress_area"] = gdf_ba_l["mean_stress_area"].fillna(0.0) >= MEAN_STRESS_THR

print(gdf_ba_l[[BA_KEY, "mean_stress_area", "area_km2", "_high_stress_area"]].head())

In [ ]:
# =========================
# FIGURE 2: HOTSPOT ATLAS (BEST PRACTICE; prefers gdf_ba_l)
# - Always shows plot in notebook
# - Exports PDF only
# - Centralized "threshold knobs" at the top
# - Uses gdf_ba_l precomputed flags when available; otherwise computes a minimal fallback
# - Legends are explicit about definitions (paper-consistent)
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# -----------------------------
# OUTPUT
# -----------------------------
OUTDIR = "images"
os.makedirs(OUTDIR, exist_ok=True)
OUT_PDF = os.path.join(OUTDIR, "Fig2_HotspotAtlas_base.pdf")

# -----------------------------
# THRESHOLD KNOBS (edit here)
# -----------------------------
# Scope 1 basin hotspot:
S1_HIGH_Q = 0.50          # 0.50 = median among basins hosting sites (paper)
STRESS_THR = 3.0          # Aqueduct baseline stress threshold (paper)
USE_NEIGHBOR_STRESS = True  # paper: stress in basin OR touching neighbor

# Scope 2 BA hotspot typology:
S2_HIGH_Q = 0.20          # top quartile among BAs hosting sites (paper)
FOSSIL_THR = 0.20         # coal+gas share threshold (paper)
MEAN_STRESS_THR = 2.0     # mean facility stress threshold (paper)

# -----------------------------
# STYLE
# -----------------------------
FIG_W, FIG_H = 14.5, 7.2

# 4-class palettes (fixed, readable)
S1_COLORS = {0:"#f0f0f0", 1:"#bdbdbd", 2:"#9ecae1", 3:"#de2d26"}  # neither, stress, high S1, hotspot
S2_COLORS = {0:"#f0f0f0", 1:"#9ecae1", 2:"#fc9272", 3:"#de2d26"}  # not high S2, high S2 only, +1 crit, triple

STATE_EDGE = "#111111"
STATE_LW   = 0.6

# Make boundaries pop (you asked for this)
HOTSPOT_EDGE = "#000000"
S1_HOT_LW = 1.25
S2_HOT_LW = 2.35

# Stress overlay hatch: make it visible
HATCH = "////"
HATCH_EDGE = "#2b2b2b"

# -----------------------------
# REQUIRE INPUTS EXIST
# -----------------------------
_assert_has(gdf_sites, [SITE_BASIN_KEY, SITE_BA_KEY, MW_COL, S1_COL_M3, S2_COL_M3], "gdf_sites")
_assert_has(gdf_basin, [BASIN_KEY, STRESS_COL_BASIN, "geometry"], "gdf_basin")
_assert_has(states48,  ["geometry"], "states48")

# Prefer gdf_ba_l if present (contains fossil share + precomputed flags)
ba_src = pick_ba_layer(gdf_ba, gdf_ba_l if "gdf_ba_l" in globals() else None)
_assert_has(ba_src, [BA_KEY, "geometry"], "BA layer (gdf_ba_l or gdf_ba)")

using_ba_l = ("gdf_ba_l" in globals()) and (ba_src is globals().get("gdf_ba_l"))

# If using gdf_ba_l, enforce expected columns exist
if using_ba_l:
    _assert_has(
        ba_src,
        ["_fossil_share", "_fossil_heavy", "_high_s2", "_high_stress_mean"],
        "gdf_ba_l"
    )

# Prep CONUS (assumes your helper handles clipping and CRS consistently)
sites, basin, ba_layer, st = _prep_conus(gdf_sites, gdf_basin, ba_src, states48)

# -----------------------------
# CLEAN NUMERICS
# -----------------------------
sites[MW_COL]   = pd.to_numeric(sites[MW_COL], errors="coerce").fillna(0.0)
sites[S1_COL_M3]= pd.to_numeric(sites[S1_COL_M3], errors="coerce").fillna(0.0)
sites[S2_COL_M3]= pd.to_numeric(sites[S2_COL_M3], errors="coerce").fillna(0.0)

# -----------------------------
# PANEL A: Scope 1 basin hotspots (paper-consistent)
# Hotspot = (S1 basin total > P50 among hosting basins) AND (stress>=3 in basin OR neighbor)
# -----------------------------
basins_with_sites = sites[SITE_BASIN_KEY].astype(str).unique()

s1_b = (
    sites.groupby(SITE_BASIN_KEY)[S1_COL_M3].sum()
    .rename("s1_m3_sum").reset_index()
    .rename(columns={SITE_BASIN_KEY: BASIN_KEY})
)

basin2 = basin.merge(s1_b, on=BASIN_KEY, how="left")
basin2["s1_m3_sum"] = basin2["s1_m3_sum"].fillna(0.0)
basin2["_hosts"] = basin2[BASIN_KEY].astype(str).isin(basins_with_sites.astype(str))

# High-S1 threshold among hosting basins
if basin2.loc[basin2["_hosts"], "s1_m3_sum"].notna().any():
    s1_thr = float(basin2.loc[basin2["_hosts"], "s1_m3_sum"].quantile(S1_HIGH_Q))
else:
    s1_thr = 0.0

basin2["_hi_s1"] = basin2["s1_m3_sum"] > s1_thr

# Neighbor-aware stress (paper)
if USE_NEIGHBOR_STRESS:
    basin2["_hi_stress_nb"] = _neighbor_aware_high_stress(basin2, STRESS_COL_BASIN, thr=STRESS_THR)
else:
    basin2["_hi_stress_nb"] = pd.to_numeric(basin2[STRESS_COL_BASIN], errors="coerce").fillna(0.0) >= STRESS_THR

# 4-class category:
# 0 neither; 1 stress only; 2 high S1 only; 3 hotspot
basin2["_cat_s1"] = basin2["_hi_s1"].astype(int) * 2 + basin2["_hi_stress_nb"].astype(int)

# -----------------------------
# PANEL B: Scope 2 BA hotspot typology (paper-consistent)
# Hotspot = high S2 (top quartile among hosting BAs) AND fossil-heavy AND mean stress>=3
# Prefer gdf_ba_l flags; fallback computes minimal hi_s2 + mean stress only (no fossil)
# -----------------------------
ba2 = ba_layer.copy()

if using_ba_l:
    # Use precomputed flags (already aligned to your data engineering)
    ba2["_hi_s2"]     = ba2["_high_s2"].astype(bool)
    ba2["_hi_stress"] = ba2["_high_stress_mean"].astype(bool)
    ba2["_hi_fossil"] = ba2["_fossil_heavy"].astype(bool)
else:
    # Fallback: compute BA totals + mean stress from sites; cannot apply fossil criterion
    if STRESS_COL_SITE not in sites.columns:
        raise KeyError(
            f"`{STRESS_COL_SITE}` missing in gdf_sites; needed to compute BA mean stress "
            "(fallback BA typology)."
        )
    sites[STRESS_COL_SITE] = pd.to_numeric(sites[STRESS_COL_SITE], errors="coerce").fillna(0.0)

    # Aggregate to BA
    r_s2 = (
        sites.groupby(SITE_BA_KEY)[S2_COL_M3].sum()
        .rename("s2_m3_sum").reset_index()
        .rename(columns={SITE_BA_KEY: BA_KEY})
    )
    r_st = (
        sites.groupby(SITE_BA_KEY)[STRESS_COL_SITE].mean()
        .rename("stress_mean").reset_index()
        .rename(columns={SITE_BA_KEY: BA_KEY})
    )

    ba2 = ba2.merge(r_s2, on=BA_KEY, how="left").merge(r_st, on=BA_KEY, how="left")
    ba2[["s2_m3_sum", "stress_mean"]] = ba2[["s2_m3_sum", "stress_mean"]].fillna(0.0)

    # High-S2 threshold among hosting BAs (those with any S2 > 0)
    if (ba2["s2_m3_sum"] > 0).any():
        s2_thr = float(ba2.loc[ba2["s2_m3_sum"] > 0, "s2_m3_sum"].quantile(S2_HIGH_Q))
    else:
        s2_thr = 0.0

    ba2["_hi_s2"]     = ba2["s2_m3_sum"] >= s2_thr
    ba2["_hi_stress"] = ba2["mean_stress_mw"] >= MEAN_STRESS_THR
    ba2["_hi_fossil"] = False  # unknown
    print("WARNING: using gdf_ba fallback; fossil-heavy criterion not applied (no fossil share fields).")

# Collapse into 4 bins:
# 0 not high S2
# 1 high S2 only
# 2 high S2 + (one of fossil-heavy or mean-stress)
# 3 high S2 + fossil-heavy + mean-stress (Scope 2 hotspot)
ba2["_crit_other"] = (ba2["_hi_fossil"].astype(int) + ba2["_hi_stress"].astype(int))
ba2["_cat_s2"] = 0
ba2.loc[ba2["_hi_s2"] & (ba2["_crit_other"]==0), "_cat_s2"] = 1
ba2.loc[ba2["_hi_s2"] & (ba2["_crit_other"]==1), "_cat_s2"] = 2
ba2.loc[ba2["_hi_s2"] & (ba2["_crit_other"]>=2), "_cat_s2"] = 3

# -----------------------------
# PLOT
# -----------------------------
minx, miny, maxx, maxy = st.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

fig = plt.figure(figsize=(FIG_W, FIG_H))
gs = fig.add_gridspec(1, 2, left=0.04, right=0.985, top=0.92, bottom=0.10, wspace=0.06)
axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])

# ---- Panel A (basins)
basin2.plot(
    ax=axA,
    color=basin2["_cat_s1"].map(S1_COLORS),
    linewidth=0.15,
    edgecolor="white",
    zorder=1
)

# Stress overlay (visible hatch)
basin2[basin2["_hi_stress_nb"]].plot(
    ax=axA,
    facecolor="none",
    edgecolor=HATCH_EDGE,
    linewidth=0.0,
    hatch=HATCH,
    zorder=3
)

# Hotspot outline (strong)
basin2[basin2["_cat_s1"] == 3].boundary.plot(
    ax=axA, color=HOTSPOT_EDGE, linewidth=S1_HOT_LW, zorder=5
)

# State outline
st.boundary.plot(ax=axA, color=STATE_EDGE, linewidth=STATE_LW, zorder=10)

axA.set_title(
    "a. Scope 1 basin hotspots (baseline)\n"
    f"High Scope 1 > P{int(S1_HIGH_Q*100)} among hosting basins; "
    f"stress = Aqueduct ≥{STRESS_THR:.0f} (basin or neighbor)",
    loc="center"
)
axA.set_axis_off()
axA.set_xlim(minx - padx, maxx + padx)
axA.set_ylim(miny - pady, maxy + pady)
axA.set_aspect("equal")

axA.legend(
    handles=[
        Patch(facecolor=S1_COLORS[3], edgecolor="black", label="Hotspot (high Scope 1 + high stress)"),
        Patch(facecolor=S1_COLORS[2], edgecolor="white", label="High Scope 1 only"),
        Patch(facecolor=S1_COLORS[1], edgecolor="white", label="High stress only"),
        Patch(facecolor=S1_COLORS[0], edgecolor="white", label="Neither"),
        Patch(facecolor="white", edgecolor=HATCH_EDGE, hatch=HATCH, label=f"Stress overlay (≥{STRESS_THR:.0f})"),
    ],
    loc="lower left",
    frameon=True,
    fontsize=9
)

# ---- Panel B (BAs)
ba2.plot(
    ax=axB,
    color=ba2["_cat_s2"].map(S2_COLORS),
    linewidth=0.25,
    edgecolor="white",
    zorder=1
)

# Triple-intersection hotspot border (very evident)
ba2[ba2["_cat_s2"] == 3].boundary.plot(
    ax=axB,
    color=HOTSPOT_EDGE,
    linewidth=S2_HOT_LW,
    zorder=6
)

# State outline
st.boundary.plot(ax=axB, color=STATE_EDGE, linewidth=STATE_LW, zorder=10)

axB.set_title(
    "b. Scope 2 BA hotspot typology (baseline)\n"
    f"High Scope 2 = top quartile; fossil-heavy >{int(FOSSIL_THR*100)}%; mean stress ≥{MEAN_STRESS_THR:.0f}",
    loc="center"
)
axB.set_axis_off()
axB.set_xlim(minx - padx, maxx + padx)
axB.set_ylim(miny - pady, maxy + pady)
axB.set_aspect("equal")

axB.legend(
    handles=[
        Patch(facecolor=S2_COLORS[3], edgecolor="black", label="Scope 2 hotspot (high S2 + fossil-heavy + high stress)"),
        Patch(facecolor=S2_COLORS[2], edgecolor="white", label="High S2 + one criterion"),
        Patch(facecolor=S2_COLORS[1], edgecolor="white", label="High S2 only"),
        Patch(facecolor=S2_COLORS[0], edgecolor="white", label="Not high Scope 2"),
    ],
    loc="lower left",
    frameon=True,
    fontsize=9
)

note = "BA fossil share/typology from gdf_ba_l" if using_ba_l else "Note: fossil-heavy criterion not applied (fallback BA layer)"
fig.text(0.5, 0.04, note, ha="center", va="center", fontsize=9)

# Export PDF only (per your preference) + show in notebook
plt.savefig(OUT_PDF, bbox_inches="tight")
plt.show()
print("Saved:", OUT_PDF)

In [ ]:
# =========================
# FIGURE 2: HOTSPOT ATLAS (BEST PRACTICE)
# Panel A: Scope 1 basin hotspots (YOUR ORIGINAL STYLE: 4-class + hatch + bold outline)
# Panel B: Scope 2 BA typology (FULL 8 combinations A,B,C)
# - Always shows plot in notebook
# - Exports PDF only
# - Centralized "threshold knobs" at the top
# - Uses gdf_ba_l (required for fossil share); prefers mean_stress_mw if present
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# -----------------------------
# OUTPUT
# -----------------------------
OUTDIR = "images"
os.makedirs(OUTDIR, exist_ok=True)
OUT_PDF = os.path.join(OUTDIR, "Fig2_HotspotAtlas_S1style_S2_8combos_base.pdf")

# -----------------------------
# THRESHOLD KNOBS (EDIT HERE)
# -----------------------------
# Scope 1 basin hotspot (paper-consistent):
S1_HIGH_Q = 0.50              # 0.50 = median among basins hosting sites
S1_STRESS_THR = 3.0           # Aqueduct baseline stress threshold
S1_USE_NEIGHBOR_STRESS = True # basin OR touching neighbor

# Scope 2 BA typology (A,B,C):
S2_QUANTILE_THR = 0.50        # A: high S2 cutoff among BAs hosting activity
FOSSIL_THR      = 0.50        # B: fossil-heavy if coal+gas share > threshold
STRESS_THR      = 3.0         # C: high stress if BA mean stress >= threshold

# Hosting definition for S2 quantile:
S2_HOST_DEF = "s2_m3_sum>0"   # or "n_sites>0" if you prefer and you have n_sites

# -----------------------------
# STYLE
# -----------------------------
FIG_W, FIG_H = 14.5, 7.2

# Panel A palette (fixed, readable)
S1_COLORS = {0:"#f0f0f0", 1:"#bdbdbd", 2:"#9ecae1", 3:"#de2d26"}  # neither, stress-only, high-S1-only, hotspot

# Panel B palette (8-class)
S2_COLORS_8 = {
    0: "#f0f0f0",
    1: "#bdbdbd",
    2: "#d9d9d9",
    3: "#969696",
    4: "#9ecae1",
    5: "#6baed6",
    6: "#fc9272",
    7: "#de2d26",
}

STATE_EDGE = "#111111"
STATE_LW   = 0.6

HOTSPOT_EDGE = "#000000"
S1_HOT_LW = 1.25
S2_HOT_LW = 2.35

# Stress overlay hatch (make it visible)
HATCH = "////"
HATCH_EDGE = "#2b2b2b"

# -----------------------------
# REQUIRE INPUTS / HELPERS
# -----------------------------
# expects your notebook already has:
# - gdf_sites, gdf_basin, gdf_ba_l, states48
# - SITE_BASIN_KEY, BASIN_KEY, SITE_BA_KEY, BA_KEY, MW_COL, S1_COL_M3, S2_COL_M3
# - STRESS_COL_BASIN (e.g., "bws_01_score")
# - _assert_has(), _prep_conus(), _neighbor_aware_high_stress()

if "_assert_has" not in globals():
    raise NameError("Expected helper `_assert_has(df, cols, name)` in globals().")
if "_prep_conus" not in globals():
    raise NameError("Expected helper `_prep_conus(gdf_sites, gdf_basin, gdf_ba, states48)` in globals().")
if "_neighbor_aware_high_stress" not in globals():
    raise NameError("Expected helper `_neighbor_aware_high_stress(basin_gdf, stress_col, thr=...)` in globals().")

_assert_has(gdf_sites, [SITE_BASIN_KEY, SITE_BA_KEY, MW_COL, S1_COL_M3, S2_COL_M3, "geometry"], "gdf_sites")
_assert_has(gdf_basin, [BASIN_KEY, "geometry", STRESS_COL_BASIN], "gdf_basin")
_assert_has(states48,  ["geometry"], "states48")

if "gdf_ba_l" not in globals():
    raise NameError("Expected `gdf_ba_l` in globals() for Scope 2 8-combo typology.")
_assert_has(gdf_ba_l, [BA_KEY, "geometry", "s2_m3_sum", "_fossil_share"], "gdf_ba_l")

# choose stress field on BA layer
if "mean_stress_mw" in gdf_ba_l.columns:
    STRESS_COL_BA = "mean_stress_mw"
elif "mean_stress" in gdf_ba_l.columns:
    STRESS_COL_BA = "mean_stress"
else:
    raise KeyError("No stress summary found on gdf_ba_l. Add `mean_stress_mw` (recommended) or `mean_stress`.")

# Prep CONUS (clip + CRS harmonization handled by your helper)
sites, basin, ba_layer, st = _prep_conus(gdf_sites, gdf_basin, gdf_ba_l, states48)

# -----------------------------
# CLEAN NUMERICS
# -----------------------------
sites[MW_COL]    = pd.to_numeric(sites[MW_COL], errors="coerce").fillna(0.0)
sites[S1_COL_M3] = pd.to_numeric(sites[S1_COL_M3], errors="coerce").fillna(0.0)
sites[S2_COL_M3] = pd.to_numeric(sites[S2_COL_M3], errors="coerce").fillna(0.0)

ba2 = ba_layer.copy()
ba2["s2_m3_sum"]      = pd.to_numeric(ba2["s2_m3_sum"], errors="coerce").fillna(0.0)
ba2["_fossil_share"]  = pd.to_numeric(ba2["_fossil_share"], errors="coerce").fillna(0.0)
ba2[STRESS_COL_BA]    = pd.to_numeric(ba2[STRESS_COL_BA], errors="coerce").fillna(0.0)

# -----------------------------
# PANEL A: Scope 1 basin hotspots (YOUR STYLE)
# Hotspot = (S1 basin total > P50 among hosting basins) AND (stress>=thr in basin OR neighbor)
# -----------------------------
basins_with_sites = sites[SITE_BASIN_KEY].astype(str).unique()

s1_b = (
    sites.groupby(SITE_BASIN_KEY)[S1_COL_M3].sum()
    .rename("s1_m3_sum").reset_index()
    .rename(columns={SITE_BASIN_KEY: BASIN_KEY})
)

basin2 = basin.merge(s1_b, on=BASIN_KEY, how="left")
basin2["s1_m3_sum"] = basin2["s1_m3_sum"].fillna(0.0)
basin2["_hosts"] = basin2[BASIN_KEY].astype(str).isin(basins_with_sites.astype(str))

if basin2.loc[basin2["_hosts"], "s1_m3_sum"].notna().any():
    s1_thr = float(basin2.loc[basin2["_hosts"], "s1_m3_sum"].quantile(S1_HIGH_Q))
else:
    s1_thr = 0.0

basin2["_hi_s1"] = basin2["s1_m3_sum"] > s1_thr

if S1_USE_NEIGHBOR_STRESS:
    basin2["_hi_stress_nb"] = _neighbor_aware_high_stress(basin2, STRESS_COL_BASIN, thr=S1_STRESS_THR)
else:
    basin2["_hi_stress_nb"] = (
        pd.to_numeric(basin2[STRESS_COL_BASIN], errors="coerce").fillna(0.0) >= S1_STRESS_THR
    )

# 4-class: 0 neither; 1 stress only; 2 high S1 only; 3 hotspot
basin2["_cat_s1"] = basin2["_hi_s1"].astype(int) * 2 + basin2["_hi_stress_nb"].astype(int)

# -----------------------------
# PANEL B: Scope 2 8-combo typology (A,B,C)
# A = high S2 (quantile among hosting BAs)
# B = fossil-heavy (share > threshold)
# C = high stress exposure (mean stress >= threshold)
# -----------------------------
if S2_HOST_DEF == "n_sites>0" and "n_sites" not in ba2.columns:
    raise KeyError("S2_HOST_DEF='n_sites>0' but gdf_ba_l has no `n_sites` column. Use s2_m3_sum>0 or add n_sites.")

if S2_HOST_DEF == "n_sites>0":
    host_mask = ba2["n_sites"] > 0
else:
    host_mask = ba2["s2_m3_sum"] > 0

s2_thr = float(ba2.loc[host_mask, "s2_m3_sum"].quantile(S2_QUANTILE_THR)) if host_mask.any() else 0.0

ba2["_A_high_s2"] = ba2["s2_m3_sum"] >= s2_thr
ba2["_B_fossil"]  = ba2["_fossil_share"] > FOSSIL_THR
ba2["_C_stress"]  = ba2[STRESS_COL_BA] >= STRESS_THR

# bitmask: bit2=A, bit1=B, bit0=C => 0..7
ba2["_code_ABC"] = (
    ba2["_A_high_s2"].astype(int) * 4 +
    ba2["_B_fossil"].astype(int)  * 2 +
    ba2["_C_stress"].astype(int)  * 1
)

code_to_label = {
    0: "¬A ¬B ¬C  (low S2, not fossil-heavy, low stress)",
    1: "¬A ¬B  C  (low S2, not fossil-heavy, high stress)",
    2: "¬A  B ¬C  (low S2, fossil-heavy, low stress)",
    3: "¬A  B  C  (low S2, fossil-heavy, high stress)",
    4: " A ¬B ¬C  (high S2 only)",
    5: " A ¬B  C  (high S2 + high stress)",
    6: " A  B ¬C  (high S2 + fossil-heavy)",
    7: " A  B  C  (high S2 + fossil-heavy + high stress)  ← HOTSPOT",
}
ba2["_label_ABC"] = ba2["_code_ABC"].map(code_to_label)

# Print “which one?” table (counts + BA IDs per combo)
summary = (
    ba2.groupby(["_code_ABC", "_label_ABC"], dropna=False)
       .agg(
           n_BA=(BA_KEY, "count"),
           BA_list=(BA_KEY, lambda x: ", ".join(map(str, sorted(x.astype(str).unique()))))
       )
       .reset_index()
       .sort_values("_code_ABC")
)
print("\n=== Scope 2 typology: 8 combinations (A,B,C) ===")
print(f"A(high S2): quantile={S2_QUANTILE_THR} among hosting BAs ({S2_HOST_DEF}) -> s2_thr={s2_thr:.3e} m3/yr")
print(f"B(fossil-heavy): fossil_share > {FOSSIL_THR}")
print(f"C(high stress): {STRESS_COL_BA} >= {STRESS_THR}\n")
print(summary.to_string(index=False))

# -----------------------------
# PLOT (Panel A style preserved)
# -----------------------------
minx, miny, maxx, maxy = st.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

fig = plt.figure(figsize=(FIG_W, FIG_H))
gs = fig.add_gridspec(1, 2, left=0.04, right=0.985, top=0.92, bottom=0.10, wspace=0.06)
axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])

# ---- Panel A (Scope 1 basins) — YOUR STYLE
basin2.plot(
    ax=axA,
    color=basin2["_cat_s1"].map(S1_COLORS),
    linewidth=0.15,
    edgecolor="white",
    zorder=1
)

# Stress overlay (visible hatch)
basin2[basin2["_hi_stress_nb"]].plot(
    ax=axA,
    facecolor="none",
    edgecolor=HATCH_EDGE,
    linewidth=0.0,
    hatch=HATCH,
    zorder=3
)

# Hotspot outline (strong)
basin2[basin2["_cat_s1"] == 3].boundary.plot(
    ax=axA,
    color=HOTSPOT_EDGE,
    linewidth=S1_HOT_LW,
    zorder=5
)

st.boundary.plot(ax=axA, color=STATE_EDGE, linewidth=STATE_LW, zorder=10)

axA.set_title(
    "a. Scope 1 basin hotspots (baseline)\n"
    f"High Scope 1 > P{int(S1_HIGH_Q*100)} among hosting basins; "
    f"stress = Aqueduct ≥{S1_STRESS_THR:.0f} ({'basin or neighbor' if S1_USE_NEIGHBOR_STRESS else 'basin only'})",
    loc="center"
)
axA.set_axis_off()
axA.set_xlim(minx - padx, maxx + padx)
axA.set_ylim(miny - pady, maxy + pady)
axA.set_aspect("equal")

axA.legend(
    handles=[
        Patch(facecolor=S1_COLORS[3], edgecolor="black", label="Hotspot (high Scope 1 + high stress)"),
        Patch(facecolor=S1_COLORS[2], edgecolor="white", label="High Scope 1 only"),
        Patch(facecolor=S1_COLORS[1], edgecolor="white", label="High stress only"),
        Patch(facecolor=S1_COLORS[0], edgecolor="white", label="Neither"),
        Patch(facecolor="white", edgecolor=HATCH_EDGE, hatch=HATCH, label=f"Stress overlay (≥{S1_STRESS_THR:.0f})"),
    ],
    loc="lower left",
    frameon=True,
    fontsize=9
)

# ---- Panel B (Scope 2 8-combo)
ba2.plot(
    ax=axB,
    color=ba2["_code_ABC"].map(S2_COLORS_8),
    linewidth=0.30,
    edgecolor="white",
    zorder=1
)

# Outline the full hotspot (A&B&C)
ba2.loc[ba2["_code_ABC"] == 7].boundary.plot(
    ax=axB,
    color=HOTSPOT_EDGE,
    linewidth=S2_HOT_LW,
    zorder=6
)

# Optional: outline all A (high S2) a bit lighter (helpful cue)
ba2.loc[ba2["_A_high_s2"]].boundary.plot(
    ax=axB,
    color="#111111",
    linewidth=0.9,
    zorder=5
)

st.boundary.plot(ax=axB, color=STATE_EDGE, linewidth=STATE_LW, zorder=10)

axB.set_title(
    "b. Scope 2 BA typology (all 8 combinations)\n"
    "A=High S2 (upper tail), B=Fossil-heavy, C=High stress exposure",
    loc="center"
)
axB.set_axis_off()
axB.set_xlim(minx - padx, maxx + padx)
axB.set_ylim(miny - pady, maxy + pady)
axB.set_aspect("equal")

legend_order = [7, 6, 5, 4, 3, 2, 1, 0]
axB.legend(
    handles=[Patch(facecolor=S2_COLORS_8[c], edgecolor="white", label=code_to_label[c]) for c in legend_order],
    loc="lower left",
    frameon=True,
    fontsize=8
)

note = (
    f"S2 thresholds: A = s2_m3_sum ≥ P{int(S2_QUANTILE_THR*100)} among hosting BAs ({S2_HOST_DEF}) "
    f"(thr={s2_thr:.2e} m³/yr); B = fossil_share>{FOSSIL_THR}; C = {STRESS_COL_BA}≥{STRESS_THR}"
)
fig.text(0.5, 0.04, note, ha="center", va="center", fontsize=8)

# Export PDF only + show
plt.savefig(OUT_PDF, bbox_inches="tight")
plt.show()
print("Saved:", OUT_PDF)

In [ ]:
# =========================
# FIGURE 2: HOTSPOT ATLAS (S1 style preserved; S2 restyled to match)
# - Panel A: Scope 1 basin hotspots (unchanged)
# - Panel B: Scope 2 BA typology (8 combos) with S1-like coloring + hatch overlays
# - Legends below maps (no overlap)
# - No "A/B/C" wording in legend labels
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# -----------------------------
# OUTPUT
# -----------------------------
OUTDIR = "images"
os.makedirs(OUTDIR, exist_ok=True)
OUT_PDF = os.path.join(OUTDIR, "Fig2_HotspotAtlas_S1style_S2style_base.pdf")

# -----------------------------
# THRESHOLD KNOBS (EDIT HERE)
# -----------------------------
# Scope 1 basin hotspot:
S1_HIGH_Q = 0.50
S1_STRESS_THR = 3.0
S1_USE_NEIGHBOR_STRESS = True

# Scope 2 BA typology:
S2_QUANTILE_THR = 0.50
FOSSIL_THR      = 0.50
STRESS_THR      = 3.0
S2_HOST_DEF     = "s2_m3_sum>0"   # or "n_sites>0" if you have n_sites

# -----------------------------
# STYLE
# -----------------------------
FIG_W, FIG_H = 14.5, 7.2

# Panel A palette (unchanged)
S1_COLORS = {0:"#f0f0f0", 1:"#bdbdbd", 2:"#9ecae1", 3:"#de2d26"}

# Panel B: S1-like palette (limited & consistent)
# base fills depend on high-S2 + fossil, hotspot gets red
S2_BASE_COLORS = {
    "low_s2":          "#f0f0f0",  # light grey
    "high_s2_only":    "#9ecae1",  # light blue
    "high_s2_fossil":  "#fc9272",  # light orange-ish (keeps your earlier cue)
    "hotspot":         "#de2d26",  # red
}

STATE_EDGE = "#111111"
STATE_LW   = 0.6

HOTSPOT_EDGE = "#000000"
S1_HOT_LW = 1.25
S2_HOT_LW = 2.35

# Hatch overlays (same “purpose” as panel A: stress exposure shown with diagonal hatch)
HATCH_STRESS = "////"
HATCH_FOSSIL = "\\\\\\\\"
HATCH_EDGE   = "#2b2b2b"

# Legend layout (below)
LEGEND_FONTSIZE_A = 9
LEGEND_FONTSIZE_B = 8

# -----------------------------
# REQUIRE INPUTS / HELPERS
# -----------------------------
if "_assert_has" not in globals():
    raise NameError("Expected helper `_assert_has(df, cols, name)` in globals().")
if "_prep_conus" not in globals():
    raise NameError("Expected helper `_prep_conus(gdf_sites, gdf_basin, gdf_ba, states48)` in globals().")
if "_neighbor_aware_high_stress" not in globals():
    raise NameError("Expected helper `_neighbor_aware_high_stress(basin_gdf, stress_col, thr=...)` in globals().")

_assert_has(gdf_sites, [SITE_BASIN_KEY, SITE_BA_KEY, MW_COL, S1_COL_M3, S2_COL_M3, "geometry"], "gdf_sites")
_assert_has(gdf_basin, [BASIN_KEY, "geometry", STRESS_COL_BASIN], "gdf_basin")
_assert_has(states48,  ["geometry"], "states48")

if "gdf_ba_l" not in globals():
    raise NameError("Expected `gdf_ba_l` in globals() for Scope 2 plotting.")
_assert_has(gdf_ba_l, [BA_KEY, "geometry", "s2_m3_sum", "_fossil_share"], "gdf_ba_l")

# pick BA stress field
if "mean_stress_mw" in gdf_ba_l.columns:
    STRESS_COL_BA = "mean_stress_mw"
elif "mean_stress" in gdf_ba_l.columns:
    STRESS_COL_BA = "mean_stress"
else:
    raise KeyError("No stress summary found on gdf_ba_l. Add `mean_stress_mw` (recommended) or `mean_stress`.")

# Prep CONUS
sites, basin, ba_layer, st = _prep_conus(gdf_sites, gdf_basin, gdf_ba_l, states48)

# -----------------------------
# CLEAN NUMERICS
# -----------------------------
sites[MW_COL]    = pd.to_numeric(sites[MW_COL], errors="coerce").fillna(0.0)
sites[S1_COL_M3] = pd.to_numeric(sites[S1_COL_M3], errors="coerce").fillna(0.0)
sites[S2_COL_M3] = pd.to_numeric(sites[S2_COL_M3], errors="coerce").fillna(0.0)

ba2 = ba_layer.copy()
ba2["s2_m3_sum"]     = pd.to_numeric(ba2["s2_m3_sum"], errors="coerce").fillna(0.0)
ba2["_fossil_share"] = pd.to_numeric(ba2["_fossil_share"], errors="coerce").fillna(0.0)
ba2[STRESS_COL_BA]   = pd.to_numeric(ba2[STRESS_COL_BA], errors="coerce").fillna(0.0)

# -----------------------------
# PANEL A: Scope 1 basin hotspots (UNCHANGED)
# -----------------------------
basins_with_sites = sites[SITE_BASIN_KEY].astype(str).unique()

s1_b = (
    sites.groupby(SITE_BASIN_KEY)[S1_COL_M3].sum()
    .rename("s1_m3_sum").reset_index()
    .rename(columns={SITE_BASIN_KEY: BASIN_KEY})
)

basin2 = basin.merge(s1_b, on=BASIN_KEY, how="left")
basin2["s1_m3_sum"] = basin2["s1_m3_sum"].fillna(0.0)
basin2["_hosts"] = basin2[BASIN_KEY].astype(str).isin(basins_with_sites.astype(str))

s1_thr = float(basin2.loc[basin2["_hosts"], "s1_m3_sum"].quantile(S1_HIGH_Q)) if basin2["_hosts"].any() else 0.0
basin2["_hi_s1"] = basin2["s1_m3_sum"] > s1_thr

if S1_USE_NEIGHBOR_STRESS:
    basin2["_hi_stress_nb"] = _neighbor_aware_high_stress(basin2, STRESS_COL_BASIN, thr=S1_STRESS_THR)
else:
    basin2["_hi_stress_nb"] = pd.to_numeric(basin2[STRESS_COL_BASIN], errors="coerce").fillna(0.0) >= S1_STRESS_THR

basin2["_cat_s1"] = basin2["_hi_s1"].astype(int) * 2 + basin2["_hi_stress_nb"].astype(int)

# -----------------------------
# PANEL B: Scope 2 8-combo flags (same logic, new styling)
# -----------------------------
if S2_HOST_DEF == "n_sites>0" and "n_sites" not in ba2.columns:
    raise KeyError("S2_HOST_DEF='n_sites>0' but gdf_ba_l has no `n_sites` column. Use s2_m3_sum>0 or add n_sites.")

host_mask = (ba2["n_sites"] > 0) if S2_HOST_DEF == "n_sites>0" else (ba2["s2_m3_sum"] > 0)
s2_thr = float(ba2.loc[host_mask, "s2_m3_sum"].quantile(S2_QUANTILE_THR)) if host_mask.any() else 0.0

ba2["_hi_s2"]     = ba2["s2_m3_sum"] >= s2_thr
ba2["_hi_fossil"] = ba2["_fossil_share"] > FOSSIL_THR
ba2["_hi_stress"] = ba2[STRESS_COL_BA] >= STRESS_THR

# bitmask 0..7 (kept for your summaries)
ba2["_code_ABC"] = ba2["_hi_s2"].astype(int)*4 + ba2["_hi_fossil"].astype(int)*2 + ba2["_hi_stress"].astype(int)*1

# Fill color: S1-like logic (few colors)
# - hotspot (hi_s2 & fossil & stress) -> red
# - hi_s2 & fossil -> orange-ish
# - hi_s2 only (regardless of stress; stress shown by hatch) -> blue
# - else -> grey
def _s2_fill(row):
    if row["_hi_s2"] and row["_hi_fossil"] and row["_hi_stress"]:
        return S2_BASE_COLORS["hotspot"]
    if row["_hi_s2"] and row["_hi_fossil"]:
        return S2_BASE_COLORS["high_s2_fossil"]
    if row["_hi_s2"]:
        return S2_BASE_COLORS["high_s2_only"]
    return S2_BASE_COLORS["low_s2"]

ba2["_fill_s2"] = ba2.apply(_s2_fill, axis=1)

# Human legend labels (no A/B/C)
code_to_label_plain = {
    0: "Low Scope 2; not fossil-heavy; low stress",
    1: "Low Scope 2; not fossil-heavy; high stress",
    2: "Low Scope 2; fossil-heavy; low stress",
    3: "Low Scope 2; fossil-heavy; high stress",
    4: "High Scope 2 only",
    5: "High Scope 2 + high stress",
    6: "High Scope 2 + fossil-heavy",
    7: "Scope 2 hotspot (high Scope 2 + fossil-heavy + high stress)",
}

# Print “which one?” table (optional but useful)
summary = (
    ba2.groupby("_code_ABC", dropna=False)
       .agg(
           n_BA=(BA_KEY, "count"),
           BA_list=(BA_KEY, lambda x: ", ".join(map(str, sorted(x.astype(str).unique()))))
       )
       .reset_index()
       .sort_values("_code_ABC")
)
summary["label"] = summary["_code_ABC"].map(code_to_label_plain)
print("\n=== Scope 2 typology: 8 combinations (plain labels) ===")
print(f"High Scope 2 threshold: P{int(S2_QUANTILE_THR*100)} among hosting BAs ({S2_HOST_DEF}) -> {s2_thr:.3e} m3/yr")
print(f"Fossil-heavy: fossil_share > {FOSSIL_THR}")
print(f"High stress: {STRESS_COL_BA} >= {STRESS_THR}\n")
print(summary[["_code_ABC","label","n_BA","BA_list"]].to_string(index=False))

# -----------------------------
# PLOT (legends below, no overlap)
# -----------------------------
minx, miny, maxx, maxy = st.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

fig = plt.figure(figsize=(FIG_W, FIG_H))

# more bottom space reserved for legends
gs = fig.add_gridspec(
    1, 2,
    left=0.04, right=0.985,
    top=0.92, bottom=0.22,
    wspace=0.06
)
axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])

# ---- Panel A (Scope 1 basins)
basin2.plot(
    ax=axA,
    color=basin2["_cat_s1"].map(S1_COLORS),
    linewidth=0.15,
    edgecolor="white",
    zorder=1
)

basin2[basin2["_hi_stress_nb"]].plot(
    ax=axA,
    facecolor="none",
    edgecolor=HATCH_EDGE,
    linewidth=0.0,
    hatch=HATCH_STRESS,
    zorder=3
)

basin2[basin2["_cat_s1"] == 3].boundary.plot(
    ax=axA, color=HOTSPOT_EDGE, linewidth=S1_HOT_LW, zorder=5
)

st.boundary.plot(ax=axA, color=STATE_EDGE, linewidth=STATE_LW, zorder=10)

axA.set_title(
    "a. Scope 1 basin hotspots (baseline)\n"
    f"High Scope 1 > P{int(S1_HIGH_Q*100)} among hosting basins; "
    f"stress ≥{S1_STRESS_THR:.0f} ({'basin or neighbor' if S1_USE_NEIGHBOR_STRESS else 'basin only'})",
    loc="center"
)
axA.set_axis_off()
axA.set_xlim(minx - padx, maxx + padx)
axA.set_ylim(miny - pady, maxy + pady)
axA.set_aspect("equal")

# ---- Panel B (Scope 2 BAs) — S1-like styling
ba2.plot(
    ax=axB,
    color=ba2["_fill_s2"],
    linewidth=0.30,
    edgecolor="white",
    zorder=1
)

# Stress overlay hatch (same purpose as Panel A)
ba2[ba2["_hi_stress"]].plot(
    ax=axB,
    facecolor="none",
    edgecolor=HATCH_EDGE,
    linewidth=0.0,
    hatch=HATCH_STRESS,
    zorder=3
)

# Fossil overlay hatch (different diagonal)
ba2[ba2["_hi_fossil"]].plot(
    ax=axB,
    facecolor="none",
    edgecolor=HATCH_EDGE,
    linewidth=0.0,
    hatch=HATCH_FOSSIL,
    zorder=4
)

# Bold outline for full hotspot (triple intersection)
ba2.loc[ba2["_code_ABC"] == 7].boundary.plot(
    ax=axB, color=HOTSPOT_EDGE, linewidth=S2_HOT_LW, zorder=6
)

st.boundary.plot(ax=axB, color=STATE_EDGE, linewidth=STATE_LW, zorder=10)

axB.set_title(
    "b. Scope 2 balancing-authority typology (baseline)\n"
    "Color = high Scope 2 and fossil intensity; hatching = stress/fossil overlays",
    loc="center"
)
axB.set_axis_off()
axB.set_xlim(minx - padx, maxx + padx)
axB.set_ylim(miny - pady, maxy + pady)
axB.set_aspect("equal")

# -----------------------------
# LEGENDS BELOW (no overlap)
# -----------------------------
# Panel A legend (below left map)
handles_A = [
    Patch(facecolor=S1_COLORS[3], edgecolor="black", label="Hotspot (high Scope 1 + high stress)"),
    Patch(facecolor=S1_COLORS[2], edgecolor="white", label="High Scope 1 only"),
    Patch(facecolor=S1_COLORS[1], edgecolor="white", label="High stress only"),
    Patch(facecolor=S1_COLORS[0], edgecolor="white", label="Neither"),
    Patch(facecolor="white", edgecolor=HATCH_EDGE, hatch=HATCH_STRESS, label=f"Stress overlay (≥{S1_STRESS_THR:.0f})"),
]
axA.legend(
    handles=handles_A,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.10),
    ncol=2,
    frameon=True,
    fontsize=LEGEND_FONTSIZE_A
)

# Panel B legend (below right map) — plain-language only
handles_B = [
    Patch(facecolor=S2_BASE_COLORS["hotspot"], edgecolor="black",
          label="Scope 2 hotspot (high Scope 2 + fossil-heavy + high stress)"),
    Patch(facecolor=S2_BASE_COLORS["high_s2_fossil"], edgecolor="white",
          label="High Scope 2 + fossil-heavy"),
    Patch(facecolor=S2_BASE_COLORS["high_s2_only"], edgecolor="white",
          label="High Scope 2 (not fossil-heavy)"),
    Patch(facecolor=S2_BASE_COLORS["low_s2"], edgecolor="white",
          label="Low Scope 2"),
    Patch(facecolor="white", edgecolor=HATCH_EDGE, hatch=HATCH_STRESS,
          label=f"High stress overlay (≥{STRESS_THR:.0f})"),
    Patch(facecolor="white", edgecolor=HATCH_EDGE, hatch=HATCH_FOSSIL,
          label=f"Fossil-heavy overlay (fossil share > {FOSSIL_THR:.2f})"),
]
axB.legend(
    handles=handles_B,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.10),
    ncol=2,
    frameon=True,
    fontsize=LEGEND_FONTSIZE_B
)

note = (
    f"Scope 2 high threshold: s2_m3_sum ≥ P{int(S2_QUANTILE_THR*100)} among hosting BAs ({S2_HOST_DEF}) "
    f"(thr={s2_thr:.2e} m³/yr); stress metric: {STRESS_COL_BA}"
)
fig.text(0.5, 0.06, note, ha="center", va="center", fontsize=8)

# Export PDF only + show
plt.savefig(OUT_PDF, bbox_inches="tight")
plt.show()
print("Saved:", OUT_PDF)

In [ ]:
# =========================
# SCOPE 2 HOTSPOTS: FULL 8-COMBO TYPOLOGY (A,B,C)
# A = high Scope 2 (upper tail)
# B = fossil-heavy (coal+gas share > threshold)
# C = high stress exposure (mean stress >= threshold)
# =========================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

OUTDIR = "images"
os.makedirs(OUTDIR, exist_ok=True)

# -----------------------------
# Thresholds (EDIT HERE)
# -----------------------------
S2_QUANTILE_THR = 0.50     # "high S2" cutoff among BAs with hyperscale activity
FOSSIL_THR      = 0.50     # fossil-heavy if coal+gas share > this
STRESS_THR      = 3.0      # high stress if mean stress >= this

# -----------------------------
# Choose BA layer
# -----------------------------
if "gdf_ba_l" in globals():
    ba2 = gdf_ba_l.copy()
else:
    raise NameError("Expected `gdf_ba_l` in globals() for Scope 2 typology.")

# Required columns in gdf_ba_l (from your note)
need = ["subregion_id", "geometry", "s2_m3_sum", "_fossil_share"]
missing = [c for c in need if c not in ba2.columns]
if missing:
    raise KeyError(f"Missing required columns in gdf_ba_l: {missing}")

# Stress column: pick one
# Preferred: capacity-weighted facility stress you computed (mean_stress_mw)
# Fallback: mean_stress (if present)
if "mean_stress_mw" in ba2.columns:
    STRESS_COL = "mean_stress_mw"
elif "mean_stress" in ba2.columns:
    STRESS_COL = "mean_stress"
else:
    raise KeyError("No stress summary found on gdf_ba_l. Add `mean_stress_mw` (recommended) or `mean_stress`.")

# Numeric cleanup
ba2["s2_m3_sum"] = pd.to_numeric(ba2["s2_m3_sum"], errors="coerce").fillna(0.0)
ba2["_fossil_share"] = pd.to_numeric(ba2["_fossil_share"], errors="coerce").fillna(0.0)
ba2[STRESS_COL] = pd.to_numeric(ba2[STRESS_COL], errors="coerce").fillna(0.0)

# -----------------------------
# Define A/B/C flags
# -----------------------------
# define "hosting hyperscale activity" as s2_m3_sum > 0 (or use n_sites>0 if you prefer)
host_mask = ba2["s2_m3_sum"] > 0

# High S2 threshold among hosting BAs
if host_mask.any():
    s2_thr = float(ba2.loc[host_mask, "s2_m3_sum"].quantile(S2_QUANTILE_THR))
else:
    s2_thr = 0.0

ba2["_A_high_s2"] = ba2["s2_m3_sum"] >= s2_thr
ba2["_B_fossil"]  = ba2["_fossil_share"] > FOSSIL_THR
ba2["_C_stress"]  = ba2[STRESS_COL] >= STRESS_THR

# -----------------------------
# 8-class code (bitmask) + label
# bit2=A, bit1=B, bit0=C  => 0..7
# -----------------------------
ba2["_code_ABC"] = (
    ba2["_A_high_s2"].astype(int) * 4 +
    ba2["_B_fossil"].astype(int)  * 2 +
    ba2["_C_stress"].astype(int)  * 1
)

code_to_label = {
    0: "¬A ¬B ¬C  (low S2, not fossil-heavy, low stress)",
    1: "¬A ¬B  C  (low S2, not fossil-heavy, high stress)",
    2: "¬A  B ¬C  (low S2, fossil-heavy, low stress)",
    3: "¬A  B  C  (low S2, fossil-heavy, high stress)",
    4: " A ¬B ¬C  (high S2 only)",
    5: " A ¬B  C  (high S2 + high stress)",
    6: " A  B ¬C  (high S2 + fossil-heavy)",
    7: " A  B  C  (high S2 + fossil-heavy + high stress)  ← HOTSPOT",
}
ba2["_label_ABC"] = ba2["_code_ABC"].map(code_to_label)

# -----------------------------
# Quick “which one?” summary (counts + BA IDs)
# -----------------------------
summary = (
    ba2.groupby(["_code_ABC", "_label_ABC"], dropna=False)
       .agg(
           n_BA=("subregion_id", "count"),
           BA_list=("subregion_id", lambda x: ", ".join(map(str, sorted(x))))
       )
       .reset_index()
       .sort_values("_code_ABC")
)
print("\n=== Scope 2 typology: 8 combinations (A,B,C) ===")
print(f"A(high S2) threshold: quantile={S2_QUANTILE_THR} -> s2_thr={s2_thr:.3e} m3/yr")
print(f"B(fossil-heavy) threshold: fossil_share > {FOSSIL_THR}")
print(f"C(high stress) threshold: {STRESS_COL} >= {STRESS_THR}\n")
print(summary.to_string(index=False))

# -----------------------------
# Plot map (8 colors + clear legend)
# -----------------------------
# A sensible palette: greys for ¬A classes, stronger colors for A classes, black outline for code=7.
colors = {
    0: "#f0f0f0",
    1: "#bdbdbd",
    2: "#d9d9d9",
    3: "#969696",
    4: "#9ecae1",
    5: "#6baed6",
    6: "#fc9272",
    7: "#de2d26",
}

fig = plt.figure(figsize=(9.5, 6.4))
ax = fig.add_subplot(1, 1, 1)

ba2.plot(
    ax=ax,
    color=ba2["_code_ABC"].map(colors),
    linewidth=0.35, edgecolor="white", zorder=1
)

# Strong outline for the full hotspot (A & B & C)
ba2.loc[ba2["_code_ABC"] == 7].boundary.plot(ax=ax, color="#000000", linewidth=2.4, zorder=5)

# Optional: also outline all A (high S2) regions lightly
ba2.loc[ba2["_A_high_s2"]].boundary.plot(ax=ax, color="#111111", linewidth=0.9, zorder=4)

# States outline if available
if "states48" in globals():
    states48.boundary.plot(ax=ax, color="#111111", linewidth=0.6, zorder=10)

ax.set_title(
    "Scope 2 balancing-authority typology (all 8 combinations)\n"
    "A=High S2 (upper tail), B=Fossil-heavy, C=High stress exposure",
    loc="center"
)
ax.set_axis_off()
ax.set_aspect("equal")

# Legend (explicit combos)
legend_order = [7, 6, 5, 4, 3, 2, 1, 0]
handles = [
    Patch(facecolor=colors[c], edgecolor="white", label=code_to_label[c])
    for c in legend_order
]
ax.legend(handles=handles, loc="lower left", frameon=True, fontsize=8)

note = (
    f"A: s2_m3_sum ≥ P{int(S2_QUANTILE_THR*100)} among hosting BAs (thr={s2_thr:.2e} m³/yr); "
    f"B: fossil_share>{FOSSIL_THR}; C: {STRESS_COL}≥{STRESS_THR}"
)
fig.text(0.5, 0.02, note, ha="center", va="center", fontsize=8)

out_pdf = os.path.join(OUTDIR, "Fig_S2_Typology_8combos_base.pdf")
plt.savefig(out_pdf, bbox_inches="tight")
plt.show()
print("Saved:", out_pdf)

In [ ]:
# =========================
# FIGURE 2: HOTSPOT ATLAS (REVISED: uses gdf_ba_l)
# =========================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

OUTDIR = "images"
os.makedirs(OUTDIR, exist_ok=True)

# --- require these exist ---
_assert_has(gdf_sites, [SITE_BASIN_KEY, SITE_BA_KEY, MW_COL, S1_COL_M3, S2_COL_M3], "gdf_sites")
_assert_has(gdf_basin, [BASIN_KEY, STRESS_COL_BASIN, "geometry"], "gdf_basin")
_assert_has(states48,  ["geometry"], "states48")

# pick the correct BA layer (this is the key change)
ba_src = pick_ba_layer(gdf_ba, gdf_ba_l if "gdf_ba_l" in globals() else None)

# must have geometry + BA key
_assert_has(ba_src, [BA_KEY, "geometry"], "gdf_ba_l (or fallback gdf_ba)")

# if we are indeed using gdf_ba_l, enforce those columns exist
using_ba_l = ("gdf_ba_l" in globals()) and (ba_src is globals().get("gdf_ba_l"))
if using_ba_l:
    _assert_has(
        ba_src,
        ["_fossil_share", "_fossil_heavy", "_high_s2", "_high_stress_mean"],
        "gdf_ba_l"
    )

sites, basin, ba_layer, st = _prep_conus(gdf_sites, gdf_basin, ba_src, states48)

# numeric cleanup
sites[MW_COL] = pd.to_numeric(sites[MW_COL], errors="coerce").fillna(0.0)
sites[S1_COL_M3] = pd.to_numeric(sites[S1_COL_M3], errors="coerce").fillna(0.0)
sites[S2_COL_M3] = pd.to_numeric(sites[S2_COL_M3], errors="coerce").fillna(0.0)

# -----------------------------
# Panel A: Scope 1 basin hotspots (unchanged definition)
# -----------------------------
basins_with_sites = sites[SITE_BASIN_KEY].astype(str).unique()

s1_b = (sites.groupby(SITE_BASIN_KEY)[S1_COL_M3].sum()
        .rename("s1_m3_sum").reset_index()
        .rename(columns={SITE_BASIN_KEY: BASIN_KEY}))

basin2 = basin.merge(s1_b, on=BASIN_KEY, how="left")
basin2["s1_m3_sum"] = basin2["s1_m3_sum"].fillna(0.0)
basin2["_hosts"] = basin2[BASIN_KEY].isin(basins_with_sites)

s1_thr = float(basin2.loc[basin2["_hosts"], "s1_m3_sum"].median())
basin2["_hi_s1"] = basin2["s1_m3_sum"] > s1_thr
basin2["_hi_stress_nb"] = _neighbor_aware_high_stress(basin2, STRESS_COL_BASIN, thr=3.0)

# 4-class category (same mapping)
basin2["_cat_s1"] = basin2["_hi_s1"].astype(int) * 2 + basin2["_hi_stress_nb"].astype(int)

# -----------------------------
# Panel B: Scope 2 BA typology from gdf_ba_l fields
# -----------------------------
# Ensure these exist even if we fell back to gdf_ba (we'll compute minimal if needed)
ba2 = ba_layer.copy()

if using_ba_l:
    # Use your precomputed flags directly
    ba2["_hi_s2"] = ba2["_high_s2"].astype(bool)
    ba2["_hi_stress"] = ba2["_high_stress_mean"].astype(bool)
    ba2["_hi_fossil"] = ba2["_fossil_heavy"].astype(bool)
else:
    # Fallback: compute hi_s2 + stress_mean from sites; fossil criterion not available
    if STRESS_COL_SITE not in sites.columns:
        raise KeyError(f"`{STRESS_COL_SITE}` missing in gdf_sites; needed to compute BA mean stress.")
    sites[STRESS_COL_SITE] = pd.to_numeric(sites[STRESS_COL_SITE], errors="coerce").fillna(0.0)

    r_s2 = sites.groupby(SITE_BA_KEY)[S2_COL_M3].sum().rename("s2_m3_sum").reset_index().rename(columns={SITE_BA_KEY: BA_KEY})
    r_st = sites.groupby(SITE_BA_KEY)[STRESS_COL_SITE].mean().rename("stress_mean").reset_index().rename(columns={SITE_BA_KEY: BA_KEY})
    ba2 = ba2.merge(r_s2, on=BA_KEY, how="left").merge(r_st, on=BA_KEY, how="left")
    ba2[["s2_m3_sum","stress_mean"]] = ba2[["s2_m3_sum","stress_mean"]].fillna(0.0)

    thr = float(ba2.loc[ba2["s2_m3_sum"]>0, "s2_m3_sum"].quantile(0.3))
    ba2["_hi_s2"] = ba2["s2_m3_sum"] >= thr
    ba2["_hi_stress"] = ba2["stress_mean"] >= 3.0
    ba2["_hi_fossil"] = False  # unknown -> treated as False
    print("WARNING: using gdf_ba fallback; fossil-heavy criterion not applied.")

# Collapse into 4 bins (same as before)
ba2["_crit_other"] = (ba2["_hi_fossil"].astype(int) + ba2["_hi_stress"].astype(int))
ba2["_cat_s2"] = 0
ba2.loc[ba2["_hi_s2"] & (ba2["_crit_other"]==0), "_cat_s2"] = 1
ba2.loc[ba2["_hi_s2"] & (ba2["_crit_other"]==1), "_cat_s2"] = 2
ba2.loc[ba2["_hi_s2"] & (ba2["_crit_other"]>=2), "_cat_s2"] = 3

# -----------------------------
# Plot Figure 2
# -----------------------------
minx, miny, maxx, maxy = st.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

s1_colors = {0:"#f0f0f0", 1:"#bdbdbd", 2:"#9ecae1", 3:"#de2d26"}
s2_colors = {0:"#f0f0f0", 1:"#9ecae1", 2:"#fc9272", 3:"#de2d26"}

fig = plt.figure(figsize=(14.5, 7.2))
gs = fig.add_gridspec(1, 2, left=0.04, right=0.985, top=0.92, bottom=0.08, wspace=0.06)
axA = fig.add_subplot(gs[0,0])
axB = fig.add_subplot(gs[0,1])

# Panel A
basin2.plot(
    ax=axA,
    color=basin2["_cat_s1"].map(s1_colors),
    linewidth=0.15, edgecolor="white", zorder=1
)

# Visible stress overlay (hatch needs edgecolor)
basin2[basin2["_hi_stress_nb"]].plot(
    ax=axA, facecolor="none", edgecolor="#4d4d4d", linewidth=0.0,
    hatch="////", zorder=3
)

# Strong hotspot outline
basin2[basin2["_cat_s1"]==3].boundary.plot(ax=axA, color="#000000", linewidth=1.2, zorder=4)

st.boundary.plot(ax=axA, color="#111111", linewidth=0.6, zorder=10)
axA.set_title("a. Scope 1 basin hotspots (baseline)\nHigh S1 > median; Stress = Aqueduct ≥3 in basin or neighbor", loc="center")
axA.set_axis_off()
axA.set_xlim(minx-padx, maxx+padx); axA.set_ylim(miny-pady, maxy+pady)
axA.set_aspect("equal")

axA.legend(handles=[
    Patch(facecolor=s1_colors[3], edgecolor="black", label="Hotspot (high S1 + high stress)"),
    Patch(facecolor=s1_colors[2], edgecolor="white", label="High S1 only"),
    Patch(facecolor=s1_colors[1], edgecolor="white", label="High stress only"),
    Patch(facecolor=s1_colors[0], edgecolor="white", label="Neither"),
    Patch(facecolor="white", edgecolor="#4d4d4d", hatch="////", label="Stress overlay (≥3, basin or neighbor)"),
], loc="lower left", frameon=True, fontsize=9)

# Panel B
ba2.plot(
    ax=axB,
    color=ba2["_cat_s2"].map(s2_colors),
    linewidth=0.25, edgecolor="white", zorder=1
)

# Very evident border for scope 2 hotspots (triple intersection)
ba2[ba2["_cat_s2"]==3].boundary.plot(ax=axB, color="#000000", linewidth=2.2, zorder=5)

st.boundary.plot(ax=axB, color="#111111", linewidth=0.6, zorder=10)
axB.set_title("b. Scope 2 BA hotspot typology (baseline)\nHigh S2 top quartile × fossil-heavy >50% × mean stress ≥3", loc="center")
axB.set_axis_off()
axB.set_xlim(minx-padx, maxx+padx); axB.set_ylim(miny-pady, maxy+pady)
axB.set_aspect("equal")

axB.legend(handles=[
    Patch(facecolor=s2_colors[3], edgecolor="black", label="Scope 2 hotspot (triple intersection)"),
    Patch(facecolor=s2_colors[2], edgecolor="white", label="High S2 + one criterion"),
    Patch(facecolor=s2_colors[1], edgecolor="white", label="High S2 only"),
    Patch(facecolor=s2_colors[0], edgecolor="white", label="Not high S2"),
], loc="lower left", frameon=True, fontsize=9)

note = "BA fossil share/typology from gdf_ba_l" if using_ba_l else "Note: fossil-heavy criterion not applied (fallback BA layer)"
fig.text(0.5, 0.02, note, ha="center", va="center", fontsize=9)

out_pdf = os.path.join(OUTDIR, "Fig2_HotspotAtlas_base.pdf")
plt.savefig(out_pdf, bbox_inches="tight")
plt.show()
print("Saved:", out_pdf)

In [ ]:
# =========================
# FIGURE 2: HOTSPOT ATLAS
# =========================
_assert_has(gdf_sites, [SITE_BASIN_KEY, SITE_BA_KEY, MW_COL, S1_COL_M3, S2_COL_M3], "gdf_sites")
_assert_has(gdf_basin, [BASIN_KEY, STRESS_COL_BASIN, "geometry"], "gdf_basin")
_assert_has(gdf_ba,    [BA_KEY, "geometry"], "gdf_ba")
_assert_has(states48,  ["geometry"], "states48")

sites, basin, ba, st = _prep_conus(gdf_sites, gdf_basin, gdf_ba, states48)

# Ensure stress exists on sites (often in the "last columns" of your gdf_sites)
if STRESS_COL_SITE not in sites.columns:
    raise KeyError(f"`{STRESS_COL_SITE}` not found in gdf_sites. It must exist to compute BA mean stress.")

sites[MW_COL]     = pd.to_numeric(sites[MW_COL], errors="coerce").fillna(0.0)
sites[S1_COL_M3]  = pd.to_numeric(sites[S1_COL_M3], errors="coerce").fillna(0.0)
sites[S2_COL_M3]  = pd.to_numeric(sites[S2_COL_M3], errors="coerce").fillna(0.0)
sites[STRESS_COL_SITE] = pd.to_numeric(sites[STRESS_COL_SITE], errors="coerce").fillna(0.0)

# -----------------------------
# Scope 1 basin aggregation + hotspot definition
# -----------------------------
basins_with_sites = sites[SITE_BASIN_KEY].astype(str).unique()
s1_b = (sites.groupby(SITE_BASIN_KEY)[S1_COL_M3].sum()).rename("s1_m3_sum").reset_index()
s1_b = s1_b.rename(columns={SITE_BASIN_KEY: BASIN_KEY})

basin2 = basin.merge(s1_b, on=BASIN_KEY, how="left")
basin2["s1_m3_sum"] = basin2["s1_m3_sum"].fillna(0.0)
basin2["_hosts"] = basin2[BASIN_KEY].isin(basins_with_sites)

# median among hosting basins
s1_thr = float(basin2.loc[basin2["_hosts"], "s1_m3_sum"].median())
basin2["_hi_s1"] = basin2["s1_m3_sum"] > s1_thr

# neighbor-aware stress (THIS is where your hatch was previously invisible)
basin2["_hi_stress_nb"] = _neighbor_aware_high_stress(basin2, STRESS_COL_BASIN, thr=3.0)

# 4-class category
# 0 neither, 1 stress only, 2 high S1 only, 3 hotspot
basin2["_cat_s1"] = (
    basin2["_hi_s1"].astype(int) * 2 +
    basin2["_hi_stress_nb"].astype(int) * 1
)
# mapping:
# hi_s1=0,stress=0 -> 0
# hi_s1=0,stress=1 -> 1
# hi_s1=1,stress=0 -> 2
# hi_s1=1,stress=1 -> 3

# -----------------------------
# Scope 2 BA aggregation + typology
# -----------------------------
bas_with_sites = sites[SITE_BA_KEY].astype(str).unique()
s2_r = (sites.groupby(SITE_BA_KEY)[S2_COL_M3].sum()).rename("s2_m3_sum").reset_index()
s2_r = s2_r.rename(columns={SITE_BA_KEY: BA_KEY})

s1_r = (sites.groupby(SITE_BA_KEY)[S1_COL_M3].sum()).rename("s1_m3_sum").reset_index()
s1_r = s1_r.rename(columns={SITE_BA_KEY: BA_KEY})

mw_r = (sites.groupby(SITE_BA_KEY)[MW_COL].sum()).rename("mw_sum").reset_index()
mw_r = mw_r.rename(columns={SITE_BA_KEY: BA_KEY})

stress_r = (sites.groupby(SITE_BA_KEY)[STRESS_COL_SITE].mean()).rename("stress_mean").reset_index()
stress_r = stress_r.rename(columns={SITE_BA_KEY: BA_KEY})

ba2 = ba.merge(s2_r, on=BA_KEY, how="left").merge(s1_r, on=BA_KEY, how="left").merge(mw_r, on=BA_KEY, how="left").merge(stress_r, on=BA_KEY, how="left")
ba2[["s2_m3_sum","s1_m3_sum","mw_sum","stress_mean"]] = ba2[["s2_m3_sum","s1_m3_sum","mw_sum","stress_mean"]].fillna(0.0)
ba2["_hosts"] = ba2[BA_KEY].isin(bas_with_sites)

# high Scope2 threshold: top quartile among BAs hosting hyperscale
s2_thr = float(ba2.loc[ba2["_hosts"], "s2_m3_sum"].quantile(0.5))
ba2["_hi_s2"] = ba2["s2_m3_sum"] >= s2_thr
ba2["_hi_stress"] = ba2["stress_mean"] >= 3.0

# fossil share detection: try BA first; else site-level and aggregate
fossil_ba, fossil_note = _find_fossil_share_series(ba2)
if fossil_ba is None:
    fossil_site, fossil_note2 = _find_fossil_share_series(sites)
    if fossil_site is not None:
        tmp = sites[[SITE_BA_KEY]].copy()
        tmp["fossil_share"] = pd.to_numeric(fossil_site, errors="coerce")
        if tmp["fossil_share"].max(skipna=True) > 1.5:
            tmp["fossil_share"] = tmp["fossil_share"] / 100.0
        fossil_ba = tmp.groupby(SITE_BA_KEY)["fossil_share"].mean()
        ba2 = ba2.set_index(BA_KEY)
        ba2["fossil_share"] = fossil_ba
        ba2 = ba2.reset_index()
        fossil_note = f"aggregated from sites ({fossil_note2})"
    else:
        print("WARNING: No fossil share column found in gdf_ba or gdf_sites. "
              "Scope 2 typology will be computed WITHOUT the fossil-heavy criterion.")
        ba2["fossil_share"] = np.nan
        fossil_note = None
else:
    ba2["fossil_share"] = fossil_ba

ba2["_hi_fossil"] = (pd.to_numeric(ba2["fossil_share"], errors="coerce") > 0.50)

# typology score: number of criteria satisfied among (hi_s2, hi_fossil, hi_stress)
# We'll collapse into 4 bins for readability:
# 0: not high consumption
# 1: high consumption only
# 2: high consumption + exactly one of (fossil,stress)
# 3: triple intersection (hotspot)
ba2["_crit_other"] = (ba2["_hi_fossil"].astype(int) + ba2["_hi_stress"].astype(int))
ba2["_cat_s2"] = 0
ba2.loc[ba2["_hi_s2"] & (ba2["_crit_other"]==0), "_cat_s2"] = 1
ba2.loc[ba2["_hi_s2"] & (ba2["_crit_other"]==1), "_cat_s2"] = 2
ba2.loc[ba2["_hi_s2"] & (ba2["_crit_other"]>=2), "_cat_s2"] = 3

# -----------------------------
# Plot Figure 2
# -----------------------------
minx, miny, maxx, maxy = st.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

# Color palettes (categorical, readable)
s1_colors = {
    0: "#f0f0f0",  # neither
    1: "#bdbdbd",  # stress only
    2: "#9ecae1",  # high S1 only
    3: "#de2d26",  # hotspot
}
s2_colors = {
    0: "#f0f0f0",  # not high consumption
    1: "#9ecae1",  # high S2 only
    2: "#fc9272",  # high S2 + one criterion
    3: "#de2d26",  # triple hotspot
}

fig = plt.figure(figsize=(14.5, 7.2))
gs = fig.add_gridspec(nrows=1, ncols=2, left=0.04, right=0.985, top=0.92, bottom=0.08, wspace=0.06)
axA = fig.add_subplot(gs[0,0])
axB = fig.add_subplot(gs[0,1])

# Panel A: Scope 1 basin hotspots
basin2.assign(_c=basin2["_cat_s1"].map(s1_colors)).plot(
    ax=axA, color=basin2["_cat_s1"].map(s1_colors),
    linewidth=0.15, edgecolor="white", zorder=1
)

# Make stress overlay visible (fix: hatch needs edgecolor)
# Overlay stress-only hatch (neighbor-aware stress)
basin2[basin2["_hi_stress_nb"]].plot(
    ax=axA, facecolor="none", edgecolor="#4d4d4d", linewidth=0.0,
    hatch="////", zorder=3
)

# Outline hotspots strongly
basin2[basin2["_cat_s1"]==3].boundary.plot(ax=axA, color="#000000", linewidth=1.2, zorder=4)

st.boundary.plot(ax=axA, color="#111111", linewidth=0.6, zorder=10)
axA.set_title("a. Scope 1 basin hotspots (baseline)\nHigh S1 = above median among hosting basins; Stress = Aqueduct ≥3 in basin or neighbor", loc="center")
axA.set_axis_off()
axA.set_xlim(minx-padx, maxx+padx); axA.set_ylim(miny-pady, maxy+pady)
axA.set_aspect("equal")

legendA = [
    Patch(facecolor=s1_colors[3], edgecolor="black", label="Hotspot (high S1 + high stress)"),
    Patch(facecolor=s1_colors[2], edgecolor="white", label="High S1 only"),
    Patch(facecolor=s1_colors[1], edgecolor="white", label="High stress only"),
    Patch(facecolor=s1_colors[0], edgecolor="white", label="Neither"),
    Patch(facecolor="white", edgecolor="#4d4d4d", hatch="////", label="Stress overlay (Aqueduct ≥3, basin or neighbor)"),
]
axA.legend(handles=legendA, loc="lower left", frameon=True, fontsize=9)

# Panel B: Scope 2 BA typology
ba2.assign(_c=ba2["_cat_s2"].map(s2_colors)).plot(
    ax=axB, color=ba2["_cat_s2"].map(s2_colors),
    linewidth=0.25, edgecolor="white", zorder=1
)

# Outline triple hotspots strongly (user asked: more evident border)
ba2[ba2["_cat_s2"]==3].boundary.plot(ax=axB, color="#000000", linewidth=2.0, zorder=5)

st.boundary.plot(ax=axB, color="#111111", linewidth=0.6, zorder=10)
axB.set_title("b. Scope 2 BA hotspot typology (baseline)\nHigh S2 = top quartile; fossil-heavy >50%; mean siting stress ≥3", loc="center")
axB.set_axis_off()
axB.set_xlim(minx-padx, maxx+padx); axB.set_ylim(miny-pady, maxy+pady)
axB.set_aspect("equal")

legendB = [
    Patch(facecolor=s2_colors[3], edgecolor="black", label="Scope 2 hotspot (triple intersection)"),
    Patch(facecolor=s2_colors[2], edgecolor="white", label="High S2 + one criterion"),
    Patch(facecolor=s2_colors[1], edgecolor="white", label="High S2 only"),
    Patch(facecolor=s2_colors[0], edgecolor="white", label="Not high S2"),
]
axB.legend(handles=legendB, loc="lower left", frameon=True, fontsize=9)

# small footnote in figure about fossil share detection
if fossil_note is None:
    fig.text(0.5, 0.02, "Note: fossil-share column not detected; typology computed without fossil-heavy criterion.",
             ha="center", va="center", fontsize=9)
else:
    fig.text(0.5, 0.02, f"Fossil share source: {fossil_note}", ha="center", va="center", fontsize=9)

out_pdf = os.path.join(OUTDIR, "Fig2_HotspotAtlas_base.pdf")
plt.savefig(out_pdf, bbox_inches="tight")
plt.show()
print("Saved:", out_pdf)

In [ ]:
# ==========================================
# FIGURE 3: CONCENTRATION + DOMINANCE MAP
# ==========================================
sites, basin, ba, st = _prep_conus(gdf_sites, gdf_basin, gdf_ba, states48)

for c in [MW_COL, S1_COL_M3, S2_COL_M3, STRESS_COL_SITE]:
    if c not in sites.columns:
        raise KeyError(f"gdf_sites missing `{c}` needed for Fig 3.")
sites[MW_COL] = pd.to_numeric(sites[MW_COL], errors="coerce").fillna(0.0)
sites[S1_COL_M3] = pd.to_numeric(sites[S1_COL_M3], errors="coerce").fillna(0.0)
sites[S2_COL_M3] = pd.to_numeric(sites[S2_COL_M3], errors="coerce").fillna(0.0)
sites[STRESS_COL_SITE] = pd.to_numeric(sites[STRESS_COL_SITE], errors="coerce").fillna(0.0)

# Aggregations
basins_with_sites = sites[SITE_BASIN_KEY].astype(str).unique()
b_s1 = sites.groupby(SITE_BASIN_KEY)[S1_COL_M3].sum().rename("s1_m3").reset_index().rename(columns={SITE_BASIN_KEY:BASIN_KEY})
basin3 = basin.merge(b_s1, on=BASIN_KEY, how="left")
basin3["s1_m3"] = basin3["s1_m3"].fillna(0.0)
basin3 = basin3[basin3[BASIN_KEY].isin(basins_with_sites)].copy()

ba_with_sites = sites[SITE_BA_KEY].astype(str).unique()
r_s1 = sites.groupby(SITE_BA_KEY)[S1_COL_M3].sum().rename("s1_m3").reset_index().rename(columns={SITE_BA_KEY:BA_KEY})
r_s2 = sites.groupby(SITE_BA_KEY)[S2_COL_M3].sum().rename("s2_m3").reset_index().rename(columns={SITE_BA_KEY:BA_KEY})
r_mw = sites.groupby(SITE_BA_KEY)[MW_COL].sum().rename("mw").reset_index().rename(columns={SITE_BA_KEY:BA_KEY})
r_st = sites.groupby(SITE_BA_KEY)[STRESS_COL_SITE].mean().rename("stress_mean").reset_index().rename(columns={SITE_BA_KEY:BA_KEY})

ba3 = ba.merge(r_s1, on=BA_KEY, how="left").merge(r_s2, on=BA_KEY, how="left").merge(r_mw, on=BA_KEY, how="left").merge(r_st, on=BA_KEY, how="left")
ba3[["s1_m3","s2_m3","mw","stress_mean"]] = ba3[["s1_m3","s2_m3","mw","stress_mean"]].fillna(0.0)
ba3 = ba3[ba3[BA_KEY].isin(ba_with_sites)].copy()

# Concentration curves
def concentration_curve(values):
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    v = v[v > 0]
    if len(v) == 0:
        return np.array([0,1]), np.array([0,1])
    v = np.sort(v)[::-1]
    cum = np.cumsum(v) / v.sum()
    x = np.arange(1, len(v)+1) / len(v)
    return x, cum

x1, c1 = concentration_curve(basin3["s1_m3"])
x2, c2 = concentration_curve(ba3["s2_m3"])

# Dominance map at BA level (share of total attributable to Scope 2)
ba3["tot_m3"] = ba3["s1_m3"] + ba3["s2_m3"]
ba3["s2_share"] = np.where(ba3["tot_m3"] > 0, ba3["s2_m3"] / ba3["tot_m3"], np.nan)

# categories for action: cooling-driven vs mixed vs electricity-driven
# (you can tweak thresholds; keep simple + interpretable)
ba3["_dom_cat"] = pd.cut(
    ba3["s2_share"],
    bins=[-0.001, 0.40, 0.60, 1.001],
    labels=["Scope 1–dominant", "Mixed", "Scope 2–dominant"]
)

dom_colors = {
    "Scope 1–dominant": "#9ecae1",
    "Mixed": "#f0f0f0",
    "Scope 2–dominant": "#fdae6b"
}

# Plot
minx, miny, maxx, maxy = st.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

fig = plt.figure(figsize=(14.5, 7.2))
gs = fig.add_gridspec(nrows=1, ncols=2, left=0.04, right=0.985, top=0.92, bottom=0.10, wspace=0.10)
axA = fig.add_subplot(gs[0,0])
axB = fig.add_subplot(gs[0,1])

# Panel A: concentration
axA.plot(x1, c1, lw=2.2, label="Scope 1 (basins)")
axA.plot(x2, c2, lw=2.2, label="Scope 2 (BAs)")
axA.set_title("a. Concentration of operational water")
axA.set_xlabel("Fraction of regions (ranked by water consumption)")
axA.set_ylabel("Cumulative fraction of water consumption")
axA.grid(alpha=0.25)
axA.set_xlim(0, 1)
axA.set_ylim(0, 1)
axA.legend(frameon=True, loc="lower right")

# Annotate a simple “top 10% share”
def top_share(values, frac=0.10):
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    v = v[v > 0]
    if len(v) == 0:
        return np.nan
    v = np.sort(v)[::-1]
    k = max(1, int(np.ceil(frac * len(v))))
    return float(v[:k].sum() / v.sum())

axA.text(0.02, 0.12, f"Top 10% basins → {100*top_share(basin3['s1_m3'],0.10):.0f}% of Scope 1",
         transform=axA.transAxes, fontsize=9)
axA.text(0.02, 0.06, f"Top 10% BAs → {100*top_share(ba3['s2_m3'],0.10):.0f}% of Scope 2",
         transform=axA.transAxes, fontsize=9)

# Panel B: dominance map
ba3.plot(
    ax=axB,
    color=ba3["_dom_cat"].astype(str).map(dom_colors),
    linewidth=0.25, edgecolor="white", zorder=1
)

# overlay high siting-stress hatch (mean stress >= 3)
ba3_hi = ba3[ba3["stress_mean"] >= 3.0]
if len(ba3_hi):
    ba3_hi.plot(ax=axB, facecolor="none", edgecolor="#4d4d4d", linewidth=0.0, hatch="////", zorder=3)

st.boundary.plot(ax=axB, color="#111111", linewidth=0.6, zorder=10)
axB.set_title("b. Pathway dominance by BA\n(fill: S2 share category; hatch: mean siting stress ≥3)")
axB.set_axis_off()
axB.set_xlim(minx-padx, maxx+padx); axB.set_ylim(miny-pady, maxy+pady)
axB.set_aspect("equal")

legendB = [
    Patch(facecolor=dom_colors["Scope 1–dominant"], edgecolor="white", label="Scope 1–dominant (S2 share < 0.4)"),
    Patch(facecolor=dom_colors["Mixed"], edgecolor="white", label="Mixed (0.4–0.6)"),
    Patch(facecolor=dom_colors["Scope 2–dominant"], edgecolor="white", label="Scope 2–dominant (S2 share > 0.6)"),
    Patch(facecolor="white", edgecolor="#4d4d4d", hatch="////", label="High siting stress (mean ≥3)"),
]
axB.legend(handles=legendB, loc="lower left", frameon=True)

out_pdf = os.path.join(OUTDIR, "Fig3_Concentration_Dominance.pdf")
plt.savefig(out_pdf, bbox_inches="tight")
plt.show()
print("Saved:", out_pdf)

In [ ]:
# ==========================================
# FIGURE 3 (REVISED): CONCENTRATION + “WHERE TO ACT” DOMINANCE MAP
# - More compelling message:
#   (a) Concentration curves + top-share callouts + “80% of water in top X% regions”
#   (b) BA “pathway dominance” map with:
#       • fill = dominance class (Scope 1 / Mixed / Scope 2)
#       • hatch = high siting stress (mean ≥ threshold)
#       • bold outline = “high-burden” BAs (top quantile of total water)  <-- action cue
# - Always shows plot in notebook
# - Exports PDF only
# - Threshold knobs at the top
# ==========================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# -----------------------------
# OUTPUT
# -----------------------------
OUTDIR = "images"
os.makedirs(OUTDIR, exist_ok=True)
OUT_PDF = os.path.join(OUTDIR, "Fig3_Concentration_Dominance_action.pdf")

# -----------------------------
# THRESHOLD KNOBS (edit here)
# -----------------------------
TOP_FRAC_ANNOT = 0.10          # annotate “top 10% regions”
PCT_EXPLAINED  = 0.80          # “80% of water occurs in top X% regions”

# Dominance bins (share of total water that is Scope 2)
DOM_LO = 0.40                  # below -> Scope 1–dominant
DOM_HI = 0.60                  # above -> Scope 2–dominant

# Stress hatch threshold (siting exposure)
STRESS_THR = 3.0               # Aqueduct baseline stress threshold

# “High burden” outline threshold (action cue) — based on total water in BA
HIGH_BURDEN_Q = 0.75           # top quartile of BA total (S1+S2) -> bold outline

# Which BA layer to use for aggregation base
# (Fig 2 uses gdf_ba_l for fossil share; here we only need geometry + BA key)
BA_LAYER = gdf_ba_l if "gdf_ba_l" in globals() else gdf_ba

# Stress aggregation: capacity-weighted mean is preferred
USE_CAP_WEIGHTED_STRESS = True  # uses MW weights if possible

# -----------------------------
# STYLE
# -----------------------------
FIG_W, FIG_H = 14.8, 7.4

# Same “paper” style: muted fills + diagonal stress hatch like Fig 2
dom_colors = {
    "Scope 1–dominant": "#9ecae1",
    "Mixed": "#f0f0f0",
    "Scope 2–dominant": "#fc9272"
}
HATCH = "////"
HATCH_EDGE = "#2b2b2b"

STATE_EDGE = "#111111"
STATE_LW = 0.6

EDGE_OTHER = "white"
LW_OTHER = 0.25

BURDEN_EDGE = "#000000"
BURDEN_LW = 2.0  # bold outline for high-burden BAs

# -----------------------------
# PREP + CHECKS
# -----------------------------
sites, basin, ba, st = _prep_conus(gdf_sites, gdf_basin, BA_LAYER, states48)

req_sites = [MW_COL, S1_COL_M3, S2_COL_M3, SITE_BASIN_KEY, SITE_BA_KEY]
for c in req_sites:
    if c not in sites.columns:
        raise KeyError(f"gdf_sites missing `{c}` needed for Fig 3.")

# stress column in sites
if STRESS_COL_SITE not in sites.columns:
    raise KeyError(f"gdf_sites missing `{STRESS_COL_SITE}` (needed for BA stress exposure).")

# numeric cleanup
for c in [MW_COL, S1_COL_M3, S2_COL_M3, STRESS_COL_SITE]:
    sites[c] = pd.to_numeric(sites[c], errors="coerce").fillna(0.0)

# -----------------------------
# AGGREGATIONS
# -----------------------------
# Basins hosting sites (Scope 1 concentration at basin level)
basins_with_sites = sites[SITE_BASIN_KEY].astype(str).unique()
b_s1 = (
    sites.groupby(SITE_BASIN_KEY)[S1_COL_M3].sum()
    .rename("s1_m3").reset_index()
    .rename(columns={SITE_BASIN_KEY: BASIN_KEY})
)
basin3 = basin.merge(b_s1, on=BASIN_KEY, how="left")
basin3["s1_m3"] = basin3["s1_m3"].fillna(0.0)
basin3 = basin3[basin3[BASIN_KEY].astype(str).isin(basins_with_sites.astype(str))].copy()

# BA hosting sites (Scope 2 concentration + dominance map at BA level)
ba_with_sites = sites[SITE_BA_KEY].astype(str).unique()

r_s1 = (
    sites.groupby(SITE_BA_KEY)[S1_COL_M3].sum()
    .rename("s1_m3").reset_index()
    .rename(columns={SITE_BA_KEY: BA_KEY})
)
r_s2 = (
    sites.groupby(SITE_BA_KEY)[S2_COL_M3].sum()
    .rename("s2_m3").reset_index()
    .rename(columns={SITE_BA_KEY: BA_KEY})
)
r_mw = (
    sites.groupby(SITE_BA_KEY)[MW_COL].sum()
    .rename("mw").reset_index()
    .rename(columns={SITE_BA_KEY: BA_KEY})
)

# Stress: capacity-weighted mean if requested + possible
if USE_CAP_WEIGHTED_STRESS:
    # avoid division by 0
    tmp = sites[[SITE_BA_KEY, MW_COL, STRESS_COL_SITE]].copy()
    tmp["_w"] = tmp[MW_COL].clip(lower=0.0)
    tmp["_wx"] = tmp["_w"] * tmp[STRESS_COL_SITE]
    r_st = (
        tmp.groupby(SITE_BA_KEY)
           .agg(wsum=("_w", "sum"), wxsum=("_wx", "sum"))
           .reset_index()
           .rename(columns={SITE_BA_KEY: BA_KEY})
    )
    r_st["stress_mean"] = np.where(r_st["wsum"] > 0, r_st["wxsum"] / r_st["wsum"], np.nan)
    r_st = r_st[[BA_KEY, "stress_mean"]]
else:
    r_st = (
        sites.groupby(SITE_BA_KEY)[STRESS_COL_SITE].mean()
        .rename("stress_mean").reset_index()
        .rename(columns={SITE_BA_KEY: BA_KEY})
    )

ba3 = (
    ba.merge(r_s1, on=BA_KEY, how="left")
      .merge(r_s2, on=BA_KEY, how="left")
      .merge(r_mw, on=BA_KEY, how="left")
      .merge(r_st, on=BA_KEY, how="left")
)
ba3[["s1_m3", "s2_m3", "mw"]] = ba3[["s1_m3", "s2_m3", "mw"]].fillna(0.0)
ba3["stress_mean"] = pd.to_numeric(ba3["stress_mean"], errors="coerce")
ba3 = ba3[ba3[BA_KEY].astype(str).isin(ba_with_sites.astype(str))].copy()

# Dominance metrics
ba3["tot_m3"] = ba3["s1_m3"] + ba3["s2_m3"]
ba3["s2_share"] = np.where(ba3["tot_m3"] > 0, ba3["s2_m3"] / ba3["tot_m3"], np.nan)

ba3["_dom_cat"] = pd.cut(
    ba3["s2_share"],
    bins=[-0.001, DOM_LO, DOM_HI, 1.001],
    labels=["Scope 1–dominant", "Mixed", "Scope 2–dominant"]
)

# High stress (hatch)
ba3["_hi_stress"] = (ba3["stress_mean"].fillna(0.0) >= STRESS_THR)

# High burden (bold outline)
mask_host = ba3["tot_m3"] > 0
if mask_host.any():
    burden_thr = float(ba3.loc[mask_host, "tot_m3"].quantile(HIGH_BURDEN_Q))
else:
    burden_thr = np.inf
ba3["_hi_burden"] = ba3["tot_m3"] >= burden_thr

# -----------------------------
# CONCENTRATION CURVES + CALLOUTS
# -----------------------------
def concentration_curve(values):
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    v = v[v > 0]
    if len(v) == 0:
        return np.array([0, 1]), np.array([0, 1])
    v = np.sort(v)[::-1]
    cum = np.cumsum(v) / v.sum()
    x = np.arange(1, len(v) + 1) / len(v)
    # prepend origin for nicer display
    x = np.concatenate([[0], x])
    cum = np.concatenate([[0], cum])
    return x, cum

def top_share(values, frac=0.10):
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    v = v[v > 0]
    if len(v) == 0:
        return np.nan
    v = np.sort(v)[::-1]
    k = max(1, int(np.ceil(frac * len(v))))
    return float(v[:k].sum() / v.sum())

def frac_needed_for_share(values, target=0.80):
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    v = v[v > 0]
    if len(v) == 0:
        return np.nan
    v = np.sort(v)[::-1]
    cum = np.cumsum(v) / v.sum()
    k = int(np.searchsorted(cum, target, side="left")) + 1
    return k / len(v)

x1, c1 = concentration_curve(basin3["s1_m3"])
x2, c2 = concentration_curve(ba3["s2_m3"])

top10_s1 = top_share(basin3["s1_m3"], TOP_FRAC_ANNOT)
top10_s2 = top_share(ba3["s2_m3"], TOP_FRAC_ANNOT)

need80_s1 = frac_needed_for_share(basin3["s1_m3"], PCT_EXPLAINED)
need80_s2 = frac_needed_for_share(ba3["s2_m3"], PCT_EXPLAINED)

# -----------------------------
# PLOT LAYOUT
# -----------------------------
minx, miny, maxx, maxy = st.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

fig = plt.figure(figsize=(FIG_W, FIG_H))
gs = fig.add_gridspec(
    nrows=1, ncols=2,
    left=0.04, right=0.985, top=0.90, bottom=0.16, wspace=0.10
)
axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])

# ---- Panel A: concentration curves
axA.plot(x1, c1, lw=2.4, label="Scope 1 (basins)")
axA.plot(x2, c2, lw=2.4, label="Scope 2 (balancing authorities)")

# visual anchors
axA.axvline(TOP_FRAC_ANNOT, lw=1.0, alpha=0.35)
axA.axhline(PCT_EXPLAINED, lw=1.0, alpha=0.35)

# callouts: “80% occurs in top X%”
if np.isfinite(need80_s1):
    axA.text(0.02, 0.92, f"Scope 1: {PCT_EXPLAINED:.0%} in top {need80_s1:.0%} of basins",
             transform=axA.transAxes, fontsize=10, va="top")
if np.isfinite(need80_s2):
    axA.text(0.02, 0.86, f"Scope 2: {PCT_EXPLAINED:.0%} in top {need80_s2:.0%} of BAs",
             transform=axA.transAxes, fontsize=10, va="top")

# callouts: top-10% shares
if np.isfinite(top10_s1):
    axA.text(0.02, 0.15, f"Top {TOP_FRAC_ANNOT:.0%} basins → {100*top10_s1:.0f}% of Scope 1",
             transform=axA.transAxes, fontsize=10)
if np.isfinite(top10_s2):
    axA.text(0.02, 0.08, f"Top {TOP_FRAC_ANNOT:.0%} BAs → {100*top10_s2:.0f}% of Scope 2",
             transform=axA.transAxes, fontsize=10)

axA.set_title("a. Water consumption is highly concentrated", loc="center")
axA.set_xlabel("Fraction of regions (ranked by water consumption)")
axA.set_ylabel("Cumulative fraction of water consumption")
axA.grid(alpha=0.25)
axA.set_xlim(0, 1)
axA.set_ylim(0, 1)

# legend below the panel
axA.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=1)

# ---- Panel B: dominance + action cue
ba3.plot(
    ax=axB,
    color=ba3["_dom_cat"].astype(str).map(dom_colors),
    linewidth=LW_OTHER, edgecolor=EDGE_OTHER, zorder=1
)

# hatch overlay for high stress exposure
ba3.loc[ba3["_hi_stress"]].plot(
    ax=axB, facecolor="none", edgecolor=HATCH_EDGE, linewidth=0.0, hatch=HATCH, zorder=3
)

# bold outline for “high burden” (top quantile of BA total water)
ba3.loc[ba3["_hi_burden"]].boundary.plot(
    ax=axB, color=BURDEN_EDGE, linewidth=BURDEN_LW, zorder=6
)

# state outline
st.boundary.plot(ax=axB, color=STATE_EDGE, linewidth=STATE_LW, zorder=10)

axB.set_title(
    "b. Which pathway dominates—and where siting stress and burden co-occur (BAs)",
    loc="center"
)
axB.set_axis_off()
axB.set_xlim(minx - padx, maxx + padx)
axB.set_ylim(miny - pady, maxy + pady)
axB.set_aspect("equal")

# -----------------------------
# FIG-LEVEL LEGEND (below maps; no overlap)
# -----------------------------
handles = [
    Patch(facecolor=dom_colors["Scope 1–dominant"], edgecolor="white",
          label=f"Scope 1–dominant (Scope 2 share < {DOM_LO:.2f})"),
    Patch(facecolor=dom_colors["Mixed"], edgecolor="white",
          label=f"Mixed ({DOM_LO:.2f}–{DOM_HI:.2f})"),
    Patch(facecolor=dom_colors["Scope 2–dominant"], edgecolor="white",
          label=f"Scope 2–dominant (Scope 2 share > {DOM_HI:.2f})"),
    Patch(facecolor="white", edgecolor=HATCH_EDGE, hatch=HATCH,
          label=f"High siting stress (mean ≥ {STRESS_THR:.0f})"),
    Patch(facecolor="none", edgecolor=BURDEN_EDGE,
          label=f"High-burden BA (top {int((1-HIGH_BURDEN_Q)*100)}% of total water)"),
]

fig.legend(
    handles=handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.04),
    ncol=3,
    frameon=False,
    fontsize=9,
    handlelength=1.8,
    columnspacing=1.6
)

# small note (optional, but helps interpretability)
stress_note = "Stress is capacity-weighted mean" if USE_CAP_WEIGHTED_STRESS else "Stress is unweighted mean"
fig.text(
    0.5, 0.01,
    f"{stress_note}. High-burden outline uses BA total water (Scope 1 + Scope 2) quantile Q={HIGH_BURDEN_Q:.2f}.",
    ha="center", va="bottom", fontsize=9
)

# -----------------------------
# SAVE (PDF only) + SHOW
# -----------------------------
plt.savefig(OUT_PDF, bbox_inches="tight")
plt.show()
print("Saved:", OUT_PDF)

In [ ]:
# ==========================================
# FIGURE 3 (IMPROVED): PARETO CONCENTRATION + DOMINANCE MAP
# - Panel A: Pareto charts (bars + cumulative line) instead of empty-looking curves
# - Legends boxed, moved to non-overlapping locations
# - Text annotations placed in axes coords (no collision)
# ==========================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

OUTDIR = "images"
os.makedirs(OUTDIR, exist_ok=True)

# -----------------------------
# knobs for panel A
# -----------------------------
TOP_N_SHOW = 35     # number of ranked regions to show as bars (rest are aggregated into "rest")
TARGET_CUM = 0.80   # show where we hit 80%

# -----------------------------
# Prep / checks (same as your code)
# -----------------------------
sites, basin, ba, st = _prep_conus(gdf_sites, gdf_basin, gdf_ba, states48)

for c in [MW_COL, S1_COL_M3, S2_COL_M3, STRESS_COL_SITE]:
    if c not in sites.columns:
        raise KeyError(f"gdf_sites missing `{c}` needed for Fig 3.")
sites[MW_COL] = pd.to_numeric(sites[MW_COL], errors="coerce").fillna(0.0)
sites[S1_COL_M3] = pd.to_numeric(sites[S1_COL_M3], errors="coerce").fillna(0.0)
sites[S2_COL_M3] = pd.to_numeric(sites[S2_COL_M3], errors="coerce").fillna(0.0)
sites[STRESS_COL_SITE] = pd.to_numeric(sites[STRESS_COL_SITE], errors="coerce").fillna(0.0)

# Aggregations
basins_with_sites = sites[SITE_BASIN_KEY].astype(str).unique()
b_s1 = (sites.groupby(SITE_BASIN_KEY)[S1_COL_M3].sum()
        .rename("s1_m3").reset_index()
        .rename(columns={SITE_BASIN_KEY: BASIN_KEY}))
basin3 = basin.merge(b_s1, on=BASIN_KEY, how="left")
basin3["s1_m3"] = basin3["s1_m3"].fillna(0.0)
basin3 = basin3[basin3[BASIN_KEY].astype(str).isin(basins_with_sites.astype(str))].copy()

ba_with_sites = sites[SITE_BA_KEY].astype(str).unique()
r_s1 = sites.groupby(SITE_BA_KEY)[S1_COL_M3].sum().rename("s1_m3").reset_index().rename(columns={SITE_BA_KEY:BA_KEY})
r_s2 = sites.groupby(SITE_BA_KEY)[S2_COL_M3].sum().rename("s2_m3").reset_index().rename(columns={SITE_BA_KEY:BA_KEY})
r_mw = sites.groupby(SITE_BA_KEY)[MW_COL].sum().rename("mw").reset_index().rename(columns={SITE_BA_KEY:BA_KEY})
r_st = sites.groupby(SITE_BA_KEY)[STRESS_COL_SITE].mean().rename("stress_mean").reset_index().rename(columns={SITE_BA_KEY:BA_KEY})

ba3 = (ba.merge(r_s1, on=BA_KEY, how="left")
         .merge(r_s2, on=BA_KEY, how="left")
         .merge(r_mw, on=BA_KEY, how="left")
         .merge(r_st, on=BA_KEY, how="left"))
ba3[["s1_m3","s2_m3","mw","stress_mean"]] = ba3[["s1_m3","s2_m3","mw","stress_mean"]].fillna(0.0)
ba3 = ba3[ba3[BA_KEY].astype(str).isin(ba_with_sites.astype(str))].copy()

# Dominance map (BA-level)
ba3["tot_m3"] = ba3["s1_m3"] + ba3["s2_m3"]
ba3["s2_share"] = np.where(ba3["tot_m3"] > 0, ba3["s2_m3"] / ba3["tot_m3"], np.nan)

ba3["_dom_cat"] = pd.cut(
    ba3["s2_share"],
    bins=[-0.001, 0.40, 0.60, 1.001],
    labels=["Scope 1–dominant", "Mixed", "Scope 2–dominant"]
)

dom_colors = {
    "Scope 1–dominant": "#9ecae1",
    "Mixed": "#f0f0f0",
    "Scope 2–dominant": "#fdae6b"
}

# High-burden outline: top quartile of BA total water (optional but powerful)
if (ba3["tot_m3"] > 0).any():
    hi_burden_thr = float(ba3.loc[ba3["tot_m3"] > 0, "tot_m3"].quantile(0.75))
else:
    hi_burden_thr = np.inf
ba3["_hi_burden"] = ba3["tot_m3"] >= hi_burden_thr

# -----------------------------
# Helper: Pareto chart builder
# -----------------------------
def pareto_components(values, top_n=TOP_N_SHOW):
    v = pd.Series(values).astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    v = v[v > 0].sort_values(ascending=False).reset_index(drop=True)
    if v.empty:
        return None

    share = v / v.sum()
    cum = share.cumsum()

    if len(share) > top_n:
        share_top = share.iloc[:top_n]
        rest = pd.Series([share.iloc[top_n:].sum()])
        share_plot = pd.concat([share_top, rest], ignore_index=True)
        cum_plot = share_plot.cumsum()
        labels = list(range(1, top_n+1)) + ["rest"]
    else:
        share_plot = share
        cum_plot = cum
        labels = list(range(1, len(share_plot)+1))

    # where do we hit TARGET_CUM in the full distribution (not just plotted)?
    k80 = int((cum >= TARGET_CUM).idxmax() + 1) if (cum >= TARGET_CUM).any() else len(cum)
    frac80 = k80 / len(cum) if len(cum) else np.nan

    return share_plot.values, cum_plot.values, labels, k80, frac80, float(TARGET_CUM)

def draw_pareto(ax, values, title):
    out = pareto_components(values)
    if out is None:
        ax.text(0.5, 0.5, "No positive values", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
        return

    share_plot, cum_plot, labels, k80, frac80, target = out
    x = np.arange(len(share_plot))

    # bars: share of total (left axis)
    ax.bar(x, share_plot, linewidth=0.0)
    ax.set_ylim(0, max(share_plot)*1.35)
    ax.set_ylabel("Share of total")
    ax.set_title(title, loc="left")
    ax.grid(axis="y", alpha=0.25)

    # cumulative (right axis)
    ax2 = ax.twinx()
    ax2.plot(x, cum_plot, lw=2.2)
    ax2.set_ylim(0, 1.0)
    ax2.set_ylabel("Cumulative share")

    # 80% line
    ax2.axhline(target, lw=1.2, alpha=0.7)
    ax2.text(0.99, target, f"{int(target*100)}%", ha="right", va="bottom", transform=ax2.get_yaxis_transform())

    # x ticks (keep sparse)
    ax.set_xticks([0, len(x)//2, len(x)-1])
    ax.set_xticklabels([str(labels[0]), str(labels[len(x)//2]), str(labels[-1])])
    ax.set_xlabel("Ranked regions (descending)")

    # annotation box (axes coords -> avoids overlap)
    ax.text(
        0.02, 0.92,
        f"{int(target*100)}% reached by top {k80} regions\n({frac80*100:.0f}% of regions)",
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#333333", lw=0.8, alpha=0.95)
    )

    # legend (boxed)
    handles = [
        Patch(facecolor="#999999", edgecolor="none", label="Region share (bars)"),
        Line2D([0],[0], lw=2.2, label="Cumulative share (line)"),
        Line2D([0],[0], lw=1.2, label=f"{int(target*100)}% threshold")
    ]
    ax.legend(handles=handles, loc="upper right", frameon=True)

# -----------------------------
# Plot layout
# -----------------------------
minx, miny, maxx, maxy = st.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

fig = plt.figure(figsize=(14.8, 7.6), constrained_layout=False)
gs = fig.add_gridspec(
    nrows=2, ncols=2,
    height_ratios=[1.0, 1.0],
    width_ratios=[1.05, 1.0],
    left=0.05, right=0.985, top=0.93, bottom=0.12,
    wspace=0.08, hspace=0.22
)

# Panel A becomes two Pareto sub-panels stacked
axA1 = fig.add_subplot(gs[0, 0])
axA2 = fig.add_subplot(gs[1, 0])

# a1: Scope 1 concentration across basins (bars only)
draw_concentration_bars(
    axA1,
    basin3["s1_m3"],
    basin3[BASIN_KEY],
    "a1. Scope 1 concentration across basins",
    target=TARGET_CUM,
    top_n=TOP_N_SHOW,
    top_label_n=TOP_LABEL_N,
    show_x_ids=SHOW_X_IDS,
    x_id_n=X_ID_N
)

# a2: Scope 2 concentration across BAs (bars only)
draw_concentration_bars(
    axA2,
    ba3["s2_m3"],
    ba3[BA_KEY],
    "a2. Scope 2 concentration across balancing authorities",
    target=TARGET_CUM,
    top_n=TOP_N_SHOW,
    top_label_n=TOP_LABEL_N,
    show_x_ids=SHOW_X_IDS,
    x_id_n=X_ID_N
)
# Panel B dominance map spans both rows
axB = fig.add_subplot(gs[:, 1])

ba3.plot(
    ax=axB,
    color=ba3["_dom_cat"].astype(str).map(dom_colors),
    linewidth=0.25, edgecolor="white", zorder=1
)

# hatch high siting-stress (mean stress >= 3)
ba3_hi = ba3[ba3["stress_mean"] >= 3.0]
if len(ba3_hi):
    ba3_hi.plot(ax=axB, facecolor="none", edgecolor="#4d4d4d", linewidth=0.0, hatch="////", zorder=3)

# outline high-burden BAs (top 25% total)
ba3.loc[ba3["_hi_burden"]].boundary.plot(ax=axB, color="#000000", linewidth=1.8, zorder=6)

st.boundary.plot(ax=axB, color="#111111", linewidth=0.6, zorder=10)
axB.set_title("b. Pathway dominance and stress co-occurrence (BAs)\nFill: dominance; hatch: mean stress ≥3; outline: top-quartile total burden", loc="center")
axB.set_axis_off()
axB.set_xlim(minx-padx, maxx+padx); axB.set_ylim(miny-pady, maxy+pady)
axB.set_aspect("equal")

# -----------------------------
# Boxed legend BELOW the map (no overlap)
# -----------------------------
legend_handles = [
    Patch(facecolor=dom_colors["Scope 1–dominant"], edgecolor="white", label="Scope 1–dominant (Scope 2 share < 0.40)"),
    Patch(facecolor=dom_colors["Mixed"], edgecolor="white", label="Mixed (0.40–0.60)"),
    Patch(facecolor=dom_colors["Scope 2–dominant"], edgecolor="white", label="Scope 2–dominant (Scope 2 share > 0.60)"),
    Patch(facecolor="white", edgecolor="#4d4d4d", hatch="////", label="High siting stress (mean ≥3)"),
    Line2D([0],[0], color="#000000", lw=1.8, label="High-burden BA (top 25% total water)")
]

# Put the legend in a dedicated figure-level box under the right panel
fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.73, 0.03),   # tuned to sit under the map column
    ncol=1,
    frameon=True,
    fontsize=9
)

out_pdf = os.path.join(OUTDIR, "Fig3_Concentration_Pareto_Dominance.pdf")
plt.savefig(out_pdf, bbox_inches="tight")
plt.show()
print("Saved:", out_pdf)

In [ ]:
# ==========================================
# FIGURE 3 (REMADE): CONCENTRATION BARS + WHERE-THEY-ARE MAPS
# - Bars only (no background curve)
# - Inset: top contributors + % share
# - Companion map: highlights top contributors and labels by rank
# - Basin labels use dominant state abbrev(s) from overlap area (no counties needed)
# ==========================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

OUTDIR = "images"
os.makedirs(OUTDIR, exist_ok=True)
OUT_PDF = os.path.join(OUTDIR, "Fig3_Concentration_BarsPlusMaps.pdf")

# -----------------------------
# KNOBS (edit here)
# -----------------------------
TARGET_CUM   = 0.80   # vertical line = smallest k reaching this cumulative share
TOP_N_SHOW   = 30     # how many ranked bars to show (rest aggregated)
TOP_ARROW_N  = 7      # how many top bars get arrows + rank labels
TOP_INSET_N  = 7      # how many shown in the inset box
SHOW_X_LABELS = False # if True, put top labels on x-axis (can crowd)

# Map highlight:
MAP_LABEL_N = 7       # label ranks 1..MAP_LABEL_N on the map (match arrows)

# Dominance bins for BA map
S2_SHARE_BINS = [-0.001, 0.40, 0.60, 1.001]
S2_SHARE_LABS = ["Scope 1–dominant", "Mixed", "Scope 2–dominant"]
DOM_COLORS = {
    "Scope 1–dominant": "#9ecae1",
    "Mixed": "#f0f0f0",
    "Scope 2–dominant": "#fdae6b",
}

# Stress hatch on BA dominance map (optional)
STRESS_THR = 3.0
HATCH = "////"
HATCH_EDGE = "#4d4d4d"

# -----------------------------
# Prep & required columns
# -----------------------------
sites, basin, ba, st = _prep_conus(gdf_sites, gdf_basin, gdf_ba_l if "gdf_ba_l" in globals() else gdf_ba, states48)

for c in [MW_COL, S1_COL_M3, S2_COL_M3, STRESS_COL_SITE, SITE_BASIN_KEY, SITE_BA_KEY]:
    if c not in sites.columns:
        raise KeyError(f"gdf_sites missing `{c}` needed for Fig 3.")
sites[MW_COL] = pd.to_numeric(sites[MW_COL], errors="coerce").fillna(0.0)
sites[S1_COL_M3] = pd.to_numeric(sites[S1_COL_M3], errors="coerce").fillna(0.0)
sites[S2_COL_M3] = pd.to_numeric(sites[S2_COL_M3], errors="coerce").fillna(0.0)
sites[STRESS_COL_SITE] = pd.to_numeric(sites[STRESS_COL_SITE], errors="coerce").fillna(0.0)

# -----------------------------
# Aggregations (same logic as your current Fig 3)
# -----------------------------
# Basin totals for Scope 1
basins_with_sites = sites[SITE_BASIN_KEY].astype(str).unique()
b_s1 = (
    sites.groupby(SITE_BASIN_KEY)[S1_COL_M3].sum()
    .rename("s1_m3").reset_index()
    .rename(columns={SITE_BASIN_KEY: BASIN_KEY})
)
basin3 = basin.merge(b_s1, on=BASIN_KEY, how="left")
basin3["s1_m3"] = basin3["s1_m3"].fillna(0.0)
basin3 = basin3[basin3[BASIN_KEY].astype(str).isin(basins_with_sites.astype(str))].copy()

# BA totals for Scope 1/2 + MW + stress mean (unweighted)
ba_with_sites = sites[SITE_BA_KEY].astype(str).unique()
r_s1 = sites.groupby(SITE_BA_KEY)[S1_COL_M3].sum().rename("s1_m3").reset_index().rename(columns={SITE_BA_KEY: BA_KEY})
r_s2 = sites.groupby(SITE_BA_KEY)[S2_COL_M3].sum().rename("s2_m3").reset_index().rename(columns={SITE_BA_KEY: BA_KEY})
r_mw = sites.groupby(SITE_BA_KEY)[MW_COL].sum().rename("mw").reset_index().rename(columns={SITE_BA_KEY: BA_KEY})
r_st = sites.groupby(SITE_BA_KEY)[STRESS_COL_SITE].mean().rename("stress_mean").reset_index().rename(columns={SITE_BA_KEY: BA_KEY})

ba3 = (
    ba.merge(r_s1, on=BA_KEY, how="left")
      .merge(r_s2, on=BA_KEY, how="left")
      .merge(r_mw, on=BA_KEY, how="left")
      .merge(r_st, on=BA_KEY, how="left")
)
ba3[["s1_m3","s2_m3","mw","stress_mean"]] = ba3[["s1_m3","s2_m3","mw","stress_mean"]].fillna(0.0)
ba3 = ba3[ba3[BA_KEY].astype(str).isin(ba_with_sites.astype(str))].copy()

# Dominance map value
ba3["tot_m3"] = ba3["s1_m3"] + ba3["s2_m3"]
ba3["s2_share"] = np.where(ba3["tot_m3"] > 0, ba3["s2_m3"] / ba3["tot_m3"], np.nan)
ba3["_dom_cat"] = pd.cut(ba3["s2_share"], bins=S2_SHARE_BINS, labels=S2_SHARE_LABS)

# -----------------------------
# Basin “geographic labels” = dominant state abbrev(s) by overlap area
# (No counties available in your layers.)
# -----------------------------
def basin_to_state_label(basin_gdf, states_gdf, basin_id_col=BASIN_KEY, state_col="STUSPS"):
    if basin_gdf.crs != states_gdf.crs:
        states_gdf = states_gdf.to_crs(basin_gdf.crs)

    tmp = basin_gdf[[basin_id_col, "geometry"]].copy()
    tmp["_rp"] = tmp.geometry.representative_point()
    pts = tmp.set_geometry("_rp")

    joined = pts.sjoin(states_gdf[[state_col, "geometry"]], how="left", predicate="within")
    out = joined[[basin_id_col, state_col]].rename(columns={state_col: "state_lbl"})
    out["state_lbl"] = out["state_lbl"].fillna("")
    return out[[basin_id_col, "state_lbl"]]

# Build labels once
basin_state_lbl = basin_to_state_label_fast(basin3, st, basin_id_col=BASIN_KEY, state_col="STUSPS")
basin3 = basin3.merge(basin_state_lbl, on=BASIN_KEY, how="left")
basin3["state_lbl"] = basin3["state_lbl"].fillna("")

# Final label for bars
basin3["label"] = basin3[BASIN_KEY].astype(str) + " (" + basin3["state_lbl"].astype(str) + ")"

# For BA labels: prefer a human-readable name if present
if "region_ful" in ba3.columns:
    ba3["label"] = ba3["region_ful"].astype(str)
else:
    ba3["label"] = ba3[BA_KEY].astype(str)

# -----------------------------
# Concentration bars + inset + arrows + vertical line
# -----------------------------
def concentration_bar_components(values, labels, target=0.80, top_n=30):
    v = pd.Series(values).astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    lab = pd.Series(labels).astype(str).loc[v.index]

    v = v[v > 0]
    lab = lab.loc[v.index]

    if v.empty:
        return None

    order = v.sort_values(ascending=False).index
    v = v.loc[order].reset_index(drop=True)
    lab = lab.loc[order].reset_index(drop=True)

    share = v / v.sum()
    cum = share.cumsum()
    k = int((cum >= target).idxmax() + 1) if (cum >= target).any() else len(cum)
    cum_at_k = float(cum.iloc[k-1])

    # Plot only top_n bars + rest
    if len(share) > top_n:
        share_plot = np.r_[share.iloc[:top_n].values, float(share.iloc[top_n:].sum())]
        lab_plot = lab.iloc[:top_n].tolist() + ["Rest of regions"]
    else:
        share_plot = share.values
        lab_plot = lab.tolist()

    return dict(
        share_plot=share_plot,
        lab_plot=lab_plot,
        share_full=share,
        lab_full=lab,
        k=k,
        cum_at_k=cum_at_k,
        n_total=len(share)
    )

def draw_concentration_bars(
    ax, values, labels, title,
    target=0.80, top_n=30, arrow_n=4, inset_n=5,
    show_x_labels=False,
    # NEW: inset placement knobs
    info_anchor=(0.02, 0.96),   # (x,y) in axes fraction
    top_anchor=(0.02, 0.62),    # (x,y) in axes fraction
    info_ha="left",             # "left" or "right"
    top_ha="left",              # "left" or "right"
):
    out = concentration_bar_components(values, labels, target=target, top_n=top_n)
    if out is None:
        ax.text(0.5, 0.5, "No positive values", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
        return out

    share_plot = out["share_plot"]
    lab_plot   = out["lab_plot"]
    share_full = out["share_full"]
    lab_full   = out["lab_full"]
    k          = out["k"]
    cum_at_k   = out["cum_at_k"]
    n_total    = out["n_total"]

    x = np.arange(len(share_plot))
    ax.bar(x, share_plot, linewidth=0.0)

    ax.set_title(title, loc="left")
    ax.set_ylabel("Share of total water")
    ax.grid(axis="y", alpha=0.25)

    # Vertical line at k (if k beyond shown bars, line on the "rest" bar)
    x_k = min(k-1, len(share_plot)-1)
    ax.axvline(x_k, lw=1.4, alpha=0.9)

    # Build inset text
    top_list = []
    for i in range(min(inset_n, len(share_full))):
        top_list.append(f"{i+1}. {lab_full.iloc[i]} — {100*share_full.iloc[i]:.1f}%")
    inset_txt = "Top contributors:\n" + "\n".join(top_list)

    # ---- INSETS (now positionable)
    ax.text(
        info_anchor[0], info_anchor[1],
        f"Top {k}/{n_total} regions reach {100*target:.0f}%\n(cum={100*cum_at_k:.0f}%)",
        transform=ax.transAxes, ha=info_ha, va="top", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#333333", lw=0.8, alpha=0.96)
    )
    ax.text(
        top_anchor[0], top_anchor[1],
        inset_txt,
        transform=ax.transAxes, ha=top_ha, va="top", fontsize=8,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#333333", lw=0.8, alpha=0.96)
    )

    # Arrows for top bars (rank labels 1..arrow_n)
    y_max = float(np.nanmax(share_plot)) if len(share_plot) else 1.0
    for i in range(min(arrow_n, len(share_plot))):
        if lab_plot[i] == "Rest of regions":
            continue
        ax.annotate(
            f"{i+1}",
            xy=(i, share_plot[i]),
            xytext=(i + 0.6, share_plot[i] + 0.12*y_max),
            ha="left", va="bottom", fontsize=9,
            arrowprops=dict(arrowstyle="->", lw=0.9),
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="#333333", lw=0.6, alpha=0.95)
        )

    # X axis
    if show_x_labels:
        show_n = min(10, len(lab_plot))
        xt = list(range(show_n))
        if lab_plot[-1] == "Rest of regions" and (len(lab_plot)-1) not in xt:
            xt.append(len(lab_plot)-1)
        ax.set_xticks(xt)
        ax.set_xticklabels([lab_plot[i] for i in xt], rotation=45, ha="right", fontsize=8)
    else:
        ax.set_xticks([0, len(x)//2, len(x)-1])
        ax.set_xticklabels(["1", f"{len(x)//2 + 1}", str(len(x))])
    ax.set_xlabel("Ranked regions (descending)")

    ax.set_xlim(-0.6, len(x)-0.4)
    ax.set_ylim(0, y_max*1.35)

    return out
# -----------------------------
# Companion maps: highlight top ranks and label by rank number
# -----------------------------
def label_ranks_on_map_with_arrows(
    ax,
    gdf,
    rank_ids,
    id_col,
    fontsize=10,
    r0=0.06,          # base radial offset as fraction of map width
    r_step=0.03,      # additional push if label overlaps prior labels
    max_tries=12,     # how hard we try to avoid overlaps
    angles_deg=None,  # optional custom angles
    arrow_lw=0.9,
):
    """
    Place rank labels as bubbles OFFSET from polygon with leader arrows.

    - Uses representative_point for anchor.
    - Places labels around a circle (angles) and pushes outward if overlapping earlier labels.
    - Offsets scale with map extent so it works in projected CRS.
    """
    if not rank_ids:
        return

    # subset
    sub = gdf[gdf[id_col].astype(str).isin([str(x) for x in rank_ids])].copy()
    if sub.empty:
        return

    # anchors
    sub["_rp"] = sub.geometry.representative_point()

    # map extents for scaling offsets
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    w = abs(x1 - x0)
    h = abs(y1 - y0)
    base = r0 * w  # scale by width

    # rank mapping
    rank_map = {str(k): i + 1 for i, k in enumerate(rank_ids)}

    # angle plan: spread around to reduce collision
    if angles_deg is None:
        angles_deg = [35, 145, 255, 325, 80, 190, 300, 10]  # cycles if >8

    placed = []  # store placed label centers to avoid overlap

    for j, (_, r) in enumerate(sub.iterrows()):
        rid = str(r[id_col])
        rk = rank_map.get(rid, None)
        if rk is None:
            continue

        ax_x, ax_y = r["_rp"].x, r["_rp"].y

        # pick base angle by rank order (stable)
        ang = np.deg2rad(angles_deg[(rk - 1) % len(angles_deg)])

        # try increasing radius until no overlap
        for t in range(max_tries):
            rad = base + t * (r_step * w)

            lx = ax_x + rad * np.cos(ang)
            ly = ax_y + rad * np.sin(ang)

            # simple overlap check in data units
            ok = True
            for (px, py) in placed:
                if (abs(lx - px) < 0.03 * w) and (abs(ly - py) < 0.03 * h):
                    ok = False
                    break
            if ok:
                placed.append((lx, ly))

                # arrow/leader line
                ax.annotate(
                    "", xy=(ax_x, ax_y), xytext=(lx, ly),
                    arrowprops=dict(arrowstyle="->", lw=arrow_lw, color="#111111"),
                    zorder=60
                )

                # bubble
                ax.text(
                    lx, ly, str(rk),
                    ha="center", va="center",
                    fontsize=fontsize,
                    bbox=dict(boxstyle="circle,pad=0.22", fc="white", ec="#111111", lw=0.9, alpha=0.96),
                    zorder=61
                )
                break
# -----------------------------
# Layout: 2 rows × 2 cols
#   Row 1: Scope 1 bars | Scope 1 map (top basins)
#   Row 2: Scope 2 bars | BA dominance map (+ stress hatch)
# -----------------------------
minx, miny, maxx, maxy = st.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

fig = plt.figure(figsize=(15.5, 10.0))
gs = fig.add_gridspec(
    nrows=2, ncols=2,
    left=0.05, right=0.985, top=0.94, bottom=0.10,
    wspace=0.10, hspace=0.22
)

axA1 = fig.add_subplot(gs[0,0])
axA2 = fig.add_subplot(gs[0,1])
axB1 = fig.add_subplot(gs[1,0])
axB2 = fig.add_subplot(gs[1,1])

# ---- Row 1: Scope 1 concentration bars
out_s1 = draw_concentration_bars(
    axA1,
    basin3["s1_m3"], basin3["label"],
    "a1. Scope 1 is concentrated across a small number of hydrologic basins",
    target=TARGET_CUM, top_n=TOP_N_SHOW, arrow_n=TOP_ARROW_N, inset_n=TOP_INSET_N,
    show_x_labels=SHOW_X_LABELS
)

# ---- Row 1: Map showing top basins (rank-labeled)
basin_rank_ids = []
if out_s1 is not None:
    basin_rank_ids = basin3.sort_values("s1_m3", ascending=False)[BASIN_KEY].astype(str).head(MAP_LABEL_N).tolist()

# Basemap: light grey all hosting basins, darker for top ones
basin3.plot(ax=axA2, color="#eeeeee", edgecolor="white", linewidth=0.2, zorder=1)
if basin_rank_ids:
    basin3[basin3[BASIN_KEY].astype(str).isin(basin_rank_ids)].plot(
        ax=axA2, color="#9ecae1", edgecolor="#111111", linewidth=0.6, zorder=3
    )

st.boundary.plot(ax=axA2, color="#111111", linewidth=0.6, zorder=10)
label_ranks_on_map_with_arrows(axA2, basin3, basin_rank_ids, BASIN_KEY, fontsize=10)#label_ranks_on_map(axA2, basin3, basin_rank_ids, BASIN_KEY, fontsize=10)

axA2.set_title("a2. Where the top Scope 1 basins are (labels = rank in a1)", loc="left")
axA2.set_axis_off()
axA2.set_xlim(minx-padx, maxx+padx); axA2.set_ylim(miny-pady, maxy+pady)
axA2.set_aspect("equal")

# Legend below map (boxed, no overlap)
axA2.legend(
    handles=[
        Patch(facecolor="#9ecae1", edgecolor="#111111", label=f"Top {MAP_LABEL_N} basins by Scope 1"),
        Patch(facecolor="#eeeeee", edgecolor="white", label="Other basins hosting hyperscale sites"),
    ],
    loc="upper center", bbox_to_anchor=(0.5, -0.05), ncol=2,
    frameon=True, fontsize=9
)

# ---- Row 2: Scope 2 concentration bars
out_s2 = draw_concentration_bars(
    axB1,
    ba3["s2_m3"], ba3["label"],
    "b1. Scope 2 is even more concentrated across balancing authorities",
    target=TARGET_CUM, top_n=TOP_N_SHOW, arrow_n=TOP_ARROW_N, inset_n=7,
    show_x_labels=SHOW_X_LABELS,
    # move both boxes to the right
    info_anchor=(0.98, 0.96), info_ha="right",
    top_anchor=(0.98, 0.62),  top_ha="right"
)

# ---- Row 2: Scope 2 "where-they-are" map (mirrors a2)
# build rank ids (same ranks highlighted in b1 arrows/inset)
ba_rank_ids = []
if out_s2 is not None:
    ba_rank_ids = ba3.sort_values("s2_m3", ascending=False)[BA_KEY].astype(str).head(MAP_LABEL_N).tolist()

# Basemap: light grey all hosting BAs, darker for top ones
ba3.plot(ax=axB2, color="#eeeeee", edgecolor="white", linewidth=0.2, zorder=1)
if ba_rank_ids:
    ba3[ba3[BA_KEY].astype(str).isin(ba_rank_ids)].plot(
        ax=axB2, color="#fdae6b", edgecolor="#111111", linewidth=0.6, zorder=3
    )

st.boundary.plot(ax=axB2, color="#111111", linewidth=0.6, zorder=10)

# IMPORTANT: set limits BEFORE calling label_ranks... (your function reads ax limits)
axB2.set_axis_off()
axB2.set_xlim(minx-padx, maxx+padx); axB2.set_ylim(miny-pady, maxy+pady)
axB2.set_aspect("equal")

label_ranks_on_map_with_arrows(axB2, ba3, ba_rank_ids, BA_KEY, fontsize=10)

axB2.set_title("b2. Where the top Scope 2 balancing authorities are (labels = rank in b1)", loc="left")

axB2.legend(
    handles=[
        Patch(facecolor="#fdae6b", edgecolor="#111111", label=f"Top {MAP_LABEL_N} BAs by Scope 2"),
        Patch(facecolor="#eeeeee", edgecolor="white", label="Other BAs hosting hyperscale sites"),
    ],
    loc="upper center", bbox_to_anchor=(0.5, -0.05), ncol=2,
    frameon=True, fontsize=9
)

# stress hatch overlay
hi_st = ba3_plot[ba3_plot["stress_mean"] >= STRESS_THR]
if len(hi_st):
    hi_st.plot(ax=axB2, facecolor="none", edgecolor=HATCH_EDGE, linewidth=0.0, hatch=HATCH, zorder=3)

# outline top BAs by total water (top quartile of tot_m3 among hosting)
host_mask = ba3_plot["tot_m3"] > 0
if host_mask.any():
    thr_tot = float(ba3_plot.loc[host_mask, "tot_m3"].quantile(0.75))
    hi_burden = ba3_plot[ba3_plot["tot_m3"] >= thr_tot]
    if len(hi_burden):
        hi_burden.boundary.plot(ax=axB2, color="#000000", linewidth=1.6, zorder=5)

st.boundary.plot(ax=axB2, color="#111111", linewidth=0.6, zorder=10)
axB2.set_title(
    "b2. Which pathway dominates—and where stress co-occurs\n"
    "(fill: dominance; hatch: mean siting stress ≥ 3; outline: top-quartile total water)",
    loc="left"
)
axB2.set_axis_off()
axB2.set_xlim(minx-padx, maxx+padx); axB2.set_ylim(miny-pady, maxy+pady)
axB2.set_aspect("equal")

# Legend below map (boxed)
axB2.legend(
    handles=[
        Patch(facecolor=DOM_COLORS["Scope 1–dominant"], edgecolor="white", label="Scope 1–dominant (Scope 2 share < 0.40)"),
        Patch(facecolor=DOM_COLORS["Mixed"], edgecolor="white", label="Mixed (0.40–0.60)"),
        Patch(facecolor=DOM_COLORS["Scope 2–dominant"], edgecolor="white", label="Scope 2–dominant (Scope 2 share > 0.60)"),
        Patch(facecolor="white", edgecolor=HATCH_EDGE, hatch=HATCH, label="High siting stress (mean ≥ 3)"),
        Patch(facecolor="none", edgecolor="#000000", label="High-burden BA (top 25% total water)"),
    ],
    loc="upper center", bbox_to_anchor=(0.5, -0.05), ncol=2,
    frameon=True, fontsize=9
)

# Export PDF + show
plt.savefig(OUT_PDF, bbox_inches="tight")
plt.show()
print("Saved:", OUT_PDF)

In [ ]:
# ==========================================
# FIGURE 3 (FINAL COHERENT): CONCENTRATION BARS + WHERE-THEY-ARE MAPS
# Story (single coherent narrative):
#   Row A = Scope 1 (Basins): concentration -> where are top basins?
#   Row B = Scope 2 (BAs):    concentration -> where are top BAs?
# Notes:
# - Bars only (no background curves)
# - Inset boxes are boxed and moveable; BA inset moved to right to avoid overlap
# - Maps highlight top contributors and label ranks with leader arrows
# - Basin labels use dominant state abbrev from representative point (no counties available)
# - Legends are BELOW maps, boxed, no overlap
# - Exports PDF only + always shows figure in notebook
# ==========================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

OUTDIR = "images"
os.makedirs(OUTDIR, exist_ok=True)
OUT_PDF = os.path.join(OUTDIR, "Fig3_Concentration_BarsPlusMaps.pdf")

BA_KEY = 'subregion_id'

# -----------------------------
# KNOBS (edit here)
# -----------------------------
TARGET_CUM    = 0.80   # vertical line = smallest k reaching this cumulative share
TOP_N_SHOW    = 30     # how many ranked bars to show (rest aggregated)
TOP_ARROW_N   = 7      # how many top bars get arrows + rank labels
TOP_INSET_N   = 7      # how many shown in the inset box
SHOW_X_LABELS = False  # if True, put top labels on x-axis (can crowd)

# Map highlight + rank labeling:
MAP_LABEL_N = 7        # label ranks 1..MAP_LABEL_N on maps (match arrows)

# Map colors
BASIN_TOP_COLOR = "#9ecae1"
BA_TOP_COLOR    = "#fdae6b"
OTHER_COLOR     = "#eeeeee"
BORDER_COLOR    = "#111111"

# -----------------------------
# Prep & required columns
# -----------------------------
# Use gdf_ba_l if present (fine), but Fig3 only needs geometry + BA key; no fossil/stress needed here.
sites, basin, ba, st = _prep_conus(
    gdf_sites,
    gdf_basin,
    gdf_ba_l if "gdf_ba_l" in globals() else gdf_ba,
    states48
)

need_sites = [MW_COL, S1_COL_M3, S2_COL_M3, STRESS_COL_SITE, SITE_BASIN_KEY, SITE_BA_KEY]
for c in need_sites:
    if c not in sites.columns:
        raise KeyError(f"gdf_sites missing `{c}` needed for Fig 3.")

sites[MW_COL]        = pd.to_numeric(sites[MW_COL], errors="coerce").fillna(0.0)
sites[S1_COL_M3]     = pd.to_numeric(sites[S1_COL_M3], errors="coerce").fillna(0.0)
sites[S2_COL_M3]     = pd.to_numeric(sites[S2_COL_M3], errors="coerce").fillna(0.0)
sites[STRESS_COL_SITE]= pd.to_numeric(sites[STRESS_COL_SITE], errors="coerce").fillna(0.0)

# -----------------------------
# Aggregations
# -----------------------------
# Basin totals for Scope 1
basins_with_sites = sites[SITE_BASIN_KEY].astype(str).unique()
b_s1 = (
    sites.groupby(SITE_BASIN_KEY)[S1_COL_M3].sum()
    .rename("s1_m3").reset_index()
    .rename(columns={SITE_BASIN_KEY: BASIN_KEY})
)
basin3 = basin.merge(b_s1, on=BASIN_KEY, how="left")
basin3["s1_m3"] = basin3["s1_m3"].fillna(0.0)
basin3 = basin3[basin3[BASIN_KEY].astype(str).isin(basins_with_sites.astype(str))].copy()

# BA totals for Scope 2
ba_with_sites = sites[SITE_BA_KEY].astype(str).unique()
r_s2 = (
    sites.groupby(SITE_BA_KEY)[S2_COL_M3].sum()
    .rename("s2_m3").reset_index()
    .rename(columns={SITE_BA_KEY: BA_KEY})
)
ba3 = ba.merge(r_s2, on=BA_KEY, how="left")
ba3["s2_m3"] = ba3["s2_m3"].fillna(0.0)
ba3 = ba3[ba3[BA_KEY].astype(str).isin(ba_with_sites.astype(str))].copy()

# -----------------------------
# Basin “geographic labels” = state abbrev from representative point
# -----------------------------
def basin_to_state_label_fast(basin_gdf, states_gdf, basin_id_col=BASIN_KEY, state_col="STUSPS"):
    if basin_gdf.crs != states_gdf.crs:
        states_gdf = states_gdf.to_crs(basin_gdf.crs)

    tmp = basin_gdf[[basin_id_col, "geometry"]].copy()
    tmp["_rp"] = tmp.geometry.representative_point()
    pts = tmp.set_geometry("_rp")

    joined = pts.sjoin(states_gdf[[state_col, "geometry"]], how="left", predicate="within")
    out = joined[[basin_id_col, state_col]].rename(columns={state_col: "state_lbl"})
    out["state_lbl"] = out["state_lbl"].fillna("")
    return out[[basin_id_col, "state_lbl"]]

basin_state_lbl = basin_to_state_label_fast(basin3, st, basin_id_col=BASIN_KEY, state_col="STUSPS")
basin3 = basin3.merge(basin_state_lbl, on=BASIN_KEY, how="left")
basin3["state_lbl"] = basin3["state_lbl"].fillna("")
basin3["label"] = basin3[BASIN_KEY].astype(str) + " (" + basin3["state_lbl"].astype(str) + ")"

# BA labels: prefer readable name if present
if "region_ful" in ba3.columns:
    ba3["label"] = ba3["region_ful"].astype(str)
else:
    ba3["label"] = ba3[BA_KEY].astype(str)

# -----------------------------
# Concentration bars helper
# -----------------------------
def concentration_bar_components(values, labels, target=0.80, top_n=30):
    v = pd.Series(values).astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    lab = pd.Series(labels).astype(str).loc[v.index]

    v = v[v > 0]
    lab = lab.loc[v.index]
    if v.empty:
        return None

    order = v.sort_values(ascending=False).index
    v = v.loc[order].reset_index(drop=True)
    lab = lab.loc[order].reset_index(drop=True)

    share = v / v.sum()
    cum = share.cumsum()
    k = int((cum >= target).idxmax() + 1) if (cum >= target).any() else len(cum)
    cum_at_k = float(cum.iloc[k-1])

    if len(share) > top_n:
        share_plot = np.r_[share.iloc[:top_n].values, float(share.iloc[top_n:].sum())]
        lab_plot = lab.iloc[:top_n].tolist() + ["Rest of regions"]
    else:
        share_plot = share.values
        lab_plot = lab.tolist()

    return dict(
        share_plot=share_plot, lab_plot=lab_plot,
        share_full=share, lab_full=lab,
        k=k, cum_at_k=cum_at_k, n_total=len(share)
    )

def draw_concentration_bars(
    ax, values, labels, title,
    target=0.80, top_n=30, arrow_n=7, inset_n=7, show_x_labels=False,
    info_anchor=(0.02, 0.96), top_anchor=(0.02, 0.62),
    info_ha="left", top_ha="left"
):
    out = concentration_bar_components(values, labels, target=target, top_n=top_n)
    if out is None:
        ax.text(0.5, 0.5, "No positive values", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
        return None

    share_plot = out["share_plot"]
    lab_plot   = out["lab_plot"]
    share_full = out["share_full"]
    lab_full   = out["lab_full"]
    k          = out["k"]
    cum_at_k   = out["cum_at_k"]
    n_total    = out["n_total"]

    x = np.arange(len(share_plot))
    ax.bar(x, share_plot, linewidth=0.0)

    ax.set_title(title, loc="left")
    ax.set_ylabel("Share of total water")
    ax.grid(axis="y", alpha=0.25)

    # vertical line at k
    x_k = min(k-1, len(share_plot)-1)
    ax.axvline(x_k, lw=1.4, alpha=0.9)

    # inset text
    top_list = []
    for i in range(min(inset_n, len(share_full))):
        top_list.append(f"{i+1}. {lab_full.iloc[i]} — {100*share_full.iloc[i]:.1f}%")
    inset_txt = "Top contributors:\n" + "\n".join(top_list)

    ax.text(
        info_anchor[0], info_anchor[1],
        f"Top {k}/{n_total} regions reach {100*target:.0f}%\n(cum={100*cum_at_k:.0f}%)",
        transform=ax.transAxes, ha=info_ha, va="top", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#333333", lw=0.8, alpha=0.96)
    )
    ax.text(
        top_anchor[0], top_anchor[1],
        inset_txt,
        transform=ax.transAxes, ha=top_ha, va="top", fontsize=8,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#333333", lw=0.8, alpha=0.96)
    )

    # arrows for top bars
    y_max = float(np.nanmax(share_plot)) if len(share_plot) else 1.0
    for i in range(min(arrow_n, len(share_plot))):
        if lab_plot[i] == "Rest of regions":
            continue
        ax.annotate(
            f"{i+1}",
            xy=(i, share_plot[i]),
            xytext=(i + 0.6, share_plot[i] + 0.12*y_max),
            ha="left", va="bottom", fontsize=9,
            arrowprops=dict(arrowstyle="->", lw=0.9),
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="#333333", lw=0.6, alpha=0.95)
        )

    # x-axis
    if show_x_labels:
        show_n = min(10, len(lab_plot))
        xt = list(range(show_n))
        if lab_plot[-1] == "Rest of regions" and (len(lab_plot)-1) not in xt:
            xt.append(len(lab_plot)-1)
        ax.set_xticks(xt)
        ax.set_xticklabels([lab_plot[i] for i in xt], rotation=45, ha="right", fontsize=8)
    else:
        ax.set_xticks([0, len(x)//2, len(x)-1])
        ax.set_xticklabels(["1", f"{len(x)//2 + 1}", str(len(x))])
    ax.set_xlabel("Ranked regions (descending)")

    ax.set_xlim(-0.6, len(x)-0.4)
    ax.set_ylim(0, y_max*1.35)

    return out

# -----------------------------
# Map rank labels with leader arrows (so labels don't hide polygons)
# IMPORTANT: call after ax limits are set (function reads ax.get_xlim/ylim)
# -----------------------------
def label_ranks_on_map_with_arrows(
    ax,
    gdf,
    rank_ids,
    id_col,
    fontsize=10,
    r0=0.06,
    r_step=0.03,
    max_tries=12,
    angles_deg=None,
    arrow_lw=0.9,
):
    if not rank_ids:
        return

    sub = gdf[gdf[id_col].astype(str).isin([str(x) for x in rank_ids])].copy()
    if sub.empty:
        return

    sub["_rp"] = sub.geometry.representative_point()

    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    w = abs(x1 - x0)
    h = abs(y1 - y0)
    base = r0 * w

    rank_map = {str(k): i + 1 for i, k in enumerate(rank_ids)}

    if angles_deg is None:
        angles_deg = [35, 145, 255, 325, 80, 190, 300, 10]  # cycles if >8

    placed = []
    # iterate in rank order (stable)
    for rk, rid in enumerate(rank_ids, start=1):
        rrows = sub[sub[id_col].astype(str) == str(rid)]
        if rrows.empty:
            continue
        r = rrows.iloc[0]
        ax_x, ax_y = r["_rp"].x, r["_rp"].y
        ang = np.deg2rad(angles_deg[(rk - 1) % len(angles_deg)])

        for t in range(max_tries):
            rad = base + t * (r_step * w)
            lx = ax_x + rad * np.cos(ang)
            ly = ax_y + rad * np.sin(ang)

            ok = True
            for (px, py) in placed:
                if (abs(lx - px) < 0.03 * w) and (abs(ly - py) < 0.03 * h):
                    ok = False
                    break
            if ok:
                placed.append((lx, ly))
                ax.annotate(
                    "", xy=(ax_x, ax_y), xytext=(lx, ly),
                    arrowprops=dict(arrowstyle="->", lw=arrow_lw, color=BORDER_COLOR),
                    zorder=60
                )
                ax.text(
                    lx, ly, str(rk),
                    ha="center", va="center",
                    fontsize=fontsize,
                    bbox=dict(boxstyle="circle,pad=0.22", fc="white", ec=BORDER_COLOR, lw=0.9, alpha=0.96),
                    zorder=61
                )
                break

# -----------------------------
# Layout: 2 rows × 2 cols
#   Row 1: Scope 1 bars | Scope 1 map (top basins)
#   Row 2: Scope 2 bars | Scope 2 map (top BAs)
# -----------------------------
minx, miny, maxx, maxy = st.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

fig = plt.figure(figsize=(15.5, 10.0))
gs = fig.add_gridspec(
    nrows=2, ncols=2,
    left=0.05, right=0.985, top=0.94, bottom=0.10,
    wspace=0.10, hspace=0.22
)

axA1 = fig.add_subplot(gs[0,0])
axA2 = fig.add_subplot(gs[0,1])
axB1 = fig.add_subplot(gs[1,0])
axB2 = fig.add_subplot(gs[1,1])

# ---- Row 1: Scope 1 concentration bars
out_s1 = draw_concentration_bars(
    axA1,
    basin3["s1_m3"], basin3["label"],
    "a1. Scope 1 is concentrated across a small number of hydrologic basins",
    target=TARGET_CUM, top_n=TOP_N_SHOW, arrow_n=TOP_ARROW_N, inset_n=TOP_INSET_N,
    show_x_labels=SHOW_X_LABELS,
    info_anchor=(0.02, 0.96), info_ha="left",
    top_anchor=(0.02, 0.62),  top_ha="left"
)

# ---- Row 1: Scope 1 map (top basins)
basin_rank_ids = []
if out_s1 is not None:
    basin_rank_ids = basin3.sort_values("s1_m3", ascending=False)[BASIN_KEY].astype(str).head(MAP_LABEL_N).tolist()

basin3.plot(ax=axA2, color=OTHER_COLOR, edgecolor="white", linewidth=0.2, zorder=1)
if basin_rank_ids:
    basin3[basin3[BASIN_KEY].astype(str).isin(basin_rank_ids)].plot(
        ax=axA2, color=BASIN_TOP_COLOR, edgecolor=BORDER_COLOR, linewidth=0.6, zorder=3
    )

st.boundary.plot(ax=axA2, color=BORDER_COLOR, linewidth=0.6, zorder=10)

axA2.set_title("a2. Where the top Scope 1 basins are (labels = rank in a1)", loc="left")
axA2.set_axis_off()
axA2.set_xlim(minx-padx, maxx+padx); axA2.set_ylim(miny-pady, maxy+pady)
axA2.set_aspect("equal")

# leader-arrow labels AFTER limits set
label_ranks_on_map_with_arrows(ax=axA2, gdf=basin3, rank_ids=basin_rank_ids, id_col=BASIN_KEY, fontsize=10)

axA2.legend(
    handles=[
        Patch(facecolor=BASIN_TOP_COLOR, edgecolor=BORDER_COLOR, label=f"Top {MAP_LABEL_N} basins by Scope 1"),
        Patch(facecolor=OTHER_COLOR, edgecolor="white", label="Other basins hosting hyperscale sites"),
    ],
    loc="upper center", bbox_to_anchor=(0.5, -0.05), ncol=2,
    frameon=True, fontsize=9
)

# ---- Row 2: Scope 2 concentration bars (move boxes right to avoid overlap)
out_s2 = draw_concentration_bars(
    axB1,
    ba3["s2_m3"], ba3["label"],
    "b1. Scope 2 is concentrated across a small number of balancing authorities",
    target=TARGET_CUM, top_n=TOP_N_SHOW, arrow_n=TOP_ARROW_N, inset_n=TOP_INSET_N,
    show_x_labels=SHOW_X_LABELS,
    info_anchor=(0.98, 0.96), info_ha="right",
    top_anchor=(0.98, 0.62),  top_ha="right"
)

# ---- Row 2: Scope 2 map (top BAs)

BA_ALL_LW  = 0.55   # <- increase this (all BA borders)
BA_TOP_LW  = 1.60   # <- increase this (top BA borders)
BA_TOP_EDGE = "#000000"

ba3.plot(ax=axB2, color=OTHER_COLOR, edgecolor="white", linewidth=BA_ALL_LW, zorder=1)

if ba_rank_ids:
    ba3[ba3[BA_KEY].astype(str).isin(ba_rank_ids)].plot(
        ax=axB2, color=BA_TOP_COLOR, edgecolor=BA_TOP_EDGE, linewidth=BA_TOP_LW, zorder=3
    )

st.boundary.plot(ax=axB2, color=BORDER_COLOR, linewidth=0.6, zorder=10)

axB2.set_title("b2. Where the top Scope 2 balancing authorities are (labels = rank in b1)", loc="left")
axB2.set_axis_off()
axB2.set_xlim(minx-padx, maxx+padx); axB2.set_ylim(miny-pady, maxy+pady)
axB2.set_aspect("equal")

label_ranks_on_map_with_arrows(ax=axB2, gdf=ba3, rank_ids=ba_rank_ids, id_col=BA_KEY, fontsize=10)

axB2.legend(
    handles=[
        Patch(facecolor=BA_TOP_COLOR, edgecolor=BORDER_COLOR, label=f"Top {MAP_LABEL_N} BAs by Scope 2"),
        Patch(facecolor=OTHER_COLOR, edgecolor="white", label="Other BAs hosting hyperscale sites"),
    ],
    loc="upper center", bbox_to_anchor=(0.5, -0.05), ncol=2,
    frameon=True, fontsize=9
)

# Export PDF + show
plt.savefig(OUT_PDF, bbox_inches="tight")
plt.show()
print("Saved:", OUT_PDF)

In [ ]:
ba3

In [ ]:
# -----------------------------
# Panel A helper: bars-only concentration (no cumulative line)
# - Bars = region share of total, ranked
# - Vertical line at k (e.g., where cumulative reaches 80%)
# - Optional arrows to top N bars, showing basin/BA IDs
# -----------------------------

TOP_LABEL_N = 4        # how many top bars get arrow labels
SHOW_X_IDS  = False    # if True, put top IDs on x-axis (can get crowded)
X_ID_N      = 10       # if SHOW_X_IDS, how many IDs to show on x-axis

def concentration_bar_components(values, ids, target=0.80, top_n=35):
    v = pd.Series(values).astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    ids = pd.Series(ids).astype(str).loc[v.index]

    v = v[v > 0]
    ids = ids.loc[v.index]

    if v.empty:
        return None

    order = v.sort_values(ascending=False).index
    v = v.loc[order].reset_index(drop=True)
    ids = ids.loc[order].reset_index(drop=True)

    share = v / v.sum()
    cum = share.cumsum()

    # k where cumulative reaches target
    k = int((cum >= target).idxmax() + 1) if (cum >= target).any() else len(cum)
    frac_k = k / len(cum)

    # plot only top_n bars, aggregate rest
    if len(share) > top_n:
        share_top = share.iloc[:top_n]
        ids_top = ids.iloc[:top_n].tolist()
        rest_share = float(share.iloc[top_n:].sum())
        share_plot = np.r_[share_top.values, rest_share]
        ids_plot = ids_top + ["rest"]
        is_rest = np.r_[np.zeros(top_n, dtype=bool), True]
    else:
        share_plot = share.values
        ids_plot = ids.tolist()
        is_rest = np.zeros(len(share_plot), dtype=bool)

    return dict(
        share_plot=share_plot,
        ids_plot=ids_plot,
        k=k,
        frac_k=frac_k,
        target=target,
        n_total=len(cum),
        cum_at_k=float(cum.iloc[k-1]) if len(cum) else np.nan
    )

def draw_concentration_bars(
    ax, values, labels, title,
    target=0.80, top_n=30, arrow_n=4, inset_n=5,
    show_x_labels=False,
    # NEW: inset placement knobs
    info_anchor=(0.02, 0.96),   # (x,y) in axes fraction
    top_anchor=(0.02, 0.62),    # (x,y) in axes fraction
    info_ha="left",             # "left" or "right"
    top_ha="left",              # "left" or "right"
):
    out = concentration_bar_components(values, labels, target=target, top_n=top_n)
    if out is None:
        ax.text(0.5, 0.5, "No positive values", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
        return out

    share_plot = out["share_plot"]
    lab_plot   = out["lab_plot"]
    share_full = out["share_full"]
    lab_full   = out["lab_full"]
    k          = out["k"]
    cum_at_k   = out["cum_at_k"]
    n_total    = out["n_total"]

    x = np.arange(len(share_plot))
    ax.bar(x, share_plot, linewidth=0.0)

    ax.set_title(title, loc="left")
    ax.set_ylabel("Share of total water")
    ax.grid(axis="y", alpha=0.25)

    # Vertical line at k (if k beyond shown bars, line on the "rest" bar)
    x_k = min(k-1, len(share_plot)-1)
    ax.axvline(x_k, lw=1.4, alpha=0.9)

    # Build inset text
    top_list = []
    for i in range(min(inset_n, len(share_full))):
        top_list.append(f"{i+1}. {lab_full.iloc[i]} — {100*share_full.iloc[i]:.1f}%")
    inset_txt = "Top contributors:\n" + "\n".join(top_list)

    # ---- INSETS (now positionable)
    ax.text(
        info_anchor[0], info_anchor[1],
        f"Top {k}/{n_total} regions reach {100*target:.0f}%\n(cum={100*cum_at_k:.0f}%)",
        transform=ax.transAxes, ha=info_ha, va="top", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#333333", lw=0.8, alpha=0.96)
    )
    ax.text(
        top_anchor[0], top_anchor[1],
        inset_txt,
        transform=ax.transAxes, ha=top_ha, va="top", fontsize=8,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#333333", lw=0.8, alpha=0.96)
    )

    # Arrows for top bars (rank labels 1..arrow_n)
    y_max = float(np.nanmax(share_plot)) if len(share_plot) else 1.0
    for i in range(min(arrow_n, len(share_plot))):
        if lab_plot[i] == "Rest of regions":
            continue
        ax.annotate(
            f"{i+1}",
            xy=(i, share_plot[i]),
            xytext=(i + 0.6, share_plot[i] + 0.12*y_max),
            ha="left", va="bottom", fontsize=9,
            arrowprops=dict(arrowstyle="->", lw=0.9),
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="#333333", lw=0.6, alpha=0.95)
        )

    # X axis
    if show_x_labels:
        show_n = min(10, len(lab_plot))
        xt = list(range(show_n))
        if lab_plot[-1] == "Rest of regions" and (len(lab_plot)-1) not in xt:
            xt.append(len(lab_plot)-1)
        ax.set_xticks(xt)
        ax.set_xticklabels([lab_plot[i] for i in xt], rotation=45, ha="right", fontsize=8)
    else:
        ax.set_xticks([0, len(x)//2, len(x)-1])
        ax.set_xticklabels(["1", f"{len(x)//2 + 1}", str(len(x))])
    ax.set_xlabel("Ranked regions (descending)")

    ax.set_xlim(-0.6, len(x)-0.4)
    ax.set_ylim(0, y_max*1.35)

    return out

In [ ]:
# a1: Scope 1 concentration across basins (bars only)
draw_concentration_bars(
    axA1,
    basin3["s1_m3"],
    basin3[BASIN_KEY],
    "a1. Scope 1 concentration across basins",
    target=TARGET_CUM,
    top_n=TOP_N_SHOW,
    top_label_n=TOP_LABEL_N,
    show_x_ids=SHOW_X_IDS,
    x_id_n=X_ID_N
)

# a2: Scope 2 concentration across BAs (bars only)
draw_concentration_bars(
    axA2,
    ba3["s2_m3"],
    ba3[BA_KEY],
    "a2. Scope 2 concentration across balancing authorities",
    target=TARGET_CUM,
    top_n=TOP_N_SHOW,
    top_label_n=TOP_LABEL_N,
    show_x_ids=SHOW_X_IDS,
    x_id_n=X_ID_N
)

In [ ]:
# ==========================================
# FIGURE 4: ACTION LEVERS (WHERE TO ACT)
# ==========================================
sites, basin, ba, st = _prep_conus(gdf_sites, gdf_basin, gdf_ba, states48)

for c in [SITE_BASIN_KEY, SITE_BA_KEY, MW_COL, S1_COL_M3, S2_COL_M3, STRESS_COL_SITE]:
    if c not in sites.columns:
        raise KeyError(f"gdf_sites missing `{c}` needed for Fig 4.")

sites[MW_COL] = pd.to_numeric(sites[MW_COL], errors="coerce").fillna(0.0)
sites[S1_COL_M3] = pd.to_numeric(sites[S1_COL_M3], errors="coerce").fillna(0.0)
sites[S2_COL_M3] = pd.to_numeric(sites[S2_COL_M3], errors="coerce").fillna(0.0)
sites[STRESS_COL_SITE] = pd.to_numeric(sites[STRESS_COL_SITE], errors="coerce").fillna(0.0)

# Baseline electricity (kWh/yr) from MW and utilization u_base
# MW * 8760 h = MWh; *1000 = kWh
sites["E_fac_kWh_base"] = sites[MW_COL] * 8760.0 * u_base * 1000.0

# -----------------------------
# Scope 1 "efficiency benchmark" savings:
# scale baseline S1 by ratio of u*(WUE/PUE)
# baseline corresponds to u_base, wue_base, pue_base
# -----------------------------
scale_s1_eff = (u_eff / u_base) * ((wue_eff / pue_eff) / (wue_base / pue_base))
sites["s1_m3_eff"] = sites[S1_COL_M3] * scale_s1_eff
sites["s1_savings_m3"] = np.maximum(sites[S1_COL_M3] - sites["s1_m3_eff"], 0.0)

# Aggregate to basin
s1sav_b = sites.groupby(SITE_BASIN_KEY)["s1_savings_m3"].sum().rename("s1_savings_m3").reset_index().rename(columns={SITE_BASIN_KEY:BASIN_KEY})
basin4 = basin.merge(s1sav_b, on=BASIN_KEY, how="left")
basin4["s1_savings_m3"] = basin4["s1_savings_m3"].fillna(0.0)

top_basin = (basin4[[BASIN_KEY, "s1_savings_m3"]]
             .sort_values("s1_savings_m3", ascending=False)
             .head(12)
             .copy())
top_basin["s1_savings_GL"] = m3_to_GL(top_basin["s1_savings_m3"])

# -----------------------------
# Scope 2 "low-water grid benchmark" savings:
# derive current BA water intensity from S2 and electricity:
# I_ba = S2_L / E_kWh
# then set I_new = min(I_ba, P25(I_ba)) and compute savings
# -----------------------------
# Aggregate BA totals: S2_L and E_kWh
tmp = sites.copy()
tmp["s2_L_base"] = tmp[S2_COL_M3] * 1000.0  # m3 -> L
baE = tmp.groupby(SITE_BA_KEY)["E_fac_kWh_base"].sum().rename("E_kWh").reset_index().rename(columns={SITE_BA_KEY:BA_KEY})
baS2 = tmp.groupby(SITE_BA_KEY)["s2_L_base"].sum().rename("S2_L").reset_index().rename(columns={SITE_BA_KEY:BA_KEY})
baStress = tmp.groupby(SITE_BA_KEY)[STRESS_COL_SITE].mean().rename("stress_mean").reset_index().rename(columns={SITE_BA_KEY:BA_KEY})

ba4 = ba.merge(baE, on=BA_KEY, how="left").merge(baS2, on=BA_KEY, how="left").merge(baStress, on=BA_KEY, how="left")
ba4[["E_kWh","S2_L","stress_mean"]] = ba4[["E_kWh","S2_L","stress_mean"]].fillna(0.0)
ba4 = ba4[ba4["E_kWh"] > 0].copy()
ba4["I_L_per_kWh"] = ba4["S2_L"] / ba4["E_kWh"]

I_target = float(ba4["I_L_per_kWh"].quantile(0.25))
ba4["I_target"] = np.minimum(ba4["I_L_per_kWh"], I_target)
ba4["S2_L_new"] = ba4["I_target"] * ba4["E_kWh"]
ba4["s2_savings_L"] = np.maximum(ba4["S2_L"] - ba4["S2_L_new"], 0.0)
ba4["s2_savings_m3"] = ba4["s2_savings_L"] / 1000.0
ba4["s2_savings_GL"] = m3_to_GL(ba4["s2_savings_m3"])

top_ba = (ba4[[BA_KEY, "s2_savings_GL", "stress_mean", "I_L_per_kWh"]]
          .sort_values("s2_savings_GL", ascending=False)
          .head(12)
          .copy())

# -----------------------------
# Plot Figure 4 (two horizontal bar panels)
# -----------------------------
fig = plt.figure(figsize=(14.5, 6.8))
gs = fig.add_gridspec(nrows=1, ncols=2, left=0.06, right=0.985, top=0.90, bottom=0.12, wspace=0.22)
axA = fig.add_subplot(gs[0,0])
axB = fig.add_subplot(gs[0,1])

# Panel A: basins
yA = np.arange(len(top_basin))[::-1]
axA.barh(yA, top_basin["s1_savings_GL"].values[::-1])
axA.set_yticks(yA)
axA.set_yticklabels(top_basin[BASIN_KEY].values[::-1], fontsize=9)
axA.set_xlabel("Potential Scope 1 reduction (GL/yr)")
axA.set_title("a. Largest potential Scope 1 reductions\n(Efficiency benchmark: u,WUE,PUE at low end)")
axA.grid(axis="x", alpha=0.25)

# Panel B: BAs
yB = np.arange(len(top_ba))[::-1]
axB.barh(yB, top_ba["s2_savings_GL"].values[::-1])
axB.set_yticks(yB)
axB.set_yticklabels(top_ba[BA_KEY].values[::-1], fontsize=9)
axB.set_xlabel("Potential Scope 2 reduction (GL/yr)")
axB.set_title(f"b. Largest potential Scope 2 reductions\n(Low-water grid benchmark: intensity → P25 = {I_target:.2f} L/kWh)")
axB.grid(axis="x", alpha=0.25)

# Add small annotations for stress and intensity (right side)
for i, (_, row) in enumerate(top_ba.iloc[::-1].reset_index(drop=True).iterrows()):
    axB.text(
        x=float(top_ba["s2_savings_GL"].max())*0.02,
        y=i,
        s=f"stress={row['stress_mean']:.2f}, I={row['I_L_per_kWh']:.2f}",
        va="center", fontsize=8
    )

out_pdf = os.path.join(OUTDIR, "Fig4_Action_Levers.pdf")
plt.savefig(out_pdf, bbox_inches="tight")
plt.show()
print("Saved:", out_pdf)

In [ ]:
# ==========================================
# FIGURE 4A: ACTION LEVERS (UPGRADED, YOUR ORIGINAL IDEA)
# - Panel a: Scope 1 potential reduction (Efficiency benchmark)
# - Panel b: Scope 2 potential reduction (Low-water grid benchmark)
# - Better labels (basin -> state abbrev; BA -> region name)
# - Stress cue (hatch overlay) + boxed legend
# - PDF export only + show in notebook
# ==========================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

OUTDIR = "images"
os.makedirs(OUTDIR, exist_ok=True)
OUT_PDF = os.path.join(OUTDIR, "Fig4A_ActionLevers_Bars.pdf")

# -----------------------------
# REQUIRED: scenario parameters must exist in env
# -----------------------------
# u_base, pue_base, wue_base
# u_eff,  pue_eff,  wue_eff
# (you already used these in your old snippet)

# -----------------------------
# KNOBS (edit here)
# -----------------------------
TOP_N = 12
STRESS_THR = 3.0          # mean siting stress threshold for highlighting
GRID_BENCH_Q = 0.25       # P25 benchmark for grid water intensity (as you used)

# -----------------------------
# Helpers
# -----------------------------
def m3_to_GL(x):  # 1 GL = 1e6 m3
    return np.asarray(x, dtype=float) / 1e6

def basin_to_state_label_fast(basin_gdf, states_gdf, basin_id_col, state_col="STUSPS"):
    """Representative-point join to get a simple, readable 'state' label for each basin."""
    if basin_gdf.crs != states_gdf.crs:
        states_gdf = states_gdf.to_crs(basin_gdf.crs)
    tmp = basin_gdf[[basin_id_col, "geometry"]].copy()
    tmp["_rp"] = tmp.geometry.representative_point()
    pts = tmp.set_geometry("_rp")
    j = pts.sjoin(states_gdf[[state_col, "geometry"]], how="left", predicate="within")
    out = j[[basin_id_col, state_col]].rename(columns={state_col: "state_lbl"})
    out["state_lbl"] = out["state_lbl"].fillna("")
    return out[[basin_id_col, "state_lbl"]]

def pick_ba_name_col(ba_gdf):
    for c in ["region_ful", "region", "region_BA"]:
        if c in ba_gdf.columns:
            return c
    return None

# -----------------------------
# Prep data (prefer gdf_ba_l if available)
# -----------------------------
ba_layer = gdf_ba_l if "gdf_ba_l" in globals() else gdf_ba
sites, basin, ba, st = _prep_conus(gdf_sites, gdf_basin, ba_layer, states48)

for c in [SITE_BASIN_KEY, SITE_BA_KEY, MW_COL, S1_COL_M3, S2_COL_M3, STRESS_COL_SITE]:
    if c not in sites.columns:
        raise KeyError(f"gdf_sites missing `{c}` needed for Fig 4A.")
sites[MW_COL] = pd.to_numeric(sites[MW_COL], errors="coerce").fillna(0.0)
sites[S1_COL_M3] = pd.to_numeric(sites[S1_COL_M3], errors="coerce").fillna(0.0)
sites[S2_COL_M3] = pd.to_numeric(sites[S2_COL_M3], errors="coerce").fillna(0.0)
sites[STRESS_COL_SITE] = pd.to_numeric(sites[STRESS_COL_SITE], errors="coerce").fillna(0.0)

# Baseline electricity (kWh/yr) from MW and utilization u_base
sites["E_fac_kWh_base"] = sites[MW_COL] * 8760.0 * float(u_base) * 1000.0

# -----------------------------
# Scope 1: efficiency benchmark savings (paper-consistent scaling)
# -----------------------------
scale_s1_eff = (float(u_eff) / float(u_base)) * ((float(wue_eff) / float(pue_eff)) / (float(wue_base) / float(pue_base)))
sites["s1_m3_eff"] = sites[S1_COL_M3] * scale_s1_eff
sites["s1_savings_m3"] = np.maximum(sites[S1_COL_M3] - sites["s1_m3_eff"], 0.0)

# Aggregate to basin
s1sav_b = (
    sites.groupby(SITE_BASIN_KEY)["s1_savings_m3"].sum()
    .rename("s1_savings_m3").reset_index()
    .rename(columns={SITE_BASIN_KEY: BASIN_KEY})
)
basin4 = basin.merge(s1sav_b, on=BASIN_KEY, how="left")
basin4["s1_savings_m3"] = basin4["s1_savings_m3"].fillna(0.0)

# Basin stress proxy (optional): mean site stress within basin
b_st = (
    sites.groupby(SITE_BASIN_KEY)[STRESS_COL_SITE].mean()
    .rename("stress_mean").reset_index()
    .rename(columns={SITE_BASIN_KEY: BASIN_KEY})
)
basin4 = basin4.merge(b_st, on=BASIN_KEY, how="left")
basin4["stress_mean"] = basin4["stress_mean"].fillna(0.0)

# Add simple readable basin label
bas_lbl = basin_to_state_label_fast(basin4, st, basin_id_col=BASIN_KEY, state_col="STUSPS")
basin4 = basin4.merge(bas_lbl, on=BASIN_KEY, how="left")
basin4["state_lbl"] = basin4["state_lbl"].fillna("")
basin4["label"] = basin4[BASIN_KEY].astype(str) + " (" + basin4["state_lbl"].astype(str) + ")"

top_basin = (
    basin4[[BASIN_KEY, "label", "s1_savings_m3", "stress_mean"]]
    .sort_values("s1_savings_m3", ascending=False)
    .head(TOP_N)
    .copy()
)
top_basin["s1_savings_GL"] = m3_to_GL(top_basin["s1_savings_m3"])

# -----------------------------
# Scope 2: low-water grid benchmark savings (P25 of observed BA intensity)
# -----------------------------
tmp = sites.copy()
tmp["s2_L_base"] = tmp[S2_COL_M3] * 1000.0  # m3 -> L

baE = tmp.groupby(SITE_BA_KEY)["E_fac_kWh_base"].sum().rename("E_kWh").reset_index().rename(columns={SITE_BA_KEY: BA_KEY})
baS2 = tmp.groupby(SITE_BA_KEY)["s2_L_base"].sum().rename("S2_L").reset_index().rename(columns={SITE_BA_KEY: BA_KEY})

# MW-weighted mean stress (better than unweighted)
tmp["_w"] = tmp[MW_COL].replace(0, np.nan)
tmp["_ws"] = tmp[STRESS_COL_SITE] * tmp[MW_COL]
baStressW = (
    tmp.groupby(SITE_BA_KEY)
       .apply(lambda g: float(g["_ws"].sum() / g[MW_COL].sum()) if g[MW_COL].sum() > 0 else np.nan)
       .rename("stress_mean_mw")
       .reset_index()
       .rename(columns={SITE_BA_KEY: BA_KEY})
)

ba4 = ba.merge(baE, on=BA_KEY, how="left").merge(baS2, on=BA_KEY, how="left").merge(baStressW, on=BA_KEY, how="left")
ba4[["E_kWh","S2_L"]] = ba4[["E_kWh","S2_L"]].fillna(0.0)
ba4["stress_mean_mw"] = ba4["stress_mean_mw"].fillna(0.0)
ba4 = ba4[ba4["E_kWh"] > 0].copy()

ba4["I_L_per_kWh"] = np.where(ba4["E_kWh"] > 0, ba4["S2_L"] / ba4["E_kWh"], np.nan)
I_target = float(ba4["I_L_per_kWh"].quantile(GRID_BENCH_Q))

ba4["I_target"] = np.minimum(ba4["I_L_per_kWh"], I_target)
ba4["S2_L_new"] = ba4["I_target"] * ba4["E_kWh"]
ba4["s2_savings_L"] = np.maximum(ba4["S2_L"] - ba4["S2_L_new"], 0.0)
ba4["s2_savings_m3"] = ba4["s2_savings_L"] / 1000.0
ba4["s2_savings_GL"] = m3_to_GL(ba4["s2_savings_m3"])

# BA display label
name_col = pick_ba_name_col(ba4)
if name_col is not None:
    ba4["label"] = ba4[name_col].astype(str)
else:
    ba4["label"] = ba4[BA_KEY].astype(str)

top_ba = (
    ba4[[BA_KEY, "label", "s2_savings_GL", "stress_mean_mw", "I_L_per_kWh"]]
    .sort_values("s2_savings_GL", ascending=False)
    .head(TOP_N)
    .copy()
)

# -----------------------------
# Plot
# -----------------------------
fig = plt.figure(figsize=(14.8, 7.2))
gs = fig.add_gridspec(nrows=1, ncols=2, left=0.07, right=0.985, top=0.90, bottom=0.18, wspace=0.30)
axA = fig.add_subplot(gs[0,0])
axB = fig.add_subplot(gs[0,1])

# Panel A: basins (Scope 1)
dfA = top_basin.sort_values("s1_savings_GL", ascending=True).reset_index(drop=True)
yA = np.arange(len(dfA))
axA.barh(yA, dfA["s1_savings_GL"], linewidth=0.0)

# Hatch overlay for high stress (mean basin stress proxy)
hiA = dfA["stress_mean"] >= STRESS_THR
axA.barh(yA[hiA.values], dfA.loc[hiA, "s1_savings_GL"], facecolor="none", edgecolor="#333333", hatch="////", linewidth=0.0)

axA.set_yticks(yA)
axA.set_yticklabels(dfA["label"], fontsize=9)
axA.set_xlabel("Potential Scope 1 reduction (GL/yr)")
axA.set_title("a. Where efficiency/cooling improvements could reduce Scope 1 most", loc="left")
axA.grid(axis="x", alpha=0.25)

# Panel B: BAs (Scope 2)
dfB = top_ba.sort_values("s2_savings_GL", ascending=True).reset_index(drop=True)
yB = np.arange(len(dfB))
axB.barh(yB, dfB["s2_savings_GL"], linewidth=0.0)

hiB = dfB["stress_mean_mw"] >= STRESS_THR
axB.barh(yB[hiB.values], dfB.loc[hiB, "s2_savings_GL"], facecolor="none", edgecolor="#333333", hatch="////", linewidth=0.0)

axB.set_yticks(yB)
axB.set_yticklabels(dfB["label"], fontsize=9)
axB.set_xlabel("Potential Scope 2 reduction (GL/yr)")
axB.set_title(f"b. Where lower-water electricity supply would reduce Scope 2 most\n(grid intensity benchmark → P{int(GRID_BENCH_Q*100)} = {I_target:.2f} L/kWh)", loc="left")
axB.grid(axis="x", alpha=0.25)

# Small right-side annotation: stress + intensity (kept compact, no overlap)
xpad = float(dfB["s2_savings_GL"].max()) * 0.02 if len(dfB) else 0.0
for i, r in dfB.iterrows():
    axB.text(xpad, i, f"stress={r['stress_mean_mw']:.2f}, I={r['I_L_per_kWh']:.2f}", va="center", fontsize=8)

# Shared legend below
fig.legend(
    handles=[
        Patch(facecolor="#4c78a8", edgecolor="none", label="Potential reduction (GL/yr)"),
        Patch(facecolor="none", edgecolor="#333333", hatch="////", label=f"High siting stress (mean ≥ {STRESS_THR:.0f})"),
    ],
    loc="lower center", bbox_to_anchor=(0.5, 0.06), ncol=2, frameon=True, fontsize=9
)

plt.savefig(OUT_PDF, bbox_inches="tight")
plt.show()
print("Saved:", OUT_PDF)

In [ ]:
# ==========================================
# FIGURE 4B: "POLICY ATLAS" (WHERE + WHAT LEVER)
# - Maps show where potential reductions concentrate
# - Side bars show top contributors (readable labels)
# - Uses MW-weighted stress where possible
# - PDF export only + show
# ==========================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

OUTDIR = "images"
os.makedirs(OUTDIR, exist_ok=True)
OUT_PDF = os.path.join(OUTDIR, "Fig4B_PolicyAtlas.pdf")

# -----------------------------
# KNOBS
# -----------------------------
TOP_N = 12
STRESS_THR = 3.0
GRID_BENCH_Q = 0.25

# -----------------------------
# Helpers
# -----------------------------
def m3_to_GL(x): return np.asarray(x, dtype=float) / 1e6

def pick_ba_name_col(df):
    for c in ["region_ful", "region", "region_BA"]:
        if c in df.columns:
            return c
    return None

def basin_to_state_label_fast(basin_gdf, states_gdf, basin_id_col, state_col="STUSPS"):
    if basin_gdf.crs != states_gdf.crs:
        states_gdf = states_gdf.to_crs(basin_gdf.crs)
    tmp = basin_gdf[[basin_id_col, "geometry"]].copy()
    tmp["_rp"] = tmp.geometry.representative_point()
    pts = tmp.set_geometry("_rp")
    j = pts.sjoin(states_gdf[[state_col, "geometry"]], how="left", predicate="within")
    out = j[[basin_id_col, state_col]].rename(columns={state_col: "state_lbl"})
    out["state_lbl"] = out["state_lbl"].fillna("")
    return out[[basin_id_col, "state_lbl"]]

# -----------------------------
# Prep
# -----------------------------
ba_layer = gdf_ba_l if "gdf_ba_l" in globals() else gdf_ba
sites, basin, ba, st = _prep_conus(gdf_sites, gdf_basin, ba_layer, states48)

for c in [SITE_BASIN_KEY, SITE_BA_KEY, MW_COL, S1_COL_M3, S2_COL_M3, STRESS_COL_SITE]:
    if c not in sites.columns:
        raise KeyError(f"gdf_sites missing `{c}` needed for Fig 4B.")
sites[MW_COL] = pd.to_numeric(sites[MW_COL], errors="coerce").fillna(0.0)
sites[S1_COL_M3] = pd.to_numeric(sites[S1_COL_M3], errors="coerce").fillna(0.0)
sites[S2_COL_M3] = pd.to_numeric(sites[S2_COL_M3], errors="coerce").fillna(0.0)
sites[STRESS_COL_SITE] = pd.to_numeric(sites[STRESS_COL_SITE], errors="coerce").fillna(0.0)

sites["E_fac_kWh_base"] = sites[MW_COL] * 8760.0 * float(u_base) * 1000.0

# -----------------------------
# Scope 1 savings -> basin
# -----------------------------
scale_s1_eff = (float(u_eff) / float(u_base)) * ((float(wue_eff) / float(pue_eff)) / (float(wue_base) / float(pue_base)))
sites["s1_m3_eff"] = sites[S1_COL_M3] * scale_s1_eff
sites["s1_savings_m3"] = np.maximum(sites[S1_COL_M3] - sites["s1_m3_eff"], 0.0)

b_sav = sites.groupby(SITE_BASIN_KEY)["s1_savings_m3"].sum().rename("s1_sav_m3").reset_index().rename(columns={SITE_BASIN_KEY: BASIN_KEY})
b_st  = sites.groupby(SITE_BASIN_KEY)[STRESS_COL_SITE].mean().rename("stress_mean").reset_index().rename(columns={SITE_BASIN_KEY: BASIN_KEY})

basin4 = basin.merge(b_sav, on=BASIN_KEY, how="left").merge(b_st, on=BASIN_KEY, how="left")
basin4[["s1_sav_m3","stress_mean"]] = basin4[["s1_sav_m3","stress_mean"]].fillna(0.0)
basin4["s1_sav_GL"] = m3_to_GL(basin4["s1_sav_m3"])

# readable basin labels (state abbrev)
bas_lbl = basin_to_state_label_fast(basin4, st, basin_id_col=BASIN_KEY)
basin4 = basin4.merge(bas_lbl, on=BASIN_KEY, how="left")
basin4["state_lbl"] = basin4["state_lbl"].fillna("")
basin4["label"] = basin4[BASIN_KEY].astype(str) + " (" + basin4["state_lbl"].astype(str) + ")"

top_basin = basin4.sort_values("s1_sav_GL", ascending=False).head(TOP_N).copy()

# -----------------------------
# Scope 2 savings -> BA
# -----------------------------
tmp = sites.copy()
tmp["s2_L_base"] = tmp[S2_COL_M3] * 1000.0

baE = tmp.groupby(SITE_BA_KEY)["E_fac_kWh_base"].sum().rename("E_kWh").reset_index().rename(columns={SITE_BA_KEY: BA_KEY})
baS2 = tmp.groupby(SITE_BA_KEY)["s2_L_base"].sum().rename("S2_L").reset_index().rename(columns={SITE_BA_KEY: BA_KEY})

# MW-weighted stress
tmp["_ws"] = tmp[STRESS_COL_SITE] * tmp[MW_COL]
baStressW = (
    tmp.groupby(SITE_BA_KEY)
       .apply(lambda g: float(g["_ws"].sum() / g[MW_COL].sum()) if g[MW_COL].sum() > 0 else np.nan)
       .rename("stress_mean_mw")
       .reset_index()
       .rename(columns={SITE_BA_KEY: BA_KEY})
)

ba4 = ba.merge(baE, on=BA_KEY, how="left").merge(baS2, on=BA_KEY, how="left").merge(baStressW, on=BA_KEY, how="left")
ba4[["E_kWh","S2_L"]] = ba4[["E_kWh","S2_L"]].fillna(0.0)
ba4["stress_mean_mw"] = ba4["stress_mean_mw"].fillna(0.0)
ba4 = ba4[ba4["E_kWh"] > 0].copy()

ba4["I_L_per_kWh"] = ba4["S2_L"] / ba4["E_kWh"]
I_target = float(ba4["I_L_per_kWh"].quantile(GRID_BENCH_Q))

ba4["I_target"] = np.minimum(ba4["I_L_per_kWh"], I_target)
ba4["S2_L_new"] = ba4["I_target"] * ba4["E_kWh"]
ba4["s2_sav_m3"] = np.maximum(ba4["S2_L"] - ba4["S2_L_new"], 0.0) / 1000.0
ba4["s2_sav_GL"] = m3_to_GL(ba4["s2_sav_m3"])

name_col = pick_ba_name_col(ba4)
ba4["label"] = ba4[name_col].astype(str) if name_col else ba4[BA_KEY].astype(str)

top_ba = ba4.sort_values("s2_sav_GL", ascending=False).head(TOP_N).copy()

# -----------------------------
# Plot: 2x2 atlas
# -----------------------------
minx, miny, maxx, maxy = st.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

fig = plt.figure(figsize=(15.5, 10.0))
gs = fig.add_gridspec(2, 2, left=0.05, right=0.985, top=0.93, bottom=0.12, wspace=0.18, hspace=0.28)

axA_map = fig.add_subplot(gs[0,0])
axA_bar = fig.add_subplot(gs[0,1])
axB_map = fig.add_subplot(gs[1,0])
axB_bar = fig.add_subplot(gs[1,1])

# ---- a1 map (Scope 1 savings)
basin4.plot(ax=axA_map, column="s1_sav_GL", legend=False, linewidth=0.2, edgecolor="white", zorder=1)
basin4[basin4["stress_mean"] >= STRESS_THR].plot(ax=axA_map, facecolor="none", edgecolor="#333333", hatch="////", linewidth=0.0, zorder=3)
st.boundary.plot(ax=axA_map, color="#111111", linewidth=0.6, zorder=10)
axA_map.set_title("a1. Scope 1 leverage: where efficiency/cooling could reduce water most\n(hatch = mean siting stress ≥ 3)", loc="left")
axA_map.set_axis_off()
axA_map.set_xlim(minx-padx, maxx+padx); axA_map.set_ylim(miny-pady, maxy+pady)
axA_map.set_aspect("equal")

# ---- a2 bar (top basins)
df = top_basin.sort_values("s1_sav_GL", ascending=True).reset_index(drop=True)
y = np.arange(len(df))
axA_bar.barh(y, df["s1_sav_GL"], linewidth=0.0)
hi = df["stress_mean"] >= STRESS_THR
axA_bar.barh(y[hi.values], df.loc[hi, "s1_sav_GL"], facecolor="none", edgecolor="#333333", hatch="////", linewidth=0.0)
axA_bar.set_yticks(y)
axA_bar.set_yticklabels(df["label"], fontsize=9)
axA_bar.set_xlabel("Potential reduction (GL/yr)")
axA_bar.set_title("a2. Top basins by potential Scope 1 reduction", loc="left")
axA_bar.grid(axis="x", alpha=0.25)

# ---- b1 map (Scope 2 savings)
ba4.plot(ax=axB_map, column="s2_sav_GL", legend=False, linewidth=0.25, edgecolor="white", zorder=1)
ba4[ba4["stress_mean_mw"] >= STRESS_THR].plot(ax=axB_map, facecolor="none", edgecolor="#333333", hatch="////", linewidth=0.0, zorder=3)
st.boundary.plot(ax=axB_map, color="#111111", linewidth=0.6, zorder=10)
axB_map.set_title(
    f"b1. Scope 2 leverage: where lower-water electricity supply could reduce water most\n"
    f"(benchmark = P{int(GRID_BENCH_Q*100)} intensity = {I_target:.2f} L/kWh; hatch = stress ≥ 3)",
    loc="left"
)
axB_map.set_axis_off()
axB_map.set_xlim(minx-padx, maxx+padx); axB_map.set_ylim(miny-pady, maxy+pady)
axB_map.set_aspect("equal")

# ---- b2 bar (top BAs)
df = top_ba.sort_values("s2_sav_GL", ascending=True).reset_index(drop=True)
y = np.arange(len(df))
axB_bar.barh(y, df["s2_sav_GL"], linewidth=0.0)
hi = df["stress_mean_mw"] >= STRESS_THR
axB_bar.barh(y[hi.values], df.loc[hi, "s2_sav_GL"], facecolor="none", edgecolor="#333333", hatch="////", linewidth=0.0)
axB_bar.set_yticks(y)
axB_bar.set_yticklabels(df["label"], fontsize=9)
axB_bar.set_xlabel("Potential reduction (GL/yr)")
axB_bar.set_title("b2. Top balancing authorities by potential Scope 2 reduction", loc="left")
axB_bar.grid(axis="x", alpha=0.25)

# shared legend
fig.legend(
    handles=[
        Patch(facecolor="none", edgecolor="#333333", hatch="////", label=f"High siting stress (≥ {STRESS_THR:.0f})"),
    ],
    loc="lower center", bbox_to_anchor=(0.5, 0.06), ncol=1, frameon=True, fontsize=9
)

plt.savefig(OUT_PDF, bbox_inches="tight")
plt.show()
print("Saved:", OUT_PDF)

In [ ]:
# ==========================================
# FIGURE 4B: "POLICY ATLAS" (RESTYLED)
# - Light background (low alpha) so borders stay readable
# - Strong highlight for top leverage regions
# - No dark purple; use lighter sequential cmap
# - Optional winsorization/log to reveal structure
# ==========================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import Normalize, LogNorm
from matplotlib.cm import ScalarMappable

OUTDIR = "images"
os.makedirs(OUTDIR, exist_ok=True)
OUT_PDF = os.path.join(OUTDIR, "Fig4B_PolicyAtlas.pdf")

# -----------------------------
# KNOBS (edit here)
# -----------------------------
TOP_N = 12
STRESS_THR = 3.0
GRID_BENCH_Q = 0.25

# Visual emphasis knobs
BG_ALPHA = 0.18            # background basins/BAs fill transparency
BG_EDGE_LW = 0.18          # thin internal borders
TOP_ALPHA = 0.95           # highlighted regions opacity
TOP_EDGE_LW = 1.25         # highlight border thickness
STATE_LW = 0.6

# Color choices (avoid dark purple)
# Good options: "YlGnBu", "PuBuGn", "GnBu", "Blues"
CMAP_S1 = "YlGnBu"
CMAP_S2 = "YlGnBu"

# Scale handling to avoid "flat" map
USE_LOG = False            # True if extreme skew makes map flat
WINSOR_PCT = 0.98          # cap vmax at this quantile (helps show variation)

# Hatch styling
HATCH = "////"
HATCH_EDGE = "#4d4d4d"

# -----------------------------
# Helpers
# -----------------------------
def m3_to_GL(x): return np.asarray(x, dtype=float) / 1e6

def pick_ba_name_col(df):
    for c in ["region_ful", "region", "region_BA"]:
        if c in df.columns:
            return c
    return None

def basin_to_state_label_fast(basin_gdf, states_gdf, basin_id_col, state_col="STUSPS"):
    if basin_gdf.crs != states_gdf.crs:
        states_gdf = states_gdf.to_crs(basin_gdf.crs)
    tmp = basin_gdf[[basin_id_col, "geometry"]].copy()
    tmp["_rp"] = tmp.geometry.representative_point()
    pts = tmp.set_geometry("_rp")
    j = pts.sjoin(states_gdf[[state_col, "geometry"]], how="left", predicate="within")
    out = j[[basin_id_col, state_col]].rename(columns={state_col: "state_lbl"})
    out["state_lbl"] = out["state_lbl"].fillna("")
    return out[[basin_id_col, "state_lbl"]]

def cap_norm(series, pct=0.98, use_log=False):
    s = pd.Series(series).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    s_pos = s[s > 0]
    if len(s_pos) == 0:
        vmax = 1.0
        return Normalize(0, vmax), vmax
    vmax = float(s_pos.quantile(pct))
    vmax = max(vmax, float(s_pos.max()), 1e-12)
    if use_log:
        vmin = float(s_pos.min())
        vmin = max(vmin, vmax * 1e-6)
        return LogNorm(vmin=vmin, vmax=vmax), vmax
    return Normalize(vmin=0.0, vmax=vmax), vmax

# -----------------------------
# Prep (same logic; prefer gdf_ba_l if available)
# -----------------------------
ba_layer = gdf_ba_l if "gdf_ba_l" in globals() else gdf_ba
sites, basin, ba, st = _prep_conus(gdf_sites, gdf_basin, ba_layer, states48)

for c in [SITE_BASIN_KEY, SITE_BA_KEY, MW_COL, S1_COL_M3, S2_COL_M3, STRESS_COL_SITE]:
    if c not in sites.columns:
        raise KeyError(f"gdf_sites missing `{c}` needed for Fig 4B.")
sites[MW_COL] = pd.to_numeric(sites[MW_COL], errors="coerce").fillna(0.0)
sites[S1_COL_M3] = pd.to_numeric(sites[S1_COL_M3], errors="coerce").fillna(0.0)
sites[S2_COL_M3] = pd.to_numeric(sites[S2_COL_M3], errors="coerce").fillna(0.0)
sites[STRESS_COL_SITE] = pd.to_numeric(sites[STRESS_COL_SITE], errors="coerce").fillna(0.0)

sites["E_fac_kWh_base"] = sites[MW_COL] * 8760.0 * float(u_base) * 1000.0

# -----------------------------
# Scope 1 savings -> basin
# -----------------------------
scale_s1_eff = (float(u_eff) / float(u_base)) * ((float(wue_eff) / float(pue_eff)) / (float(wue_base) / float(pue_base)))
sites["s1_m3_eff"] = sites[S1_COL_M3] * scale_s1_eff
sites["s1_savings_m3"] = np.maximum(sites[S1_COL_M3] - sites["s1_m3_eff"], 0.0)

b_sav = sites.groupby(SITE_BASIN_KEY)["s1_savings_m3"].sum().rename("s1_sav_m3").reset_index().rename(columns={SITE_BASIN_KEY: BASIN_KEY})
b_st  = sites.groupby(SITE_BASIN_KEY)[STRESS_COL_SITE].mean().rename("stress_mean").reset_index().rename(columns={SITE_BASIN_KEY: BASIN_KEY})

basin4 = basin.merge(b_sav, on=BASIN_KEY, how="left").merge(b_st, on=BASIN_KEY, how="left")
basin4[["s1_sav_m3","stress_mean"]] = basin4[["s1_sav_m3","stress_mean"]].fillna(0.0)
basin4["s1_sav_GL"] = m3_to_GL(basin4["s1_sav_m3"])

bas_lbl = basin_to_state_label_fast(basin4, st, basin_id_col=BASIN_KEY)
basin4 = basin4.merge(bas_lbl, on=BASIN_KEY, how="left")
basin4["label"] = basin4[BASIN_KEY].astype(str) + " (" + basin4["state_lbl"].fillna("").astype(str) + ")"

top_basin = basin4.sort_values("s1_sav_GL", ascending=False).head(TOP_N).copy()
top_basin_ids = set(top_basin[BASIN_KEY].astype(str))

# -----------------------------
# Scope 2 savings -> BA
# -----------------------------
tmp = sites.copy()
tmp["s2_L_base"] = tmp[S2_COL_M3] * 1000.0
baE = tmp.groupby(SITE_BA_KEY)["E_fac_kWh_base"].sum().rename("E_kWh").reset_index().rename(columns={SITE_BA_KEY: BA_KEY})
baS2 = tmp.groupby(SITE_BA_KEY)["s2_L_base"].sum().rename("S2_L").reset_index().rename(columns={SITE_BA_KEY: BA_KEY})

tmp["_ws"] = tmp[STRESS_COL_SITE] * tmp[MW_COL]
baStressW = (
    tmp.groupby(SITE_BA_KEY)
       .apply(lambda g: float(g["_ws"].sum() / g[MW_COL].sum()) if g[MW_COL].sum() > 0 else np.nan)
       .rename("stress_mean_mw")
       .reset_index()
       .rename(columns={SITE_BA_KEY: BA_KEY})
)

ba4 = ba.merge(baE, on=BA_KEY, how="left").merge(baS2, on=BA_KEY, how="left").merge(baStressW, on=BA_KEY, how="left")
ba4[["E_kWh","S2_L"]] = ba4[["E_kWh","S2_L"]].fillna(0.0)
ba4["stress_mean_mw"] = ba4["stress_mean_mw"].fillna(0.0)
ba4 = ba4[ba4["E_kWh"] > 0].copy()

ba4["I_L_per_kWh"] = ba4["S2_L"] / ba4["E_kWh"]
I_target = float(ba4["I_L_per_kWh"].quantile(GRID_BENCH_Q))

ba4["I_target"] = np.minimum(ba4["I_L_per_kWh"], I_target)
ba4["S2_L_new"] = ba4["I_target"] * ba4["E_kWh"]
ba4["s2_sav_m3"] = np.maximum(ba4["S2_L"] - ba4["S2_L_new"], 0.0) / 1000.0
ba4["s2_sav_GL"] = m3_to_GL(ba4["s2_sav_m3"])

name_col = pick_ba_name_col(ba4)
ba4["label"] = ba4[name_col].astype(str) if name_col else ba4[BA_KEY].astype(str)

top_ba = ba4.sort_values("s2_sav_GL", ascending=False).head(TOP_N).copy()
top_ba_ids = set(top_ba[BA_KEY].astype(str))

# -----------------------------
# Norms for colored highlights
# -----------------------------
norm_s1, vmax_s1 = cap_norm(basin4["s1_sav_GL"], pct=WINSOR_PCT, use_log=USE_LOG)
norm_s2, vmax_s2 = cap_norm(ba4["s2_sav_GL"], pct=WINSOR_PCT, use_log=USE_LOG)

# -----------------------------
# Plot: 2x2 atlas
# -----------------------------
minx, miny, maxx, maxy = st.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

fig = plt.figure(figsize=(15.5, 10.0))
gs = fig.add_gridspec(2, 2, left=0.05, right=0.985, top=0.93, bottom=0.12, wspace=0.18, hspace=0.28)

axA_map = fig.add_subplot(gs[0,0])
axA_bar = fig.add_subplot(gs[0,1])
axB_map = fig.add_subplot(gs[1,0])
axB_bar = fig.add_subplot(gs[1,1])

# ---- a1 map (Scope 1 savings): background faint + top highlighted
basin4.plot(ax=axA_map, color="#f3f3f3", edgecolor="white", linewidth=BG_EDGE_LW, alpha=1.0, zorder=1)

topmask = basin4[BASIN_KEY].astype(str).isin(top_basin_ids)
basin4.loc[topmask].plot(
    ax=axA_map,
    column="s1_sav_GL",
    cmap=CMAP_S1,
    norm=norm_s1,
    edgecolor="#111111",
    linewidth=TOP_EDGE_LW,
    alpha=TOP_ALPHA,
    zorder=3
)

# stress hatch overlay (over all basins)
basin4[basin4["stress_mean"] >= STRESS_THR].plot(
    ax=axA_map, facecolor="none", edgecolor=HATCH_EDGE, hatch=HATCH, linewidth=0.0, zorder=4
)

st.boundary.plot(ax=axA_map, color="#111111", linewidth=STATE_LW, zorder=10)
axA_map.set_title("a1. Scope 1 leverage: where efficiency/cooling could reduce water most\n(hatch = mean siting stress ≥ 3)", loc="left")
axA_map.set_axis_off()
axA_map.set_xlim(minx-padx, maxx+padx); axA_map.set_ylim(miny-pady, maxy+pady)
axA_map.set_aspect("equal")

# add a clean colorbar for the highlighted values only
sm1 = ScalarMappable(norm=norm_s1, cmap=CMAP_S1); sm1.set_array([])
cb1 = fig.colorbar(sm1, ax=axA_map, fraction=0.030, pad=0.01)
cb1.set_label("Potential Scope 1 reduction (GL/yr)", fontsize=9)
cb1.ax.tick_params(labelsize=8)

# ---- a2 bar (top basins)
df = top_basin.sort_values("s1_sav_GL", ascending=True).reset_index(drop=True)
y = np.arange(len(df))
axA_bar.barh(y, df["s1_sav_GL"], linewidth=0.0)
hi = df["stress_mean"] >= STRESS_THR
axA_bar.barh(y[hi.values], df.loc[hi, "s1_sav_GL"], facecolor="none", edgecolor="#333333", hatch="////", linewidth=0.0)
axA_bar.set_yticks(y)
axA_bar.set_yticklabels(df["label"], fontsize=9)
axA_bar.set_xlabel("Potential reduction (GL/yr)")
axA_bar.set_title("a2. Top basins by potential Scope 1 reduction", loc="left")
axA_bar.grid(axis="x", alpha=0.25)

# ---- b1 map (Scope 2 savings): background faint + top highlighted
ba4.plot(ax=axB_map, color="#f3f3f3", edgecolor="white", linewidth=BG_EDGE_LW, alpha=1.0, zorder=1)

topmask = ba4[BA_KEY].astype(str).isin(top_ba_ids)
ba4.loc[topmask].plot(
    ax=axB_map,
    column="s2_sav_GL",
    cmap=CMAP_S2,
    norm=norm_s2,
    edgecolor="#111111",
    linewidth=TOP_EDGE_LW,
    alpha=TOP_ALPHA,
    zorder=3
)

ba4[ba4["stress_mean_mw"] >= STRESS_THR].plot(
    ax=axB_map, facecolor="none", edgecolor=HATCH_EDGE, hatch=HATCH, linewidth=0.0, zorder=4
)

st.boundary.plot(ax=axB_map, color="#111111", linewidth=STATE_LW, zorder=10)
axB_map.set_title(
    f"b1. Scope 2 leverage: where lower-water electricity supply could reduce water most\n"
    f"(benchmark = P{int(GRID_BENCH_Q*100)} intensity = {I_target:.2f} L/kWh; hatch = stress ≥ 3)",
    loc="left"
)
axB_map.set_axis_off()
axB_map.set_xlim(minx-padx, maxx+padx); axB_map.set_ylim(miny-pady, maxy+pady)
axB_map.set_aspect("equal")

sm2 = ScalarMappable(norm=norm_s2, cmap=CMAP_S2); sm2.set_array([])
cb2 = fig.colorbar(sm2, ax=axB_map, fraction=0.030, pad=0.01)
cb2.set_label("Potential Scope 2 reduction (GL/yr)", fontsize=9)
cb2.ax.tick_params(labelsize=8)

# ---- b2 bar (top BAs)
df = top_ba.sort_values("s2_sav_GL", ascending=True).reset_index(drop=True)
y = np.arange(len(df))
axB_bar.barh(y, df["s2_sav_GL"], linewidth=0.0)
hi = df["stress_mean_mw"] >= STRESS_THR
axB_bar.barh(y[hi.values], df.loc[hi, "s2_sav_GL"], facecolor="none", edgecolor="#333333", hatch="////", linewidth=0.0)
axB_bar.set_yticks(y)
axB_bar.set_yticklabels(df["label"], fontsize=9)
axB_bar.set_xlabel("Potential reduction (GL/yr)")
axB_bar.set_title("b2. Top balancing authorities by potential Scope 2 reduction", loc="left")
axB_bar.grid(axis="x", alpha=0.25)

# shared legend (simple, below)
fig.legend(
    handles=[
        Patch(facecolor="#f3f3f3", edgecolor="white", label="All regions (background)"),
        Patch(facecolor="none", edgecolor="#111111", label=f"Top {TOP_N} by leverage (outlined)"),
        Patch(facecolor="none", edgecolor=HATCH_EDGE, hatch="////", label=f"High siting stress (≥ {STRESS_THR:.0f})"),
    ],
    loc="lower center", bbox_to_anchor=(0.5, 0.06), ncol=3, frameon=True, fontsize=9
)

plt.savefig(OUT_PDF, bbox_inches="tight")
plt.show()
print("Saved:", OUT_PDF)

In [ ]:
# ==========================================
# FIGURE 4B: "POLICY ATLAS" (RESTYLED — RUN-ALL)
# - Light background so borders stay readable
# - Strong highlight for top leverage regions
# - No dark purple; uses light sequential cmap
# - Maps larger than bars (Nature-friendly)
# - Fixes overlapping labels on BA map by (a) thicker borders and (b) optional label shortening
# - Exports PDF only + shows in notebook
# ==========================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import Normalize, LogNorm
from matplotlib.cm import ScalarMappable

# -----------------------------
# OUTPUT
# -----------------------------
OUTDIR = "images"
os.makedirs(OUTDIR, exist_ok=True)
OUT_PDF = os.path.join(OUTDIR, "Fig4B_PolicyAtlas.pdf")

# -----------------------------
# KNOBS (edit here)
# -----------------------------
TOP_N = 12
STRESS_THR = 3.0
GRID_BENCH_Q = 0.25  # BA intensity benchmark quantile (e.g., P25)

# Visual emphasis knobs
BG_ALPHA   = 0.22     # background fill transparency
BG_EDGE_LW = 0.22     # thin internal borders
TOP_ALPHA  = 0.95     # highlighted regions opacity
TOP_EDGE_LW = 1.35    # highlight border thickness
STATE_LW   = 0.60
BA_BORDER_LW = 0.95   # make BA borders more visible (you asked)

# Color choices (avoid dark purple)
# Recommended for Nature: "YlGnBu" or "PuBuGn" (both read well in print)
CMAP_S1 = "YlGnBu"
CMAP_S2 = "YlGnBu"

# Scale handling to avoid "flat" map
USE_LOG = False        # True if extreme skew makes map flat
WINSOR_PCT = 0.98      # cap vmax at this quantile (helps show variation)

# Hatch styling
HATCH = "////"
HATCH_EDGE = "#4d4d4d"

# Bars typography
BAR_Y_FONTSIZE = 9

# If BA names are too long, shorten them a bit (keeps them from colliding with map margins)
SHORTEN_BA_NAMES = True
MAX_BA_CHARS = 26

# -----------------------------
# Helpers
# -----------------------------
def m3_to_GL(x):
    return np.asarray(x, dtype=float) / 1e6

def pick_ba_name_col(df):
    # choose the most human-readable BA name if available
    for c in ["region_ful", "region", "region_BA", "region_BA_"]:
        if c in df.columns:
            return c
    return None

def shorten(s, n=26):
    s = "" if s is None else str(s)
    if len(s) <= n:
        return s
    return s[: max(0, n - 1)] + "…"

def basin_to_state_label_fast(basin_gdf, states_gdf, basin_id_col, state_col="STUSPS"):
    """Representative-point join to states; returns a single state abbrev label per basin."""
    if basin_gdf.crs != states_gdf.crs:
        states_gdf = states_gdf.to_crs(basin_gdf.crs)

    tmp = basin_gdf[[basin_id_col, "geometry"]].copy()
    tmp["_rp"] = tmp.geometry.representative_point()
    pts = tmp.set_geometry("_rp")

    j = pts.sjoin(states_gdf[[state_col, "geometry"]], how="left", predicate="within")
    out = j[[basin_id_col, state_col]].rename(columns={state_col: "state_lbl"})
    out["state_lbl"] = out["state_lbl"].fillna("")
    return out[[basin_id_col, "state_lbl"]]

def cap_norm(series, pct=0.98, use_log=False):
    """Winsorize vmax; optionally log-scale (positive-only)."""
    s = pd.Series(series).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    s_pos = s[s > 0]

    if len(s_pos) == 0:
        vmax = 1.0
        return Normalize(0.0, vmax), vmax

    vmax = float(s_pos.quantile(pct))
    vmax = max(vmax, 1e-12)

    if use_log:
        vmin = float(s_pos.min())
        vmin = max(vmin, vmax * 1e-6)
        return LogNorm(vmin=vmin, vmax=vmax), vmax

    return Normalize(vmin=0.0, vmax=vmax), vmax

# -----------------------------
# Prep (prefer gdf_ba_l if available)
# -----------------------------
ba_layer = gdf_ba_l if "gdf_ba_l" in globals() else gdf_ba
sites, basin, ba, st = _prep_conus(gdf_sites, gdf_basin, ba_layer, states48)

# Required columns
need_sites = [SITE_BASIN_KEY, SITE_BA_KEY, MW_COL, S1_COL_M3, S2_COL_M3, STRESS_COL_SITE]
missing = [c for c in need_sites if c not in sites.columns]
if missing:
    raise KeyError(f"gdf_sites missing required columns for Fig4B: {missing}")

# Numeric cleanup
sites[MW_COL] = pd.to_numeric(sites[MW_COL], errors="coerce").fillna(0.0)
sites[S1_COL_M3] = pd.to_numeric(sites[S1_COL_M3], errors="coerce").fillna(0.0)
sites[S2_COL_M3] = pd.to_numeric(sites[S2_COL_M3], errors="coerce").fillna(0.0)
sites[STRESS_COL_SITE] = pd.to_numeric(sites[STRESS_COL_SITE], errors="coerce").fillna(0.0)

# Baseline electricity (kWh/yr): MW * 8760 h = MWh; *1000 = kWh
sites["E_fac_kWh_base"] = sites[MW_COL] * 8760.0 * float(u_base) * 1000.0

# -----------------------------
# Scope 1 savings -> basin
# -----------------------------
scale_s1_eff = (float(u_eff) / float(u_base)) * ((float(wue_eff) / float(pue_eff)) / (float(wue_base) / float(pue_base)))
sites["s1_m3_eff"] = sites[S1_COL_M3] * scale_s1_eff
sites["s1_sav_m3"] = np.maximum(sites[S1_COL_M3] - sites["s1_m3_eff"], 0.0)

b_sav = (
    sites.groupby(SITE_BASIN_KEY)["s1_sav_m3"].sum()
    .rename("s1_sav_m3").reset_index()
    .rename(columns={SITE_BASIN_KEY: BASIN_KEY})
)
b_st = (
    sites.groupby(SITE_BASIN_KEY)[STRESS_COL_SITE].mean()
    .rename("stress_mean").reset_index()
    .rename(columns={SITE_BASIN_KEY: BASIN_KEY})
)

basin4 = basin.merge(b_sav, on=BASIN_KEY, how="left").merge(b_st, on=BASIN_KEY, how="left")
basin4[["s1_sav_m3", "stress_mean"]] = basin4[["s1_sav_m3", "stress_mean"]].fillna(0.0)
basin4["s1_sav_GL"] = m3_to_GL(basin4["s1_sav_m3"])

# Add state abbrev label (for bar y-ticks)
bas_lbl = basin_to_state_label_fast(basin4, st, basin_id_col=BASIN_KEY)
basin4 = basin4.merge(bas_lbl, on=BASIN_KEY, how="left")
basin4["state_lbl"] = basin4["state_lbl"].fillna("")
basin4["label"] = basin4[BASIN_KEY].astype(str) + " (" + basin4["state_lbl"].astype(str) + ")"

top_basin = basin4.sort_values("s1_sav_GL", ascending=False).head(TOP_N).copy()
top_basin_ids = set(top_basin[BASIN_KEY].astype(str))

# -----------------------------
# Scope 2 savings -> BA
# -----------------------------
tmp = sites.copy()
tmp["s2_L_base"] = tmp[S2_COL_M3] * 1000.0  # m3 -> L

baE = (
    tmp.groupby(SITE_BA_KEY)["E_fac_kWh_base"].sum()
    .rename("E_kWh").reset_index()
    .rename(columns={SITE_BA_KEY: BA_KEY})
)
baS2 = (
    tmp.groupby(SITE_BA_KEY)["s2_L_base"].sum()
    .rename("S2_L").reset_index()
    .rename(columns={SITE_BA_KEY: BA_KEY})
)

# MW-weighted mean stress (more defensible for siting exposure)
tmp["_ws"] = tmp[STRESS_COL_SITE] * tmp[MW_COL]
baStressW = (
    tmp.groupby(SITE_BA_KEY)
       .apply(lambda g: float(g["_ws"].sum() / g[MW_COL].sum()) if g[MW_COL].sum() > 0 else np.nan)
       .rename("stress_mean_mw")
       .reset_index()
       .rename(columns={SITE_BA_KEY: BA_KEY})
)

ba4 = ba.merge(baE, on=BA_KEY, how="left").merge(baS2, on=BA_KEY, how="left").merge(baStressW, on=BA_KEY, how="left")
ba4[["E_kWh", "S2_L"]] = ba4[["E_kWh", "S2_L"]].fillna(0.0)
ba4["stress_mean_mw"] = ba4["stress_mean_mw"].fillna(0.0)

# Keep BAs with electricity > 0
ba4 = ba4[ba4["E_kWh"] > 0].copy()

# Intensity + benchmark target
ba4["I_L_per_kWh"] = ba4["S2_L"] / ba4["E_kWh"]
I_target = float(ba4["I_L_per_kWh"].quantile(GRID_BENCH_Q))

ba4["I_target"] = np.minimum(ba4["I_L_per_kWh"], I_target)
ba4["S2_L_new"] = ba4["I_target"] * ba4["E_kWh"]
ba4["s2_sav_GL"] = m3_to_GL(np.maximum(ba4["S2_L"] - ba4["S2_L_new"], 0.0) / 1000.0)

# BA labels
# --- Replace the BA label block in Fig4B with this (prints BA initials/codes) ---

# BA initials / codes: prefer region_BA / region_BA_ / region_B_1 / BA_region, else fall back to BA_KEY
def pick_ba_code_col(df):
    for c in ["region_BA", "region_BA_", "region_B_1", "BA_region"]:
        if c in df.columns:
            return c
    return None

ba_code_col = pick_ba_code_col(ba4)

# Use initials/codes ONLY
if ba_code_col:
    ba4["label"] = ba4[ba_code_col].astype(str)
else:
    ba4["label"] = ba4[BA_KEY].astype(str)

# (Optional) also apply to the TOP table used for b2 bars
# If you create `top_ba` AFTER assigning ba4["label"], you're done.
# If not, re-create it:
top_ba = ba4.sort_values("s2_sav_GL", ascending=False).head(TOP_N).copy()

print("BA code column used:", ba_code_col if ba_code_col else f"(fallback) {BA_KEY}")

# -----------------------------
# Norms for colored highlights
# -----------------------------
norm_s1, _ = cap_norm(basin4["s1_sav_GL"], pct=WINSOR_PCT, use_log=USE_LOG)
norm_s2, _ = cap_norm(ba4["s2_sav_GL"], pct=WINSOR_PCT, use_log=USE_LOG)

# -----------------------------
# Plot: 2x2 atlas (maps larger than bars)
# -----------------------------
minx, miny, maxx, maxy = st.total_bounds
padx = 0.02 * (maxx - minx)
pady = 0.02 * (maxy - miny)

fig = plt.figure(figsize=(15.8, 10.2))

# Make maps wider than bars: col ratios [1.25, 0.95]
gs = fig.add_gridspec(
    2, 2,
    width_ratios=[1.25, 0.95],
    left=0.05, right=0.985, top=0.93, bottom=0.13,
    wspace=0.18, hspace=0.28
)

axA_map = fig.add_subplot(gs[0, 0])
axA_bar = fig.add_subplot(gs[0, 1])
axB_map = fig.add_subplot(gs[1, 0])
axB_bar = fig.add_subplot(gs[1, 1])

# ---- a1 map (Scope 1 savings)
basin4.plot(
    ax=axA_map,
    color="#f3f3f3",
    edgecolor="white",
    linewidth=BG_EDGE_LW,
    alpha=1.0,
    zorder=1
)

topmask = basin4[BASIN_KEY].astype(str).isin(top_basin_ids)
basin4.loc[topmask].plot(
    ax=axA_map,
    column="s1_sav_GL",
    cmap=CMAP_S1,
    norm=norm_s1,
    edgecolor="#111111",
    linewidth=TOP_EDGE_LW,
    alpha=TOP_ALPHA,
    zorder=3
)

# stress hatch overlay
basin4.loc[basin4["stress_mean"] >= STRESS_THR].plot(
    ax=axA_map,
    facecolor="none",
    edgecolor=HATCH_EDGE,
    hatch=HATCH,
    linewidth=0.0,
    zorder=4
)

st.boundary.plot(ax=axA_map, color="#111111", linewidth=STATE_LW, zorder=10)

axA_map.set_title(
    "a1. Scope 1 leverage: where efficiency/cooling could reduce water most\n"
    f"(highlighted = top {TOP_N}; hatch = mean siting stress ≥ {STRESS_THR:.0f})",
    loc="left"
)
axA_map.set_axis_off()
axA_map.set_xlim(minx - padx, maxx + padx)
axA_map.set_ylim(miny - pady, maxy + pady)
axA_map.set_aspect("equal")

sm1 = ScalarMappable(norm=norm_s1, cmap=CMAP_S1); sm1.set_array([])
cb1 = fig.colorbar(sm1, ax=axA_map, fraction=0.030, pad=0.01)
cb1.set_label("Potential Scope 1 reduction (GL/yr)", fontsize=9)
cb1.ax.tick_params(labelsize=8)

# ---- a2 bar (top basins)
dfA = top_basin.sort_values("s1_sav_GL", ascending=True).reset_index(drop=True)
yA = np.arange(len(dfA))
axA_bar.barh(yA, dfA["s1_sav_GL"].values, linewidth=0.0)

# hatch the stressed ones
hiA = dfA["stress_mean"] >= STRESS_THR
axA_bar.barh(
    yA[hiA.values],
    dfA.loc[hiA, "s1_sav_GL"].values,
    facecolor="none",
    edgecolor="#333333",
    hatch=HATCH,
    linewidth=0.0
)

axA_bar.set_yticks(yA)
axA_bar.set_yticklabels(dfA["label"].tolist(), fontsize=BAR_Y_FONTSIZE)
axA_bar.set_xlabel("Potential reduction (GL/yr)")
axA_bar.set_title("a2. Top basins by potential Scope 1 reduction", loc="left")
axA_bar.grid(axis="x", alpha=0.25)

# ---- b1 map (Scope 2 savings)
ba4.plot(
    ax=axB_map,
    color="#f3f3f3",
    edgecolor="white",
    linewidth=BG_EDGE_LW,
    alpha=1.0,
    zorder=1
)

topmask = ba4[BA_KEY].astype(str).isin(top_ba_ids)
ba4.loc[topmask].plot(
    ax=axB_map,
    column="s2_sav_GL",
    cmap=CMAP_S2,
    norm=norm_s2,
    edgecolor="#111111",
    linewidth=TOP_EDGE_LW,
    alpha=TOP_ALPHA,
    zorder=3
)

# show *all* BA borders slightly thicker for readability
ba4.boundary.plot(ax=axB_map, color="#111111", linewidth=BA_BORDER_LW, zorder=2)

# hatch high stress
ba4.loc[ba4["stress_mean_mw"] >= STRESS_THR].plot(
    ax=axB_map,
    facecolor="none",
    edgecolor=HATCH_EDGE,
    hatch=HATCH,
    linewidth=0.0,
    zorder=4
)

st.boundary.plot(ax=axB_map, color="#111111", linewidth=STATE_LW, zorder=10)

axB_map.set_title(
    "b1. Scope 2 leverage: where lower-water electricity supply could reduce water most\n"
    f"(benchmark = P{int(GRID_BENCH_Q*100)} intensity = {I_target:.2f} L/kWh; "
    f"highlighted = top {TOP_N}; hatch = stress ≥ {STRESS_THR:.0f})",
    loc="left"
)
axB_map.set_axis_off()
axB_map.set_xlim(minx - padx, maxx + padx)
axB_map.set_ylim(miny - pady, maxy + pady)
axB_map.set_aspect("equal")

sm2 = ScalarMappable(norm=norm_s2, cmap=CMAP_S2); sm2.set_array([])
cb2 = fig.colorbar(sm2, ax=axB_map, fraction=0.030, pad=0.01)
cb2.set_label("Potential Scope 2 reduction (GL/yr)", fontsize=9)
cb2.ax.tick_params(labelsize=8)

# ---- b2 bar (top BAs)
dfB = top_ba.sort_values("s2_sav_GL", ascending=True).reset_index(drop=True)
yB = np.arange(len(dfB))
axB_bar.barh(yB, dfB["s2_sav_GL"].values, linewidth=0.0)

hiB = dfB["stress_mean_mw"] >= STRESS_THR
axB_bar.barh(
    yB[hiB.values],
    dfB.loc[hiB, "s2_sav_GL"].values,
    facecolor="none",
    edgecolor="#333333",
    hatch=HATCH,
    linewidth=0.0
)

axB_bar.set_yticks(yB)
axB_bar.set_yticklabels(dfB["label"].tolist(), fontsize=BAR_Y_FONTSIZE)
axB_bar.set_xlabel("Potential reduction (GL/yr)")
axB_bar.set_title("b2. Top balancing authorities by potential Scope 2 reduction", loc="left")
axB_bar.grid(axis="x", alpha=0.25)

# ---- shared legend (boxed, below; no overlap)
fig.legend(
    handles=[
        Patch(facecolor="#f3f3f3", edgecolor="white", label="All regions (background)"),
        Patch(facecolor="none", edgecolor="#111111", label=f"Top {TOP_N} by leverage (highlighted)"),
        Patch(facecolor="none", edgecolor=HATCH_EDGE, hatch=HATCH, label=f"High siting stress (≥ {STRESS_THR:.0f})"),
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, 0.06),
    ncol=3,
    frameon=True,
    fontsize=9
)

plt.savefig(OUT_PDF, bbox_inches="tight")
plt.show()
print("Saved:", OUT_PDF)

# -----------------------------
# Quick printout: which BA label column used?
# -----------------------------
print("BA label column used:", name_col if name_col else f"(fallback) {BA_KEY}")
print("Scope 2 benchmark I_target (L/kWh):", round(I_target, 3))

In [ ]:
# ==========================================
# SI ROBUSTNESS FIGURES
# ==========================================
sites, basin, ba, st = _prep_conus(gdf_sites, gdf_basin, gdf_ba, states48)

for c in [SITE_BASIN_KEY, SITE_BA_KEY, S1_COL_M3, S2_COL_M3, STRESS_COL_SITE]:
    if c not in sites.columns:
        raise KeyError(f"gdf_sites missing `{c}` needed for SI robustness.")
for c in [BASIN_KEY, STRESS_COL_BASIN]:
    if c not in basin.columns:
        raise KeyError(f"gdf_basin missing `{c}` needed for SI robustness.")

sites[S1_COL_M3] = pd.to_numeric(sites[S1_COL_M3], errors="coerce").fillna(0.0)
sites[S2_COL_M3] = pd.to_numeric(sites[S2_COL_M3], errors="coerce").fillna(0.0)
sites[STRESS_COL_SITE] = pd.to_numeric(sites[STRESS_COL_SITE], errors="coerce").fillna(0.0)
basin[STRESS_COL_BASIN] = pd.to_numeric(basin[STRESS_COL_BASIN], errors="coerce").fillna(0.0)

# -----------------------------
# Build basin-level S1 totals
# -----------------------------
basins_with_sites = sites[SITE_BASIN_KEY].astype(str).unique()
s1_b = sites.groupby(SITE_BASIN_KEY)[S1_COL_M3].sum().rename("s1_m3").reset_index().rename(columns={SITE_BASIN_KEY:BASIN_KEY})
bas = basin.merge(s1_b, on=BASIN_KEY, how="left")
bas["s1_m3"] = bas["s1_m3"].fillna(0.0)
bas = bas[bas[BASIN_KEY].isin(basins_with_sites)].copy()

hi_stress_in = bas[STRESS_COL_BASIN].values >= 3.0
hi_stress_nb = _neighbor_aware_high_stress(bas, STRESS_COL_BASIN, thr=3.0).values

# helper
def jaccard(A, B):
    A = set(A); B = set(B)
    if len(A)==0 and len(B)==0:
        return 1.0
    return len(A & B) / len(A | B)

# -----------------------------
# Scope 1 robustness (threshold sensitivity)
# baseline: P50 + neighbor-aware stress
# -----------------------------
p_grid_s1 = np.arange(0.50, 0.91, 0.05)
baseline_thr_s1 = float(np.quantile(bas["s1_m3"].values, 0.50))
baseline_hot_s1 = set(bas.loc[(bas["s1_m3"] > baseline_thr_s1) & hi_stress_nb, BASIN_KEY])

jac_s1 = []
count_s1 = []
for p in p_grid_s1:
    thr = float(np.quantile(bas["s1_m3"].values, p))
    hot = set(bas.loc[(bas["s1_m3"] > thr) & hi_stress_nb, BASIN_KEY])
    jac_s1.append(jaccard(hot, baseline_hot_s1))
    count_s1.append(len(hot))

# Neighbor sensitivity (in-basin vs neighbor-aware)
jac_s1_in_vs_nb = []
for p in p_grid_s1:
    thr = float(np.quantile(bas["s1_m3"].values, p))
    hot_in = set(bas.loc[(bas["s1_m3"] > thr) & hi_stress_in, BASIN_KEY])
    hot_nb = set(bas.loc[(bas["s1_m3"] > thr) & hi_stress_nb, BASIN_KEY])
    jac_s1_in_vs_nb.append(jaccard(hot_in, hot_nb))

# -----------------------------
# Scope 2 robustness (threshold + fossil + stress)
# -----------------------------
# BA aggregation
ba_with_sites = sites[SITE_BA_KEY].astype(str).unique()
r_s2 = sites.groupby(SITE_BA_KEY)[S2_COL_M3].sum().rename("s2_m3").reset_index().rename(columns={SITE_BA_KEY:BA_KEY})
r_st = sites.groupby(SITE_BA_KEY)[STRESS_COL_SITE].mean().rename("stress_mean").reset_index().rename(columns={SITE_BA_KEY:BA_KEY})
rb = ba.merge(r_s2, on=BA_KEY, how="left").merge(r_st, on=BA_KEY, how="left")
rb[["s2_m3","stress_mean"]] = rb[["s2_m3","stress_mean"]].fillna(0.0)
rb = rb[rb[BA_KEY].isin(ba_with_sites)].copy()

# fossil share detection
fossil_ba, note = _find_fossil_share_series(rb)
if fossil_ba is None:
    fossil_site, note2 = _find_fossil_share_series(sites)
    if fossil_site is not None:
        tmp = sites[[SITE_BA_KEY]].copy()
        tmp["fossil_share"] = pd.to_numeric(fossil_site, errors="coerce")
        if tmp["fossil_share"].max(skipna=True) > 1.5:
            tmp["fossil_share"] = tmp["fossil_share"] / 100.0
        rb = rb.set_index(BA_KEY)
        rb["fossil_share"] = tmp.groupby(SITE_BA_KEY)["fossil_share"].mean()
        rb = rb.reset_index()
        note = f"aggregated from sites ({note2})"
    else:
        rb["fossil_share"] = np.nan
        note = None
else:
    rb["fossil_share"] = fossil_ba

p_grid_s2 = np.arange(0.70, 0.96, 0.05)
baseline_thr_s2 = float(np.quantile(rb["s2_m3"].values, 0.75))
baseline_hot_s2 = set(rb.loc[
    (rb["s2_m3"] >= baseline_thr_s2) &
    (rb["stress_mean"] >= 3.0) &
    (pd.to_numeric(rb["fossil_share"], errors="coerce") > 0.50),
    BA_KEY
]) if rb["fossil_share"].notna().any() else set()

jac_s2 = []
count_s2 = []
for p in p_grid_s2:
    thr = float(np.quantile(rb["s2_m3"].values, p))
    if rb["fossil_share"].notna().any():
        hot = set(rb.loc[
            (rb["s2_m3"] >= thr) &
            (rb["stress_mean"] >= 3.0) &
            (pd.to_numeric(rb["fossil_share"], errors="coerce") > 0.50),
            BA_KEY
        ])
    else:
        # fallback: without fossil criterion
        hot = set(rb.loc[
            (rb["s2_m3"] >= thr) &
            (rb["stress_mean"] >= 3.0),
            BA_KEY
        ])
    jac_s2.append(jaccard(hot, baseline_hot_s2) if len(baseline_hot_s2) else np.nan)
    count_s2.append(len(hot))

# Fossil threshold sensitivity (if fossil share exists)
fossil_grid = np.arange(0.40, 0.61, 0.05)
jac_fossil = []
count_fossil = []
if rb["fossil_share"].notna().any():
    for fthr in fossil_grid:
        hot = set(rb.loc[
            (rb["s2_m3"] >= baseline_thr_s2) &
            (rb["stress_mean"] >= 3.0) &
            (pd.to_numeric(rb["fossil_share"], errors="coerce") > fthr),
            BA_KEY
        ])
        jac_fossil.append(jaccard(hot, baseline_hot_s2) if len(baseline_hot_s2) else np.nan)
        count_fossil.append(len(hot))

# -----------------------------
# Plot SI_Hotspot_ThresholdRobustness.pdf (2 panels)
# -----------------------------
fig = plt.figure(figsize=(14.5, 5.8))
gs = fig.add_gridspec(nrows=1, ncols=2, left=0.06, right=0.985, top=0.88, bottom=0.14, wspace=0.22)
axA = fig.add_subplot(gs[0,0])
axB = fig.add_subplot(gs[0,1])

axA.plot(p_grid_s1, jac_s1, lw=2.2, marker="o")
axA.set_title("a. Scope 1 hotspot stability vs threshold\n(Jaccard vs baseline: P50 + neighbor-stress)")
axA.set_xlabel("Consumption threshold percentile")
axA.set_ylabel("Jaccard similarity")
axA.set_ylim(0, 1.02)
axA.grid(alpha=0.25)
axA2 = axA.twinx()
axA2.plot(p_grid_s1, count_s1, lw=1.6, linestyle="--", marker="s")
axA2.set_ylabel("Hotspot count")

axB.plot(p_grid_s2, jac_s2, lw=2.2, marker="o")
titleB = "b. Scope 2 hotspot stability vs threshold\n(Jaccard vs baseline: P75 + fossil>0.5 + stress>=3)"
if note is None:
    titleB = "b. Scope 2 stability vs threshold (no fossil criterion detected)"
axB.set_title(titleB)
axB.set_xlabel("Consumption threshold percentile")
axB.set_ylabel("Jaccard similarity")
axB.set_ylim(0, 1.02)
axB.grid(alpha=0.25)
axB2 = axB.twinx()
axB2.plot(p_grid_s2, count_s2, lw=1.6, linestyle="--", marker="s")
axB2.set_ylabel("Hotspot count")

out_pdf = os.path.join(OUTDIR, "SI_Hotspot_ThresholdRobustness.pdf")
plt.savefig(out_pdf, bbox_inches="tight")
plt.show()
print("Saved:", out_pdf)

# -----------------------------
# Plot SI_NeighborStress_Sensitivity.pdf
# -----------------------------
fig = plt.figure(figsize=(8.8, 5.4))
ax = fig.add_subplot(111)
ax.plot(p_grid_s1, jac_s1_in_vs_nb, lw=2.2, marker="o")
ax.set_title("Scope 1 stress definition sensitivity\n(Jaccard: in-basin stress vs neighbor-aware stress)")
ax.set_xlabel("Consumption threshold percentile")
ax.set_ylabel("Jaccard similarity")
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.25)

out_pdf = os.path.join(OUTDIR, "SI_NeighborStress_Sensitivity.pdf")
plt.savefig(out_pdf, bbox_inches="tight")
plt.show()
print("Saved:", out_pdf)

# -----------------------------
# Plot SI_FossilThreshold_Sensitivity.pdf (if possible)
# -----------------------------
if rb["fossil_share"].notna().any():
    fig = plt.figure(figsize=(8.8, 5.4))
    ax = fig.add_subplot(111)
    ax.plot(fossil_grid, jac_fossil, lw=2.2, marker="o")
    ax2 = ax.twinx()
    ax2.plot(fossil_grid, count_fossil, lw=1.6, linestyle="--", marker="s")

    ax.set_title("Scope 2 fossil-share threshold sensitivity\n(baseline P75 + stress>=3; vary fossil threshold)")
    ax.set_xlabel("Fossil-heavy threshold (coal+gas share)")
    ax.set_ylabel("Jaccard vs baseline hotspot set")
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.25)
    ax2.set_ylabel("Hotspot count")

    out_pdf = os.path.join(OUTDIR, "SI_FossilThreshold_Sensitivity.pdf")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()
    print("Saved:", out_pdf)
else:
    print("Skipped SI_FossilThreshold_Sensitivity.pdf (no fossil_share detected).")

# export

In [ ]:
# ============================================================
# WATER PAPER AUDIT EXPORT CELL
# ============================================================
# What this does:
# 1) Exports the key DataFrames / GeoDataFrames used in the notebook
# 2) Writes schema summaries for every DataFrame currently in memory
# 3) Saves sample rows and headline totals for fast debugging
# ============================================================

from pathlib import Path
import os
import json
import math
import warnings

import numpy as np
import pandas as pd

try:
    import geopandas as gpd
    HAS_GPD = True
except Exception:
    HAS_GPD = False
    gpd = None

warnings.filterwarnings("ignore")

# -----------------------------
# Settings
# -----------------------------
OUTDIR = Path("water_audit_export")
OUTDIR.mkdir(exist_ok=True, parents=True)

EXPORT_ALL_DATAFRAMES = False   # set True only if you want every df exported fully
MAX_ROWS_FOR_AUTO_EXPORT = 2_000_000
SAMPLE_N = 10

# -----------------------------
# Helper functions
# -----------------------------
def is_df(x):
    return isinstance(x, pd.DataFrame)

def is_gdf(x):
    return HAS_GPD and isinstance(x, gpd.GeoDataFrame)

def safe_json_value(x):
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, (np.bool_,)):
        return bool(x)
    if isinstance(x, (pd.Timestamp,)):
        return x.isoformat()
    if isinstance(x, (str, int, float, bool)) or x is None:
        return x
    return str(x)

def series_preview(s, n=5):
    vals = []
    try:
        uniq = s.dropna().astype(str).unique().tolist()[:n]
        vals = uniq
    except Exception:
        try:
            vals = [str(v) for v in s.dropna().head(n).tolist()]
        except Exception:
            vals = []
    return vals

def summarize_df(name, df):
    info = {
        "name": name,
        "type": "GeoDataFrame" if is_gdf(df) else "DataFrame",
        "n_rows": int(len(df)),
        "n_cols": int(len(df.columns)),
        "columns": list(map(str, df.columns)),
        "dtypes": {str(c): str(df[c].dtype) for c in df.columns},
        "non_null": {str(c): int(df[c].notna().sum()) for c in df.columns},
        "n_unique": {},
        "sample_values": {},
    }
    if is_gdf(df):
        try:
            info["crs"] = str(df.crs)
        except Exception:
            info["crs"] = None
        info["has_geometry"] = "geometry" in df.columns
    else:
        info["crs"] = None
        info["has_geometry"] = False

    for c in df.columns:
        c_str = str(c)
        try:
            info["n_unique"][c_str] = int(df[c].nunique(dropna=True))
        except Exception:
            info["n_unique"][c_str] = None
        info["sample_values"][c_str] = series_preview(df[c], n=5)
    return info

def export_dataframe(name, df, folder):
    """
    Export a DataFrame/GeoDataFrame in a robust way.
    Writes:
      - schema json
      - head csv
      - parquet/geoparquet if possible
      - csv fallback
    """
    base = folder / name
    schema = summarize_df(name, df)

    # schema
    with open(base.with_suffix(".schema.json"), "w") as f:
        json.dump(schema, f, indent=2)

    # head/sample
    try:
        df.head(50).to_csv(base.with_suffix(".head.csv"), index=False)
    except Exception:
        pass

    # random sample (deterministic)
    try:
        n = min(SAMPLE_N, len(df))
        if n > 0:
            df.sample(n=n, random_state=42).to_csv(base.with_suffix(".sample.csv"), index=False)
    except Exception:
        pass

    # main export
    exported = []
    if is_gdf(df):
        # geoparquet first
        try:
            df.to_parquet(base.with_suffix(".parquet"), index=False)
            exported.append(str(base.with_suffix(".parquet")))
        except Exception:
            pass
        # geopackage fallback
        try:
            df.to_file(base.with_suffix(".gpkg"), driver="GPKG")
            exported.append(str(base.with_suffix(".gpkg")))
        except Exception:
            pass
        # csv fallback with WKT geometry
        try:
            dfx = df.copy()
            if "geometry" in dfx.columns:
                dfx["geometry_wkt"] = dfx.geometry.astype(str)
            dfx.to_csv(base.with_suffix(".csv"), index=False)
            exported.append(str(base.with_suffix(".csv")))
        except Exception:
            pass
    else:
        try:
            df.to_parquet(base.with_suffix(".parquet"), index=False)
            exported.append(str(base.with_suffix(".parquet")))
        except Exception:
            pass
        try:
            df.to_csv(base.with_suffix(".csv"), index=False)
            exported.append(str(base.with_suffix(".csv")))
        except Exception:
            pass

    return {
        "name": name,
        "rows": int(len(df)),
        "cols": int(len(df.columns)),
        "exported_files": exported,
    }

def try_get(name):
    return globals().get(name, None)

def numeric_sum(df, col):
    if col in df.columns:
        return float(pd.to_numeric(df[col], errors="coerce").sum())
    return None

def numeric_count_nonnull(df, col):
    if col in df.columns:
        return int(pd.to_numeric(df[col], errors="coerce").notna().sum())
    return None

def maybe_add_headline_from_df(tag, df, out):
    if not is_df(df):
        return
    row = {"source_df": tag, "n_rows": int(len(df)), "n_cols": int(len(df.columns))}
    for c in [
        "current_mw", "mwh_fac_A_mid",
        "s1_m3_mid", "s2_m3_mid", "total_m3_mid",
        "grid_L_per_kwh", "bws_01_score",
        "pfaf_id", "subregion_id", "state", "STATE", "State"
    ]:
        if c in df.columns:
            if c in ["subregion_id", "pfaf_id", "state", "STATE", "State"]:
                try:
                    row[f"{c}_nunique"] = int(df[c].nunique(dropna=True))
                except Exception:
                    row[f"{c}_nunique"] = None
            else:
                row[f"{c}_sum"] = numeric_sum(df, c)
                row[f"{c}_nonnull"] = numeric_count_nonnull(df, c)
    out[tag] = row

# -----------------------------
# 1) Inventory all dataframes in memory
# -----------------------------
all_frames = {}
for name, obj in list(globals().items()):
    if is_df(obj):
        all_frames[name] = obj

inventory = []
for name, df_ in sorted(all_frames.items()):
    inventory.append(summarize_df(name, df_))

with open(OUTDIR / "ALL_DATAFRAME_INVENTORY.json", "w") as f:
    json.dump(inventory, f, indent=2)

# also save a compact CSV inventory
inv_rows = []
for item in inventory:
    inv_rows.append({
        "name": item["name"],
        "type": item["type"],
        "n_rows": item["n_rows"],
        "n_cols": item["n_cols"],
        "crs": item.get("crs"),
        "columns_pipe_sep": "|".join(item["columns"])
    })
pd.DataFrame(inv_rows).to_csv(OUTDIR / "ALL_DATAFRAME_INVENTORY.csv", index=False)

# -----------------------------
# 2) Export the exact targets I need
# -----------------------------
target_names = [
    # raw / early-stage
    "df",
    "mix_raw",
    "ba_polys",
    "gdf_basins",
    "aq",
    "gdf_h",
    "mix",
    "ba_polys_m",
    "joined",
    "gdf_p",
    "ba_p",
    "states48",
    "states_p",

    # CFG objects / late-stage
    "gdf_sites",
    "gdf_ba",
    "gdf_basin",
    "df_ba",
    "basin_tab",
    "basins_m",
    "states",
]

# Also pull from CFG if present
cfg_exports = {}
if "CFG" in globals() and isinstance(CFG, dict):
    for k in ["gdf_sites", "gdf_ba", "gdf_basin"]:
        if k in CFG and is_df(CFG[k]):
            cfg_exports[f"CFG__{k}"] = CFG[k]

export_manifest = []

# named variables in globals()
for name in target_names:
    obj = try_get(name)
    if is_df(obj):
        export_manifest.append(export_dataframe(name, obj, OUTDIR))

# CFG-derived objects
for name, obj in cfg_exports.items():
    export_manifest.append(export_dataframe(name, obj, OUTDIR))

# Optional: export every dataframe fully
if EXPORT_ALL_DATAFRAMES:
    for name, obj in sorted(all_frames.items()):
        if name not in target_names and f"CFG__{name}" not in cfg_exports:
            if len(obj) <= MAX_ROWS_FOR_AUTO_EXPORT:
                export_manifest.append(export_dataframe(f"AUTO__{name}", obj, OUTDIR))

with open(OUTDIR / "EXPORT_MANIFEST.json", "w") as f:
    json.dump(export_manifest, f, indent=2)

# -----------------------------
# 3) Headline checks for the national totals
# -----------------------------
headline = {}

# try key frames
for nm in ["df", "gdf_h", "gdf_sites", "CFG__gdf_sites", "df_ba"]:
    obj = None
    if nm.startswith("CFG__"):
        obj = cfg_exports.get(nm, None)
    else:
        obj = try_get(nm)
    if is_df(obj):
        maybe_add_headline_from_df(nm, obj, headline)

# Try direct calculations from candidate site-level frames
candidate_site_frames = []
for nm in ["gdf_h", "gdf_sites"]:
    obj = try_get(nm)
    if is_df(obj):
        candidate_site_frames.append((nm, obj))
if "CFG" in globals() and isinstance(CFG, dict) and "gdf_sites" in CFG and is_df(CFG["gdf_sites"]):
    candidate_site_frames.append(("CFG__gdf_sites", CFG["gdf_sites"]))

derived_checks = {}
for nm, df_ in candidate_site_frames:
    rec = {"source_df": nm}

    if "company_type" in df_.columns:
        try:
            rec["hyperscale_rows"] = int(
                df_["company_type"].astype(str).str.contains(
                    r"hyperscale|hyperscaler|hyper\s*scale",
                    case=False, na=False
                ).sum()
            )
        except Exception:
            rec["hyperscale_rows"] = None

    if "current_mw" in df_.columns:
        mw = pd.to_numeric(df_["current_mw"], errors="coerce")
        rec["mw_sum"] = float(mw.sum())
        rec["mw_nonnull"] = int(mw.notna().sum())

        # utilization-based annual electricity if 0.66
        rec["twh_from_mw_u066"] = float(mw.sum() * 8760 * 0.66 / 1e6)
        rec["twh_from_mw_u070"] = float(mw.sum() * 8760 * 0.70 / 1e6)
        rec["twh_from_mw_u085"] = float(mw.sum() * 8760 * 0.85 / 1e6)

    for c in ["mwh_fac_A_mid", "s1_m3_mid", "s2_m3_mid", "total_m3_mid"]:
        if c in df_.columns:
            x = pd.to_numeric(df_[c], errors="coerce")
            rec[f"{c}_sum"] = float(x.sum())
            rec[f"{c}_GL_if_m3"] = float(x.sum() / 1e6) if "m3" in c else None
            rec[f"{c}_TWh_if_MWh"] = float(x.sum() / 1e6) if "mwh" in c.lower() else None

    derived_checks[nm] = rec

with open(OUTDIR / "headline_checks.json", "w") as f:
    json.dump({
        "headline_from_frames": headline,
        "derived_checks": derived_checks,
    }, f, indent=2)

# -----------------------------
# 4) Save a plain-text columns dump for quick copy/paste
# -----------------------------
with open(OUTDIR / "COLUMNS_DUMP.txt", "w") as f:
    for item in inventory:
        f.write("=" * 100 + "\n")
        f.write(f"{item['name']}  [{item['type']}]  rows={item['n_rows']} cols={item['n_cols']} crs={item.get('crs')}\n")
        f.write("-" * 100 + "\n")
        for c in item["columns"]:
            nn = item["non_null"].get(c)
            nu = item["n_unique"].get(c)
            dt = item["dtypes"].get(c)
            sm = item["sample_values"].get(c)
            f.write(f"{c} | dtype={dt} | non_null={nn} | nunique={nu} | sample={sm}\n")
        f.write("\n")

# -----------------------------
# 5) Print a compact on-screen summary
# -----------------------------
print(f"\nExport folder: {OUTDIR.resolve()}\n")

print("Exported key tables:")
for row in export_manifest:
    print(f" - {row['name']}: {row['rows']:,} x {row['cols']:,}")

print("\nAll DataFrames currently in memory:")
for item in inventory:
    print(f" - {item['name']}: {item['type']} | {item['n_rows']:,} x {item['n_cols']:,}")

print("\nQuick headline checks:")
for k, v in derived_checks.items():
    print(f"\n[{k}]")
    for kk, vv in v.items():
        print(f"  {kk}: {vv}")

print("\nDone.")

In [ ]:
import pandas as pd

try:
    import geopandas as gpd
    HAS_GPD = True
except ImportError:
    HAS_GPD = False
    gpd = None

for name, obj in globals().items():
    is_gdf = HAS_GPD and isinstance(obj, gpd.GeoDataFrame)
    is_df = isinstance(obj, pd.DataFrame)

    if is_gdf or is_df:
        kind = "GeoDataFrame" if is_gdf else "DataFrame"
        print("=" * 100)
        print(f"{name}  |  {kind}  |  shape={obj.shape}")
        print("- columns -")
        for col in obj.columns:
            print(f"  {col}")
        print()

# SELF REVISION

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np

def compute_ba_supply_stress(CFG, ba_id_col="subregion_id", basin_stress_col="bws_01_score"):
    """
    Compute BA-level 'supply-region' scarcity proxy:
    area-weighted mean Aqueduct baseline stress across HydroBASINS polygons
    intersecting each BA.

    Returns:
      df_ba_supply: DataFrame with [ba_id, stress_supply_mean, stress_supply_p50, stress_supply_p90, n_parts, area_km2]
      gdf_ba_out: gdf_ba with stress_supply_mean joined (for mapping)
    """
    gdf_ba = CFG["gdf_ba"].copy()
    gdf_basin = CFG["gdf_basin"].copy()

    # --- Defensive checks
    assert ba_id_col in gdf_ba.columns, f"Missing {ba_id_col} in gdf_ba"
    assert basin_stress_col in gdf_basin.columns, f"Missing {basin_stress_col} in gdf_basin"

    # --- Project to equal-area CRS for area weights
    ea = "EPSG:5070"  # NAD83 / Conus Albers
    ba_ea = gdf_ba.to_crs(ea)
    basin_ea = gdf_basin.to_crs(ea)[[basin_stress_col, "geometry"]].copy()

    # Drop missing/invalid stress if any
    basin_ea = basin_ea.dropna(subset=[basin_stress_col])

    # --- Overlay BA × basin
    inter = gpd.overlay(
        ba_ea[[ba_id_col, "geometry"]],
        basin_ea,
        how="intersection",
        keep_geom_type=False
    )

    # --- Area weights
    inter["area_m2"] = inter.geometry.area
    inter["w"] = inter["area_m2"] / inter.groupby(ba_id_col)["area_m2"].transform("sum")

    # Weighted mean stress
    inter["stress_w"] = inter[basin_stress_col] * inter["w"]
    df_mean = inter.groupby(ba_id_col, as_index=False)["stress_w"].sum().rename(columns={"stress_w": "stress_supply_mean"})

    # Extra distribution summaries (useful for robustness / reviewer questions)
    # NOTE: these are unweighted quantiles; if you want area-weighted quantiles later, we can add.
    df_q = (
        inter.groupby(ba_id_col)[basin_stress_col]
        .quantile([0.5, 0.9])
        .unstack()
        .reset_index()
        .rename(columns={0.5: "stress_supply_p50", 0.9: "stress_supply_p90"})
    )

    df_meta = inter.groupby(ba_id_col, as_index=False).agg(
        n_parts=("area_m2", "size"),
        area_m2=("area_m2", "sum")
    )
    df_meta["area_km2"] = df_meta["area_m2"] / 1e6
    df_meta = df_meta.drop(columns=["area_m2"])

    df_ba_supply = df_mean.merge(df_q, on=ba_id_col, how="left").merge(df_meta, on=ba_id_col, how="left")

    # Join back to BA geodataframe for mapping
    gdf_ba_out = gdf_ba.merge(df_ba_supply[[ba_id_col, "stress_supply_mean"]], on=ba_id_col, how="left")

    return df_ba_supply, gdf_ba_out

In [ ]:
df_ba_supply, gdf_ba_supply = compute_ba_supply_stress(CFG)

# Example: merge into your BA summary table
# df_ba must contain 'subregion_id'
df_ba = ba_stats.merge(df_ba_supply, on="subregion_id", how="left")

In [ ]:
def classify_scope2_hotspots(df_ba, 
                            ba_id="subregion_id",
                            s2_col="s2_m3_mid",
                            grid_col="grid_L_per_kwh",
                            stress_supply_col="stress_supply_mean",
                            fossil_col="share_fossil",
                            q_s2=0.75,
                            q_grid=0.75,
                            stress_cut=3.0):
    """
    Returns df_ba with a categorical column 's2_hotspot_class' that is
    physically consistent: stress is supply-region stress.
    """
    out = df_ba.copy()

    # thresholds among BAs with hyperscale activity (assumes df_ba is already filtered that way)
    t_s2 = out[s2_col].quantile(q_s2)
    t_grid = out[grid_col].quantile(q_grid)

    out["high_s2"] = out[s2_col] >= t_s2
    out["high_grid_intensity"] = out[grid_col] >= t_grid
    out["high_supply_stress"] = out[stress_supply_col] >= stress_cut

    # Optional descriptor (not a rule)
    out["fossil_heavy"] = out[fossil_col] >= 0.5

    # 3-way intersection hotspot
    out["s2_hotspot"] = out["high_s2"] & out["high_grid_intensity"] & out["high_supply_stress"]

    # More informative class label for maps
    def _label(r):
        if r["s2_hotspot"]:
            return "Hotspot (high S2 × high grid-intensity × high supply-stress)"
        if r["high_s2"] and r["high_grid_intensity"]:
            return "High S2 × high grid-intensity"
        if r["high_s2"] and r["high_supply_stress"]:
            return "High S2 × high supply-stress"
        if r["high_grid_intensity"] and r["high_supply_stress"]:
            return "High grid-intensity × high supply-stress"
        if r["high_s2"]:
            return "High S2 only"
        if r["high_grid_intensity"]:
            return "High grid-intensity only"
        if r["high_supply_stress"]:
            return "High supply-stress only"
        return "Other"

    out["s2_hotspot_class"] = out.apply(_label, axis=1)

    # store thresholds (useful to print into Methods/SI)
    thresholds = {"t_s2": float(t_s2), "t_grid": float(t_grid), "stress_cut": float(stress_cut),
                  "q_s2": q_s2, "q_grid": q_grid}
    return out, thresholds

In [ ]:
import matplotlib.pyplot as plt

def plot_ba_scope2_hotspots(gdf_ba_supply, df_ba_classified,
                            ba_id="subregion_id",
                            class_col="s2_hotspot_class",
                            states_gdf=None,
                            figsize=(11,7),
                            title="Scope 2 hotspot typology (using supply-region stress proxy)"):
    g = gdf_ba_supply.merge(df_ba_classified[[ba_id, class_col]], on=ba_id, how="left")

    fig, ax = plt.subplots(1,1, figsize=figsize)
    if states_gdf is not None:
        states_gdf.boundary.plot(ax=ax, linewidth=0.4)

    # Let geopandas pick colors automatically (review-safe; you can later define palette)
    g.plot(column=class_col, ax=ax, legend=True, linewidth=0.2, edgecolor="white")

    ax.set_title(title)
    ax.set_axis_off()
    plt.tight_layout()
    return fig, ax

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# =========================
# 4) Map in Lambert Conformal Conic + states clearly shown
# =========================

# Lambert Conformal Conic for CONUS (common choice)
LCC = (
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 "
    "+lon_0=-96 +datum=WGS84 +units=m +no_defs"
)

# Prepare map layers
gdf_ba_map = gdf_ba.merge(df_ba[[BA_ID, "s2_hotspot_class", "s2_hotspot"]], on=BA_ID, how="left")

# Use contiguous states if available; else use full states
if "states48" in globals():
    states_outline = states48.copy()
elif "states" in globals():
    states_outline = states.copy()
else:
    states_outline = None

# Reproject to Lambert
gdf_ba_lcc = gdf_ba_map.to_crs(LCC)

states_lcc = states_outline.to_crs(LCC) if states_outline is not None else None

# Optional: also plot sites (helps reviewers visually validate allocation)
plot_sites = True
sites_lcc = gdf_sites.to_crs(LCC) if plot_sites else None

# ---- plot
fig, ax = plt.subplots(1, 1, figsize=(13, 8))

# 1) States fill (very light) + thicker boundaries
if states_lcc is not None:
    # Light state fill to distinguish states clearly
    states_lcc.plot(ax=ax, color="whitesmoke", edgecolor="none", linewidth=0)

    # Stronger state boundaries
    states_lcc.boundary.plot(ax=ax, linewidth=0.6)

# 2) BA typology polygons
# Let GeoPandas choose colors automatically (review-safe). If you want a fixed palette later, we can add it.
gdf_ba_lcc.plot(
    column="s2_hotspot_class",
    ax=ax,
    legend=True,
    linewidth=0.35,
    edgecolor="white"
)

# 3) State boundaries again on top (so they are not buried under BA colors)
if states_lcc is not None:
    states_lcc.boundary.plot(ax=ax, linewidth=0.6)

# 4) Optional site points
if plot_sites and sites_lcc is not None:
    # make points subtle (small) so they don't clutter
    sites_lcc.plot(ax=ax, markersize=6, color="black", alpha=0.55)

# 5) Framing
ax.set_title("Scope 2 hotspot typology by balancing authority (Lambert Conformal Conic)\n(using supply-region stress proxy)")
ax.set_axis_off()

# 6) Tight map bounds (optional)
# If you want to zoom to CONUS extent based on states
if states_lcc is not None:
    xmin, ymin, xmax, ymax = states_lcc.total_bounds
    pad_x = 0.02 * (xmax - xmin)
    pad_y = 0.02 * (ymax - ymin)
    ax.set_xlim(xmin - pad_x, xmax + pad_x)
    ax.set_ylim(ymin - pad_y, ymax + pad_y)

plt.tight_layout()
plt.show()

# Optional save
# fig.savefig("Fig_BA_scope2_hotspots_supply_stress_LCC.png", dpi=300, bbox_inches="tight")

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================
# 0) Inputs
# =========================
gdf_sites = CFG["gdf_sites"].copy()
gdf_ba = CFG["gdf_ba"].copy()
gdf_basin = CFG["gdf_basin"].copy()

# choose a state boundary layer if you have it
# (use states48 if you want contiguous US; else states)
states_outline = None
if "states48" in globals():
    states_outline = states48.copy()
elif "states" in globals():
    states_outline = states.copy()

# Column names (from your inventory)
BA_ID = "subregion_id"
S2 = "s2_m3_mid"
MW = "current_mw"
GRID = "grid_L_per_kwh"
FOSSIL = "share_fossil"
SITING_STRESS = "bws_01_score"   # facility-level Aqueduct baseline stress (0–5)
BASIN_STRESS = "bws_01_score"    # basin-level Aqueduct baseline stress (0–5)

# Basic sanity checks
required_site_cols = [BA_ID, S2, MW, GRID, FOSSIL, SITING_STRESS]
missing = [c for c in required_site_cols if c not in gdf_sites.columns]
if missing:
    raise ValueError(f"gdf_sites is missing required columns: {missing}")

if BA_ID not in gdf_ba.columns:
    raise ValueError(f"gdf_ba missing BA id column: {BA_ID}")

if BASIN_STRESS not in gdf_basin.columns:
    raise ValueError(f"gdf_basin missing basin stress column: {BASIN_STRESS}")


# =========================
# 1) Build BA summary table from site-level data
# =========================
# Note:
# - sum Scope 2 volume across sites
# - sum MW across sites
# - use MW-weighted means for grid intensity / fossil share / siting stress (better than simple mean)
tmp = gdf_sites[[BA_ID, S2, MW, GRID, FOSSIL, SITING_STRESS]].copy()

# Ensure numeric
for col in [S2, MW, GRID, FOSSIL, SITING_STRESS]:
    tmp[col] = pd.to_numeric(tmp[col], errors="coerce")

# Keep rows with valid BA and positive MW
tmp = tmp.dropna(subset=[BA_ID, MW])
tmp = tmp[tmp[MW] > 0].copy()

def mw_weighted_mean(df, value_col, weight_col=MW):
    v = df[value_col].to_numpy()
    w = df[weight_col].to_numpy()
    m = np.isfinite(v) & np.isfinite(w) & (w > 0)
    if m.sum() == 0:
        return np.nan
    return float(np.average(v[m], weights=w[m]))

df_ba = (
    tmp.groupby(BA_ID, as_index=False)
       .agg(
           n_sites=(BA_ID, "size"),
           mw_total=(MW, "sum"),
           s2_m3=(S2, "sum"),
       )
)

# Add MW-weighted descriptors
df_desc = (
    tmp.groupby(BA_ID)
       .apply(lambda d: pd.Series({
           "grid_L_per_kwh_mw_wtd": mw_weighted_mean(d, GRID),
           "share_fossil_mw_wtd": mw_weighted_mean(d, FOSSIL),
           "siting_stress_mw_wtd": mw_weighted_mean(d, SITING_STRESS),
           "siting_stress_simple_mean": float(np.nanmean(d[SITING_STRESS].to_numpy()))
       }))
       .reset_index()
)

df_ba = df_ba.merge(df_desc, on=BA_ID, how="left")

# Helpful derived metrics
df_ba["s2_GL_yr"] = df_ba["s2_m3"] / 1e6   # 1 GL = 1e6 m3
df_ba["s2_ML_per_MWyr"] = (df_ba["s2_m3"] * 1000.0) / df_ba["mw_total"] / 1e6
# Explanation: m3 -> L (×1000), then /MW, then /1e6 => ML/MW-yr

print("BA summary table built:")
print(df_ba.sort_values("s2_m3", ascending=False).head(8))


# =========================
# 2) Compute BA supply-region stress proxy (area-weighted basin stress within BA footprint)
# =========================
# Use equal-area projection for weighting by area
EA = "EPSG:5070"  # NAD83 / Conus Albers (good for US area stats)

ba_ea = gdf_ba[[BA_ID, "geometry"]].to_crs(EA).copy()
basin_ea = gdf_basin[[BASIN_STRESS, "geometry"]].to_crs(EA).copy()
basin_ea = basin_ea.dropna(subset=[BASIN_STRESS])

# Overlay BA x basins
inter = gpd.overlay(
    ba_ea,
    basin_ea,
    how="intersection",
    keep_geom_type=False
)

inter["area_m2"] = inter.geometry.area
# Normalize area weights within each BA
inter["w"] = inter["area_m2"] / inter.groupby(BA_ID)["area_m2"].transform("sum")

# Weighted mean stress
inter["stress_w"] = inter[BASIN_STRESS] * inter["w"]
df_supply_mean = (
    inter.groupby(BA_ID, as_index=False)["stress_w"]
         .sum()
         .rename(columns={"stress_w": "stress_supply_mean"})
)

# Optional extra summaries (unweighted quantiles; useful diagnostics)
df_supply_q = (
    inter.groupby(BA_ID)[BASIN_STRESS]
         .quantile([0.5, 0.9])
         .unstack()
         .reset_index()
         .rename(columns={0.5: "stress_supply_p50", 0.9: "stress_supply_p90"})
)

df_supply_meta = (
    inter.groupby(BA_ID, as_index=False)
         .agg(
             n_parts=("area_m2", "size"),
             area_m2=("area_m2", "sum")
         )
)
df_supply_meta["area_km2"] = df_supply_meta["area_m2"] / 1e6
df_supply_meta = df_supply_meta.drop(columns=["area_m2"])

df_ba_supply = df_supply_mean.merge(df_supply_q, on=BA_ID, how="left").merge(df_supply_meta, on=BA_ID, how="left")

# Merge into BA summary
df_ba = df_ba.merge(df_ba_supply, on=BA_ID, how="left")

print("\nSupply-region stress computed (joined to df_ba):")
print(df_ba[[BA_ID, "stress_supply_mean", "stress_supply_p50", "stress_supply_p90"]].head(8))


# =========================
# 3) Classify Scope 2 hotspot typology using supply stress (physically consistent)
# =========================
# Thresholds:
q_s2 = 0.75
q_grid = 0.75
stress_cut = 3.0

# Use MW-weighted grid intensity for BA-level classification
GRID_BA = "grid_L_per_kwh_mw_wtd"

t_s2 = df_ba["s2_m3"].quantile(q_s2)
t_grid = df_ba[GRID_BA].quantile(q_grid)

df_ba["high_s2"] = df_ba["s2_m3"] >= t_s2
df_ba["high_grid_intensity"] = df_ba[GRID_BA] >= t_grid
df_ba["high_supply_stress"] = df_ba["stress_supply_mean"] >= stress_cut

# fossil-heavy is an informative descriptor (not required for hotspot)
df_ba["fossil_heavy"] = df_ba["share_fossil_mw_wtd"] >= 0.5

df_ba["s2_hotspot"] = df_ba["high_s2"] & df_ba["high_grid_intensity"] & df_ba["high_supply_stress"]

def label_row(r):
    if r["s2_hotspot"]:
        return "Hotspot (high S2 × high grid-intensity × high supply-stress)"
    if r["high_s2"] and r["high_grid_intensity"]:
        return "High S2 × high grid-intensity"
    if r["high_s2"] and r["high_supply_stress"]:
        return "High S2 × high supply-stress"
    if r["high_grid_intensity"] and r["high_supply_stress"]:
        return "High grid-intensity × high supply-stress"
    if r["high_s2"]:
        return "High S2 only"
    if r["high_grid_intensity"]:
        return "High grid-intensity only"
    if r["high_supply_stress"]:
        return "High supply-stress only"
    return "Other"

df_ba["s2_hotspot_class"] = df_ba.apply(label_row, axis=1)

print("\nHotspot thresholds:")
print({"t_s2_m3": float(t_s2), "t_grid_L_per_kwh": float(t_grid), "stress_cut": stress_cut})
print("\nCounts by class:")
print(df_ba["s2_hotspot_class"].value_counts())

print("\nTop BAs by S2 (with new supply-stress):")
print(df_ba.sort_values("s2_m3", ascending=False)[
    [BA_ID, "s2_GL_yr", "mw_total", GRID_BA, "stress_supply_mean", "share_fossil_mw_wtd", "s2_hotspot"]
].head(12))


# =========================
# 4) Map: BA polygons colored by typology
# =========================
gdf_ba_map = gdf_ba.merge(df_ba[[BA_ID, "s2_hotspot_class", "s2_hotspot"]], on=BA_ID, how="left")

fig, ax = plt.subplots(1, 1, figsize=(12, 7))

if states_outline is not None:
    states_outline.boundary.plot(ax=ax, linewidth=0.4)

gdf_ba_map.plot(
    column="s2_hotspot_class",
    ax=ax,
    legend=True,
    linewidth=0.2,
    edgecolor="white"
)

ax.set_title("Scope 2 hotspot typology (using supply-region stress proxy)")
ax.set_axis_off()
plt.tight_layout()
plt.show()

# =========================
# 5) Optional: save outputs (table + map)
# =========================
# df_ba.to_csv("BA_scope2_hotspots_supply_stress.csv", index=False)
# fig.savefig("Fig_BA_scope2_hotspots_supply_stress.png", dpi=300, bbox_inches="tight")

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

# -------------------------
# Lambert Conformal Conic (CONUS)
# -------------------------
LCC = (
    "+proj=lcc +lat_1=33 +lat_2=45 +lat_0=39 "
    "+lon_0=-96 +datum=WGS84 +units=m +no_defs"
)

BA_ID = "subregion_id"

# -------------------------
# Build BA map dataframe
# -------------------------
gdf_ba_map = CFG["gdf_ba"].copy().merge(
    df_ba[[BA_ID, "s2_hotspot_class", "s2_hotspot"]],
    on=BA_ID,
    how="left"
)

# -------------------------
# CONUS states (no Alaska/Hawaii)
# -------------------------
if "states48" in globals():
    conus_states = states48.copy()  # usually CONUS + DC
elif "states" in globals():
    conus_states = states.copy()
    conus_states = conus_states[~conus_states["STUSPS"].isin(["AK", "HI", "PR", "GU", "VI", "MP", "AS"])].copy()
else:
    conus_states = None

# -------------------------
# Reproject layers
# -------------------------
gdf_ba_lcc = gdf_ba_map.to_crs(LCC)

conus_states_lcc = conus_states.to_crs(LCC) if conus_states is not None else None

# Optional: sites (helps visually show clustering)
plot_sites = True
sites_lcc = CFG["gdf_sites"].to_crs(LCC) if plot_sites else None

# -------------------------
# Optional: clip BA polygons to CONUS outline (cleaner)
# -------------------------
if conus_states_lcc is not None:
    conus_union = conus_states_lcc.unary_union
    gdf_ba_lcc = gdf_ba_lcc.copy()
    gdf_ba_lcc["geometry"] = gdf_ba_lcc.geometry.intersection(conus_union)

# -------------------------
# Plot
# -------------------------
fig, ax = plt.subplots(1, 1, figsize=(13, 8))

# 1) States: light fill + clear boundaries
if conus_states_lcc is not None:
    conus_states_lcc.plot(ax=ax, color="whitesmoke", edgecolor="none", linewidth=0)
    conus_states_lcc.boundary.plot(ax=ax, linewidth=0.7)

# 2) BA typology polygons
gdf_ba_lcc.plot(
    column="s2_hotspot_class",
    ax=ax,
    legend=True,
    linewidth=0.35,
    edgecolor="white"
)

# 3) Draw state boundaries on top again (so they pop)
if conus_states_lcc is not None:
    conus_states_lcc.boundary.plot(ax=ax, linewidth=0.8)

# 4) Optional site points
if plot_sites and sites_lcc is not None:
    # Clip sites to CONUS bounds for cleanliness
    if conus_states_lcc is not None:
        sites_lcc = sites_lcc[sites_lcc.geometry.within(conus_states_lcc.unary_union)]
    sites_lcc.plot(ax=ax, markersize=6, color="black", alpha=0.55)

# 5) Frame
ax.set_title(
    "Scope 2 hotspot typology by balancing authority (CONUS, Lambert)\n"
    "using supply-region stress proxy (HydroBASINS × Aqueduct within BA)"
)
ax.set_axis_off()

# 6) Tight bounds to CONUS
if conus_states_lcc is not None:
    xmin, ymin, xmax, ymax = conus_states_lcc.total_bounds
    pad_x = 0.02 * (xmax - xmin)
    pad_y = 0.02 * (ymax - ymin)
    ax.set_xlim(xmin - pad_x, xmax + pad_x)
    ax.set_ylim(ymin - pad_y, ymax + pad_y)

plt.tight_layout()
plt.show()

# Optional save
# fig.savefig("Fig_BA_scope2_hotspots_supply_stress_CONUS_LCC.png", dpi=300, bbox_inches="tight")